# Verified same-image alignment — F, hai GPU

Một nhánh trên **GPU T4 ×2**, Internet ON, **Save & Run All**. Bạn tự chạy notebook.
Giữ kiến trúc E và nền caption SCST+CE+thưởng đối tượng; thay loss giả nhãn bằng nhãn V-COCO xác nhận trên cùng ảnh.
Không cần thêm dataset: nhãn và mã đã nhúng sẵn. Giữ sáu Input giống A/E.

503 cặp fit / 310 ảnh; 140 cặp calibration / 79 ảnh; probe cũ 62 cặp / 59 ảnh.
Không dùng nhãn calibration/probe để huấn luyện hay chọn checkpoint. Không dùng test caption.
Tập fit nhỏ và được lặp lại; chưa có bằng chứng phương pháp này cải thiện caption.


In [ ]:
print('VERIFIED SAME IMAGE F: BOOTSTRAP START', flush=True)
from pathlib import Path

ARM = 'F'
CHECKPOINT = Path('/kaggle/input/datasets/ducanh2403/qformer-scst-object-w05-legacy/qformer_scst_object_w05_legacy.pth')
LEGACY_PROMPT = Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
UPT_PROMPT = Path('/kaggle/input/datasets/ducanh2403/upt-prompts-full-all-actions-v2/prompt_cache.json')
VISUAL_CACHE = Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON = Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS = Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT = Path('/kaggle/working/semantic_evidence_verified_pairs_F')

# Bạn của bạn sửa các mount ở trên nếu khác; giữ nguyên nội dung đầu vào.
# Không cần COCO_IMAGES hay raw UPT cache: dùng visual HDF5 + prompt JSON đã có.

# Khởi tạo từ checkpoint nền gốc (SHA 5f8...), KHÔNG dùng checkpoint E/bin.


## 1. Chuẩn bị môi trường và mã nguồn
Kho mã đặt trong `/tmp` để không chiếm danh sách Output. Không cài lại PyTorch của Kaggle.

In [ ]:
import json, os, shutil, subprocess, sys
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', CUBLAS_WORKSPACE_CONFIG=':4096:8',
                  HF_HUB_DISABLE_XET='1', HF_HUB_DOWNLOAD_TIMEOUT='120', TOKENIZERS_PARALLELISM='false',
                  PYTHONHASHSEED='42', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
REPO = Path('/tmp/Image_Captioning-verified-same-image')
CHAIR = Path('/tmp/semantic-evidence-pilot-chair')
SETUP_LOG = Path('/tmp/semantic_evidence_pilot_setup_' + ARM + '.log')
PROJECT_COMMIT = 'c09823178a9b28186e71554d368e2716c22af176'

def run(args, cwd=None, log=None):
    args = list(map(str,args))
    print('\nRunning:', ' '.join(args), flush=True)
    with Path(log or SETUP_LOG).open('a',encoding='utf-8') as output:
        output.write('\nRunning: ' + ' '.join(args) + '\n'); output.flush()
        with subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,errors='replace',bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True); output.write(line); output.flush()
            status = process.wait()
    if status: raise RuntimeError(f'Command failed ({status}); full log: {log or SETUP_LOG}')

def checkout(url,path,revision):
    if not path.exists(): run(['git','clone','--no-checkout',url,path])
    assert (path/'.git').is_dir(),path
    run(['git','fetch','--depth','1','origin',revision],cwd=path)
    run(['git','checkout','--detach',revision],cwd=path)

for file in (CHECKPOINT,LEGACY_PROMPT,UPT_PROMPT,COCO_JSON,DETECTIONS):
    assert file.is_file(),f'Missing exact input: {file}. Copy mounted path from Input.'
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
assert not OUTPUT.exists(),f'Choose a fresh OUTPUT or restart session: {OUTPUT}'
import torch
assert torch.cuda.device_count()==2,'Select GPU T4 x2: ONE notebook, ONE arm, TWO GPUs.'
for i in range(2): print('GPU',i,torch.cuda.get_device_name(i),flush=True)
checkout('https://github.com/Supzxjee/Image_Captioning.git',REPO,PROJECT_COMMIT)
checkout('https://github.com/Maxlinn/CHAIR-metric-standalone.git',CHAIR,'4087a26211aa2339b9a76307cb8f0321ef691d0a')
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocotools','pycocoevalcap==1.2',
     'h5py','pandas','Pillow','tqdm','scipy','nltk==3.9.2','future'])
run([sys.executable,'-m','pip','install','-q','--no-deps',
     'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
run(['java','-version'])

SOURCES = {'pilot_model.py': '"""Same evidence architecture as E; replace cross-image pseudo-pairs with verified pairs."""\nimport torch\nfrom torch.nn import functional as F\nfrom evidence_core import EvidenceMemory, PilotModel as OriginalPilotModel\n\n\ndef pair_loss(scores, positive_index, valid):\n    positive=scores.gather(1,positive_index[:,None]).squeeze(1)\n    negative=scores.gather(1,(1-positive_index)[:,None]).squeeze(1)\n    margin=positive-negative\n    if not bool(valid.any()): return scores.sum()*0,dict(pairs=0,accuracy=None,margin=None)\n    return F.relu(.2-margin[valid]).mean(),dict(pairs=int(valid.sum()),\n        accuracy=float((margin[valid]>0).float().mean().detach()),margin=float(margin[valid].mean().detach()))\n\n\nclass PilotModel(OriginalPilotModel):\n    def __init__(self,base,arm):\n        if arm not in (\'A\',\'F\'): raise ValueError(arm)\n        super().__init__(base,\'A\' if arm==\'A\' else \'E\')\n        self.arm=arm\n\n    def forward(self,batch,sample,active,advantage,scst_weight,pad_idx):\n        from captioning.scst import policy_loss\n        memory,padding=self.memory(batch)\n        targets=batch[\'captions\'].flatten(0,1); masks=batch[\'caption_mask\'].flatten(0,1)\n        logits=self.teacher(memory,padding,targets[:,:-1],masks[:,:-1])\n        ce=F.cross_entropy(logits.flatten(0,1),targets[:,1:].reshape(-1),ignore_index=pad_idx)\n        rl=ce.new_zeros(())\n        if scst_weight: rl=policy_loss(self.teacher(memory,padding,sample[:,:-1]),sample[:,1:],active,advantage)\n        alignment=ce.new_zeros(());info=dict(pairs=0,accuracy=None,margin=None)\n        if self.extra is not None:\n            n=len(batch[\'aux_visual\'])\n            patches=batch[\'aux_visual\'][:,None,1:].expand(-1,2,-1,-1).reshape(n*2,196,768)\n            text=batch[\'aux_text\'].reshape(n*2,512)\n            boxes=batch[\'aux_bounds\'].reshape(n*2,3,4)\n            roles=torch.full((n*2,),2,dtype=torch.long,device=text.device)\n            _,scores,_=self.extra.attend(patches,text,roles,boxes)\n            alignment,info=pair_loss(scores.reshape(n,2),batch[\'aux_positive\'],batch[\'aux_valid\'])\n        self.last_alignment=dict(loss=float(alignment.detach()),**info)\n        return ce+scst_weight*rl+.1*alignment,ce.detach(),rl.detach()\n', 'run_verified.py': '"""Reuse the audited caption runner; add fit-only GT pairs and matcher diagnostics."""\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport numpy as np\nimport base_runner as runner\nfrom pilot_data import digest,identity,save\nfrom verified_data import training_schedule,geometry_summary\nfrom probe_data import summarize\n\nORIGINAL_LOADER=runner.make_loader\nORIGINAL_PREDICT=runner.predict\n\n\ndef manifest(cfg):\n    path=Path(cfg[\'verified_manifest\'])\n    if digest(path)!=cfg[\'verified_manifest_sha256\']: raise ValueError(\'Verified labels changed\')\n    data=json.loads(path.read_text())\n    assert list(data[\'splits\'])==[\'fit\',\'calibration\',\'probe\']\n    ids={k:{r[\'coco_id\'] for r in rows} for k,rows in data[\'splits\'].items()}\n    assert not ids[\'fit\']&ids[\'calibration\'] and not ids[\'fit\']&ids[\'probe\'] and not ids[\'calibration\']&ids[\'probe\']\n    for key,rows in data[\'splits\'].items(): assert identity(rows)==data[\'split_sha256\'][key]\n    assert data[\'schedule\']==training_schedule(data[\'splits\'][\'fit\'])\n    assert len(data[\'schedule\'])==10000 and all(0<=i<len(data[\'splits\'][\'fit\']) for i in data[\'schedule\'])\n    assert not data[\'test_used\']\n    return data\n\n\ndef prepare(config):\n    import h5py\n    import torch\n    from transformers import CLIPTextModel,AutoTokenizer\n    from captioning.visual_cache import VisualCache\n    cfg=json.loads(Path(config).read_text()); labels=manifest(cfg)\n    runner.prepare(config)\n    protocol=Path(cfg[\'output\'])/\'protocol.json\';cfg=json.loads(protocol.read_text())\n    stage=Path(cfg[\'aux_stage\'])\n    if stage.exists(): raise FileExistsError(stage)\n    rows=[r for split in (\'fit\',\'calibration\',\'probe\') for r in labels[\'splits\'][split]]\n    ids=sorted({r[\'coco_id\'] for r in rows});index={iid:i for i,iid in enumerate(ids)}\n    required=len(ids)*197*768*4+len(rows)*2*512*4\n    if shutil.disk_usage(stage.parent).free<required+1024**3: raise RuntimeError(\'Not enough space for auxiliary stage\')\n    cache=VisualCache([cfg[\'visual_cache\']],id_key=\'coco_id\');cache.require_ids(ids,\'verified relation partitions\')\n    try:\n        with h5py.File(stage,\'w\') as f:\n            visual=f.create_dataset(\'visual\',(len(ids),197,768),dtype=\'f4\',chunks=(1,197,768))\n            for count,iid in enumerate(sorted(ids,key=lambda x:cache.index[x]),1):\n                visual[index[iid]]=cache.read(iid)\n                if count%100==0 or count==len(ids): print(f\'Verified image features: {count}/{len(ids)}\',flush=True)\n            f.create_dataset(\'image_index\',data=np.asarray([index[r[\'coco_id\']] for r in rows],dtype=np.int64))\n            f.create_dataset(\'bounds\',data=np.asarray([[c[\'bounds\'] for c in r[\'candidates\']] for r in rows],dtype=np.float32))\n            f.create_dataset(\'positive\',data=np.asarray([r[\'positive_index\'] for r in rows],dtype=np.int64))\n    finally: cache.close()\n    encoder=CLIPTextModel.from_pretrained(runner.CLIP,revision=runner.CLIP_REVISION).cuda().eval()\n    tokenizer=AutoTokenizer.from_pretrained(runner.CLIP,revision=runner.CLIP_REVISION)\n    texts=sorted({c[\'text\'] for r in rows for c in r[\'candidates\']});embedding={}\n    with torch.inference_mode():\n        for start in range(0,len(texts),128):\n            batch=texts[start:start+128]\n            token=tokenizer(batch,padding=True,truncation=True,max_length=77,return_tensors=\'pt\').to(\'cuda\')\n            for text,value in zip(batch,encoder(**token).pooler_output.float().cpu().numpy()): embedding[text]=value\n    with h5py.File(stage,\'r+\') as f:\n        f.create_dataset(\'text\',data=np.asarray([[embedding[c[\'text\']] for c in r[\'candidates\']] for r in rows],dtype=np.float32))\n        f.attrs[\'verified_manifest_sha256\']=cfg[\'verified_manifest_sha256\'];f.attrs[\'complete\']=True\n    # The underlying runner\'s generic E description is replaced before workers lock protocol SHA.\n    cfg.update(method=\'verified_same_image_F\',alignment_weight=.1,alignment_margin=.2,aux_stage_sha256=digest(stage),\n        negative_policy=labels[\'policy\'],verified_counts=labels[\'counts\'],\n        loss=\'CE + scheduled(.05)*SCST(CIDEr-D+.5*object_count_v2) + .1*verified_same_image_R_hinge_margin.2\',\n        comparison_limits=\'Same caption cohort/updates/architecture as E; additional verified fit-only relation labels and auxiliary image exposures. Not equal annotation/compute budget to A/E.\',\n        module_claim=\'Unchanged E module; modified relation supervision, not an established novel architecture\')\n    fit_ids={r[\'coco_id\'] for r in labels[\'splits\'][\'fit\']}\n    probe_ids={r[\'coco_id\'] for r in labels[\'splits\'][\'probe\']}\n    main_train={r[\'coco_id\'] for r in cfg[\'groups\'][\'train\']}\n    assert not main_train&probe_ids\n    cfg[\'auxiliary_main_caption_overlap\']=len(main_train&fit_ids)\n    save(protocol,cfg);save(Path(cfg[\'output\'])/\'verified_manifest.json\',labels)\n    print(\'VERIFIED RELATION PREPARATION PASS\',labels[\'counts\'],flush=True)\n\n\ndef make_loader(cfg,dataset,tokenizer,split,rank,world):\n    import h5py\n    import torch\n    from torch.utils.data import Dataset,DataLoader,DistributedSampler\n    original=ORIGINAL_LOADER(cfg,dataset,tokenizer,split,rank,world)\n    original.verified_cfg=cfg\n    if split!=\'train\': return original\n    labels=manifest(cfg);schedule=labels[\'schedule\']\n    class Augmented(Dataset):\n        def __init__(self):\n            self.aux=h5py.File(cfg[\'aux_stage\'],\'r\')\n            assert self.aux.attrs.get(\'complete\') and self.aux.attrs[\'verified_manifest_sha256\']==cfg[\'verified_manifest_sha256\']\n        def __len__(self):return len(original.dataset)\n        def __getitem__(self,index):\n            row=original.dataset[index];i=schedule[index]\n            for key,source in [(\'aux_visual\',self.aux[\'visual\'][int(self.aux[\'image_index\'][i])]),\n                               (\'aux_text\',self.aux[\'text\'][i]),(\'aux_bounds\',self.aux[\'bounds\'][i])]:\n                row[key]=torch.from_numpy(source)\n            row[\'aux_positive\']=torch.tensor(int(self.aux[\'positive\'][i]),dtype=torch.long)\n            row[\'aux_valid\']=torch.tensor(True)\n            return row\n    data=Augmented();sampler=DistributedSampler(data,num_replicas=world,rank=rank,shuffle=True,seed=42,drop_last=False)\n    return DataLoader(data,batch_size=cfg[\'global_batch_size\']//world,sampler=sampler,num_workers=0,pin_memory=True)\n\n\ndef score_matcher(model,cfg,phase):\n    import h5py\n    import torch\n    rank=int(os.environ[\'RANK\']);labels=manifest(cfg);offset=0\n    with h5py.File(cfg[\'aux_stage\'],\'r\') as f,torch.no_grad():\n        for split,rows in labels[\'splits\'].items():\n            if split==\'fit\': offset+=len(rows);continue\n            selected=set(sorted({r[\'coco_id\'] for r in rows})[rank::2]);records=[]\n            for j,row in enumerate(rows):\n                if row[\'coco_id\'] not in selected:continue\n                i=offset+j\n                visual=torch.from_numpy(f[\'visual\'][int(f[\'image_index\'][i])][1:]).cuda()[None].expand(2,-1,-1)\n                text=torch.from_numpy(f[\'text\'][i]).cuda();boxes=torch.from_numpy(f[\'bounds\'][i]).cuda()\n                roles=torch.full((2,),2,dtype=torch.long,device=text.device);scores={}\n                for method in (\'boxed\',\'global_view\',\'zero_visual\'):\n                    p,b=visual,boxes\n                    if method==\'global_view\': b=torch.zeros_like(boxes);b[:,2,2:]=1\n                    if method==\'zero_visual\':p=torch.zeros_like(visual)\n                    _,values,_=model.extra.attend(p,text,roles,b)\n                    if not torch.isfinite(values).all():raise FloatingPointError(\'Nonfinite verified diagnostic\')\n                    scores[method]=values.cpu().tolist()\n                records.append(dict(row,scores=scores))\n            save(Path(cfg[\'output\'])/f\'matcher_{phase}_{split}_rank{rank}.json\',records)\n            offset+=len(rows)\n\n\ndef predict(model,loader,tokenizer,intervention):\n    result=ORIGINAL_PREDICT(model,loader,tokenizer,intervention)\n    if not getattr(model,\'_verified_before\',False):\n        score_matcher(model,loader.verified_cfg,\'before\');model._verified_before=True\n    elif intervention==\'on\': score_matcher(model,loader.verified_cfg,\'after\')\n    return result\n\n\ndef evaluate(protocol):\n    runner.evaluate(protocol)\n    cfg=json.loads(Path(protocol).read_text());labels=manifest(cfg);out=Path(cfg[\'output\']);result={}\n    for phase in (\'before\',\'after\'):\n        result[phase]={}\n        for split in (\'calibration\',\'probe\'):\n            rows=sum((json.loads((out/f\'matcher_{phase}_{split}_rank{r}.json\').read_text()) for r in (0,1)),[])\n            rows.sort(key=lambda r:(r[\'task\'],r[\'case_id\']))\n            assert [{k:v for k,v in r.items() if k!=\'scores\'} for r in rows]==labels[\'splits\'][split]\n            result[phase][split]=summarize(rows)\n            save(out/f\'matcher_{phase}_{split}.json\',rows)\n    result[\'geometry_controls\']={split:geometry_summary(labels[\'splits\'][split]) for split in (\'calibration\',\'probe\')}\n    save(out/\'verified_matcher_summary.json\',result)\n    summary=json.loads((out/\'run_summary.json\').read_text());summary.update(method=cfg[\'method\'],verified_counts=cfg[\'verified_counts\'],verified_matcher=result)\n    save(out/\'run_summary.json\',summary)\n\n\nrunner.make_loader=make_loader\nrunner.predict=predict\n\n\ndef worker(protocol):\n    cfg=json.loads(Path(protocol).read_text());manifest(cfg)\n    assert cfg[\'arm\']==\'F\' and cfg[\'alignment_weight\']==.1 and cfg[\'alignment_margin\']==.2\n    if digest(cfg[\'aux_stage\'])!=cfg[\'aux_stage_sha256\']:raise ValueError(\'Auxiliary features changed\')\n    runner.worker(protocol)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=[\'prepare\',\'worker\',\'evaluate\',\'archive\']);p.add_argument(\'--config\',required=True)\n    a=p.parse_args()\n    dict(prepare=prepare,worker=worker,evaluate=evaluate,archive=runner.archive)[a.mode](a.config)\n', 'verified_data.py': '"""Explicit same-image contrasts. Missing roles/actions remain unknown."""\nfrom collections import defaultdict, Counter\nimport numpy as np\nfrom probe_data import bounds, identity\n\nSPECS = ((\'hold\',\'obj\',\'holding\'),(\'ride\',\'instr\',\'riding\'),(\'sit\',\'instr\',\'sitting_on\'),\n         (\'carry\',\'obj\',\'carrying\'),(\'eat\',\'obj\',\'eating\'),(\'talk_on_phone\',\'instr\',\'talking_on\'))\nPROMPT_NAMES = dict(hold=\'holding\',ride=\'riding\',sit=\'sit_on\',carry=\'carry\',eat=\'eat\',talk_on_phone=\'talk_on\')\n\n\ndef iou(candidate):\n    s,o=np.asarray(candidate[\'bounds\'][:2])\n    inter=np.maximum(0,np.minimum(s[2:],o[2:])-np.maximum(s[:2],o[:2])).prod()\n    return float(inter/((s[2:]-s[:2]).prod()+(o[2:]-o[:2]).prod()-inter))\n\n\ndef construct(actions, instances, captions, allowed_ids, forbidden_ids, compatibility, decode_action, normalized_box, relation_text):\n    """One actor contrast per positive edge; harder geometric distractor first, never invented labels."""\n    decoded={row[\'action_name\']:decode_action(row) for row in actions}\n    images={r[\'id\']:r for r in instances[\'images\']}\n    anns={r[\'id\']:r for r in instances[\'annotations\']}\n    categories={r[\'id\']:r[\'name\'] for r in instances[\'categories\']}\n    allowed=set(allowed_ids)-set(forbidden_ids)\n    boxes={}\n    def box(aid, iid):\n        if aid not in anns or anns[aid][\'image_id\']!=iid or anns[aid].get(\'iscrowd\',0): return None\n        if aid not in boxes:\n            try: boxes[aid]=normalized_box(anns[aid],images[iid])\n            except ValueError: return None\n        return boxes[aid]\n    positives=[]\n    for action,role,predicate in SPECS:\n        for sid,row in decoded[action].items():\n            iid,oid=row[\'image_id\'],row[\'roles\'].get(role,0)\n            if iid not in allowed or row[\'label\']!=1 or oid==sid or oid not in anns: continue\n            if iid not in captions or captions[iid][\'split\'] not in (\'train\',\'restval\'): raise ValueError(\'Caption heldout image in relation fitting\')\n            s,o=box(sid,iid),box(oid,iid)\n            if s is None or o is None: continue\n            assert categories[anns[sid][\'category_id\']]==\'person\'\n            positives.append((action,predicate,sid,oid,iid,categories[anns[oid][\'category_id\']],s,o))\n    # Compatibility is learned ONLY from fit positives, never calibration/probe.\n    if compatibility is None: compatibility={(obj,action) for action,_,_,_,_,obj,_,_ in positives}\n    by_image_action=defaultdict(list)\n    for action,_,_ in SPECS:\n        for sid,row in decoded[action].items():\n            if row[\'image_id\'] in allowed and row[\'label\']==0:\n                if sid in anns and categories[anns[sid][\'category_id\']]==\'person\': by_image_action[row[\'image_id\'],action].append(sid)\n    verb_candidates=defaultdict(lambda:defaultdict(list)); actor_candidates=[]\n    for action,predicate,sid,oid,iid,obj,s,o in positives:\n        positive=dict(text=relation_text(PROMPT_NAMES[action],obj),bounds=bounds(s,o))\n        common=dict(filename=captions[iid][\'filename\'],coco_id=iid,caption_split=captions[iid][\'split\'],\n                    predicate=predicate,object_label=obj,positive_index=0,\n                    source_positive=dict(subject_id=sid,object_id=oid,action=action,label=1),label_source=\'vcoco_explicit_role_positive_and_actor_zero\')\n        for alternate,_,_ in SPECS:\n            wrong=decoded[alternate].get(sid)\n            if alternate==action or (obj,alternate) not in compatibility or wrong is None or wrong[\'label\']!=0: continue\n            negative=dict(text=relation_text(PROMPT_NAMES[alternate],obj),bounds=positive[\'bounds\'])\n            pair=tuple(sorted((action,alternate)))\n            item=dict(common,case_id=f\'verb_{iid}_{sid}_{oid}_{action}_{alternate}\',task=\'wrong_predicate\',\n                      source_negative=dict(subject_id=sid,object_id=oid,action=alternate,label=0),candidates=[positive,negative])\n            verb_candidates[obj,pair][action].append(item)\n        possible=[]\n        for other in by_image_action[iid,action]:\n            if other==sid: continue\n            other_box=box(other,iid)\n            if other_box is None: continue\n            wrong=dict(text=positive[\'text\'],bounds=bounds(other_box,o))\n            item=dict(common,case_id=f\'actor_{iid}_{sid}_{oid}_{other}_{action}\',task=\'wrong_subject_instance\',\n                      source_negative=dict(subject_id=other,object_id=oid,action=action,label=0),candidates=[positive,wrong])\n            possible.append(item)\n        if possible:\n            # Annotation-confirmed zero first; geometry only selects difficulty, not truth.\n            possible.sort(key=lambda r:(-iou(r[\'candidates\'][1]),identity(r[\'case_id\'])))\n            actor_candidates.append(possible[0])\n    verbs=[]\n    for (obj,pair),directions in sorted(verb_candidates.items()):\n        groups=[sorted(directions[a],key=lambda r:identity(r[\'case_id\'])) for a in pair]\n        count=min(len(g) for g in groups)\n        verbs.extend(row for group in groups for row in group[:count])\n    # Stable orientation prevents a positive-side implementation shortcut.\n    rows=verbs+actor_candidates\n    for row in rows:\n        if int(identity(row[\'case_id\'])[:2],16)%2:\n            row[\'candidates\']=row[\'candidates\'][::-1];row[\'positive_index\']=1\n    assert len({r[\'case_id\'] for r in rows})==len(rows)\n    return sorted(rows,key=lambda r:(r[\'task\'],r[\'case_id\'])),compatibility\n\n\ndef counts(rows):\n    return dict(pairs=len(rows),images=len({r[\'coco_id\'] for r in rows}),tasks=dict(Counter(r[\'task\'] for r in rows)),\n                predicates=dict(Counter(r[\'predicate\'] for r in rows)))\n\n\ndef training_schedule(rows, size=10000):\n    """Equal task exposure; deterministic cycle over every pair, no score-based mining."""\n    groups=[[i for i,r in enumerate(rows) if r[\'task\']==task] for task in (\'wrong_predicate\',\'wrong_subject_instance\')]\n    if any(not group for group in groups): raise ValueError(\'Both verified contrast tasks are required\')\n    return [groups[i%2][(i//2)%len(groups[i%2])] for i in range(size)]\n\n\ndef geometry_summary(rows):\n    """Untrained geometric controls; ties count half, report both pair and image means."""\n    result={}\n    for task in sorted({r[\'task\'] for r in rows}):\n        result[task]={}\n        for method in (\'box_iou\',\'negative_center_distance\'):\n            by_image=defaultdict(list)\n            for row in rows:\n                if row[\'task\']!=task:continue\n                scores=[]\n                for candidate in row[\'candidates\']:\n                    s,o=np.asarray(candidate[\'bounds\'][:2])\n                    scores.append(iou(candidate) if method==\'box_iou\' else -float(np.linalg.norm((s[:2]+s[2:]-o[:2]-o[2:])/2)))\n                margin=scores[row[\'positive_index\']]-scores[1-row[\'positive_index\']]\n                by_image[row[\'coco_id\']].append(.5 if abs(margin)<=1e-7 else float(margin>0))\n            result[task][method]=dict(pair_accuracy=float(np.mean([v for group in by_image.values() for v in group])),\n                image_macro_pair_accuracy=float(np.mean([np.mean(v) for v in by_image.values()])))\n    return result\n', 'test_verified_data.py': '"""Annotation policy tests; run without PyTorch or model downloads."""\nfrom collections import Counter\nimport json\nfrom pathlib import Path\nimport sys\nimport unittest\nHERE=Path(__file__).resolve().parent\nif (HERE.parent/\'semantic_evidence_probe\').exists():sys.path.insert(0,str(HERE.parent/\'semantic_evidence_probe\'))\nfrom verified_data import construct,training_schedule,geometry_summary,SPECS\nfrom probe_data import identity\n\n\ndef fixture():\n    images=[dict(id=i,width=100,height=100) for i in (1,2)]\n    anns=[dict(id=i*10+j,image_id=i,category_id=1 if j<3 else 2,bbox=[j/10,0,j/10+.1,.5]) for i in (1,2) for j in (1,2,3)]\n    instances=dict(images=images,annotations=anns,categories=[dict(id=1,name=\'person\'),dict(id=2,name=\'bicycle\')])\n    actions=[]\n    for action,role,_ in SPECS:\n        rows={}\n        for i,positive in ((1,\'hold\'),(2,\'ride\')):\n            rows[i*10+1]=dict(image_id=i,label=int(action==positive),roles={role:i*10+3} if action==positive else {})\n            rows[i*10+2]=dict(image_id=i,label=0,roles={})\n        actions.append(dict(action_name=action,decoded=rows))\n    captions={i:dict(filename=f\'{i}.jpg\',split=\'train\') for i in (1,2)}\n    return actions,instances,captions\n\n\ndef build(actions,instances,captions,allowed=(1,2),forbidden=(),compatibility=None):\n    return construct(actions,instances,captions,allowed,forbidden,compatibility,\n                     lambda row:row[\'decoded\'],lambda ann,img:ann[\'bbox\'],lambda verb,obj:f\'{verb} {obj}\')\n\n\nclass PolicyTests(unittest.TestCase):\n    def test_explicit_zero_and_direction_balance(self):\n        actions,instances,captions=fixture();rows,_=build(actions,instances,captions)\n        self.assertEqual(Counter(r[\'task\'] for r in rows),dict(wrong_predicate=2,wrong_subject_instance=2))\n        self.assertEqual({r[\'source_positive\'][\'action\'] for r in rows if r[\'task\']==\'wrong_predicate\'},{\'hold\',\'ride\'})\n        for row in rows:\n            self.assertEqual(row[\'source_negative\'][\'label\'],0)\n            self.assertIn(row[\'positive_index\'],(0,1))\n\n    def test_missing_unknown_and_another_positive_are_not_negatives(self):\n        for label in (-1,1):\n            actions,instances,captions=fixture()\n            for action in actions: action[\'decoded\'][12][\'label\']=label\n            rows,_=build(actions,instances,captions)\n            self.assertFalse(any(r[\'source_negative\'][\'subject_id\']==12 for r in rows))\n        actions,instances,captions=fixture()\n        # Unknown alternate action cannot become a predicate negative.\n        next(a for a in actions if a[\'action_name\']==\'ride\')[\'decoded\'][11][\'label\']=-1\n        rows,_=build(actions,instances,captions)\n        self.assertFalse(any(r[\'task\']==\'wrong_predicate\' for r in rows))\n        actions,instances,captions=fixture()\n        next(a for a in actions if a[\'action_name\']==\'hold\')[\'decoded\'][11][\'roles\']={}\n        rows,_=build(actions,instances,captions)\n        self.assertFalse(any(r[\'source_positive\'][\'subject_id\']==11 for r in rows))\n\n    def test_partition_and_fit_only_compatibility(self):\n        a,i,c=fixture();rows,compat=build(a,i,c,allowed=(1,))\n        self.assertEqual(compat,{(\'bicycle\',\'hold\')})\n        heldout,_=build(a,i,c,allowed=(2,),compatibility=compat)\n        self.assertFalse(any(r[\'task\']==\'wrong_predicate\' for r in heldout))\n        forbidden,_=build(a,i,c,forbidden=(1,2));self.assertEqual(forbidden,[])\n        c[1][\'split\']=\'val\'\n        with self.assertRaises(ValueError):build(a,i,c)\n\n    def test_locked_manifest_and_schedule(self):\n        m=json.loads((HERE/\'verified_manifest.json\').read_text(encoding=\'utf-8\'))\n        ids={k:{r[\'coco_id\'] for r in rows} for k,rows in m[\'splits\'].items()}\n        self.assertFalse(ids[\'fit\']&ids[\'calibration\'] or ids[\'fit\']&ids[\'probe\'] or ids[\'calibration\']&ids[\'probe\'])\n        for key,rows in m[\'splits\'].items():self.assertEqual(identity(rows),m[\'split_sha256\'][key])\n        fit=m[\'splits\'][\'fit\'];schedule=training_schedule(fit)\n        self.assertEqual(schedule,m[\'schedule\']);self.assertEqual(set(schedule),set(range(len(fit))))\n        self.assertEqual(Counter(fit[i][\'task\'] for i in schedule),dict(wrong_predicate=5000,wrong_subject_instance=5000))\n        self.assertTrue(all(r[\'caption_split\'] in (\'train\',\'restval\') for rows in m[\'splits\'].values() for r in rows))\n        control=geometry_summary(fit)\n        self.assertEqual(control[\'wrong_predicate\'][\'box_iou\'][\'pair_accuracy\'],.5)\n        self.assertEqual(control[\'wrong_predicate\'][\'negative_center_distance\'][\'pair_accuracy\'],.5)\n        self.assertEqual(geometry_summary(m[\'splits\'][\'probe\'])[\'wrong_object_instance\'][\'box_iou\'][\'pair_accuracy\'],1.)\n\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'test_verified_model.py': '"""Synthetic CPU preflight, mandatory in the manual Kaggle notebook."""\nimport unittest\nimport torch\nfrom pilot_model import PilotModel,pair_loss\nfrom synthetic_fixtures import FakeBase,batch as caption_batch\n\n\ndef batch(device=\'cpu\'):\n    d=caption_batch(device);n=len(d[\'visual\'])\n    d.update(aux_visual=torch.randn(n,197,768,device=device),aux_text=torch.randn(n,2,512,device=device),\n             aux_bounds=d[\'bounds\'][:,:2].clone(),aux_positive=torch.tensor([0,1],device=device),\n             aux_valid=torch.ones(n,dtype=torch.bool,device=device))\n    return d\n\n\nclass VerifiedChecks(unittest.TestCase):\n    def test_positive_orientation_and_empty_loss(self):\n        scores=torch.tensor([[.4,.1],[.5,.6]],requires_grad=True)\n        value,info=pair_loss(scores,torch.tensor([0,1]),torch.ones(2,dtype=torch.bool))\n        torch.testing.assert_close(value,torch.tensor(.05));self.assertEqual(info[\'accuracy\'],1.)\n        flipped,_=pair_loss(scores.flip(1),torch.tensor([1,0]),torch.ones(2,dtype=torch.bool))\n        torch.testing.assert_close(value,flipped)\n        zero,info=pair_loss(scores,torch.tensor([0,1]),torch.zeros(2,dtype=torch.bool))\n        zero.backward();self.assertEqual(info[\'pairs\'],0);torch.testing.assert_close(scores.grad,torch.zeros_like(scores))\n\n    def test_forward_backward_ce_scst_and_replaced_loss(self):\n        for weight in (0.,.05):\n            torch.manual_seed(42);m=PilotModel(FakeBase(),\'F\').eval();d=batch()\n            # The old cross-image pseudo loss must never run.\n            def forbidden(*args):raise AssertionError(\'Old pseudo loss used\')\n            m.extra.alignment=forbidden\n            loss,ce,rl=m(d,d[\'captions\'][:,0,:4],torch.ones(2,3,dtype=torch.bool),torch.tensor([1.,-.3]),weight,0)\n            torch.testing.assert_close(loss.detach(),ce+weight*rl+.1*m.last_alignment[\'loss\'])\n            self.assertEqual(m.last_alignment[\'pairs\'],2);loss.backward()\n            for part in (m.extra.query,m.extra.key,m.extra.value):\n                self.assertGreater(sum(float(p.grad.abs().sum()) for p in part.parameters() if p.grad is not None),0)\n            self.assertTrue(all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None))\n            d[\'aux_valid\'].zero_();m.zero_grad();loss,_,_=m(d,None,None,None,0.,0)\n            self.assertEqual(m.last_alignment[\'pairs\'],0);loss.backward();self.assertTrue(torch.isfinite(loss))\n\n    def test_off_is_original_decoder(self):\n        torch.manual_seed(42);m=PilotModel(FakeBase(),\'F\').eval();d=batch()\n        old,pad=m.base.build_memory(d[\'visual\'],d[\'legacy\'],d[\'legacy_mask\']);mem,mask=m.memory(d,\'off\')\n        t=d[\'captions\'][:,0,:-1]\n        torch.testing.assert_close(m.decoder(t,old,memory_key_padding_mask=pad),m.decoder(t,mem,memory_key_padding_mask=mask))\n\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'test_verified_ddp.py': '"""Three synthetic synchronized updates; not a caption experiment."""\nimport os\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel\nfrom pilot_model import PilotModel\nfrom synthetic_fixtures import FakeBase\nfrom test_verified_model import batch\n\n\ndef main():\n    rank=int(os.environ[\'LOCAL_RANK\']);torch.cuda.set_device(rank);dist.init_process_group(\'nccl\')\n    try:\n        torch.manual_seed(42);model=PilotModel(FakeBase(),\'F\').cuda().eval()\n        ddp=DistributedDataParallel(model,device_ids=[rank],find_unused_parameters=True,broadcast_buffers=False)\n        optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4)\n        for i,weight in enumerate((0.,.05,0.)):\n            torch.manual_seed(90+rank);data=batch(f\'cuda:{rank}\')\n            if i==2 and rank==1:data[\'valid\'].zero_();data[\'aux_valid\'].zero_()\n            optimizer.zero_grad(set_to_none=True)\n            loss,_,_=ddp(data,data[\'captions\'][:,0,:4],torch.ones(2,3,dtype=torch.bool,device=f\'cuda:{rank}\'),\n                        torch.tensor([1.,-.3],device=f\'cuda:{rank}\'),weight,0)\n            loss.backward();optimizer.step()\n            vector=torch.cat([p.detach().flatten() for p in model.parameters()]);other=vector.clone();dist.broadcast(other,0)\n            torch.testing.assert_close(vector,other,rtol=0,atol=0)\n        if rank==0:print(\'F: two-GPU synchronized CE/SCST/verified-pair synthetic smoke PASS\',flush=True)\n    finally:dist.destroy_process_group()\n\n\nif __name__==\'__main__\':main()\n', 'pilot_data.py': '"""Deterministic image cohorts and box pooling; no ML/runtime side effects."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nSEED = 42\nSLOTS = 9\nINPUT_DIM = 768 * 3 + 8\nTRAIN_COUNT = 10000\nVAL_COUNT = 1000\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef identity(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()).hexdigest()\n\n\ndef save(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding=\'utf-8\')\n\n\ndef cohorts(dataset, train_count=TRAIN_COUNT, val_count=VAL_COUNT):\n    result = {}\n    for split, count in [(\'train\', train_count), (\'val\', val_count)]:\n        rows = []\n        for i, row in enumerate(dataset[\'images\']):\n            selected = row[\'split\'] in (\'train\', \'restval\') if split == \'train\' else row[\'split\'] == \'val\'\n            if selected:\n                rows.append({\'image_id\': i, \'coco_id\': int(Path(row[\'filename\']).stem.rsplit(\'_\', 1)[1]),\n                             \'filename\': row[\'filename\'], \'split\': split})\n        if not 1 < count <= len(rows):\n            raise ValueError(f\'Insufficient {split} images.\')\n        rows.sort(key=lambda r: hashlib.sha256((\'upt-alignment-pilot-v1:\' + split + \':\' + r[\'filename\']).encode()).hexdigest())\n        result[split] = sorted(rows[:count], key=lambda r: r[\'image_id\'])\n    all_rows = result[\'train\'] + result[\'val\']\n    if len({r[\'coco_id\'] for r in all_rows}) != len(all_rows):\n        raise ValueError(\'Overlapping/duplicate COCO IDs.\')\n    return result\n\n\ndef box(value):\n    b = np.asarray(value, dtype=np.float32)\n    if b.shape != (4,) or not np.isfinite(b).all() or (b < 0).any() or (b > 1).any() or (b[2:] <= b[:2]).any():\n        raise ValueError(\'Expected a positive-area normalized xyxy box.\')\n    return b\n\n\ndef pool(patches, bounds):\n    if patches.shape != (196, 768):\n        raise ValueError(\'Expected 14x14 spatial CLIP patch tokens, excluding CLS.\')\n    b = box(bounds)\n    y, x = np.meshgrid((np.arange(14) + .5) / 14, (np.arange(14) + .5) / 14, indexing=\'ij\')\n    x, y = x.ravel(), y.ravel()\n    mask = (x >= b[0]) & (x <= b[2]) & (y >= b[1]) & (y <= b[3])\n    if mask.any():\n        return patches[mask].mean(0)\n    nearest = np.argmin((x - (b[0] + b[2]) / 2)**2 + (y - (b[1] + b[3]) / 2)**2)\n    return patches[nearest].copy()\n\n\ndef geometry(s, o):\n    sw, sh = np.maximum(s[2:] - s[:2], 1e-6)\n    ow, oh = np.maximum(o[2:] - o[:2], 1e-6)\n    sc, oc = (s[:2] + s[2:]) / 2, (o[:2] + o[2:]) / 2\n    inter = np.maximum(np.minimum(s[2:], o[2:]) - np.maximum(s[:2], o[:2]), 0).prod()\n    return np.asarray([(oc[0]-sc[0])/sw, (oc[1]-sc[1])/sh, np.log(ow/sw), np.log(oh/sh),\n                       inter/max(sw*sh+ow*oh-inter, 1e-6), inter/(sw*sh), inter/(ow*oh),\n                       np.linalg.norm(oc-sc)], dtype=np.float32)\n\n\ndef region_slots(entry, visual):\n    """Keep every selected triplet, with separate S/O/R slots; object fallback otherwise."""\n    if visual.shape != (197, 768) or not np.isfinite(visual).all():\n        raise ValueError(\'Invalid visual cache entry.\')\n    features = np.zeros((SLOTS, INPUT_DIM), dtype=np.float32)\n    roles = np.ones(SLOTS, dtype=np.int64)\n    valid = np.zeros(SLOTS, dtype=np.bool_)\n    texts = [\'\'] * SLOTS\n    objects = {o[\'index\']: o for o in entry[\'objects\']}\n    relations = entry[\'relations\']\n    if len(relations) > 3:\n        raise ValueError(\'Expected at most 3 selected relations.\')\n    if relations:\n        for i, relation in enumerate(relations):\n            if relation[\'score\'] < relation[\'threshold\']:\n                raise ValueError(\'A selected relation is below its threshold.\')\n            subject, obj = (objects[relation[key + \'_index\']] for key in (\'subject\', \'object\'))\n            if subject[\'instance_id\'] != relation[\'subject_instance\'] or obj[\'instance_id\'] != relation[\'object_instance\']:\n                raise ValueError(\'Instance-link mismatch.\')\n            s, o = box(subject[\'bbox\']), box(obj[\'bbox\'])\n            union = np.r_[np.minimum(s[:2], o[:2]), np.maximum(s[2:], o[2:])]\n            vs, vo, vu = [pool(visual[1:], b) for b in (s, o, union)]\n            n = i * 3\n            features[n, :768], features[n+1, :768] = vs, vo\n            features[n+2] = np.r_[vs, vo, vu, geometry(s, o)]\n            roles[n:n+3], valid[n:n+3] = (0, 1, 2), True\n            texts[n:n+3] = (subject[\'label\'], obj[\'label\'], relation[\'text\'])\n    else:\n        accepted = sorted((o for o in objects.values() if o[\'confidence\'] >= .5),\n                          key=lambda o: (-o[\'confidence\'], o[\'index\']))[:8]\n        for i, obj in enumerate(accepted):\n            features[i, :768] = pool(visual[1:], obj[\'bbox\'])\n            valid[i], texts[i] = True, obj[\'label\']\n    if bool(valid.any()) != (entry[\'mode\'] != \'no_objects\'):\n        raise ValueError(\'Prompt mode and region slots disagree.\')\n    return features, roles, valid, texts\n\n\ndef evidence_bounds(entry):\n    """Normalized subject/object/allowed boxes, in exactly region_slots order."""\n    result = np.zeros((SLOTS, 3, 4), dtype=np.float32)\n    objects = {o[\'index\']:o for o in entry[\'objects\']}\n    if entry[\'relations\']:\n        for i,r in enumerate(entry[\'relations\']):\n            s,o = (box(objects[r[k+\'_index\']][\'bbox\']) for k in (\'subject\',\'object\'))\n            union=np.r_[np.minimum(s[:2],o[:2]),np.maximum(s[2:],o[2:])]\n            result[i*3]=np.stack((s,o,s));result[i*3+1]=np.stack((s,o,o))\n            result[i*3+2]=np.stack((s,o,union))\n    else:\n        accepted=sorted((o for o in objects.values() if o[\'confidence\']>=.5),key=lambda o:(-o[\'confidence\'],o[\'index\']))[:8]\n        for i,o in enumerate(accepted): result[i]=np.tile(box(o[\'bbox\']),(3,1))\n    return result\n\n\ndef phrase_identity(text):\n    return int(hashlib.sha256(text.strip().casefold().encode()).hexdigest()[:15],16) if text else 0\n', 'test_pilot_data.py': "import unittest\nimport numpy as np\nfrom pilot_data import box, pool, geometry, region_slots, cohorts, identity, evidence_bounds, phrase_identity\n\n\nclass DataChecks(unittest.TestCase):\n    def test_spatial_pool_and_tiny_box(self):\n        patches = np.repeat(np.arange(196, dtype=np.float32)[:, None], 768, axis=1)\n        np.testing.assert_allclose(pool(patches, [0, 0, 1, 1]), 97.5)\n        np.testing.assert_allclose(pool(patches, [0, 0, .001, .001]), 0)\n        np.testing.assert_allclose(geometry(box([0, 0, 1, 1]), box([0, 0, 1, 1])), [0,0,0,0,1,1,1,0])\n        for b in ([0,0,0,1], [-1,0,1,1], [0,0,float('nan'),1]):\n            with self.assertRaises(ValueError): box(b)\n\n    def test_instance_link_fallback_and_no_objects(self):\n        objects = [dict(index=0, instance_id='p0', label='person', confidence=.9, bbox=[0,0,.4,1]),\n                   dict(index=1, instance_id='o1', label='horse', confidence=.8, bbox=[.4,0,1,1])]\n        relation = dict(subject_index=0, object_index=1, subject_instance='p0', object_instance='o1',\n                        score=.7, threshold=.5, text='person riding horse')\n        entry = dict(objects=objects, relations=[relation], mode='object_relation')\n        visual = np.ones((197, 768), dtype=np.float32)\n        f, roles, valid, texts = region_slots(entry, visual)\n        self.assertEqual(valid.sum(), 3)\n        self.assertEqual(texts[:3], ['person','horse','person riding horse'])\n        bounds=evidence_bounds(entry)\n        np.testing.assert_array_equal(bounds[0,2],bounds[0,0])\n        np.testing.assert_array_equal(bounds[1,2],bounds[1,1])\n        np.testing.assert_array_equal(bounds[2,2],[0,0,1,1])\n        self.assertEqual(phrase_identity('Person'),phrase_identity(' person '))\n        self.assertNotEqual(phrase_identity('horse'),phrase_identity('person'))\n        np.testing.assert_array_equal(roles[:3], [0,1,2])\n        np.testing.assert_array_equal(f[2,:2304], 1)\n        entry['relations'] = []; entry['mode'] = 'object_only'\n        self.assertEqual(region_slots(entry, visual)[2].sum(), 2)\n        entry['objects'] = []; entry['mode'] = 'no_objects'\n        self.assertFalse(region_slots(entry, visual)[2].any())\n        entry.update(objects=objects, relations=[dict(relation, subject_instance='wrong')], mode='object_relation')\n        with self.assertRaises(ValueError): region_slots(entry, visual)\n\n    def test_locked_split_is_deterministic_and_excludes_test(self):\n        data = {'images': [dict(filename=f'COCO_train2014_{i:012d}.jpg', split=split)\n                           for i, split in enumerate(['train']*6+['restval']*2+['val']*5+['test']*3)]}\n        a = cohorts(data, 4, 3); b = cohorts(data, 4, 3)\n        self.assertEqual(identity(a), identity(b))\n        self.assertTrue(all(x['image_id'] < 8 for x in a['train']))\n        self.assertTrue(all(8 <= x['image_id'] < 13 for x in a['val']))\n        self.assertFalse({r['coco_id'] for r in a['train']} & {r['coco_id'] for r in a['val']})\n\n\nif __name__ == '__main__': unittest.main(verbosity=2)\n", 'pilot_metric_smoke.py': '"""Fail on metric/tokenizer dependencies before staging or training."""\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\nimport numpy as np\nfrom captioning.metrics import score_files\nfrom captioning.scst import TrainingCider\nfrom captioning.object_reward import _AnnotationReader\n\n\ndef main(chair):\n    references = {1: [\'a person holding an umbrella\']*5, 2: [\'a dog running through grass\']*5}\n    reward = TrainingCider(references, reference_cache_size=2).score([1,2],[r[0] for r in references.values()])\n    assert np.isfinite(reward).all() and (reward > 0).all()\n    with tempfile.TemporaryDirectory() as folder:\n        p = Path(folder)\n        gt = dict(info={},licenses=[],type=\'captions\',images=[{\'id\':i} for i in references],\n                  annotations=[dict(id=i*5+j,image_id=i,caption=text) for i,refs in references.items() for j,text in enumerate(refs)])\n        (p/\'gt.json\').write_text(json.dumps(gt))\n        (p/\'pred.json\').write_text(json.dumps([dict(image_id=i,caption=refs[0]) for i,refs in references.items()]))\n        result = score_files(p/\'pred.json\',p/\'gt.json\',p/\'metrics.json\')\n        assert np.isfinite(result[\'CIDEr\'])\n        with (Path(chair)/\'chair.pkl\').open(\'rb\') as stream: annotation = _AnnotationReader(stream).load()\n        ids = sorted(annotation.imid_to_objects)[:2]\n        (p/\'chair_input.json\').write_text(json.dumps([dict(image_id=i,caption=\'a person\') for i in ids]))\n        subprocess.run([sys.executable,\'-u\',str(Path(chair)/\'chair.py\'),\'--cache\',str(Path(chair)/\'chair.pkl\'),\n                        \'--cap_file\',str(p/\'chair_input.json\'),\'--save_path\',str(p/\'chair_output.json\')],cwd=chair,check=True)\n        assert \'overall_metrics\' in json.loads((p/\'chair_output.json\').read_text())\n    print(\'CIDEr reward, official caption metrics and CHAIR smoke PASS\',flush=True)\n\n\nif __name__ == \'__main__\': main(sys.argv[1])\n', 'evidence_core.py': '"""Text queries image values; no text residual in visual evidence or matching score."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass EvidenceMemory(nn.Module):\n    def __init__(self, dim=512):\n        super().__init__()\n        self.query = nn.Linear(512, dim)\n        self.role = nn.Embedding(3, dim)\n        self.key = nn.Linear(768, dim)\n        self.value = nn.Linear(768, dim)\n        self.position = nn.Linear(4, dim, bias=False)\n        self.norm = nn.LayerNorm(dim)\n        axis = (torch.arange(14, dtype=torch.float32) + .5) / 14\n        y, x = torch.meshgrid(axis, axis, indexing=\'ij\')\n        self.register_buffer(\'xy\', torch.stack((x.flatten(), y.flatten()), -1))\n\n    def attend(self, patches, text, roles, bounds):\n        # bounds: subject, object, allowed box; values contain image features only.\n        xy = self.xy[None].expand(len(text), -1, -1)\n        inside = (xy[:, None] >= bounds[:, :, None, :2]).all(-1) & (xy[:, None] <= bounds[:, :, None, 2:]).all(-1)\n        allowed = inside[:, 2].clone()\n        empty = ~allowed.any(-1)\n        center = (bounds[:, 2, :2] + bounds[:, 2, 2:]) / 2\n        nearest = ((xy-center[:, None])**2).sum(-1).argmin(-1)\n        allowed[torch.arange(len(text), device=text.device)[empty], nearest[empty]] = True\n        spatial = torch.cat((xy, inside[:, :2].transpose(1, 2).float()), -1)\n        q = self.query(text) + self.role(roles)\n        keys = self.key(patches) + self.position(spatial)\n        weights = ((keys*q[:, None]).sum(-1) / q.shape[-1]**.5).masked_fill(~allowed, -torch.inf).softmax(-1)\n        evidence = self.norm((weights[:, :, None]*self.value(patches)).sum(1))\n        score = (F.normalize(q, dim=-1)*F.normalize(evidence, dim=-1)).sum(-1)\n        return evidence, score, weights\n\n    def forward(self, patches, text, roles, valid, bounds):\n        shape = valid.shape\n        tokens, scores, attention = self.attend(patches.flatten(0,1), text.flatten(0,1), roles.flatten(), bounds.flatten(0,1))\n        return tokens.reshape(*shape, -1).masked_fill(~valid[..., None], 0), scores.reshape(shape), attention.reshape(*shape,196)\n\n    def alignment(self, patches, text, roles, valid, bounds, phrase_ids, image_ids, positive_scores):\n        """Local-rank pseudo-pair hinge loss; uncertain negatives are a stated limit."""\n        n, slots = valid.shape\n        words, role, ok = phrase_ids.flatten(), roles.flatten(), valid.flatten()\n        owner = torch.arange(n, device=text.device).repeat_interleave(slots)\n        # Exclude every exact phrase known anywhere in the target image, not just its positive slot.\n        known = ((words[None, :, None] == phrase_ids[:, None, :]) & valid[:, None, :]).any(-1)\n        eligible = ok[:, None] & ok[None, :] & (role[:, None] == role[None, :])\n        eligible &= image_ids[owner, None] != image_ids[owner][None, :]\n        eligible &= ~known[owner] & ~known[owner].T\n        has = eligible.any(-1)\n        donor = eligible.long().argmax(-1)\n        index = torch.nonzero(has, as_tuple=False).flatten()\n        if not len(index):\n            return positive_scores.sum()*0, dict(pairs=0, accuracy=None, margin=None)\n        donor = donor[index]\n        p, t, r, b = patches.flatten(0,1), text.flatten(0,1), roles.flatten(), bounds.flatten(0,1)\n        _, wrong_text, _ = self.attend(p[index], t[donor], r[index], b[index])\n        _, wrong_region, _ = self.attend(p[donor], t[index], r[index], b[donor])\n        positive = positive_scores.flatten()[index]\n        losses = (F.relu(.2-positive+wrong_text)+F.relu(.2-positive+wrong_region)) / 2\n        margin = torch.minimum(positive-wrong_text, positive-wrong_region)\n        return losses.mean(), dict(pairs=len(index), accuracy=float((margin>0).float().mean().detach()),\n                                   margin=float(margin.mean().detach()))\n\n\nclass PilotModel(nn.Module):\n    def __init__(self, base, arm):\n        super().__init__()\n        if arm not in (\'A\',\'E\'): raise ValueError(arm)\n        self.base, self.arm = base, arm\n        self.extra = EvidenceMemory() if arm==\'E\' else None\n        self.last_gates = None\n        self.last_alignment = dict(loss=0., pairs=0, accuracy=None, margin=None)\n\n    @property\n    def decoder(self): return self.base.decoder\n\n    def memory(self, batch, intervention=\'on\'):\n        memory, padding = self.base.build_memory(batch[\'visual\'], batch[\'legacy\'], batch[\'legacy_mask\'])\n        if self.extra is None: return memory, padding\n        shape = batch[\'valid\'].shape\n        patches = batch[\'visual\'][:, None, 1:].expand(-1, shape[1], -1, -1)\n        text, bounds = batch[\'text\'], batch[\'bounds\']\n        if intervention==\'text_swap\': text=batch[\'swap_text\']\n        elif intervention==\'region_swap\':\n            joint = batch[\'swap_valid\']\n            patches = torch.where(joint[...,None,None], batch[\'swap_visual\'][:,None,1:].expand_as(patches), patches)\n            bounds = torch.where(joint[...,None,None], batch[\'swap_bounds\'], bounds)\n        elif intervention not in (\'on\',\'off\'): raise ValueError(intervention)\n        tokens, scores, attention = self.extra(patches, text, batch[\'roles\'], batch[\'valid\'], bounds)\n        self.last_gates = torch.ones_like(scores)\n        self.last_attention = attention.detach()\n        self._evidence_inputs = (patches, text, batch[\'roles\'], batch[\'valid\'], bounds, scores)\n        extra_padding = ~batch[\'valid\']\n        if intervention==\'off\':\n            tokens=torch.zeros_like(tokens);extra_padding=torch.ones_like(extra_padding)\n        return torch.cat((memory,tokens),1), torch.cat((padding,extra_padding),1)\n\n    def teacher(self, memory, padding, targets, masks=None):\n        b, slots, dim = memory.shape\n        repeats = targets.size(0)//b\n        if repeats*b != targets.size(0): raise ValueError(\'Caption/image batch mismatch\')\n        m=memory[:,None].expand(b,repeats,slots,dim).reshape(-1,slots,dim)\n        p=padding[:,None].expand(b,repeats,slots).reshape(-1,slots)\n        return self.decoder(targets,m,tgt_key_padding_mask=masks,memory_key_padding_mask=p)\n\n    def forward(self, batch, sample, active, advantage, scst_weight, pad_idx):\n        from captioning.scst import policy_loss\n        memory,padding=self.memory(batch)\n        targets=batch[\'captions\'].flatten(0,1); masks=batch[\'caption_mask\'].flatten(0,1)\n        logits=self.teacher(memory,padding,targets[:,:-1],masks[:,:-1])\n        ce=F.cross_entropy(logits.flatten(0,1),targets[:,1:].reshape(-1),ignore_index=pad_idx)\n        rl=ce.new_zeros(())\n        if scst_weight:\n            rl=policy_loss(self.teacher(memory,padding,sample[:,:-1]),sample[:,1:],active,advantage)\n        alignment=ce.new_zeros(())\n        info=dict(pairs=0,accuracy=None,margin=None)\n        if self.extra is not None:\n            p,t,r,v,b,s=self._evidence_inputs\n            alignment,info=self.extra.alignment(p,t,r,v,b,batch[\'phrase_ids\'],batch[\'image_id\'],s)\n        self.last_alignment=dict(loss=float(alignment.detach()),**info)\n        return ce+scst_weight*rl+.1*alignment, ce.detach(), rl.detach()\n', 'base_runner.py': '"""Manual, one-arm, two-GPU UPT caption pilot. Importing this file never trains."""\nimport argparse\nfrom collections import Counter\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport zipfile\n\nimport numpy as np\n\nfrom pilot_data import cohorts, digest, identity, region_slots, evidence_bounds, phrase_identity, save, SLOTS, INPUT_DIM\n\nCLIP = \'openai/clip-vit-base-patch16\'\nCLIP_REVISION = \'57c216476eefef5ab752ec549e440a49ae4ae5f3\'\nCHECKPOINT_SHA = \'5f8eead3f4e960189345b1a9bf492763fe52ec71f70115b73244ffe9d03d2e75\'\n\n\ndef prepare(config_path):\n    import h5py\n    import torch\n    from transformers import CLIPTextModel, AutoTokenizer\n    from captioning.visual_cache import VisualCache\n    from captioning.semantic_prompts import align_prompt_cache\n    from captioning.object_reward import make_targets, _AnnotationReader\n    from captioning.scst_runtime import configure_strict_runtime\n    cfg = json.loads(Path(config_path).read_text())\n    out = Path(cfg[\'output\'])\n    if out.exists():\n        raise FileExistsError(f\'Use a new OUTPUT: {out}\')\n    for key in (\'checkpoint\', \'legacy_prompt\', \'upt_prompt\', \'dataset_json\', \'detections\'):\n        if not Path(cfg[key]).is_file():\n            raise FileNotFoundError(f\'{key}: {cfg[key]}\')\n    hashes = {}\n    for key in (\'checkpoint\', \'legacy_prompt\', \'upt_prompt\', \'dataset_json\', \'detections\'):\n        print(f\'Checking pinned input: {key}\', flush=True)\n        hashes[key] = digest(cfg[key])\n    if hashes != cfg[\'expected_hashes\']:\n        raise ValueError(f\'Pinned inputs differ: {hashes}. Do not silently replace the baseline/cache.\')\n    dataset = json.loads(Path(cfg[\'dataset_json\']).read_text())\n    groups = cohorts(dataset)\n    all_rows = groups[\'train\'] + groups[\'val\']\n    prompts = json.loads(Path(cfg[\'upt_prompt\']).read_text())\n    if len(prompts) != 123287 or any(r[\'filename\'] not in prompts for r in all_rows):\n        raise ValueError(\'Incomplete UPT prompt mapping.\')\n    for rows in groups.values():\n        for row in rows:\n            row[\'mode\'] = prompts[row[\'filename\']][\'mode\']\n    if identity(groups) != \'6bbdf3b75869927334ec6c2f868edf9f52346c0e79401aa0e118ba5a02fb711b\':\n        raise ValueError(\'Cohort differs from the locally reviewed 10000/1000 image manifest.\')\n    chair = Path(cfg[\'chair\'])\n    if digest(chair / \'chair.pkl\') != \'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3\':\n        raise ValueError(\'Pinned CHAIR annotation cache differs.\')\n    # Read validation annotation coverage for evaluation only, not training targets.\n    with (chair / \'chair.pkl\').open(\'rb\') as stream:\n        annotation = _AnnotationReader(stream).load()\n    if any(r[\'coco_id\'] not in annotation.imid_to_objects for r in groups[\'val\']):\n        raise ValueError(\'CHAIR validation coverage incomplete.\')\n    del annotation\n    out.mkdir(parents=True)\n    cfg.update(groups=groups, cohort_sha256=identity(groups), test_used=False,\n               alignment_weight=.1 if cfg[\'arm\']==\'E\' else 0., alignment_margin=.2,\n               negative_policy=\'Within-rank same-role, other image, exclude exact known phrases in either image; deterministic first eligible; may contain false negatives\',\n               validation_scope=\'Historically reused development validation, not an independent unseen test\',\n               source_checkpoint_sha256=hashes[\'checkpoint\'], supported_actions=116,\n               loss=\'CE + scheduled 0.05*SCST(CIDEr-D+0.5*object_count_v2) + E-only 0.1*pseudo_pair_hinge_margin0.2\',\n               evaluation_policy=\'Raw greedy, all arms and start checkpoint; no reranking in this screening pilot\',\n               comparison_limits=\'A continued-training control; E text-query patch attention plus alignment loss. Whole-method comparison only; UPT pseudo-pairs and cross-image negatives can be noisy.\',\n               module_claim=\'Text-conditioned visual evidence with pseudo-pair supervision; not calibrated relation truth or established novelty\')\n    save(out / \'groups.json\', groups)\n    targets = make_targets(groups, dataset, chair / \'chair.pkl\', json.loads(Path(cfg[\'detections\']).read_text()),\n                           \'aliases_v2\', \'unique_count_v2\')\n    save(out / \'train_object_targets.json\', targets)\n    cfg[\'object_targets_sha256\'] = digest(out / \'train_object_targets.json\')\n    bundle = torch.load(cfg[\'legacy_prompt\'], map_location=\'cpu\', weights_only=False)\n    names = [str(Path(\'/images\') / dataset[\'images\'][r[\'image_id\']][\'filepath\'] / r[\'filename\']) for r in all_rows]\n    legacy = align_prompt_cache(bundle[\'data\'], names)\n    if len(legacy) != len(all_rows):\n        raise ValueError(\'Missing baseline prompt entries; UPT JSON must not replace this cache.\')\n    first = legacy[names[0]]\n    length = first[\'tokens\'].shape[0]\n    if first[\'tokens\'].shape != (length, 512):\n        raise ValueError(\'Legacy prompt shape differs.\')\n    stage = Path(cfg[\'stage\'])\n    if stage.exists():\n        raise FileExistsError(f\'Use a fresh stage path: {stage}\')\n    stage.parent.mkdir(parents=True, exist_ok=True)\n    required = len(all_rows) * (197 * 768 + SLOTS * INPUT_DIM + length * 512 + SLOTS * 512) * 4\n    if shutil.disk_usage(stage.parent).free < required + 2 * 1024**3:\n        raise RuntimeError(\'Not enough disk space to stage the 11000-image subset without quantizing.\')\n    cache = VisualCache([cfg[\'visual_cache\']], id_key=\'coco_id\')\n    cache.require_ids([r[\'coco_id\'] for r in all_rows], \'pilot train/val\')\n    text_rows = []\n    n = len(all_rows)\n    try:\n        with h5py.File(stage, \'w\') as f:\n            arrays = {\'visual\': ((n, 197, 768), \'f4\'), \'legacy\': ((n, length, 512), \'f4\'),\n                      \'legacy_mask\': ((n, length), \'i8\'), \'regions\': ((n, SLOTS, INPUT_DIM), \'f4\'),\n                      \'roles\': ((n, SLOTS), \'i8\'), \'valid\': ((n, SLOTS), \'?\'),\n                      \'text\': ((n, SLOTS, 512), \'f4\'), \'bounds\': ((n,SLOTS,3,4),\'f4\'), \'phrase_ids\': ((n,SLOTS),\'i8\')}\n            for key, (shape, dtype) in arrays.items():\n                f.create_dataset(key, shape=shape, dtype=dtype, chunks=(1, *shape[1:]))\n            # Read source HDF5 in source-row order rather than random image order.\n            order = sorted(range(n), key=lambda i: cache.index[all_rows[i][\'coco_id\']])\n            text_rows = [None] * n\n            for count, i in enumerate(order, 1):\n                row = all_rows[i]\n                visual = cache.read(row[\'coco_id\'])\n                features, roles, valid, texts = region_slots(prompts[row[\'filename\']], visual)\n                old = legacy[names[i]]\n                if old[\'tokens\'].shape != (length, 512) or old[\'mask\'].shape != (length,):\n                    raise ValueError(\'Inconsistent legacy prompt length.\')\n                for key, value in [(\'visual\', visual), (\'legacy\', old[\'tokens\'].float().numpy()),\n                                   (\'legacy_mask\', old[\'mask\'].long().numpy()), (\'regions\', features),\n                                   (\'roles\', roles), (\'valid\', valid), (\'bounds\',evidence_bounds(prompts[row[\'filename\']])),\n                                   (\'phrase_ids\',np.asarray([phrase_identity(t) for t in texts],dtype=np.int64))]:\n                    f[key][i] = value\n                text_rows[i] = texts\n                if count % 250 == 0 or count == n:\n                    print(f\'Staged spatial tokens/regions: {count}/{n}\', flush=True)\n    finally:\n        cache.close()\n    del bundle, legacy, prompts, targets\n    # Frozen EOS hidden states (512D), shared across repeated phrases; no trainable text encoder.\n    torch.manual_seed(42)\n    configure_strict_runtime(True)\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError(\'Select GPU T4 x2. This notebook runs ONE arm using two GPUs.\')\n    encoder = CLIPTextModel.from_pretrained(CLIP, revision=CLIP_REVISION).cuda().eval()\n    tokenizer = AutoTokenizer.from_pretrained(CLIP, revision=CLIP_REVISION)\n    unique = sorted({t for row in text_rows for t in row if t})\n    embedding = {}\n    with torch.no_grad():\n        for start in range(0, len(unique), 128):\n            batch = unique[start:start+128]\n            tokens = tokenizer(batch, padding=True, truncation=True, max_length=77, return_tensors=\'pt\').to(\'cuda\')\n            vectors = encoder(**tokens).pooler_output.float().cpu().numpy()\n            for text, vector in zip(batch, vectors):\n                embedding[text] = vector\n            if start % 1024 == 0:\n                print(f\'Frozen text embeddings: {min(start+128, len(unique))}/{len(unique)}\', flush=True)\n    with h5py.File(stage, \'r+\') as f:\n        for i, texts in enumerate(text_rows):\n            values = np.zeros((SLOTS, 512), dtype=np.float32)\n            for j, text in enumerate(texts):\n                if text:\n                    values[j] = embedding[text]\n            f[\'text\'][i] = values\n        f.attrs[\'cohort_sha256\'] = cfg[\'cohort_sha256\']\n        f.attrs[\'complete\'] = True\n    cfg[\'stage_sha256\'] = digest(stage)\n    cfg[\'counts_by_mode\'] = {key: dict(Counter(r[\'mode\'] for r in rows)) for key, rows in groups.items()}\n    save(out / \'slot_texts.json\', [{\'filename\': r[\'filename\'], \'texts\': texts} for r, texts in zip(all_rows, text_rows)])\n    save(out / \'protocol.json\', cfg)\n    print(\'PREPARATION PASS:\', json.dumps(cfg[\'counts_by_mode\']), \'cohort:\', cfg[\'cohort_sha256\'], flush=True)\n\n\ndef seed_all(seed):\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\ndef build_model(cfg, tokenizer):\n    from captioning.models import UniversalVisionEncoder, CaptionDecoder, ImageCaptioningModel\n    from captioning.checkpoints import load_checkpoint\n    from pilot_model import PilotModel\n    base = ImageCaptioningModel(UniversalVisionEncoder(load_backbone=False, visual_adapter=\'qformer\',\n                               num_visual_queries=32, qformer_layers=2), CaptionDecoder(len(tokenizer.word2idx)))\n    load_checkpoint(base, cfg[\'checkpoint\'], tokenizer)\n    return PilotModel(base, cfg[\'arm\']).cuda().eval()\n\n\ndef corpus(cfg):\n    from captioning.tokenizer import CaptionTokenizer\n    dataset = json.loads(Path(cfg[\'dataset_json\']).read_text())\n    train = [r for r in dataset[\'images\'] if r[\'split\'] in (\'train\', \'restval\')]\n    tokenizer = CaptionTokenizer([s[\'raw\'] for r in train for s in r[\'sentences\'][:5]])\n    return dataset, tokenizer\n\n\ndef make_loader(cfg, dataset, tokenizer, split, rank, world):\n    import h5py\n    import torch\n    from torch.utils.data import Dataset, DataLoader, DistributedSampler\n    rows = cfg[\'groups\'][split]\n    train_count = len(cfg[\'groups\'][\'train\'])\n\n    class Staged(Dataset):\n        def __init__(self):\n            self.f = h5py.File(cfg[\'stage\'], \'r\')\n            if not self.f.attrs.get(\'complete\') or self.f.attrs[\'cohort_sha256\'] != cfg[\'cohort_sha256\']:\n                raise ValueError(\'Incomplete/different staged cohort.\')\n\n        def __len__(self):\n            return len(rows)\n\n        def __getitem__(self, index):\n            row = rows[index]\n            i = index if split == \'train\' else index + train_count\n            result = {key: torch.from_numpy(self.f[key][i]) for key in\n                      (\'visual\', \'legacy\', \'legacy_mask\', \'roles\', \'valid\', \'text\', \'bounds\', \'phrase_ids\')}\n            refs = dataset[\'images\'][row[\'image_id\']][\'sentences\'][:5]\n            encoded = [tokenizer.encode(s[\'raw\']) for s in refs]\n            result[\'captions\'] = torch.stack([x[0] for x in encoded])\n            result[\'caption_mask\'] = torch.stack([x[1] for x in encoded])\n            result[\'image_id\'] = torch.tensor(row[\'image_id\'])\n            # Same-mode deterministic donor. Missing slots are not filled with fake evidence.\n            candidates = donors[row[\'mode\']]\n            pos = candidates.index(index)\n            donor = candidates[(pos + 1) % len(candidates)]\n            di = donor if split == \'train\' else donor + train_count\n            donor_valid = self.f[\'valid\'][di]\n            result[\'swap_visual\']=torch.from_numpy(self.f[\'visual\'][di])\n            result[\'swap_bounds\']=torch.from_numpy(self.f[\'bounds\'][di])\n            joint = self.f[\'valid\'][i] & donor_valid\n            result[\'swap_text\'] = torch.from_numpy(np.where(joint[:, None], self.f[\'text\'][di], self.f[\'text\'][i]))\n            result[\'swap_valid\']=torch.from_numpy(joint)\n            return result\n\n    donors = {mode: [i for i, r in enumerate(rows) if r[\'mode\'] == mode] for mode in set(r[\'mode\'] for r in rows)}\n    staged = Staged()\n    if split == \'train\':\n        sampler = DistributedSampler(staged, num_replicas=world, rank=rank, shuffle=True, seed=42, drop_last=False)\n        return DataLoader(staged, batch_size=cfg[\'global_batch_size\']//world, sampler=sampler,\n                          num_workers=0, pin_memory=True, drop_last=False)\n    # Explicit rank slice: no padding or duplicate validation IDs from DistributedSampler.\n    subset = torch.utils.data.Subset(staged, list(range(rank, len(rows), world)))\n    return DataLoader(subset, batch_size=cfg[\'eval_batch_size\'], shuffle=False, num_workers=0, pin_memory=True)\n\n\ndef cuda_batch(batch):\n    return {key: value.cuda(non_blocking=True) for key, value in batch.items()}\n\n\ndef ground_truth(cfg, dataset, out):\n    rows = cfg[\'groups\'][\'val\']\n    gt = {\'info\': {}, \'licenses\': [], \'type\': \'captions\', \'images\': [{\'id\': r[\'image_id\']} for r in rows],\n          \'annotations\': [{\'id\': i*5+j, \'image_id\': r[\'image_id\'], \'caption\': ref[\'raw\']}\n                          for i, r in enumerate(rows) for j, ref in enumerate(dataset[\'images\'][r[\'image_id\']][\'sentences\'][:5])]}\n    save(out / \'val_gt.json\', gt)\n\n\ndef predict(model, loader, tokenizer, intervention):\n    import torch\n    from captioning.scst import rollout, decode\n    records, diagnostics = [], []\n    with torch.no_grad():\n        for index, batch in enumerate(loader):\n            batch = cuda_batch(batch)\n            memory, padding = model.memory(batch, intervention)\n            seq, _ = rollout(model, memory, padding, tokenizer, sample=False)\n            texts = decode(seq, tokenizer)\n            target = batch[\'captions\'].flatten(0, 1)\n            mask = batch[\'caption_mask\'].flatten(0, 1)\n            logits = model.teacher(memory, padding, target[:, :-1], mask[:, :-1])\n            logp = logits.log_softmax(-1).gather(-1, target[:, 1:, None]).squeeze(-1)\n            valid = target[:, 1:].ne(tokenizer.pad_idx)\n            nll = (-(logp * valid).sum(1) / valid.sum(1).clamp_min(1)).reshape(-1, 5).mean(1).cpu().tolist()\n            ids = batch[\'image_id\'].cpu().tolist()\n            for i, (image_id, caption, value) in enumerate(zip(ids, texts, nll)):\n                records.append({\'image_id\': image_id, \'caption\': caption})\n                row = {\'image_id\': image_id, \'reference_nll_per_token\': value}\n                if model.extra is not None:\n                    v = batch[\'valid\'][i]\n                    row[\'valid_evidence_slots\']=int(v.sum())\n                    att=model.last_attention[i]\n                    row[\'attention_entropy\']=float((-(att.clamp_min(1e-12).log()*att).sum(-1))[v].mean()) if v.any() else None\n                diagnostics.append(row)\n            if index % 10 == 0:\n                print(f\'Evaluation {intervention}: {len(records)} local images\', flush=True)\n    return records, diagnostics\n\n\ndef worker(protocol):\n    import torch\n    import torch.distributed as dist\n    from torch.nn.parallel import DistributedDataParallel as DDP\n    from captioning.scst import TrainingCider, rollout, decode\n    from captioning.object_reward import TrainingObjectReward, combine_rewards\n    from captioning.scst_runtime import configure_strict_runtime, runtime_manifest, model_digest\n    cfg = json.loads(Path(protocol).read_text())\n    rank, world = int(os.environ[\'RANK\']), int(os.environ[\'WORLD_SIZE\'])\n    local_rank = int(os.environ[\'LOCAL_RANK\'])\n    if world != 2:\n        raise RuntimeError(\'This pilot requires exactly two GPU ranks for ONE arm.\')\n    torch.cuda.set_device(local_rank)\n    torch.set_num_threads(2)\n    configure_strict_runtime(True)\n    # Both GPUs belong to this single arm; NCCL synchronizes its gradients.\n    from datetime import timedelta\n    dist.init_process_group(\'nccl\', timeout=timedelta(minutes=20))\n    out = Path(cfg[\'output\'])\n    save(out / f\'runtime_rank{rank}.json\', runtime_manifest())\n    seed_all(42)\n    dataset, tokenizer = corpus(cfg)\n    model = build_model(cfg, tokenizer)\n    initial_base_hash = model_digest(model.base)\n    train = make_loader(cfg, dataset, tokenizer, \'train\', rank, world)\n    validation = make_loader(cfg, dataset, tokenizer, \'val\', rank, world)\n    if len(train) != math.ceil(len(cfg[\'groups\'][\'train\'])/cfg[\'global_batch_size\']):\n        raise ValueError(\'Unexpected DDP training step count.\')\n    # Inspect the frozen starting system with extra tokens masked off, on the same GPU/profile.\n    records, detail = predict(model, validation, tokenizer, \'off\')\n    save(out / f\'origin_rank{rank}.json\', records)\n    save(out / f\'origin_diagnostic_rank{rank}.json\', detail)\n    # Every rank uses identical source initialization; different sampling RNG streams are explicit.\n    seed_all(42 + rank)\n    if digest(out / \'train_object_targets.json\') != cfg[\'object_targets_sha256\']:\n        raise ValueError(\'Object reward targets changed.\')\n    targets = json.loads((out / \'train_object_targets.json\').read_text())[\'data\']\n    object_reward = TrainingObjectReward(targets, \'aliases_v2\', \'unique_count_v2\')\n    # Retain full TRAIN IDF like the successful full-train baseline, even in this 10k pilot.\n    references = {i: [s[\'raw\'] for s in row[\'sentences\'][:5]] for i, row in enumerate(dataset[\'images\'])\n                  if row[\'split\'] in (\'train\', \'restval\')}\n    print(f\'[rank {rank}] Building fixed TRAIN CIDEr-D IDF: {len(references)} images; CPU work, please wait.\', flush=True)\n    reward = TrainingCider(references, reference_cache_size=256)\n    print(f\'[rank {rank}] TRAIN reward ready; starting synchronized updates.\', flush=True)\n    params = [{\'params\': [p for p in model.base.parameters() if p.requires_grad], \'lr\': cfg[\'base_lr\']}]\n    if model.extra is not None:\n        params.append({\'params\': list(model.extra.parameters()), \'lr\': cfg[\'branch_lr\']})\n    optimizer = torch.optim.AdamW(params)\n    # No collective calls during sampled/greedy decoding: those use the unwrapped model in no_grad.\n    # All ranks execute exactly ONE synchronized loss backward per update.\n    ddp = DDP(model, device_ids=[local_rank], output_device=local_rank, broadcast_buffers=False,\n              find_unused_parameters=True)\n    history = []\n    begin = time.monotonic()\n    for step, batch in enumerate(train, 1):\n        batch = cuda_batch(batch)\n        optimizer.zero_grad(set_to_none=True)\n        weight = 0. if step <= cfg[\'warmup_updates\'] else cfg[\'scst_weight\']\n        sample = active = advantage = None\n        record = {\'step\': step, \'global_images_seen\': min(step*cfg[\'global_batch_size\'], 10000),\n                  \'scst_weight\': weight}\n        if weight:\n            with torch.no_grad():\n                memory, padding = model.memory(batch)\n                sample, active = rollout(model, memory, padding, tokenizer, sample=True)\n                greedy, _ = rollout(model, memory, padding, tokenizer, sample=False)\n            ids = batch[\'image_id\'].cpu().tolist()\n            sampled_text, greedy_text = decode(sample, tokenizer), decode(greedy, tokenizer)\n            sample_cider, greedy_cider = reward.score(ids, sampled_text), reward.score(ids, greedy_text)\n            sample_object, _ = object_reward.score(ids, sampled_text)\n            greedy_object, _ = object_reward.score(ids, greedy_text)\n            sample_reward = combine_rewards(sample_cider, sample_object, .5)\n            greedy_reward = combine_rewards(greedy_cider, greedy_object, .5)\n            advantage = torch.from_numpy(sample_reward - greedy_reward).cuda()\n            record.update(sample_reward=float(sample_reward.mean()), greedy_reward=float(greedy_reward.mean()),\n                          sample_cider=float(sample_cider.mean()), greedy_cider=float(greedy_cider.mean()),\n                          sample_object=float(sample_object.mean()), greedy_object=float(greedy_object.mean()))\n        loss, ce, rl = ddp(batch, sample, active, advantage, weight, tokenizer.pad_idx)\n        if not torch.isfinite(loss):\n            raise FloatingPointError(f\'Nonfinite loss, rank {rank}, step {step}.\')\n        loss.backward()\n        branch_grad = (sum(float(p.grad.detach().square().sum()) for p in model.extra.parameters() if p.grad is not None)**.5\n                       if model.extra is not None else 0.)\n        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)\n        optimizer.step()\n        record.update(alignment=model.last_alignment,\n                      submodule_grad_norms={name:sum(float(p.grad.detach().square().sum()) for p in part.parameters() if p.grad is not None)**.5 for name,part in model.extra.named_children()} if model.extra is not None else {},\n                      loss=float(loss.detach()), ce_loss=float(ce), scst_loss=float(rl),\n                      branch_grad_norm=branch_grad, grad_norm=float(grad_norm), seconds=time.monotonic()-begin)\n        history.append(record)\n        if step == 1 or step % 25 == 0 or step == len(train):\n            save(out / f\'history_rank{rank}.json\', history)\n            print(f\'[{cfg["arm"]} rank {rank}] {json.dumps(record)}\', flush=True)\n    if rank == 0:\n        state = {\'format\': \'semantic_evidence_pilot_v1\',\n                 \'model_state_dict\': {k: v.detach().cpu() for k, v in model.state_dict().items()},\n                 \'arm\': cfg[\'arm\'], \'caption_word2idx\': tokenizer.word2idx, \'cohort_sha256\': cfg[\'cohort_sha256\'],\n                 \'source_checkpoint_sha256\': cfg[\'source_checkpoint_sha256\'],\n                 \'protocol_sha256\': digest(protocol), \'completed_updates\': len(train)}\n        temporary = out / \'checkpoint.pth.partial\'\n        # Legacy serialization avoids Kaggle treating a .pth checkpoint as a ZIP to unpack.\n        torch.save(state, temporary, _use_new_zipfile_serialization=False)\n        temporary.replace(out / \'checkpoint.pth\')\n    # Evaluation has no DDP communication; different rank runtimes cannot stall loss collectives.\n    conditions = [\'on\'] if cfg[\'arm\'] == \'A\' else [\'on\', \'off\', \'text_swap\', \'region_swap\']\n    for condition in conditions:\n        records, detail = predict(model, validation, tokenizer, condition)\n        save(out / f\'{condition}_rank{rank}.json\', records)\n        save(out / f\'{condition}_diagnostic_rank{rank}.json\', detail)\n    save(out / f\'worker_complete_rank{rank}.json\', {\'status\': \'COMPLETE\', \'updates\': len(train),\n         \'initial_base_sha256\': initial_base_hash, \'rank\': rank, \'source_arm\': cfg[\'arm\'],\n         \'trainable_base_parameters\': sum(p.numel() for p in model.base.parameters() if p.requires_grad),\n         \'trainable_branch_parameters\': sum(p.numel() for p in model.extra.parameters()) if model.extra is not None else 0,\n         \'test_used\': False})\n    dist.destroy_process_group()\n\n\ndef evaluate(protocol):\n    from captioning.metrics import score_files\n    cfg = json.loads(Path(protocol).read_text())\n    out = Path(cfg[\'output\'])\n    dataset = json.loads(Path(cfg[\'dataset_json\']).read_text())\n    ground_truth(cfg, dataset, out)\n    expected = {r[\'image_id\']: r for r in cfg[\'groups\'][\'val\']}\n    conditions = [\'origin\', \'on\'] + ([] if cfg[\'arm\'] == \'A\' else [\'off\', \'text_swap\', \'region_swap\'])\n    results, predictions, details = {}, {}, {}\n    for condition in conditions:\n        rows, detail = [], []\n        for rank in (0, 1):\n            completion = json.loads((out / f\'worker_complete_rank{rank}.json\').read_text())\n            if completion[\'status\'] != \'COMPLETE\':\n                raise RuntimeError(\'A GPU worker did not finish.\')\n            rows.extend(json.loads((out / f\'{condition}_rank{rank}.json\').read_text()))\n            detail.extend(json.loads((out / f\'{condition}_diagnostic_rank{rank}.json\').read_text()))\n        rows.sort(key=lambda r: r[\'image_id\'])\n        if len(rows) != len(expected) or {r[\'image_id\'] for r in rows} != expected.keys():\n            raise ValueError(\'Missing/duplicated validation predictions.\')\n        path = out / f\'{condition}_predictions.json\'\n        save(path, rows)\n        predictions[condition] = {r[\'image_id\']: r[\'caption\'] for r in rows}\n        details[condition] = {r[\'image_id\']: r for r in detail}\n        if len(details[condition]) != len(expected):\n            raise ValueError(\'Incomplete validation diagnostics.\')\n        # Score branch removal as well: a caption change alone is not evidence of a useful branch.\n        if condition not in (\'origin\', \'on\', \'off\'):\n            continue\n        caption = score_files(path, out / \'val_gt.json\', out / f\'{condition}_metrics.json\')\n        chair_input = out / f\'{condition}_chair_input.json\'\n        chair_output = out / f\'{condition}_chair_output.json\'\n        save(chair_input, [{\'image_id\': expected[r[\'image_id\']][\'coco_id\'], \'caption\': r[\'caption\']} for r in rows])\n        subprocess.run([sys.executable, \'-u\', str(Path(cfg[\'chair\'])/\'chair.py\'), \'--cap_file\', str(chair_input),\n                        \'--cache\', str(Path(cfg[\'chair\'])/\'chair.pkl\'), \'--save_path\', str(chair_output)],\n                       cwd=cfg[\'chair\'], check=True)\n        chair = json.loads(chair_output.read_text())\n        if {r[\'image_id\'] for r in chair[\'sentences\']} != {r[\'coco_id\'] for r in expected.values()}:\n            raise ValueError(\'CHAIR IDs differ.\')\n        # Per-image CIDEr uses the SAME full validation IDF for all mode subsets.\n        from pycocotools.coco import COCO\n        from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n        from pycocoevalcap.cider.cider import Cider\n        gt = COCO(str(out / \'val_gt.json\'))\n        tokenizer = PTBTokenizer()\n        ids = [r[\'image_id\'] for r in rows]\n        refs = tokenizer.tokenize({i: gt.imgToAnns[i] for i in ids})\n        hyp = tokenizer.tokenize({r[\'image_id\']: [{\'caption\': r[\'caption\']}] for r in rows})\n        _, per_image = Cider().compute_score(refs, hyp)\n        grouped = {}\n        for mode in (\'object_relation\', \'object_only\', \'no_objects\'):\n            indices = [i for i, row in enumerate(rows) if expected[row[\'image_id\']][\'mode\'] == mode]\n            grouped[mode] = {\'images\': len(indices), \'CIDEr_full_val_IDF\': float(np.mean(np.asarray(per_image)[indices])) if indices else None}\n        save(out / f\'{condition}_per_image_cider.json\', [{\'image_id\': r[\'image_id\'], \'CIDEr\': float(v)} for r, v in zip(rows, per_image)])\n        results[condition] = {\'caption\': caption, \'chair\': chair[\'overall_metrics\'], \'by_mode\': grouped}\n    sensitivity = {}\n    for condition in conditions:\n        if condition in (\'origin\', \'on\'):\n            continue\n        sensitivity[condition] = {}\n        for mode in (\'all\', \'object_relation\', \'object_only\', \'no_objects\'):\n            ids = [i for i, row in expected.items() if mode == \'all\' or row[\'mode\'] == mode]\n            sensitivity[condition][mode] = {\'images\': len(ids),\n                \'changed_captions\': sum(predictions[condition][i] != predictions[\'on\'][i] for i in ids),\n                \'mean_reference_nll_delta\': float(np.mean([details[condition][i][\'reference_nll_per_token\'] -\n                                                          details[\'on\'][i][\'reference_nll_per_token\'] for i in ids])) if ids else None}\n    summary = {\'status\': \'PILOT_COMPLETE\', \'arm\': cfg[\'arm\'], \'metrics\': results, \'sensitivity\': sensitivity,\n               \'cohort_sha256\': cfg[\'cohort_sha256\'], \'stage_sha256\': cfg[\'stage_sha256\'],\n               \'protocol_sha256\': digest(protocol), \'checkpoint_sha256\': digest(out / \'checkpoint.pth\'),\n               \'test_used\': False, \'selection\': \'End checkpoint only, fixed before metrics; no automatic promotion\',\n               \'policy\': cfg[\'evaluation_policy\'], \'limits\': [cfg[\'validation_scope\'], cfg[\'comparison_limits\'],\n               \'Greedy pilot metrics must not be compared directly to historical beam5/fixed test metrics.\',\n               \'Text/region swaps are sensitivity probes, not verified false relations.\',\n               \'CHAIR measures objects, not predicate correctness; contrastive scores are not calibrated probabilities.\',\n               \'Different accounts/GPUs may differ numerically; compare input/cohort/runtime manifests.\']}\n    save(out / \'run_summary.json\', summary)\n    print(json.dumps(summary, indent=2), flush=True)\n\n\ndef archive(protocol):\n    cfg = json.loads(Path(protocol).read_text())\n    out = Path(cfg[\'output\'])\n    with zipfile.ZipFile(str(out) + \'_review.zip\', \'w\', zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(out.rglob(\'*\')):\n            if path.is_file() and path.suffix in (\'.json\', \'.jsonl\', \'.log\', \'.py\'):\n                z.write(path, path.relative_to(out))\n    print(\'Return:\', str(out) + \'_review.zip\', flush=True)\n    print(\'Keep checkpoint separately in saved notebook output:\', out / \'checkpoint.pth\', flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'mode\', choices=[\'prepare\', \'worker\', \'evaluate\', \'archive\'])\n    parser.add_argument(\'--config\', required=True)\n    args = parser.parse_args()\n    globals()[args.mode](args.config)\n', 'probe_data.py': '"""Verified pair construction and tie-aware summaries; no model or training imports."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport numpy as np\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef identity(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()).hexdigest()\n\n\ndef bounds(subject, obj):\n    s, o = np.asarray(subject, dtype=np.float32), np.asarray(obj, dtype=np.float32)\n    for value in (s, o):\n        if value.shape != (4,) or not np.isfinite(value).all() or (value < 0).any() or (value > 1).any() or (value[2:] <= value[:2]).any():\n            raise ValueError(\'Invalid normalized xyxy box\')\n    return np.stack((s, o, np.r_[np.minimum(s[:2], o[:2]), np.maximum(s[2:], o[2:])])).tolist()\n\n\ndef human_pairs(cases, annotations, excluded, captions, relation_text):\n    result = []\n    for case in cases[\'cases\']:\n        if case[\'relation\'] not in (\'holding\', \'riding\'):\n            continue\n        a = annotations[\'annotations\'].get(case[\'case_id\'], {})\n        if case[\'filename\'] in excluded or captions[case[\'filename\']][\'split\'] not in (\'train\', \'restval\'):\n            continue\n        if any(a.get(key) != \'clear\' for key in (\'subject_box_status\', \'A_box_status\', \'B_box_status\')):\n            continue\n        status = (a.get(\'A_status\'), a.get(\'B_status\'))\n        if status not in ((\'supported\', \'contradicted\'), (\'contradicted\', \'supported\')):\n            continue\n        targets = case[\'targets\']\n        if targets[0][\'label\'] != targets[1][\'label\']:\n            continue\n        candidates = [dict(text=relation_text(case[\'relation\'], t[\'label\']), bounds=bounds(case[\'subject_box\'], t[\'bbox\'])) for t in targets]\n        result.append(dict(case_id=case[\'case_id\'], filename=case[\'filename\'], task=\'wrong_object_instance\',\n                           predicate=case[\'relation\'], object_label=targets[0][\'label\'], candidates=candidates,\n                           positive_index=status.index(\'supported\'), label_source=\'human_explicit_supported_vs_contradicted\'))\n    return result\n\n\ndef vcoco_pairs(records, excluded, captions, relation_text, per_predicate=50):\n    candidates = defaultdict(list)\n    # Restrict verb swaps to classes plausibly associated with riding; no phone-vs-horse shortcut.\n    ride_objects = {\'bicycle\', \'motorcycle\', \'horse\', \'elephant\', \'skateboard\', \'surfboard\', \'snowboard\', \'skis\', \'boat\', \'car\', \'bus\', \'truck\', \'train\'}\n    for image in records:\n        if image[\'filename\'] in excluded or captions[image[\'filename\']][\'split\'] not in (\'train\', \'restval\'):\n            continue\n        objects = {r[\'instance_id\']: r for r in image[\'instances\']}\n        labels = {(r[\'subject_id\'], r[\'predicate\']): r[\'label\'] for r in image[\'actor_action_labels\']}\n        for edge in image[\'relations\']:\n            sid, oid, verb = edge[\'subject_id\'], edge[\'object_id\'], edge[\'predicate\']\n            if verb not in (\'holding\', \'riding\'):\n                continue\n            subject, obj = objects[sid], objects[oid]\n            assert labels.get((sid, verb)) == 1 and subject[\'category\'] == \'person\'\n            b = bounds(subject[\'box\'], obj[\'box\'])\n            positive = dict(text=relation_text(verb, obj[\'category\']), bounds=b)\n            alternate = \'riding\' if verb == \'holding\' else \'holding\'\n            common = dict(filename=image[\'filename\'], predicate=verb, object_label=obj[\'category\'], label_source=\'vcoco_positive_role_and_explicit_actor_zero\')\n            if obj[\'category\'] in ride_objects and labels.get((sid, alternate)) == 0:\n                candidates[(\'wrong_predicate\', verb)].append(dict(common, case_id=f\'vcoco_verb_{image["image_id"]}_{sid}_{oid}_{verb}\',\n                    task=\'wrong_predicate\', candidates=[positive, dict(text=relation_text(alternate, obj[\'category\']), bounds=b)], positive_index=0))\n            for other in objects.values():\n                if other[\'instance_id\'] == sid or other[\'category\'] != \'person\' or labels.get((other[\'instance_id\'], verb)) != 0:\n                    continue\n                candidates[(\'wrong_subject_instance\', verb)].append(dict(common,\n                    case_id=f\'vcoco_actor_{image["image_id"]}_{sid}_{oid}_{other["instance_id"]}_{verb}\', task=\'wrong_subject_instance\',\n                    candidates=[positive, dict(text=positive[\'text\'], bounds=bounds(other[\'box\'], obj[\'box\']))], positive_index=0))\n    result = []\n    for task in (\'wrong_predicate\', \'wrong_subject_instance\'):\n        groups = []\n        for verb in (\'holding\', \'riding\'):\n            # One pair per image/predicate; selection depends only on stable IDs, never model scores.\n            ordered = sorted(candidates[(task, verb)], key=lambda c: identity(c[\'case_id\']))\n            seen, selected = set(), []\n            for row in ordered:\n                if row[\'filename\'] not in seen:\n                    seen.add(row[\'filename\']); selected.append(row)\n            groups.append(selected)\n        if task == \'wrong_predicate\':\n            # Balance positive verbs WITHIN each object class, before seeing any model score.\n            remaining = per_predicate\n            for label in sorted(ride_objects):\n                per_class = [[r for r in g if r[\'object_label\'] == label] for g in groups]\n                count = min(remaining, *(len(g) for g in per_class))\n                result.extend(row for group in per_class for row in group[:count])\n                remaining -= count\n        else:\n            count = min(per_predicate, *(len(g) for g in groups))\n            result.extend(row for group in groups for row in group[:count])\n    return result\n\n\ndef finalize(rows, captions):\n    result = []\n    for original in rows:\n        row = dict(original)\n        row[\'coco_id\'] = int(Path(row[\'filename\']).stem.rsplit(\'_\', 1)[1])\n        row[\'caption_split\'] = captions[row[\'filename\']][\'split\']\n        if int(identity(row[\'case_id\'])[:2], 16) % 2:\n            row[\'candidates\'] = row[\'candidates\'][::-1]\n            row[\'positive_index\'] = 1-row[\'positive_index\']\n        result.append(row)\n    assert len({r[\'case_id\'] for r in result}) == len(result)\n    return sorted(result, key=lambda r: (r[\'task\'], r[\'case_id\']))\n\n\ndef summarize(rows):\n    output = {}\n    for task in sorted({r[\'task\'] for r in rows}):\n        selected = [r for r in rows if r[\'task\'] == task]\n        methods = {}\n        for method in (\'boxed\', \'global_view\', \'zero_visual\'):\n            by_image = defaultdict(list)\n            margins, ties = [], 0\n            for row in selected:\n                positive = row[\'positive_index\']\n                scores = row[\'scores\'][method]\n                if len(scores) != 2 or not all(math.isfinite(s) for s in scores):\n                    raise ValueError(\'Nonfinite or incomplete scores\')\n                margin = scores[positive]-scores[1-positive]\n                tie = abs(margin) <= 1e-7\n                margins.append(margin); ties += tie\n                by_image[row[\'coco_id\']].append(.5 if tie else float(margin > 0))\n            values = np.asarray([np.mean(v) for v in by_image.values()])\n            rng = np.random.default_rng(20261019)\n            intervals = np.percentile([values[rng.integers(0, len(values), len(values))].mean() for _ in range(2000)], [2.5, 97.5])\n            methods[method] = dict(image_macro_pair_accuracy=float(values.mean()), image_cluster_bootstrap95=intervals.tolist(),\n                                   pair_accuracy=float(sum(sum(v) for v in by_image.values())/len(selected)),\n                                   ties=ties, mean_cosine_margin=float(np.mean(margins)))\n        output[task] = dict(pairs=len(selected), images=len({r[\'coco_id\'] for r in selected}),\n                            predicates=dict(Counter(r[\'predicate\'] for r in selected)), methods=methods)\n    return output\n', 'synthetic_fixtures.py': '"""Kaggle CPU preflight; no pretrained downloads or notebook launch."""\nimport unittest\nimport torch\nfrom torch import nn\nfrom pilot_model import EvidenceMemory, PilotModel\n\n\nclass FakeDecoder(nn.Module):\n    def __init__(self):\n        super().__init__();self.embedding=nn.Embedding(9,512);self.out=nn.Linear(512,9)\n    def forward(self,tgt,memory,tgt_key_padding_mask=None,memory_key_padding_mask=None):\n        keep=(~memory_key_padding_mask)[...,None]\n        value=(memory*keep).sum(1)/keep.sum(1).clamp_min(1)\n        return self.out(self.embedding(tgt)+value[:,None])\n\n\nclass FakeBase(nn.Module):\n    def __init__(self):\n        super().__init__();self.projection=nn.Linear(768,512);self.decoder=FakeDecoder()\n    def build_memory(self,visual,legacy,mask):\n        memory=torch.cat((legacy,self.projection(visual[:,:2])),1)\n        padding=torch.cat((mask.eq(0),torch.zeros(len(visual),2,dtype=torch.bool,device=visual.device)),1)\n        return memory,padding\n\n\ndef batch(device=\'cpu\'):\n    b=2\n    bounds=torch.tensor([0.,0.,1.,1.],device=device).expand(b,9,3,4).clone()\n    d=dict(visual=torch.randn(b,197,768,device=device),legacy=torch.randn(b,2,512,device=device),\n           legacy_mask=torch.ones(b,2,dtype=torch.long,device=device),text=torch.randn(b,9,512,device=device),\n           roles=torch.tensor([[0,1,2]*3]*b,device=device),valid=torch.ones(b,9,dtype=torch.bool,device=device),bounds=bounds,\n           phrase_ids=torch.arange(1,19,device=device).reshape(b,9),image_id=torch.arange(b,device=device),\n           captions=torch.tensor([[[1,4,5,2,0]]*5]*b,device=device),\n           caption_mask=torch.tensor([[[False,False,False,False,True]]*5]*b,device=device))\n    d.update(swap_text=d[\'text\'].flip(0),swap_visual=d[\'visual\'].flip(0),swap_bounds=bounds.flip(0),swap_valid=d[\'valid\'].clone())\n    return d\n\n\n'}
SOURCE_HASHES = {'pilot_model.py': '88cb1f4587b88d1400ab05ac7aa8f8f0535a9e16c698f60036270290df9a3d59', 'run_verified.py': '3e3be190b8b3e337efd1357824dcaa7e30f75ce59227d9c0d78bac0a579d918f', 'verified_data.py': '3b197271e103fb7f5b8513b02f4152b972136f4f243e13e829ff0395f9521878', 'test_verified_data.py': '79b5cab158c9ce5658eeaab0c7b0540df6b447ba42b2bfc6f5bb3e91ffb95a20', 'test_verified_model.py': '02f0bda7591229022df3b379dce6fedb9533f139bc0a8a76b45828794482bc01', 'test_verified_ddp.py': 'cb4884a036b736268cccd51bea0eb92e6818a731bf25a1c24094bf77c65ed6a6', 'pilot_data.py': 'aac539b7c23f4acc3ac05b99496ef8eb79f740e6c99d11742b6de08fd2b892c0', 'test_pilot_data.py': '5481c322e2266447028d93c694d765b3dbefb77428d6a9a9eea31d75422e1f8f', 'pilot_metric_smoke.py': '1f33a26f3edcd21574189464633ac2060ac8cff0a69ec0af27505b07cecbadf6', 'evidence_core.py': '4f8189de7d183bb062247043d6a2ce7bcb6fbd9fb7d0d485922c065d8cbe962b', 'base_runner.py': 'b8a58ecdccf89151aedbedad3ca0ad73832438db8bae707cdcf1aa5800830801', 'probe_data.py': 'f8dfb24cf825ecc2935bfc062731009197459cd55ebb9b0cbb146d248d574e9f', 'synthetic_fixtures.py': '44671927a64ed6d9d0c3a09137fcf9e680641ebba64e29d1d233945026ea1988'}
for name, source in SOURCES.items(): (REPO/name).write_text(source,encoding='utf-8')
VERIFIED_LABELS = '{\n  "schema_version": 1,\n  "method": "verified_same_image_F",\n  "splits": {\n    "fit": [\n      {\n        "filename": "COCO_train2014_000000100668.jpg",\n        "coco_id": 100668,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "banana",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 444814,\n          "object_id": 1547320,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_100668_444814_1547320_carry_eat",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 444814,\n          "object_id": 1547320,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.6030468940734863,\n                0.22634804248809814,\n                0.8399531245231628,\n                0.9380637407302856\n              ],\n              [\n                0.6506562232971191,\n                0.06948529183864594,\n                0.8397343754768372,\n                0.22468137741088867\n              ],\n              [\n                0.6030468940734863,\n                0.06948529183864594,\n                0.8399531245231628,\n                0.9380637407302856\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a banana",\n            "bounds": [\n              [\n                0.6030468940734863,\n                0.22634804248809814,\n                0.8399531245231628,\n                0.9380637407302856\n              ],\n              [\n                0.6506562232971191,\n                0.06948529183864594,\n                0.8397343754768372,\n                0.22468137741088867\n              ],\n              [\n                0.6030468940734863,\n                0.06948529183864594,\n                0.8399531245231628,\n                0.9380637407302856\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000107843.jpg",\n        "coco_id": 107843,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "umbrella",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 522539,\n          "object_id": 280479,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_107843_522539_280479_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 522539,\n          "object_id": 280479,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.4559687376022339,\n                0.34872570633888245,\n                0.5707343816757202,\n                0.7178833484649658\n              ],\n              [\n                0.441468745470047,\n                0.32555076479911804,\n                0.5842187404632568,\n                0.40673866868019104\n              ],\n              [\n                0.441468745470047,\n                0.32555076479911804,\n                0.5842187404632568,\n                0.7178833484649658\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.4559687376022339,\n                0.34872570633888245,\n                0.5707343816757202,\n                0.7178833484649658\n              ],\n              [\n                0.441468745470047,\n                0.32555076479911804,\n                0.5842187404632568,\n                0.40673866868019104\n              ],\n              [\n                0.441468745470047,\n                0.32555076479911804,\n                0.5842187404632568,\n                0.7178833484649658\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000114540.jpg",\n        "coco_id": 114540,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1720105,\n          "object_id": 241661,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_114540_1720105_241661_hold_ride",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1720105,\n          "object_id": 241661,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.4153437614440918,\n                0.4627962112426758,\n                0.5554375052452087,\n                0.7991153001785278\n              ],\n              [\n                0.4738749861717224,\n                0.600300133228302,\n                0.6109218597412109,\n                0.8242496252059937\n              ],\n              [\n                0.4153437614440918,\n                0.4627962112426758,\n                0.6109218597412109,\n                0.8242496252059937\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.4153437614440918,\n                0.4627962112426758,\n                0.5554375052452087,\n                0.7991153001785278\n              ],\n              [\n                0.4738749861717224,\n                0.600300133228302,\n                0.6109218597412109,\n                0.8242496252059937\n              ],\n              [\n                0.4153437614440918,\n                0.4627962112426758,\n                0.6109218597412109,\n                0.8242496252059937\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000138103.jpg",\n        "coco_id": 138103,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "banana",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 430103,\n          "object_id": 1041498,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_138103_430103_1041498_eat_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 430103,\n          "object_id": 1041498,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.0029843749944120646,\n                0.006894117686897516,\n                0.7371875047683716,\n                0.9844235181808472\n              ],\n              [\n                0.3753125071525574,\n                0.5432705879211426,\n                0.6376718878746033,\n                0.9080941081047058\n              ],\n              [\n                0.0029843749944120646,\n                0.006894117686897516,\n                0.7371875047683716,\n                0.9844235181808472\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a banana",\n            "bounds": [\n              [\n                0.0029843749944120646,\n                0.006894117686897516,\n                0.7371875047683716,\n                0.9844235181808472\n              ],\n              [\n                0.3753125071525574,\n                0.5432705879211426,\n                0.6376718878746033,\n                0.9080941081047058\n              ],\n              [\n                0.0029843749944120646,\n                0.006894117686897516,\n                0.7371875047683716,\n                0.9844235181808472\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000159948.jpg",\n        "coco_id": 159948,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 252836,\n          "object_id": 127084,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_159948_252836_127084_hold_sit",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 252836,\n          "object_id": 127084,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.6111875176429749,\n                0.5051093697547913,\n                0.7046666741371155,\n                0.7314687371253967\n              ],\n              [\n                0.5790208578109741,\n                0.5962656140327454,\n                0.7522500157356262,\n                0.7404062747955322\n              ],\n              [\n                0.5790208578109741,\n                0.5051093697547913,\n                0.7522500157356262,\n                0.7404062747955322\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.6111875176429749,\n                0.5051093697547913,\n                0.7046666741371155,\n                0.7314687371253967\n              ],\n              [\n                0.5790208578109741,\n                0.5962656140327454,\n                0.7522500157356262,\n                0.7404062747955322\n              ],\n              [\n                0.5790208578109741,\n                0.5051093697547913,\n                0.7522500157356262,\n                0.7404062747955322\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000168900.jpg",\n        "coco_id": 168900,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 212940,\n          "object_id": 131141,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_168900_212940_131141_ride_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 212940,\n          "object_id": 131141,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7388443350791931\n              ],\n              [\n                0.27892187237739563,\n                0.6338443160057068,\n                0.3210781216621399,\n                0.7727358341217041\n              ],\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7727358341217041\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7388443350791931\n              ],\n              [\n                0.27892187237739563,\n                0.6338443160057068,\n                0.3210781216621399,\n                0.7727358341217041\n              ],\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7727358341217041\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000168900.jpg",\n        "coco_id": 168900,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 212940,\n          "object_id": 131141,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_168900_212940_131141_sit_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 212940,\n          "object_id": 131141,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7388443350791931\n              ],\n              [\n                0.27892187237739563,\n                0.6338443160057068,\n                0.3210781216621399,\n                0.7727358341217041\n              ],\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7727358341217041\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7388443350791931\n              ],\n              [\n                0.27892187237739563,\n                0.6338443160057068,\n                0.3210781216621399,\n                0.7727358341217041\n              ],\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7727358341217041\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000175188.jpg",\n        "coco_id": 175188,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "truck",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1741700,\n          "object_id": 395017,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_175188_1741700_395017_hold_ride",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1741700,\n          "object_id": 395017,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a truck",\n            "bounds": [\n              [\n                0.5491406321525574,\n                0.2676442265510559,\n                0.8437187671661377,\n                0.8735576868057251\n              ],\n              [\n                0.0043750000186264515,\n                0.0022355769760906696,\n                0.5813437700271606,\n                0.8988701701164246\n              ],\n              [\n                0.0043750000186264515,\n                0.0022355769760906696,\n                0.8437187671661377,\n                0.8988701701164246\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a truck",\n            "bounds": [\n              [\n                0.5491406321525574,\n                0.2676442265510559,\n                0.8437187671661377,\n                0.8735576868057251\n              ],\n              [\n                0.0043750000186264515,\n                0.0022355769760906696,\n                0.5813437700271606,\n                0.8988701701164246\n              ],\n              [\n                0.0043750000186264515,\n                0.0022355769760906696,\n                0.8437187671661377,\n                0.8988701701164246\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000183790.jpg",\n        "coco_id": 183790,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1278622,\n          "object_id": 1184026,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_183790_1278622_1184026_carry_sit",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1278622,\n          "object_id": 1184026,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.3387968838214874,\n                0.4948946237564087,\n                0.44600000977516174,\n                0.9338407516479492\n              ],\n              [\n                0.34028124809265137,\n                0.7413114905357361,\n                0.4270625114440918,\n                0.8733723759651184\n              ],\n              [\n                0.3387968838214874,\n                0.4948946237564087,\n                0.44600000977516174,\n                0.9338407516479492\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.3387968838214874,\n                0.4948946237564087,\n                0.44600000977516174,\n                0.9338407516479492\n              ],\n              [\n                0.34028124809265137,\n                0.7413114905357361,\n                0.4270625114440918,\n                0.8733723759651184\n              ],\n              [\n                0.3387968838214874,\n                0.4948946237564087,\n                0.44600000977516174,\n                0.9338407516479492\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000018554.jpg",\n        "coco_id": 18554,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "banana",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 196720,\n          "object_id": 1041972,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_18554_196720_1041972_eat_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 196720,\n          "object_id": 1041972,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.0,\n                0.10337500274181366,\n                0.852154552936554,\n                1.0\n              ],\n              [\n                0.2987353503704071,\n                0.46357813477516174,\n                0.6775878071784973,\n                0.9980937242507935\n              ],\n              [\n                0.0,\n                0.10337500274181366,\n                0.852154552936554,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a banana",\n            "bounds": [\n              [\n                0.0,\n                0.10337500274181366,\n                0.852154552936554,\n                1.0\n              ],\n              [\n                0.2987353503704071,\n                0.46357813477516174,\n                0.6775878071784973,\n                0.9980937242507935\n              ],\n              [\n                0.0,\n                0.10337500274181366,\n                0.852154552936554,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000018554.jpg",\n        "coco_id": 18554,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "banana",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 196720,\n          "object_id": 1041972,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_18554_196720_1041972_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 196720,\n          "object_id": 1041972,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a banana",\n            "bounds": [\n              [\n                0.0,\n                0.10337500274181366,\n                0.852154552936554,\n                1.0\n              ],\n              [\n                0.2987353503704071,\n                0.46357813477516174,\n                0.6775878071784973,\n                0.9980937242507935\n              ],\n              [\n                0.0,\n                0.10337500274181366,\n                0.852154552936554,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.0,\n                0.10337500274181366,\n                0.852154552936554,\n                1.0\n              ],\n              [\n                0.2987353503704071,\n                0.46357813477516174,\n                0.6775878071784973,\n                0.9980937242507935\n              ],\n              [\n                0.0,\n                0.10337500274181366,\n                0.852154552936554,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000190229.jpg",\n        "coco_id": 190229,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1211614,\n          "object_id": 1825699,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_190229_1211614_1825699_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1211614,\n          "object_id": 1825699,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.34034374356269836,\n                0.4449414610862732,\n                0.4677812457084656,\n                0.7932552695274353\n              ],\n              [\n                0.2708437442779541,\n                0.44344261288642883,\n                0.3578281104564667,\n                0.6182201504707336\n              ],\n              [\n                0.2708437442779541,\n                0.44344261288642883,\n                0.4677812457084656,\n                0.7932552695274353\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.34034374356269836,\n                0.4449414610862732,\n                0.4677812457084656,\n                0.7932552695274353\n              ],\n              [\n                0.2708437442779541,\n                0.44344261288642883,\n                0.3578281104564667,\n                0.6182201504707336\n              ],\n              [\n                0.2708437442779541,\n                0.44344261288642883,\n                0.4677812457084656,\n                0.7932552695274353\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000190229.jpg",\n        "coco_id": 190229,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1257269,\n          "object_id": 1824859,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_190229_1257269_1824859_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1257269,\n          "object_id": 1824859,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.8551718592643738,\n                0.40295082330703735,\n                0.9452812671661377,\n                0.8722482323646545\n              ],\n              [\n                0.9084218740463257,\n                0.4771662652492523,\n                0.9959999918937683,\n                0.6172365546226501\n              ],\n              [\n                0.8551718592643738,\n                0.40295082330703735,\n                0.9959999918937683,\n                0.8722482323646545\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.8551718592643738,\n                0.40295082330703735,\n                0.9452812671661377,\n                0.8722482323646545\n              ],\n              [\n                0.9084218740463257,\n                0.4771662652492523,\n                0.9959999918937683,\n                0.6172365546226501\n              ],\n              [\n                0.8551718592643738,\n                0.40295082330703735,\n                0.9959999918937683,\n                0.8722482323646545\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000190229.jpg",\n        "coco_id": 190229,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1697749,\n          "object_id": 1170451,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_190229_1697749_1170451_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1697749,\n          "object_id": 1170451,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.7720780968666077,\n                0.360117107629776,\n                0.866937518119812,\n                0.9137470722198486\n              ],\n              [\n                0.8164687752723694,\n                0.4400702714920044,\n                0.8879374861717224,\n                0.6108196973800659\n              ],\n              [\n                0.7720780968666077,\n                0.360117107629776,\n                0.8879374861717224,\n                0.9137470722198486\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.7720780968666077,\n                0.360117107629776,\n                0.866937518119812,\n                0.9137470722198486\n              ],\n              [\n                0.8164687752723694,\n                0.4400702714920044,\n                0.8879374861717224,\n                0.6108196973800659\n              ],\n              [\n                0.7720780968666077,\n                0.360117107629776,\n                0.8879374861717224,\n                0.9137470722198486\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000194842.jpg",\n        "coco_id": 194842,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "cake",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 462630,\n          "object_id": 1088064,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_194842_462630_1088064_eat_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 462630,\n          "object_id": 1088064,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.7670156359672546,\n                0.3617991507053375,\n                1.0,\n                0.6471966505050659\n              ],\n              [\n                0.5602656006813049,\n                0.4748953878879547,\n                0.7485156059265137,\n                0.6691213250160217\n              ],\n              [\n                0.5602656006813049,\n                0.3617991507053375,\n                1.0,\n                0.6691213250160217\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.7670156359672546,\n                0.3617991507053375,\n                1.0,\n                0.6471966505050659\n              ],\n              [\n                0.5602656006813049,\n                0.4748953878879547,\n                0.7485156059265137,\n                0.6691213250160217\n              ],\n              [\n                0.5602656006813049,\n                0.3617991507053375,\n                1.0,\n                0.6691213250160217\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000211205.jpg",\n        "coco_id": 211205,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1236480,\n          "object_id": 1445610,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_211205_1236480_1445610_sit_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1236480,\n          "object_id": 1445610,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.3102107644081116,\n                0.5369062423706055,\n                0.6312177777290344,\n                0.9137656092643738\n              ],\n              [\n                0.045761123299598694,\n                0.7095937728881836,\n                0.5110772848129272,\n                0.9785468578338623\n              ],\n              [\n                0.045761123299598694,\n                0.5369062423706055,\n                0.6312177777290344,\n                0.9785468578338623\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.3102107644081116,\n                0.5369062423706055,\n                0.6312177777290344,\n                0.9137656092643738\n              ],\n              [\n                0.045761123299598694,\n                0.7095937728881836,\n                0.5110772848129272,\n                0.9785468578338623\n              ],\n              [\n                0.045761123299598694,\n                0.5369062423706055,\n                0.6312177777290344,\n                0.9785468578338623\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000211205.jpg",\n        "coco_id": 211205,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1236480,\n          "object_id": 1445610,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_211205_1236480_1445610_sit_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1236480,\n          "object_id": 1445610,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.3102107644081116,\n                0.5369062423706055,\n                0.6312177777290344,\n                0.9137656092643738\n              ],\n              [\n                0.045761123299598694,\n                0.7095937728881836,\n                0.5110772848129272,\n                0.9785468578338623\n              ],\n              [\n                0.045761123299598694,\n                0.5369062423706055,\n                0.6312177777290344,\n                0.9785468578338623\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.3102107644081116,\n                0.5369062423706055,\n                0.6312177777290344,\n                0.9137656092643738\n              ],\n              [\n                0.045761123299598694,\n                0.7095937728881836,\n                0.5110772848129272,\n                0.9785468578338623\n              ],\n              [\n                0.045761123299598694,\n                0.5369062423706055,\n                0.6312177777290344,\n                0.9785468578338623\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000221311.jpg",\n        "coco_id": 221311,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1731597,\n          "object_id": 1182941,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_221311_1731597_1182941_carry_sit",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1731597,\n          "object_id": 1182941,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.1529218703508377,\n                0.49877646565437317,\n                0.2336718738079071,\n                0.7925176620483398\n              ],\n              [\n                0.10445312410593033,\n                0.7348235249519348,\n                0.2238437533378601,\n                0.9258353114128113\n              ],\n              [\n                0.10445312410593033,\n                0.49877646565437317,\n                0.2336718738079071,\n                0.9258353114128113\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.1529218703508377,\n                0.49877646565437317,\n                0.2336718738079071,\n                0.7925176620483398\n              ],\n              [\n                0.10445312410593033,\n                0.7348235249519348,\n                0.2238437533378601,\n                0.9258353114128113\n              ],\n              [\n                0.10445312410593033,\n                0.49877646565437317,\n                0.2336718738079071,\n                0.9258353114128113\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000231019.jpg",\n        "coco_id": 231019,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 526738,\n          "object_id": 1435531,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_231019_526738_1435531_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 526738,\n          "object_id": 1435531,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.12484374642372131,\n                0.08375000208616257,\n                0.28914061188697815,\n                0.7540094256401062\n              ],\n              [\n                0.15990625321865082,\n                0.3785141408443451,\n                0.2993437647819519,\n                0.566674530506134\n              ],\n              [\n                0.12484374642372131,\n                0.08375000208616257,\n                0.2993437647819519,\n                0.7540094256401062\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.12484374642372131,\n                0.08375000208616257,\n                0.28914061188697815,\n                0.7540094256401062\n              ],\n              [\n                0.15990625321865082,\n                0.3785141408443451,\n                0.2993437647819519,\n                0.566674530506134\n              ],\n              [\n                0.12484374642372131,\n                0.08375000208616257,\n                0.2993437647819519,\n                0.7540094256401062\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000249435.jpg",\n        "coco_id": 249435,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "cake",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 498692,\n          "object_id": 1087324,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_249435_498692_1087324_eat_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 498692,\n          "object_id": 1087324,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.37415623664855957,\n                0.11685416847467422,\n                0.8292187452316284,\n                0.48089584708213806\n              ],\n              [\n                0.45228123664855957,\n                0.4350416660308838,\n                0.5814843773841858,\n                0.543541669845581\n              ],\n              [\n                0.37415623664855957,\n                0.11685416847467422,\n                0.8292187452316284,\n                0.543541669845581\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.37415623664855957,\n                0.11685416847467422,\n                0.8292187452316284,\n                0.48089584708213806\n              ],\n              [\n                0.45228123664855957,\n                0.4350416660308838,\n                0.5814843773841858,\n                0.543541669845581\n              ],\n              [\n                0.37415623664855957,\n                0.11685416847467422,\n                0.8292187452316284,\n                0.543541669845581\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000251314.jpg",\n        "coco_id": 251314,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 216304,\n          "object_id": 1431441,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_251314_216304_1431441_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 216304,\n          "object_id": 1431441,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.9306718707084656,\n                0.6456131935119629,\n                1.0,\n                0.9785377383232117\n              ],\n              [\n                0.9821249842643738,\n                0.7652594447135925,\n                1.0,\n                0.8223820924758911\n              ],\n              [\n                0.9306718707084656,\n                0.6456131935119629,\n                1.0,\n                0.9785377383232117\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.9306718707084656,\n                0.6456131935119629,\n                1.0,\n                0.9785377383232117\n              ],\n              [\n                0.9821249842643738,\n                0.7652594447135925,\n                1.0,\n                0.8223820924758911\n              ],\n              [\n                0.9306718707084656,\n                0.6456131935119629,\n                1.0,\n                0.9785377383232117\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000254199.jpg",\n        "coco_id": 254199,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 472542,\n          "object_id": 1432207,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_254199_472542_1432207_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 472542,\n          "object_id": 1432207,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.17295774817466736,\n                0.09876562654972076,\n                0.9970422387123108,\n                0.9904062747955322\n              ],\n              [\n                0.4504929482936859,\n                0.6622499823570251,\n                0.5404459834098816,\n                0.9090468883514404\n              ],\n              [\n                0.17295774817466736,\n                0.09876562654972076,\n                0.9970422387123108,\n                0.9904062747955322\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.17295774817466736,\n                0.09876562654972076,\n                0.9970422387123108,\n                0.9904062747955322\n              ],\n              [\n                0.4504929482936859,\n                0.6622499823570251,\n                0.5404459834098816,\n                0.9090468883514404\n              ],\n              [\n                0.17295774817466736,\n                0.09876562654972076,\n                0.9970422387123108,\n                0.9904062747955322\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000265810.jpg",\n        "coco_id": 265810,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "pizza",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1720215,\n          "object_id": 1072259,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_265810_1720215_1072259_hold_eat",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1720215,\n          "object_id": 1072259,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a pizza",\n            "bounds": [\n              [\n                0.41235432028770447,\n                0.17078125476837158,\n                0.9755710959434509,\n                0.9887655973434448\n              ],\n              [\n                0.0903729572892189,\n                0.6570624709129333,\n                0.5252214670181274,\n                0.8350937366485596\n              ],\n              [\n                0.0903729572892189,\n                0.17078125476837158,\n                0.9755710959434509,\n                0.9887655973434448\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.41235432028770447,\n                0.17078125476837158,\n                0.9755710959434509,\n                0.9887655973434448\n              ],\n              [\n                0.0903729572892189,\n                0.6570624709129333,\n                0.5252214670181274,\n                0.8350937366485596\n              ],\n              [\n                0.0903729572892189,\n                0.17078125476837158,\n                0.9755710959434509,\n                0.9887655973434448\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000276265.jpg",\n        "coco_id": 276265,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "banana",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 441306,\n          "object_id": 1045237,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_276265_441306_1045237_carry_eat",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 441306,\n          "object_id": 1045237,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a banana",\n            "bounds": [\n              [\n                0.33128124475479126,\n                0.49322351813316345,\n                0.6115937232971191,\n                1.0\n              ],\n              [\n                0.3265624940395355,\n                0.2988235354423523,\n                0.4375,\n                0.46117648482322693\n              ],\n              [\n                0.3265624940395355,\n                0.2988235354423523,\n                0.6115937232971191,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.33128124475479126,\n                0.49322351813316345,\n                0.6115937232971191,\n                1.0\n              ],\n              [\n                0.3265624940395355,\n                0.2988235354423523,\n                0.4375,\n                0.46117648482322693\n              ],\n              [\n                0.3265624940395355,\n                0.2988235354423523,\n                0.6115937232971191,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000276835.jpg",\n        "coco_id": 276835,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 469395,\n          "object_id": 1436774,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_276835_469395_1436774_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 469395,\n          "object_id": 1436774,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.13043749332427979,\n                0.2022482454776764,\n                0.47828125953674316,\n                0.9865105152130127\n              ],\n              [\n                0.43479686975479126,\n                0.6943793892860413,\n                0.5907187461853027,\n                0.9887588024139404\n              ],\n              [\n                0.13043749332427979,\n                0.2022482454776764,\n                0.5907187461853027,\n                0.9887588024139404\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.13043749332427979,\n                0.2022482454776764,\n                0.47828125953674316,\n                0.9865105152130127\n              ],\n              [\n                0.43479686975479126,\n                0.6943793892860413,\n                0.5907187461853027,\n                0.9887588024139404\n              ],\n              [\n                0.13043749332427979,\n                0.2022482454776764,\n                0.5907187461853027,\n                0.9887588024139404\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000276835.jpg",\n        "coco_id": 276835,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 476736,\n          "object_id": 1436774,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_276835_476736_1436774_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 476736,\n          "object_id": 1436774,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.49809375405311584,\n                0.2623419165611267,\n                0.7926250100135803,\n                0.9954332709312439\n              ],\n              [\n                0.43479686975479126,\n                0.6943793892860413,\n                0.5907187461853027,\n                0.9887588024139404\n              ],\n              [\n                0.43479686975479126,\n                0.2623419165611267,\n                0.7926250100135803,\n                0.9954332709312439\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.49809375405311584,\n                0.2623419165611267,\n                0.7926250100135803,\n                0.9954332709312439\n              ],\n              [\n                0.43479686975479126,\n                0.6943793892860413,\n                0.5907187461853027,\n                0.9887588024139404\n              ],\n              [\n                0.43479686975479126,\n                0.2623419165611267,\n                0.7926250100135803,\n                0.9954332709312439\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000281215.jpg",\n        "coco_id": 281215,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "umbrella",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 479041,\n          "object_id": 280415,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_281215_479041_280415_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 479041,\n          "object_id": 280415,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.10636655986309052,\n                0.37303125858306885,\n                0.9711254239082336,\n                1.0\n              ],\n              [\n                0.013826366513967514,\n                0.0021250001154839993,\n                0.9920257329940796,\n                0.5155781507492065\n              ],\n              [\n                0.013826366513967514,\n                0.0021250001154839993,\n                0.9920257329940796,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.10636655986309052,\n                0.37303125858306885,\n                0.9711254239082336,\n                1.0\n              ],\n              [\n                0.013826366513967514,\n                0.0021250001154839993,\n                0.9920257329940796,\n                0.5155781507492065\n              ],\n              [\n                0.013826366513967514,\n                0.0021250001154839993,\n                0.9920257329940796,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000281563.jpg",\n        "coco_id": 281563,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "banana",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 435260,\n          "object_id": 1042342,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_281563_435260_1042342_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 435260,\n          "object_id": 1042342,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.34474000334739685,\n                0.5818373560905457,\n                0.6355199813842773,\n                0.988162636756897\n              ],\n              [\n                0.5017200112342834,\n                0.11210843175649643,\n                0.6059399843215942,\n                0.3296084403991699\n              ],\n              [\n                0.34474000334739685,\n                0.11210843175649643,\n                0.6355199813842773,\n                0.988162636756897\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a banana",\n            "bounds": [\n              [\n                0.34474000334739685,\n                0.5818373560905457,\n                0.6355199813842773,\n                0.988162636756897\n              ],\n              [\n                0.5017200112342834,\n                0.11210843175649643,\n                0.6059399843215942,\n                0.3296084403991699\n              ],\n              [\n                0.34474000334739685,\n                0.11210843175649643,\n                0.6355199813842773,\n                0.988162636756897\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000289569.jpg",\n        "coco_id": 289569,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1723069,\n          "object_id": 150835,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_289569_1723069_150835_sit_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1723069,\n          "object_id": 150835,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.2942500114440918,\n                0.47292739152908325,\n                0.4144531190395355,\n                0.7010070085525513\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.27464061975479126,\n                0.47292739152908325,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.2942500114440918,\n                0.47292739152908325,\n                0.4144531190395355,\n                0.7010070085525513\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.27464061975479126,\n                0.47292739152908325,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000291262.jpg",\n        "coco_id": 291262,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "boat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 183604,\n          "object_id": 180103,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_291262_183604_180103_hold_ride",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 183604,\n          "object_id": 180103,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a boat",\n            "bounds": [\n              [\n                0.5480208396911621,\n                0.3740312457084656,\n                0.6953333616256714,\n                0.48121875524520874\n              ],\n              [\n                0.5183541774749756,\n                0.175843745470047,\n                0.6831458210945129,\n                0.4927031397819519\n              ],\n              [\n                0.5183541774749756,\n                0.175843745470047,\n                0.6953333616256714,\n                0.4927031397819519\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a boat",\n            "bounds": [\n              [\n                0.5480208396911621,\n                0.3740312457084656,\n                0.6953333616256714,\n                0.48121875524520874\n              ],\n              [\n                0.5183541774749756,\n                0.175843745470047,\n                0.6831458210945129,\n                0.4927031397819519\n              ],\n              [\n                0.5183541774749756,\n                0.175843745470047,\n                0.6953333616256714,\n                0.4927031397819519\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000304295.jpg",\n        "coco_id": 304295,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 427285,\n          "object_id": 1176637,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_304295_427285_1176637_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 427285,\n          "object_id": 1176637,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.5342656373977661,\n                0.5776875019073486,\n                0.6303437352180481,\n                0.9731875061988831\n              ],\n              [\n                0.6075624823570251,\n                0.7027083039283752,\n                0.6256874799728394,\n                0.7722083330154419\n              ],\n              [\n                0.5342656373977661,\n                0.5776875019073486,\n                0.6303437352180481,\n                0.9731875061988831\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.5342656373977661,\n                0.5776875019073486,\n                0.6303437352180481,\n                0.9731875061988831\n              ],\n              [\n                0.6075624823570251,\n                0.7027083039283752,\n                0.6256874799728394,\n                0.7722083330154419\n              ],\n              [\n                0.5342656373977661,\n                0.5776875019073486,\n                0.6303437352180481,\n                0.9731875061988831\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000326229.jpg",\n        "coco_id": 326229,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 450887,\n          "object_id": 324219,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_326229_450887_324219_talk_on_phone_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 450887,\n          "object_id": 324219,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.5319687724113464,\n                0.20266978442668915,\n                1.0,\n                0.9866276383399963\n              ],\n              [\n                0.6688281297683716,\n                0.4741217792034149,\n                0.721875011920929,\n                0.6704215407371521\n              ],\n              [\n                0.5319687724113464,\n                0.20266978442668915,\n                1.0,\n                0.9866276383399963\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.5319687724113464,\n                0.20266978442668915,\n                1.0,\n                0.9866276383399963\n              ],\n              [\n                0.6688281297683716,\n                0.4741217792034149,\n                0.721875011920929,\n                0.6704215407371521\n              ],\n              [\n                0.5319687724113464,\n                0.20266978442668915,\n                1.0,\n                0.9866276383399963\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000329386.jpg",\n        "coco_id": 329386,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "pizza",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1216534,\n          "object_id": 1072051,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_329386_1216534_1072051_eat_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1216534,\n          "object_id": 1072051,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a pizza",\n            "bounds": [\n              [\n                0.05898437649011612,\n                0.10562500357627869,\n                0.6370781064033508,\n                0.8696666955947876\n              ],\n              [\n                0.569656252861023,\n                0.768541693687439,\n                0.8275312781333923,\n                0.9617916941642761\n              ],\n              [\n                0.05898437649011612,\n                0.10562500357627869,\n                0.8275312781333923,\n                0.9617916941642761\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.05898437649011612,\n                0.10562500357627869,\n                0.6370781064033508,\n                0.8696666955947876\n              ],\n              [\n                0.569656252861023,\n                0.768541693687439,\n                0.8275312781333923,\n                0.9617916941642761\n              ],\n              [\n                0.05898437649011612,\n                0.10562500357627869,\n                0.8275312781333923,\n                0.9617916941642761\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000336683.jpg",\n        "coco_id": 336683,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cake",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 466214,\n          "object_id": 1086911,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_336683_466214_1086911_hold_eat",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 466214,\n          "object_id": 1086911,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.502240002155304,\n                0.1258399933576584,\n                0.9185400009155273,\n                0.9865066409111023\n              ],\n              [\n                0.2983199954032898,\n                0.6333333253860474,\n                0.6640400290489197,\n                0.864799976348877\n              ],\n              [\n                0.2983199954032898,\n                0.1258399933576584,\n                0.9185400009155273,\n                0.9865066409111023\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.502240002155304,\n                0.1258399933576584,\n                0.9185400009155273,\n                0.9865066409111023\n              ],\n              [\n                0.2983199954032898,\n                0.6333333253860474,\n                0.6640400290489197,\n                0.864799976348877\n              ],\n              [\n                0.2983199954032898,\n                0.1258399933576584,\n                0.9185400009155273,\n                0.9865066409111023\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000336683.jpg",\n        "coco_id": 336683,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cake",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 472273,\n          "object_id": 1086911,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_336683_472273_1086911_hold_eat",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 472273,\n          "object_id": 1086911,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.021220000460743904,\n                0.012319999746978283,\n                0.5339599847793579,\n                1.0\n              ],\n              [\n                0.2983199954032898,\n                0.6333333253860474,\n                0.6640400290489197,\n                0.864799976348877\n              ],\n              [\n                0.021220000460743904,\n                0.012319999746978283,\n                0.6640400290489197,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.021220000460743904,\n                0.012319999746978283,\n                0.5339599847793579,\n                1.0\n              ],\n              [\n                0.2983199954032898,\n                0.6333333253860474,\n                0.6640400290489197,\n                0.864799976348877\n              ],\n              [\n                0.021220000460743904,\n                0.012319999746978283,\n                0.6640400290489197,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000340282.jpg",\n        "coco_id": 340282,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 188191,\n          "object_id": 1177717,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_340282_188191_1177717_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 188191,\n          "object_id": 1177717,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.5740600228309631,\n                0.5022898316383362,\n                0.7008399963378906,\n                0.9995072484016418\n              ],\n              [\n                0.6092399954795837,\n                0.7051014304161072,\n                0.6720600128173828,\n                0.8539130687713623\n              ],\n              [\n                0.5740600228309631,\n                0.5022898316383362,\n                0.7008399963378906,\n                0.9995072484016418\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.5740600228309631,\n                0.5022898316383362,\n                0.7008399963378906,\n                0.9995072484016418\n              ],\n              [\n                0.6092399954795837,\n                0.7051014304161072,\n                0.6720600128173828,\n                0.8539130687713623\n              ],\n              [\n                0.5740600228309631,\n                0.5022898316383362,\n                0.7008399963378906,\n                0.9995072484016418\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000345879.jpg",\n        "coco_id": 345879,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "boat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 189792,\n          "object_id": 177764,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_345879_189792_177764_ride_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 189792,\n          "object_id": 177764,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a boat",\n            "bounds": [\n              [\n                0.4460468888282776,\n                0.5199764370918274,\n                0.5262500047683716,\n                0.6996933817863464\n              ],\n              [\n                0.15589062869548798,\n                0.5545754432678223,\n                0.8609218597412109,\n                0.7724056839942932\n              ],\n              [\n                0.15589062869548798,\n                0.5199764370918274,\n                0.8609218597412109,\n                0.7724056839942932\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a boat",\n            "bounds": [\n              [\n                0.4460468888282776,\n                0.5199764370918274,\n                0.5262500047683716,\n                0.6996933817863464\n              ],\n              [\n                0.15589062869548798,\n                0.5545754432678223,\n                0.8609218597412109,\n                0.7724056839942932\n              ],\n              [\n                0.15589062869548798,\n                0.5199764370918274,\n                0.8609218597412109,\n                0.7724056839942932\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000036433.jpg",\n        "coco_id": 36433,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 517171,\n          "object_id": 1825595,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_36433_517171_1825595_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 517171,\n          "object_id": 1825595,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.4398750066757202,\n                0.3560781180858612,\n                0.7616249918937683,\n                0.9305468797683716\n              ],\n              [\n                0.3917500078678131,\n                0.3372499942779541,\n                0.5987708568572998,\n                0.5955312252044678\n              ],\n              [\n                0.3917500078678131,\n                0.3372499942779541,\n                0.7616249918937683,\n                0.9305468797683716\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.4398750066757202,\n                0.3560781180858612,\n                0.7616249918937683,\n                0.9305468797683716\n              ],\n              [\n                0.3917500078678131,\n                0.3372499942779541,\n                0.5987708568572998,\n                0.5955312252044678\n              ],\n              [\n                0.3917500078678131,\n                0.3372499942779541,\n                0.7616249918937683,\n                0.9305468797683716\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000368219.jpg",\n        "coco_id": 368219,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cake",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 489798,\n          "object_id": 1085563,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_368219_489798_1085563_hold_eat",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 489798,\n          "object_id": 1085563,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.0,\n                0.0022499999031424522,\n                0.8790866732597351,\n                0.9865156412124634\n              ],\n              [\n                0.3233489394187927,\n                0.6179843544960022,\n                0.8049882650375366,\n                0.7101093530654907\n              ],\n              [\n                0.0,\n                0.0022499999031424522,\n                0.8790866732597351,\n                0.9865156412124634\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.0,\n                0.0022499999031424522,\n                0.8790866732597351,\n                0.9865156412124634\n              ],\n              [\n                0.3233489394187927,\n                0.6179843544960022,\n                0.8049882650375366,\n                0.7101093530654907\n              ],\n              [\n                0.0,\n                0.0022499999031424522,\n                0.8790866732597351,\n                0.9865156412124634\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000368586.jpg",\n        "coco_id": 368586,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 497904,\n          "object_id": 1184525,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_368586_497904_1184525_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 497904,\n          "object_id": 1184525,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.2923593819141388,\n                0.19775176048278809,\n                0.6192031502723694,\n                1.0\n              ],\n              [\n                0.4887656271457672,\n                0.5883840918540955,\n                0.7031718492507935,\n                0.826580822467804\n              ],\n              [\n                0.2923593819141388,\n                0.19775176048278809,\n                0.7031718492507935,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.2923593819141388,\n                0.19775176048278809,\n                0.6192031502723694,\n                1.0\n              ],\n              [\n                0.4887656271457672,\n                0.5883840918540955,\n                0.7031718492507935,\n                0.826580822467804\n              ],\n              [\n                0.2923593819141388,\n                0.19775176048278809,\n                0.7031718492507935,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000372130.jpg",\n        "coco_id": 372130,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1317949,\n          "object_id": 1176699,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_372130_1317949_1176699_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1317949,\n          "object_id": 1176699,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.606374979019165,\n                0.41917645931243896,\n                0.7324844002723694,\n                1.0\n              ],\n              [\n                0.6008750200271606,\n                0.5416235327720642,\n                0.6951562762260437,\n                0.7947294116020203\n              ],\n              [\n                0.6008750200271606,\n                0.41917645931243896,\n                0.7324844002723694,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.606374979019165,\n                0.41917645931243896,\n                0.7324844002723694,\n                1.0\n              ],\n              [\n                0.6008750200271606,\n                0.5416235327720642,\n                0.6951562762260437,\n                0.7947294116020203\n              ],\n              [\n                0.6008750200271606,\n                0.41917645931243896,\n                0.7324844002723694,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000372130.jpg",\n        "coco_id": 372130,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1705207,\n          "object_id": 1432030,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_372130_1705207_1432030_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1705207,\n          "object_id": 1432030,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.7900937795639038,\n                0.5112705826759338,\n                0.8764687776565552,\n                0.9050588011741638\n              ],\n              [\n                0.8699843883514404,\n                0.6270353198051453,\n                0.9122187495231628,\n                0.7659294009208679\n              ],\n              [\n                0.7900937795639038,\n                0.5112705826759338,\n                0.9122187495231628,\n                0.9050588011741638\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.7900937795639038,\n                0.5112705826759338,\n                0.8764687776565552,\n                0.9050588011741638\n              ],\n              [\n                0.8699843883514404,\n                0.6270353198051453,\n                0.9122187495231628,\n                0.7659294009208679\n              ],\n              [\n                0.7900937795639038,\n                0.5112705826759338,\n                0.9122187495231628,\n                0.9050588011741638\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000380440.jpg",\n        "coco_id": 380440,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 470048,\n          "object_id": 1421553,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_380440_470048_1421553_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 470048,\n          "object_id": 1421553,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.3577812612056732,\n                0.11010638624429703,\n                0.5795781016349792,\n                0.8292021155357361\n              ],\n              [\n                0.45093750953674316,\n                0.20125000178813934,\n                0.5364999771118164,\n                0.27553191781044006\n              ],\n              [\n                0.3577812612056732,\n                0.11010638624429703,\n                0.5795781016349792,\n                0.8292021155357361\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.3577812612056732,\n                0.11010638624429703,\n                0.5795781016349792,\n                0.8292021155357361\n              ],\n              [\n                0.45093750953674316,\n                0.20125000178813934,\n                0.5364999771118164,\n                0.27553191781044006\n              ],\n              [\n                0.3577812612056732,\n                0.11010638624429703,\n                0.5795781016349792,\n                0.8292021155357361\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000386880.jpg",\n        "coco_id": 386880,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 209085,\n          "object_id": 326718,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_386880_209085_326718_talk_on_phone_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 209085,\n          "object_id": 326718,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.41871875524520874,\n                0.2264583259820938,\n                0.9282500147819519,\n                0.9865416884422302\n              ],\n              [\n                0.6346874833106995,\n                0.41070833802223206,\n                0.6728281378746033,\n                0.5047916769981384\n              ],\n              [\n                0.41871875524520874,\n                0.2264583259820938,\n                0.9282500147819519,\n                0.9865416884422302\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.41871875524520874,\n                0.2264583259820938,\n                0.9282500147819519,\n                0.9865416884422302\n              ],\n              [\n                0.6346874833106995,\n                0.41070833802223206,\n                0.6728281378746033,\n                0.5047916769981384\n              ],\n              [\n                0.41871875524520874,\n                0.2264583259820938,\n                0.9282500147819519,\n                0.9865416884422302\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000395198.jpg",\n        "coco_id": 395198,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 226730,\n          "object_id": 1416190,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_395198_226730_1416190_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 226730,\n          "object_id": 1416190,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.5013281106948853,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ],\n              [\n                0.5135937333106995,\n                0.36056336760520935,\n                0.596609354019165,\n                0.5369952917098999\n              ],\n              [\n                0.5013281106948853,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.5013281106948853,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ],\n              [\n                0.5135937333106995,\n                0.36056336760520935,\n                0.596609354019165,\n                0.5369952917098999\n              ],\n              [\n                0.5013281106948853,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000395241.jpg",\n        "coco_id": 395241,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 508445,\n          "object_id": 2230268,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_395241_508445_2230268_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 508445,\n          "object_id": 2230268,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.6847187280654907,\n                0.25702500343322754,\n                0.7991406321525574,\n                0.9675750136375427\n              ],\n              [\n                0.708078145980835,\n                0.5552250146865845,\n                0.8184999823570251,\n                0.6649500131607056\n              ],\n              [\n                0.6847187280654907,\n                0.25702500343322754,\n                0.8184999823570251,\n                0.9675750136375427\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.6847187280654907,\n                0.25702500343322754,\n                0.7991406321525574,\n                0.9675750136375427\n              ],\n              [\n                0.708078145980835,\n                0.5552250146865845,\n                0.8184999823570251,\n                0.6649500131607056\n              ],\n              [\n                0.6847187280654907,\n                0.25702500343322754,\n                0.8184999823570251,\n                0.9675750136375427\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000426500.jpg",\n        "coco_id": 426500,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 515141,\n          "object_id": 1167919,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_426500_515141_1167919_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 515141,\n          "object_id": 1167919,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.29710936546325684,\n                0.49140143394470215,\n                0.39115625619888306,\n                0.8634917140007019\n              ],\n              [\n                0.33079686760902405,\n                0.6052019000053406,\n                0.37107813358306885,\n                0.6686936020851135\n              ],\n              [\n                0.29710936546325684,\n                0.49140143394470215,\n                0.39115625619888306,\n                0.8634917140007019\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.29710936546325684,\n                0.49140143394470215,\n                0.39115625619888306,\n                0.8634917140007019\n              ],\n              [\n                0.33079686760902405,\n                0.6052019000053406,\n                0.37107813358306885,\n                0.6686936020851135\n              ],\n              [\n                0.29710936546325684,\n                0.49140143394470215,\n                0.39115625619888306,\n                0.8634917140007019\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000433232.jpg",\n        "coco_id": 433232,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "truck",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1265466,\n          "object_id": 2051110,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_433232_1265466_2051110_ride_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1265466,\n          "object_id": 2051110,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a truck",\n            "bounds": [\n              [\n                0.37627172470092773,\n                0.14359374344348907,\n                0.44110584259033203,\n                0.2677343785762787\n              ],\n              [\n                0.05135861039161682,\n                0.21587499976158142,\n                0.9147709608078003,\n                0.9365156292915344\n              ],\n              [\n                0.05135861039161682,\n                0.14359374344348907,\n                0.9147709608078003,\n                0.9365156292915344\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a truck",\n            "bounds": [\n              [\n                0.37627172470092773,\n                0.14359374344348907,\n                0.44110584259033203,\n                0.2677343785762787\n              ],\n              [\n                0.05135861039161682,\n                0.21587499976158142,\n                0.9147709608078003,\n                0.9365156292915344\n              ],\n              [\n                0.05135861039161682,\n                0.14359374344348907,\n                0.9147709608078003,\n                0.9365156292915344\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000445964.jpg",\n        "coco_id": 445964,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "banana",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 453604,\n          "object_id": 1042544,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_445964_453604_1042544_carry_eat",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 453604,\n          "object_id": 1042544,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a banana",\n            "bounds": [\n              [\n                0.30023810267448425,\n                0.41410937905311584,\n                0.4567687213420868,\n                0.8722344040870667\n              ],\n              [\n                0.25926870107650757,\n                0.30562499165534973,\n                0.4891836643218994,\n                0.4247187376022339\n              ],\n              [\n                0.25926870107650757,\n                0.30562499165534973,\n                0.4891836643218994,\n                0.8722344040870667\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.30023810267448425,\n                0.41410937905311584,\n                0.4567687213420868,\n                0.8722344040870667\n              ],\n              [\n                0.25926870107650757,\n                0.30562499165534973,\n                0.4891836643218994,\n                0.4247187376022339\n              ],\n              [\n                0.25926870107650757,\n                0.30562499165534973,\n                0.4891836643218994,\n                0.8722344040870667\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000445964.jpg",\n        "coco_id": 445964,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "banana",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 453604,\n          "object_id": 1042544,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_445964_453604_1042544_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 453604,\n          "object_id": 1042544,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a banana",\n            "bounds": [\n              [\n                0.30023810267448425,\n                0.41410937905311584,\n                0.4567687213420868,\n                0.8722344040870667\n              ],\n              [\n                0.25926870107650757,\n                0.30562499165534973,\n                0.4891836643218994,\n                0.4247187376022339\n              ],\n              [\n                0.25926870107650757,\n                0.30562499165534973,\n                0.4891836643218994,\n                0.8722344040870667\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.30023810267448425,\n                0.41410937905311584,\n                0.4567687213420868,\n                0.8722344040870667\n              ],\n              [\n                0.25926870107650757,\n                0.30562499165534973,\n                0.4891836643218994,\n                0.4247187376022339\n              ],\n              [\n                0.25926870107650757,\n                0.30562499165534973,\n                0.4891836643218994,\n                0.8722344040870667\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000446956.jpg",\n        "coco_id": 446956,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1252373,\n          "object_id": 1179873,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_446956_1252373_1179873_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1252373,\n          "object_id": 1179873,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.6379374861717224,\n                0.7239791750907898,\n                0.7214687466621399,\n                0.9511874914169312\n              ],\n              [\n                0.6375156044960022,\n                0.7778541445732117,\n                0.6646562218666077,\n                0.8242708444595337\n              ],\n              [\n                0.6375156044960022,\n                0.7239791750907898,\n                0.7214687466621399,\n                0.9511874914169312\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.6379374861717224,\n                0.7239791750907898,\n                0.7214687466621399,\n                0.9511874914169312\n              ],\n              [\n                0.6375156044960022,\n                0.7778541445732117,\n                0.6646562218666077,\n                0.8242708444595337\n              ],\n              [\n                0.6375156044960022,\n                0.7239791750907898,\n                0.7214687466621399,\n                0.9511874914169312\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000045084.jpg",\n        "coco_id": 45084,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2159991,\n          "object_id": 1842424,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_45084_2159991_1842424_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 2159991,\n          "object_id": 1842424,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.09493750333786011,\n                0.03696335107088089,\n                0.32326561212539673,\n                0.6952181458473206\n              ],\n              [\n                0.0008750000270083547,\n                0.4463699758052826,\n                0.08742187172174454,\n                0.6504537463188171\n              ],\n              [\n                0.0008750000270083547,\n                0.03696335107088089,\n                0.32326561212539673,\n                0.6952181458473206\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.09493750333786011,\n                0.03696335107088089,\n                0.32326561212539673,\n                0.6952181458473206\n              ],\n              [\n                0.0008750000270083547,\n                0.4463699758052826,\n                0.08742187172174454,\n                0.6504537463188171\n              ],\n              [\n                0.0008750000270083547,\n                0.03696335107088089,\n                0.32326561212539673,\n                0.6952181458473206\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000456302.jpg",\n        "coco_id": 456302,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "banana",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2165028,\n          "object_id": 1041578,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_456302_2165028_1041578_eat_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 2165028,\n          "object_id": 1041578,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.19807812571525574,\n                0.08332552760839462,\n                0.6637187600135803,\n                0.9817798733711243\n              ],\n              [\n                0.5154218673706055,\n                0.5123887658119202,\n                0.7092656493186951,\n                0.8975175619125366\n              ],\n              [\n                0.19807812571525574,\n                0.08332552760839462,\n                0.7092656493186951,\n                0.9817798733711243\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a banana",\n            "bounds": [\n              [\n                0.19807812571525574,\n                0.08332552760839462,\n                0.6637187600135803,\n                0.9817798733711243\n              ],\n              [\n                0.5154218673706055,\n                0.5123887658119202,\n                0.7092656493186951,\n                0.8975175619125366\n              ],\n              [\n                0.19807812571525574,\n                0.08332552760839462,\n                0.7092656493186951,\n                0.9817798733711243\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000456302.jpg",\n        "coco_id": 456302,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "banana",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2165028,\n          "object_id": 1041578,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_456302_2165028_1041578_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 2165028,\n          "object_id": 1041578,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a banana",\n            "bounds": [\n              [\n                0.19807812571525574,\n                0.08332552760839462,\n                0.6637187600135803,\n                0.9817798733711243\n              ],\n              [\n                0.5154218673706055,\n                0.5123887658119202,\n                0.7092656493186951,\n                0.8975175619125366\n              ],\n              [\n                0.19807812571525574,\n                0.08332552760839462,\n                0.7092656493186951,\n                0.9817798733711243\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.19807812571525574,\n                0.08332552760839462,\n                0.6637187600135803,\n                0.9817798733711243\n              ],\n              [\n                0.5154218673706055,\n                0.5123887658119202,\n                0.7092656493186951,\n                0.8975175619125366\n              ],\n              [\n                0.19807812571525574,\n                0.08332552760839462,\n                0.7092656493186951,\n                0.9817798733711243\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000461870.jpg",\n        "coco_id": 461870,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "cake",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 184952,\n          "object_id": 1087980,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_461870_184952_1087980_eat_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 184952,\n          "object_id": 1087980,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.0,\n                0.0044999998062849045,\n                1.0,\n                0.6988750100135803\n              ],\n              [\n                0.5044843554496765,\n                0.7264583110809326,\n                0.8593124747276306,\n                0.9731041789054871\n              ],\n              [\n                0.0,\n                0.0044999998062849045,\n                1.0,\n                0.9731041789054871\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.0,\n                0.0044999998062849045,\n                1.0,\n                0.6988750100135803\n              ],\n              [\n                0.5044843554496765,\n                0.7264583110809326,\n                0.8593124747276306,\n                0.9731041789054871\n              ],\n              [\n                0.0,\n                0.0044999998062849045,\n                1.0,\n                0.9731041789054871\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000462381.jpg",\n        "coco_id": 462381,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1225442,\n          "object_id": 2212429,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_462381_1225442_2212429_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1225442,\n          "object_id": 2212429,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.25603407621383667,\n                0.17679686844348907,\n                0.7750851511955261,\n                0.965093731880188\n              ],\n              [\n                0.2614598572254181,\n                0.4864218831062317,\n                0.772822380065918,\n                0.9012343883514404\n              ],\n              [\n                0.25603407621383667,\n                0.17679686844348907,\n                0.7750851511955261,\n                0.965093731880188\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.25603407621383667,\n                0.17679686844348907,\n                0.7750851511955261,\n                0.965093731880188\n              ],\n              [\n                0.2614598572254181,\n                0.4864218831062317,\n                0.772822380065918,\n                0.9012343883514404\n              ],\n              [\n                0.25603407621383667,\n                0.17679686844348907,\n                0.7750851511955261,\n                0.965093731880188\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000046441.jpg",\n        "coco_id": 46441,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "sandwich",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 434992,\n          "object_id": 2218486,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_46441_434992_2218486_eat_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 434992,\n          "object_id": 2218486,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a sandwich",\n            "bounds": [\n              [\n                0.10654321312904358,\n                0.14650000631809235,\n                0.7256378531455994,\n                0.7078593969345093\n              ],\n              [\n                0.46481481194496155,\n                0.6624062657356262,\n                0.5821399092674255,\n                0.7763749957084656\n              ],\n              [\n                0.10654321312904358,\n                0.14650000631809235,\n                0.7256378531455994,\n                0.7763749957084656\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a sandwich",\n            "bounds": [\n              [\n                0.10654321312904358,\n                0.14650000631809235,\n                0.7256378531455994,\n                0.7078593969345093\n              ],\n              [\n                0.46481481194496155,\n                0.6624062657356262,\n                0.5821399092674255,\n                0.7763749957084656\n              ],\n              [\n                0.10654321312904358,\n                0.14650000631809235,\n                0.7256378531455994,\n                0.7763749957084656\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000465529.jpg",\n        "coco_id": 465529,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 437303,\n          "object_id": 322681,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_465529_437303_322681_hold_talk_on_phone",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 437303,\n          "object_id": 322681,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.4582187533378601,\n                0.139133483171463,\n                0.850781261920929,\n                0.9042154550552368\n              ],\n              [\n                0.4522656202316284,\n                0.5160890221595764,\n                0.5123593807220459,\n                0.6709133386611938\n              ],\n              [\n                0.4522656202316284,\n                0.139133483171463,\n                0.850781261920929,\n                0.9042154550552368\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.4582187533378601,\n                0.139133483171463,\n                0.850781261920929,\n                0.9042154550552368\n              ],\n              [\n                0.4522656202316284,\n                0.5160890221595764,\n                0.5123593807220459,\n                0.6709133386611938\n              ],\n              [\n                0.4522656202316284,\n                0.139133483171463,\n                0.850781261920929,\n                0.9042154550552368\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000499865.jpg",\n        "coco_id": 499865,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 234005,\n          "object_id": 1180432,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_499865_234005_1180432_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 234005,\n          "object_id": 1180432,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.29183998703956604,\n                0.1528228223323822,\n                0.4624600112438202,\n                0.9056156277656555\n              ],\n              [\n                0.30441999435424805,\n                0.37378379702568054,\n                0.4136599898338318,\n                0.5737837553024292\n              ],\n              [\n                0.29183998703956604,\n                0.1528228223323822,\n                0.4624600112438202,\n                0.9056156277656555\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.29183998703956604,\n                0.1528228223323822,\n                0.4624600112438202,\n                0.9056156277656555\n              ],\n              [\n                0.30441999435424805,\n                0.37378379702568054,\n                0.4136599898338318,\n                0.5737837553024292\n              ],\n              [\n                0.29183998703956604,\n                0.1528228223323822,\n                0.4624600112438202,\n                0.9056156277656555\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000504494.jpg",\n        "coco_id": 504494,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1720416,\n          "object_id": 2167975,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_504494_1720416_2167975_ride_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1720416,\n          "object_id": 2167975,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.02590624988079071,\n                0.30246537923812866,\n                0.14057812094688416,\n                0.749279797077179\n              ],\n              [\n                0.04807812348008156,\n                0.4166482090950012,\n                0.16679687798023224,\n                0.7675623297691345\n              ],\n              [\n                0.02590624988079071,\n                0.30246537923812866,\n                0.16679687798023224,\n                0.7675623297691345\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.02590624988079071,\n                0.30246537923812866,\n                0.14057812094688416,\n                0.749279797077179\n              ],\n              [\n                0.04807812348008156,\n                0.4166482090950012,\n                0.16679687798023224,\n                0.7675623297691345\n              ],\n              [\n                0.02590624988079071,\n                0.30246537923812866,\n                0.16679687798023224,\n                0.7675623297691345\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000504494.jpg",\n        "coco_id": 504494,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1720416,\n          "object_id": 2167975,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_504494_1720416_2167975_sit_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1720416,\n          "object_id": 2167975,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.02590624988079071,\n                0.30246537923812866,\n                0.14057812094688416,\n                0.749279797077179\n              ],\n              [\n                0.04807812348008156,\n                0.4166482090950012,\n                0.16679687798023224,\n                0.7675623297691345\n              ],\n              [\n                0.02590624988079071,\n                0.30246537923812866,\n                0.16679687798023224,\n                0.7675623297691345\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.02590624988079071,\n                0.30246537923812866,\n                0.14057812094688416,\n                0.749279797077179\n              ],\n              [\n                0.04807812348008156,\n                0.4166482090950012,\n                0.16679687798023224,\n                0.7675623297691345\n              ],\n              [\n                0.02590624988079071,\n                0.30246537923812866,\n                0.16679687798023224,\n                0.7675623297691345\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000515123.jpg",\n        "coco_id": 515123,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1730874,\n          "object_id": 1186172,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_515123_1730874_1186172_carry_sit",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1730874,\n          "object_id": 1186172,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.3658199906349182,\n                0.3020800054073334,\n                0.5418000221252441,\n                0.9394400119781494\n              ],\n              [\n                0.5176799893379211,\n                0.6362400054931641,\n                0.667739987373352,\n                0.9719200134277344\n              ],\n              [\n                0.3658199906349182,\n                0.3020800054073334,\n                0.667739987373352,\n                0.9719200134277344\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.3658199906349182,\n                0.3020800054073334,\n                0.5418000221252441,\n                0.9394400119781494\n              ],\n              [\n                0.5176799893379211,\n                0.6362400054931641,\n                0.667739987373352,\n                0.9719200134277344\n              ],\n              [\n                0.3658199906349182,\n                0.3020800054073334,\n                0.667739987373352,\n                0.9719200134277344\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000051899.jpg",\n        "coco_id": 51899,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 496658,\n          "object_id": 1177051,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_51899_496658_1177051_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 496658,\n          "object_id": 1177051,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.380921870470047,\n                0.4012500047683716,\n                0.5248281359672546,\n                0.9633125066757202\n              ],\n              [\n                0.41042187809944153,\n                0.47995832562446594,\n                0.46403124928474426,\n                0.6746458411216736\n              ],\n              [\n                0.380921870470047,\n                0.4012500047683716,\n                0.5248281359672546,\n                0.9633125066757202\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.380921870470047,\n                0.4012500047683716,\n                0.5248281359672546,\n                0.9633125066757202\n              ],\n              [\n                0.41042187809944153,\n                0.47995832562446594,\n                0.46403124928474426,\n                0.6746458411216736\n              ],\n              [\n                0.380921870470047,\n                0.4012500047683716,\n                0.5248281359672546,\n                0.9633125066757202\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000522661.jpg",\n        "coco_id": 522661,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 198829,\n          "object_id": 1843103,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_522661_198829_1843103_hold_sit",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 198829,\n          "object_id": 1843103,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.8724218606948853,\n                0.5874166488647461,\n                0.996343731880188,\n                0.9161249995231628\n              ],\n              [\n                0.9652968645095825,\n                0.7327916622161865,\n                0.9999687671661377,\n                0.8645625114440918\n              ],\n              [\n                0.8724218606948853,\n                0.5874166488647461,\n                0.9999687671661377,\n                0.9161249995231628\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.8724218606948853,\n                0.5874166488647461,\n                0.996343731880188,\n                0.9161249995231628\n              ],\n              [\n                0.9652968645095825,\n                0.7327916622161865,\n                0.9999687671661377,\n                0.8645625114440918\n              ],\n              [\n                0.8724218606948853,\n                0.5874166488647461,\n                0.9999687671661377,\n                0.9161249995231628\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000530896.jpg",\n        "coco_id": 530896,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 224036,\n          "object_id": 130392,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_530896_224036_130392_hold_ride",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 224036,\n          "object_id": 130392,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.621218740940094,\n                0.18027397990226746,\n                0.7110781073570251,\n                0.49803653359413147\n              ],\n              [\n                0.561718761920929,\n                0.3054109513759613,\n                0.644656240940094,\n                0.5509132146835327\n              ],\n              [\n                0.561718761920929,\n                0.18027397990226746,\n                0.7110781073570251,\n                0.5509132146835327\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.621218740940094,\n                0.18027397990226746,\n                0.7110781073570251,\n                0.49803653359413147\n              ],\n              [\n                0.561718761920929,\n                0.3054109513759613,\n                0.644656240940094,\n                0.5509132146835327\n              ],\n              [\n                0.561718761920929,\n                0.18027397990226746,\n                0.7110781073570251,\n                0.5509132146835327\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000530896.jpg",\n        "coco_id": 530896,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 224036,\n          "object_id": 130392,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_530896_224036_130392_hold_sit",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 224036,\n          "object_id": 130392,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.621218740940094,\n                0.18027397990226746,\n                0.7110781073570251,\n                0.49803653359413147\n              ],\n              [\n                0.561718761920929,\n                0.3054109513759613,\n                0.644656240940094,\n                0.5509132146835327\n              ],\n              [\n                0.561718761920929,\n                0.18027397990226746,\n                0.7110781073570251,\n                0.5509132146835327\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.621218740940094,\n                0.18027397990226746,\n                0.7110781073570251,\n                0.49803653359413147\n              ],\n              [\n                0.561718761920929,\n                0.3054109513759613,\n                0.644656240940094,\n                0.5509132146835327\n              ],\n              [\n                0.561718761920929,\n                0.18027397990226746,\n                0.7110781073570251,\n                0.5509132146835327\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000053939.jpg",\n        "coco_id": 53939,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1268968,\n          "object_id": 1415443,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_53939_1268968_1415443_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1268968,\n          "object_id": 1415443,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.19740624725818634,\n                0.2776041626930237,\n                0.3386250138282776,\n                0.6082291603088379\n              ],\n              [\n                0.23045311868190765,\n                0.3217083215713501,\n                0.3200937509536743,\n                0.4310624897480011\n              ],\n              [\n                0.19740624725818634,\n                0.2776041626930237,\n                0.3386250138282776,\n                0.6082291603088379\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.19740624725818634,\n                0.2776041626930237,\n                0.3386250138282776,\n                0.6082291603088379\n              ],\n              [\n                0.23045311868190765,\n                0.3217083215713501,\n                0.3200937509536743,\n                0.4310624897480011\n              ],\n              [\n                0.19740624725818634,\n                0.2776041626930237,\n                0.3386250138282776,\n                0.6082291603088379\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000543521.jpg",\n        "coco_id": 543521,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "sandwich",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 481708,\n          "object_id": 311612,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_543521_481708_311612_hold_eat",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 481708,\n          "object_id": 311612,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a sandwich",\n            "bounds": [\n              [\n                0.14096875488758087,\n                0.049929577857255936,\n                0.804812490940094,\n                1.0\n              ],\n              [\n                0.18892186880111694,\n                0.5870891809463501,\n                0.40607813000679016,\n                0.9876291155815125\n              ],\n              [\n                0.14096875488758087,\n                0.049929577857255936,\n                0.804812490940094,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a sandwich",\n            "bounds": [\n              [\n                0.14096875488758087,\n                0.049929577857255936,\n                0.804812490940094,\n                1.0\n              ],\n              [\n                0.18892186880111694,\n                0.5870891809463501,\n                0.40607813000679016,\n                0.9876291155815125\n              ],\n              [\n                0.14096875488758087,\n                0.049929577857255936,\n                0.804812490940094,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000556838.jpg",\n        "coco_id": 556838,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 506928,\n          "object_id": 2138264,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_556838_506928_2138264_carry_talk_on_phone",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 506928,\n          "object_id": 2138264,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.016917647793889046,\n                0.28053125739097595,\n                0.3824000060558319,\n                0.9883906245231628\n              ],\n              [\n                0.17489412426948547,\n                0.4039374887943268,\n                0.1965411752462387,\n                0.42625001072883606\n              ],\n              [\n                0.016917647793889046,\n                0.28053125739097595,\n                0.3824000060558319,\n                0.9883906245231628\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.016917647793889046,\n                0.28053125739097595,\n                0.3824000060558319,\n                0.9883906245231628\n              ],\n              [\n                0.17489412426948547,\n                0.4039374887943268,\n                0.1965411752462387,\n                0.42625001072883606\n              ],\n              [\n                0.016917647793889046,\n                0.28053125739097595,\n                0.3824000060558319,\n                0.9883906245231628\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000564349.jpg",\n        "coco_id": 564349,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 444940,\n          "object_id": 1183087,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_564349_444940_1183087_sit_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 444940,\n          "object_id": 1183087,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.3109859228134155,\n                0.5126093626022339,\n                0.7843661904335022,\n                0.8791875243186951\n              ],\n              [\n                0.37258216738700867,\n                0.6309999823570251,\n                0.6897417902946472,\n                0.9251875281333923\n              ],\n              [\n                0.3109859228134155,\n                0.5126093626022339,\n                0.7843661904335022,\n                0.9251875281333923\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.3109859228134155,\n                0.5126093626022339,\n                0.7843661904335022,\n                0.8791875243186951\n              ],\n              [\n                0.37258216738700867,\n                0.6309999823570251,\n                0.6897417902946472,\n                0.9251875281333923\n              ],\n              [\n                0.3109859228134155,\n                0.5126093626022339,\n                0.7843661904335022,\n                0.9251875281333923\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000579729.jpg",\n        "coco_id": 579729,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 196525,\n          "object_id": 152520,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_579729_196525_152520_hold_sit",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 196525,\n          "object_id": 152520,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.4985312521457672,\n                0.1612880527973175,\n                0.6567343473434448,\n                0.7228103280067444\n              ],\n              [\n                0.2413906306028366,\n                0.4241686165332794,\n                0.7476562261581421,\n                0.8211475610733032\n              ],\n              [\n                0.2413906306028366,\n                0.1612880527973175,\n                0.7476562261581421,\n                0.8211475610733032\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.4985312521457672,\n                0.1612880527973175,\n                0.6567343473434448,\n                0.7228103280067444\n              ],\n              [\n                0.2413906306028366,\n                0.4241686165332794,\n                0.7476562261581421,\n                0.8211475610733032\n              ],\n              [\n                0.2413906306028366,\n                0.1612880527973175,\n                0.7476562261581421,\n                0.8211475610733032\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000579729.jpg",\n        "coco_id": 579729,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 196525,\n          "object_id": 152520,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_579729_196525_152520_ride_sit",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 196525,\n          "object_id": 152520,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.4985312521457672,\n                0.1612880527973175,\n                0.6567343473434448,\n                0.7228103280067444\n              ],\n              [\n                0.2413906306028366,\n                0.4241686165332794,\n                0.7476562261581421,\n                0.8211475610733032\n              ],\n              [\n                0.2413906306028366,\n                0.1612880527973175,\n                0.7476562261581421,\n                0.8211475610733032\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.4985312521457672,\n                0.1612880527973175,\n                0.6567343473434448,\n                0.7228103280067444\n              ],\n              [\n                0.2413906306028366,\n                0.4241686165332794,\n                0.7476562261581421,\n                0.8211475610733032\n              ],\n              [\n                0.2413906306028366,\n                0.1612880527973175,\n                0.7476562261581421,\n                0.8211475610733032\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000006253.jpg",\n        "coco_id": 6253,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1709656,\n          "object_id": 1187940,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_6253_1709656_1187940_hold_sit",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1709656,\n          "object_id": 1187940,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.7165156006813049,\n                0.12964583933353424,\n                1.0,\n                0.9984999895095825\n              ],\n              [\n                0.6064218878746033,\n                0.604729175567627,\n                0.7871562242507935,\n                0.9831041693687439\n              ],\n              [\n                0.6064218878746033,\n                0.12964583933353424,\n                1.0,\n                0.9984999895095825\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.7165156006813049,\n                0.12964583933353424,\n                1.0,\n                0.9984999895095825\n              ],\n              [\n                0.6064218878746033,\n                0.604729175567627,\n                0.7871562242507935,\n                0.9831041693687439\n              ],\n              [\n                0.6064218878746033,\n                0.12964583933353424,\n                1.0,\n                0.9984999895095825\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000072646.jpg",\n        "coco_id": 72646,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "cake",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 442458,\n          "object_id": 1084677,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_72646_442458_1084677_eat_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 442458,\n          "object_id": 1084677,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.32984375953674316,\n                0.30074942111968994,\n                0.7346562743186951,\n                0.7748945951461792\n              ],\n              [\n                0.3013593852519989,\n                0.6853864192962646,\n                0.5142655968666077,\n                0.898875892162323\n              ],\n              [\n                0.3013593852519989,\n                0.30074942111968994,\n                0.7346562743186951,\n                0.898875892162323\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.32984375953674316,\n                0.30074942111968994,\n                0.7346562743186951,\n                0.7748945951461792\n              ],\n              [\n                0.3013593852519989,\n                0.6853864192962646,\n                0.5142655968666077,\n                0.898875892162323\n              ],\n              [\n                0.3013593852519989,\n                0.30074942111968994,\n                0.7346562743186951,\n                0.898875892162323\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000073172.jpg",\n        "coco_id": 73172,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 439461,\n          "object_id": 323433,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_73172_439461_323433_carry_talk_on_phone",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 439461,\n          "object_id": 323433,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.3070468604564667,\n                0.18531645834445953,\n                0.5059999823570251,\n                0.8852742910385132\n              ],\n              [\n                0.4742499887943268,\n                0.3550421893596649,\n                0.5002968907356262,\n                0.3951265811920166\n              ],\n              [\n                0.3070468604564667,\n                0.18531645834445953,\n                0.5059999823570251,\n                0.8852742910385132\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.3070468604564667,\n                0.18531645834445953,\n                0.5059999823570251,\n                0.8852742910385132\n              ],\n              [\n                0.4742499887943268,\n                0.3550421893596649,\n                0.5002968907356262,\n                0.3951265811920166\n              ],\n              [\n                0.3070468604564667,\n                0.18531645834445953,\n                0.5059999823570251,\n                0.8852742910385132\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000074650.jpg",\n        "coco_id": 74650,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cake",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 478267,\n          "object_id": 1089382,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_74650_478267_1089382_hold_eat",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 478267,\n          "object_id": 1089382,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.21839062869548798,\n                0.0022535212337970734,\n                0.8585781455039978,\n                0.984272301197052\n              ],\n              [\n                0.5579218864440918,\n                0.3820187747478485,\n                0.9034531116485596,\n                0.6516901254653931\n              ],\n              [\n                0.21839062869548798,\n                0.0022535212337970734,\n                0.9034531116485596,\n                0.984272301197052\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.21839062869548798,\n                0.0022535212337970734,\n                0.8585781455039978,\n                0.984272301197052\n              ],\n              [\n                0.5579218864440918,\n                0.3820187747478485,\n                0.9034531116485596,\n                0.6516901254653931\n              ],\n              [\n                0.21839062869548798,\n                0.0022535212337970734,\n                0.9034531116485596,\n                0.984272301197052\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000080553.jpg",\n        "coco_id": 80553,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 452775,\n          "object_id": 1841790,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_80553_452775_1841790_sit_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 452775,\n          "object_id": 1841790,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.300656259059906,\n                0.40772300958633423,\n                0.42210936546325684,\n                0.8224647641181946\n              ],\n              [\n                0.349609375,\n                0.6357980966567993,\n                0.4331406354904175,\n                0.8246009349822998\n              ],\n              [\n                0.300656259059906,\n                0.40772300958633423,\n                0.4331406354904175,\n                0.8246009349822998\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.300656259059906,\n                0.40772300958633423,\n                0.42210936546325684,\n                0.8224647641181946\n              ],\n              [\n                0.349609375,\n                0.6357980966567993,\n                0.4331406354904175,\n                0.8246009349822998\n              ],\n              [\n                0.300656259059906,\n                0.40772300958633423,\n                0.4331406354904175,\n                0.8246009349822998\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000080553.jpg",\n        "coco_id": 80553,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 452775,\n          "object_id": 1841790,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_80553_452775_1841790_sit_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 452775,\n          "object_id": 1841790,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.300656259059906,\n                0.40772300958633423,\n                0.42210936546325684,\n                0.8224647641181946\n              ],\n              [\n                0.349609375,\n                0.6357980966567993,\n                0.4331406354904175,\n                0.8246009349822998\n              ],\n              [\n                0.300656259059906,\n                0.40772300958633423,\n                0.4331406354904175,\n                0.8246009349822998\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.300656259059906,\n                0.40772300958633423,\n                0.42210936546325684,\n                0.8224647641181946\n              ],\n              [\n                0.349609375,\n                0.6357980966567993,\n                0.4331406354904175,\n                0.8246009349822998\n              ],\n              [\n                0.300656259059906,\n                0.40772300958633423,\n                0.4331406354904175,\n                0.8246009349822998\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000094681.jpg",\n        "coco_id": 94681,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 217133,\n          "object_id": 150642,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_94681_217133_150642_sit_ride",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 217133,\n          "object_id": 150642,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.06881249696016312,\n                0.38875511288642883,\n                0.2597968876361847,\n                0.7617959380149841\n              ],\n              [\n                0.018906250596046448,\n                0.42963266372680664,\n                0.6412187218666077,\n                0.937816321849823\n              ],\n              [\n                0.018906250596046448,\n                0.38875511288642883,\n                0.6412187218666077,\n                0.937816321849823\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.06881249696016312,\n                0.38875511288642883,\n                0.2597968876361847,\n                0.7617959380149841\n              ],\n              [\n                0.018906250596046448,\n                0.42963266372680664,\n                0.6412187218666077,\n                0.937816321849823\n              ],\n              [\n                0.018906250596046448,\n                0.38875511288642883,\n                0.6412187218666077,\n                0.937816321849823\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000009669.jpg",\n        "coco_id": 9669,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1700706,\n          "object_id": 321902,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_9669_1700706_321902_talk_on_phone_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1700706,\n          "object_id": 321902,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.17647196352481842,\n                0.7288437485694885,\n                0.6961682438850403,\n                1.0\n              ],\n              [\n                0.48649531602859497,\n                0.5219374895095825,\n                0.6251401901245117,\n                0.5786093473434448\n              ],\n              [\n                0.17647196352481842,\n                0.5219374895095825,\n                0.6961682438850403,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.17647196352481842,\n                0.7288437485694885,\n                0.6961682438850403,\n                1.0\n              ],\n              [\n                0.48649531602859497,\n                0.5219374895095825,\n                0.6251401901245117,\n                0.5786093473434448\n              ],\n              [\n                0.17647196352481842,\n                0.5219374895095825,\n                0.6961682438850403,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000101575.jpg",\n        "coco_id": 101575,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1309881,\n          "object_id": 2177970,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_101575_1309881_2177970_1326747_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1326747,\n          "object_id": 2177970,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.2461874932050705,\n                0.16346603631973267,\n                0.36589062213897705,\n                0.7505620718002319\n              ],\n              [\n                0.5087656378746033,\n                0.658501148223877,\n                0.5962812304496765,\n                0.6958313584327698\n              ],\n              [\n                0.2461874932050705,\n                0.16346603631973267,\n                0.5962812304496765,\n                0.7505620718002319\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.377015620470047,\n                0.45913347601890564,\n                0.5466562509536743,\n                0.8488993048667908\n              ],\n              [\n                0.5087656378746033,\n                0.658501148223877,\n                0.5962812304496765,\n                0.6958313584327698\n              ],\n              [\n                0.377015620470047,\n                0.45913347601890564,\n                0.5962812304496765,\n                0.8488993048667908\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000010265.jpg",\n        "coco_id": 10265,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 493301,\n          "object_id": 657811,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_10265_493301_657811_495760_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 495760,\n          "object_id": 657811,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.193359375,\n                0.10140515118837357,\n                0.31850001215934753,\n                0.3650819659233093\n              ],\n              [\n                0.40754687786102295,\n                0.6367681622505188,\n                0.4945937395095825,\n                0.7002341747283936\n              ],\n              [\n                0.193359375,\n                0.10140515118837357,\n                0.4945937395095825,\n                0.7002341747283936\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.4684687554836273,\n                0.47163933515548706,\n                0.7081249952316284,\n                0.7380561828613281\n              ],\n              [\n                0.40754687786102295,\n                0.6367681622505188,\n                0.4945937395095825,\n                0.7002341747283936\n              ],\n              [\n                0.40754687786102295,\n                0.47163933515548706,\n                0.7081249952316284,\n                0.7380561828613281\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000103437.jpg",\n        "coco_id": 103437,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "fork",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 188754,\n          "object_id": 685874,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_103437_188754_685874_2159205_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2159205,\n          "object_id": 685874,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.0034687500447034836,\n                0.012051282450556755,\n                0.33256250619888306,\n                0.6683915853500366\n              ],\n              [\n                0.5365781188011169,\n                0.46937063336372375,\n                0.7937187552452087,\n                0.701025664806366\n              ],\n              [\n                0.0034687500447034836,\n                0.012051282450556755,\n                0.7937187552452087,\n                0.701025664806366\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.21190625429153442,\n                0.06235431134700775,\n                0.935546875,\n                0.8408624529838562\n              ],\n              [\n                0.5365781188011169,\n                0.46937063336372375,\n                0.7937187552452087,\n                0.701025664806366\n              ],\n              [\n                0.21190625429153442,\n                0.06235431134700775,\n                0.935546875,\n                0.8408624529838562\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000105184.jpg",\n        "coco_id": 105184,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 522206,\n          "object_id": 2198735,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_105184_522206_2198735_2158881_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2158881,\n          "object_id": 2198735,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.2861799895763397,\n                0.39928191900253296,\n                0.5259199738502502,\n                0.7439627647399902\n              ],\n              [\n                0.25106000900268555,\n                0.5117819309234619,\n                0.29047998785972595,\n                0.5617819428443909\n              ],\n              [\n                0.25106000900268555,\n                0.39928191900253296,\n                0.5259199738502502,\n                0.7439627647399902\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.7234600186347961,\n                0.4619680941104889,\n                0.9575200080871582,\n                0.893856406211853\n              ],\n              [\n                0.25106000900268555,\n                0.5117819309234619,\n                0.29047998785972595,\n                0.5617819428443909\n              ],\n              [\n                0.25106000900268555,\n                0.4619680941104889,\n                0.9575200080871582,\n                0.893856406211853\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000105219.jpg",\n        "coco_id": 105219,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "teddy bear",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 557438,\n          "object_id": 1159092,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_105219_557438_1159092_445041_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 445041,\n          "object_id": 1159092,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a teddy bear",\n            "bounds": [\n              [\n                0.5414791703224182,\n                0.23119445145130157,\n                1.0,\n                0.9913055300712585\n              ],\n              [\n                0.5709375143051147,\n                0.5923333168029785,\n                0.9729791879653931,\n                0.9887499809265137\n              ],\n              [\n                0.5414791703224182,\n                0.23119445145130157,\n                1.0,\n                0.9913055300712585\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a teddy bear",\n            "bounds": [\n              [\n                0.0016666667070239782,\n                0.15808333456516266,\n                0.45368748903274536,\n                0.9839722514152527\n              ],\n              [\n                0.5709375143051147,\n                0.5923333168029785,\n                0.9729791879653931,\n                0.9887499809265137\n              ],\n              [\n                0.0016666667070239782,\n                0.15808333456516266,\n                0.9729791879653931,\n                0.9887499809265137\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000107843.jpg",\n        "coco_id": 107843,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 479452,\n          "object_id": 1185204,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_107843_479452_1185204_1717022_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1717022,\n          "object_id": 1185204,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.0357656255364418,\n                0.40449243783950806,\n                0.16420312225818634,\n                0.7325918078422546\n              ],\n              [\n                0.29695311188697815,\n                0.6219654679298401,\n                0.39628124237060547,\n                0.7677322030067444\n              ],\n              [\n                0.0357656255364418,\n                0.40449243783950806,\n                0.39628124237060547,\n                0.7677322030067444\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.1518281251192093,\n                0.3442980647087097,\n                0.2768906354904175,\n                0.7499352097511292\n              ],\n              [\n                0.29695311188697815,\n                0.6219654679298401,\n                0.39628124237060547,\n                0.7677322030067444\n              ],\n              [\n                0.1518281251192093,\n                0.3442980647087097,\n                0.39628124237060547,\n                0.7677322030067444\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000107843.jpg",\n        "coco_id": 107843,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 479452,\n          "object_id": 1185204,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_107843_479452_1185204_522539_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 522539,\n          "object_id": 1185204,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.4559687376022339,\n                0.34872570633888245,\n                0.5707343816757202,\n                0.7178833484649658\n              ],\n              [\n                0.29695311188697815,\n                0.6219654679298401,\n                0.39628124237060547,\n                0.7677322030067444\n              ],\n              [\n                0.29695311188697815,\n                0.34872570633888245,\n                0.5707343816757202,\n                0.7677322030067444\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.1518281251192093,\n                0.3442980647087097,\n                0.2768906354904175,\n                0.7499352097511292\n              ],\n              [\n                0.29695311188697815,\n                0.6219654679298401,\n                0.39628124237060547,\n                0.7677322030067444\n              ],\n              [\n                0.1518281251192093,\n                0.3442980647087097,\n                0.39628124237060547,\n                0.7677322030067444\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000107843.jpg",\n        "coco_id": 107843,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "umbrella",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 502289,\n          "object_id": 279936,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_107843_502289_279936_1717022_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1717022,\n          "object_id": 279936,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.5800625085830688,\n                0.32183584570884705,\n                0.7607812285423279,\n                0.786457896232605\n              ],\n              [\n                0.5706250071525574,\n                0.29887688159942627,\n                0.7429531216621399,\n                0.4112311005592346\n              ],\n              [\n                0.5706250071525574,\n                0.29887688159942627,\n                0.7607812285423279,\n                0.786457896232605\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.0357656255364418,\n                0.40449243783950806,\n                0.16420312225818634,\n                0.7325918078422546\n              ],\n              [\n                0.5706250071525574,\n                0.29887688159942627,\n                0.7429531216621399,\n                0.4112311005592346\n              ],\n              [\n                0.0357656255364418,\n                0.29887688159942627,\n                0.7429531216621399,\n                0.7325918078422546\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000107843.jpg",\n        "coco_id": 107843,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "umbrella",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 502289,\n          "object_id": 279936,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_107843_502289_279936_522539_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 522539,\n          "object_id": 279936,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.4559687376022339,\n                0.34872570633888245,\n                0.5707343816757202,\n                0.7178833484649658\n              ],\n              [\n                0.5706250071525574,\n                0.29887688159942627,\n                0.7429531216621399,\n                0.4112311005592346\n              ],\n              [\n                0.4559687376022339,\n                0.29887688159942627,\n                0.7429531216621399,\n                0.7178833484649658\n              ]\n            ]\n          },\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.5800625085830688,\n                0.32183584570884705,\n                0.7607812285423279,\n                0.786457896232605\n              ],\n              [\n                0.5706250071525574,\n                0.29887688159942627,\n                0.7429531216621399,\n                0.4112311005592346\n              ],\n              [\n                0.5706250071525574,\n                0.29887688159942627,\n                0.7607812285423279,\n                0.786457896232605\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000107843.jpg",\n        "coco_id": 107843,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "umbrella",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 522539,\n          "object_id": 280479,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_107843_522539_280479_1717022_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1717022,\n          "object_id": 280479,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.4559687376022339,\n                0.34872570633888245,\n                0.5707343816757202,\n                0.7178833484649658\n              ],\n              [\n                0.441468745470047,\n                0.32555076479911804,\n                0.5842187404632568,\n                0.40673866868019104\n              ],\n              [\n                0.441468745470047,\n                0.32555076479911804,\n                0.5842187404632568,\n                0.7178833484649658\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.0357656255364418,\n                0.40449243783950806,\n                0.16420312225818634,\n                0.7325918078422546\n              ],\n              [\n                0.441468745470047,\n                0.32555076479911804,\n                0.5842187404632568,\n                0.40673866868019104\n              ],\n              [\n                0.0357656255364418,\n                0.32555076479911804,\n                0.5842187404632568,\n                0.7325918078422546\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000110423.jpg",\n        "coco_id": 110423,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "boat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1244289,\n          "object_id": 178726,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_110423_1244289_178726_1249981_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1249981,\n          "object_id": 178726,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a boat",\n            "bounds": [\n              [\n                0.228515625,\n                0.2105555534362793,\n                0.2672500014305115,\n                0.507888913154602\n              ],\n              [\n                0.003796875011175871,\n                0.5529166460037231,\n                0.15962499380111694,\n                0.8592222332954407\n              ],\n              [\n                0.003796875011175871,\n                0.2105555534362793,\n                0.2672500014305115,\n                0.8592222332954407\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a boat",\n            "bounds": [\n              [\n                0.06357812136411667,\n                0.30799999833106995,\n                0.1478593796491623,\n                0.5565277934074402\n              ],\n              [\n                0.003796875011175871,\n                0.5529166460037231,\n                0.15962499380111694,\n                0.8592222332954407\n              ],\n              [\n                0.003796875011175871,\n                0.30799999833106995,\n                0.15962499380111694,\n                0.8592222332954407\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000110423.jpg",\n        "coco_id": 110423,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "boat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2025885,\n          "object_id": 249456,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_110423_2025885_249456_1249981_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1249981,\n          "object_id": 249456,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a boat",\n            "bounds": [\n              [\n                0.228515625,\n                0.2105555534362793,\n                0.2672500014305115,\n                0.507888913154602\n              ],\n              [\n                0.5991562604904175,\n                0.5614166855812073,\n                0.7344062328338623,\n                0.6625555753707886\n              ],\n              [\n                0.228515625,\n                0.2105555534362793,\n                0.7344062328338623,\n                0.6625555753707886\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a boat",\n            "bounds": [\n              [\n                0.5992500185966492,\n                0.41377776861190796,\n                0.6961718797683716,\n                0.5956666469573975\n              ],\n              [\n                0.5991562604904175,\n                0.5614166855812073,\n                0.7344062328338623,\n                0.6625555753707886\n              ],\n              [\n                0.5991562604904175,\n                0.41377776861190796,\n                0.7344062328338623,\n                0.6625555753707886\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000111922.jpg",\n        "coco_id": 111922,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "remote",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 188034,\n          "object_id": 1109873,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_111922_188034_1109873_224400_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 224400,\n          "object_id": 1109873,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.001687500043772161,\n                0.47085416316986084,\n                0.19843749701976776,\n                0.8811666369438171\n              ],\n              [\n                0.5402656197547913,\n                0.35843750834465027,\n                0.560296893119812,\n                0.3941250145435333\n              ],\n              [\n                0.001687500043772161,\n                0.35843750834465027,\n                0.560296893119812,\n                0.8811666369438171\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.4071093797683716,\n                0.2709791660308838,\n                0.5659219026565552,\n                0.9716458320617676\n              ],\n              [\n                0.5402656197547913,\n                0.35843750834465027,\n                0.560296893119812,\n                0.3941250145435333\n              ],\n              [\n                0.4071093797683716,\n                0.2709791660308838,\n                0.5659219026565552,\n                0.9716458320617676\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000111922.jpg",\n        "coco_id": 111922,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 200603,\n          "object_id": 110549,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_111922_200603_110549_231459_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 231459,\n          "object_id": 110549,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.07078125327825546,\n                0.5078750252723694,\n                0.4162968695163727,\n                0.9123541712760925\n              ],\n              [\n                0.0787656232714653,\n                0.8473749756813049,\n                0.4028593897819519,\n                1.0\n              ],\n              [\n                0.07078125327825546,\n                0.5078750252723694,\n                0.4162968695163727,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.748828113079071,\n                0.31295832991600037,\n                0.9396874904632568,\n                0.895479142665863\n              ],\n              [\n                0.0787656232714653,\n                0.8473749756813049,\n                0.4028593897819519,\n                1.0\n              ],\n              [\n                0.0787656232714653,\n                0.31295832991600037,\n                0.9396874904632568,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000111922.jpg",\n        "coco_id": 111922,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 224400,\n          "object_id": 1940761,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_111922_224400_1940761_188034_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 188034,\n          "object_id": 1940761,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.001687500043772161,\n                0.47085416316986084,\n                0.19843749701976776,\n                0.8811666369438171\n              ],\n              [\n                0.0,\n                0.8869791626930237,\n                0.08357812464237213,\n                1.0\n              ],\n              [\n                0.0,\n                0.47085416316986084,\n                0.19843749701976776,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.4071093797683716,\n                0.2709791660308838,\n                0.5659219026565552,\n                0.9716458320617676\n              ],\n              [\n                0.0,\n                0.8869791626930237,\n                0.08357812464237213,\n                1.0\n              ],\n              [\n                0.0,\n                0.2709791660308838,\n                0.5659219026565552,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000111936.jpg",\n        "coco_id": 111936,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 259548,\n          "object_id": 1430702,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_111936_259548_1430702_190044_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 190044,\n          "object_id": 1430702,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.7652812600135803,\n                0.46350589394569397,\n                0.9040625095367432,\n                0.9983294010162354\n              ],\n              [\n                0.7745781540870667,\n                0.6627764701843262,\n                0.8348593711853027,\n                0.7525647282600403\n              ],\n              [\n                0.7652812600135803,\n                0.46350589394569397,\n                0.9040625095367432,\n                0.9983294010162354\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.5824843645095825,\n                0.3987058699131012,\n                0.7070000171661377,\n                0.9849176406860352\n              ],\n              [\n                0.7745781540870667,\n                0.6627764701843262,\n                0.8348593711853027,\n                0.7525647282600403\n              ],\n              [\n                0.5824843645095825,\n                0.3987058699131012,\n                0.8348593711853027,\n                0.9849176406860352\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000111936.jpg",\n        "coco_id": 111936,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 259548,\n          "object_id": 1430702,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_111936_259548_1430702_262790_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 262790,\n          "object_id": 1430702,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.8943281173706055,\n                0.40512940287590027,\n                1.0,\n                1.0\n              ],\n              [\n                0.7745781540870667,\n                0.6627764701843262,\n                0.8348593711853027,\n                0.7525647282600403\n              ],\n              [\n                0.7745781540870667,\n                0.40512940287590027,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.7652812600135803,\n                0.46350589394569397,\n                0.9040625095367432,\n                0.9983294010162354\n              ],\n              [\n                0.7745781540870667,\n                0.6627764701843262,\n                0.8348593711853027,\n                0.7525647282600403\n              ],\n              [\n                0.7652812600135803,\n                0.46350589394569397,\n                0.9040625095367432,\n                0.9983294010162354\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000114540.jpg",\n        "coco_id": 114540,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1720105,\n          "object_id": 1437025,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_114540_1720105_1437025_1718717_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1718717,\n          "object_id": 1437025,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.2467343807220459,\n                0.4354660212993622,\n                0.4034687578678131,\n                0.6809794902801514\n              ],\n              [\n                0.48884373903274536,\n                0.5092259049415588,\n                0.5264999866485596,\n                0.6402369737625122\n              ],\n              [\n                0.2467343807220459,\n                0.4354660212993622,\n                0.5264999866485596,\n                0.6809794902801514\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.4153437614440918,\n                0.4627962112426758,\n                0.5554375052452087,\n                0.7991153001785278\n              ],\n              [\n                0.48884373903274536,\n                0.5092259049415588,\n                0.5264999866485596,\n                0.6402369737625122\n              ],\n              [\n                0.4153437614440918,\n                0.4627962112426758,\n                0.5554375052452087,\n                0.7991153001785278\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000114540.jpg",\n        "coco_id": 114540,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1720105,\n          "object_id": 241661,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_114540_1720105_241661_1718717_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1718717,\n          "object_id": 241661,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.2467343807220459,\n                0.4354660212993622,\n                0.4034687578678131,\n                0.6809794902801514\n              ],\n              [\n                0.4738749861717224,\n                0.600300133228302,\n                0.6109218597412109,\n                0.8242496252059937\n              ],\n              [\n                0.2467343807220459,\n                0.4354660212993622,\n                0.6109218597412109,\n                0.8242496252059937\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.4153437614440918,\n                0.4627962112426758,\n                0.5554375052452087,\n                0.7991153001785278\n              ],\n              [\n                0.4738749861717224,\n                0.600300133228302,\n                0.6109218597412109,\n                0.8242496252059937\n              ],\n              [\n                0.4153437614440918,\n                0.4627962112426758,\n                0.6109218597412109,\n                0.8242496252059937\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000118186.jpg",\n        "coco_id": 118186,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 514280,\n          "object_id": 634524,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_118186_514280_634524_512243_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 512243,\n          "object_id": 634524,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.1874062567949295,\n                0.4265807867050171,\n                0.5787343978881836,\n                0.9591803550720215\n              ],\n              [\n                0.1899062544107437,\n                0.5966510772705078,\n                0.2623281180858612,\n                0.6666979193687439\n              ],\n              [\n                0.1874062567949295,\n                0.4265807867050171,\n                0.5787343978881836,\n                0.9591803550720215\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.257874995470047,\n                0.02920374646782875,\n                0.4182968735694885,\n                0.579765796661377\n              ],\n              [\n                0.1899062544107437,\n                0.5966510772705078,\n                0.2623281180858612,\n                0.6666979193687439\n              ],\n              [\n                0.1899062544107437,\n                0.02920374646782875,\n                0.4182968735694885,\n                0.6666979193687439\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000125550.jpg",\n        "coco_id": 125550,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "cake",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 425379,\n          "object_id": 1089023,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_125550_425379_1089023_474150_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 474150,\n          "object_id": 1089023,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.39784374833106995,\n                0.07207059115171432,\n                0.6401249766349792,\n                0.6779294013977051\n              ],\n              [\n                0.3297656178474426,\n                0.5494823455810547,\n                0.40828123688697815,\n                0.6369646787643433\n              ],\n              [\n                0.3297656178474426,\n                0.07207059115171432,\n                0.6401249766349792,\n                0.6779294013977051\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.1809687465429306,\n                0.08221176266670227,\n                0.553390622138977,\n                0.6655294299125671\n              ],\n              [\n                0.3297656178474426,\n                0.5494823455810547,\n                0.40828123688697815,\n                0.6369646787643433\n              ],\n              [\n                0.1809687465429306,\n                0.08221176266670227,\n                0.553390622138977,\n                0.6655294299125671\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000125550.jpg",\n        "coco_id": 125550,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cake",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 425379,\n          "object_id": 1089023,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_125550_425379_1089023_474150_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 474150,\n          "object_id": 1089023,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.39784374833106995,\n                0.07207059115171432,\n                0.6401249766349792,\n                0.6779294013977051\n              ],\n              [\n                0.3297656178474426,\n                0.5494823455810547,\n                0.40828123688697815,\n                0.6369646787643433\n              ],\n              [\n                0.3297656178474426,\n                0.07207059115171432,\n                0.6401249766349792,\n                0.6779294013977051\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.1809687465429306,\n                0.08221176266670227,\n                0.553390622138977,\n                0.6655294299125671\n              ],\n              [\n                0.3297656178474426,\n                0.5494823455810547,\n                0.40828123688697815,\n                0.6369646787643433\n              ],\n              [\n                0.1809687465429306,\n                0.08221176266670227,\n                0.553390622138977,\n                0.6655294299125671\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000125550.jpg",\n        "coco_id": 125550,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 518190,\n          "object_id": 683926,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_125550_518190_683926_474150_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 474150,\n          "object_id": 683926,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.39784374833106995,\n                0.07207059115171432,\n                0.6401249766349792,\n                0.6779294013977051\n              ],\n              [\n                0.6707343459129333,\n                0.5654117465019226,\n                0.7149999737739563,\n                0.653388261795044\n              ],\n              [\n                0.39784374833106995,\n                0.07207059115171432,\n                0.7149999737739563,\n                0.6779294013977051\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.6521718502044678,\n                0.09642352908849716,\n                0.9597031474113464,\n                0.7697882056236267\n              ],\n              [\n                0.6707343459129333,\n                0.5654117465019226,\n                0.7149999737739563,\n                0.653388261795044\n              ],\n              [\n                0.6521718502044678,\n                0.09642352908849716,\n                0.9597031474113464,\n                0.7697882056236267\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000127986.jpg",\n        "coco_id": 127986,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 481415,\n          "object_id": 658458,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_127986_481415_658458_534374_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 534374,\n          "object_id": 658458,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.6863124966621399,\n                0.00912735890597105,\n                0.7845781445503235,\n                0.2540801763534546\n              ],\n              [\n                0.5809062719345093,\n                0.34271225333213806,\n                0.6739062666893005,\n                0.4051651060581207\n              ],\n              [\n                0.5809062719345093,\n                0.00912735890597105,\n                0.7845781445503235,\n                0.4051651060581207\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.4691562354564667,\n                0.24566037952899933,\n                0.5987812280654907,\n                0.6356368064880371\n              ],\n              [\n                0.5809062719345093,\n                0.34271225333213806,\n                0.6739062666893005,\n                0.4051651060581207\n              ],\n              [\n                0.4691562354564667,\n                0.24566037952899933,\n                0.6739062666893005,\n                0.6356368064880371\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000128615.jpg",\n        "coco_id": 128615,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 477530,\n          "object_id": 635046,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_128615_477530_635046_516016_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 516016,\n          "object_id": 635046,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.2560235261917114,\n                0.2084687501192093,\n                0.6973646879196167,\n                0.6984062790870667\n              ],\n              [\n                0.23792941868305206,\n                0.17926561832427979,\n                0.3595764636993408,\n                0.2265312522649765\n              ],\n              [\n                0.23792941868305206,\n                0.17926561832427979,\n                0.6973646879196167,\n                0.6984062790870667\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.28519999980926514,\n                0.5534999966621399,\n                0.7275294065475464,\n                0.7580468654632568\n              ],\n              [\n                0.23792941868305206,\n                0.17926561832427979,\n                0.3595764636993408,\n                0.2265312522649765\n              ],\n              [\n                0.23792941868305206,\n                0.17926561832427979,\n                0.7275294065475464,\n                0.7580468654632568\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000132626.jpg",\n        "coco_id": 132626,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 189077,\n          "object_id": 153165,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_132626_189077_153165_189965_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 189965,\n          "object_id": 153165,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.28062498569488525,\n                0.10826697945594788,\n                0.5424374938011169,\n                0.9820609092712402\n              ],\n              [\n                0.22742187976837158,\n                0.1800234168767929,\n                0.7063437700271606,\n                0.988149881362915\n              ],\n              [\n                0.22742187976837158,\n                0.10826697945594788,\n                0.7063437700271606,\n                0.988149881362915\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.1685781180858612,\n                0.05709601938724518,\n                0.39003124833106995,\n                0.8857845664024353\n              ],\n              [\n                0.22742187976837158,\n                0.1800234168767929,\n                0.7063437700271606,\n                0.988149881362915\n              ],\n              [\n                0.1685781180858612,\n                0.05709601938724518,\n                0.7063437700271606,\n                0.988149881362915\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000136300.jpg",\n        "coco_id": 136300,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 209940,\n          "object_id": 2230092,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_136300_209940_2230092_206156_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 206156,\n          "object_id": 2230092,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.34184375405311584,\n                0.3393208384513855,\n                0.42729687690734863,\n                0.6651756167411804\n              ],\n              [\n                0.6640468835830688,\n                0.4598594903945923,\n                0.8941718935966492,\n                0.8128337264060974\n              ],\n              [\n                0.34184375405311584,\n                0.3393208384513855,\n                0.8941718935966492,\n                0.8128337264060974\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.7488437294960022,\n                0.34765806794166565,\n                0.8147343993186951,\n                0.654004693031311\n              ],\n              [\n                0.6640468835830688,\n                0.4598594903945923,\n                0.8941718935966492,\n                0.8128337264060974\n              ],\n              [\n                0.6640468835830688,\n                0.34765806794166565,\n                0.8941718935966492,\n                0.8128337264060974\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000136953.jpg",\n        "coco_id": 136953,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 519571,\n          "object_id": 669987,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_136953_519571_669987_530897_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 530897,\n          "object_id": 669987,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.3573000133037567,\n                0.4000000059604645,\n                0.6977599859237671,\n                0.9887733459472656\n              ],\n              [\n                0.4553599953651428,\n                0.6363199949264526,\n                0.4969800114631653,\n                0.7878933548927307\n              ],\n              [\n                0.3573000133037567,\n                0.4000000059604645,\n                0.6977599859237671,\n                0.9887733459472656\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.0017000000225380063,\n                0.12442667037248611,\n                0.36142000555992126,\n                0.9864266514778137\n              ],\n              [\n                0.4553599953651428,\n                0.6363199949264526,\n                0.4969800114631653,\n                0.7878933548927307\n              ],\n              [\n                0.0017000000225380063,\n                0.12442667037248611,\n                0.4969800114631653,\n                0.9864266514778137\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000136953.jpg",\n        "coco_id": 136953,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "remote",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 528152,\n          "object_id": 1108291,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_136953_528152_1108291_530897_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 530897,\n          "object_id": 1108291,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.6938199996948242,\n                0.09213333576917648,\n                1.0,\n                0.9870133399963379\n              ],\n              [\n                0.7363399863243103,\n                0.5845866799354553,\n                0.796280026435852,\n                0.6861600279808044\n              ],\n              [\n                0.6938199996948242,\n                0.09213333576917648,\n                1.0,\n                0.9870133399963379\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.0017000000225380063,\n                0.12442667037248611,\n                0.36142000555992126,\n                0.9864266514778137\n              ],\n              [\n                0.7363399863243103,\n                0.5845866799354553,\n                0.796280026435852,\n                0.6861600279808044\n              ],\n              [\n                0.0017000000225380063,\n                0.12442667037248611,\n                0.796280026435852,\n                0.9864266514778137\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000137150.jpg",\n        "coco_id": 137150,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "banana",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 187930,\n          "object_id": 1542575,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_137150_187930_1542575_191414_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 191414,\n          "object_id": 1542575,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a banana",\n            "bounds": [\n              [\n                0.3235156238079071,\n                0.2774178385734558,\n                0.6212499737739563,\n                1.0\n              ],\n              [\n                0.41792187094688416,\n                0.0862441286444664,\n                0.5162343978881836,\n                0.18014083802700043\n              ],\n              [\n                0.3235156238079071,\n                0.0862441286444664,\n                0.6212499737739563,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a banana",\n            "bounds": [\n              [\n                0.0002656250144354999,\n                0.0012206572573632002,\n                0.1888750046491623,\n                1.0\n              ],\n              [\n                0.41792187094688416,\n                0.0862441286444664,\n                0.5162343978881836,\n                0.18014083802700043\n              ],\n              [\n                0.0002656250144354999,\n                0.0012206572573632002,\n                0.5162343978881836,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000137150.jpg",\n        "coco_id": 137150,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "banana",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 187930,\n          "object_id": 1543067,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_137150_187930_1543067_200808_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 200808,\n          "object_id": 1543067,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.7858281135559082,\n                0.3154694736003876,\n                0.9586250185966492,\n                0.9881455302238464\n              ],\n              [\n                0.3874843716621399,\n                0.15870891511440277,\n                0.48814061284065247,\n                0.2847183048725128\n              ],\n              [\n                0.3874843716621399,\n                0.15870891511440277,\n                0.9586250185966492,\n                0.9881455302238464\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.3235156238079071,\n                0.2774178385734558,\n                0.6212499737739563,\n                1.0\n              ],\n              [\n                0.3874843716621399,\n                0.15870891511440277,\n                0.48814061284065247,\n                0.2847183048725128\n              ],\n              [\n                0.3235156238079071,\n                0.15870891511440277,\n                0.6212499737739563,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000138131.jpg",\n        "coco_id": 138131,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1725659,\n          "object_id": 605162,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_138131_1725659_605162_536516_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 536516,\n          "object_id": 605162,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.44820868968963623,\n                0.49592188000679016,\n                0.6336348056793213,\n                0.9067656397819519\n              ],\n              [\n                0.44975653290748596,\n                0.7067031264305115,\n                0.5373391509056091,\n                0.7671562433242798\n              ],\n              [\n                0.44820868968963623,\n                0.49592188000679016,\n                0.6336348056793213,\n                0.9067656397819519\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.6651825904846191,\n                0.4348124861717224,\n                1.0,\n                0.848312497138977\n              ],\n              [\n                0.44975653290748596,\n                0.7067031264305115,\n                0.5373391509056091,\n                0.7671562433242798\n              ],\n              [\n                0.44975653290748596,\n                0.4348124861717224,\n                1.0,\n                0.848312497138977\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000140074.jpg",\n        "coco_id": 140074,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 555717,\n          "object_id": 1448627,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_140074_555717_1448627_559888_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 559888,\n          "object_id": 1448627,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.07953125238418579,\n                0.2195884734392166,\n                0.2026718705892563,\n                0.8704938292503357\n              ],\n              [\n                0.14103125035762787,\n                0.1357201635837555,\n                0.179625004529953,\n                0.24806584417819977\n              ],\n              [\n                0.07953125238418579,\n                0.1357201635837555,\n                0.2026718705892563,\n                0.8704938292503357\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.7947656512260437,\n                0.2512757182121277,\n                0.8827187418937683,\n                0.8911111354827881\n              ],\n              [\n                0.14103125035762787,\n                0.1357201635837555,\n                0.179625004529953,\n                0.24806584417819977\n              ],\n              [\n                0.14103125035762787,\n                0.1357201635837555,\n                0.8827187418937683,\n                0.8911111354827881\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000140990.jpg",\n        "coco_id": 140990,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 423994,\n          "object_id": 627832,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_140990_423994_627832_1211228_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1211228,\n          "object_id": 627832,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.5258749723434448,\n                0.05617570877075195,\n                0.9607031345367432,\n                0.9460723400115967\n              ],\n              [\n                0.39039063453674316,\n                0.1800258457660675,\n                0.5719218850135803,\n                0.2635400593280792\n              ],\n              [\n                0.39039063453674316,\n                0.05617570877075195,\n                0.9607031345367432,\n                0.9460723400115967\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.014953125268220901,\n                0.3842635750770569,\n                0.39134374260902405,\n                1.0\n              ],\n              [\n                0.39039063453674316,\n                0.1800258457660675,\n                0.5719218850135803,\n                0.2635400593280792\n              ],\n              [\n                0.014953125268220901,\n                0.1800258457660675,\n                0.5719218850135803,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000142380.jpg",\n        "coco_id": 142380,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "snowboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2205689,\n          "object_id": 618897,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_142380_2205689_618897_423693_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 423693,\n          "object_id": 618897,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a snowboard",\n            "bounds": [\n              [\n                0.2837812602519989,\n                0.40304964780807495,\n                0.35718750953674316,\n                0.8544917106628418\n              ],\n              [\n                0.19062499701976776,\n                0.4890070855617523,\n                0.4232812523841858,\n                0.7957446575164795\n              ],\n              [\n                0.19062499701976776,\n                0.40304964780807495,\n                0.4232812523841858,\n                0.8544917106628418\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a snowboard",\n            "bounds": [\n              [\n                0.32395312190055847,\n                0.4165484607219696,\n                0.4332500100135803,\n                0.8450827598571777\n              ],\n              [\n                0.19062499701976776,\n                0.4890070855617523,\n                0.4232812523841858,\n                0.7957446575164795\n              ],\n              [\n                0.19062499701976776,\n                0.4165484607219696,\n                0.4332500100135803,\n                0.8450827598571777\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000143348.jpg",\n        "coco_id": 143348,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1278119,\n          "object_id": 655501,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_143348_1278119_655501_1314521_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1314521,\n          "object_id": 655501,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.3421406149864197,\n                0.4089791774749756,\n                0.6353906393051147,\n                0.9775208234786987\n              ],\n              [\n                0.552984356880188,\n                0.07760416716337204,\n                0.6435624957084656,\n                0.265583336353302\n              ],\n              [\n                0.3421406149864197,\n                0.07760416716337204,\n                0.6435624957084656,\n                0.9775208234786987\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.001687500043772161,\n                0.4490624964237213,\n                0.15337499976158142,\n                0.8445624709129333\n              ],\n              [\n                0.552984356880188,\n                0.07760416716337204,\n                0.6435624957084656,\n                0.265583336353302\n              ],\n              [\n                0.001687500043772161,\n                0.07760416716337204,\n                0.6435624957084656,\n                0.8445624709129333\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000149896.jpg",\n        "coco_id": 149896,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 453292,\n          "object_id": 1844705,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_149896_453292_1844705_447945_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 447945,\n          "object_id": 1844705,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.09799418598413467,\n                0.26965999603271484,\n                0.9080232381820679,\n                0.9393200278282166\n              ],\n              [\n                0.3201162815093994,\n                0.41260001063346863,\n                0.4932267367839813,\n                0.5227000117301941\n              ],\n              [\n                0.09799418598413467,\n                0.26965999603271484,\n                0.9080232381820679,\n                0.9393200278282166\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.03593023121356964,\n                0.18201999366283417,\n                0.8448546528816223,\n                0.7258399724960327\n              ],\n              [\n                0.3201162815093994,\n                0.41260001063346863,\n                0.4932267367839813,\n                0.5227000117301941\n              ],\n              [\n                0.03593023121356964,\n                0.18201999366283417,\n                0.8448546528816223,\n                0.7258399724960327\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000154020.jpg",\n        "coco_id": 154020,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 476076,\n          "object_id": 1448607,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_154020_476076_1448607_525316_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 525316,\n          "object_id": 1448607,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.0,\n                0.07280000299215317,\n                0.22516000270843506,\n                0.9915466904640198\n              ],\n              [\n                0.33737999200820923,\n                0.6161066889762878,\n                0.47929999232292175,\n                0.6517333388328552\n              ],\n              [\n                0.0,\n                0.07280000299215317,\n                0.47929999232292175,\n                0.9915466904640198\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.3910199999809265,\n                0.2563199996948242,\n                0.9084200263023376,\n                0.9731733202934265\n              ],\n              [\n                0.33737999200820923,\n                0.6161066889762878,\n                0.47929999232292175,\n                0.6517333388328552\n              ],\n              [\n                0.33737999200820923,\n                0.2563199996948242,\n                0.9084200263023376,\n                0.9731733202934265\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000154911.jpg",\n        "coco_id": 154911,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 464414,\n          "object_id": 1591030,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_154911_464414_1591030_547068_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 547068,\n          "object_id": 1591030,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.011796874925494194,\n                0.5101041793823242,\n                0.4365156292915344,\n                0.9146041870117188\n              ],\n              [\n                0.0,\n                0.6266666650772095,\n                0.08190625160932541,\n                0.799791693687439\n              ],\n              [\n                0.0,\n                0.5101041793823242,\n                0.4365156292915344,\n                0.9146041870117188\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.0,\n                0.00020833333837799728,\n                0.1509999930858612,\n                0.6400416493415833\n              ],\n              [\n                0.0,\n                0.6266666650772095,\n                0.08190625160932541,\n                0.799791693687439\n              ],\n              [\n                0.0,\n                0.00020833333837799728,\n                0.1509999930858612,\n                0.799791693687439\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000155191.jpg",\n        "coco_id": 155191,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2149552,\n          "object_id": 2082264,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_155191_2149552_2082264_2149462_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2149462,\n          "object_id": 2082264,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.6797663569450378,\n                0.46860936284065247,\n                1.0,\n                0.7614374756813049\n              ],\n              [\n                0.1934579461812973,\n                0.5329218506813049,\n                0.4491121470928192,\n                0.7377499938011169\n              ],\n              [\n                0.1934579461812973,\n                0.46860936284065247,\n                1.0,\n                0.7614374756813049\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.3270093500614166,\n                0.37617188692092896,\n                0.8009812831878662,\n                0.8218281269073486\n              ],\n              [\n                0.1934579461812973,\n                0.5329218506813049,\n                0.4491121470928192,\n                0.7377499938011169\n              ],\n              [\n                0.1934579461812973,\n                0.37617188692092896,\n                0.8009812831878662,\n                0.8218281269073486\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000157017.jpg",\n        "coco_id": 157017,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skateboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2155161,\n          "object_id": 645461,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_157017_2155161_645461_184931_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 184931,\n          "object_id": 645461,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.21590624749660492,\n                0.05100702494382858,\n                0.5277500152587891,\n                0.5004449486732483\n              ],\n              [\n                0.010781249962747097,\n                0.3121545612812042,\n                0.08364062756299973,\n                0.5929273962974548\n              ],\n              [\n                0.010781249962747097,\n                0.05100702494382858,\n                0.5277500152587891,\n                0.5929273962974548\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.01385937537997961,\n                0.03377049043774605,\n                0.10917187482118607,\n                0.5428571701049805\n              ],\n              [\n                0.010781249962747097,\n                0.3121545612812042,\n                0.08364062756299973,\n                0.5929273962974548\n              ],\n              [\n                0.010781249962747097,\n                0.03377049043774605,\n                0.10917187482118607,\n                0.5929273962974548\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000159577.jpg",\n        "coco_id": 159577,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "scissors",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 487065,\n          "object_id": 1158299,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_159577_487065_1158299_426271_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 426271,\n          "object_id": 1158299,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding scissors",\n            "bounds": [\n              [\n                0.5375800132751465,\n                0.00249249255284667,\n                0.9990000128746033,\n                0.9868168234825134\n              ],\n              [\n                0.34393998980522156,\n                0.33648648858070374,\n                0.5816799998283386,\n                0.6287387609481812\n              ],\n              [\n                0.34393998980522156,\n                0.00249249255284667,\n                0.9990000128746033,\n                0.9868168234825134\n              ]\n            ]\n          },\n          {\n            "text": "a person holding scissors",\n            "bounds": [\n              [\n                0.003000000026077032,\n                0.018138138577342033,\n                0.7737600207328796,\n                0.9799399375915527\n              ],\n              [\n                0.34393998980522156,\n                0.33648648858070374,\n                0.5816799998283386,\n                0.6287387609481812\n              ],\n              [\n                0.003000000026077032,\n                0.018138138577342033,\n                0.7737600207328796,\n                0.9799399375915527\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000016069.jpg",\n        "coco_id": 16069,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "umbrella",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 211754,\n          "object_id": 282107,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_16069_211754_282107_207888_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 207888,\n          "object_id": 282107,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.8062499761581421,\n                0.2385377287864685,\n                0.9426406025886536,\n                0.7922877073287964\n              ],\n              [\n                0.7428905963897705,\n                0.15521226823329926,\n                0.9200624823570251,\n                0.26757076382637024\n              ],\n              [\n                0.7428905963897705,\n                0.15521226823329926,\n                0.9426406025886536,\n                0.7922877073287964\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.49799999594688416,\n                0.3086792528629303,\n                0.6475468873977661,\n                0.8256132006645203\n              ],\n              [\n                0.7428905963897705,\n                0.15521226823329926,\n                0.9200624823570251,\n                0.26757076382637024\n              ],\n              [\n                0.49799999594688416,\n                0.15521226823329926,\n                0.9200624823570251,\n                0.8256132006645203\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000016069.jpg",\n        "coco_id": 16069,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "umbrella",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 211754,\n          "object_id": 282107,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_16069_211754_282107_207888_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 207888,\n          "object_id": 282107,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.49799999594688416,\n                0.3086792528629303,\n                0.6475468873977661,\n                0.8256132006645203\n              ],\n              [\n                0.7428905963897705,\n                0.15521226823329926,\n                0.9200624823570251,\n                0.26757076382637024\n              ],\n              [\n                0.49799999594688416,\n                0.15521226823329926,\n                0.9200624823570251,\n                0.8256132006645203\n              ]\n            ]\n          },\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.8062499761581421,\n                0.2385377287864685,\n                0.9426406025886536,\n                0.7922877073287964\n              ],\n              [\n                0.7428905963897705,\n                0.15521226823329926,\n                0.9200624823570251,\n                0.26757076382637024\n              ],\n              [\n                0.7428905963897705,\n                0.15521226823329926,\n                0.9426406025886536,\n                0.7922877073287964\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000164572.jpg",\n        "coco_id": 164572,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 505564,\n          "object_id": 321763,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_164572_505564_321763_483955_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 483955,\n          "object_id": 321763,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.05093749985098839,\n                0.38195833563804626,\n                0.18793749809265137,\n                0.987666666507721\n              ],\n              [\n                0.08121874928474426,\n                0.44062501192092896,\n                0.11040624976158142,\n                0.48639583587646484\n              ],\n              [\n                0.05093749985098839,\n                0.38195833563804626,\n                0.18793749809265137,\n                0.987666666507721\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.6593906283378601,\n                0.2866666615009308,\n                0.8674530982971191,\n                0.9868958592414856\n              ],\n              [\n                0.08121874928474426,\n                0.44062501192092896,\n                0.11040624976158142,\n                0.48639583587646484\n              ],\n              [\n                0.08121874928474426,\n                0.2866666615009308,\n                0.8674530982971191,\n                0.9868958592414856\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000164572.jpg",\n        "coco_id": 164572,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 505564,\n          "object_id": 321763,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_164572_505564_321763_483955_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 483955,\n          "object_id": 321763,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.6593906283378601,\n                0.2866666615009308,\n                0.8674530982971191,\n                0.9868958592414856\n              ],\n              [\n                0.08121874928474426,\n                0.44062501192092896,\n                0.11040624976158142,\n                0.48639583587646484\n              ],\n              [\n                0.08121874928474426,\n                0.2866666615009308,\n                0.8674530982971191,\n                0.9868958592414856\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.05093749985098839,\n                0.38195833563804626,\n                0.18793749809265137,\n                0.987666666507721\n              ],\n              [\n                0.08121874928474426,\n                0.44062501192092896,\n                0.11040624976158142,\n                0.48639583587646484\n              ],\n              [\n                0.05093749985098839,\n                0.38195833563804626,\n                0.18793749809265137,\n                0.987666666507721\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000164663.jpg",\n        "coco_id": 164663,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2206476,\n          "object_id": 605395,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_164663_2206476_605395_494354_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 494354,\n          "object_id": 605395,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.7618125081062317,\n                0.4313125014305115,\n                0.963546872138977,\n                0.988656222820282\n              ],\n              [\n                0.8493906259536743,\n                0.4009062647819519,\n                0.8904687762260437,\n                0.4781562387943268\n              ],\n              [\n                0.7618125081062317,\n                0.4009062647819519,\n                0.963546872138977,\n                0.988656222820282\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.0023593748919665813,\n                0.7074375152587891,\n                0.19529686868190765,\n                0.9898124933242798\n              ],\n              [\n                0.8493906259536743,\n                0.4009062647819519,\n                0.8904687762260437,\n                0.4781562387943268\n              ],\n              [\n                0.0023593748919665813,\n                0.4009062647819519,\n                0.8904687762260437,\n                0.9898124933242798\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000164663.jpg",\n        "coco_id": 164663,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 434719,\n          "object_id": 604817,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_164663_434719_604817_494354_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 494354,\n          "object_id": 604817,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.0023593748919665813,\n                0.7074375152587891,\n                0.19529686868190765,\n                0.9898124933242798\n              ],\n              [\n                0.47749999165534973,\n                0.3909687399864197,\n                0.5041406154632568,\n                0.4534062445163727\n              ],\n              [\n                0.0023593748919665813,\n                0.3909687399864197,\n                0.5041406154632568,\n                0.9898124933242798\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.39774999022483826,\n                0.43974998593330383,\n                0.551687479019165,\n                0.9858437776565552\n              ],\n              [\n                0.47749999165534973,\n                0.3909687399864197,\n                0.5041406154632568,\n                0.4534062445163727\n              ],\n              [\n                0.39774999022483826,\n                0.3909687399864197,\n                0.551687479019165,\n                0.9858437776565552\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000168763.jpg",\n        "coco_id": 168763,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 190387,\n          "object_id": 53720,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_168763_190387_53720_196117_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 196117,\n          "object_id": 53720,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.6422343850135803,\n                0.4975263178348541,\n                0.8238750100135803,\n                0.9800876975059509\n              ],\n              [\n                0.49924999475479126,\n                0.6031578779220581,\n                0.8951562643051147,\n                1.0\n              ],\n              [\n                0.49924999475479126,\n                0.4975263178348541,\n                0.8951562643051147,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.12134374678134918,\n                0.4796140491962433,\n                0.4192187488079071,\n                0.9208947420120239\n              ],\n              [\n                0.49924999475479126,\n                0.6031578779220581,\n                0.8951562643051147,\n                1.0\n              ],\n              [\n                0.12134374678134918,\n                0.4796140491962433,\n                0.8951562643051147,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000168900.jpg",\n        "coco_id": 168900,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1288889,\n          "object_id": 148071,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_168900_1288889_148071_1718678_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1718678,\n          "object_id": 148071,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.06628125160932541,\n                0.3420754671096802,\n                0.12790624797344208,\n                0.5830660462379456\n              ],\n              [\n                0.33873438835144043,\n                0.5909669995307922,\n                0.41624999046325684,\n                0.7529953122138977\n              ],\n              [\n                0.06628125160932541,\n                0.3420754671096802,\n                0.41624999046325684,\n                0.7529953122138977\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.3336718678474426,\n                0.5656132102012634,\n                0.41817188262939453,\n                0.7548348903656006\n              ],\n              [\n                0.33873438835144043,\n                0.5909669995307922,\n                0.41624999046325684,\n                0.7529953122138977\n              ],\n              [\n                0.3336718678474426,\n                0.5656132102012634,\n                0.41817188262939453,\n                0.7548348903656006\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000168900.jpg",\n        "coco_id": 168900,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1288889,\n          "object_id": 148071,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_168900_1288889_148071_1718678_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1718678,\n          "object_id": 148071,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.06628125160932541,\n                0.3420754671096802,\n                0.12790624797344208,\n                0.5830660462379456\n              ],\n              [\n                0.33873438835144043,\n                0.5909669995307922,\n                0.41624999046325684,\n                0.7529953122138977\n              ],\n              [\n                0.06628125160932541,\n                0.3420754671096802,\n                0.41624999046325684,\n                0.7529953122138977\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.3336718678474426,\n                0.5656132102012634,\n                0.41817188262939453,\n                0.7548348903656006\n              ],\n              [\n                0.33873438835144043,\n                0.5909669995307922,\n                0.41624999046325684,\n                0.7529953122138977\n              ],\n              [\n                0.3336718678474426,\n                0.5656132102012634,\n                0.41817188262939453,\n                0.7548348903656006\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000168900.jpg",\n        "coco_id": 168900,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 212940,\n          "object_id": 131141,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_168900_212940_131141_2023053_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2023053,\n          "object_id": 131141,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7388443350791931\n              ],\n              [\n                0.27892187237739563,\n                0.6338443160057068,\n                0.3210781216621399,\n                0.7727358341217041\n              ],\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7727358341217041\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.014515625312924385,\n                0.27346697449684143,\n                0.07474999874830246,\n                0.4663679301738739\n              ],\n              [\n                0.27892187237739563,\n                0.6338443160057068,\n                0.3210781216621399,\n                0.7727358341217041\n              ],\n              [\n                0.014515625312924385,\n                0.27346697449684143,\n                0.3210781216621399,\n                0.7727358341217041\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000168900.jpg",\n        "coco_id": 168900,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 212940,\n          "object_id": 131141,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_168900_212940_131141_2023053_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2023053,\n          "object_id": 131141,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.014515625312924385,\n                0.27346697449684143,\n                0.07474999874830246,\n                0.4663679301738739\n              ],\n              [\n                0.27892187237739563,\n                0.6338443160057068,\n                0.3210781216621399,\n                0.7727358341217041\n              ],\n              [\n                0.014515625312924385,\n                0.27346697449684143,\n                0.3210781216621399,\n                0.7727358341217041\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7388443350791931\n              ],\n              [\n                0.27892187237739563,\n                0.6338443160057068,\n                0.3210781216621399,\n                0.7727358341217041\n              ],\n              [\n                0.27764061093330383,\n                0.5382782816886902,\n                0.32875001430511475,\n                0.7727358341217041\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000170525.jpg",\n        "coco_id": 170525,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "knife",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2157457,\n          "object_id": 1892376,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_170525_2157457_1892376_195773_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 195773,\n          "object_id": 1892376,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.12034375220537186,\n                0.1389496773481369,\n                0.29685938358306885,\n                0.7232166528701782\n              ],\n              [\n                0.5521249771118164,\n                0.6366083025932312,\n                0.6621719002723694,\n                0.9678336977958679\n              ],\n              [\n                0.12034375220537186,\n                0.1389496773481369,\n                0.6621719002723694,\n                0.9678336977958679\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.5423593521118164,\n                0.05496717616915703,\n                0.807015597820282,\n                0.7255579829216003\n              ],\n              [\n                0.5521249771118164,\n                0.6366083025932312,\n                0.6621719002723694,\n                0.9678336977958679\n              ],\n              [\n                0.5423593521118164,\n                0.05496717616915703,\n                0.807015597820282,\n                0.9678336977958679\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000172874.jpg",\n        "coco_id": 172874,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "remote",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 445080,\n          "object_id": 1113199,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_172874_445080_1113199_1224702_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1224702,\n          "object_id": 1113199,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.6151718497276306,\n                0.21535415947437286,\n                0.7921406030654907,\n                0.9838958382606506\n              ],\n              [\n                0.7494999766349792,\n                0.25462499260902405,\n                0.7776562571525574,\n                0.2958333194255829\n              ],\n              [\n                0.6151718497276306,\n                0.21535415947437286,\n                0.7921406030654907,\n                0.9838958382606506\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.2292187511920929,\n                0.21122916042804718,\n                0.37584376335144043,\n                0.7977499961853027\n              ],\n              [\n                0.7494999766349792,\n                0.25462499260902405,\n                0.7776562571525574,\n                0.2958333194255829\n              ],\n              [\n                0.2292187511920929,\n                0.21122916042804718,\n                0.7776562571525574,\n                0.7977499961853027\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000174228.jpg",\n        "coco_id": 174228,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 187520,\n          "object_id": 1509509,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_174228_187520_1509509_1749556_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1749556,\n          "object_id": 1509509,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.435734361410141,\n                0.36584508419036865,\n                0.6483906507492065,\n                0.8345774412155151\n              ],\n              [\n                0.5505156517028809,\n                0.7507981061935425,\n                0.5814531445503235,\n                0.8185446262359619\n              ],\n              [\n                0.435734361410141,\n                0.36584508419036865,\n                0.6483906507492065,\n                0.8345774412155151\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.0,\n                0.41974177956581116,\n                0.18735937774181366,\n                0.987652599811554\n              ],\n              [\n                0.5505156517028809,\n                0.7507981061935425,\n                0.5814531445503235,\n                0.8185446262359619\n              ],\n              [\n                0.0,\n                0.41974177956581116,\n                0.5814531445503235,\n                0.987652599811554\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000175188.jpg",\n        "coco_id": 175188,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "truck",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1741700,\n          "object_id": 395017,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_175188_1741700_395017_1726048_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1726048,\n          "object_id": 395017,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a truck",\n            "bounds": [\n              [\n                0.5491406321525574,\n                0.2676442265510559,\n                0.8437187671661377,\n                0.8735576868057251\n              ],\n              [\n                0.0043750000186264515,\n                0.0022355769760906696,\n                0.5813437700271606,\n                0.8988701701164246\n              ],\n              [\n                0.0043750000186264515,\n                0.0022355769760906696,\n                0.8437187671661377,\n                0.8988701701164246\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a truck",\n            "bounds": [\n              [\n                0.7946093678474426,\n                0.36367788910865784,\n                0.9976406097412109,\n                0.9771634340286255\n              ],\n              [\n                0.0043750000186264515,\n                0.0022355769760906696,\n                0.5813437700271606,\n                0.8988701701164246\n              ],\n              [\n                0.0043750000186264515,\n                0.0022355769760906696,\n                0.9976406097412109,\n                0.9771634340286255\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000176736.jpg",\n        "coco_id": 176736,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1211433,\n          "object_id": 1481347,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_176736_1211433_1481347_1250695_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1250695,\n          "object_id": 1481347,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.13251562416553497,\n                0.6995529532432556,\n                0.550906240940094,\n                0.9013411998748779\n              ],\n              [\n                6.25000029685907e-05,\n                0.8060235381126404,\n                0.5635156035423279,\n                0.9241411685943604\n              ],\n              [\n                6.25000029685907e-05,\n                0.6995529532432556,\n                0.5635156035423279,\n                0.9241411685943604\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.6857343912124634,\n                0.49480000138282776,\n                0.9370937347412109,\n                0.7546352744102478\n              ],\n              [\n                6.25000029685907e-05,\n                0.8060235381126404,\n                0.5635156035423279,\n                0.9241411685943604\n              ],\n              [\n                6.25000029685907e-05,\n                0.49480000138282776,\n                0.9370937347412109,\n                0.9241411685943604\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000176736.jpg",\n        "coco_id": 176736,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1287582,\n          "object_id": 1481371,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_176736_1287582_1481371_1249977_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1249977,\n          "object_id": 1481371,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.3824218809604645,\n                0.6600000262260437,\n                0.7448124885559082,\n                0.8326117396354675\n              ],\n              [\n                0.2024843692779541,\n                0.7594823241233826,\n                0.7468906044960022,\n                0.8542117476463318\n              ],\n              [\n                0.2024843692779541,\n                0.6600000262260437,\n                0.7468906044960022,\n                0.8542117476463318\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.5606094002723694,\n                0.6223529577255249,\n                0.8501093983650208,\n                0.7814587950706482\n              ],\n              [\n                0.2024843692779541,\n                0.7594823241233826,\n                0.7468906044960022,\n                0.8542117476463318\n              ],\n              [\n                0.2024843692779541,\n                0.6223529577255249,\n                0.8501093983650208,\n                0.8542117476463318\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000178874.jpg",\n        "coco_id": 178874,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bottle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1735994,\n          "object_id": 1867085,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_178874_1735994_1867085_458564_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 458564,\n          "object_id": 1867085,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bottle",\n            "bounds": [\n              [\n                0.03896874934434891,\n                0.010729411616921425,\n                0.4512031376361847,\n                0.9836235046386719\n              ],\n              [\n                0.7904531359672546,\n                0.4620705842971802,\n                0.8513906002044678,\n                0.7447293996810913\n              ],\n              [\n                0.03896874934434891,\n                0.010729411616921425,\n                0.8513906002044678,\n                0.9836235046386719\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bottle",\n            "bounds": [\n              [\n                0.5105312466621399,\n                0.06745882332324982,\n                1.0,\n                0.894352912902832\n              ],\n              [\n                0.7904531359672546,\n                0.4620705842971802,\n                0.8513906002044678,\n                0.7447293996810913\n              ],\n              [\n                0.5105312466621399,\n                0.06745882332324982,\n                1.0,\n                0.894352912902832\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000179153.jpg",\n        "coco_id": 179153,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skis",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 457022,\n          "object_id": 605934,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_179153_457022_605934_2151670_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2151670,\n          "object_id": 605934,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding skis",\n            "bounds": [\n              [\n                0.4752812385559082,\n                0.453249990940094,\n                0.6067343950271606,\n                0.729645848274231\n              ],\n              [\n                0.48551562428474426,\n                0.692145824432373,\n                0.6443437337875366,\n                0.7544375061988831\n              ],\n              [\n                0.4752812385559082,\n                0.453249990940094,\n                0.6443437337875366,\n                0.7544375061988831\n              ]\n            ]\n          },\n          {\n            "text": "a person holding skis",\n            "bounds": [\n              [\n                0.9174374938011169,\n                0.4729999899864197,\n                1.0,\n                1.0\n              ],\n              [\n                0.48551562428474426,\n                0.692145824432373,\n                0.6443437337875366,\n                0.7544375061988831\n              ],\n              [\n                0.48551562428474426,\n                0.4729999899864197,\n                1.0,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000181038.jpg",\n        "coco_id": 181038,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 191181,\n          "object_id": 576634,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_181038_191181_576634_202140_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 202140,\n          "object_id": 576634,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.28220313787460327,\n                0.3033806085586548,\n                0.515375018119812,\n                0.9393144249916077\n              ],\n              [\n                0.4143906235694885,\n                0.48439717292785645,\n                0.5798906087875366,\n                0.9595271944999695\n              ],\n              [\n                0.28220313787460327,\n                0.3033806085586548,\n                0.5798906087875366,\n                0.9595271944999695\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.5086718797683716,\n                0.07420803606510162,\n                0.6352656483650208,\n                0.5600472688674927\n              ],\n              [\n                0.4143906235694885,\n                0.48439717292785645,\n                0.5798906087875366,\n                0.9595271944999695\n              ],\n              [\n                0.4143906235694885,\n                0.07420803606510162,\n                0.6352656483650208,\n                0.9595271944999695\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000181038.jpg",\n        "coco_id": 181038,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 223603,\n          "object_id": 576634,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_181038_223603_576634_202140_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 202140,\n          "object_id": 576634,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.14070312678813934,\n                0.30534279346466064,\n                0.4207812547683716,\n                0.9039952754974365\n              ],\n              [\n                0.4143906235694885,\n                0.48439717292785645,\n                0.5798906087875366,\n                0.9595271944999695\n              ],\n              [\n                0.14070312678813934,\n                0.30534279346466064,\n                0.5798906087875366,\n                0.9595271944999695\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.5086718797683716,\n                0.07420803606510162,\n                0.6352656483650208,\n                0.5600472688674927\n              ],\n              [\n                0.4143906235694885,\n                0.48439717292785645,\n                0.5798906087875366,\n                0.9595271944999695\n              ],\n              [\n                0.4143906235694885,\n                0.07420803606510162,\n                0.6352656483650208,\n                0.9595271944999695\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000181475.jpg",\n        "coco_id": 181475,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "fork",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 556336,\n          "object_id": 2103095,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_181475_556336_2103095_428521_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 428521,\n          "object_id": 2103095,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.5919687747955322,\n                0.5286029577255249,\n                1.0,\n                0.9594117403030396\n              ],\n              [\n                0.6116874814033508,\n                0.5634007453918457,\n                0.6621562242507935,\n                0.7167279124259949\n              ],\n              [\n                0.5919687747955322,\n                0.5286029577255249,\n                1.0,\n                0.9594117403030396\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.508468747138977,\n                0.25,\n                0.7022500038146973,\n                0.5457169413566589\n              ],\n              [\n                0.6116874814033508,\n                0.5634007453918457,\n                0.6621562242507935,\n                0.7167279124259949\n              ],\n              [\n                0.508468747138977,\n                0.25,\n                0.7022500038146973,\n                0.7167279124259949\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000181837.jpg",\n        "coco_id": 181837,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 204302,\n          "object_id": 127970,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_181837_204302_127970_1219659_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1219659,\n          "object_id": 127970,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.07312499731779099,\n                0.4786588251590729\n              ],\n              [\n                0.4521562457084656,\n                0.3910352885723114,\n                0.5091562271118164,\n                0.567058801651001\n              ],\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.5091562271118164,\n                0.567058801651001\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.44471874833106995,\n                0.2814352810382843,\n                0.5210312604904175,\n                0.48585882782936096\n              ],\n              [\n                0.4521562457084656,\n                0.3910352885723114,\n                0.5091562271118164,\n                0.567058801651001\n              ],\n              [\n                0.44471874833106995,\n                0.2814352810382843,\n                0.5210312604904175,\n                0.567058801651001\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000181837.jpg",\n        "coco_id": 181837,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 204302,\n          "object_id": 127970,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_181837_204302_127970_1219659_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1219659,\n          "object_id": 127970,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.07312499731779099,\n                0.4786588251590729\n              ],\n              [\n                0.4521562457084656,\n                0.3910352885723114,\n                0.5091562271118164,\n                0.567058801651001\n              ],\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.5091562271118164,\n                0.567058801651001\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.44471874833106995,\n                0.2814352810382843,\n                0.5210312604904175,\n                0.48585882782936096\n              ],\n              [\n                0.4521562457084656,\n                0.3910352885723114,\n                0.5091562271118164,\n                0.567058801651001\n              ],\n              [\n                0.44471874833106995,\n                0.2814352810382843,\n                0.5210312604904175,\n                0.567058801651001\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000181837.jpg",\n        "coco_id": 181837,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 204302,\n          "object_id": 127970,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_181837_204302_127970_1219659_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1219659,\n          "object_id": 127970,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.07312499731779099,\n                0.4786588251590729\n              ],\n              [\n                0.4521562457084656,\n                0.3910352885723114,\n                0.5091562271118164,\n                0.567058801651001\n              ],\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.5091562271118164,\n                0.567058801651001\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.44471874833106995,\n                0.2814352810382843,\n                0.5210312604904175,\n                0.48585882782936096\n              ],\n              [\n                0.4521562457084656,\n                0.3910352885723114,\n                0.5091562271118164,\n                0.567058801651001\n              ],\n              [\n                0.44471874833106995,\n                0.2814352810382843,\n                0.5210312604904175,\n                0.567058801651001\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000181837.jpg",\n        "coco_id": 181837,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 216088,\n          "object_id": 129415,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_181837_216088_129415_1219659_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1219659,\n          "object_id": 129415,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.07312499731779099,\n                0.4786588251590729\n              ],\n              [\n                0.6265624761581421,\n                0.37131765484809875,\n                0.6987968683242798,\n                0.5521647334098816\n              ],\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.6987968683242798,\n                0.5521647334098816\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.6182187795639038,\n                0.279482364654541,\n                0.6883593797683716,\n                0.5171764492988586\n              ],\n              [\n                0.6265624761581421,\n                0.37131765484809875,\n                0.6987968683242798,\n                0.5521647334098816\n              ],\n              [\n                0.6182187795639038,\n                0.279482364654541,\n                0.6987968683242798,\n                0.5521647334098816\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000181837.jpg",\n        "coco_id": 181837,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 216088,\n          "object_id": 129415,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_181837_216088_129415_1219659_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1219659,\n          "object_id": 129415,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.6182187795639038,\n                0.279482364654541,\n                0.6883593797683716,\n                0.5171764492988586\n              ],\n              [\n                0.6265624761581421,\n                0.37131765484809875,\n                0.6987968683242798,\n                0.5521647334098816\n              ],\n              [\n                0.6182187795639038,\n                0.279482364654541,\n                0.6987968683242798,\n                0.5521647334098816\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.07312499731779099,\n                0.4786588251590729\n              ],\n              [\n                0.6265624761581421,\n                0.37131765484809875,\n                0.6987968683242798,\n                0.5521647334098816\n              ],\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.6987968683242798,\n                0.5521647334098816\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000181837.jpg",\n        "coco_id": 181837,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 216088,\n          "object_id": 129415,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_181837_216088_129415_1219659_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1219659,\n          "object_id": 129415,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.6182187795639038,\n                0.279482364654541,\n                0.6883593797683716,\n                0.5171764492988586\n              ],\n              [\n                0.6265624761581421,\n                0.37131765484809875,\n                0.6987968683242798,\n                0.5521647334098816\n              ],\n              [\n                0.6182187795639038,\n                0.279482364654541,\n                0.6987968683242798,\n                0.5521647334098816\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.07312499731779099,\n                0.4786588251590729\n              ],\n              [\n                0.6265624761581421,\n                0.37131765484809875,\n                0.6987968683242798,\n                0.5521647334098816\n              ],\n              [\n                0.001500000013038516,\n                0.27865883708000183,\n                0.6987968683242798,\n                0.5521647334098816\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000001837.jpg",\n        "coco_id": 1837,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skateboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1284207,\n          "object_id": 1860644,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_1837_1284207_1860644_1252864_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1252864,\n          "object_id": 1860644,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.682200014591217,\n                0.3011331558227539,\n                0.8805199861526489,\n                0.8809065222740173\n              ],\n              [\n                0.5101000070571899,\n                0.5945042371749878,\n                0.5658000111579895,\n                0.7952691316604614\n              ],\n              [\n                0.5101000070571899,\n                0.3011331558227539,\n                0.8805199861526489,\n                0.8809065222740173\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.5054000020027161,\n                0.31728044152259827,\n                0.6188399791717529,\n                0.8056657314300537\n              ],\n              [\n                0.5101000070571899,\n                0.5945042371749878,\n                0.5658000111579895,\n                0.7952691316604614\n              ],\n              [\n                0.5054000020027161,\n                0.31728044152259827,\n                0.6188399791717529,\n                0.8056657314300537\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000001837.jpg",\n        "coco_id": 1837,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skateboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1303454,\n          "object_id": 1861614,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_1837_1303454_1861614_1309321_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1309321,\n          "object_id": 1861614,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.4119200110435486,\n                0.2891501486301422,\n                0.5155400037765503,\n                0.7843343019485474\n              ],\n              [\n                0.40742000937461853,\n                0.5879603624343872,\n                0.47088000178337097,\n                0.7929461598396301\n              ],\n              [\n                0.40742000937461853,\n                0.2891501486301422,\n                0.5155400037765503,\n                0.7929461598396301\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.1943800002336502,\n                0.37657225131988525,\n                0.3504999876022339,\n                0.8240793347358704\n              ],\n              [\n                0.40742000937461853,\n                0.5879603624343872,\n                0.47088000178337097,\n                0.7929461598396301\n              ],\n              [\n                0.1943800002336502,\n                0.37657225131988525,\n                0.47088000178337097,\n                0.8240793347358704\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000001837.jpg",\n        "coco_id": 1837,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skateboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1725698,\n          "object_id": 1860956,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_1837_1725698_1860956_1309321_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1309321,\n          "object_id": 1860956,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.1943800002336502,\n                0.37657225131988525,\n                0.3504999876022339,\n                0.8240793347358704\n              ],\n              [\n                0.15437999367713928,\n                0.5857790112495422,\n                0.21724000573158264,\n                0.7925495505332947\n              ],\n              [\n                0.15437999367713928,\n                0.37657225131988525,\n                0.3504999876022339,\n                0.8240793347358704\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.13928000628948212,\n                0.3056657314300537,\n                0.2604199945926666,\n                0.7941359877586365\n              ],\n              [\n                0.15437999367713928,\n                0.5857790112495422,\n                0.21724000573158264,\n                0.7925495505332947\n              ],\n              [\n                0.13928000628948212,\n                0.3056657314300537,\n                0.2604199945926666,\n                0.7941359877586365\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000185328.jpg",\n        "coco_id": 185328,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 225747,\n          "object_id": 601666,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_185328_225747_601666_236694_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 236694,\n          "object_id": 601666,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.32677778601646423,\n                0.2876306474208832,\n                0.47440001368522644,\n                0.8426828980445862\n              ],\n              [\n                0.34779998660087585,\n                0.29153311252593994,\n                0.3806222081184387,\n                0.36425086855888367\n              ],\n              [\n                0.32677778601646423,\n                0.2876306474208832,\n                0.47440001368522644,\n                0.8426828980445862\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.4700888991355896,\n                0.23595818877220154,\n                0.9057777523994446,\n                0.6966202259063721\n              ],\n              [\n                0.34779998660087585,\n                0.29153311252593994,\n                0.3806222081184387,\n                0.36425086855888367\n              ],\n              [\n                0.34779998660087585,\n                0.23595818877220154,\n                0.9057777523994446,\n                0.6966202259063721\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000185991.jpg",\n        "coco_id": 185991,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bottle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1311302,\n          "object_id": 288157,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_185991_1311302_288157_1202450_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1202450,\n          "object_id": 288157,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bottle",\n            "bounds": [\n              [\n                0.23207812011241913,\n                0.19750000536441803,\n                0.5788437724113464,\n                1.0\n              ],\n              [\n                0.7361875176429749,\n                0.921791672706604,\n                0.7939062714576721,\n                1.0\n              ],\n              [\n                0.23207812011241913,\n                0.19750000536441803,\n                0.7939062714576721,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bottle",\n            "bounds": [\n              [\n                0.5749375224113464,\n                0.4494999945163727,\n                0.9388437271118164,\n                1.0\n              ],\n              [\n                0.7361875176429749,\n                0.921791672706604,\n                0.7939062714576721,\n                1.0\n              ],\n              [\n                0.5749375224113464,\n                0.4494999945163727,\n                0.9388437271118164,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000188478.jpg",\n        "coco_id": 188478,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1232405,\n          "object_id": 58036,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_188478_1232405_58036_1302215_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1302215,\n          "object_id": 58036,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.7399374842643738,\n                0.049442898482084274,\n                0.8912031054496765,\n                0.6741504073143005\n              ],\n              [\n                0.48268750309944153,\n                0.14077994227409363,\n                1.0,\n                0.9819777011871338\n              ],\n              [\n                0.48268750309944153,\n                0.049442898482084274,\n                1.0,\n                0.9819777011871338\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.06089062616229057,\n                0.399136483669281,\n                0.14423437416553497,\n                0.8538718819618225\n              ],\n              [\n                0.48268750309944153,\n                0.14077994227409363,\n                1.0,\n                0.9819777011871338\n              ],\n              [\n                0.06089062616229057,\n                0.14077994227409363,\n                1.0,\n                0.9819777011871338\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000188478.jpg",\n        "coco_id": 188478,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1232405,\n          "object_id": 58036,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_188478_1232405_58036_1302215_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1302215,\n          "object_id": 58036,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.06089062616229057,\n                0.399136483669281,\n                0.14423437416553497,\n                0.8538718819618225\n              ],\n              [\n                0.48268750309944153,\n                0.14077994227409363,\n                1.0,\n                0.9819777011871338\n              ],\n              [\n                0.06089062616229057,\n                0.14077994227409363,\n                1.0,\n                0.9819777011871338\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.7399374842643738,\n                0.049442898482084274,\n                0.8912031054496765,\n                0.6741504073143005\n              ],\n              [\n                0.48268750309944153,\n                0.14077994227409363,\n                1.0,\n                0.9819777011871338\n              ],\n              [\n                0.48268750309944153,\n                0.049442898482084274,\n                1.0,\n                0.9819777011871338\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000188478.jpg",\n        "coco_id": 188478,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1232405,\n          "object_id": 58036,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_188478_1232405_58036_1314968_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1314968,\n          "object_id": 58036,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.7399374842643738,\n                0.049442898482084274,\n                0.8912031054496765,\n                0.6741504073143005\n              ],\n              [\n                0.48268750309944153,\n                0.14077994227409363,\n                1.0,\n                0.9819777011871338\n              ],\n              [\n                0.48268750309944153,\n                0.049442898482084274,\n                1.0,\n                0.9819777011871338\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.0,\n                0.4337047338485718,\n                0.060499999672174454,\n                0.8561838269233704\n              ],\n              [\n                0.48268750309944153,\n                0.14077994227409363,\n                1.0,\n                0.9819777011871338\n              ],\n              [\n                0.0,\n                0.14077994227409363,\n                1.0,\n                0.9819777011871338\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000188478.jpg",\n        "coco_id": 188478,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1233548,\n          "object_id": 274075,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_188478_1233548_274075_1293037_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1293037,\n          "object_id": 274075,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.16512499749660492,\n                0.37077993154525757,\n                0.24454687535762787,\n                0.8179665803909302\n              ],\n              [\n                0.20351561903953552,\n                0.2584958076477051,\n                0.6563281416893005,\n                0.9954596161842346\n              ],\n              [\n                0.16512499749660492,\n                0.2584958076477051,\n                0.6563281416893005,\n                0.9954596161842346\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.4040468633174896,\n                0.0972980484366417,\n                0.5285468697547913,\n                0.6696378588676453\n              ],\n              [\n                0.20351561903953552,\n                0.2584958076477051,\n                0.6563281416893005,\n                0.9954596161842346\n              ],\n              [\n                0.20351561903953552,\n                0.0972980484366417,\n                0.6563281416893005,\n                0.9954596161842346\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000188478.jpg",\n        "coco_id": 188478,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1233548,\n          "object_id": 274075,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_188478_1233548_274075_1293037_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1293037,\n          "object_id": 274075,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.4040468633174896,\n                0.0972980484366417,\n                0.5285468697547913,\n                0.6696378588676453\n              ],\n              [\n                0.20351561903953552,\n                0.2584958076477051,\n                0.6563281416893005,\n                0.9954596161842346\n              ],\n              [\n                0.20351561903953552,\n                0.0972980484366417,\n                0.6563281416893005,\n                0.9954596161842346\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.16512499749660492,\n                0.37077993154525757,\n                0.24454687535762787,\n                0.8179665803909302\n              ],\n              [\n                0.20351561903953552,\n                0.2584958076477051,\n                0.6563281416893005,\n                0.9954596161842346\n              ],\n              [\n                0.16512499749660492,\n                0.2584958076477051,\n                0.6563281416893005,\n                0.9954596161842346\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000188478.jpg",\n        "coco_id": 188478,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1233548,\n          "object_id": 274075,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_188478_1233548_274075_1314968_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1314968,\n          "object_id": 274075,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.0,\n                0.4337047338485718,\n                0.060499999672174454,\n                0.8561838269233704\n              ],\n              [\n                0.20351561903953552,\n                0.2584958076477051,\n                0.6563281416893005,\n                0.9954596161842346\n              ],\n              [\n                0.0,\n                0.2584958076477051,\n                0.6563281416893005,\n                0.9954596161842346\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.4040468633174896,\n                0.0972980484366417,\n                0.5285468697547913,\n                0.6696378588676453\n              ],\n              [\n                0.20351561903953552,\n                0.2584958076477051,\n                0.6563281416893005,\n                0.9954596161842346\n              ],\n              [\n                0.20351561903953552,\n                0.0972980484366417,\n                0.6563281416893005,\n                0.9954596161842346\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000190229.jpg",\n        "coco_id": 190229,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1211614,\n          "object_id": 1825699,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_190229_1211614_1825699_1697749_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1697749,\n          "object_id": 1825699,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.34034374356269836,\n                0.4449414610862732,\n                0.4677812457084656,\n                0.7932552695274353\n              ],\n              [\n                0.2708437442779541,\n                0.44344261288642883,\n                0.3578281104564667,\n                0.6182201504707336\n              ],\n              [\n                0.2708437442779541,\n                0.44344261288642883,\n                0.4677812457084656,\n                0.7932552695274353\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.7720780968666077,\n                0.360117107629776,\n                0.866937518119812,\n                0.9137470722198486\n              ],\n              [\n                0.2708437442779541,\n                0.44344261288642883,\n                0.3578281104564667,\n                0.6182201504707336\n              ],\n              [\n                0.2708437442779541,\n                0.360117107629776,\n                0.866937518119812,\n                0.9137470722198486\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000190229.jpg",\n        "coco_id": 190229,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1257269,\n          "object_id": 1824859,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_190229_1257269_1824859_1268108_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1268108,\n          "object_id": 1824859,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.9370625019073486,\n                0.41573771834373474,\n                1.0,\n                0.8404449820518494\n              ],\n              [\n                0.9084218740463257,\n                0.4771662652492523,\n                0.9959999918937683,\n                0.6172365546226501\n              ],\n              [\n                0.9084218740463257,\n                0.41573771834373474,\n                1.0,\n                0.8404449820518494\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.8551718592643738,\n                0.40295082330703735,\n                0.9452812671661377,\n                0.8722482323646545\n              ],\n              [\n                0.9084218740463257,\n                0.4771662652492523,\n                0.9959999918937683,\n                0.6172365546226501\n              ],\n              [\n                0.8551718592643738,\n                0.40295082330703735,\n                0.9959999918937683,\n                0.8722482323646545\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000190229.jpg",\n        "coco_id": 190229,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1697749,\n          "object_id": 1170451,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_190229_1697749_1170451_1268108_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1268108,\n          "object_id": 1170451,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.7720780968666077,\n                0.360117107629776,\n                0.866937518119812,\n                0.9137470722198486\n              ],\n              [\n                0.8164687752723694,\n                0.4400702714920044,\n                0.8879374861717224,\n                0.6108196973800659\n              ],\n              [\n                0.7720780968666077,\n                0.360117107629776,\n                0.8879374861717224,\n                0.9137470722198486\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.9370625019073486,\n                0.41573771834373474,\n                1.0,\n                0.8404449820518494\n              ],\n              [\n                0.8164687752723694,\n                0.4400702714920044,\n                0.8879374861717224,\n                0.6108196973800659\n              ],\n              [\n                0.8164687752723694,\n                0.41573771834373474,\n                1.0,\n                0.8404449820518494\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000190229.jpg",\n        "coco_id": 190229,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1716586,\n          "object_id": 1825699,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_190229_1716586_1825699_1697749_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1697749,\n          "object_id": 1825699,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.7720780968666077,\n                0.360117107629776,\n                0.866937518119812,\n                0.9137470722198486\n              ],\n              [\n                0.2708437442779541,\n                0.44344261288642883,\n                0.3578281104564667,\n                0.6182201504707336\n              ],\n              [\n                0.2708437442779541,\n                0.360117107629776,\n                0.866937518119812,\n                0.9137470722198486\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.20399999618530273,\n                0.31400468945503235,\n                0.34303125739097595,\n                0.6148946285247803\n              ],\n              [\n                0.2708437442779541,\n                0.44344261288642883,\n                0.3578281104564667,\n                0.6182201504707336\n              ],\n              [\n                0.20399999618530273,\n                0.31400468945503235,\n                0.3578281104564667,\n                0.6182201504707336\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000192955.jpg",\n        "coco_id": 192955,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 251870,\n          "object_id": 274302,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_192955_251870_274302_198243_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 198243,\n          "object_id": 274302,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.1468925178050995,\n                0.5082656145095825,\n                0.3371962606906891,\n                0.6429687738418579\n              ],\n              [\n                0.1780841052532196,\n                0.6307812333106995,\n                0.42004671692848206,\n                0.958859384059906\n              ],\n              [\n                0.1468925178050995,\n                0.5082656145095825,\n                0.42004671692848206,\n                0.958859384059906\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.5311448574066162,\n                0.22849999368190765,\n                0.7299299240112305,\n                0.4372812509536743\n              ],\n              [\n                0.1780841052532196,\n                0.6307812333106995,\n                0.42004671692848206,\n                0.958859384059906\n              ],\n              [\n                0.1780841052532196,\n                0.22849999368190765,\n                0.7299299240112305,\n                0.958859384059906\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000001947.jpg",\n        "coco_id": 1947,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 427086,\n          "object_id": 1636127,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_1947_427086_1636127_496805_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 496805,\n          "object_id": 1636127,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.6432399749755859,\n                0.11807999759912491,\n                1.0,\n                1.0\n              ],\n              [\n                0.24911999702453613,\n                0.47784000635147095,\n                0.2858999967575073,\n                0.5125333070755005\n              ],\n              [\n                0.24911999702453613,\n                0.11807999759912491,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.0,\n                0.2107200026512146,\n                0.2768400013446808,\n                0.8303999900817871\n              ],\n              [\n                0.24911999702453613,\n                0.47784000635147095,\n                0.2858999967575073,\n                0.5125333070755005\n              ],\n              [\n                0.0,\n                0.2107200026512146,\n                0.2858999967575073,\n                0.8303999900817871\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000194842.jpg",\n        "coco_id": 194842,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "cake",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 462630,\n          "object_id": 1088064,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_194842_462630_1088064_445118_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 445118,\n          "object_id": 1088064,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.20737500488758087,\n                0.0016945606330409646,\n                0.42317187786102295,\n                0.44638076424598694\n              ],\n              [\n                0.5602656006813049,\n                0.4748953878879547,\n                0.7485156059265137,\n                0.6691213250160217\n              ],\n              [\n                0.20737500488758087,\n                0.0016945606330409646,\n                0.7485156059265137,\n                0.6691213250160217\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.7670156359672546,\n                0.3617991507053375,\n                1.0,\n                0.6471966505050659\n              ],\n              [\n                0.5602656006813049,\n                0.4748953878879547,\n                0.7485156059265137,\n                0.6691213250160217\n              ],\n              [\n                0.5602656006813049,\n                0.3617991507053375,\n                1.0,\n                0.6691213250160217\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000198424.jpg",\n        "coco_id": 198424,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 218807,\n          "object_id": 147915,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_198424_218807_147915_218900_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 218900,\n          "object_id": 147915,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.4708215892314911,\n                0.43126562237739563,\n                0.5668075084686279,\n                0.6448749899864197\n              ],\n              [\n                0.18333333730697632,\n                0.46217188239097595,\n                0.6368779540061951,\n                0.7034687399864197\n              ],\n              [\n                0.18333333730697632,\n                0.43126562237739563,\n                0.6368779540061951,\n                0.7034687399864197\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.38652580976486206,\n                0.42435938119888306,\n                0.528075098991394,\n                0.6605468988418579\n              ],\n              [\n                0.18333333730697632,\n                0.46217188239097595,\n                0.6368779540061951,\n                0.7034687399864197\n              ],\n              [\n                0.18333333730697632,\n                0.42435938119888306,\n                0.6368779540061951,\n                0.7034687399864197\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000205159.jpg",\n        "coco_id": 205159,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 442694,\n          "object_id": 1184976,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_205159_442694_1184976_464680_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 464680,\n          "object_id": 1184976,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.2533906102180481,\n                0.038468748331069946,\n                0.5633437633514404,\n                0.8846094012260437\n              ],\n              [\n                0.5787500143051147,\n                0.6129843592643738,\n                0.947515606880188,\n                0.9318593740463257\n              ],\n              [\n                0.2533906102180481,\n                0.038468748331069946,\n                0.947515606880188,\n                0.9318593740463257\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.08090624958276749,\n                0.07640624791383743,\n                0.27640625834465027,\n                0.7595468759536743\n              ],\n              [\n                0.5787500143051147,\n                0.6129843592643738,\n                0.947515606880188,\n                0.9318593740463257\n              ],\n              [\n                0.08090624958276749,\n                0.07640624791383743,\n                0.947515606880188,\n                0.9318593740463257\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000205159.jpg",\n        "coco_id": 205159,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 442694,\n          "object_id": 1445332,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_205159_442694_1445332_464680_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 464680,\n          "object_id": 1445332,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.08090624958276749,\n                0.07640624791383743,\n                0.27640625834465027,\n                0.7595468759536743\n              ],\n              [\n                0.5093125104904175,\n                0.528124988079071,\n                0.7570156455039978,\n                0.7064062356948853\n              ],\n              [\n                0.08090624958276749,\n                0.07640624791383743,\n                0.7570156455039978,\n                0.7595468759536743\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.2533906102180481,\n                0.038468748331069946,\n                0.5633437633514404,\n                0.8846094012260437\n              ],\n              [\n                0.5093125104904175,\n                0.528124988079071,\n                0.7570156455039978,\n                0.7064062356948853\n              ],\n              [\n                0.2533906102180481,\n                0.038468748331069946,\n                0.7570156455039978,\n                0.8846094012260437\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000207647.jpg",\n        "coco_id": 207647,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 447450,\n          "object_id": 627658,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_207647_447450_627658_449645_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 449645,\n          "object_id": 627658,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.2562745213508606,\n                0.0696675181388855,\n                0.8334313631057739,\n                0.8763938546180725\n              ],\n              [\n                0.4845833480358124,\n                0.2658056318759918,\n                0.6122058629989624,\n                0.33125320076942444\n              ],\n              [\n                0.2562745213508606,\n                0.0696675181388855,\n                0.8334313631057739,\n                0.8763938546180725\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.6568382382392883,\n                0.020230179652571678,\n                0.9066421389579773,\n                0.7168542146682739\n              ],\n              [\n                0.4845833480358124,\n                0.2658056318759918,\n                0.6122058629989624,\n                0.33125320076942444\n              ],\n              [\n                0.4845833480358124,\n                0.020230179652571678,\n                0.9066421389579773,\n                0.7168542146682739\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000207734.jpg",\n        "coco_id": 207734,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 450936,\n          "object_id": 632288,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_207734_450936_632288_2204826_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2204826,\n          "object_id": 632288,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.5357187390327454,\n                0.2722916603088379,\n                0.8142812252044678,\n                0.8307291865348816\n              ],\n              [\n                0.5268750190734863,\n                0.5372083187103271,\n                0.7493593692779541,\n                0.7355833053588867\n              ],\n              [\n                0.5268750190734863,\n                0.2722916603088379,\n                0.8142812252044678,\n                0.8307291865348816\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.2292187511920929,\n                0.17079167068004608,\n                0.5325781106948853,\n                0.9011250138282776\n              ],\n              [\n                0.5268750190734863,\n                0.5372083187103271,\n                0.7493593692779541,\n                0.7355833053588867\n              ],\n              [\n                0.2292187511920929,\n                0.17079167068004608,\n                0.7493593692779541,\n                0.9011250138282776\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000210472.jpg",\n        "coco_id": 210472,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "laptop",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1283570,\n          "object_id": 1102709,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_210472_1283570_1102709_1232809_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1232809,\n          "object_id": 1102709,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a laptop",\n            "bounds": [\n              [\n                0.41449999809265137,\n                0.01344467606395483,\n                0.8759687542915344,\n                0.869958221912384\n              ],\n              [\n                0.3421874940395355,\n                0.7421085834503174,\n                0.8293437361717224,\n                0.9290605187416077\n              ],\n              [\n                0.3421874940395355,\n                0.01344467606395483,\n                0.8759687542915344,\n                0.9290605187416077\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a laptop",\n            "bounds": [\n              [\n                0.06742187589406967,\n                0.3457202613353729,\n                0.38939061760902405,\n                1.0\n              ],\n              [\n                0.3421874940395355,\n                0.7421085834503174,\n                0.8293437361717224,\n                0.9290605187416077\n              ],\n              [\n                0.06742187589406967,\n                0.3457202613353729,\n                0.8293437361717224,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000210751.jpg",\n        "coco_id": 210751,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "donut",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 215917,\n          "object_id": 1078528,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_210751_215917_1078528_183193_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 183193,\n          "object_id": 1078528,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a donut",\n            "bounds": [\n              [\n                0.0,\n                0.024453332647681236,\n                0.5023999810218811,\n                0.9866399765014648\n              ],\n              [\n                0.3514400124549866,\n                0.38365334272384644,\n                0.48201999068260193,\n                0.5342666506767273\n              ],\n              [\n                0.0,\n                0.024453332647681236,\n                0.5023999810218811,\n                0.9866399765014648\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a donut",\n            "bounds": [\n              [\n                0.5455999970436096,\n                0.2736533284187317,\n                0.9780399799346924,\n                0.9898666739463806\n              ],\n              [\n                0.3514400124549866,\n                0.38365334272384644,\n                0.48201999068260193,\n                0.5342666506767273\n              ],\n              [\n                0.3514400124549866,\n                0.2736533284187317,\n                0.9780399799346924,\n                0.9898666739463806\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000211205.jpg",\n        "coco_id": 211205,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1236480,\n          "object_id": 1445610,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_211205_1236480_1445610_2166570_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2166570,\n          "object_id": 1445610,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.5460655689239502,\n                0.3262031376361847,\n                0.9565105438232422,\n                0.9774062633514404\n              ],\n              [\n                0.045761123299598694,\n                0.7095937728881836,\n                0.5110772848129272,\n                0.9785468578338623\n              ],\n              [\n                0.045761123299598694,\n                0.3262031376361847,\n                0.9565105438232422,\n                0.9785468578338623\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.3102107644081116,\n                0.5369062423706055,\n                0.6312177777290344,\n                0.9137656092643738\n              ],\n              [\n                0.045761123299598694,\n                0.7095937728881836,\n                0.5110772848129272,\n                0.9785468578338623\n              ],\n              [\n                0.045761123299598694,\n                0.5369062423706055,\n                0.6312177777290344,\n                0.9785468578338623\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000215732.jpg",\n        "coco_id": 215732,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 464259,\n          "object_id": 602200,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_215732_464259_602200_521558_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 521558,\n          "object_id": 602200,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.2602187395095825,\n                0.10434272140264511,\n                0.4913593828678131,\n                0.9543427228927612\n              ],\n              [\n                0.4168125092983246,\n                0.4070657193660736,\n                0.4875468611717224,\n                0.5491549372673035\n              ],\n              [\n                0.2602187395095825,\n                0.10434272140264511,\n                0.4913593828678131,\n                0.9543427228927612\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.7387499809265137,\n                0.020070422440767288,\n                0.9984375238418579,\n                0.970751166343689\n              ],\n              [\n                0.4168125092983246,\n                0.4070657193660736,\n                0.4875468611717224,\n                0.5491549372673035\n              ],\n              [\n                0.4168125092983246,\n                0.020070422440767288,\n                0.9984375238418579,\n                0.970751166343689\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000216059.jpg",\n        "coco_id": 216059,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 238229,\n          "object_id": 1396851,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_216059_238229_1396851_201187_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 201187,\n          "object_id": 1396851,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.5271562337875366,\n                0.4258541762828827,\n                0.6021562218666077,\n                0.6318541765213013\n              ],\n              [\n                0.8413749933242798,\n                0.5542916655540466,\n                0.947015643119812,\n                0.6491666436195374\n              ],\n              [\n                0.5271562337875366,\n                0.4258541762828827,\n                0.947015643119812,\n                0.6491666436195374\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.8031874895095825,\n                0.4936041533946991,\n                0.8969218730926514,\n                0.6385625004768372\n              ],\n              [\n                0.8413749933242798,\n                0.5542916655540466,\n                0.947015643119812,\n                0.6491666436195374\n              ],\n              [\n                0.8031874895095825,\n                0.4936041533946991,\n                0.947015643119812,\n                0.6491666436195374\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000216899.jpg",\n        "coco_id": 216899,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "knife",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 478733,\n          "object_id": 1892214,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_216899_478733_1892214_446936_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 446936,\n          "object_id": 1892214,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.0,\n                0.12809374928474426,\n                0.31235936284065247,\n                0.6269687414169312\n              ],\n              [\n                0.3858281373977661,\n                0.5945937633514404,\n                0.4462343752384186,\n                0.6563281416893005\n              ],\n              [\n                0.0,\n                0.12809374928474426,\n                0.4462343752384186,\n                0.6563281416893005\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.33809375762939453,\n                0.12376562505960464,\n                0.7859531044960022,\n                0.6339531540870667\n              ],\n              [\n                0.3858281373977661,\n                0.5945937633514404,\n                0.4462343752384186,\n                0.6563281416893005\n              ],\n              [\n                0.33809375762939453,\n                0.12376562505960464,\n                0.7859531044960022,\n                0.6563281416893005\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000217653.jpg",\n        "coco_id": 217653,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1287418,\n          "object_id": 659234,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_217653_1287418_659234_516400_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 516400,\n          "object_id": 659234,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.22139061987400055,\n                0.00885245855897665,\n                0.4628593623638153,\n                0.8888524770736694\n              ],\n              [\n                0.4138124883174896,\n                0.28248244524002075,\n                0.5982187390327454,\n                0.4038173258304596\n              ],\n              [\n                0.22139061987400055,\n                0.00885245855897665,\n                0.5982187390327454,\n                0.8888524770736694\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.40931248664855957,\n                0.0025058547034859657,\n                0.6072187423706055,\n                0.6449414491653442\n              ],\n              [\n                0.4138124883174896,\n                0.28248244524002075,\n                0.5982187390327454,\n                0.4038173258304596\n              ],\n              [\n                0.40931248664855957,\n                0.0025058547034859657,\n                0.6072187423706055,\n                0.6449414491653442\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000021983.jpg",\n        "coco_id": 21983,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "sports ball",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 547193,\n          "object_id": 301439,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_21983_547193_301439_538264_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 538264,\n          "object_id": 301439,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a sports ball",\n            "bounds": [\n              [\n                0.45231249928474426,\n                0.1396487057209015,\n                0.8655468821525574,\n                0.9955034852027893\n              ],\n              [\n                0.7390156388282776,\n                0.4299531579017639,\n                0.845828115940094,\n                0.5462061166763306\n              ],\n              [\n                0.45231249928474426,\n                0.1396487057209015,\n                0.8655468821525574,\n                0.9955034852027893\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a sports ball",\n            "bounds": [\n              [\n                0.0,\n                0.22557376325130463,\n                0.39295312762260437,\n                0.9993442893028259\n              ],\n              [\n                0.7390156388282776,\n                0.4299531579017639,\n                0.845828115940094,\n                0.5462061166763306\n              ],\n              [\n                0.0,\n                0.22557376325130463,\n                0.845828115940094,\n                0.9993442893028259\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000021983.jpg",\n        "coco_id": 21983,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "sports ball",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 547193,\n          "object_id": 301439,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_21983_547193_301439_538264_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 538264,\n          "object_id": 301439,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.45231249928474426,\n                0.1396487057209015,\n                0.8655468821525574,\n                0.9955034852027893\n              ],\n              [\n                0.7390156388282776,\n                0.4299531579017639,\n                0.845828115940094,\n                0.5462061166763306\n              ],\n              [\n                0.45231249928474426,\n                0.1396487057209015,\n                0.8655468821525574,\n                0.9955034852027893\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.0,\n                0.22557376325130463,\n                0.39295312762260437,\n                0.9993442893028259\n              ],\n              [\n                0.7390156388282776,\n                0.4299531579017639,\n                0.845828115940094,\n                0.5462061166763306\n              ],\n              [\n                0.0,\n                0.22557376325130463,\n                0.845828115940094,\n                0.9993442893028259\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000220152.jpg",\n        "coco_id": 220152,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 426184,\n          "object_id": 1165867,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_220152_426184_1165867_425986_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 425986,\n          "object_id": 1165867,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.6399531364440918,\n                0.35610416531562805,\n                0.794015645980835,\n                0.8301458358764648\n              ],\n              [\n                0.6817187666893005,\n                0.36270833015441895,\n                0.8021406531333923,\n                0.5521458387374878\n              ],\n              [\n                0.6399531364440918,\n                0.35610416531562805,\n                0.8021406531333923,\n                0.8301458358764648\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.27809375524520874,\n                0.2539375126361847,\n                0.38257813453674316,\n                0.703374981880188\n              ],\n              [\n                0.6817187666893005,\n                0.36270833015441895,\n                0.8021406531333923,\n                0.5521458387374878\n              ],\n              [\n                0.27809375524520874,\n                0.2539375126361847,\n                0.8021406531333923,\n                0.703374981880188\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000220152.jpg",\n        "coco_id": 220152,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 426184,\n          "object_id": 1165867,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_220152_426184_1165867_425986_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 425986,\n          "object_id": 1165867,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.6399531364440918,\n                0.35610416531562805,\n                0.794015645980835,\n                0.8301458358764648\n              ],\n              [\n                0.6817187666893005,\n                0.36270833015441895,\n                0.8021406531333923,\n                0.5521458387374878\n              ],\n              [\n                0.6399531364440918,\n                0.35610416531562805,\n                0.8021406531333923,\n                0.8301458358764648\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.27809375524520874,\n                0.2539375126361847,\n                0.38257813453674316,\n                0.703374981880188\n              ],\n              [\n                0.6817187666893005,\n                0.36270833015441895,\n                0.8021406531333923,\n                0.5521458387374878\n              ],\n              [\n                0.27809375524520874,\n                0.2539375126361847,\n                0.8021406531333923,\n                0.703374981880188\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000220772.jpg",\n        "coco_id": 220772,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1240903,\n          "object_id": 605235,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_220772_1240903_605235_1693657_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1693657,\n          "object_id": 605235,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.10849999636411667,\n                0.030030030757188797,\n                0.34766000509262085,\n                0.8893693685531616\n              ],\n              [\n                0.4420199990272522,\n                0.27876877784729004,\n                0.5443000197410583,\n                0.31105104088783264\n              ],\n              [\n                0.10849999636411667,\n                0.030030030757188797,\n                0.5443000197410583,\n                0.8893693685531616\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.296999990940094,\n                0.0439339354634285,\n                0.5759999752044678,\n                0.9470871090888977\n              ],\n              [\n                0.4420199990272522,\n                0.27876877784729004,\n                0.5443000197410583,\n                0.31105104088783264\n              ],\n              [\n                0.296999990940094,\n                0.0439339354634285,\n                0.5759999752044678,\n                0.9470871090888977\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000226298.jpg",\n        "coco_id": 226298,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 478003,\n          "object_id": 633592,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_226298_478003_633592_443113_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 443113,\n          "object_id": 633592,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.36061999201774597,\n                0.5749152302742004,\n                0.9387000203132629,\n                0.7838982939720154\n              ],\n              [\n                0.3942599892616272,\n                0.3782033920288086,\n                0.4572800099849701,\n                0.47444066405296326\n              ],\n              [\n                0.36061999201774597,\n                0.3782033920288086,\n                0.9387000203132629,\n                0.7838982939720154\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.09573999792337418,\n                0.20898304879665375,\n                0.40318000316619873,\n                0.790372908115387\n              ],\n              [\n                0.3942599892616272,\n                0.3782033920288086,\n                0.4572800099849701,\n                0.47444066405296326\n              ],\n              [\n                0.09573999792337418,\n                0.20898304879665375,\n                0.4572800099849701,\n                0.790372908115387\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000227109.jpg",\n        "coco_id": 227109,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1687934,\n          "object_id": 2232055,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_227109_1687934_2232055_1226226_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1226226,\n          "object_id": 2232055,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.17087499797344208,\n                0.6387074589729309,\n                0.31882813572883606,\n                1.0\n              ],\n              [\n                0.6068437695503235,\n                0.23884353041648865,\n                0.6551562547683716,\n                0.3034467101097107\n              ],\n              [\n                0.17087499797344208,\n                0.23884353041648865,\n                0.6551562547683716,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.5285000205039978,\n                0.23832198977470398,\n                0.6546093821525574,\n                0.8345351219177246\n              ],\n              [\n                0.6068437695503235,\n                0.23884353041648865,\n                0.6551562547683716,\n                0.3034467101097107\n              ],\n              [\n                0.5285000205039978,\n                0.23832198977470398,\n                0.6551562547683716,\n                0.8345351219177246\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000227109.jpg",\n        "coco_id": 227109,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1693931,\n          "object_id": 2232054,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_227109_1693931_2232054_1226226_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1226226,\n          "object_id": 2232054,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.17087499797344208,\n                0.6387074589729309,\n                0.31882813572883606,\n                1.0\n              ],\n              [\n                0.8960781097412109,\n                0.49582767486572266,\n                0.9345625042915344,\n                0.5341496467590332\n              ],\n              [\n                0.17087499797344208,\n                0.49582767486572266,\n                0.9345625042915344,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.8280781507492065,\n                0.29832199215888977,\n                0.9401562213897705,\n                0.5683220028877258\n              ],\n              [\n                0.8960781097412109,\n                0.49582767486572266,\n                0.9345625042915344,\n                0.5341496467590332\n              ],\n              [\n                0.8280781507492065,\n                0.29832199215888977,\n                0.9401562213897705,\n                0.5683220028877258\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000231019.jpg",\n        "coco_id": 231019,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 526738,\n          "object_id": 1435531,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_231019_526738_1435531_539003_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 539003,\n          "object_id": 1435531,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.27306249737739563,\n                0.5788207650184631,\n                0.8251405954360962,\n                1.0\n              ],\n              [\n                0.15990625321865082,\n                0.3785141408443451,\n                0.2993437647819519,\n                0.566674530506134\n              ],\n              [\n                0.15990625321865082,\n                0.3785141408443451,\n                0.8251405954360962,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.12484374642372131,\n                0.08375000208616257,\n                0.28914061188697815,\n                0.7540094256401062\n              ],\n              [\n                0.15990625321865082,\n                0.3785141408443451,\n                0.2993437647819519,\n                0.566674530506134\n              ],\n              [\n                0.12484374642372131,\n                0.08375000208616257,\n                0.2993437647819519,\n                0.7540094256401062\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000023127.jpg",\n        "coco_id": 23127,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "sandwich",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 478176,\n          "object_id": 310440,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_23127_478176_310440_1730622_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1730622,\n          "object_id": 310440,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a sandwich",\n            "bounds": [\n              [\n                0.14326561987400055,\n                0.640458345413208,\n                0.7011250257492065,\n                0.9887708425521851\n              ],\n              [\n                0.1661093682050705,\n                0.11675000190734863,\n                0.8859062790870667,\n                0.6290624737739563\n              ],\n              [\n                0.14326561987400055,\n                0.11675000190734863,\n                0.8859062790870667,\n                0.9887708425521851\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a sandwich",\n            "bounds": [\n              [\n                0.001687500043772161,\n                0.003375000087544322,\n                0.39189061522483826,\n                0.8006666898727417\n              ],\n              [\n                0.1661093682050705,\n                0.11675000190734863,\n                0.8859062790870667,\n                0.6290624737739563\n              ],\n              [\n                0.001687500043772161,\n                0.003375000087544322,\n                0.8859062790870667,\n                0.8006666898727417\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000234671.jpg",\n        "coco_id": 234671,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "pizza",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 428524,\n          "object_id": 1923395,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_234671_428524_1923395_424057_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 424057,\n          "object_id": 1923395,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.0033802816178649664,\n                0.0494375005364418,\n                0.4996478855609894,\n                0.5483124852180481\n              ],\n              [\n                0.3389436602592468,\n                0.6369062662124634,\n                0.4799061119556427,\n                0.698171854019165\n              ],\n              [\n                0.0033802816178649664,\n                0.0494375005364418,\n                0.4996478855609894,\n                0.698171854019165\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.4355164170265198,\n                0.0018749999580904841,\n                0.9959389567375183,\n                0.6917656064033508\n              ],\n              [\n                0.3389436602592468,\n                0.6369062662124634,\n                0.4799061119556427,\n                0.698171854019165\n              ],\n              [\n                0.3389436602592468,\n                0.0018749999580904841,\n                0.9959389567375183,\n                0.698171854019165\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000234671.jpg",\n        "coco_id": 234671,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "knife",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 428524,\n          "object_id": 699235,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_234671_428524_699235_424057_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 424057,\n          "object_id": 699235,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.4355164170265198,\n                0.0018749999580904841,\n                0.9959389567375183,\n                0.6917656064033508\n              ],\n              [\n                0.436784029006958,\n                0.5049999952316284,\n                0.5490140914916992,\n                0.6486093997955322\n              ],\n              [\n                0.4355164170265198,\n                0.0018749999580904841,\n                0.9959389567375183,\n                0.6917656064033508\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.0033802816178649664,\n                0.0494375005364418,\n                0.4996478855609894,\n                0.5483124852180481\n              ],\n              [\n                0.436784029006958,\n                0.5049999952316284,\n                0.5490140914916992,\n                0.6486093997955322\n              ],\n              [\n                0.0033802816178649664,\n                0.0494375005364418,\n                0.5490140914916992,\n                0.6486093997955322\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000236057.jpg",\n        "coco_id": 236057,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bed",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 437583,\n          "object_id": 314513,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_236057_437583_314513_427558_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 427558,\n          "object_id": 314513,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bed",\n            "bounds": [\n              [\n                0.07176562398672104,\n                0.10077083110809326,\n                0.38093748688697815,\n                0.8695833086967468\n              ],\n              [\n                0.04214062541723251,\n                0.343812495470047,\n                0.930343747138977,\n                0.9887708425521851\n              ],\n              [\n                0.04214062541723251,\n                0.10077083110809326,\n                0.930343747138977,\n                0.9887708425521851\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bed",\n            "bounds": [\n              [\n                0.3792187571525574,\n                0.31460416316986084,\n                0.7466250061988831,\n                0.9910208582878113\n              ],\n              [\n                0.04214062541723251,\n                0.343812495470047,\n                0.930343747138977,\n                0.9887708425521851\n              ],\n              [\n                0.04214062541723251,\n                0.31460416316986084,\n                0.930343747138977,\n                0.9910208582878113\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000023639.jpg",\n        "coco_id": 23639,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "sports ball",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 508464,\n          "object_id": 307645,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_23639_508464_307645_517957_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 517957,\n          "object_id": 307645,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.14673437178134918,\n                0.21424999833106995,\n                0.7434218525886536,\n                1.0\n              ],\n              [\n                0.6353124976158142,\n                0.2472916692495346,\n                0.6658281087875366,\n                0.2956874966621399\n              ],\n              [\n                0.14673437178134918,\n                0.21424999833106995,\n                0.7434218525886536,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.5342656373977661,\n                0.0,\n                1.0,\n                0.5797708630561829\n              ],\n              [\n                0.6353124976158142,\n                0.2472916692495346,\n                0.6658281087875366,\n                0.2956874966621399\n              ],\n              [\n                0.5342656373977661,\n                0.0,\n                1.0,\n                0.5797708630561829\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000237467.jpg",\n        "coco_id": 237467,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 421618,\n          "object_id": 321847,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_237467_421618_321847_444801_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 444801,\n          "object_id": 321847,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.48249998688697815,\n                0.2545066773891449,\n                0.8973000049591064,\n                0.9881600141525269\n              ],\n              [\n                0.29732000827789307,\n                0.37834665179252625,\n                0.33748000860214233,\n                0.4172533452510834\n              ],\n              [\n                0.29732000827789307,\n                0.2545066773891449,\n                0.8973000049591064,\n                0.9881600141525269\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.2039400041103363,\n                0.2760266661643982,\n                0.47360000014305115,\n                0.9883999824523926\n              ],\n              [\n                0.29732000827789307,\n                0.37834665179252625,\n                0.33748000860214233,\n                0.4172533452510834\n              ],\n              [\n                0.2039400041103363,\n                0.2760266661643982,\n                0.47360000014305115,\n                0.9883999824523926\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000237467.jpg",\n        "coco_id": 237467,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 421618,\n          "object_id": 321847,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_237467_421618_321847_444801_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 444801,\n          "object_id": 321847,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.2039400041103363,\n                0.2760266661643982,\n                0.47360000014305115,\n                0.9883999824523926\n              ],\n              [\n                0.29732000827789307,\n                0.37834665179252625,\n                0.33748000860214233,\n                0.4172533452510834\n              ],\n              [\n                0.2039400041103363,\n                0.2760266661643982,\n                0.47360000014305115,\n                0.9883999824523926\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.48249998688697815,\n                0.2545066773891449,\n                0.8973000049591064,\n                0.9881600141525269\n              ],\n              [\n                0.29732000827789307,\n                0.37834665179252625,\n                0.33748000860214233,\n                0.4172533452510834\n              ],\n              [\n                0.29732000827789307,\n                0.2545066773891449,\n                0.8973000049591064,\n                0.9881600141525269\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000238709.jpg",\n        "coco_id": 238709,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 471833,\n          "object_id": 627378,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_238709_471833_627378_449269_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 449269,\n          "object_id": 627378,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.04053125157952309,\n                0.3213513493537903,\n                0.25099998712539673,\n                0.914617121219635\n              ],\n              [\n                0.5331718921661377,\n                0.09887387603521347,\n                0.6095625162124634,\n                0.2966216206550598\n              ],\n              [\n                0.04053125157952309,\n                0.09887387603521347,\n                0.6095625162124634,\n                0.914617121219635\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.463015615940094,\n                0.20450450479984283,\n                0.8247030973434448,\n                0.8404504656791687\n              ],\n              [\n                0.5331718921661377,\n                0.09887387603521347,\n                0.6095625162124634,\n                0.2966216206550598\n              ],\n              [\n                0.463015615940094,\n                0.09887387603521347,\n                0.8247030973434448,\n                0.8404504656791687\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000243176.jpg",\n        "coco_id": 243176,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1247605,\n          "object_id": 602585,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_243176_1247605_602585_1288968_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1288968,\n          "object_id": 602585,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.10878124833106995,\n                0.09880281984806061,\n                0.9015468955039978,\n                0.9772534966468811\n              ],\n              [\n                0.5639062523841858,\n                0.5723474025726318,\n                0.6850625276565552,\n                0.7206572890281677\n              ],\n              [\n                0.10878124833106995,\n                0.09880281984806061,\n                0.9015468955039978,\n                0.9772534966468811\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.34296876192092896,\n                0.23727700114250183,\n                0.6847968697547913,\n                1.0\n              ],\n              [\n                0.5639062523841858,\n                0.5723474025726318,\n                0.6850625276565552,\n                0.7206572890281677\n              ],\n              [\n                0.34296876192092896,\n                0.23727700114250183,\n                0.6850625276565552,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000245118.jpg",\n        "coco_id": 245118,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1217130,\n          "object_id": 323654,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_245118_1217130_323654_1293817_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1293817,\n          "object_id": 323654,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.7863593697547913,\n                0.27488282322883606,\n                0.9952968955039978,\n                0.958691418170929\n              ],\n              [\n                0.8567031025886536,\n                0.33433592319488525,\n                0.9277187585830688,\n                0.4066796898841858\n              ],\n              [\n                0.7863593697547913,\n                0.27488282322883606,\n                0.9952968955039978,\n                0.958691418170929\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.0,\n                0.20673827826976776,\n                0.24809375405311584,\n                0.9842773675918579\n              ],\n              [\n                0.8567031025886536,\n                0.33433592319488525,\n                0.9277187585830688,\n                0.4066796898841858\n              ],\n              [\n                0.0,\n                0.20673827826976776,\n                0.9277187585830688,\n                0.9842773675918579\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000245118.jpg",\n        "coco_id": 245118,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1217130,\n          "object_id": 323654,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_245118_1217130_323654_1742185_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1742185,\n          "object_id": 323654,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.19762499630451202,\n                0.31117188930511475,\n                0.4900781214237213,\n                1.0\n              ],\n              [\n                0.8567031025886536,\n                0.33433592319488525,\n                0.9277187585830688,\n                0.4066796898841858\n              ],\n              [\n                0.19762499630451202,\n                0.31117188930511475,\n                0.9277187585830688,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.7863593697547913,\n                0.27488282322883606,\n                0.9952968955039978,\n                0.958691418170929\n              ],\n              [\n                0.8567031025886536,\n                0.33433592319488525,\n                0.9277187585830688,\n                0.4066796898841858\n              ],\n              [\n                0.7863593697547913,\n                0.27488282322883606,\n                0.9952968955039978,\n                0.958691418170929\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000246980.jpg",\n        "coco_id": 246980,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "sports ball",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 424762,\n          "object_id": 300977,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_246980_424762_300977_459742_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 459742,\n          "object_id": 300977,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a sports ball",\n            "bounds": [\n              [\n                0.32314062118530273,\n                0.1864553987979889,\n                0.5672031044960022,\n                0.6508215665817261\n              ],\n              [\n                0.4451562464237213,\n                0.27025821805000305,\n                0.49842187762260437,\n                0.3773708939552307\n              ],\n              [\n                0.32314062118530273,\n                0.1864553987979889,\n                0.5672031044960022,\n                0.6508215665817261\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a sports ball",\n            "bounds": [\n              [\n                0.47715625166893005,\n                0.1596948355436325,\n                0.7419062256813049,\n                0.7934037446975708\n              ],\n              [\n                0.4451562464237213,\n                0.27025821805000305,\n                0.49842187762260437,\n                0.3773708939552307\n              ],\n              [\n                0.4451562464237213,\n                0.1596948355436325,\n                0.7419062256813049,\n                0.7934037446975708\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000246980.jpg",\n        "coco_id": 246980,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "sports ball",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 424762,\n          "object_id": 300977,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_246980_424762_300977_459742_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 459742,\n          "object_id": 300977,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.32314062118530273,\n                0.1864553987979889,\n                0.5672031044960022,\n                0.6508215665817261\n              ],\n              [\n                0.4451562464237213,\n                0.27025821805000305,\n                0.49842187762260437,\n                0.3773708939552307\n              ],\n              [\n                0.32314062118530273,\n                0.1864553987979889,\n                0.5672031044960022,\n                0.6508215665817261\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.47715625166893005,\n                0.1596948355436325,\n                0.7419062256813049,\n                0.7934037446975708\n              ],\n              [\n                0.4451562464237213,\n                0.27025821805000305,\n                0.49842187762260437,\n                0.3773708939552307\n              ],\n              [\n                0.4451562464237213,\n                0.1596948355436325,\n                0.7419062256813049,\n                0.7934037446975708\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000248052.jpg",\n        "coco_id": 248052,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1254111,\n          "object_id": 1584016,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_248052_1254111_1584016_1260835_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1260835,\n          "object_id": 1584016,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.15337499976158142,\n                0.056187499314546585,\n                0.3792187571525574,\n                0.9865208268165588\n              ],\n              [\n                0.629281222820282,\n                0.5295833349227905,\n                0.7942187786102295,\n                0.940500020980835\n              ],\n              [\n                0.15337499976158142,\n                0.056187499314546585,\n                0.7942187786102295,\n                0.9865208268165588\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.6084218621253967,\n                0.2966249883174896,\n                0.9353906512260437,\n                0.8696666955947876\n              ],\n              [\n                0.629281222820282,\n                0.5295833349227905,\n                0.7942187786102295,\n                0.940500020980835\n              ],\n              [\n                0.6084218621253967,\n                0.2966249883174896,\n                0.9353906512260437,\n                0.940500020980835\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000249435.jpg",\n        "coco_id": 249435,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "fork",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 465111,\n          "object_id": 689715,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_249435_465111_689715_498692_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 498692,\n          "object_id": 689715,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.17159375548362732,\n                0.0,\n                0.4481562376022339,\n                0.4189999997615814\n              ],\n              [\n                0.31315624713897705,\n                0.26887500286102295,\n                0.4039531350135803,\n                0.382645845413208\n              ],\n              [\n                0.17159375548362732,\n                0.0,\n                0.4481562376022339,\n                0.4189999997615814\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.37415623664855957,\n                0.11685416847467422,\n                0.8292187452316284,\n                0.48089584708213806\n              ],\n              [\n                0.31315624713897705,\n                0.26887500286102295,\n                0.4039531350135803,\n                0.382645845413208\n              ],\n              [\n                0.31315624713897705,\n                0.11685416847467422,\n                0.8292187452316284,\n                0.48089584708213806\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000250026.jpg",\n        "coco_id": 250026,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "snowboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2157536,\n          "object_id": 620869,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_250026_2157536_620869_492102_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 492102,\n          "object_id": 620869,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.29176563024520874,\n                0.8028749823570251,\n                0.3695937395095825,\n                0.9357083439826965\n              ],\n              [\n                0.44626563787460327,\n                0.78885418176651,\n                0.48564061522483826,\n                0.9427083134651184\n              ],\n              [\n                0.29176563024520874,\n                0.78885418176651,\n                0.48564061522483826,\n                0.9427083134651184\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.45985937118530273,\n                0.7498124837875366,\n                0.5194687247276306,\n                0.957854151725769\n              ],\n              [\n                0.44626563787460327,\n                0.78885418176651,\n                0.48564061522483826,\n                0.9427083134651184\n              ],\n              [\n                0.44626563787460327,\n                0.7498124837875366,\n                0.5194687247276306,\n                0.957854151725769\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000250192.jpg",\n        "coco_id": 250192,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "elephant",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 511633,\n          "object_id": 581568,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_250192_511633_581568_474919_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 474919,\n          "object_id": 581568,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.30268749594688416,\n                0.36125001311302185,\n                0.3720625042915344,\n                0.6460000276565552\n              ],\n              [\n                0.4240781366825104,\n                0.16472221910953522,\n                0.7532968521118164,\n                0.8117499947547913\n              ],\n              [\n                0.30268749594688416,\n                0.16472221910953522,\n                0.7532968521118164,\n                0.8117499947547913\n              ]\n            ]\n          },\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.44487500190734863,\n                0.0016666667070239782,\n                0.5784531235694885,\n                0.33319443464279175\n              ],\n              [\n                0.4240781366825104,\n                0.16472221910953522,\n                0.7532968521118164,\n                0.8117499947547913\n              ],\n              [\n                0.4240781366825104,\n                0.0016666667070239782,\n                0.7532968521118164,\n                0.8117499947547913\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000250192.jpg",\n        "coco_id": 250192,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "elephant",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 511633,\n          "object_id": 581568,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_250192_511633_581568_474919_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 474919,\n          "object_id": 581568,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on an elephant",\n            "bounds": [\n              [\n                0.44487500190734863,\n                0.0016666667070239782,\n                0.5784531235694885,\n                0.33319443464279175\n              ],\n              [\n                0.4240781366825104,\n                0.16472221910953522,\n                0.7532968521118164,\n                0.8117499947547913\n              ],\n              [\n                0.4240781366825104,\n                0.0016666667070239782,\n                0.7532968521118164,\n                0.8117499947547913\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on an elephant",\n            "bounds": [\n              [\n                0.30268749594688416,\n                0.36125001311302185,\n                0.3720625042915344,\n                0.6460000276565552\n              ],\n              [\n                0.4240781366825104,\n                0.16472221910953522,\n                0.7532968521118164,\n                0.8117499947547913\n              ],\n              [\n                0.30268749594688416,\n                0.16472221910953522,\n                0.7532968521118164,\n                0.8117499947547913\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000250749.jpg",\n        "coco_id": 250749,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "knife",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 488963,\n          "object_id": 695993,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_250749_488963_695993_496854_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 496854,\n          "object_id": 695993,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.28496503829956055,\n                0.24045312404632568,\n                0.9956876635551453,\n                1.0\n              ],\n              [\n                0.17398601770401,\n                0.35899999737739563,\n                0.3349650204181671,\n                0.4039531350135803\n              ],\n              [\n                0.17398601770401,\n                0.24045312404632568,\n                0.9956876635551453,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.3043822944164276,\n                0.15918749570846558,\n                0.7559440732002258,\n                0.948437511920929\n              ],\n              [\n                0.17398601770401,\n                0.35899999737739563,\n                0.3349650204181671,\n                0.4039531350135803\n              ],\n              [\n                0.17398601770401,\n                0.15918749570846558,\n                0.7559440732002258,\n                0.948437511920929\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000253264.jpg",\n        "coco_id": 253264,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 442154,\n          "object_id": 635833,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_253264_442154_635833_474112_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 474112,\n          "object_id": 635833,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.5319530963897705,\n                0.16667447984218597,\n                0.6927343606948853,\n                0.8873770236968994\n              ],\n              [\n                0.1372968703508377,\n                0.6554332375526428,\n                0.21620312333106995,\n                0.747002363204956\n              ],\n              [\n                0.1372968703508377,\n                0.16667447984218597,\n                0.6927343606948853,\n                0.8873770236968994\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.13643750548362732,\n                0.4584309160709381,\n                0.3268437385559082,\n                0.9887588024139404\n              ],\n              [\n                0.1372968703508377,\n                0.6554332375526428,\n                0.21620312333106995,\n                0.747002363204956\n              ],\n              [\n                0.13643750548362732,\n                0.4584309160709381,\n                0.3268437385559082,\n                0.9887588024139404\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000253538.jpg",\n        "coco_id": 253538,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 433862,\n          "object_id": 604634,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_253538_433862_604634_502522_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 502522,\n          "object_id": 604634,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.9019200205802917,\n                0.486156165599823,\n                1.0,\n                1.0\n              ],\n              [\n                0.5032200217247009,\n                0.4324324429035187,\n                0.5619000196456909,\n                0.49204203486442566\n              ],\n              [\n                0.5032200217247009,\n                0.4324324429035187,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.5200999975204468,\n                0.2290990948677063,\n                0.635420024394989,\n                0.8418018221855164\n              ],\n              [\n                0.5032200217247009,\n                0.4324324429035187,\n                0.5619000196456909,\n                0.49204203486442566\n              ],\n              [\n                0.5032200217247009,\n                0.2290990948677063,\n                0.635420024394989,\n                0.8418018221855164\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000256606.jpg",\n        "coco_id": 256606,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 506154,\n          "object_id": 631527,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_256606_506154_631527_473372_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 473372,\n          "object_id": 631527,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.4996874928474426,\n                0.2343091368675232,\n                0.6431406140327454,\n                0.6497657895088196\n              ],\n              [\n                0.40095311403274536,\n                0.44283372163772583,\n                0.5005156397819519,\n                0.4849883019924164\n              ],\n              [\n                0.40095311403274536,\n                0.2343091368675232,\n                0.6431406140327454,\n                0.6497657895088196\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.7440780997276306,\n                0.31817328929901123,\n                0.8827968835830688,\n                0.6832084059715271\n              ],\n              [\n                0.40095311403274536,\n                0.44283372163772583,\n                0.5005156397819519,\n                0.4849883019924164\n              ],\n              [\n                0.40095311403274536,\n                0.31817328929901123,\n                0.8827968835830688,\n                0.6832084059715271\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000267417.jpg",\n        "coco_id": 267417,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2154683,\n          "object_id": 2100708,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_267417_2154683_2100708_481757_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 481757,\n          "object_id": 2100708,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.208921879529953,\n                0.3688333332538605,\n                0.3740312457084656,\n                0.7506389021873474\n              ],\n              [\n                0.29032811522483826,\n                0.594944417476654,\n                0.32996875047683716,\n                0.659250020980835\n              ],\n              [\n                0.208921879529953,\n                0.3688333332538605,\n                0.3740312457084656,\n                0.7506389021873474\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.37331250309944153,\n                0.32511112093925476,\n                0.6154687404632568,\n                0.7219722270965576\n              ],\n              [\n                0.29032811522483826,\n                0.594944417476654,\n                0.32996875047683716,\n                0.659250020980835\n              ],\n              [\n                0.29032811522483826,\n                0.32511112093925476,\n                0.6154687404632568,\n                0.7219722270965576\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000270809.jpg",\n        "coco_id": 270809,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 490143,\n          "object_id": 636283,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_270809_490143_636283_489968_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 489968,\n          "object_id": 636283,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.5700156092643738,\n                0.07191570848226547,\n                0.8962656259536743,\n                0.8494444489479065\n              ],\n              [\n                0.8287500143051147,\n                0.31145593523979187,\n                0.8729062676429749,\n                0.38917624950408936\n              ],\n              [\n                0.5700156092643738,\n                0.07191570848226547,\n                0.8962656259536743,\n                0.8494444489479065\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.0462343767285347,\n                0.20862068235874176,\n                0.5363593697547913,\n                0.7052107453346252\n              ],\n              [\n                0.8287500143051147,\n                0.31145593523979187,\n                0.8729062676429749,\n                0.38917624950408936\n              ],\n              [\n                0.0462343767285347,\n                0.20862068235874176,\n                0.8729062676429749,\n                0.7052107453346252\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000271250.jpg",\n        "coco_id": 271250,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 528612,\n          "object_id": 2082322,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_271250_528612_2082322_1273174_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1273174,\n          "object_id": 2082322,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.007515625096857548,\n                0.39046838879585266,\n                0.1450156271457672,\n                0.7537002563476562\n              ],\n              [\n                0.4279531240463257,\n                0.5572365522384644,\n                0.47206249833106995,\n                0.5980796217918396\n              ],\n              [\n                0.007515625096857548,\n                0.39046838879585266,\n                0.47206249833106995,\n                0.7537002563476562\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.34184375405311584,\n                0.4134894609451294,\n                0.5007656216621399,\n                0.8359484672546387\n              ],\n              [\n                0.4279531240463257,\n                0.5572365522384644,\n                0.47206249833106995,\n                0.5980796217918396\n              ],\n              [\n                0.34184375405311584,\n                0.4134894609451294,\n                0.5007656216621399,\n                0.8359484672546387\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000272632.jpg",\n        "coco_id": 272632,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bed",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 507521,\n          "object_id": 317978,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_272632_507521_317978_1711049_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1711049,\n          "object_id": 317978,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bed",\n            "bounds": [\n              [\n                0.5601750612258911,\n                0.7386406064033508,\n                0.771028459072113,\n                0.9835781455039978\n              ],\n              [\n                0.025185994803905487,\n                0.04456249997019768,\n                1.0,\n                0.9883906245231628\n              ],\n              [\n                0.025185994803905487,\n                0.04456249997019768,\n                1.0,\n                0.9883906245231628\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bed",\n            "bounds": [\n              [\n                0.42608314752578735,\n                0.012781250290572643,\n                0.7127571105957031,\n                0.33018749952316284\n              ],\n              [\n                0.025185994803905487,\n                0.04456249997019768,\n                1.0,\n                0.9883906245231628\n              ],\n              [\n                0.025185994803905487,\n                0.012781250290572643,\n                1.0,\n                0.9883906245231628\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000273200.jpg",\n        "coco_id": 273200,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1203861,\n          "object_id": 628533,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_273200_1203861_628533_1228983_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1228983,\n          "object_id": 628533,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.29887500405311584,\n                0.343729168176651,\n                0.5593593716621399,\n                0.7634999752044678\n              ],\n              [\n                0.5195156335830688,\n                0.2860625088214874,\n                0.5548593997955322,\n                0.499854177236557\n              ],\n              [\n                0.29887500405311584,\n                0.2860625088214874,\n                0.5593593716621399,\n                0.7634999752044678\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.8091562390327454,\n                0.43575000762939453,\n                0.9470781087875366,\n                0.78885418176651\n              ],\n              [\n                0.5195156335830688,\n                0.2860625088214874,\n                0.5548593997955322,\n                0.499854177236557\n              ],\n              [\n                0.5195156335830688,\n                0.2860625088214874,\n                0.9470781087875366,\n                0.78885418176651\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000273200.jpg",\n        "coco_id": 273200,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2164695,\n          "object_id": 636107,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_273200_2164695_636107_1228983_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1228983,\n          "object_id": 636107,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.8091562390327454,\n                0.43575000762939453,\n                0.9470781087875366,\n                0.78885418176651\n              ],\n              [\n                0.7183125019073486,\n                0.38206249475479126,\n                0.7622656226158142,\n                0.44218748807907104\n              ],\n              [\n                0.7183125019073486,\n                0.38206249475479126,\n                0.9470781087875366,\n                0.78885418176651\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.7210312485694885,\n                0.3880625069141388,\n                0.8495468497276306,\n                0.7821875214576721\n              ],\n              [\n                0.7183125019073486,\n                0.38206249475479126,\n                0.7622656226158142,\n                0.44218748807907104\n              ],\n              [\n                0.7183125019073486,\n                0.38206249475479126,\n                0.8495468497276306,\n                0.7821875214576721\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000002752.jpg",\n        "coco_id": 2752,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 197058,\n          "object_id": 669623,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_2752_197058_669623_1708115_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1708115,\n          "object_id": 669623,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.6756799817085266,\n                0.4922666549682617,\n                0.9434400200843811,\n                0.7349333167076111\n              ],\n              [\n                0.37786000967025757,\n                0.6087466478347778,\n                0.40281999111175537,\n                0.652400016784668\n              ],\n              [\n                0.37786000967025757,\n                0.4922666549682617,\n                0.9434400200843811,\n                0.7349333167076111\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.22529999911785126,\n                0.5289066433906555,\n                0.43101999163627625,\n                0.93149334192276\n              ],\n              [\n                0.37786000967025757,\n                0.6087466478347778,\n                0.40281999111175537,\n                0.652400016784668\n              ],\n              [\n                0.22529999911785126,\n                0.5289066433906555,\n                0.43101999163627625,\n                0.93149334192276\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000275544.jpg",\n        "coco_id": 275544,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 219766,\n          "object_id": 147869,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_275544_219766_147869_190590_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 190590,\n          "object_id": 147869,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.5894062519073486,\n                0.2866666615009308,\n                0.7375156283378601,\n                0.8090277910232544\n              ],\n              [\n                0.49803125858306885,\n                0.4041111171245575,\n                0.8620781302452087,\n                0.804111123085022\n              ],\n              [\n                0.49803125858306885,\n                0.2866666615009308,\n                0.8620781302452087,\n                0.8090277910232544\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.18387499451637268,\n                0.3505277633666992,\n                0.3302343785762787,\n                0.8236666917800903\n              ],\n              [\n                0.49803125858306885,\n                0.4041111171245575,\n                0.8620781302452087,\n                0.804111123085022\n              ],\n              [\n                0.18387499451637268,\n                0.3505277633666992,\n                0.8620781302452087,\n                0.8236666917800903\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000276265.jpg",\n        "coco_id": 276265,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "banana",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 441306,\n          "object_id": 1045237,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_276265_441306_1045237_478210_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 478210,\n          "object_id": 1045237,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.33128124475479126,\n                0.49322351813316345,\n                0.6115937232971191,\n                1.0\n              ],\n              [\n                0.3265624940395355,\n                0.2988235354423523,\n                0.4375,\n                0.46117648482322693\n              ],\n              [\n                0.3265624940395355,\n                0.2988235354423523,\n                0.6115937232971191,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.7759843468666077,\n                0.2853882312774658,\n                1.0,\n                0.9865176677703857\n              ],\n              [\n                0.3265624940395355,\n                0.2988235354423523,\n                0.4375,\n                0.46117648482322693\n              ],\n              [\n                0.3265624940395355,\n                0.2853882312774658,\n                1.0,\n                0.9865176677703857\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000277156.jpg",\n        "coco_id": 277156,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "pizza",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 424383,\n          "object_id": 1071238,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_277156_424383_1071238_1265347_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1265347,\n          "object_id": 1071238,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.5443750023841858,\n                0.01347916666418314,\n                0.8797812461853027,\n                0.6629166603088379\n              ],\n              [\n                0.3061249852180481,\n                0.8035416603088379,\n                0.47004687786102295,\n                0.9535416960716248\n              ],\n              [\n                0.3061249852180481,\n                0.01347916666418314,\n                0.8797812461853027,\n                0.9535416960716248\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.001687500043772161,\n                0.23527082800865173,\n                0.6000000238418579,\n                0.929645836353302\n              ],\n              [\n                0.3061249852180481,\n                0.8035416603088379,\n                0.47004687786102295,\n                0.9535416960716248\n              ],\n              [\n                0.001687500043772161,\n                0.23527082800865173,\n                0.6000000238418579,\n                0.9535416960716248\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000277156.jpg",\n        "coco_id": 277156,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 424383,\n          "object_id": 1880673,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_277156_424383_1880673_1265347_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1265347,\n          "object_id": 1880673,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.001687500043772161,\n                0.23527082800865173,\n                0.6000000238418579,\n                0.929645836353302\n              ],\n              [\n                0.17734375596046448,\n                0.5340208411216736,\n                0.2741093635559082,\n                0.706041693687439\n              ],\n              [\n                0.001687500043772161,\n                0.23527082800865173,\n                0.6000000238418579,\n                0.929645836353302\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.5443750023841858,\n                0.01347916666418314,\n                0.8797812461853027,\n                0.6629166603088379\n              ],\n              [\n                0.17734375596046448,\n                0.5340208411216736,\n                0.2741093635559082,\n                0.706041693687439\n              ],\n              [\n                0.17734375596046448,\n                0.01347916666418314,\n                0.8797812461853027,\n                0.706041693687439\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000277575.jpg",\n        "coco_id": 277575,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 493419,\n          "object_id": 1170734,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_277575_493419_1170734_554217_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 554217,\n          "object_id": 1170734,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.30943751335144043,\n                0.2995208203792572,\n                0.43162500858306885,\n                0.7297499775886536\n              ],\n              [\n                0.1407187432050705,\n                0.25222915410995483,\n                0.2571093738079071,\n                0.45524999499320984\n              ],\n              [\n                0.1407187432050705,\n                0.25222915410995483,\n                0.43162500858306885,\n                0.7297499775886536\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.12471874803304672,\n                0.20874999463558197,\n                0.30550000071525574,\n                0.8066666722297668\n              ],\n              [\n                0.1407187432050705,\n                0.25222915410995483,\n                0.2571093738079071,\n                0.45524999499320984\n              ],\n              [\n                0.12471874803304672,\n                0.20874999463558197,\n                0.30550000071525574,\n                0.8066666722297668\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000277729.jpg",\n        "coco_id": 277729,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1204276,\n          "object_id": 1166587,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_277729_1204276_1166587_1305428_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1305428,\n          "object_id": 1166587,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.5100937485694885,\n                0.8101392388343811,\n                0.5396406054496765,\n                0.9961484670639038\n              ],\n              [\n                0.53515625,\n                0.8459164500236511,\n                0.5498594045639038,\n                0.9050115942955017\n              ],\n              [\n                0.5100937485694885,\n                0.8101392388343811,\n                0.5498594045639038,\n                0.9961484670639038\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.1509374976158142,\n                0.8327610492706299,\n                0.2179843783378601,\n                1.0\n              ],\n              [\n                0.53515625,\n                0.8459164500236511,\n                0.5498594045639038,\n                0.9050115942955017\n              ],\n              [\n                0.1509374976158142,\n                0.8327610492706299,\n                0.5498594045639038,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000277729.jpg",\n        "coco_id": 277729,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1316698,\n          "object_id": 1190901,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_277729_1316698_1190901_1305428_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1305428,\n          "object_id": 1190901,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.7873437404632568,\n                0.8193967342376709,\n                0.8573125004768372,\n                1.0\n              ],\n              [\n                0.7270625233650208,\n                0.9066821336746216,\n                0.7964218854904175,\n                0.9998607635498047\n              ],\n              [\n                0.7270625233650208,\n                0.8193967342376709,\n                0.8573125004768372,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.1509374976158142,\n                0.8327610492706299,\n                0.2179843783378601,\n                1.0\n              ],\n              [\n                0.7270625233650208,\n                0.9066821336746216,\n                0.7964218854904175,\n                0.9998607635498047\n              ],\n              [\n                0.1509374976158142,\n                0.8327610492706299,\n                0.7964218854904175,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000281563.jpg",\n        "coco_id": 281563,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "banana",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 435260,\n          "object_id": 1042342,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_281563_435260_1042342_474294_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 474294,\n          "object_id": 1042342,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.34474000334739685,\n                0.5818373560905457,\n                0.6355199813842773,\n                0.988162636756897\n              ],\n              [\n                0.5017200112342834,\n                0.11210843175649643,\n                0.6059399843215942,\n                0.3296084403991699\n              ],\n              [\n                0.34474000334739685,\n                0.11210843175649643,\n                0.6355199813842773,\n                0.988162636756897\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a banana",\n            "bounds": [\n              [\n                0.8326200246810913,\n                0.4539457857608795,\n                1.0,\n                0.9662951827049255\n              ],\n              [\n                0.5017200112342834,\n                0.11210843175649643,\n                0.6059399843215942,\n                0.3296084403991699\n              ],\n              [\n                0.5017200112342834,\n                0.11210843175649643,\n                1.0,\n                0.9662951827049255\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000283678.jpg",\n        "coco_id": 283678,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 489751,\n          "object_id": 629600,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_283678_489751_629600_470243_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 470243,\n          "object_id": 629600,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.29340624809265137,\n                0.3822014033794403,\n                0.47276562452316284,\n                0.8350585699081421\n              ],\n              [\n                0.4241406321525574,\n                0.4532552659511566,\n                0.48353123664855957,\n                0.4737236499786377\n              ],\n              [\n                0.29340624809265137,\n                0.3822014033794403,\n                0.48353123664855957,\n                0.8350585699081421\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.5803906321525574,\n                0.5002107620239258,\n                0.7542499899864197,\n                0.8330913186073303\n              ],\n              [\n                0.4241406321525574,\n                0.4532552659511566,\n                0.48353123664855957,\n                0.4737236499786377\n              ],\n              [\n                0.4241406321525574,\n                0.4532552659511566,\n                0.7542499899864197,\n                0.8330913186073303\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000028759.jpg",\n        "coco_id": 28759,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 481067,\n          "object_id": 1172860,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_28759_481067_1172860_475363_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 475363,\n          "object_id": 1172860,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.36618751287460327,\n                0.41195833683013916,\n                0.4944531321525574,\n                0.6666666865348816\n              ],\n              [\n                0.23207812011241913,\n                0.5316458344459534,\n                0.2757031321525574,\n                0.629479169845581\n              ],\n              [\n                0.23207812011241913,\n                0.41195833683013916,\n                0.4944531321525574,\n                0.6666666865348816\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.25404688715934753,\n                0.3853333294391632,\n                0.3427031338214874,\n                0.6776458621025085\n              ],\n              [\n                0.23207812011241913,\n                0.5316458344459534,\n                0.2757031321525574,\n                0.629479169845581\n              ],\n              [\n                0.23207812011241913,\n                0.3853333294391632,\n                0.3427031338214874,\n                0.6776458621025085\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000028759.jpg",\n        "coco_id": 28759,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 481067,\n          "object_id": 1172860,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_28759_481067_1172860_475363_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 475363,\n          "object_id": 1172860,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.25404688715934753,\n                0.3853333294391632,\n                0.3427031338214874,\n                0.6776458621025085\n              ],\n              [\n                0.23207812011241913,\n                0.5316458344459534,\n                0.2757031321525574,\n                0.629479169845581\n              ],\n              [\n                0.23207812011241913,\n                0.3853333294391632,\n                0.3427031338214874,\n                0.6776458621025085\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.36618751287460327,\n                0.41195833683013916,\n                0.4944531321525574,\n                0.6666666865348816\n              ],\n              [\n                0.23207812011241913,\n                0.5316458344459534,\n                0.2757031321525574,\n                0.629479169845581\n              ],\n              [\n                0.23207812011241913,\n                0.41195833683013916,\n                0.4944531321525574,\n                0.6666666865348816\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000028782.jpg",\n        "coco_id": 28782,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "pizza",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1200636,\n          "object_id": 1075232,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_28782_1200636_1075232_542942_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 542942,\n          "object_id": 1075232,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.0,\n                0.23423437774181366,\n                0.45645833015441895,\n                0.7072031497955322\n              ],\n              [\n                0.09814583510160446,\n                0.6819531321525574,\n                0.3387500047683716,\n                0.814078152179718\n              ],\n              [\n                0.0,\n                0.23423437774181366,\n                0.45645833015441895,\n                0.814078152179718\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.0,\n                0.7196875214576721,\n                0.42110416293144226,\n                1.0\n              ],\n              [\n                0.09814583510160446,\n                0.6819531321525574,\n                0.3387500047683716,\n                0.814078152179718\n              ],\n              [\n                0.0,\n                0.6819531321525574,\n                0.42110416293144226,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000288548.jpg",\n        "coco_id": 288548,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "laptop",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1220922,\n          "object_id": 1103647,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_288548_1220922_1103647_1676451_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1676451,\n          "object_id": 1103647,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a laptop",\n            "bounds": [\n              [\n                0.09721875190734863,\n                0.3123708963394165,\n                0.41432812809944153,\n                0.9775351881980896\n              ],\n              [\n                0.5844843983650208,\n                0.5569953322410583,\n                0.8203906416893005,\n                0.7172769904136658\n              ],\n              [\n                0.09721875190734863,\n                0.3123708963394165,\n                0.8203906416893005,\n                0.9775351881980896\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a laptop",\n            "bounds": [\n              [\n                0.49659374356269836,\n                0.226971834897995,\n                0.9587968587875366,\n                0.831455409526825\n              ],\n              [\n                0.5844843983650208,\n                0.5569953322410583,\n                0.8203906416893005,\n                0.7172769904136658\n              ],\n              [\n                0.49659374356269836,\n                0.226971834897995,\n                0.9587968587875366,\n                0.831455409526825\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000289012.jpg",\n        "coco_id": 289012,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "sports ball",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 492052,\n          "object_id": 304078,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_289012_492052_304078_513820_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 513820,\n          "object_id": 304078,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a sports ball",\n            "bounds": [\n              [\n                0.6205781102180481,\n                0.1925586313009262,\n                0.8565937280654907,\n                0.9358208775520325\n              ],\n              [\n                0.8097968697547913,\n                0.5636247396469116,\n                0.847781240940094,\n                0.622302770614624\n              ],\n              [\n                0.6205781102180481,\n                0.1925586313009262,\n                0.8565937280654907,\n                0.9358208775520325\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a sports ball",\n            "bounds": [\n              [\n                0.04774999991059303,\n                0.1977611929178238,\n                0.3820468783378601,\n                0.943816602230072\n              ],\n              [\n                0.8097968697547913,\n                0.5636247396469116,\n                0.847781240940094,\n                0.622302770614624\n              ],\n              [\n                0.04774999991059303,\n                0.1977611929178238,\n                0.847781240940094,\n                0.943816602230072\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000289012.jpg",\n        "coco_id": 289012,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "sports ball",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 492052,\n          "object_id": 304078,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_289012_492052_304078_513820_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 513820,\n          "object_id": 304078,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.6205781102180481,\n                0.1925586313009262,\n                0.8565937280654907,\n                0.9358208775520325\n              ],\n              [\n                0.8097968697547913,\n                0.5636247396469116,\n                0.847781240940094,\n                0.622302770614624\n              ],\n              [\n                0.6205781102180481,\n                0.1925586313009262,\n                0.8565937280654907,\n                0.9358208775520325\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.04774999991059303,\n                0.1977611929178238,\n                0.3820468783378601,\n                0.943816602230072\n              ],\n              [\n                0.8097968697547913,\n                0.5636247396469116,\n                0.847781240940094,\n                0.622302770614624\n              ],\n              [\n                0.04774999991059303,\n                0.1977611929178238,\n                0.847781240940094,\n                0.943816602230072\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000289569.jpg",\n        "coco_id": 289569,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1723069,\n          "object_id": 150835,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_289569_1723069_150835_190890_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 190890,\n          "object_id": 150835,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.16068750619888306,\n                0.46896955370903015,\n                0.2756093740463257,\n                0.8078688383102417\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.16068750619888306,\n                0.46896955370903015,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.2942500114440918,\n                0.47292739152908325,\n                0.4144531190395355,\n                0.7010070085525513\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.27464061975479126,\n                0.47292739152908325,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000289569.jpg",\n        "coco_id": 289569,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1723069,\n          "object_id": 150835,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_289569_1723069_150835_190890_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 190890,\n          "object_id": 150835,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.16068750619888306,\n                0.46896955370903015,\n                0.2756093740463257,\n                0.8078688383102417\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.16068750619888306,\n                0.46896955370903015,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.2942500114440918,\n                0.47292739152908325,\n                0.4144531190395355,\n                0.7010070085525513\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.27464061975479126,\n                0.47292739152908325,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000289569.jpg",\n        "coco_id": 289569,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 258348,\n          "object_id": 150835,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_289569_258348_150835_1723069_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1723069,\n          "object_id": 150835,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.2942500114440918,\n                0.47292739152908325,\n                0.4144531190395355,\n                0.7010070085525513\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.27464061975479126,\n                0.47292739152908325,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.37867188453674316,\n                0.47185012698173523,\n                0.4853593707084656,\n                0.827704906463623\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.27464061975479126,\n                0.47185012698173523,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000289569.jpg",\n        "coco_id": 289569,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 258348,\n          "object_id": 150835,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_289569_258348_150835_190890_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 190890,\n          "object_id": 150835,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.37867188453674316,\n                0.47185012698173523,\n                0.4853593707084656,\n                0.827704906463623\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.27464061975479126,\n                0.47185012698173523,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.16068750619888306,\n                0.46896955370903015,\n                0.2756093740463257,\n                0.8078688383102417\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.16068750619888306,\n                0.46896955370903015,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000289569.jpg",\n        "coco_id": 289569,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 258348,\n          "object_id": 150835,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_289569_258348_150835_190890_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 190890,\n          "object_id": 150835,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.16068750619888306,\n                0.46896955370903015,\n                0.2756093740463257,\n                0.8078688383102417\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.16068750619888306,\n                0.46896955370903015,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.37867188453674316,\n                0.47185012698173523,\n                0.4853593707084656,\n                0.827704906463623\n              ],\n              [\n                0.27464061975479126,\n                0.6020140647888184,\n                0.5806249976158142,\n                0.8570491671562195\n              ],\n              [\n                0.27464061975479126,\n                0.47185012698173523,\n                0.5806249976158142,\n                0.8570491671562195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000292226.jpg",\n        "coco_id": 292226,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1308118,\n          "object_id": 127229,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_292226_1308118_127229_1331628_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1331628,\n          "object_id": 127229,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.3434426188468933\n              ],\n              [\n                0.1764843761920929,\n                0.46721312403678894,\n                0.5495312213897705,\n                0.8306557536125183\n              ],\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.8306557536125183\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.24384374916553497,\n                0.18758782744407654,\n                0.4562656283378601,\n                0.7772833704948425\n              ],\n              [\n                0.1764843761920929,\n                0.46721312403678894,\n                0.5495312213897705,\n                0.8306557536125183\n              ],\n              [\n                0.1764843761920929,\n                0.18758782744407654,\n                0.5495312213897705,\n                0.8306557536125183\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000292226.jpg",\n        "coco_id": 292226,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1308118,\n          "object_id": 127229,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_292226_1308118_127229_1331628_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1331628,\n          "object_id": 127229,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.3434426188468933\n              ],\n              [\n                0.1764843761920929,\n                0.46721312403678894,\n                0.5495312213897705,\n                0.8306557536125183\n              ],\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.8306557536125183\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.24384374916553497,\n                0.18758782744407654,\n                0.4562656283378601,\n                0.7772833704948425\n              ],\n              [\n                0.1764843761920929,\n                0.46721312403678894,\n                0.5495312213897705,\n                0.8306557536125183\n              ],\n              [\n                0.1764843761920929,\n                0.18758782744407654,\n                0.5495312213897705,\n                0.8306557536125183\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000292226.jpg",\n        "coco_id": 292226,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1308118,\n          "object_id": 127229,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_292226_1308118_127229_1331628_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1331628,\n          "object_id": 127229,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.24384374916553497,\n                0.18758782744407654,\n                0.4562656283378601,\n                0.7772833704948425\n              ],\n              [\n                0.1764843761920929,\n                0.46721312403678894,\n                0.5495312213897705,\n                0.8306557536125183\n              ],\n              [\n                0.1764843761920929,\n                0.18758782744407654,\n                0.5495312213897705,\n                0.8306557536125183\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.3434426188468933\n              ],\n              [\n                0.1764843761920929,\n                0.46721312403678894,\n                0.5495312213897705,\n                0.8306557536125183\n              ],\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.8306557536125183\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000292226.jpg",\n        "coco_id": 292226,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1330527,\n          "object_id": 127991,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_292226_1330527_127991_1331628_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1331628,\n          "object_id": 127991,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.6791874766349792,\n                0.1707962602376938,\n                0.9475625157356262,\n                0.7730444669723511\n              ],\n              [\n                0.6013281345367432,\n                0.4635128676891327,\n                0.9964843988418579,\n                0.8133020997047424\n              ],\n              [\n                0.6013281345367432,\n                0.1707962602376938,\n                0.9964843988418579,\n                0.8133020997047424\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.3434426188468933\n              ],\n              [\n                0.6013281345367432,\n                0.4635128676891327,\n                0.9964843988418579,\n                0.8133020997047424\n              ],\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.9964843988418579,\n                0.8133020997047424\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000292226.jpg",\n        "coco_id": 292226,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1330527,\n          "object_id": 127991,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_292226_1330527_127991_1331628_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1331628,\n          "object_id": 127991,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.6791874766349792,\n                0.1707962602376938,\n                0.9475625157356262,\n                0.7730444669723511\n              ],\n              [\n                0.6013281345367432,\n                0.4635128676891327,\n                0.9964843988418579,\n                0.8133020997047424\n              ],\n              [\n                0.6013281345367432,\n                0.1707962602376938,\n                0.9964843988418579,\n                0.8133020997047424\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.3434426188468933\n              ],\n              [\n                0.6013281345367432,\n                0.4635128676891327,\n                0.9964843988418579,\n                0.8133020997047424\n              ],\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.9964843988418579,\n                0.8133020997047424\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000292226.jpg",\n        "coco_id": 292226,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1330527,\n          "object_id": 127991,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_292226_1330527_127991_1331628_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1331628,\n          "object_id": 127991,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.6791874766349792,\n                0.1707962602376938,\n                0.9475625157356262,\n                0.7730444669723511\n              ],\n              [\n                0.6013281345367432,\n                0.4635128676891327,\n                0.9964843988418579,\n                0.8133020997047424\n              ],\n              [\n                0.6013281345367432,\n                0.1707962602376938,\n                0.9964843988418579,\n                0.8133020997047424\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.3434426188468933\n              ],\n              [\n                0.6013281345367432,\n                0.4635128676891327,\n                0.9964843988418579,\n                0.8133020997047424\n              ],\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.9964843988418579,\n                0.8133020997047424\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000292226.jpg",\n        "coco_id": 292226,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1714472,\n          "object_id": 1770315,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_292226_1714472_1770315_1331628_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1331628,\n          "object_id": 1770315,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.532546877861023,\n                0.20538641512393951,\n                0.7000937461853027,\n                0.7681499123573303\n              ],\n              [\n                0.5339375138282776,\n                0.4345199167728424,\n                0.7147031426429749,\n                0.615948498249054\n              ],\n              [\n                0.532546877861023,\n                0.20538641512393951,\n                0.7147031426429749,\n                0.7681499123573303\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.3434426188468933\n              ],\n              [\n                0.5339375138282776,\n                0.4345199167728424,\n                0.7147031426429749,\n                0.615948498249054\n              ],\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.615948498249054\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000292226.jpg",\n        "coco_id": 292226,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1714472,\n          "object_id": 1770315,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_292226_1714472_1770315_1331628_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1331628,\n          "object_id": 1770315,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.532546877861023,\n                0.20538641512393951,\n                0.7000937461853027,\n                0.7681499123573303\n              ],\n              [\n                0.5339375138282776,\n                0.4345199167728424,\n                0.7147031426429749,\n                0.615948498249054\n              ],\n              [\n                0.532546877861023,\n                0.20538641512393951,\n                0.7147031426429749,\n                0.7681499123573303\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.3434426188468933\n              ],\n              [\n                0.5339375138282776,\n                0.4345199167728424,\n                0.7147031426429749,\n                0.615948498249054\n              ],\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.615948498249054\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000292226.jpg",\n        "coco_id": 292226,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1714472,\n          "object_id": 1770315,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_292226_1714472_1770315_1331628_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1331628,\n          "object_id": 1770315,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.532546877861023,\n                0.20538641512393951,\n                0.7000937461853027,\n                0.7681499123573303\n              ],\n              [\n                0.5339375138282776,\n                0.4345199167728424,\n                0.7147031426429749,\n                0.615948498249054\n              ],\n              [\n                0.532546877861023,\n                0.20538641512393951,\n                0.7147031426429749,\n                0.7681499123573303\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.3434426188468933\n              ],\n              [\n                0.5339375138282776,\n                0.4345199167728424,\n                0.7147031426429749,\n                0.615948498249054\n              ],\n              [\n                0.15592187643051147,\n                0.001873536268249154,\n                0.8845781087875366,\n                0.615948498249054\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000297811.jpg",\n        "coco_id": 297811,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skateboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1313967,\n          "object_id": 643155,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_297811_1313967_643155_1267861_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1267861,\n          "object_id": 643155,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.5486619472503662,\n                0.31953126192092896,\n                0.6778638362884521,\n                0.5864062309265137\n              ],\n              [\n                0.48352113366127014,\n                0.5083437561988831,\n                0.5711502432823181,\n                0.6505468487739563\n              ],\n              [\n                0.48352113366127014,\n                0.31953126192092896,\n                0.6778638362884521,\n                0.6505468487739563\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.33032864332199097,\n                0.3570156395435333,\n                0.5664788484573364,\n                0.6028749942779541\n              ],\n              [\n                0.48352113366127014,\n                0.5083437561988831,\n                0.5711502432823181,\n                0.6505468487739563\n              ],\n              [\n                0.33032864332199097,\n                0.3570156395435333,\n                0.5711502432823181,\n                0.6505468487739563\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000300624.jpg",\n        "coco_id": 300624,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 479241,\n          "object_id": 602518,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_300624_479241_602518_473208_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 473208,\n          "object_id": 602518,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.4694531261920929,\n                0.19952940940856934,\n                0.8230156302452087,\n                0.9937176704406738\n              ],\n              [\n                0.7580780982971191,\n                0.35056471824645996,\n                0.8505937457084656,\n                0.48315295577049255\n              ],\n              [\n                0.4694531261920929,\n                0.19952940940856934,\n                0.8505937457084656,\n                0.9937176704406738\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.053593751043081284,\n                0.02007058821618557,\n                0.5002812743186951,\n                0.9886823296546936\n              ],\n              [\n                0.7580780982971191,\n                0.35056471824645996,\n                0.8505937457084656,\n                0.48315295577049255\n              ],\n              [\n                0.053593751043081284,\n                0.02007058821618557,\n                0.8505937457084656,\n                0.9886823296546936\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000030238.jpg",\n        "coco_id": 30238,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1204143,\n          "object_id": 603776,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_30238_1204143_603776_522995_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 522995,\n          "object_id": 603776,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.6025599837303162,\n                0.20925332605838776,\n                0.8326399922370911,\n                0.8698933124542236\n              ],\n              [\n                0.5905799865722656,\n                0.5333600044250488,\n                0.7170400023460388,\n                0.6065866947174072\n              ],\n              [\n                0.5905799865722656,\n                0.20925332605838776,\n                0.8326399922370911,\n                0.8698933124542236\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.09798000007867813,\n                0.33157333731651306,\n                0.3665600121021271,\n                0.7189599871635437\n              ],\n              [\n                0.5905799865722656,\n                0.5333600044250488,\n                0.7170400023460388,\n                0.6065866947174072\n              ],\n              [\n                0.09798000007867813,\n                0.33157333731651306,\n                0.7170400023460388,\n                0.7189599871635437\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000304295.jpg",\n        "coco_id": 304295,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 427285,\n          "object_id": 1176637,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_304295_427285_1176637_462832_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 462832,\n          "object_id": 1176637,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.5342656373977661,\n                0.5776875019073486,\n                0.6303437352180481,\n                0.9731875061988831\n              ],\n              [\n                0.6075624823570251,\n                0.7027083039283752,\n                0.6256874799728394,\n                0.7722083330154419\n              ],\n              [\n                0.5342656373977661,\n                0.5776875019073486,\n                0.6303437352180481,\n                0.9731875061988831\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.9293281435966492,\n                0.5205000042915344,\n                1.0,\n                1.0\n              ],\n              [\n                0.6075624823570251,\n                0.7027083039283752,\n                0.6256874799728394,\n                0.7722083330154419\n              ],\n              [\n                0.6075624823570251,\n                0.5205000042915344,\n                1.0,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000309285.jpg",\n        "coco_id": 309285,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 469668,\n          "object_id": 670843,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_309285_469668_670843_504642_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 504642,\n          "object_id": 670843,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.1254843771457672,\n                0.4596041738986969,\n                0.29862499237060547,\n                0.9866458177566528\n              ],\n              [\n                0.4378125071525574,\n                0.8823124766349792,\n                0.504562497138977,\n                0.9735000133514404\n              ],\n              [\n                0.1254843771457672,\n                0.4596041738986969,\n                0.504562497138977,\n                0.9866458177566528\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.22753125429153442,\n                0.5775208473205566,\n                0.7837031483650208,\n                0.9820208549499512\n              ],\n              [\n                0.4378125071525574,\n                0.8823124766349792,\n                0.504562497138977,\n                0.9735000133514404\n              ],\n              [\n                0.22753125429153442,\n                0.5775208473205566,\n                0.7837031483650208,\n                0.9820208549499512\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000313601.jpg",\n        "coco_id": 313601,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1265837,\n          "object_id": 602058,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_313601_1265837_602058_1240696_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1240696,\n          "object_id": 602058,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.4728749990463257,\n                0.22883720695972443,\n                0.7052500247955322,\n                0.7479327917098999\n              ],\n              [\n                0.662484347820282,\n                0.39379844069480896,\n                0.7176718711853027,\n                0.4417312741279602\n              ],\n              [\n                0.4728749990463257,\n                0.22883720695972443,\n                0.7176718711853027,\n                0.7479327917098999\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.6673281192779541,\n                0.06418604403734207,\n                0.9015781283378601,\n                0.782661497592926\n              ],\n              [\n                0.662484347820282,\n                0.39379844069480896,\n                0.7176718711853027,\n                0.4417312741279602\n              ],\n              [\n                0.662484347820282,\n                0.06418604403734207,\n                0.9015781283378601,\n                0.782661497592926\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000315381.jpg",\n        "coco_id": 315381,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2148674,\n          "object_id": 602052,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_315381_2148674_602052_527035_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 527035,\n          "object_id": 602052,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.14509375393390656,\n                0.1844131499528885,\n                0.5848593711853027,\n                0.9349765181541443\n              ],\n              [\n                0.5113281011581421,\n                0.43201878666877747,\n                0.5974843502044678,\n                0.4684976637363434\n              ],\n              [\n                0.14509375393390656,\n                0.1844131499528885,\n                0.5974843502044678,\n                0.9349765181541443\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.21562500298023224,\n                0.09302816540002823,\n                0.5796562433242798,\n                0.9073004722595215\n              ],\n              [\n                0.5113281011581421,\n                0.43201878666877747,\n                0.5974843502044678,\n                0.4684976637363434\n              ],\n              [\n                0.21562500298023224,\n                0.09302816540002823,\n                0.5974843502044678,\n                0.9073004722595215\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000318445.jpg",\n        "coco_id": 318445,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1285696,\n          "object_id": 1179216,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_318445_1285696_1179216_1299859_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1299859,\n          "object_id": 1179216,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.5942968726158142,\n                0.4855492413043976,\n                0.6812499761581421,\n                0.7236742377281189\n              ],\n              [\n                0.8792031407356262,\n                0.609431803226471,\n                0.907828152179718,\n                0.6501514911651611\n              ],\n              [\n                0.5942968726158142,\n                0.4855492413043976,\n                0.907828152179718,\n                0.7236742377281189\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.8486875295639038,\n                0.5160984992980957,\n                0.8989843726158142,\n                0.6891098618507385\n              ],\n              [\n                0.8792031407356262,\n                0.609431803226471,\n                0.907828152179718,\n                0.6501514911651611\n              ],\n              [\n                0.8486875295639038,\n                0.5160984992980957,\n                0.907828152179718,\n                0.6891098618507385\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000318445.jpg",\n        "coco_id": 318445,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1285696,\n          "object_id": 1179216,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_318445_1285696_1179216_1312564_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1312564,\n          "object_id": 1179216,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.04378125071525574,\n                0.46570074558258057,\n                0.08726562559604645,\n                0.6857007741928101\n              ],\n              [\n                0.8792031407356262,\n                0.609431803226471,\n                0.907828152179718,\n                0.6501514911651611\n              ],\n              [\n                0.04378125071525574,\n                0.46570074558258057,\n                0.907828152179718,\n                0.6857007741928101\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.8486875295639038,\n                0.5160984992980957,\n                0.8989843726158142,\n                0.6891098618507385\n              ],\n              [\n                0.8792031407356262,\n                0.609431803226471,\n                0.907828152179718,\n                0.6501514911651611\n              ],\n              [\n                0.8486875295639038,\n                0.5160984992980957,\n                0.907828152179718,\n                0.6891098618507385\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000032206.jpg",\n        "coco_id": 32206,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "sports ball",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1302158,\n          "object_id": 300618,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_32206_1302158_300618_1738470_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1738470,\n          "object_id": 300618,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.46729686856269836,\n                0.5755624771118164,\n                0.5530937314033508,\n                0.8269166946411133\n              ],\n              [\n                0.5246875286102295,\n                0.6680625081062317,\n                0.5566250085830688,\n                0.7097916603088379\n              ],\n              [\n                0.46729686856269836,\n                0.5755624771118164,\n                0.5566250085830688,\n                0.8269166946411133\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.43482813239097595,\n                0.8046875,\n                0.5568437576293945,\n                0.960895836353302\n              ],\n              [\n                0.5246875286102295,\n                0.6680625081062317,\n                0.5566250085830688,\n                0.7097916603088379\n              ],\n              [\n                0.43482813239097595,\n                0.6680625081062317,\n                0.5568437576293945,\n                0.960895836353302\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000003236.jpg",\n        "coco_id": 3236,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bottle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1207438,\n          "object_id": 2093276,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_3236_1207438_2093276_1268300_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1268300,\n          "object_id": 2093276,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bottle",\n            "bounds": [\n              [\n                0.3246093690395355,\n                0.38008153438568115,\n                0.4618437588214874,\n                0.9464402198791504\n              ],\n              [\n                0.41801562905311584,\n                0.5773641467094421,\n                0.4493750035762787,\n                0.6248641014099121\n              ],\n              [\n                0.3246093690395355,\n                0.38008153438568115,\n                0.4618437588214874,\n                0.9464402198791504\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bottle",\n            "bounds": [\n              [\n                0.341984361410141,\n                0.7984510660171509,\n                0.5227187275886536,\n                1.0\n              ],\n              [\n                0.41801562905311584,\n                0.5773641467094421,\n                0.4493750035762787,\n                0.6248641014099121\n              ],\n              [\n                0.341984361410141,\n                0.5773641467094421,\n                0.5227187275886536,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000003236.jpg",\n        "coco_id": 3236,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1245196,\n          "object_id": 1940425,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_3236_1245196_1940425_1207438_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1207438,\n          "object_id": 1940425,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.3246093690395355,\n                0.38008153438568115,\n                0.4618437588214874,\n                0.9464402198791504\n              ],\n              [\n                0.44551563262939453,\n                0.6600815057754517,\n                0.5683906078338623,\n                0.923505425453186\n              ],\n              [\n                0.3246093690395355,\n                0.38008153438568115,\n                0.5683906078338623,\n                0.9464402198791504\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.45225000381469727,\n                0.5775271654129028,\n                0.608593761920929,\n                0.9011141061782837\n              ],\n              [\n                0.44551563262939453,\n                0.6600815057754517,\n                0.5683906078338623,\n                0.923505425453186\n              ],\n              [\n                0.44551563262939453,\n                0.5775271654129028,\n                0.608593761920929,\n                0.923505425453186\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000003236.jpg",\n        "coco_id": 3236,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1247136,\n          "object_id": 373464,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_3236_1247136_373464_1207438_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1207438,\n          "object_id": 373464,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.014218750409781933,\n                0.6067391037940979,\n                0.20545312762260437,\n                0.9617934823036194\n              ],\n              [\n                0.00015624999650754035,\n                0.7870380282402039,\n                0.14710937440395355,\n                1.0\n              ],\n              [\n                0.00015624999650754035,\n                0.6067391037940979,\n                0.20545312762260437,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.3246093690395355,\n                0.38008153438568115,\n                0.4618437588214874,\n                0.9464402198791504\n              ],\n              [\n                0.00015624999650754035,\n                0.7870380282402039,\n                0.14710937440395355,\n                1.0\n              ],\n              [\n                0.00015624999650754035,\n                0.38008153438568115,\n                0.4618437588214874,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000003236.jpg",\n        "coco_id": 3236,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "book",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1268949,\n          "object_id": 1985183,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_3236_1268949_1985183_1276398_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1276398,\n          "object_id": 1985183,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a book",\n            "bounds": [\n              [\n                0.8152031302452087,\n                0.6522554159164429,\n                0.9374687671661377,\n                1.0\n              ],\n              [\n                0.6201093792915344,\n                0.6741847991943359,\n                0.6656562685966492,\n                0.743070662021637\n              ],\n              [\n                0.6201093792915344,\n                0.6522554159164429,\n                0.9374687671661377,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a book",\n            "bounds": [\n              [\n                0.5514843463897705,\n                0.5325543284416199,\n                0.6678125262260437,\n                0.8866032361984253\n              ],\n              [\n                0.6201093792915344,\n                0.6741847991943359,\n                0.6656562685966492,\n                0.743070662021637\n              ],\n              [\n                0.5514843463897705,\n                0.5325543284416199,\n                0.6678125262260437,\n                0.8866032361984253\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000003236.jpg",\n        "coco_id": 3236,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "couch",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1276398,\n          "object_id": 99729,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_3236_1276398_99729_1207438_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1207438,\n          "object_id": 99729,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a couch",\n            "bounds": [\n              [\n                0.3246093690395355,\n                0.38008153438568115,\n                0.4618437588214874,\n                0.9464402198791504\n              ],\n              [\n                0.8841406106948853,\n                0.6969293355941772,\n                1.0,\n                0.950679361820221\n              ],\n              [\n                0.3246093690395355,\n                0.38008153438568115,\n                1.0,\n                0.950679361820221\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a couch",\n            "bounds": [\n              [\n                0.8152031302452087,\n                0.6522554159164429,\n                0.9374687671661377,\n                1.0\n              ],\n              [\n                0.8841406106948853,\n                0.6969293355941772,\n                1.0,\n                0.950679361820221\n              ],\n              [\n                0.8152031302452087,\n                0.6522554159164429,\n                1.0,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000003236.jpg",\n        "coco_id": 3236,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "couch",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1281250,\n          "object_id": 99729,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_3236_1281250_99729_1207438_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1207438,\n          "object_id": 99729,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a couch",\n            "bounds": [\n              [\n                0.7798125147819519,\n                0.5116575956344604,\n                0.9044062495231628,\n                0.9549999833106995\n              ],\n              [\n                0.8841406106948853,\n                0.6969293355941772,\n                1.0,\n                0.950679361820221\n              ],\n              [\n                0.7798125147819519,\n                0.5116575956344604,\n                1.0,\n                0.9549999833106995\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a couch",\n            "bounds": [\n              [\n                0.3246093690395355,\n                0.38008153438568115,\n                0.4618437588214874,\n                0.9464402198791504\n              ],\n              [\n                0.8841406106948853,\n                0.6969293355941772,\n                1.0,\n                0.950679361820221\n              ],\n              [\n                0.3246093690395355,\n                0.38008153438568115,\n                1.0,\n                0.950679361820221\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000003236.jpg",\n        "coco_id": 3236,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1293339,\n          "object_id": 1942454,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_3236_1293339_1942454_1207438_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1207438,\n          "object_id": 1942454,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.181640625,\n                0.7776358723640442,\n                0.347531259059906,\n                0.9992119669914246\n              ],\n              [\n                0.07218749821186066,\n                0.9149184823036194,\n                0.2445937544107437,\n                1.0\n              ],\n              [\n                0.07218749821186066,\n                0.7776358723640442,\n                0.347531259059906,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.3246093690395355,\n                0.38008153438568115,\n                0.4618437588214874,\n                0.9464402198791504\n              ],\n              [\n                0.07218749821186066,\n                0.9149184823036194,\n                0.2445937544107437,\n                1.0\n              ],\n              [\n                0.07218749821186066,\n                0.38008153438568115,\n                0.4618437588214874,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000324817.jpg",\n        "coco_id": 324817,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 191405,\n          "object_id": 1171688,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_324817_191405_1171688_196515_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 196515,\n          "object_id": 1171688,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.4517799913883209,\n                0.44651415944099426,\n                0.7488399744033813,\n                0.988082766532898\n              ],\n              [\n                0.5760200023651123,\n                0.588518500328064,\n                0.8527600169181824,\n                0.890915036201477\n              ],\n              [\n                0.4517799913883209,\n                0.44651415944099426,\n                0.8527600169181824,\n                0.988082766532898\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.1444000005722046,\n                0.4044880270957947,\n                0.5631800293922424,\n                0.986514151096344\n              ],\n              [\n                0.5760200023651123,\n                0.588518500328064,\n                0.8527600169181824,\n                0.890915036201477\n              ],\n              [\n                0.1444000005722046,\n                0.4044880270957947,\n                0.8527600169181824,\n                0.986514151096344\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000324817.jpg",\n        "coco_id": 324817,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "person",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 191405,\n          "object_id": 196515,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_324817_191405_196515_196515_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 196515,\n          "object_id": 196515,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.1444000005722046,\n                0.4044880270957947,\n                0.5631800293922424,\n                0.986514151096344\n              ],\n              [\n                0.1444000005722046,\n                0.4044880270957947,\n                0.5631800293922424,\n                0.986514151096344\n              ],\n              [\n                0.1444000005722046,\n                0.4044880270957947,\n                0.5631800293922424,\n                0.986514151096344\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.4517799913883209,\n                0.44651415944099426,\n                0.7488399744033813,\n                0.988082766532898\n              ],\n              [\n                0.1444000005722046,\n                0.4044880270957947,\n                0.5631800293922424,\n                0.986514151096344\n              ],\n              [\n                0.1444000005722046,\n                0.4044880270957947,\n                0.7488399744033813,\n                0.988082766532898\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000325530.jpg",\n        "coco_id": 325530,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 559564,\n          "object_id": 627704,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_325530_559564_627704_518314_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 518314,\n          "object_id": 627704,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.668681800365448,\n                0.41096875071525574,\n                0.8070454597473145,\n                0.6105156540870667\n              ],\n              [\n                0.9398409128189087,\n                0.5016093850135803,\n                0.94968181848526,\n                0.5776093602180481\n              ],\n              [\n                0.668681800365448,\n                0.41096875071525574,\n                0.94968181848526,\n                0.6105156540870667\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.8909772634506226,\n                0.40575000643730164,\n                0.9621590971946716,\n                0.5790781378746033\n              ],\n              [\n                0.9398409128189087,\n                0.5016093850135803,\n                0.94968181848526,\n                0.5776093602180481\n              ],\n              [\n                0.8909772634506226,\n                0.40575000643730164,\n                0.9621590971946716,\n                0.5790781378746033\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000326480.jpg",\n        "coco_id": 326480,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 508546,\n          "object_id": 329574,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_326480_508546_329574_512096_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 512096,\n          "object_id": 329574,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.6064218878746033,\n                0.1576666682958603,\n                0.7972968816757202,\n                0.8963958621025085\n              ],\n              [\n                0.684249997138977,\n                0.21695832908153534,\n                0.6981250047683716,\n                0.2617500126361847\n              ],\n              [\n                0.6064218878746033,\n                0.1576666682958603,\n                0.7972968816757202,\n                0.8963958621025085\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.34273436665534973,\n                0.10766666382551193,\n                0.5599843859672546,\n                0.8883749842643738\n              ],\n              [\n                0.684249997138977,\n                0.21695832908153534,\n                0.6981250047683716,\n                0.2617500126361847\n              ],\n              [\n                0.34273436665534973,\n                0.10766666382551193,\n                0.6981250047683716,\n                0.8883749842643738\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000003272.jpg",\n        "coco_id": 3272,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "umbrella",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 467673,\n          "object_id": 280713,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_3272_467673_280713_2165377_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2165377,\n          "object_id": 280713,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.1054375022649765,\n                0.32808858156204224,\n                0.6567500233650208,\n                0.9280886054039001\n              ],\n              [\n                0.22193750739097595,\n                0.0,\n                1.0,\n                0.2956177294254303\n              ],\n              [\n                0.1054375022649765,\n                0.0,\n                1.0,\n                0.9280886054039001\n              ]\n            ]\n          },\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.8130781054496765,\n                0.36104893684387207,\n                1.0,\n                0.9870163202285767\n              ],\n              [\n                0.22193750739097595,\n                0.0,\n                1.0,\n                0.2956177294254303\n              ],\n              [\n                0.22193750739097595,\n                0.0,\n                1.0,\n                0.9870163202285767\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000327843.jpg",\n        "coco_id": 327843,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 451107,\n          "object_id": 373870,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_327843_451107_373870_425843_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 425843,\n          "object_id": 373870,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.5069687366485596,\n                0.3258398473262787,\n                0.8431406021118164,\n                0.793261706829071\n              ],\n              [\n                0.46021875739097595,\n                0.5722070336341858,\n                0.5552031397819519,\n                0.7909179925918579\n              ],\n              [\n                0.46021875739097595,\n                0.3258398473262787,\n                0.8431406021118164,\n                0.793261706829071\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.2434062510728836,\n                0.04496093839406967,\n                0.4957500100135803,\n                0.7496484518051147\n              ],\n              [\n                0.46021875739097595,\n                0.5722070336341858,\n                0.5552031397819519,\n                0.7909179925918579\n              ],\n              [\n                0.2434062510728836,\n                0.04496093839406967,\n                0.5552031397819519,\n                0.7909179925918579\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000331278.jpg",\n        "coco_id": 331278,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bed",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2150925,\n          "object_id": 318074,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_331278_2150925_318074_1714957_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1714957,\n          "object_id": 318074,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bed",\n            "bounds": [\n              [\n                0.1594531238079071,\n                0.3226666748523712,\n                0.38917186856269836,\n                0.8573958277702332\n              ],\n              [\n                0.06404687464237213,\n                0.44718751311302185,\n                0.9168593883514404,\n                0.9730416536331177\n              ],\n              [\n                0.06404687464237213,\n                0.3226666748523712,\n                0.9168593883514404,\n                0.9730416536331177\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bed",\n            "bounds": [\n              [\n                0.24931250512599945,\n                0.5015000104904175,\n                0.7073125243186951,\n                0.9971666932106018\n              ],\n              [\n                0.06404687464237213,\n                0.44718751311302185,\n                0.9168593883514404,\n                0.9730416536331177\n              ],\n              [\n                0.06404687464237213,\n                0.44718751311302185,\n                0.9168593883514404,\n                0.9971666932106018\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000333492.jpg",\n        "coco_id": 333492,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1254825,\n          "object_id": 632756,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_333492_1254825_632756_1302178_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1302178,\n          "object_id": 632756,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.4503600001335144,\n                0.22246719896793365,\n                0.6352800130844116,\n                0.9887664318084717\n              ],\n              [\n                0.5317999720573425,\n                0.7416797876358032,\n                0.6331999897956848,\n                0.8677427768707275\n              ],\n              [\n                0.4503600001335144,\n                0.22246719896793365,\n                0.6352800130844116,\n                0.9887664318084717\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.5284799933433533,\n                0.2262992113828659,\n                0.83788001537323,\n                1.0\n              ],\n              [\n                0.5317999720573425,\n                0.7416797876358032,\n                0.6331999897956848,\n                0.8677427768707275\n              ],\n              [\n                0.5284799933433533,\n                0.2262992113828659,\n                0.83788001537323,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000335518.jpg",\n        "coco_id": 335518,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1732697,\n          "object_id": 602891,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_335518_1732697_602891_490872_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 490872,\n          "object_id": 602891,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.46854686737060547,\n                0.3460625112056732,\n                0.8089843988418579,\n                0.9438124895095825\n              ],\n              [\n                0.4494687616825104,\n                0.32506251335144043,\n                0.5161562561988831,\n                0.42266666889190674\n              ],\n              [\n                0.4494687616825104,\n                0.32506251335144043,\n                0.8089843988418579,\n                0.9438124895095825\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.6163281202316284,\n                0.5064374804496765,\n                0.7749218940734863,\n                0.9634791612625122\n              ],\n              [\n                0.4494687616825104,\n                0.32506251335144043,\n                0.5161562561988831,\n                0.42266666889190674\n              ],\n              [\n                0.4494687616825104,\n                0.32506251335144043,\n                0.7749218940734863,\n                0.9634791612625122\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000033593.jpg",\n        "coco_id": 33593,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 526020,\n          "object_id": 326911,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_33593_526020_326911_543734_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 543734,\n          "object_id": 326911,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.4750156104564667,\n                0.2093442678451538,\n                0.7372499704360962,\n                0.652341902256012\n              ],\n              [\n                0.6085312366485596,\n                0.32337236404418945,\n                0.6789531111717224,\n                0.39592504501342773\n              ],\n              [\n                0.4750156104564667,\n                0.2093442678451538,\n                0.7372499704360962,\n                0.652341902256012\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.34110936522483826,\n                0.21735362708568573,\n                0.49739062786102295,\n                0.6317564249038696\n              ],\n              [\n                0.6085312366485596,\n                0.32337236404418945,\n                0.6789531111717224,\n                0.39592504501342773\n              ],\n              [\n                0.34110936522483826,\n                0.21735362708568573,\n                0.6789531111717224,\n                0.6317564249038696\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000033593.jpg",\n        "coco_id": 33593,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 526020,\n          "object_id": 326911,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_33593_526020_326911_543734_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 543734,\n          "object_id": 326911,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.4750156104564667,\n                0.2093442678451538,\n                0.7372499704360962,\n                0.652341902256012\n              ],\n              [\n                0.6085312366485596,\n                0.32337236404418945,\n                0.6789531111717224,\n                0.39592504501342773\n              ],\n              [\n                0.4750156104564667,\n                0.2093442678451538,\n                0.7372499704360962,\n                0.652341902256012\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.34110936522483826,\n                0.21735362708568573,\n                0.49739062786102295,\n                0.6317564249038696\n              ],\n              [\n                0.6085312366485596,\n                0.32337236404418945,\n                0.6789531111717224,\n                0.39592504501342773\n              ],\n              [\n                0.34110936522483826,\n                0.21735362708568573,\n                0.6789531111717224,\n                0.6317564249038696\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000336552.jpg",\n        "coco_id": 336552,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2165112,\n          "object_id": 636829,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_336552_2165112_636829_512249_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 512249,\n          "object_id": 636829,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.4917812645435333,\n                0.5038970708847046,\n                0.625906229019165,\n                0.7870097756385803\n              ],\n              [\n                0.5126562714576721,\n                0.5242646932601929,\n                0.5333437323570251,\n                0.5477451086044312\n              ],\n              [\n                0.4917812645435333,\n                0.5038970708847046,\n                0.625906229019165,\n                0.7870097756385803\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.669546902179718,\n                0.48980391025543213,\n                0.7401875257492065,\n                0.7625490427017212\n              ],\n              [\n                0.5126562714576721,\n                0.5242646932601929,\n                0.5333437323570251,\n                0.5477451086044312\n              ],\n              [\n                0.5126562714576721,\n                0.48980391025543213,\n                0.7401875257492065,\n                0.7625490427017212\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000336552.jpg",\n        "coco_id": 336552,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 421058,\n          "object_id": 630569,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_336552_421058_630569_512249_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 512249,\n          "object_id": 630569,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.4505937397480011,\n                0.4508088231086731,\n                0.5188906192779541,\n                0.7547793984413147\n              ],\n              [\n                0.4345625042915344,\n                0.6082352995872498,\n                0.4594687521457672,\n                0.698774516582489\n              ],\n              [\n                0.4345625042915344,\n                0.4508088231086731,\n                0.5188906192779541,\n                0.7547793984413147\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.669546902179718,\n                0.48980391025543213,\n                0.7401875257492065,\n                0.7625490427017212\n              ],\n              [\n                0.4345625042915344,\n                0.6082352995872498,\n                0.4594687521457672,\n                0.698774516582489\n              ],\n              [\n                0.4345625042915344,\n                0.48980391025543213,\n                0.7401875257492065,\n                0.7625490427017212\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000339248.jpg",\n        "coco_id": 339248,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 498919,\n          "object_id": 324354,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_339248_498919_324354_535785_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 535785,\n          "object_id": 324354,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.001687500043772161,\n                0.2917500138282776,\n                0.4533750116825104,\n                1.0\n              ],\n              [\n                0.20106250047683716,\n                0.47333332896232605,\n                0.2358749955892563,\n                0.5763750076293945\n              ],\n              [\n                0.001687500043772161,\n                0.2917500138282776,\n                0.4533750116825104,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.7009062767028809,\n                0.42268750071525574,\n                1.0,\n                0.989270806312561\n              ],\n              [\n                0.20106250047683716,\n                0.47333332896232605,\n                0.2358749955892563,\n                0.5763750076293945\n              ],\n              [\n                0.20106250047683716,\n                0.42268750071525574,\n                1.0,\n                0.989270806312561\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000342340.jpg",\n        "coco_id": 342340,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "dog",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 183921,\n          "object_id": 12224,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_342340_183921_12224_183707_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 183707,\n          "object_id": 12224,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a dog",\n            "bounds": [\n              [\n                0.3789375126361847,\n                0.13627038896083832,\n                0.8031718730926514,\n                0.84797203540802\n              ],\n              [\n                0.5267656445503235,\n                0.38787877559661865,\n                0.9370781183242798,\n                0.9313053488731384\n              ],\n              [\n                0.3789375126361847,\n                0.13627038896083832,\n                0.9370781183242798,\n                0.9313053488731384\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a dog",\n            "bounds": [\n              [\n                0.14493750035762787,\n                0.17904429137706757,\n                0.7533437609672546,\n                0.9898601174354553\n              ],\n              [\n                0.5267656445503235,\n                0.38787877559661865,\n                0.9370781183242798,\n                0.9313053488731384\n              ],\n              [\n                0.14493750035762787,\n                0.17904429137706757,\n                0.9370781183242798,\n                0.9898601174354553\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000345879.jpg",\n        "coco_id": 345879,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 184247,\n          "object_id": 1594007,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_345879_184247_1594007_203209_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 203209,\n          "object_id": 1594007,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.5710312724113464,\n                0.4428773522377014,\n                0.6501250267028809,\n                0.6919811367988586\n              ],\n              [\n                0.29176563024520874,\n                0.6187971830368042,\n                0.3285312354564667,\n                0.700754702091217\n              ],\n              [\n                0.29176563024520874,\n                0.4428773522377014,\n                0.6501250267028809,\n                0.700754702091217\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.3095781207084656,\n                0.5281603932380676,\n                0.4269687533378601,\n                0.7004480957984924\n              ],\n              [\n                0.29176563024520874,\n                0.6187971830368042,\n                0.3285312354564667,\n                0.700754702091217\n              ],\n              [\n                0.29176563024520874,\n                0.5281603932380676,\n                0.4269687533378601,\n                0.700754702091217\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000345879.jpg",\n        "coco_id": 345879,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 189792,\n          "object_id": 1593712,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_345879_189792_1593712_203209_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 203209,\n          "object_id": 1593712,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.5710312724113464,\n                0.4428773522377014,\n                0.6501250267028809,\n                0.6919811367988586\n              ],\n              [\n                0.4463750123977661,\n                0.6225707530975342,\n                0.5113593935966492,\n                0.6958019137382507\n              ],\n              [\n                0.4463750123977661,\n                0.4428773522377014,\n                0.6501250267028809,\n                0.6958019137382507\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.4460468888282776,\n                0.5199764370918274,\n                0.5262500047683716,\n                0.6996933817863464\n              ],\n              [\n                0.4463750123977661,\n                0.6225707530975342,\n                0.5113593935966492,\n                0.6958019137382507\n              ],\n              [\n                0.4460468888282776,\n                0.5199764370918274,\n                0.5262500047683716,\n                0.6996933817863464\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000348370.jpg",\n        "coco_id": 348370,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "snowboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1189395,\n          "object_id": 619738,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_348370_1189395_619738_525561_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 525561,\n          "object_id": 619738,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.8295312523841858,\n                0.4334791600704193,\n                1.0,\n                0.9805208444595337\n              ],\n              [\n                0.8692343831062317,\n                0.5005833506584167,\n                0.9817812442779541,\n                0.999916672706604\n              ],\n              [\n                0.8295312523841858,\n                0.4334791600704193,\n                1.0,\n                0.999916672706604\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.7042499780654907,\n                0.3986458480358124,\n                0.8371718525886536,\n                0.9991250038146973\n              ],\n              [\n                0.8692343831062317,\n                0.5005833506584167,\n                0.9817812442779541,\n                0.999916672706604\n              ],\n              [\n                0.7042499780654907,\n                0.3986458480358124,\n                0.9817812442779541,\n                0.999916672706604\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000348370.jpg",\n        "coco_id": 348370,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "snowboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 492077,\n          "object_id": 619329,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_348370_492077_619329_475764_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 475764,\n          "object_id": 619329,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.38071873784065247,\n                0.3723958432674408,\n                0.5218437314033508,\n                0.9844791889190674\n              ],\n              [\n                0.3933750092983246,\n                0.4841666519641876,\n                0.48801562190055847,\n                1.0\n              ],\n              [\n                0.38071873784065247,\n                0.3723958432674408,\n                0.5218437314033508,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.21790625154972076,\n                0.34685418009757996,\n                0.38514062762260437,\n                0.9842291474342346\n              ],\n              [\n                0.3933750092983246,\n                0.4841666519641876,\n                0.48801562190055847,\n                1.0\n              ],\n              [\n                0.21790625154972076,\n                0.34685418009757996,\n                0.48801562190055847,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000348370.jpg",\n        "coco_id": 348370,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skis",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 561557,\n          "object_id": 1849399,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_348370_561557_1849399_525561_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 525561,\n          "object_id": 1849399,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding skis",\n            "bounds": [\n              [\n                0.7042499780654907,\n                0.3986458480358124,\n                0.8371718525886536,\n                0.9991250038146973\n              ],\n              [\n                0.025828124955296516,\n                0.4463750123977661,\n                0.1021406278014183,\n                1.0\n              ],\n              [\n                0.025828124955296516,\n                0.3986458480358124,\n                0.8371718525886536,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding skis",\n            "bounds": [\n              [\n                0.06845312565565109,\n                0.37562501430511475,\n                0.21385937929153442,\n                0.9994166493415833\n              ],\n              [\n                0.025828124955296516,\n                0.4463750123977661,\n                0.1021406278014183,\n                1.0\n              ],\n              [\n                0.025828124955296516,\n                0.37562501430511475,\n                0.21385937929153442,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000349934.jpg",\n        "coco_id": 349934,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 490320,\n          "object_id": 1844603,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_349934_490320_1844603_2205485_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2205485,\n          "object_id": 1844603,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.29692187905311584,\n                0.3843578100204468,\n                0.5676718950271606,\n                0.9544495344161987\n              ],\n              [\n                0.2584843635559082,\n                0.7221559882164001,\n                0.3578437566757202,\n                0.8160091638565063\n              ],\n              [\n                0.2584843635559082,\n                0.3843578100204468,\n                0.5676718950271606,\n                0.9544495344161987\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.4311562478542328,\n                0.32864677906036377,\n                0.6291249990463257,\n                0.9535320997238159\n              ],\n              [\n                0.2584843635559082,\n                0.7221559882164001,\n                0.3578437566757202,\n                0.8160091638565063\n              ],\n              [\n                0.2584843635559082,\n                0.32864677906036377,\n                0.6291249990463257,\n                0.9535320997238159\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000350560.jpg",\n        "coco_id": 350560,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1331870,\n          "object_id": 330124,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_350560_1331870_330124_1242011_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1242011,\n          "object_id": 330124,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.5028799772262573,\n                0.6359580755233765,\n                0.7790799736976624,\n                0.988772451877594\n              ],\n              [\n                0.2202800065279007,\n                0.8735329508781433,\n                0.26646000146865845,\n                1.0\n              ],\n              [\n                0.2202800065279007,\n                0.6359580755233765,\n                0.7790799736976624,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.00022000000171829015,\n                0.7099401354789734,\n                0.25745999813079834,\n                1.0\n              ],\n              [\n                0.2202800065279007,\n                0.8735329508781433,\n                0.26646000146865845,\n                1.0\n              ],\n              [\n                0.00022000000171829015,\n                0.7099401354789734,\n                0.26646000146865845,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000353809.jpg",\n        "coco_id": 353809,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1305132,\n          "object_id": 1415936,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_353809_1305132_1415936_1274270_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1274270,\n          "object_id": 1415936,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.1596718728542328,\n                0.348305881023407,\n                0.28779688477516174,\n                0.8561882376670837\n              ],\n              [\n                0.4850468635559082,\n                0.676800012588501,\n                0.5457812547683716,\n                0.7996705770492554\n              ],\n              [\n                0.1596718728542328,\n                0.348305881023407,\n                0.5457812547683716,\n                0.8561882376670837\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.46950000524520874,\n                0.5225411653518677,\n                0.6314843893051147,\n                0.7963058948516846\n              ],\n              [\n                0.4850468635559082,\n                0.676800012588501,\n                0.5457812547683716,\n                0.7996705770492554\n              ],\n              [\n                0.46950000524520874,\n                0.5225411653518677,\n                0.6314843893051147,\n                0.7996705770492554\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000355064.jpg",\n        "coco_id": 355064,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 190554,\n          "object_id": 1790232,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_355064_190554_1790232_212862_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 212862,\n          "object_id": 1790232,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.14839062094688416,\n                0.2865625023841858,\n                0.49678125977516174,\n                0.987625002861023\n              ],\n              [\n                0.0,\n                0.3967708349227905,\n                0.9277812242507935,\n                0.8930624723434448\n              ],\n              [\n                0.0,\n                0.2865625023841858,\n                0.9277812242507935,\n                0.987625002861023\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.3622968792915344,\n                0.24716666340827942,\n                0.7162031531333923,\n                1.0\n              ],\n              [\n                0.0,\n                0.3967708349227905,\n                0.9277812242507935,\n                0.8930624723434448\n              ],\n              [\n                0.0,\n                0.24716666340827942,\n                0.9277812242507935,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000355555.jpg",\n        "coco_id": 355555,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 204016,\n          "object_id": 322796,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_355555_204016_322796_225921_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 225921,\n          "object_id": 322796,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.39787501096725464,\n                0.06307511776685715,\n                0.6581405997276306,\n                0.8630751371383667\n              ],\n              [\n                0.5871250033378601,\n                0.16265258193016052,\n                0.6000156402587891,\n                0.19239436089992523\n              ],\n              [\n                0.39787501096725464,\n                0.06307511776685715,\n                0.6581405997276306,\n                0.8630751371383667\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.0,\n                0.006361502222716808,\n                0.38890624046325684,\n                0.7367135882377625\n              ],\n              [\n                0.5871250033378601,\n                0.16265258193016052,\n                0.6000156402587891,\n                0.19239436089992523\n              ],\n              [\n                0.0,\n                0.006361502222716808,\n                0.6000156402587891,\n                0.7367135882377625\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000355555.jpg",\n        "coco_id": 355555,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "dog",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 204016,\n          "object_id": 5656,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_355555_204016_5656_225921_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 225921,\n          "object_id": 5656,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a dog",\n            "bounds": [\n              [\n                0.39787501096725464,\n                0.06307511776685715,\n                0.6581405997276306,\n                0.8630751371383667\n              ],\n              [\n                0.44401562213897705,\n                0.16129107773303986,\n                0.594656229019165,\n                0.5190845131874084\n              ],\n              [\n                0.39787501096725464,\n                0.06307511776685715,\n                0.6581405997276306,\n                0.8630751371383667\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a dog",\n            "bounds": [\n              [\n                0.0,\n                0.006361502222716808,\n                0.38890624046325684,\n                0.7367135882377625\n              ],\n              [\n                0.44401562213897705,\n                0.16129107773303986,\n                0.594656229019165,\n                0.5190845131874084\n              ],\n              [\n                0.0,\n                0.006361502222716808,\n                0.594656229019165,\n                0.7367135882377625\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000358706.jpg",\n        "coco_id": 358706,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1271640,\n          "object_id": 150118,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_358706_1271640_150118_1305079_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1305079,\n          "object_id": 150118,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.6398593783378601,\n                0.17986559867858887,\n                0.8295781016349792,\n                0.7334946393966675\n              ],\n              [\n                0.11573437601327896,\n                0.31129032373428345,\n                0.8686249852180481,\n                0.9041128754615784\n              ],\n              [\n                0.11573437601327896,\n                0.17986559867858887,\n                0.8686249852180481,\n                0.9041128754615784\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.8359531164169312,\n                0.05392473191022873,\n                0.9992343783378601,\n                0.7505645155906677\n              ],\n              [\n                0.11573437601327896,\n                0.31129032373428345,\n                0.8686249852180481,\n                0.9041128754615784\n              ],\n              [\n                0.11573437601327896,\n                0.05392473191022873,\n                0.9992343783378601,\n                0.9041128754615784\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000358706.jpg",\n        "coco_id": 358706,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1271640,\n          "object_id": 150118,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_358706_1271640_150118_1305079_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1305079,\n          "object_id": 150118,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.6398593783378601,\n                0.17986559867858887,\n                0.8295781016349792,\n                0.7334946393966675\n              ],\n              [\n                0.11573437601327896,\n                0.31129032373428345,\n                0.8686249852180481,\n                0.9041128754615784\n              ],\n              [\n                0.11573437601327896,\n                0.17986559867858887,\n                0.8686249852180481,\n                0.9041128754615784\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.8359531164169312,\n                0.05392473191022873,\n                0.9992343783378601,\n                0.7505645155906677\n              ],\n              [\n                0.11573437601327896,\n                0.31129032373428345,\n                0.8686249852180481,\n                0.9041128754615784\n              ],\n              [\n                0.11573437601327896,\n                0.05392473191022873,\n                0.9992343783378601,\n                0.9041128754615784\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000358706.jpg",\n        "coco_id": 358706,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1302846,\n          "object_id": 150118,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_358706_1302846_150118_1305079_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1305079,\n          "object_id": 150118,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.4454062581062317,\n                0.22145161032676697,\n                0.7366874814033508,\n                0.8855376243591309\n              ],\n              [\n                0.11573437601327896,\n                0.31129032373428345,\n                0.8686249852180481,\n                0.9041128754615784\n              ],\n              [\n                0.11573437601327896,\n                0.22145161032676697,\n                0.8686249852180481,\n                0.9041128754615784\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.8359531164169312,\n                0.05392473191022873,\n                0.9992343783378601,\n                0.7505645155906677\n              ],\n              [\n                0.11573437601327896,\n                0.31129032373428345,\n                0.8686249852180481,\n                0.9041128754615784\n              ],\n              [\n                0.11573437601327896,\n                0.05392473191022873,\n                0.9992343783378601,\n                0.9041128754615784\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000358706.jpg",\n        "coco_id": 358706,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1302846,\n          "object_id": 150118,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_358706_1302846_150118_1305079_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1305079,\n          "object_id": 150118,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.8359531164169312,\n                0.05392473191022873,\n                0.9992343783378601,\n                0.7505645155906677\n              ],\n              [\n                0.11573437601327896,\n                0.31129032373428345,\n                0.8686249852180481,\n                0.9041128754615784\n              ],\n              [\n                0.11573437601327896,\n                0.05392473191022873,\n                0.9992343783378601,\n                0.9041128754615784\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.4454062581062317,\n                0.22145161032676697,\n                0.7366874814033508,\n                0.8855376243591309\n              ],\n              [\n                0.11573437601327896,\n                0.31129032373428345,\n                0.8686249852180481,\n                0.9041128754615784\n              ],\n              [\n                0.11573437601327896,\n                0.22145161032676697,\n                0.8686249852180481,\n                0.9041128754615784\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000365259.jpg",\n        "coco_id": 365259,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "person",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 481913,\n          "object_id": 477368,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_365259_481913_477368_477368_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 477368,\n          "object_id": 477368,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.45762500166893005,\n                0.3929166793823242,\n                0.7845312356948853,\n                1.0\n              ],\n              [\n                0.6471874713897705,\n                0.559541642665863,\n                0.8595468997955322,\n                1.0\n              ],\n              [\n                0.45762500166893005,\n                0.3929166793823242,\n                0.8595468997955322,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.6471874713897705,\n                0.559541642665863,\n                0.8595468997955322,\n                1.0\n              ],\n              [\n                0.6471874713897705,\n                0.559541642665863,\n                0.8595468997955322,\n                1.0\n              ],\n              [\n                0.6471874713897705,\n                0.559541642665863,\n                0.8595468997955322,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000368460.jpg",\n        "coco_id": 368460,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 472503,\n          "object_id": 604110,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_368460_472503_604110_469974_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 469974,\n          "object_id": 604110,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.6194687485694885,\n                0.43197181820869446,\n                0.7430624961853027,\n                0.8816431760787964\n              ],\n              [\n                0.34956249594688416,\n                0.43725353479385376,\n                0.4054843783378601,\n                0.4915492832660675\n              ],\n              [\n                0.34956249594688416,\n                0.43197181820869446,\n                0.7430624961853027,\n                0.8816431760787964\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.35151562094688416,\n                0.29213616251945496,\n                0.5085625052452087,\n                0.813474178314209\n              ],\n              [\n                0.34956249594688416,\n                0.43725353479385376,\n                0.4054843783378601,\n                0.4915492832660675\n              ],\n              [\n                0.34956249594688416,\n                0.29213616251945496,\n                0.5085625052452087,\n                0.813474178314209\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000372130.jpg",\n        "coco_id": 372130,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1305966,\n          "object_id": 1168332,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_372130_1305966_1168332_1728177_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1728177,\n          "object_id": 1168332,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.3024531304836273,\n                0.4542117714881897,\n                0.4421406388282776,\n                0.8602588176727295\n              ],\n              [\n                0.2660468816757202,\n                0.5309882164001465,\n                0.38881251215934753,\n                0.7305411696434021\n              ],\n              [\n                0.2660468816757202,\n                0.4542117714881897,\n                0.4421406388282776,\n                0.8602588176727295\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.0,\n                0.4472000002861023,\n                0.11042187362909317,\n                0.9280940890312195\n              ],\n              [\n                0.2660468816757202,\n                0.5309882164001465,\n                0.38881251215934753,\n                0.7305411696434021\n              ],\n              [\n                0.0,\n                0.4472000002861023,\n                0.38881251215934753,\n                0.9280940890312195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000372130.jpg",\n        "coco_id": 372130,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1305966,\n          "object_id": 2177647,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_372130_1305966_2177647_1705207_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1705207,\n          "object_id": 2177647,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.7900937795639038,\n                0.5112705826759338,\n                0.8764687776565552,\n                0.9050588011741638\n              ],\n              [\n                0.1724843680858612,\n                0.8753176331520081,\n                0.30701562762260437,\n                0.9844235181808472\n              ],\n              [\n                0.1724843680858612,\n                0.5112705826759338,\n                0.8764687776565552,\n                0.9844235181808472\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.3024531304836273,\n                0.4542117714881897,\n                0.4421406388282776,\n                0.8602588176727295\n              ],\n              [\n                0.1724843680858612,\n                0.8753176331520081,\n                0.30701562762260437,\n                0.9844235181808472\n              ],\n              [\n                0.1724843680858612,\n                0.4542117714881897,\n                0.4421406388282776,\n                0.9844235181808472\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000372130.jpg",\n        "coco_id": 372130,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1317949,\n          "object_id": 1176699,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_372130_1317949_1176699_1728177_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1728177,\n          "object_id": 1176699,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.606374979019165,\n                0.41917645931243896,\n                0.7324844002723694,\n                1.0\n              ],\n              [\n                0.6008750200271606,\n                0.5416235327720642,\n                0.6951562762260437,\n                0.7947294116020203\n              ],\n              [\n                0.6008750200271606,\n                0.41917645931243896,\n                0.7324844002723694,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.0,\n                0.4472000002861023,\n                0.11042187362909317,\n                0.9280940890312195\n              ],\n              [\n                0.6008750200271606,\n                0.5416235327720642,\n                0.6951562762260437,\n                0.7947294116020203\n              ],\n              [\n                0.0,\n                0.4472000002861023,\n                0.6951562762260437,\n                0.9280940890312195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000372130.jpg",\n        "coco_id": 372130,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1323839,\n          "object_id": 1187169,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_372130_1323839_1187169_1317949_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1317949,\n          "object_id": 1187169,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.1204531267285347,\n                0.47501176595687866,\n                0.2528750002384186,\n                0.97785884141922\n              ],\n              [\n                0.050734374672174454,\n                0.8044941425323486,\n                0.12087500095367432,\n                0.9101176261901855\n              ],\n              [\n                0.050734374672174454,\n                0.47501176595687866,\n                0.2528750002384186,\n                0.97785884141922\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.606374979019165,\n                0.41917645931243896,\n                0.7324844002723694,\n                1.0\n              ],\n              [\n                0.050734374672174454,\n                0.8044941425323486,\n                0.12087500095367432,\n                0.9101176261901855\n              ],\n              [\n                0.050734374672174454,\n                0.41917645931243896,\n                0.7324844002723694,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000372130.jpg",\n        "coco_id": 372130,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1323839,\n          "object_id": 1187169,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_372130_1323839_1187169_1728177_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1728177,\n          "object_id": 1187169,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.1204531267285347,\n                0.47501176595687866,\n                0.2528750002384186,\n                0.97785884141922\n              ],\n              [\n                0.050734374672174454,\n                0.8044941425323486,\n                0.12087500095367432,\n                0.9101176261901855\n              ],\n              [\n                0.050734374672174454,\n                0.47501176595687866,\n                0.2528750002384186,\n                0.97785884141922\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.0,\n                0.4472000002861023,\n                0.11042187362909317,\n                0.9280940890312195\n              ],\n              [\n                0.050734374672174454,\n                0.8044941425323486,\n                0.12087500095367432,\n                0.9101176261901855\n              ],\n              [\n                0.0,\n                0.4472000002861023,\n                0.12087500095367432,\n                0.9280940890312195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000372130.jpg",\n        "coco_id": 372130,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1705207,\n          "object_id": 1432030,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_372130_1705207_1432030_1696473_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1696473,\n          "object_id": 1432030,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.7900937795639038,\n                0.5112705826759338,\n                0.8764687776565552,\n                0.9050588011741638\n              ],\n              [\n                0.8699843883514404,\n                0.6270353198051453,\n                0.9122187495231628,\n                0.7659294009208679\n              ],\n              [\n                0.7900937795639038,\n                0.5112705826759338,\n                0.9122187495231628,\n                0.9050588011741638\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.7182968854904175,\n                0.486141175031662,\n                0.8053281307220459,\n                0.9341411590576172\n              ],\n              [\n                0.8699843883514404,\n                0.6270353198051453,\n                0.9122187495231628,\n                0.7659294009208679\n              ],\n              [\n                0.7182968854904175,\n                0.486141175031662,\n                0.9122187495231628,\n                0.9341411590576172\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000372130.jpg",\n        "coco_id": 372130,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1728177,\n          "object_id": 1187169,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_372130_1728177_1187169_1240197_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1240197,\n          "object_id": 1187169,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.0,\n                0.4472000002861023,\n                0.11042187362909317,\n                0.9280940890312195\n              ],\n              [\n                0.050734374672174454,\n                0.8044941425323486,\n                0.12087500095367432,\n                0.9101176261901855\n              ],\n              [\n                0.0,\n                0.4472000002861023,\n                0.12087500095367432,\n                0.9280940890312195\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.48728126287460327,\n                0.502752959728241,\n                0.6077812314033508,\n                0.9597882628440857\n              ],\n              [\n                0.050734374672174454,\n                0.8044941425323486,\n                0.12087500095367432,\n                0.9101176261901855\n              ],\n              [\n                0.050734374672174454,\n                0.502752959728241,\n                0.6077812314033508,\n                0.9597882628440857\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000373686.jpg",\n        "coco_id": 373686,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 479414,\n          "object_id": 650118,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_373686_479414_650118_499048_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 499048,\n          "object_id": 650118,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.09948437660932541,\n                0.6320624947547913,\n                0.2880156338214874,\n                0.7044374942779541\n              ],\n              [\n                0.579828143119812,\n                0.46554166078567505,\n                0.6721093654632568,\n                0.6199583411216736\n              ],\n              [\n                0.09948437660932541,\n                0.46554166078567505,\n                0.6721093654632568,\n                0.7044374942779541\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.5375468730926514,\n                0.437583327293396,\n                0.6236562728881836,\n                0.6151041388511658\n              ],\n              [\n                0.579828143119812,\n                0.46554166078567505,\n                0.6721093654632568,\n                0.6199583411216736\n              ],\n              [\n                0.5375468730926514,\n                0.437583327293396,\n                0.6721093654632568,\n                0.6199583411216736\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000375759.jpg",\n        "coco_id": 375759,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1207493,\n          "object_id": 605281,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_375759_1207493_605281_1286446_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1286446,\n          "object_id": 605281,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.0,\n                0.49213114380836487,\n                0.24889062345027924,\n                1.0\n              ],\n              [\n                0.23732812702655792,\n                0.628641664981842,\n                0.2835937440395355,\n                0.7695784568786621\n              ],\n              [\n                0.0,\n                0.49213114380836487,\n                0.2835937440395355,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.18443749845027924,\n                0.2982669770717621,\n                0.3184531331062317,\n                0.9535596966743469\n              ],\n              [\n                0.23732812702655792,\n                0.628641664981842,\n                0.2835937440395355,\n                0.7695784568786621\n              ],\n              [\n                0.18443749845027924,\n                0.2982669770717621,\n                0.3184531331062317,\n                0.9535596966743469\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000375759.jpg",\n        "coco_id": 375759,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1223279,\n          "object_id": 604910,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_375759_1223279_604910_1222979_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1222979,\n          "object_id": 604910,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.43479686975479126,\n                0.15056206285953522,\n                0.5937187671661377,\n                0.8426932096481323\n              ],\n              [\n                0.41564062237739563,\n                0.03887587785720825,\n                0.48579686880111694,\n                0.14402809739112854\n              ],\n              [\n                0.41564062237739563,\n                0.03887587785720825,\n                0.5937187671661377,\n                0.8426932096481323\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.00010937500337604433,\n                0.20683841407299042,\n                0.24931250512599945,\n                0.6553863883018494\n              ],\n              [\n                0.41564062237739563,\n                0.03887587785720825,\n                0.48579686880111694,\n                0.14402809739112854\n              ],\n              [\n                0.00010937500337604433,\n                0.03887587785720825,\n                0.48579686880111694,\n                0.6553863883018494\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000379777.jpg",\n        "coco_id": 379777,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 516387,\n          "object_id": 682769,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_379777_516387_682769_524324_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 524324,\n          "object_id": 682769,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.0,\n                0.3089374899864197,\n                0.37415623664855957,\n                0.8950625061988831\n              ],\n              [\n                0.6481719017028809,\n                0.6939374804496765,\n                0.7018749713897705,\n                0.8841041922569275\n              ],\n              [\n                0.0,\n                0.3089374899864197,\n                0.7018749713897705,\n                0.8950625061988831\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.1672343760728836,\n                0.42004168033599854,\n                0.6672343611717224,\n                0.9876041412353516\n              ],\n              [\n                0.6481719017028809,\n                0.6939374804496765,\n                0.7018749713897705,\n                0.8841041922569275\n              ],\n              [\n                0.1672343760728836,\n                0.42004168033599854,\n                0.7018749713897705,\n                0.9876041412353516\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000380429.jpg",\n        "coco_id": 380429,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "knife",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1715808,\n          "object_id": 693093,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_380429_1715808_693093_547302_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 547302,\n          "object_id": 693093,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.10068749636411667,\n                0.12516355514526367,\n                0.6507187485694885,\n                0.8779672980308533\n              ],\n              [\n                0.6872344017028809,\n                0.6807944178581238,\n                0.7202968597412109,\n                0.8111215233802795\n              ],\n              [\n                0.10068749636411667,\n                0.12516355514526367,\n                0.7202968597412109,\n                0.8779672980308533\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.5102031230926514,\n                0.0,\n                0.9202499985694885,\n                0.8321962356567383\n              ],\n              [\n                0.6872344017028809,\n                0.6807944178581238,\n                0.7202968597412109,\n                0.8111215233802795\n              ],\n              [\n                0.5102031230926514,\n                0.0,\n                0.9202499985694885,\n                0.8321962356567383\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000038370.jpg",\n        "coco_id": 38370,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 188476,\n          "object_id": 2183441,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_38370_188476_2183441_254347_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 254347,\n          "object_id": 2183441,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.5580624938011169,\n                0.16183333098888397,\n                0.8725781440734863,\n                0.7059166431427002\n              ],\n              [\n                0.5521875023841858,\n                0.37424999475479126,\n                0.579421877861023,\n                0.45279166102409363\n              ],\n              [\n                0.5521875023841858,\n                0.16183333098888397,\n                0.8725781440734863,\n                0.7059166431427002\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.06925000250339508,\n                0.2241041660308838,\n                0.2787187397480011,\n                0.6137291789054871\n              ],\n              [\n                0.5521875023841858,\n                0.37424999475479126,\n                0.579421877861023,\n                0.45279166102409363\n              ],\n              [\n                0.06925000250339508,\n                0.2241041660308838,\n                0.579421877861023,\n                0.6137291789054871\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000038370.jpg",\n        "coco_id": 38370,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "hot dog",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 257505,\n          "object_id": 1069130,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_38370_257505_1069130_254347_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 254347,\n          "object_id": 1069130,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a hot dog",\n            "bounds": [\n              [\n                0.22584375739097595,\n                0.36629167199134827,\n                0.4971874952316284,\n                0.8741666674613953\n              ],\n              [\n                0.45246875286102295,\n                0.4975833296775818,\n                0.5278437733650208,\n                0.5399583578109741\n              ],\n              [\n                0.22584375739097595,\n                0.36629167199134827,\n                0.5278437733650208,\n                0.8741666674613953\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a hot dog",\n            "bounds": [\n              [\n                0.06925000250339508,\n                0.2241041660308838,\n                0.2787187397480011,\n                0.6137291789054871\n              ],\n              [\n                0.45246875286102295,\n                0.4975833296775818,\n                0.5278437733650208,\n                0.5399583578109741\n              ],\n              [\n                0.06925000250339508,\n                0.2241041660308838,\n                0.5278437733650208,\n                0.6137291789054871\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000038370.jpg",\n        "coco_id": 38370,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "fork",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 257505,\n          "object_id": 687846,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_38370_257505_687846_254347_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 254347,\n          "object_id": 687846,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.06925000250339508,\n                0.2241041660308838,\n                0.2787187397480011,\n                0.6137291789054871\n              ],\n              [\n                0.3897656202316284,\n                0.5056250095367432,\n                0.4379531145095825,\n                0.5941666960716248\n              ],\n              [\n                0.06925000250339508,\n                0.2241041660308838,\n                0.4379531145095825,\n                0.6137291789054871\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.22584375739097595,\n                0.36629167199134827,\n                0.4971874952316284,\n                0.8741666674613953\n              ],\n              [\n                0.3897656202316284,\n                0.5056250095367432,\n                0.4379531145095825,\n                0.5941666960716248\n              ],\n              [\n                0.22584375739097595,\n                0.36629167199134827,\n                0.4971874952316284,\n                0.8741666674613953\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000387124.jpg",\n        "coco_id": 387124,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 200738,\n          "object_id": 1765174,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_387124_200738_1765174_192322_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 192322,\n          "object_id": 1765174,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.26055556535720825,\n                0.39774999022483826,\n                0.5905555486679077,\n                0.9078593850135803\n              ],\n              [\n                0.218478262424469,\n                0.7108281254768372,\n                0.5491062998771667,\n                0.8989218473434448\n              ],\n              [\n                0.218478262424469,\n                0.39774999022483826,\n                0.5905555486679077,\n                0.9078593850135803\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.4435748755931854,\n                0.48173436522483826,\n                0.780434787273407,\n                0.9016249775886536\n              ],\n              [\n                0.218478262424469,\n                0.7108281254768372,\n                0.5491062998771667,\n                0.8989218473434448\n              ],\n              [\n                0.218478262424469,\n                0.48173436522483826,\n                0.780434787273407,\n                0.9016249775886536\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000387333.jpg",\n        "coco_id": 387333,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "wine glass",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2156958,\n          "object_id": 667527,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_387333_2156958_667527_488814_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 488814,\n          "object_id": 667527,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a wine glass",\n            "bounds": [\n              [\n                0.2768799960613251,\n                0.2966366410255432,\n                0.5522599816322327,\n                0.9887687563896179\n              ],\n              [\n                0.5766199827194214,\n                0.4743843972682953,\n                0.6104999780654907,\n                0.5981681942939758\n              ],\n              [\n                0.2768799960613251,\n                0.2966366410255432,\n                0.6104999780654907,\n                0.9887687563896179\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a wine glass",\n            "bounds": [\n              [\n                0.518060028553009,\n                0.2035435438156128,\n                0.7224799990653992,\n                1.0\n              ],\n              [\n                0.5766199827194214,\n                0.4743843972682953,\n                0.6104999780654907,\n                0.5981681942939758\n              ],\n              [\n                0.518060028553009,\n                0.2035435438156128,\n                0.7224799990653992,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000387976.jpg",\n        "coco_id": 387976,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "sandwich",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 470450,\n          "object_id": 2217908,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_387976_470450_2217908_424097_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 424097,\n          "object_id": 2217908,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a sandwich",\n            "bounds": [\n              [\n                0.5157679915428162,\n                0.293921560049057,\n                1.0,\n                0.7218464016914368\n              ],\n              [\n                0.6943300366401672,\n                0.6077777743339539,\n                0.7647548913955688,\n                0.6532843112945557\n              ],\n              [\n                0.5157679915428162,\n                0.293921560049057,\n                1.0,\n                0.7218464016914368\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a sandwich",\n            "bounds": [\n              [\n                0.002254901919513941,\n                0.007908497005701065,\n                0.5395098328590393,\n                0.6580065488815308\n              ],\n              [\n                0.6943300366401672,\n                0.6077777743339539,\n                0.7647548913955688,\n                0.6532843112945557\n              ],\n              [\n                0.002254901919513941,\n                0.007908497005701065,\n                0.7647548913955688,\n                0.6580065488815308\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000039078.jpg",\n        "coco_id": 39078,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 190243,\n          "object_id": 1180579,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_39078_190243_1180579_196818_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 196818,\n          "object_id": 1180579,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.8722344040870667,\n                0.5468149781227112,\n                0.968093752861023,\n                0.912599503993988\n              ],\n              [\n                0.829703152179718,\n                0.6571428775787354,\n                0.9026562571525574,\n                0.7991334795951843\n              ],\n              [\n                0.829703152179718,\n                0.5468149781227112,\n                0.968093752861023,\n                0.912599503993988\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.7975468635559082,\n                0.5772599577903748,\n                0.9032031297683716,\n                0.9213582873344421\n              ],\n              [\n                0.829703152179718,\n                0.6571428775787354,\n                0.9026562571525574,\n                0.7991334795951843\n              ],\n              [\n                0.7975468635559082,\n                0.5772599577903748,\n                0.9032031297683716,\n                0.9213582873344421\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000039078.jpg",\n        "coco_id": 39078,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 190243,\n          "object_id": 1180579,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_39078_190243_1180579_196818_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 196818,\n          "object_id": 1180579,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.8722344040870667,\n                0.5468149781227112,\n                0.968093752861023,\n                0.912599503993988\n              ],\n              [\n                0.829703152179718,\n                0.6571428775787354,\n                0.9026562571525574,\n                0.7991334795951843\n              ],\n              [\n                0.829703152179718,\n                0.5468149781227112,\n                0.968093752861023,\n                0.912599503993988\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.7975468635559082,\n                0.5772599577903748,\n                0.9032031297683716,\n                0.9213582873344421\n              ],\n              [\n                0.829703152179718,\n                0.6571428775787354,\n                0.9026562571525574,\n                0.7991334795951843\n              ],\n              [\n                0.7975468635559082,\n                0.5772599577903748,\n                0.9032031297683716,\n                0.9213582873344421\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000392793.jpg",\n        "coco_id": 392793,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 205979,\n          "object_id": 56841,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_392793_205979_56841_206116_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 206116,\n          "object_id": 56841,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.466921865940094,\n                0.09572429955005646,\n                0.8058124780654907,\n                0.9876168370246887\n              ],\n              [\n                0.23543749749660492,\n                0.7294859886169434,\n                0.5546406507492065,\n                0.9886682033538818\n              ],\n              [\n                0.23543749749660492,\n                0.09572429955005646,\n                0.8058124780654907,\n                0.9886682033538818\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.003000000026077032,\n                0.287266343832016,\n                0.3238750100135803,\n                0.9867990612983704\n              ],\n              [\n                0.23543749749660492,\n                0.7294859886169434,\n                0.5546406507492065,\n                0.9886682033538818\n              ],\n              [\n                0.003000000026077032,\n                0.287266343832016,\n                0.5546406507492065,\n                0.9886682033538818\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000394905.jpg",\n        "coco_id": 394905,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skateboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 516233,\n          "object_id": 643907,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_394905_516233_643907_442812_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 442812,\n          "object_id": 643907,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.23079624772071838,\n                0.17406250536441803,\n                0.9076581001281738,\n                0.8310937285423279\n              ],\n              [\n                0.01613583229482174,\n                0.36153125762939453,\n                0.15644028782844543,\n                0.6810469031333923\n              ],\n              [\n                0.01613583229482174,\n                0.17406250536441803,\n                0.9076581001281738,\n                0.8310937285423279\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.003372365375980735,\n                0.07657812535762787,\n                0.2835597097873688,\n                0.6846874952316284\n              ],\n              [\n                0.01613583229482174,\n                0.36153125762939453,\n                0.15644028782844543,\n                0.6810469031333923\n              ],\n              [\n                0.003372365375980735,\n                0.07657812535762787,\n                0.2835597097873688,\n                0.6846874952316284\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000395198.jpg",\n        "coco_id": 395198,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 222452,\n          "object_id": 1764830,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_395198_222452_1764830_226730_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 226730,\n          "object_id": 1764830,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.5013281106948853,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ],\n              [\n                0.3936718702316284,\n                0.4530985951423645,\n                0.423953115940094,\n                0.7188262939453125\n              ],\n              [\n                0.3936718702316284,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.3482031226158142,\n                0.17896713316440582,\n                0.46031248569488525,\n                0.6610328555107117\n              ],\n              [\n                0.3936718702316284,\n                0.4530985951423645,\n                0.423953115940094,\n                0.7188262939453125\n              ],\n              [\n                0.3482031226158142,\n                0.17896713316440582,\n                0.46031248569488525,\n                0.7188262939453125\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000395198.jpg",\n        "coco_id": 395198,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 222452,\n          "object_id": 1764830,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_395198_222452_1764830_226730_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 226730,\n          "object_id": 1764830,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.3482031226158142,\n                0.17896713316440582,\n                0.46031248569488525,\n                0.6610328555107117\n              ],\n              [\n                0.3936718702316284,\n                0.4530985951423645,\n                0.423953115940094,\n                0.7188262939453125\n              ],\n              [\n                0.3482031226158142,\n                0.17896713316440582,\n                0.46031248569488525,\n                0.7188262939453125\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.5013281106948853,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ],\n              [\n                0.3936718702316284,\n                0.4530985951423645,\n                0.423953115940094,\n                0.7188262939453125\n              ],\n              [\n                0.3936718702316284,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000395198.jpg",\n        "coco_id": 395198,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 222452,\n          "object_id": 1764830,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_395198_222452_1764830_226730_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 226730,\n          "object_id": 1764830,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.5013281106948853,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ],\n              [\n                0.3936718702316284,\n                0.4530985951423645,\n                0.423953115940094,\n                0.7188262939453125\n              ],\n              [\n                0.3936718702316284,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.3482031226158142,\n                0.17896713316440582,\n                0.46031248569488525,\n                0.6610328555107117\n              ],\n              [\n                0.3936718702316284,\n                0.4530985951423645,\n                0.423953115940094,\n                0.7188262939453125\n              ],\n              [\n                0.3482031226158142,\n                0.17896713316440582,\n                0.46031248569488525,\n                0.7188262939453125\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000395198.jpg",\n        "coco_id": 395198,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 226730,\n          "object_id": 1416190,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_395198_226730_1416190_222452_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 222452,\n          "object_id": 1416190,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.3482031226158142,\n                0.17896713316440582,\n                0.46031248569488525,\n                0.6610328555107117\n              ],\n              [\n                0.5135937333106995,\n                0.36056336760520935,\n                0.596609354019165,\n                0.5369952917098999\n              ],\n              [\n                0.3482031226158142,\n                0.17896713316440582,\n                0.596609354019165,\n                0.6610328555107117\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.5013281106948853,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ],\n              [\n                0.5135937333106995,\n                0.36056336760520935,\n                0.596609354019165,\n                0.5369952917098999\n              ],\n              [\n                0.5013281106948853,\n                0.2753990590572357,\n                0.7356406450271606,\n                0.8766666650772095\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000395216.jpg",\n        "coco_id": 395216,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skateboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1317589,\n          "object_id": 644077,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_395216_1317589_644077_1258817_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1258817,\n          "object_id": 644077,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.0009062499739229679,\n                0.3478403687477112,\n                0.11307812482118607,\n                0.498920202255249\n              ],\n              [\n                0.318296879529953,\n                0.5153756141662598,\n                0.46315625309944153,\n                0.5525117516517639\n              ],\n              [\n                0.0009062499739229679,\n                0.3478403687477112,\n                0.46315625309944153,\n                0.5525117516517639\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.3118593692779541,\n                0.35042253136634827,\n                0.4458906352519989,\n                0.5310563445091248\n              ],\n              [\n                0.318296879529953,\n                0.5153756141662598,\n                0.46315625309944153,\n                0.5525117516517639\n              ],\n              [\n                0.3118593692779541,\n                0.35042253136634827,\n                0.46315625309944153,\n                0.5525117516517639\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000396106.jpg",\n        "coco_id": 396106,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "couch",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 445186,\n          "object_id": 111980,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_396106_445186_111980_431820_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 431820,\n          "object_id": 111980,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a couch",\n            "bounds": [\n              [\n                0.2455156296491623,\n                0.18187500536441803,\n                0.45740625262260437,\n                0.9711041450500488\n              ],\n              [\n                0.15645311772823334,\n                0.01289583370089531,\n                1.0,\n                0.8128958344459534\n              ],\n              [\n                0.15645311772823334,\n                0.01289583370089531,\n                1.0,\n                0.9711041450500488\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a couch",\n            "bounds": [\n              [\n                0.36742186546325684,\n                0.11235416680574417,\n                0.8393281102180481,\n                0.9146041870117188\n              ],\n              [\n                0.15645311772823334,\n                0.01289583370089531,\n                1.0,\n                0.8128958344459534\n              ],\n              [\n                0.15645311772823334,\n                0.01289583370089531,\n                1.0,\n                0.9146041870117188\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000039911.jpg",\n        "coco_id": 39911,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2176947,\n          "object_id": 636171,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_39911_2176947_636171_2176940_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2176940,\n          "object_id": 636171,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.11989062279462814,\n                0.06641685962677002,\n                0.4430468678474426,\n                0.9856674671173096\n              ],\n              [\n                0.775640606880188,\n                0.18203747272491455,\n                0.8686406016349792,\n                0.34981265664100647\n              ],\n              [\n                0.11989062279462814,\n                0.06641685962677002,\n                0.8686406016349792,\n                0.9856674671173096\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.6341719031333923,\n                0.011709601618349552,\n                0.8617812395095825,\n                0.8606791496276855\n              ],\n              [\n                0.775640606880188,\n                0.18203747272491455,\n                0.8686406016349792,\n                0.34981265664100647\n              ],\n              [\n                0.6341719031333923,\n                0.011709601618349552,\n                0.8686406016349792,\n                0.8606791496276855\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000400175.jpg",\n        "coco_id": 400175,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1750722,\n          "object_id": 1181082,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_400175_1750722_1181082_1720412_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1720412,\n          "object_id": 1181082,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.7671562433242798,\n                0.10113289952278137,\n                0.9895625114440918,\n                0.8674073815345764\n              ],\n              [\n                0.21757812798023224,\n                0.3101089298725128,\n                0.36745312809944153,\n                0.6157298684120178\n              ],\n              [\n                0.21757812798023224,\n                0.10113289952278137,\n                0.9895625114440918,\n                0.8674073815345764\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.007078125141561031,\n                0.004945533815771341,\n                0.28687500953674316,\n                0.9827232956886292\n              ],\n              [\n                0.21757812798023224,\n                0.3101089298725128,\n                0.36745312809944153,\n                0.6157298684120178\n              ],\n              [\n                0.007078125141561031,\n                0.004945533815771341,\n                0.36745312809944153,\n                0.9827232956886292\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000401291.jpg",\n        "coco_id": 401291,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1202604,\n          "object_id": 1938158,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_401291_1202604_1938158_1321428_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1321428,\n          "object_id": 1938158,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.05420312657952309,\n                0.6572365164756775,\n                0.13673438131809235,\n                0.9705620408058167\n              ],\n              [\n                0.2823750078678131,\n                0.8655503392219543,\n                0.3811874985694885,\n                1.0\n              ],\n              [\n                0.05420312657952309,\n                0.6572365164756775,\n                0.3811874985694885,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.2971875071525574,\n                0.649625301361084,\n                0.4270625114440918,\n                0.889274001121521\n              ],\n              [\n                0.2823750078678131,\n                0.8655503392219543,\n                0.3811874985694885,\n                1.0\n              ],\n              [\n                0.2823750078678131,\n                0.649625301361084,\n                0.4270625114440918,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000401291.jpg",\n        "coco_id": 401291,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1260609,\n          "object_id": 1592589,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_401291_1260609_1592589_1270631_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1270631,\n          "object_id": 1592589,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.5156093835830688,\n                0.6910070180892944,\n                0.6856406331062317,\n                1.0\n              ],\n              [\n                0.6329843997955322,\n                0.8899765610694885,\n                0.708609402179718,\n                0.9993442893028259\n              ],\n              [\n                0.5156093835830688,\n                0.6910070180892944,\n                0.708609402179718,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.16545312106609344,\n                0.5936299562454224,\n                0.2546718716621399,\n                0.956159234046936\n              ],\n              [\n                0.6329843997955322,\n                0.8899765610694885,\n                0.708609402179718,\n                0.9993442893028259\n              ],\n              [\n                0.16545312106609344,\n                0.5936299562454224,\n                0.708609402179718,\n                0.9993442893028259\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000401291.jpg",\n        "coco_id": 401291,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1268002,\n          "object_id": 1943350,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_401291_1268002_1943350_1270631_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1270631,\n          "object_id": 1943350,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.16545312106609344,\n                0.5936299562454224,\n                0.2546718716621399,\n                0.956159234046936\n              ],\n              [\n                0.7103124856948853,\n                0.8277751803398132,\n                0.7638280987739563,\n                1.0\n              ],\n              [\n                0.16545312106609344,\n                0.5936299562454224,\n                0.7638280987739563,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.6826093792915344,\n                0.713348925113678,\n                0.8087812662124634,\n                0.9097892045974731\n              ],\n              [\n                0.7103124856948853,\n                0.8277751803398132,\n                0.7638280987739563,\n                1.0\n              ],\n              [\n                0.6826093792915344,\n                0.713348925113678,\n                0.8087812662124634,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000402407.jpg",\n        "coco_id": 402407,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 423397,\n          "object_id": 627349,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_402407_423397_627349_516518_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 516518,\n          "object_id": 627349,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.6967999935150146,\n                0.2799733281135559,\n                0.8517000079154968,\n                0.6439733505249023\n              ],\n              [\n                0.3406200110912323,\n                0.23106667399406433,\n                0.40224000811576843,\n                0.33375999331474304\n              ],\n              [\n                0.3406200110912323,\n                0.23106667399406433,\n                0.8517000079154968,\n                0.6439733505249023\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.16685999929904938,\n                0.19325333833694458,\n                0.41124001145362854,\n                0.6764000058174133\n              ],\n              [\n                0.3406200110912323,\n                0.23106667399406433,\n                0.40224000811576843,\n                0.33375999331474304\n              ],\n              [\n                0.16685999929904938,\n                0.19325333833694458,\n                0.41124001145362854,\n                0.6764000058174133\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000404865.jpg",\n        "coco_id": 404865,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "cake",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1210490,\n          "object_id": 1085936,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_404865_1210490_1085936_2153535_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2153535,\n          "object_id": 1085936,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.3165000081062317,\n                0.04872916638851166,\n                0.7662031054496765,\n                0.721875011920929\n              ],\n              [\n                0.2460625022649765,\n                0.41347917914390564,\n                0.3724687397480011,\n                0.6966249942779541\n              ],\n              [\n                0.2460625022649765,\n                0.04872916638851166,\n                0.7662031054496765,\n                0.721875011920929\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.0,\n                0.1588541716337204,\n                0.44999998807907104,\n                0.9835833311080933\n              ],\n              [\n                0.2460625022649765,\n                0.41347917914390564,\n                0.3724687397480011,\n                0.6966249942779541\n              ],\n              [\n                0.0,\n                0.1588541716337204,\n                0.44999998807907104,\n                0.9835833311080933\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000404865.jpg",\n        "coco_id": 404865,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cake",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1210490,\n          "object_id": 1085936,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_404865_1210490_1085936_2153535_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2153535,\n          "object_id": 1085936,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.3165000081062317,\n                0.04872916638851166,\n                0.7662031054496765,\n                0.721875011920929\n              ],\n              [\n                0.2460625022649765,\n                0.41347917914390564,\n                0.3724687397480011,\n                0.6966249942779541\n              ],\n              [\n                0.2460625022649765,\n                0.04872916638851166,\n                0.7662031054496765,\n                0.721875011920929\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cake",\n            "bounds": [\n              [\n                0.0,\n                0.1588541716337204,\n                0.44999998807907104,\n                0.9835833311080933\n              ],\n              [\n                0.2460625022649765,\n                0.41347917914390564,\n                0.3724687397480011,\n                0.6966249942779541\n              ],\n              [\n                0.0,\n                0.1588541716337204,\n                0.44999998807907104,\n                0.9835833311080933\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000413012.jpg",\n        "coco_id": 413012,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 504168,\n          "object_id": 325706,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_413012_504168_325706_1707713_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1707713,\n          "object_id": 325706,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.7092441916465759,\n                0.030687499791383743,\n                1.0,\n                1.0\n              ],\n              [\n                0.3020930290222168,\n                0.19507811963558197,\n                0.3213081359863281,\n                0.2757500112056732\n              ],\n              [\n                0.3020930290222168,\n                0.030687499791383743,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.050058141350746155,\n                0.056046873331069946,\n                0.7508720755577087,\n                0.9843124747276306\n              ],\n              [\n                0.3020930290222168,\n                0.19507811963558197,\n                0.3213081359863281,\n                0.2757500112056732\n              ],\n              [\n                0.050058141350746155,\n                0.056046873331069946,\n                0.7508720755577087,\n                0.9843124747276306\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000413012.jpg",\n        "coco_id": 413012,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 504168,\n          "object_id": 325706,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_413012_504168_325706_1707713_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1707713,\n          "object_id": 325706,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.7092441916465759,\n                0.030687499791383743,\n                1.0,\n                1.0\n              ],\n              [\n                0.3020930290222168,\n                0.19507811963558197,\n                0.3213081359863281,\n                0.2757500112056732\n              ],\n              [\n                0.3020930290222168,\n                0.030687499791383743,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.050058141350746155,\n                0.056046873331069946,\n                0.7508720755577087,\n                0.9843124747276306\n              ],\n              [\n                0.3020930290222168,\n                0.19507811963558197,\n                0.3213081359863281,\n                0.2757500112056732\n              ],\n              [\n                0.050058141350746155,\n                0.056046873331069946,\n                0.7508720755577087,\n                0.9843124747276306\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000413073.jpg",\n        "coco_id": 413073,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 194161,\n          "object_id": 58732,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_413073_194161_58732_240609_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 240609,\n          "object_id": 58732,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.16746875643730164,\n                0.3324374854564667,\n                0.2982187569141388,\n                0.734250009059906\n              ],\n              [\n                0.1971093714237213,\n                0.5418333411216736,\n                0.3344062566757202,\n                0.9509791731834412\n              ],\n              [\n                0.16746875643730164,\n                0.3324374854564667,\n                0.3344062566757202,\n                0.9509791731834412\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.7459531426429749,\n                0.4907291531562805,\n                0.833984375,\n                0.7753750085830688\n              ],\n              [\n                0.1971093714237213,\n                0.5418333411216736,\n                0.3344062566757202,\n                0.9509791731834412\n              ],\n              [\n                0.1971093714237213,\n                0.4907291531562805,\n                0.833984375,\n                0.9509791731834412\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000413073.jpg",\n        "coco_id": 413073,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 194161,\n          "object_id": 58732,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_413073_194161_58732_240609_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 240609,\n          "object_id": 58732,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.16746875643730164,\n                0.3324374854564667,\n                0.2982187569141388,\n                0.734250009059906\n              ],\n              [\n                0.1971093714237213,\n                0.5418333411216736,\n                0.3344062566757202,\n                0.9509791731834412\n              ],\n              [\n                0.16746875643730164,\n                0.3324374854564667,\n                0.3344062566757202,\n                0.9509791731834412\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.7459531426429749,\n                0.4907291531562805,\n                0.833984375,\n                0.7753750085830688\n              ],\n              [\n                0.1971093714237213,\n                0.5418333411216736,\n                0.3344062566757202,\n                0.9509791731834412\n              ],\n              [\n                0.1971093714237213,\n                0.4907291531562805,\n                0.833984375,\n                0.9509791731834412\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000413073.jpg",\n        "coco_id": 413073,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 240417,\n          "object_id": 57179,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_413073_240417_57179_240609_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 240609,\n          "object_id": 57179,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.7459531426429749,\n                0.4907291531562805,\n                0.833984375,\n                0.7753750085830688\n              ],\n              [\n                0.3736250102519989,\n                0.6081249713897705,\n                0.5112344026565552,\n                0.9553958177566528\n              ],\n              [\n                0.3736250102519989,\n                0.4907291531562805,\n                0.833984375,\n                0.9553958177566528\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.35359373688697815,\n                0.4518125057220459,\n                0.47257813811302185,\n                0.7261041402816772\n              ],\n              [\n                0.3736250102519989,\n                0.6081249713897705,\n                0.5112344026565552,\n                0.9553958177566528\n              ],\n              [\n                0.35359373688697815,\n                0.4518125057220459,\n                0.5112344026565552,\n                0.9553958177566528\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000413073.jpg",\n        "coco_id": 413073,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 240417,\n          "object_id": 57179,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_413073_240417_57179_240609_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 240609,\n          "object_id": 57179,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.7459531426429749,\n                0.4907291531562805,\n                0.833984375,\n                0.7753750085830688\n              ],\n              [\n                0.3736250102519989,\n                0.6081249713897705,\n                0.5112344026565552,\n                0.9553958177566528\n              ],\n              [\n                0.3736250102519989,\n                0.4907291531562805,\n                0.833984375,\n                0.9553958177566528\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.35359373688697815,\n                0.4518125057220459,\n                0.47257813811302185,\n                0.7261041402816772\n              ],\n              [\n                0.3736250102519989,\n                0.6081249713897705,\n                0.5112344026565552,\n                0.9553958177566528\n              ],\n              [\n                0.35359373688697815,\n                0.4518125057220459,\n                0.5112344026565552,\n                0.9553958177566528\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000424692.jpg",\n        "coco_id": 424692,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "remote",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 185915,\n          "object_id": 1112196,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_424692_185915_1112196_217848_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 217848,\n          "object_id": 1112196,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.5494999885559082,\n                0.307855486869812,\n                1.0,\n                1.0\n              ],\n              [\n                0.4009000062942505,\n                0.5545221567153931,\n                0.42691999673843384,\n                0.5868065357208252\n              ],\n              [\n                0.4009000062942505,\n                0.307855486869812,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.17005999386310577,\n                0.12386946380138397,\n                0.4632999897003174,\n                1.0\n              ],\n              [\n                0.4009000062942505,\n                0.5545221567153931,\n                0.42691999673843384,\n                0.5868065357208252\n              ],\n              [\n                0.17005999386310577,\n                0.12386946380138397,\n                0.4632999897003174,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000424692.jpg",\n        "coco_id": 424692,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "couch",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 217848,\n          "object_id": 116183,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_424692_217848_116183_185915_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 185915,\n          "object_id": 116183,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a couch",\n            "bounds": [\n              [\n                0.17005999386310577,\n                0.12386946380138397,\n                0.4632999897003174,\n                1.0\n              ],\n              [\n                0.6794000267982483,\n                0.46780887246131897,\n                1.0,\n                0.9952680468559265\n              ],\n              [\n                0.17005999386310577,\n                0.12386946380138397,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a couch",\n            "bounds": [\n              [\n                0.5494999885559082,\n                0.307855486869812,\n                1.0,\n                1.0\n              ],\n              [\n                0.6794000267982483,\n                0.46780887246131897,\n                1.0,\n                0.9952680468559265\n              ],\n              [\n                0.5494999885559082,\n                0.307855486869812,\n                1.0,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000426500.jpg",\n        "coco_id": 426500,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 515141,\n          "object_id": 1167919,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_426500_515141_1167919_2157878_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2157878,\n          "object_id": 1167919,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.29710936546325684,\n                0.49140143394470215,\n                0.39115625619888306,\n                0.8634917140007019\n              ],\n              [\n                0.33079686760902405,\n                0.6052019000053406,\n                0.37107813358306885,\n                0.6686936020851135\n              ],\n              [\n                0.29710936546325684,\n                0.49140143394470215,\n                0.39115625619888306,\n                0.8634917140007019\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.3804062604904175,\n                0.5370308756828308,\n                0.44487500190734863,\n                0.8505463004112244\n              ],\n              [\n                0.33079686760902405,\n                0.6052019000053406,\n                0.37107813358306885,\n                0.6686936020851135\n              ],\n              [\n                0.33079686760902405,\n                0.5370308756828308,\n                0.44487500190734863,\n                0.8505463004112244\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000427040.jpg",\n        "coco_id": 427040,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bed",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 470951,\n          "object_id": 315354,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_427040_470951_315354_500954_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 500954,\n          "object_id": 315354,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bed",\n            "bounds": [\n              [\n                0.2705624997615814,\n                0.16854910552501678,\n                0.49235936999320984,\n                0.5325892567634583\n              ],\n              [\n                0.0031406250782310963,\n                0.029218750074505806,\n                1.0,\n                0.9865178465843201\n              ],\n              [\n                0.0031406250782310963,\n                0.029218750074505806,\n                1.0,\n                0.9865178465843201\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bed",\n            "bounds": [\n              [\n                0.6676406264305115,\n                0.2964508831501007,\n                0.787234365940094,\n                0.6582589149475098\n              ],\n              [\n                0.0031406250782310963,\n                0.029218750074505806,\n                1.0,\n                0.9865178465843201\n              ],\n              [\n                0.0031406250782310963,\n                0.029218750074505806,\n                1.0,\n                0.9865178465843201\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000428019.jpg",\n        "coco_id": 428019,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 465956,\n          "object_id": 1476314,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_428019_465956_1476314_422912_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 422912,\n          "object_id": 1476314,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.4525468647480011,\n                0.4145433306694031,\n                0.578781247138977,\n                0.7096487283706665\n              ],\n              [\n                0.5596250295639038,\n                0.41025760769844055,\n                0.6158593893051147,\n                0.4485245943069458\n              ],\n              [\n                0.4525468647480011,\n                0.41025760769844055,\n                0.6158593893051147,\n                0.7096487283706665\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.2578125,\n                0.5086650848388672,\n                0.36778125166893005,\n                0.7066042423248291\n              ],\n              [\n                0.5596250295639038,\n                0.41025760769844055,\n                0.6158593893051147,\n                0.4485245943069458\n              ],\n              [\n                0.2578125,\n                0.41025760769844055,\n                0.6158593893051147,\n                0.7066042423248291\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000428164.jpg",\n        "coco_id": 428164,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 422134,\n          "object_id": 605396,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_428164_422134_605396_449984_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 449984,\n          "object_id": 605396,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.7224799990653992,\n                0.1125200018286705,\n                0.986519992351532,\n                0.9462000131607056\n              ],\n              [\n                0.35618001222610474,\n                0.13707999885082245,\n                0.4202199876308441,\n                0.34832000732421875\n              ],\n              [\n                0.35618001222610474,\n                0.1125200018286705,\n                0.986519992351532,\n                0.9462000131607056\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.3967199921607971,\n                0.17163999378681183,\n                0.5868600010871887,\n                0.9627199769020081\n              ],\n              [\n                0.35618001222610474,\n                0.13707999885082245,\n                0.4202199876308441,\n                0.34832000732421875\n              ],\n              [\n                0.35618001222610474,\n                0.13707999885082245,\n                0.5868600010871887,\n                0.9627199769020081\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000428164.jpg",\n        "coco_id": 428164,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 472834,\n          "object_id": 605259,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_428164_472834_605259_449984_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 449984,\n          "object_id": 605259,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.11020000278949738,\n                0.17475999891757965,\n                0.27619999647140503,\n                0.9599599838256836\n              ],\n              [\n                0.06747999787330627,\n                0.28780001401901245,\n                0.1406800001859665,\n                0.4115599989891052\n              ],\n              [\n                0.06747999787330627,\n                0.17475999891757965,\n                0.27619999647140503,\n                0.9599599838256836\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.7224799990653992,\n                0.1125200018286705,\n                0.986519992351532,\n                0.9462000131607056\n              ],\n              [\n                0.06747999787330627,\n                0.28780001401901245,\n                0.1406800001859665,\n                0.4115599989891052\n              ],\n              [\n                0.06747999787330627,\n                0.1125200018286705,\n                0.986519992351532,\n                0.9462000131607056\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000428683.jpg",\n        "coco_id": 428683,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "umbrella",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1732257,\n          "object_id": 282748,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_428683_1732257_282748_1740936_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1740936,\n          "object_id": 282748,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.2574218809604645,\n                0.637374997138977,\n                0.4752500057220459,\n                0.9888593554496765\n              ],\n              [\n                0.26906248927116394,\n                0.19282811880111694,\n                0.9125624895095825,\n                0.6659218668937683\n              ],\n              [\n                0.2574218809604645,\n                0.19282811880111694,\n                0.9125624895095825,\n                0.9888593554496765\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.42021873593330383,\n                0.6202187538146973,\n                0.6606718897819519,\n                0.9887655973434448\n              ],\n              [\n                0.26906248927116394,\n                0.19282811880111694,\n                0.9125624895095825,\n                0.6659218668937683\n              ],\n              [\n                0.26906248927116394,\n                0.19282811880111694,\n                0.9125624895095825,\n                0.9887655973434448\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000428683.jpg",\n        "coco_id": 428683,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "umbrella",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1732257,\n          "object_id": 282748,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_428683_1732257_282748_1740936_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1740936,\n          "object_id": 282748,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.42021873593330383,\n                0.6202187538146973,\n                0.6606718897819519,\n                0.9887655973434448\n              ],\n              [\n                0.26906248927116394,\n                0.19282811880111694,\n                0.9125624895095825,\n                0.6659218668937683\n              ],\n              [\n                0.26906248927116394,\n                0.19282811880111694,\n                0.9125624895095825,\n                0.9887655973434448\n              ]\n            ]\n          },\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.2574218809604645,\n                0.637374997138977,\n                0.4752500057220459,\n                0.9888593554496765\n              ],\n              [\n                0.26906248927116394,\n                0.19282811880111694,\n                0.9125624895095825,\n                0.6659218668937683\n              ],\n              [\n                0.2574218809604645,\n                0.19282811880111694,\n                0.9125624895095825,\n                0.9888593554496765\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000428683.jpg",\n        "coco_id": 428683,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 485631,\n          "object_id": 1639460,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_428683_485631_1639460_1740936_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1740936,\n          "object_id": 1639460,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.0852343738079071,\n                0.34007811546325684,\n                0.28379687666893005,\n                0.7947656512260437\n              ],\n              [\n                0.10885937511920929,\n                0.37990623712539673,\n                0.12998437881469727,\n                0.40864062309265137\n              ],\n              [\n                0.0852343738079071,\n                0.34007811546325684,\n                0.28379687666893005,\n                0.7947656512260437\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.2574218809604645,\n                0.637374997138977,\n                0.4752500057220459,\n                0.9888593554496765\n              ],\n              [\n                0.10885937511920929,\n                0.37990623712539673,\n                0.12998437881469727,\n                0.40864062309265137\n              ],\n              [\n                0.10885937511920929,\n                0.37990623712539673,\n                0.4752500057220459,\n                0.9888593554496765\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000428683.jpg",\n        "coco_id": 428683,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 485631,\n          "object_id": 1639460,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_428683_485631_1639460_1740936_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1740936,\n          "object_id": 1639460,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.0852343738079071,\n                0.34007811546325684,\n                0.28379687666893005,\n                0.7947656512260437\n              ],\n              [\n                0.10885937511920929,\n                0.37990623712539673,\n                0.12998437881469727,\n                0.40864062309265137\n              ],\n              [\n                0.0852343738079071,\n                0.34007811546325684,\n                0.28379687666893005,\n                0.7947656512260437\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.2574218809604645,\n                0.637374997138977,\n                0.4752500057220459,\n                0.9888593554496765\n              ],\n              [\n                0.10885937511920929,\n                0.37990623712539673,\n                0.12998437881469727,\n                0.40864062309265137\n              ],\n              [\n                0.10885937511920929,\n                0.37990623712539673,\n                0.4752500057220459,\n                0.9888593554496765\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000430621.jpg",\n        "coco_id": 430621,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 518357,\n          "object_id": 630542,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_430621_518357_630542_547886_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 547886,\n          "object_id": 630542,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.6541000008583069,\n                0.2709091007709503,\n                0.8342199921607971,\n                0.8480908870697021\n              ],\n              [\n                0.6867200136184692,\n                0.16703030467033386,\n                0.7434200048446655,\n                0.36606061458587646\n              ],\n              [\n                0.6541000008583069,\n                0.16703030467033386,\n                0.8342199921607971,\n                0.8480908870697021\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.15309999883174896,\n                0.17793940007686615,\n                0.2943199872970581,\n                0.9099090695381165\n              ],\n              [\n                0.6867200136184692,\n                0.16703030467033386,\n                0.7434200048446655,\n                0.36606061458587646\n              ],\n              [\n                0.15309999883174896,\n                0.16703030467033386,\n                0.7434200048446655,\n                0.9099090695381165\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000433232.jpg",\n        "coco_id": 433232,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "truck",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1265466,\n          "object_id": 2051110,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_433232_1265466_2051110_220424_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 220424,\n          "object_id": 2051110,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a truck",\n            "bounds": [\n              [\n                0.37627172470092773,\n                0.14359374344348907,\n                0.44110584259033203,\n                0.2677343785762787\n              ],\n              [\n                0.05135861039161682,\n                0.21587499976158142,\n                0.9147709608078003,\n                0.9365156292915344\n              ],\n              [\n                0.05135861039161682,\n                0.14359374344348907,\n                0.9147709608078003,\n                0.9365156292915344\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a truck",\n            "bounds": [\n              [\n                0.9091311097145081,\n                0.6329531073570251,\n                0.9913427829742432,\n                0.8362343907356262\n              ],\n              [\n                0.05135861039161682,\n                0.21587499976158142,\n                0.9147709608078003,\n                0.9365156292915344\n              ],\n              [\n                0.05135861039161682,\n                0.21587499976158142,\n                0.9913427829742432,\n                0.9365156292915344\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000433232.jpg",\n        "coco_id": 433232,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bus",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1753772,\n          "object_id": 169414,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_433232_1753772_169414_220424_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 220424,\n          "object_id": 169414,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bus",\n            "bounds": [\n              [\n                0.27211689949035645,\n                0.08148437738418579,\n                0.3744707703590393,\n                0.2691406309604645\n              ],\n              [\n                0.0,\n                0.22184374928474426,\n                0.906303346157074,\n                0.8705000281333923\n              ],\n              [\n                0.0,\n                0.08148437738418579,\n                0.906303346157074,\n                0.8705000281333923\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bus",\n            "bounds": [\n              [\n                0.9091311097145081,\n                0.6329531073570251,\n                0.9913427829742432,\n                0.8362343907356262\n              ],\n              [\n                0.0,\n                0.22184374928474426,\n                0.906303346157074,\n                0.8705000281333923\n              ],\n              [\n                0.0,\n                0.22184374928474426,\n                0.9913427829742432,\n                0.8705000281333923\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000433232.jpg",\n        "coco_id": 433232,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "truck",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 220304,\n          "object_id": 2051110,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_433232_220304_2051110_220424_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 220424,\n          "object_id": 2051110,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a truck",\n            "bounds": [\n              [\n                0.9091311097145081,\n                0.6329531073570251,\n                0.9913427829742432,\n                0.8362343907356262\n              ],\n              [\n                0.05135861039161682,\n                0.21587499976158142,\n                0.9147709608078003,\n                0.9365156292915344\n              ],\n              [\n                0.05135861039161682,\n                0.21587499976158142,\n                0.9913427829742432,\n                0.9365156292915344\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a truck",\n            "bounds": [\n              [\n                0.4618799388408661,\n                0.11256249994039536,\n                0.6589889526367188,\n                0.2643125057220459\n              ],\n              [\n                0.05135861039161682,\n                0.21587499976158142,\n                0.9147709608078003,\n                0.9365156292915344\n              ],\n              [\n                0.05135861039161682,\n                0.11256249994039536,\n                0.9147709608078003,\n                0.9365156292915344\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000434370.jpg",\n        "coco_id": 434370,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "donut",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 210051,\n          "object_id": 1925711,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_434370_210051_1925711_203317_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 203317,\n          "object_id": 1925711,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a donut",\n            "bounds": [\n              [\n                0.0,\n                0.013112032786011696,\n                0.5314218997955322,\n                0.9883817434310913\n              ],\n              [\n                0.8117656111717224,\n                0.5438796877861023,\n                0.8658124804496765,\n                0.6277385950088501\n              ],\n              [\n                0.0,\n                0.013112032786011696,\n                0.8658124804496765,\n                0.9883817434310913\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a donut",\n            "bounds": [\n              [\n                0.510015606880188,\n                0.0015352696646004915,\n                1.0,\n                1.0\n              ],\n              [\n                0.8117656111717224,\n                0.5438796877861023,\n                0.8658124804496765,\n                0.6277385950088501\n              ],\n              [\n                0.510015606880188,\n                0.0015352696646004915,\n                1.0,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000435096.jpg",\n        "coco_id": 435096,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 204164,\n          "object_id": 58733,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_435096_204164_58733_194534_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 194534,\n          "object_id": 58733,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.08604687452316284,\n                0.09645739942789078,\n                0.4423281252384186,\n                0.5989686250686646\n              ],\n              [\n                0.47532811760902405,\n                0.3002242147922516,\n                0.9656093716621399,\n                0.8949999809265137\n              ],\n              [\n                0.08604687452316284,\n                0.09645739942789078,\n                0.9656093716621399,\n                0.8949999809265137\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.5245781540870667,\n                0.11941704154014587,\n                0.8037656545639038,\n                0.6327354311943054\n              ],\n              [\n                0.47532811760902405,\n                0.3002242147922516,\n                0.9656093716621399,\n                0.8949999809265137\n              ],\n              [\n                0.47532811760902405,\n                0.11941704154014587,\n                0.9656093716621399,\n                0.8949999809265137\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000435716.jpg",\n        "coco_id": 435716,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "spoon",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2204515,\n          "object_id": 705193,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_435716_2204515_705193_476877_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 476877,\n          "object_id": 705193,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a spoon",\n            "bounds": [\n              [\n                0.18717187643051147,\n                0.1782008409500122,\n                0.4136562645435333,\n                0.6056485176086426\n              ],\n              [\n                0.1674218773841858,\n                0.41416317224502563,\n                0.1952187567949295,\n                0.44131800532341003\n              ],\n              [\n                0.1674218773841858,\n                0.1782008409500122,\n                0.4136562645435333,\n                0.6056485176086426\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a spoon",\n            "bounds": [\n              [\n                0.0,\n                0.15508368611335754,\n                0.2784687578678131,\n                0.6194769740104675\n              ],\n              [\n                0.1674218773841858,\n                0.41416317224502563,\n                0.1952187567949295,\n                0.44131800532341003\n              ],\n              [\n                0.0,\n                0.15508368611335754,\n                0.2784687578678131,\n                0.6194769740104675\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000439084.jpg",\n        "coco_id": 439084,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1293207,\n          "object_id": 628183,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_439084_1293207_628183_1265027_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1265027,\n          "object_id": 628183,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.3357812464237213,\n                0.3369617164134979,\n                0.48704686760902405,\n                0.6574880480766296\n              ],\n              [\n                0.2664843797683716,\n                0.42287081480026245,\n                0.35695311427116394,\n                0.47205740213394165\n              ],\n              [\n                0.2664843797683716,\n                0.3369617164134979,\n                0.48704686760902405,\n                0.6574880480766296\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.6957343816757202,\n                0.3981339633464813,\n                0.7826094031333923,\n                0.6788756251335144\n              ],\n              [\n                0.2664843797683716,\n                0.42287081480026245,\n                0.35695311427116394,\n                0.47205740213394165\n              ],\n              [\n                0.2664843797683716,\n                0.3981339633464813,\n                0.7826094031333923,\n                0.6788756251335144\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000439938.jpg",\n        "coco_id": 439938,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "surfboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1273239,\n          "object_id": 648526,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_439938_1273239_648526_1284068_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1284068,\n          "object_id": 648526,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a surfboard",\n            "bounds": [\n              [\n                0.5040000081062317,\n                0.40959569811820984,\n                0.6216406226158142,\n                0.810458242893219\n              ],\n              [\n                0.12760937213897705,\n                0.48757413029670715,\n                0.16371874511241913,\n                0.6646361351013184\n              ],\n              [\n                0.12760937213897705,\n                0.40959569811820984,\n                0.6216406226158142,\n                0.810458242893219\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a surfboard",\n            "bounds": [\n              [\n                0.11635937541723251,\n                0.34978437423706055,\n                0.247984379529953,\n                0.8780593276023865\n              ],\n              [\n                0.12760937213897705,\n                0.48757413029670715,\n                0.16371874511241913,\n                0.6646361351013184\n              ],\n              [\n                0.11635937541723251,\n                0.34978437423706055,\n                0.247984379529953,\n                0.8780593276023865\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000439938.jpg",\n        "coco_id": 439938,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1273239,\n          "object_id": 648526,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_439938_1273239_648526_1298777_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1298777,\n          "object_id": 648526,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.11635937541723251,\n                0.34978437423706055,\n                0.247984379529953,\n                0.8780593276023865\n              ],\n              [\n                0.12760937213897705,\n                0.48757413029670715,\n                0.16371874511241913,\n                0.6646361351013184\n              ],\n              [\n                0.11635937541723251,\n                0.34978437423706055,\n                0.247984379529953,\n                0.8780593276023865\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.6713906526565552,\n                0.2730458080768585,\n                0.7593593597412109,\n                0.7567654848098755\n              ],\n              [\n                0.12760937213897705,\n                0.48757413029670715,\n                0.16371874511241913,\n                0.6646361351013184\n              ],\n              [\n                0.12760937213897705,\n                0.2730458080768585,\n                0.7593593597412109,\n                0.7567654848098755\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000439938.jpg",\n        "coco_id": 439938,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "surfboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1312895,\n          "object_id": 1479995,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_439938_1312895_1479995_1282432_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1282432,\n          "object_id": 1479995,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a surfboard",\n            "bounds": [\n              [\n                0.4014218747615814,\n                0.42741239070892334,\n                0.5090937614440918,\n                0.9823989272117615\n              ],\n              [\n                0.4597812592983246,\n                0.8613746762275696,\n                0.5470625162124634,\n                0.999622642993927\n              ],\n              [\n                0.4014218747615814,\n                0.42741239070892334,\n                0.5470625162124634,\n                0.999622642993927\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a surfboard",\n            "bounds": [\n              [\n                0.5132499933242798,\n                0.671913743019104,\n                0.6865000128746033,\n                0.9887601137161255\n              ],\n              [\n                0.4597812592983246,\n                0.8613746762275696,\n                0.5470625162124634,\n                0.999622642993927\n              ],\n              [\n                0.4597812592983246,\n                0.671913743019104,\n                0.6865000128746033,\n                0.999622642993927\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000440066.jpg",\n        "coco_id": 440066,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 432638,\n          "object_id": 651940,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_440066_432638_651940_424318_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 424318,\n          "object_id": 651940,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.42381250858306885,\n                0.6982969045639038,\n                0.9432500004768372,\n                0.8188906311988831\n              ],\n              [\n                0.3685416579246521,\n                0.7393749952316284,\n                0.8793958425521851,\n                0.8391406536102295\n              ],\n              [\n                0.3685416579246521,\n                0.6982969045639038,\n                0.9432500004768372,\n                0.8391406536102295\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.2758958339691162,\n                0.433656245470047,\n                0.5850208401679993,\n                0.6905624866485596\n              ],\n              [\n                0.3685416579246521,\n                0.7393749952316284,\n                0.8793958425521851,\n                0.8391406536102295\n              ],\n              [\n                0.2758958339691162,\n                0.433656245470047,\n                0.8793958425521851,\n                0.8391406536102295\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000444362.jpg",\n        "coco_id": 444362,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "remote",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1724551,\n          "object_id": 1111937,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_444362_1724551_1111937_435206_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 435206,\n          "object_id": 1111937,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.1483124941587448,\n                0.1394791603088379,\n                0.5190937519073486,\n                0.9866666793823242\n              ],\n              [\n                0.5181718468666077,\n                0.5946249961853027,\n                0.5725312232971191,\n                0.723604142665863\n              ],\n              [\n                0.1483124941587448,\n                0.1394791603088379,\n                0.5725312232971191,\n                0.9866666793823242\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.5190937519073486,\n                0.19325000047683716,\n                0.9994375109672546,\n                0.9865208268165588\n              ],\n              [\n                0.5181718468666077,\n                0.5946249961853027,\n                0.5725312232971191,\n                0.723604142665863\n              ],\n              [\n                0.5181718468666077,\n                0.19325000047683716,\n                0.9994375109672546,\n                0.9865208268165588\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000446956.jpg",\n        "coco_id": 446956,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1252373,\n          "object_id": 1179873,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_446956_1252373_1179873_2161789_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2161789,\n          "object_id": 1179873,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.6379374861717224,\n                0.7239791750907898,\n                0.7214687466621399,\n                0.9511874914169312\n              ],\n              [\n                0.6375156044960022,\n                0.7778541445732117,\n                0.6646562218666077,\n                0.8242708444595337\n              ],\n              [\n                0.6375156044960022,\n                0.7239791750907898,\n                0.7214687466621399,\n                0.9511874914169312\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.9188593626022339,\n                0.7082083225250244,\n                0.9629531502723694,\n                0.898145854473114\n              ],\n              [\n                0.6375156044960022,\n                0.7778541445732117,\n                0.6646562218666077,\n                0.8242708444595337\n              ],\n              [\n                0.6375156044960022,\n                0.7082083225250244,\n                0.9629531502723694,\n                0.898145854473114\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000446956.jpg",\n        "coco_id": 446956,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1308772,\n          "object_id": 1390779,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_446956_1308772_1390779_2161789_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2161789,\n          "object_id": 1390779,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.1412343680858612,\n                0.8015000224113464,\n                0.21528124809265137,\n                0.9331666827201843\n              ],\n              [\n                0.12395312637090683,\n                0.8351041674613953,\n                0.2570468783378601,\n                0.9525416493415833\n              ],\n              [\n                0.12395312637090683,\n                0.8015000224113464,\n                0.2570468783378601,\n                0.9525416493415833\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.9188593626022339,\n                0.7082083225250244,\n                0.9629531502723694,\n                0.898145854473114\n              ],\n              [\n                0.12395312637090683,\n                0.8351041674613953,\n                0.2570468783378601,\n                0.9525416493415833\n              ],\n              [\n                0.12395312637090683,\n                0.7082083225250244,\n                0.9629531502723694,\n                0.9525416493415833\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000045084.jpg",\n        "coco_id": 45084,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1746894,\n          "object_id": 1185247,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_45084_1746894_1185247_2159991_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2159991,\n          "object_id": 1185247,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.09493750333786011,\n                0.03696335107088089,\n                0.32326561212539673,\n                0.6952181458473206\n              ],\n              [\n                0.6230937242507935,\n                0.39188480377197266,\n                0.8166718482971191,\n                0.7342408299446106\n              ],\n              [\n                0.09493750333786011,\n                0.03696335107088089,\n                0.8166718482971191,\n                0.7342408299446106\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.6498749852180481,\n                0.12631762027740479,\n                0.8236406445503235,\n                0.4416404962539673\n              ],\n              [\n                0.6230937242507935,\n                0.39188480377197266,\n                0.8166718482971191,\n                0.7342408299446106\n              ],\n              [\n                0.6230937242507935,\n                0.12631762027740479,\n                0.8236406445503235,\n                0.7342408299446106\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000045084.jpg",\n        "coco_id": 45084,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1746894,\n          "object_id": 1185247,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_45084_1746894_1185247_449676_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 449676,\n          "object_id": 1185247,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.6498749852180481,\n                0.12631762027740479,\n                0.8236406445503235,\n                0.4416404962539673\n              ],\n              [\n                0.6230937242507935,\n                0.39188480377197266,\n                0.8166718482971191,\n                0.7342408299446106\n              ],\n              [\n                0.6230937242507935,\n                0.12631762027740479,\n                0.8236406445503235,\n                0.7342408299446106\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.35439062118530273,\n                0.4402792453765869,\n                0.5947187542915344,\n                0.7500872611999512\n              ],\n              [\n                0.6230937242507935,\n                0.39188480377197266,\n                0.8166718482971191,\n                0.7342408299446106\n              ],\n              [\n                0.35439062118530273,\n                0.39188480377197266,\n                0.8166718482971191,\n                0.7500872611999512\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000045084.jpg",\n        "coco_id": 45084,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2159991,\n          "object_id": 1842424,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_45084_2159991_1842424_449676_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 449676,\n          "object_id": 1842424,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.09493750333786011,\n                0.03696335107088089,\n                0.32326561212539673,\n                0.6952181458473206\n              ],\n              [\n                0.0008750000270083547,\n                0.4463699758052826,\n                0.08742187172174454,\n                0.6504537463188171\n              ],\n              [\n                0.0008750000270083547,\n                0.03696335107088089,\n                0.32326561212539673,\n                0.6952181458473206\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.35439062118530273,\n                0.4402792453765869,\n                0.5947187542915344,\n                0.7500872611999512\n              ],\n              [\n                0.0008750000270083547,\n                0.4463699758052826,\n                0.08742187172174454,\n                0.6504537463188171\n              ],\n              [\n                0.0008750000270083547,\n                0.4402792453765869,\n                0.5947187542915344,\n                0.7500872611999512\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000450878.jpg",\n        "coco_id": 450878,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 444397,\n          "object_id": 631622,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_450878_444397_631622_554043_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 554043,\n          "object_id": 631622,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.6791562438011169,\n                0.1659250557422638,\n                0.9035468697547913,\n                0.6547072529792786\n              ],\n              [\n                0.38423436880111694,\n                0.4418267011642456,\n                0.47379687428474426,\n                0.5508899092674255\n              ],\n              [\n                0.38423436880111694,\n                0.1659250557422638,\n                0.9035468697547913,\n                0.6547072529792786\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.18549999594688416,\n                0.2155035138130188,\n                0.4682343602180481,\n                0.9105854630470276\n              ],\n              [\n                0.38423436880111694,\n                0.4418267011642456,\n                0.47379687428474426,\n                0.5508899092674255\n              ],\n              [\n                0.18549999594688416,\n                0.2155035138130188,\n                0.47379687428474426,\n                0.9105854630470276\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000045128.jpg",\n        "coco_id": 45128,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 441523,\n          "object_id": 603403,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_45128_441523_603403_480157_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 480157,\n          "object_id": 603403,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.9426249861717224,\n                0.5228958129882812,\n                1.0,\n                0.8647291660308838\n              ],\n              [\n                0.32270312309265137,\n                0.5249166488647461,\n                0.34696874022483826,\n                0.5669999718666077\n              ],\n              [\n                0.32270312309265137,\n                0.5228958129882812,\n                1.0,\n                0.8647291660308838\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.3359062373638153,\n                0.4932291805744171,\n                0.4069531261920929,\n                0.7208958268165588\n              ],\n              [\n                0.32270312309265137,\n                0.5249166488647461,\n                0.34696874022483826,\n                0.5669999718666077\n              ],\n              [\n                0.32270312309265137,\n                0.4932291805744171,\n                0.4069531261920929,\n                0.7208958268165588\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000453853.jpg",\n        "coco_id": 453853,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1745677,\n          "object_id": 1390489,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_453853_1745677_1390489_1735026_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1735026,\n          "object_id": 1390489,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.0,\n                0.39011791348457336,\n                0.24373437464237213,\n                0.9851886630058289\n              ],\n              [\n                0.23671874403953552,\n                0.7348349094390869,\n                0.2888124883174896,\n                0.833702802658081\n              ],\n              [\n                0.0,\n                0.39011791348457336,\n                0.2888124883174896,\n                0.9851886630058289\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.3410624861717224,\n                0.1468632072210312,\n                0.612375020980835,\n                0.7321698069572449\n              ],\n              [\n                0.23671874403953552,\n                0.7348349094390869,\n                0.2888124883174896,\n                0.833702802658081\n              ],\n              [\n                0.23671874403953552,\n                0.1468632072210312,\n                0.612375020980835,\n                0.833702802658081\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000453853.jpg",\n        "coco_id": 453853,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2154424,\n          "object_id": 2061220,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_453853_2154424_2061220_1735026_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1735026,\n          "object_id": 2061220,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.5317187309265137,\n                0.4155896306037903,\n                0.7691875100135803,\n                0.9402594566345215\n              ],\n              [\n                0.24573437869548798,\n                0.6834198236465454,\n                0.30149999260902405,\n                0.7389622926712036\n              ],\n              [\n                0.24573437869548798,\n                0.4155896306037903,\n                0.7691875100135803,\n                0.9402594566345215\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.3410624861717224,\n                0.1468632072210312,\n                0.612375020980835,\n                0.7321698069572449\n              ],\n              [\n                0.24573437869548798,\n                0.6834198236465454,\n                0.30149999260902405,\n                0.7389622926712036\n              ],\n              [\n                0.24573437869548798,\n                0.1468632072210312,\n                0.612375020980835,\n                0.7389622926712036\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000045437.jpg",\n        "coco_id": 45437,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 210043,\n          "object_id": 322779,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_45437_210043_322779_210331_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 210331,\n          "object_id": 322779,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.4336549639701843,\n                0.22696000337600708,\n                0.765497088432312,\n                0.9460600018501282\n              ],\n              [\n                0.5379239916801453,\n                0.2808400094509125,\n                0.5776315927505493,\n                0.32558000087738037\n              ],\n              [\n                0.4336549639701843,\n                0.22696000337600708,\n                0.765497088432312,\n                0.9460600018501282\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.003304093610495329,\n                0.17081999778747559,\n                0.4939473569393158,\n                0.9826200008392334\n              ],\n              [\n                0.5379239916801453,\n                0.2808400094509125,\n                0.5776315927505493,\n                0.32558000087738037\n              ],\n              [\n                0.003304093610495329,\n                0.17081999778747559,\n                0.5776315927505493,\n                0.9826200008392334\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000045437.jpg",\n        "coco_id": 45437,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 210043,\n          "object_id": 322779,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_45437_210043_322779_210331_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 210331,\n          "object_id": 322779,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.4336549639701843,\n                0.22696000337600708,\n                0.765497088432312,\n                0.9460600018501282\n              ],\n              [\n                0.5379239916801453,\n                0.2808400094509125,\n                0.5776315927505493,\n                0.32558000087738037\n              ],\n              [\n                0.4336549639701843,\n                0.22696000337600708,\n                0.765497088432312,\n                0.9460600018501282\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.003304093610495329,\n                0.17081999778747559,\n                0.4939473569393158,\n                0.9826200008392334\n              ],\n              [\n                0.5379239916801453,\n                0.2808400094509125,\n                0.5776315927505493,\n                0.32558000087738037\n              ],\n              [\n                0.003304093610495329,\n                0.17081999778747559,\n                0.5776315927505493,\n                0.9826200008392334\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000455073.jpg",\n        "coco_id": 455073,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 482731,\n          "object_id": 324313,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_455073_482731_324313_487429_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 487429,\n          "object_id": 324313,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.3835781216621399,\n                0.15181207656860352,\n                0.5763906240463257,\n                0.9752516746520996\n              ],\n              [\n                0.7037500143051147,\n                0.13845637440681458,\n                0.7461405992507935,\n                0.17018456757068634\n              ],\n              [\n                0.3835781216621399,\n                0.13845637440681458,\n                0.7461405992507935,\n                0.9752516746520996\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.5393593907356262,\n                0.0814429521560669,\n                0.8174843788146973,\n                0.9841610789299011\n              ],\n              [\n                0.7037500143051147,\n                0.13845637440681458,\n                0.7461405992507935,\n                0.17018456757068634\n              ],\n              [\n                0.5393593907356262,\n                0.0814429521560669,\n                0.8174843788146973,\n                0.9841610789299011\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000455123.jpg",\n        "coco_id": 455123,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2154597,\n          "object_id": 1578990,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_455123_2154597_1578990_252095_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 252095,\n          "object_id": 1578990,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a chair",\n            "bounds": [\n              [\n                0.6297500133514404,\n                0.38344261050224304,\n                0.7214218974113464,\n                0.6088524460792542\n              ],\n              [\n                0.5394687652587891,\n                0.4319906234741211,\n                0.6238593459129333,\n                0.5571897029876709\n              ],\n              [\n                0.5394687652587891,\n                0.38344261050224304,\n                0.7214218974113464,\n                0.6088524460792542\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a chair",\n            "bounds": [\n              [\n                0.5218124985694885,\n                0.3299531638622284,\n                0.6540625095367432,\n                0.550187349319458\n              ],\n              [\n                0.5394687652587891,\n                0.4319906234741211,\n                0.6238593459129333,\n                0.5571897029876709\n              ],\n              [\n                0.5218124985694885,\n                0.3299531638622284,\n                0.6540625095367432,\n                0.5571897029876709\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000456545.jpg",\n        "coco_id": 456545,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1277729,\n          "object_id": 125746,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_456545_1277729_125746_1277990_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1277990,\n          "object_id": 125746,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.5364687442779541,\n                0.364340215921402,\n                0.735281229019165,\n                0.7817965149879456\n              ],\n              [\n                0.5361562371253967,\n                0.5379173159599304,\n                0.6960468888282776,\n                0.8559141755104065\n              ],\n              [\n                0.5361562371253967,\n                0.364340215921402,\n                0.735281229019165,\n                0.8559141755104065\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.6948750019073486,\n                0.3269316256046295,\n                0.8498437404632568,\n                0.6646581888198853\n              ],\n              [\n                0.5361562371253967,\n                0.5379173159599304,\n                0.6960468888282776,\n                0.8559141755104065\n              ],\n              [\n                0.5361562371253967,\n                0.3269316256046295,\n                0.8498437404632568,\n                0.8559141755104065\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000456545.jpg",\n        "coco_id": 456545,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1277729,\n          "object_id": 125746,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_456545_1277729_125746_1312842_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1312842,\n          "object_id": 125746,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.4205937385559082,\n                0.5036247968673706,\n                0.6771718859672546,\n                0.8187758326530457\n              ],\n              [\n                0.5361562371253967,\n                0.5379173159599304,\n                0.6960468888282776,\n                0.8559141755104065\n              ],\n              [\n                0.4205937385559082,\n                0.5036247968673706,\n                0.6960468888282776,\n                0.8559141755104065\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.5364687442779541,\n                0.364340215921402,\n                0.735281229019165,\n                0.7817965149879456\n              ],\n              [\n                0.5361562371253967,\n                0.5379173159599304,\n                0.6960468888282776,\n                0.8559141755104065\n              ],\n              [\n                0.5361562371253967,\n                0.364340215921402,\n                0.735281229019165,\n                0.8559141755104065\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000456545.jpg",\n        "coco_id": 456545,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1277729,\n          "object_id": 125746,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_456545_1277729_125746_1312842_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1312842,\n          "object_id": 125746,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.5364687442779541,\n                0.364340215921402,\n                0.735281229019165,\n                0.7817965149879456\n              ],\n              [\n                0.5361562371253967,\n                0.5379173159599304,\n                0.6960468888282776,\n                0.8559141755104065\n              ],\n              [\n                0.5361562371253967,\n                0.364340215921402,\n                0.735281229019165,\n                0.8559141755104065\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.4205937385559082,\n                0.5036247968673706,\n                0.6771718859672546,\n                0.8187758326530457\n              ],\n              [\n                0.5361562371253967,\n                0.5379173159599304,\n                0.6960468888282776,\n                0.8559141755104065\n              ],\n              [\n                0.4205937385559082,\n                0.5036247968673706,\n                0.6960468888282776,\n                0.8559141755104065\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000456545.jpg",\n        "coco_id": 456545,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1312842,\n          "object_id": 125746,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_456545_1312842_125746_1277990_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1277990,\n          "object_id": 125746,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.6948750019073486,\n                0.3269316256046295,\n                0.8498437404632568,\n                0.6646581888198853\n              ],\n              [\n                0.5361562371253967,\n                0.5379173159599304,\n                0.6960468888282776,\n                0.8559141755104065\n              ],\n              [\n                0.5361562371253967,\n                0.3269316256046295,\n                0.8498437404632568,\n                0.8559141755104065\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.4205937385559082,\n                0.5036247968673706,\n                0.6771718859672546,\n                0.8187758326530457\n              ],\n              [\n                0.5361562371253967,\n                0.5379173159599304,\n                0.6960468888282776,\n                0.8559141755104065\n              ],\n              [\n                0.4205937385559082,\n                0.5036247968673706,\n                0.6960468888282776,\n                0.8559141755104065\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000457190.jpg",\n        "coco_id": 457190,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 476427,\n          "object_id": 603792,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_457190_476427_603792_481964_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 481964,\n          "object_id": 603792,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.8449062705039978,\n                0.006206089165061712,\n                1.0,\n                0.9836300015449524\n              ],\n              [\n                0.5382656455039978,\n                0.40318500995635986,\n                0.604812502861023,\n                0.47918033599853516\n              ],\n              [\n                0.5382656455039978,\n                0.006206089165061712,\n                1.0,\n                0.9836300015449524\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.37632811069488525,\n                0.12360655516386032,\n                0.5622344017028809,\n                0.7146135568618774\n              ],\n              [\n                0.5382656455039978,\n                0.40318500995635986,\n                0.604812502861023,\n                0.47918033599853516\n              ],\n              [\n                0.37632811069488525,\n                0.12360655516386032,\n                0.604812502861023,\n                0.7146135568618774\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000462899.jpg",\n        "coco_id": 462899,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 461023,\n          "object_id": 629330,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_462899_461023_629330_471631_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 471631,\n          "object_id": 629330,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.01782812550663948,\n                0.4320608973503113,\n                0.38634374737739563,\n                0.9175878167152405\n              ],\n              [\n                0.5468437671661377,\n                0.42777517437934875,\n                0.7314062714576721,\n                0.49578455090522766\n              ],\n              [\n                0.01782812550663948,\n                0.42777517437934875,\n                0.7314062714576721,\n                0.9175878167152405\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.3246249854564667,\n                0.25810304284095764,\n                0.6462500095367432,\n                0.9240280985832214\n              ],\n              [\n                0.5468437671661377,\n                0.42777517437934875,\n                0.7314062714576721,\n                0.49578455090522766\n              ],\n              [\n                0.3246249854564667,\n                0.25810304284095764,\n                0.7314062714576721,\n                0.9240280985832214\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000463429.jpg",\n        "coco_id": 463429,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 483755,\n          "object_id": 632473,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_463429_483755_632473_511114_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 511114,\n          "object_id": 632473,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.2038281261920929,\n                0.6514102816581726,\n                0.46037501096725464,\n                0.9986538290977478\n              ],\n              [\n                0.3524531126022339,\n                0.5378044843673706,\n                0.5077031254768372,\n                0.6816506385803223\n              ],\n              [\n                0.2038281261920929,\n                0.5378044843673706,\n                0.5077031254768372,\n                0.9986538290977478\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.36671873927116394,\n                0.5698237419128418,\n                0.5402031540870667,\n                0.9414423108100891\n              ],\n              [\n                0.3524531126022339,\n                0.5378044843673706,\n                0.5077031254768372,\n                0.6816506385803223\n              ],\n              [\n                0.3524531126022339,\n                0.5378044843673706,\n                0.5402031540870667,\n                0.9414423108100891\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000463505.jpg",\n        "coco_id": 463505,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 427017,\n          "object_id": 326560,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_463505_427017_326560_458610_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 458610,\n          "object_id": 326560,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.44999998807907104,\n                0.4426933228969574,\n                0.8915799856185913,\n                0.9887733459472656\n              ],\n              [\n                0.5958200097084045,\n                0.6478933095932007,\n                0.693560004234314,\n                0.8393599987030029\n              ],\n              [\n                0.44999998807907104,\n                0.4426933228969574,\n                0.8915799856185913,\n                0.9887733459472656\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.06418000161647797,\n                0.13469333946704865,\n                0.5675600171089172,\n                0.9837866425514221\n              ],\n              [\n                0.5958200097084045,\n                0.6478933095932007,\n                0.693560004234314,\n                0.8393599987030029\n              ],\n              [\n                0.06418000161647797,\n                0.13469333946704865,\n                0.693560004234314,\n                0.9837866425514221\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000464340.jpg",\n        "coco_id": 464340,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "knife",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 200328,\n          "object_id": 1521414,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_464340_200328_1521414_188871_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 188871,\n          "object_id": 1521414,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.5625,\n                0.009694444015622139,\n                0.7394999861717224,\n                0.6346944570541382\n              ],\n              [\n                0.4573906362056732,\n                0.589388906955719,\n                0.5365937352180481,\n                0.673027753829956\n              ],\n              [\n                0.4573906362056732,\n                0.009694444015622139,\n                0.7394999861717224,\n                0.673027753829956\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.25981250405311584,\n                0.004749999847263098,\n                0.5713281035423279,\n                0.9330000281333923\n              ],\n              [\n                0.4573906362056732,\n                0.589388906955719,\n                0.5365937352180481,\n                0.673027753829956\n              ],\n              [\n                0.25981250405311584,\n                0.004749999847263098,\n                0.5713281035423279,\n                0.9330000281333923\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000464854.jpg",\n        "coco_id": 464854,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1308541,\n          "object_id": 376742,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_464854_1308541_376742_2153362_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2153362,\n          "object_id": 376742,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.21879687905311584,\n                0.1704791635274887,\n                0.3390781283378601,\n                0.4188750088214874\n              ],\n              [\n                0.7413437366485596,\n                0.38324999809265137,\n                0.7736250162124634,\n                0.4866875112056732\n              ],\n              [\n                0.21879687905311584,\n                0.1704791635274887,\n                0.7736250162124634,\n                0.4866875112056732\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.5882031321525574,\n                0.29887500405311584,\n                0.6842656135559082,\n                0.5954999923706055\n              ],\n              [\n                0.7413437366485596,\n                0.38324999809265137,\n                0.7736250162124634,\n                0.4866875112056732\n              ],\n              [\n                0.5882031321525574,\n                0.29887500405311584,\n                0.7736250162124634,\n                0.5954999923706055\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000464854.jpg",\n        "coco_id": 464854,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1316633,\n          "object_id": 375992,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_464854_1316633_375992_2153362_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2153362,\n          "object_id": 375992,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.6192968487739563,\n                0.3609791696071625,\n                0.7434062361717224,\n                0.6154583096504211\n              ],\n              [\n                0.5692343711853027,\n                0.46954166889190674,\n                0.7859531044960022,\n                0.8419374823570251\n              ],\n              [\n                0.5692343711853027,\n                0.3609791696071625,\n                0.7859531044960022,\n                0.8419374823570251\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.21879687905311584,\n                0.1704791635274887,\n                0.3390781283378601,\n                0.4188750088214874\n              ],\n              [\n                0.5692343711853027,\n                0.46954166889190674,\n                0.7859531044960022,\n                0.8419374823570251\n              ],\n              [\n                0.21879687905311584,\n                0.1704791635274887,\n                0.7859531044960022,\n                0.8419374823570251\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000464854.jpg",\n        "coco_id": 464854,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1322165,\n          "object_id": 375992,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_464854_1322165_375992_2153362_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2153362,\n          "object_id": 375992,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.21879687905311584,\n                0.1704791635274887,\n                0.3390781283378601,\n                0.4188750088214874\n              ],\n              [\n                0.5692343711853027,\n                0.46954166889190674,\n                0.7859531044960022,\n                0.8419374823570251\n              ],\n              [\n                0.21879687905311584,\n                0.1704791635274887,\n                0.7859531044960022,\n                0.8419374823570251\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.5403749942779541,\n                0.5315208435058594,\n                0.7162968516349792,\n                0.7979999780654907\n              ],\n              [\n                0.5692343711853027,\n                0.46954166889190674,\n                0.7859531044960022,\n                0.8419374823570251\n              ],\n              [\n                0.5403749942779541,\n                0.46954166889190674,\n                0.7859531044960022,\n                0.8419374823570251\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000464854.jpg",\n        "coco_id": 464854,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1325863,\n          "object_id": 375414,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_464854_1325863_375414_2153362_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2153362,\n          "object_id": 375414,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.21879687905311584,\n                0.1704791635274887,\n                0.3390781283378601,\n                0.4188750088214874\n              ],\n              [\n                0.1001875028014183,\n                0.4554166793823242,\n                0.20289061963558197,\n                0.898520827293396\n              ],\n              [\n                0.1001875028014183,\n                0.1704791635274887,\n                0.3390781283378601,\n                0.898520827293396\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.18715624511241913,\n                0.43700000643730164,\n                0.31299999356269836,\n                0.7180416584014893\n              ],\n              [\n                0.1001875028014183,\n                0.4554166793823242,\n                0.20289061963558197,\n                0.898520827293396\n              ],\n              [\n                0.1001875028014183,\n                0.43700000643730164,\n                0.31299999356269836,\n                0.898520827293396\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000464854.jpg",\n        "coco_id": 464854,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2159171,\n          "object_id": 383556,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_464854_2159171_383556_2153362_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2153362,\n          "object_id": 383556,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.3937656283378601,\n                0.31593748927116394,\n                0.4930937588214874,\n                0.4989791810512543\n              ],\n              [\n                0.35948437452316284,\n                0.3935000002384186,\n                0.4054531157016754,\n                0.5061249732971191\n              ],\n              [\n                0.35948437452316284,\n                0.31593748927116394,\n                0.4930937588214874,\n                0.5061249732971191\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.21879687905311584,\n                0.1704791635274887,\n                0.3390781283378601,\n                0.4188750088214874\n              ],\n              [\n                0.35948437452316284,\n                0.3935000002384186,\n                0.4054531157016754,\n                0.5061249732971191\n              ],\n              [\n                0.21879687905311584,\n                0.1704791635274887,\n                0.4054531157016754,\n                0.5061249732971191\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000465235.jpg",\n        "coco_id": 465235,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1236873,\n          "object_id": 148752,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_465235_1236873_148752_1214844_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1214844,\n          "object_id": 148752,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.803390622138977,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.6206250190734863\n              ],\n              [\n                0.541015625,\n                0.49437499046325684,\n                0.8325781226158142,\n                0.7662916779518127\n              ],\n              [\n                0.541015625,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.7662916779518127\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.6137968897819519,\n                0.34977084398269653,\n                0.7668125033378601,\n                0.6928333044052124\n              ],\n              [\n                0.541015625,\n                0.49437499046325684,\n                0.8325781226158142,\n                0.7662916779518127\n              ],\n              [\n                0.541015625,\n                0.34977084398269653,\n                0.8325781226158142,\n                0.7662916779518127\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000465235.jpg",\n        "coco_id": 465235,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1236873,\n          "object_id": 148752,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_465235_1236873_148752_1214844_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1214844,\n          "object_id": 148752,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.6137968897819519,\n                0.34977084398269653,\n                0.7668125033378601,\n                0.6928333044052124\n              ],\n              [\n                0.541015625,\n                0.49437499046325684,\n                0.8325781226158142,\n                0.7662916779518127\n              ],\n              [\n                0.541015625,\n                0.34977084398269653,\n                0.8325781226158142,\n                0.7662916779518127\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.803390622138977,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.6206250190734863\n              ],\n              [\n                0.541015625,\n                0.49437499046325684,\n                0.8325781226158142,\n                0.7662916779518127\n              ],\n              [\n                0.541015625,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.7662916779518127\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000465235.jpg",\n        "coco_id": 465235,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1236873,\n          "object_id": 148752,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_465235_1236873_148752_1214844_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1214844,\n          "object_id": 148752,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.803390622138977,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.6206250190734863\n              ],\n              [\n                0.541015625,\n                0.49437499046325684,\n                0.8325781226158142,\n                0.7662916779518127\n              ],\n              [\n                0.541015625,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.7662916779518127\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.6137968897819519,\n                0.34977084398269653,\n                0.7668125033378601,\n                0.6928333044052124\n              ],\n              [\n                0.541015625,\n                0.49437499046325684,\n                0.8325781226158142,\n                0.7662916779518127\n              ],\n              [\n                0.541015625,\n                0.34977084398269653,\n                0.8325781226158142,\n                0.7662916779518127\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000465235.jpg",\n        "coco_id": 465235,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1754790,\n          "object_id": 148136,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_465235_1754790_148136_1214844_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1214844,\n          "object_id": 148136,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.803390622138977,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.6206250190734863\n              ],\n              [\n                0.020218750461935997,\n                0.5007500052452087,\n                0.3876406252384186,\n                0.7681666612625122\n              ],\n              [\n                0.020218750461935997,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.7681666612625122\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.20667187869548798,\n                0.3513333201408386,\n                0.3672812581062317,\n                0.6507708430290222\n              ],\n              [\n                0.020218750461935997,\n                0.5007500052452087,\n                0.3876406252384186,\n                0.7681666612625122\n              ],\n              [\n                0.020218750461935997,\n                0.3513333201408386,\n                0.3876406252384186,\n                0.7681666612625122\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000465235.jpg",\n        "coco_id": 465235,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1754790,\n          "object_id": 148136,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_465235_1754790_148136_1214844_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1214844,\n          "object_id": 148136,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.20667187869548798,\n                0.3513333201408386,\n                0.3672812581062317,\n                0.6507708430290222\n              ],\n              [\n                0.020218750461935997,\n                0.5007500052452087,\n                0.3876406252384186,\n                0.7681666612625122\n              ],\n              [\n                0.020218750461935997,\n                0.3513333201408386,\n                0.3876406252384186,\n                0.7681666612625122\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.803390622138977,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.6206250190734863\n              ],\n              [\n                0.020218750461935997,\n                0.5007500052452087,\n                0.3876406252384186,\n                0.7681666612625122\n              ],\n              [\n                0.020218750461935997,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.7681666612625122\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000465235.jpg",\n        "coco_id": 465235,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1754790,\n          "object_id": 148136,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_465235_1754790_148136_1214844_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1214844,\n          "object_id": 148136,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.803390622138977,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.6206250190734863\n              ],\n              [\n                0.020218750461935997,\n                0.5007500052452087,\n                0.3876406252384186,\n                0.7681666612625122\n              ],\n              [\n                0.020218750461935997,\n                0.3373333215713501,\n                0.8842187523841858,\n                0.7681666612625122\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.20667187869548798,\n                0.3513333201408386,\n                0.3672812581062317,\n                0.6507708430290222\n              ],\n              [\n                0.020218750461935997,\n                0.5007500052452087,\n                0.3876406252384186,\n                0.7681666612625122\n              ],\n              [\n                0.020218750461935997,\n                0.3513333201408386,\n                0.3876406252384186,\n                0.7681666612625122\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000465265.jpg",\n        "coco_id": 465265,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1733583,\n          "object_id": 322753,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_465265_1733583_322753_1247909_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1247909,\n          "object_id": 322753,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.5592187643051147,\n                0.0881497785449028,\n                0.643750011920929,\n                0.5516740083694458\n              ],\n              [\n                0.4253906309604645,\n                0.36224669218063354,\n                0.44935938715934753,\n                0.4007709324359894\n              ],\n              [\n                0.4253906309604645,\n                0.0881497785449028,\n                0.643750011920929,\n                0.5516740083694458\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.2837499976158142,\n                0.30337005853652954,\n                0.5069218873977661,\n                0.9550660848617554\n              ],\n              [\n                0.4253906309604645,\n                0.36224669218063354,\n                0.44935938715934753,\n                0.4007709324359894\n              ],\n              [\n                0.2837499976158142,\n                0.30337005853652954,\n                0.5069218873977661,\n                0.9550660848617554\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000465265.jpg",\n        "coco_id": 465265,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1733583,\n          "object_id": 322753,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_465265_1733583_322753_1247909_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1247909,\n          "object_id": 322753,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.5592187643051147,\n                0.0881497785449028,\n                0.643750011920929,\n                0.5516740083694458\n              ],\n              [\n                0.4253906309604645,\n                0.36224669218063354,\n                0.44935938715934753,\n                0.4007709324359894\n              ],\n              [\n                0.4253906309604645,\n                0.0881497785449028,\n                0.643750011920929,\n                0.5516740083694458\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.2837499976158142,\n                0.30337005853652954,\n                0.5069218873977661,\n                0.9550660848617554\n              ],\n              [\n                0.4253906309604645,\n                0.36224669218063354,\n                0.44935938715934753,\n                0.4007709324359894\n              ],\n              [\n                0.2837499976158142,\n                0.30337005853652954,\n                0.5069218873977661,\n                0.9550660848617554\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000467889.jpg",\n        "coco_id": 467889,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "hot dog",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1319601,\n          "object_id": 1067652,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_467889_1319601_1067652_487861_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 487861,\n          "object_id": 1067652,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a hot dog",\n            "bounds": [\n              [\n                0.3588593602180481,\n                0.16480842232704163,\n                0.9747031331062317,\n                0.9895402193069458\n              ],\n              [\n                0.39149999618530273,\n                0.5811685919761658,\n                0.5795781016349792,\n                0.8494061231613159\n              ],\n              [\n                0.3588593602180481,\n                0.16480842232704163,\n                0.9747031331062317,\n                0.9895402193069458\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a hot dog",\n            "bounds": [\n              [\n                0.0,\n                0.0,\n                0.575515627861023,\n                1.0\n              ],\n              [\n                0.39149999618530273,\n                0.5811685919761658,\n                0.5795781016349792,\n                0.8494061231613159\n              ],\n              [\n                0.0,\n                0.0,\n                0.5795781016349792,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000467979.jpg",\n        "coco_id": 467979,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 446758,\n          "object_id": 628796,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_467979_446758_628796_1720760_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1720760,\n          "object_id": 628796,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.8073281049728394,\n                0.17303529381752014,\n                0.9595312476158142,\n                0.6988705992698669\n              ],\n              [\n                0.2924843728542328,\n                0.24757647514343262,\n                0.38279688358306885,\n                0.33258822560310364\n              ],\n              [\n                0.2924843728542328,\n                0.17303529381752014,\n                0.9595312476158142,\n                0.6988705992698669\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.27607813477516174,\n                0.1865176409482956,\n                0.5685625076293945,\n                0.880894124507904\n              ],\n              [\n                0.2924843728542328,\n                0.24757647514343262,\n                0.38279688358306885,\n                0.33258822560310364\n              ],\n              [\n                0.27607813477516174,\n                0.1865176409482956,\n                0.5685625076293945,\n                0.880894124507904\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000469941.jpg",\n        "coco_id": 469941,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 490354,\n          "object_id": 603228,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_469941_490354_603228_518865_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 518865,\n          "object_id": 603228,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.2629687488079071,\n                0.35473066568374634,\n                0.6145937442779541,\n                0.9831147789955139\n              ],\n              [\n                0.22682812809944153,\n                0.44826698303222656,\n                0.29926562309265137,\n                0.5274707078933716\n              ],\n              [\n                0.22682812809944153,\n                0.35473066568374634,\n                0.6145937442779541,\n                0.9831147789955139\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.47699999809265137,\n                0.3122248351573944,\n                0.852859377861023,\n                0.9751054048538208\n              ],\n              [\n                0.22682812809944153,\n                0.44826698303222656,\n                0.29926562309265137,\n                0.5274707078933716\n              ],\n              [\n                0.22682812809944153,\n                0.3122248351573944,\n                0.852859377861023,\n                0.9751054048538208\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000470097.jpg",\n        "coco_id": 470097,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "sports ball",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 425604,\n          "object_id": 301017,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_470097_425604_301017_487008_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 487008,\n          "object_id": 301017,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.6166093945503235,\n                0.6608984470367432,\n                0.9775156378746033,\n                1.0\n              ],\n              [\n                0.6323906183242798,\n                0.2409375011920929,\n                0.6484843492507935,\n                0.2653124928474426\n              ],\n              [\n                0.6166093945503235,\n                0.2409375011920929,\n                0.9775156378746033,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.15101562440395355,\n                0.16179686784744263,\n                0.6813437342643738,\n                0.7168554663658142\n              ],\n              [\n                0.6323906183242798,\n                0.2409375011920929,\n                0.6484843492507935,\n                0.2653124928474426\n              ],\n              [\n                0.15101562440395355,\n                0.16179686784744263,\n                0.6813437342643738,\n                0.7168554663658142\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000471900.jpg",\n        "coco_id": 471900,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skis",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 514746,\n          "object_id": 2198839,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_471900_514746_2198839_451104_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 451104,\n          "object_id": 2198839,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding skis",\n            "bounds": [\n              [\n                0.0951249971985817,\n                0.12616707384586334,\n                0.3880937397480011,\n                1.0\n              ],\n              [\n                0.756515622138977,\n                0.0,\n                0.8440625071525574,\n                0.987002432346344\n              ],\n              [\n                0.0951249971985817,\n                0.0,\n                0.8440625071525574,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding skis",\n            "bounds": [\n              [\n                0.3350468873977661,\n                0.03447174280881882,\n                0.9287656545639038,\n                0.9987223744392395\n              ],\n              [\n                0.756515622138977,\n                0.0,\n                0.8440625071525574,\n                0.987002432346344\n              ],\n              [\n                0.3350468873977661,\n                0.0,\n                0.9287656545639038,\n                0.9987223744392395\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000475664.jpg",\n        "coco_id": 475664,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "wine glass",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 186776,\n          "object_id": 663064,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_475664_186776_663064_189658_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 189658,\n          "object_id": 663064,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a wine glass",\n            "bounds": [\n              [\n                0.5697343945503235,\n                0.07309192419052124,\n                0.782296895980835,\n                0.9856545925140381\n              ],\n              [\n                0.5915937423706055,\n                0.4171866178512573,\n                0.6287343502044678,\n                0.5835654735565186\n              ],\n              [\n                0.5697343945503235,\n                0.07309192419052124,\n                0.782296895980835,\n                0.9856545925140381\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a wine glass",\n            "bounds": [\n              [\n                0.4456250071525574,\n                0.2314763218164444,\n                0.6269999742507935,\n                0.9894986152648926\n              ],\n              [\n                0.5915937423706055,\n                0.4171866178512573,\n                0.6287343502044678,\n                0.5835654735565186\n              ],\n              [\n                0.4456250071525574,\n                0.2314763218164444,\n                0.6287343502044678,\n                0.9894986152648926\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000475664.jpg",\n        "coco_id": 475664,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bottle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 206930,\n          "object_id": 1492583,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_475664_206930_1492583_1213983_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1213983,\n          "object_id": 1492583,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bottle",\n            "bounds": [\n              [\n                0.8748124837875366,\n                0.3325905203819275,\n                1.0,\n                1.0\n              ],\n              [\n                0.1478281319141388,\n                0.8029526472091675,\n                0.1808750033378601,\n                0.9901949763298035\n              ],\n              [\n                0.1478281319141388,\n                0.3325905203819275,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bottle",\n            "bounds": [\n              [\n                0.0012656250037252903,\n                0.224345400929451,\n                0.2672343850135803,\n                0.9501950144767761\n              ],\n              [\n                0.1478281319141388,\n                0.8029526472091675,\n                0.1808750033378601,\n                0.9901949763298035\n              ],\n              [\n                0.0012656250037252903,\n                0.224345400929451,\n                0.2672343850135803,\n                0.9901949763298035\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000477471.jpg",\n        "coco_id": 477471,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 198258,\n          "object_id": 53729,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_477471_198258_53729_216407_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 216407,\n          "object_id": 53729,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.21235936880111694,\n                0.20897915959358215,\n                0.4297812581062317,\n                0.7046250104904175\n              ],\n              [\n                0.44999998807907104,\n                0.37597915530204773,\n                0.8467812538146973,\n                0.8577083349227905\n              ],\n              [\n                0.21235936880111694,\n                0.20897915959358215,\n                0.8467812538146973,\n                0.8577083349227905\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.3630625009536743,\n                0.19227083027362823,\n                0.5172656178474426,\n                0.5007500052452087\n              ],\n              [\n                0.44999998807907104,\n                0.37597915530204773,\n                0.8467812538146973,\n                0.8577083349227905\n              ],\n              [\n                0.3630625009536743,\n                0.19227083027362823,\n                0.8467812538146973,\n                0.8577083349227905\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000480939.jpg",\n        "coco_id": 480939,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 472952,\n          "object_id": 323159,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_480939_472952_323159_495075_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 495075,\n          "object_id": 323159,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.5474531054496765,\n                0.09213615208864212,\n                0.8092187643051147,\n                0.7752816677093506\n              ],\n              [\n                0.7065469026565552,\n                0.16617371141910553,\n                0.7535781264305115,\n                0.2700938880443573\n              ],\n              [\n                0.5474531054496765,\n                0.09213615208864212,\n                0.8092187643051147,\n                0.7752816677093506\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.022953124716877937,\n                0.3650469481945038,\n                0.5398749709129333,\n                0.5662206411361694\n              ],\n              [\n                0.7065469026565552,\n                0.16617371141910553,\n                0.7535781264305115,\n                0.2700938880443573\n              ],\n              [\n                0.022953124716877937,\n                0.16617371141910553,\n                0.7535781264305115,\n                0.5662206411361694\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000480939.jpg",\n        "coco_id": 480939,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 472952,\n          "object_id": 323159,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_480939_472952_323159_495075_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 495075,\n          "object_id": 323159,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.5474531054496765,\n                0.09213615208864212,\n                0.8092187643051147,\n                0.7752816677093506\n              ],\n              [\n                0.7065469026565552,\n                0.16617371141910553,\n                0.7535781264305115,\n                0.2700938880443573\n              ],\n              [\n                0.5474531054496765,\n                0.09213615208864212,\n                0.8092187643051147,\n                0.7752816677093506\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.022953124716877937,\n                0.3650469481945038,\n                0.5398749709129333,\n                0.5662206411361694\n              ],\n              [\n                0.7065469026565552,\n                0.16617371141910553,\n                0.7535781264305115,\n                0.2700938880443573\n              ],\n              [\n                0.022953124716877937,\n                0.16617371141910553,\n                0.7535781264305115,\n                0.5662206411361694\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000480939.jpg",\n        "coco_id": 480939,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 472952,\n          "object_id": 571690,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_480939_472952_571690_495075_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 495075,\n          "object_id": 571690,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.5474531054496765,\n                0.09213615208864212,\n                0.8092187643051147,\n                0.7752816677093506\n              ],\n              [\n                0.01653124950826168,\n                0.28657278418540955,\n                0.9976875185966492,\n                0.8328403830528259\n              ],\n              [\n                0.01653124950826168,\n                0.09213615208864212,\n                0.9976875185966492,\n                0.8328403830528259\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.022953124716877937,\n                0.3650469481945038,\n                0.5398749709129333,\n                0.5662206411361694\n              ],\n              [\n                0.01653124950826168,\n                0.28657278418540955,\n                0.9976875185966492,\n                0.8328403830528259\n              ],\n              [\n                0.01653124950826168,\n                0.28657278418540955,\n                0.9976875185966492,\n                0.8328403830528259\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000481222.jpg",\n        "coco_id": 481222,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1262498,\n          "object_id": 628379,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_481222_1262498_628379_1249948_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1249948,\n          "object_id": 628379,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.519656240940094,\n                0.24270270764827728,\n                0.8431562781333923,\n                0.8561891913414001\n              ],\n              [\n                0.8366562724113464,\n                0.10972972959280014,\n                0.9392969012260437,\n                0.29624325037002563\n              ],\n              [\n                0.519656240940094,\n                0.10972972959280014,\n                0.9392969012260437,\n                0.8561891913414001\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.09396874904632568,\n                0.35237836837768555,\n                0.3834531307220459,\n                0.8680270314216614\n              ],\n              [\n                0.8366562724113464,\n                0.10972972959280014,\n                0.9392969012260437,\n                0.29624325037002563\n              ],\n              [\n                0.09396874904632568,\n                0.10972972959280014,\n                0.9392969012260437,\n                0.8680270314216614\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000481667.jpg",\n        "coco_id": 481667,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "person",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1740126,\n          "object_id": 2153347,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_481667_1740126_2153347_2153347_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2153347,\n          "object_id": 2153347,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.10525000095367432,\n                0.09120312333106995,\n                0.46024999022483826,\n                0.7684218883514404\n              ],\n              [\n                0.10525000095367432,\n                0.09120312333106995,\n                0.46024999022483826,\n                0.7684218883514404\n              ],\n              [\n                0.10525000095367432,\n                0.09120312333106995,\n                0.46024999022483826,\n                0.7684218883514404\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.24931250512599945,\n                0.0891406238079071,\n                0.5571718811988831,\n                0.7264999747276306\n              ],\n              [\n                0.10525000095367432,\n                0.09120312333106995,\n                0.46024999022483826,\n                0.7684218883514404\n              ],\n              [\n                0.10525000095367432,\n                0.0891406238079071,\n                0.5571718811988831,\n                0.7684218883514404\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000483833.jpg",\n        "coco_id": 483833,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 555651,\n          "object_id": 627744,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_483833_555651_627744_436548_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 436548,\n          "object_id": 627744,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.531220018863678,\n                0.12483483552932739,\n                0.8027200102806091,\n                0.7166366577148438\n              ],\n              [\n                0.40384000539779663,\n                0.13732732832431793,\n                0.5437800288200378,\n                0.16009008884429932\n              ],\n              [\n                0.40384000539779663,\n                0.12483483552932739,\n                0.8027200102806091,\n                0.7166366577148438\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.16978000104427338,\n                0.15816816687583923,\n                0.38238000869750977,\n                0.620660662651062\n              ],\n              [\n                0.40384000539779663,\n                0.13732732832431793,\n                0.5437800288200378,\n                0.16009008884429932\n              ],\n              [\n                0.16978000104427338,\n                0.13732732832431793,\n                0.5437800288200378,\n                0.620660662651062\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000484362.jpg",\n        "coco_id": 484362,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1232771,\n          "object_id": 602602,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_484362_1232771_602602_437481_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 437481,\n          "object_id": 602602,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.12999999523162842,\n                0.3808281123638153,\n                0.6003051400184631,\n                0.9467656016349792\n              ],\n              [\n                0.18830986320972443,\n                0.5958906412124634,\n                0.36460092663764954,\n                0.6987344026565552\n              ],\n              [\n                0.12999999523162842,\n                0.3808281123638153,\n                0.6003051400184631,\n                0.9467656016349792\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.7506572604179382,\n                0.2553125023841858,\n                0.9494835734367371,\n                0.8694687485694885\n              ],\n              [\n                0.18830986320972443,\n                0.5958906412124634,\n                0.36460092663764954,\n                0.6987344026565552\n              ],\n              [\n                0.18830986320972443,\n                0.2553125023841858,\n                0.9494835734367371,\n                0.8694687485694885\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000486620.jpg",\n        "coco_id": 486620,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skateboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1209105,\n          "object_id": 640945,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_486620_1209105_640945_1717409_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1717409,\n          "object_id": 640945,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.32279688119888306,\n                0.27281031012535095,\n                0.6194843649864197,\n                0.8979390859603882\n              ],\n              [\n                0.5668437480926514,\n                0.5597658157348633,\n                0.6462812423706055,\n                0.9327166080474854\n              ],\n              [\n                0.32279688119888306,\n                0.27281031012535095,\n                0.6462812423706055,\n                0.9327166080474854\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.7558437585830688,\n                0.0844496488571167,\n                0.9271562695503235,\n                0.8164402842521667\n              ],\n              [\n                0.5668437480926514,\n                0.5597658157348633,\n                0.6462812423706055,\n                0.9327166080474854\n              ],\n              [\n                0.5668437480926514,\n                0.0844496488571167,\n                0.9271562695503235,\n                0.9327166080474854\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000487502.jpg",\n        "coco_id": 487502,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 462106,\n          "object_id": 326255,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_487502_462106_326255_427658_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 427658,\n          "object_id": 326255,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.0,\n                0.1550450474023819,\n                0.31279999017715454,\n                0.986516535282135\n              ],\n              [\n                0.6238800287246704,\n                0.36912912130355835,\n                0.6547399759292603,\n                0.43345344066619873\n              ],\n              [\n                0.0,\n                0.1550450474023819,\n                0.6547399759292603,\n                0.986516535282135\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.527679979801178,\n                0.2889789640903473,\n                0.7498000264167786,\n                0.994654655456543\n              ],\n              [\n                0.6238800287246704,\n                0.36912912130355835,\n                0.6547399759292603,\n                0.43345344066619873\n              ],\n              [\n                0.527679979801178,\n                0.2889789640903473,\n                0.7498000264167786,\n                0.994654655456543\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000487502.jpg",\n        "coco_id": 487502,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 462106,\n          "object_id": 326255,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_487502_462106_326255_427658_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 427658,\n          "object_id": 326255,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.527679979801178,\n                0.2889789640903473,\n                0.7498000264167786,\n                0.994654655456543\n              ],\n              [\n                0.6238800287246704,\n                0.36912912130355835,\n                0.6547399759292603,\n                0.43345344066619873\n              ],\n              [\n                0.527679979801178,\n                0.2889789640903473,\n                0.7498000264167786,\n                0.994654655456543\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.0,\n                0.1550450474023819,\n                0.31279999017715454,\n                0.986516535282135\n              ],\n              [\n                0.6238800287246704,\n                0.36912912130355835,\n                0.6547399759292603,\n                0.43345344066619873\n              ],\n              [\n                0.0,\n                0.1550450474023819,\n                0.6547399759292603,\n                0.986516535282135\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000488203.jpg",\n        "coco_id": 488203,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 194196,\n          "object_id": 1579924,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_488203_194196_1579924_206079_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 206079,\n          "object_id": 1579924,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.1384272277355194,\n                0.555062472820282,\n                0.6313145756721497,\n                0.9595468640327454\n              ],\n              [\n                0.21812206506729126,\n                0.8627656102180481,\n                0.33262911438941956,\n                0.9504218697547913\n              ],\n              [\n                0.1384272277355194,\n                0.555062472820282,\n                0.6313145756721497,\n                0.9595468640327454\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.30584508180618286,\n                0.4096093773841858,\n                0.8469718098640442,\n                0.9890312552452087\n              ],\n              [\n                0.21812206506729126,\n                0.8627656102180481,\n                0.33262911438941956,\n                0.9504218697547913\n              ],\n              [\n                0.21812206506729126,\n                0.4096093773841858,\n                0.8469718098640442,\n                0.9890312552452087\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000489827.jpg",\n        "coco_id": 489827,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "pizza",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 483346,\n          "object_id": 1075488,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_489827_483346_1075488_428751_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 428751,\n          "object_id": 1075488,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.0,\n                0.3724583387374878,\n                0.34809374809265137,\n                0.9845625162124634\n              ],\n              [\n                0.3778750002384186,\n                0.6076250076293945,\n                0.4925312399864197,\n                0.703000009059906\n              ],\n              [\n                0.0,\n                0.3724583387374878,\n                0.4925312399864197,\n                0.9845625162124634\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.38506248593330383,\n                0.3831041753292084,\n                0.6902499794960022,\n                1.0\n              ],\n              [\n                0.3778750002384186,\n                0.6076250076293945,\n                0.4925312399864197,\n                0.703000009059906\n              ],\n              [\n                0.3778750002384186,\n                0.3831041753292084,\n                0.6902499794960022,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000489827.jpg",\n        "coco_id": 489827,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "pizza",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 483346,\n          "object_id": 1075488,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_489827_483346_1075488_428751_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 428751,\n          "object_id": 1075488,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a pizza",\n            "bounds": [\n              [\n                0.38506248593330383,\n                0.3831041753292084,\n                0.6902499794960022,\n                1.0\n              ],\n              [\n                0.3778750002384186,\n                0.6076250076293945,\n                0.4925312399864197,\n                0.703000009059906\n              ],\n              [\n                0.3778750002384186,\n                0.3831041753292084,\n                0.6902499794960022,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a pizza",\n            "bounds": [\n              [\n                0.0,\n                0.3724583387374878,\n                0.34809374809265137,\n                0.9845625162124634\n              ],\n              [\n                0.3778750002384186,\n                0.6076250076293945,\n                0.4925312399864197,\n                0.703000009059906\n              ],\n              [\n                0.0,\n                0.3724583387374878,\n                0.4925312399864197,\n                0.9845625162124634\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000491900.jpg",\n        "coco_id": 491900,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1235130,\n          "object_id": 628041,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_491900_1235130_628041_1317146_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1317146,\n          "object_id": 628041,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.740359365940094,\n                0.3400000035762787,\n                0.9414374828338623,\n                0.8400290608406067\n              ],\n              [\n                0.056953124701976776,\n                0.24520349502563477,\n                0.11807812750339508,\n                0.29622092843055725\n              ],\n              [\n                0.056953124701976776,\n                0.24520349502563477,\n                0.9414374828338623,\n                0.8400290608406067\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.11742187291383743,\n                0.1835465133190155,\n                0.394656240940094,\n                0.845726728439331\n              ],\n              [\n                0.056953124701976776,\n                0.24520349502563477,\n                0.11807812750339508,\n                0.29622092843055725\n              ],\n              [\n                0.056953124701976776,\n                0.1835465133190155,\n                0.394656240940094,\n                0.845726728439331\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000492096.jpg",\n        "coco_id": 492096,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1243144,\n          "object_id": 321676,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_492096_1243144_321676_1259440_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1259440,\n          "object_id": 321676,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.06998437643051147,\n                0.07403755933046341,\n                0.49735936522483826,\n                0.9926995038986206\n              ],\n              [\n                0.610281229019165,\n                0.14762911200523376,\n                0.7000312209129333,\n                0.27347418665885925\n              ],\n              [\n                0.06998437643051147,\n                0.07403755933046341,\n                0.7000312209129333,\n                0.9926995038986206\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.4547187387943268,\n                0.0,\n                0.9199062585830688,\n                0.9835915565490723\n              ],\n              [\n                0.610281229019165,\n                0.14762911200523376,\n                0.7000312209129333,\n                0.27347418665885925\n              ],\n              [\n                0.4547187387943268,\n                0.0,\n                0.9199062585830688,\n                0.9835915565490723\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000492096.jpg",\n        "coco_id": 492096,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1259440,\n          "object_id": 1174048,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_492096_1259440_1174048_1243144_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1243144,\n          "object_id": 1174048,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.4547187387943268,\n                0.0,\n                0.9199062585830688,\n                0.9835915565490723\n              ],\n              [\n                0.3529999852180481,\n                0.3348356783390045,\n                0.49659374356269836,\n                0.9887558817863464\n              ],\n              [\n                0.3529999852180481,\n                0.0,\n                0.9199062585830688,\n                0.9887558817863464\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.06998437643051147,\n                0.07403755933046341,\n                0.49735936522483826,\n                0.9926995038986206\n              ],\n              [\n                0.3529999852180481,\n                0.3348356783390045,\n                0.49659374356269836,\n                0.9887558817863464\n              ],\n              [\n                0.06998437643051147,\n                0.07403755933046341,\n                0.49735936522483826,\n                0.9926995038986206\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000494128.jpg",\n        "coco_id": 494128,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "snowboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 428480,\n          "object_id": 616327,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_494128_428480_616327_427355_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 427355,\n          "object_id": 616327,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.1913125067949295,\n                0.3786666691303253,\n                0.6077812314033508,\n                0.9272083044052124\n              ],\n              [\n                0.07584375143051147,\n                0.34568750858306885,\n                0.47359374165534973,\n                0.5614166855812073\n              ],\n              [\n                0.07584375143051147,\n                0.34568750858306885,\n                0.6077812314033508,\n                0.9272083044052124\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.1100468784570694,\n                0.15293750166893005,\n                0.31489062309265137,\n                0.8504583239555359\n              ],\n              [\n                0.07584375143051147,\n                0.34568750858306885,\n                0.47359374165534973,\n                0.5614166855812073\n              ],\n              [\n                0.07584375143051147,\n                0.15293750166893005,\n                0.47359374165534973,\n                0.8504583239555359\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000496058.jpg",\n        "coco_id": 496058,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 458433,\n          "object_id": 651295,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_496058_458433_651295_425940_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 425940,\n          "object_id": 651295,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.2624218761920929,\n                0.3058750033378601,\n                0.37245312333106995,\n                0.682854175567627\n              ],\n              [\n                0.2627499997615814,\n                0.4326874911785126,\n                0.30076563358306885,\n                0.6763333082199097\n              ],\n              [\n                0.2624218761920929,\n                0.3058750033378601,\n                0.37245312333106995,\n                0.682854175567627\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.5047343969345093,\n                0.5171041488647461,\n                0.6858437657356262,\n                0.8322499990463257\n              ],\n              [\n                0.2627499997615814,\n                0.4326874911785126,\n                0.30076563358306885,\n                0.6763333082199097\n              ],\n              [\n                0.2627499997615814,\n                0.4326874911785126,\n                0.6858437657356262,\n                0.8322499990463257\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000496902.jpg",\n        "coco_id": 496902,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "fork",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 426162,\n          "object_id": 1517447,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_496902_426162_1517447_527691_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 527691,\n          "object_id": 1517447,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.7762656211853027,\n                0.2953958213329315,\n                0.958203136920929,\n                0.6647291779518127\n              ],\n              [\n                0.6133750081062317,\n                0.5687708258628845,\n                0.6662187576293945,\n                0.6236875057220459\n              ],\n              [\n                0.6133750081062317,\n                0.2953958213329315,\n                0.958203136920929,\n                0.6647291779518127\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.44187501072883606,\n                0.2251666635274887,\n                0.7076718807220459,\n                0.7330625057220459\n              ],\n              [\n                0.6133750081062317,\n                0.5687708258628845,\n                0.6662187576293945,\n                0.6236875057220459\n              ],\n              [\n                0.44187501072883606,\n                0.2251666635274887,\n                0.7076718807220459,\n                0.7330625057220459\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000496902.jpg",\n        "coco_id": 496902,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "cake",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 426162,\n          "object_id": 2119299,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_496902_426162_2119299_447834_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 447834,\n          "object_id": 2119299,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.44187501072883606,\n                0.2251666635274887,\n                0.7076718807220459,\n                0.7330625057220459\n              ],\n              [\n                0.5915156006813049,\n                0.7086041569709778,\n                0.6852656006813049,\n                0.7597708106040955\n              ],\n              [\n                0.44187501072883606,\n                0.2251666635274887,\n                0.7076718807220459,\n                0.7597708106040955\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a cake",\n            "bounds": [\n              [\n                0.6455000042915344,\n                0.22472916543483734,\n                0.8224687576293945,\n                0.6089791655540466\n              ],\n              [\n                0.5915156006813049,\n                0.7086041569709778,\n                0.6852656006813049,\n                0.7597708106040955\n              ],\n              [\n                0.5915156006813049,\n                0.22472916543483734,\n                0.8224687576293945,\n                0.7597708106040955\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000498218.jpg",\n        "coco_id": 498218,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 226712,\n          "object_id": 245751,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_498218_226712_245751_195150_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 195150,\n          "object_id": 245751,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.6678000092506409,\n                0.013475609943270683,\n                0.8697599768638611,\n                0.6561890244483948\n              ],\n              [\n                0.5678600072860718,\n                0.2442987859249115,\n                1.0,\n                0.7480487823486328\n              ],\n              [\n                0.5678600072860718,\n                0.013475609943270683,\n                1.0,\n                0.7480487823486328\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.19016000628471375,\n                0.16405487060546875,\n                0.5985000133514404,\n                0.7213414907455444\n              ],\n              [\n                0.5678600072860718,\n                0.2442987859249115,\n                1.0,\n                0.7480487823486328\n              ],\n              [\n                0.19016000628471375,\n                0.16405487060546875,\n                1.0,\n                0.7480487823486328\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000498218.jpg",\n        "coco_id": 498218,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 226712,\n          "object_id": 245751,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_498218_226712_245751_195150_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 195150,\n          "object_id": 245751,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.19016000628471375,\n                0.16405487060546875,\n                0.5985000133514404,\n                0.7213414907455444\n              ],\n              [\n                0.5678600072860718,\n                0.2442987859249115,\n                1.0,\n                0.7480487823486328\n              ],\n              [\n                0.19016000628471375,\n                0.16405487060546875,\n                1.0,\n                0.7480487823486328\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.6678000092506409,\n                0.013475609943270683,\n                0.8697599768638611,\n                0.6561890244483948\n              ],\n              [\n                0.5678600072860718,\n                0.2442987859249115,\n                1.0,\n                0.7480487823486328\n              ],\n              [\n                0.5678600072860718,\n                0.013475609943270683,\n                1.0,\n                0.7480487823486328\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000498218.jpg",\n        "coco_id": 498218,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 226712,\n          "object_id": 245751,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_498218_226712_245751_227750_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 227750,\n          "object_id": 245751,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.6678000092506409,\n                0.013475609943270683,\n                0.8697599768638611,\n                0.6561890244483948\n              ],\n              [\n                0.5678600072860718,\n                0.2442987859249115,\n                1.0,\n                0.7480487823486328\n              ],\n              [\n                0.5678600072860718,\n                0.013475609943270683,\n                1.0,\n                0.7480487823486328\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.7724599838256836,\n                0.04874999821186066,\n                0.9744200110435486,\n                0.637530505657196\n              ],\n              [\n                0.5678600072860718,\n                0.2442987859249115,\n                1.0,\n                0.7480487823486328\n              ],\n              [\n                0.5678600072860718,\n                0.04874999821186066,\n                1.0,\n                0.7480487823486328\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000498218.jpg",\n        "coco_id": 498218,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 227750,\n          "object_id": 245751,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_498218_227750_245751_195150_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 195150,\n          "object_id": 245751,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.7724599838256836,\n                0.04874999821186066,\n                0.9744200110435486,\n                0.637530505657196\n              ],\n              [\n                0.5678600072860718,\n                0.2442987859249115,\n                1.0,\n                0.7480487823486328\n              ],\n              [\n                0.5678600072860718,\n                0.04874999821186066,\n                1.0,\n                0.7480487823486328\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.19016000628471375,\n                0.16405487060546875,\n                0.5985000133514404,\n                0.7213414907455444\n              ],\n              [\n                0.5678600072860718,\n                0.2442987859249115,\n                1.0,\n                0.7480487823486328\n              ],\n              [\n                0.19016000628471375,\n                0.16405487060546875,\n                1.0,\n                0.7480487823486328\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000498218.jpg",\n        "coco_id": 498218,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 227750,\n          "object_id": 245751,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_498218_227750_245751_195150_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 195150,\n          "object_id": 245751,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.7724599838256836,\n                0.04874999821186066,\n                0.9744200110435486,\n                0.637530505657196\n              ],\n              [\n                0.5678600072860718,\n                0.2442987859249115,\n                1.0,\n                0.7480487823486328\n              ],\n              [\n                0.5678600072860718,\n                0.04874999821186066,\n                1.0,\n                0.7480487823486328\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.19016000628471375,\n                0.16405487060546875,\n                0.5985000133514404,\n                0.7213414907455444\n              ],\n              [\n                0.5678600072860718,\n                0.2442987859249115,\n                1.0,\n                0.7480487823486328\n              ],\n              [\n                0.19016000628471375,\n                0.16405487060546875,\n                1.0,\n                0.7480487823486328\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000499865.jpg",\n        "coco_id": 499865,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 234005,\n          "object_id": 1180432,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_499865_234005_1180432_184715_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 184715,\n          "object_id": 1180432,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.29183998703956604,\n                0.1528228223323822,\n                0.4624600112438202,\n                0.9056156277656555\n              ],\n              [\n                0.30441999435424805,\n                0.37378379702568054,\n                0.4136599898338318,\n                0.5737837553024292\n              ],\n              [\n                0.29183998703956604,\n                0.1528228223323822,\n                0.4624600112438202,\n                0.9056156277656555\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.0,\n                0.25804805755615234,\n                0.1601399928331375,\n                0.9030029773712158\n              ],\n              [\n                0.30441999435424805,\n                0.37378379702568054,\n                0.4136599898338318,\n                0.5737837553024292\n              ],\n              [\n                0.0,\n                0.25804805755615234,\n                0.4136599898338318,\n                0.9030029773712158\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000500076.jpg",\n        "coco_id": 500076,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 189279,\n          "object_id": 55673,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_500076_189279_55673_218969_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 218969,\n          "object_id": 55673,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.166015625,\n                0.1634455919265747,\n                0.28404688835144043,\n                0.6279533505439758\n              ],\n              [\n                0.01626562513411045,\n                0.29212436079978943,\n                0.3916875123977661,\n                0.8809067606925964\n              ],\n              [\n                0.01626562513411045,\n                0.1634455919265747,\n                0.3916875123977661,\n                0.8809067606925964\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.5516250133514404,\n                0.15056994557380676,\n                0.6736093759536743,\n                0.7033678889274597\n              ],\n              [\n                0.01626562513411045,\n                0.29212436079978943,\n                0.3916875123977661,\n                0.8809067606925964\n              ],\n              [\n                0.01626562513411045,\n                0.15056994557380676,\n                0.6736093759536743,\n                0.8809067606925964\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000501647.jpg",\n        "coco_id": 501647,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 435839,\n          "object_id": 631610,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_501647_435839_631610_504136_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 504136,\n          "object_id": 631610,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.6827187538146973,\n                0.45754098892211914,\n                0.9267030954360962,\n                0.9314988255500793\n              ],\n              [\n                0.30959373712539673,\n                0.41079625487327576,\n                0.40495312213897705,\n                0.46264636516571045\n              ],\n              [\n                0.30959373712539673,\n                0.41079625487327576,\n                0.9267030954360962,\n                0.9314988255500793\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.3140625059604645,\n                0.288290411233902,\n                0.5439687371253967,\n                0.8243325352668762\n              ],\n              [\n                0.30959373712539673,\n                0.41079625487327576,\n                0.40495312213897705,\n                0.46264636516571045\n              ],\n              [\n                0.30959373712539673,\n                0.288290411233902,\n                0.5439687371253967,\n                0.8243325352668762\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000502134.jpg",\n        "coco_id": 502134,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 222517,\n          "object_id": 149868,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_502134_222517_149868_228325_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 228325,\n          "object_id": 149868,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.271359384059906,\n                0.2045433223247528,\n                0.6034062504768372,\n                0.9979156851768494\n              ],\n              [\n                0.0,\n                0.3670960068702698,\n                0.9492812752723694,\n                0.9890398383140564\n              ],\n              [\n                0.0,\n                0.2045433223247528,\n                0.9492812752723694,\n                0.9979156851768494\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.07196874916553497,\n                0.1280796229839325,\n                0.32984375953674316,\n                0.9820140600204468\n              ],\n              [\n                0.0,\n                0.3670960068702698,\n                0.9492812752723694,\n                0.9890398383140564\n              ],\n              [\n                0.0,\n                0.1280796229839325,\n                0.9492812752723694,\n                0.9890398383140564\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000050274.jpg",\n        "coco_id": 50274,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 560894,\n          "object_id": 604408,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_50274_560894_604408_438852_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 438852,\n          "object_id": 604408,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.3943749964237213,\n                0.041999999433755875,\n                0.9391875267028809,\n                0.9835833311080933\n              ],\n              [\n                0.38945311307907104,\n                0.27130556106567383,\n                0.5655156373977661,\n                0.3701111078262329\n              ],\n              [\n                0.38945311307907104,\n                0.041999999433755875,\n                0.9391875267028809,\n                0.9835833311080933\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.09228125214576721,\n                0.32136112451553345,\n                0.3589843809604645,\n                1.0\n              ],\n              [\n                0.38945311307907104,\n                0.27130556106567383,\n                0.5655156373977661,\n                0.3701111078262329\n              ],\n              [\n                0.09228125214576721,\n                0.27130556106567383,\n                0.5655156373977661,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000503535.jpg",\n        "coco_id": 503535,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1232684,\n          "object_id": 1165273,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_503535_1232684_1165273_552217_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 552217,\n          "object_id": 1165273,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.2583906352519989,\n                0.25077083706855774,\n                0.4144687354564667,\n                1.0\n              ],\n              [\n                0.8177812695503235,\n                0.42391666769981384,\n                0.8820156455039978,\n                0.659500002861023\n              ],\n              [\n                0.2583906352519989,\n                0.25077083706855774,\n                0.8820156455039978,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.6809062361717224,\n                0.257750004529953,\n                0.8612343668937683,\n                0.9858333468437195\n              ],\n              [\n                0.8177812695503235,\n                0.42391666769981384,\n                0.8820156455039978,\n                0.659500002861023\n              ],\n              [\n                0.6809062361717224,\n                0.257750004529953,\n                0.8820156455039978,\n                0.9858333468437195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000503535.jpg",\n        "coco_id": 503535,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "snowboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1232684,\n          "object_id": 620949,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_503535_1232684_620949_552217_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 552217,\n          "object_id": 620949,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.2583906352519989,\n                0.25077083706855774,\n                0.4144687354564667,\n                1.0\n              ],\n              [\n                0.6644218564033508,\n                0.3648749887943268,\n                0.7281875014305115,\n                0.8682291507720947\n              ],\n              [\n                0.2583906352519989,\n                0.25077083706855774,\n                0.7281875014305115,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.6809062361717224,\n                0.257750004529953,\n                0.8612343668937683,\n                0.9858333468437195\n              ],\n              [\n                0.6644218564033508,\n                0.3648749887943268,\n                0.7281875014305115,\n                0.8682291507720947\n              ],\n              [\n                0.6644218564033508,\n                0.257750004529953,\n                0.8612343668937683,\n                0.9858333468437195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000503535.jpg",\n        "coco_id": 503535,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "snowboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2161551,\n          "object_id": 619068,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_503535_2161551_619068_552217_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 552217,\n          "object_id": 619068,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.2583906352519989,\n                0.25077083706855774,\n                0.4144687354564667,\n                1.0\n              ],\n              [\n                0.46364063024520874,\n                0.31714582443237305,\n                0.5514843463897705,\n                0.9941458106040955\n              ],\n              [\n                0.2583906352519989,\n                0.25077083706855774,\n                0.5514843463897705,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.53515625,\n                0.2903749942779541,\n                0.697265625,\n                0.9830833077430725\n              ],\n              [\n                0.46364063024520874,\n                0.31714582443237305,\n                0.5514843463897705,\n                0.9941458106040955\n              ],\n              [\n                0.46364063024520874,\n                0.2903749942779541,\n                0.697265625,\n                0.9941458106040955\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000511328.jpg",\n        "coco_id": 511328,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "wine glass",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1753009,\n          "object_id": 666342,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_511328_1753009_666342_1700871_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1700871,\n          "object_id": 666342,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a wine glass",\n            "bounds": [\n              [\n                0.5407343506813049,\n                0.6370416879653931,\n                0.9629687666893005,\n                0.9876458048820496\n              ],\n              [\n                0.5943437218666077,\n                0.8334375023841858,\n                0.7017187476158142,\n                1.0\n              ],\n              [\n                0.5407343506813049,\n                0.6370416879653931,\n                0.9629687666893005,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a wine glass",\n            "bounds": [\n              [\n                0.4971874952316284,\n                0.0494375005364418,\n                0.7617968916893005,\n                0.5820208191871643\n              ],\n              [\n                0.5943437218666077,\n                0.8334375023841858,\n                0.7017187476158142,\n                1.0\n              ],\n              [\n                0.4971874952316284,\n                0.0494375005364418,\n                0.7617968916893005,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000511328.jpg",\n        "coco_id": 511328,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "wine glass",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1755680,\n          "object_id": 666432,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_511328_1755680_666432_1750455_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1750455,\n          "object_id": 666432,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a wine glass",\n            "bounds": [\n              [\n                0.033906251192092896,\n                0.3008125126361847,\n                0.43382811546325684,\n                0.9465625286102295\n              ],\n              [\n                0.3781718611717224,\n                0.6845208406448364,\n                0.4544062614440918,\n                0.921708345413208\n              ],\n              [\n                0.033906251192092896,\n                0.3008125126361847,\n                0.4544062614440918,\n                0.9465625286102295\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a wine glass",\n            "bounds": [\n              [\n                0.2409999966621399,\n                0.3059374988079071,\n                0.5008281469345093,\n                0.8187916874885559\n              ],\n              [\n                0.3781718611717224,\n                0.6845208406448364,\n                0.4544062614440918,\n                0.921708345413208\n              ],\n              [\n                0.2409999966621399,\n                0.3059374988079071,\n                0.5008281469345093,\n                0.921708345413208\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000513015.jpg",\n        "coco_id": 513015,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1235661,\n          "object_id": 632236,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_513015_1235661_632236_1242045_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1242045,\n          "object_id": 632236,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.20995312929153442,\n                0.34418749809265137,\n                0.5186406373977661,\n                0.804979145526886\n              ],\n              [\n                0.17893749475479126,\n                0.3233333230018616,\n                0.28023436665534973,\n                0.44443750381469727\n              ],\n              [\n                0.17893749475479126,\n                0.3233333230018616,\n                0.5186406373977661,\n                0.804979145526886\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.6093437671661377,\n                0.46835416555404663,\n                0.8534374833106995,\n                0.7712500095367432\n              ],\n              [\n                0.17893749475479126,\n                0.3233333230018616,\n                0.28023436665534973,\n                0.44443750381469727\n              ],\n              [\n                0.17893749475479126,\n                0.3233333230018616,\n                0.8534374833106995,\n                0.7712500095367432\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000517113.jpg",\n        "coco_id": 517113,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 434743,\n          "object_id": 632256,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_517113_434743_632256_497490_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 497490,\n          "object_id": 632256,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.02096875011920929,\n                0.5249648690223694,\n                0.2994999885559082,\n                0.9110538363456726\n              ],\n              [\n                0.6057656407356262,\n                0.4746370017528534,\n                0.7252812385559082,\n                0.752997636795044\n              ],\n              [\n                0.02096875011920929,\n                0.4746370017528534,\n                0.7252812385559082,\n                0.9110538363456726\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.5352500081062317,\n                0.3303278684616089,\n                0.7811406254768372,\n                0.8876346349716187\n              ],\n              [\n                0.6057656407356262,\n                0.4746370017528534,\n                0.7252812385559082,\n                0.752997636795044\n              ],\n              [\n                0.5352500081062317,\n                0.3303278684616089,\n                0.7811406254768372,\n                0.8876346349716187\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000518124.jpg",\n        "coco_id": 518124,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bottle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 476542,\n          "object_id": 1483139,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_518124_476542_1483139_455178_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 455178,\n          "object_id": 1483139,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bottle",\n            "bounds": [\n              [\n                0.07773437350988388,\n                0.16817757487297058,\n                0.1867968738079071,\n                0.6010981202125549\n              ],\n              [\n                0.42774999141693115,\n                0.25212615728378296,\n                0.44090625643730164,\n                0.3214252293109894\n              ],\n              [\n                0.07773437350988388,\n                0.16817757487297058,\n                0.44090625643730164,\n                0.6010981202125549\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bottle",\n            "bounds": [\n              [\n                0.3348906338214874,\n                0.20385514199733734,\n                0.44321873784065247,\n                0.5953037142753601\n              ],\n              [\n                0.42774999141693115,\n                0.25212615728378296,\n                0.44090625643730164,\n                0.3214252293109894\n              ],\n              [\n                0.3348906338214874,\n                0.20385514199733734,\n                0.44321873784065247,\n                0.5953037142753601\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000518649.jpg",\n        "coco_id": 518649,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 496069,\n          "object_id": 1433754,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_518649_496069_1433754_522299_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 522299,\n          "object_id": 1433754,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.003078124951571226,\n                0.5618631839752197,\n                0.1008281260728836,\n                0.9834433794021606\n              ],\n              [\n                0.9538906216621399,\n                0.9025235772132874,\n                0.9777187705039978,\n                0.9892924427986145\n              ],\n              [\n                0.003078124951571226,\n                0.5618631839752197,\n                0.9777187705039978,\n                0.9892924427986145\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.8460312485694885,\n                0.604740560054779,\n                0.974359393119812,\n                0.9898585081100464\n              ],\n              [\n                0.9538906216621399,\n                0.9025235772132874,\n                0.9777187705039978,\n                0.9892924427986145\n              ],\n              [\n                0.8460312485694885,\n                0.604740560054779,\n                0.9777187705039978,\n                0.9898585081100464\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000518649.jpg",\n        "coco_id": 518649,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 496069,\n          "object_id": 1436139,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_518649_496069_1436139_522299_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 522299,\n          "object_id": 1436139,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.003078124951571226,\n                0.5618631839752197,\n                0.1008281260728836,\n                0.9834433794021606\n              ],\n              [\n                0.826296865940094,\n                0.8210613131523132,\n                0.8704531192779541,\n                0.9426179528236389\n              ],\n              [\n                0.003078124951571226,\n                0.5618631839752197,\n                0.8704531192779541,\n                0.9834433794021606\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.8460312485694885,\n                0.604740560054779,\n                0.974359393119812,\n                0.9898585081100464\n              ],\n              [\n                0.826296865940094,\n                0.8210613131523132,\n                0.8704531192779541,\n                0.9426179528236389\n              ],\n              [\n                0.826296865940094,\n                0.604740560054779,\n                0.974359393119812,\n                0.9898585081100464\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000522256.jpg",\n        "coco_id": 522256,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 466801,\n          "object_id": 627689,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_522256_466801_627689_477242_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 477242,\n          "object_id": 627689,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.5883281230926514,\n                0.25470954179763794,\n                0.7867968678474426,\n                0.9985062479972839\n              ],\n              [\n                0.3099375069141388,\n                0.5468049645423889,\n                0.5823749899864197,\n                0.8329460620880127\n              ],\n              [\n                0.3099375069141388,\n                0.25470954179763794,\n                0.7867968678474426,\n                0.9985062479972839\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.13142187893390656,\n                0.12325726449489594,\n                0.4616406261920929,\n                0.8749377727508545\n              ],\n              [\n                0.3099375069141388,\n                0.5468049645423889,\n                0.5823749899864197,\n                0.8329460620880127\n              ],\n              [\n                0.13142187893390656,\n                0.12325726449489594,\n                0.5823749899864197,\n                0.8749377727508545\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000522719.jpg",\n        "coco_id": 522719,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1251645,\n          "object_id": 326887,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_522719_1251645_326887_1310890_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1310890,\n          "object_id": 326887,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.004484375007450581,\n                0.35730046033859253,\n                0.18248437345027924,\n                0.98652583360672\n              ],\n              [\n                0.3995625078678131,\n                0.37192487716674805,\n                0.5647500157356262,\n                0.5052347183227539\n              ],\n              [\n                0.004484375007450581,\n                0.35730046033859253,\n                0.5647500157356262,\n                0.98652583360672\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.3253124952316284,\n                0.2984272241592407,\n                0.6491405963897705,\n                0.9898591637611389\n              ],\n              [\n                0.3995625078678131,\n                0.37192487716674805,\n                0.5647500157356262,\n                0.5052347183227539\n              ],\n              [\n                0.3253124952316284,\n                0.2984272241592407,\n                0.6491405963897705,\n                0.9898591637611389\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000528230.jpg",\n        "coco_id": 528230,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1235701,\n          "object_id": 628132,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_528230_1235701_628132_1322781_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1322781,\n          "object_id": 628132,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.36851561069488525,\n                0.49133333563804626,\n                0.5453125238418579,\n                0.8444374799728394\n              ],\n              [\n                0.28053125739097595,\n                0.5600416660308838,\n                0.3744843602180481,\n                0.6448958516120911\n              ],\n              [\n                0.28053125739097595,\n                0.49133333563804626,\n                0.5453125238418579,\n                0.8444374799728394\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.5832968950271606,\n                0.5942291617393494,\n                0.7014531493186951,\n                0.8450000286102295\n              ],\n              [\n                0.28053125739097595,\n                0.5600416660308838,\n                0.3744843602180481,\n                0.6448958516120911\n              ],\n              [\n                0.28053125739097595,\n                0.5600416660308838,\n                0.7014531493186951,\n                0.8450000286102295\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000530758.jpg",\n        "coco_id": 530758,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 198525,\n          "object_id": 241223,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_530758_198525_241223_1746560_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1746560,\n          "object_id": 241223,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.7226094007492065,\n                0.5129976868629456,\n                0.9328437447547913,\n                0.7874239087104797\n              ],\n              [\n                0.0788593739271164,\n                0.6158782243728638,\n                0.24646875262260437,\n                0.9630913138389587\n              ],\n              [\n                0.0788593739271164,\n                0.5129976868629456,\n                0.9328437447547913,\n                0.9630913138389587\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.07928124815225601,\n                0.3007025718688965,\n                0.2543124854564667,\n                0.7984777688980103\n              ],\n              [\n                0.0788593739271164,\n                0.6158782243728638,\n                0.24646875262260437,\n                0.9630913138389587\n              ],\n              [\n                0.0788593739271164,\n                0.3007025718688965,\n                0.2543124854564667,\n                0.9630913138389587\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000532704.jpg",\n        "coco_id": 532704,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 208088,\n          "object_id": 1930813,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_532704_208088_1930813_214801_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 214801,\n          "object_id": 1930813,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.17881250381469727,\n                0.23976580798625946,\n                0.5850937366485596,\n                1.0\n              ],\n              [\n                0.023984374478459358,\n                0.6199531555175781,\n                0.26384374499320984,\n                0.9014285802841187\n              ],\n              [\n                0.023984374478459358,\n                0.23976580798625946,\n                0.5850937366485596,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.06926562637090683,\n                0.393325537443161,\n                0.24904687702655792,\n                0.9288524389266968\n              ],\n              [\n                0.023984374478459358,\n                0.6199531555175781,\n                0.26384374499320984,\n                0.9014285802841187\n              ],\n              [\n                0.023984374478459358,\n                0.393325537443161,\n                0.26384374499320984,\n                0.9288524389266968\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000532704.jpg",\n        "coco_id": 532704,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "remote",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 214801,\n          "object_id": 1113297,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_532704_214801_1113297_208088_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 208088,\n          "object_id": 1113297,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.17881250381469727,\n                0.23976580798625946,\n                0.5850937366485596,\n                1.0\n              ],\n              [\n                0.5577031373977661,\n                0.43149882555007935,\n                0.5832343697547913,\n                0.46234193444252014\n              ],\n              [\n                0.17881250381469727,\n                0.23976580798625946,\n                0.5850937366485596,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.06926562637090683,\n                0.393325537443161,\n                0.24904687702655792,\n                0.9288524389266968\n              ],\n              [\n                0.5577031373977661,\n                0.43149882555007935,\n                0.5832343697547913,\n                0.46234193444252014\n              ],\n              [\n                0.06926562637090683,\n                0.393325537443161,\n                0.5832343697547913,\n                0.9288524389266968\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000534655.jpg",\n        "coco_id": 534655,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 544054,\n          "object_id": 1480612,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_534655_544054_1480612_528966_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 528966,\n          "object_id": 1480612,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.016499999910593033,\n                0.321358323097229,\n                0.0794687494635582,\n                0.6584309339523315\n              ],\n              [\n                0.004812499973922968,\n                0.3936065435409546,\n                0.13554687798023224,\n                0.43714284896850586\n              ],\n              [\n                0.004812499973922968,\n                0.321358323097229,\n                0.13554687798023224,\n                0.6584309339523315\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.12132812291383743,\n                0.36138173937797546,\n                0.22426562011241913,\n                0.7257845401763916\n              ],\n              [\n                0.004812499973922968,\n                0.3936065435409546,\n                0.13554687798023224,\n                0.43714284896850586\n              ],\n              [\n                0.004812499973922968,\n                0.36138173937797546,\n                0.22426562011241913,\n                0.7257845401763916\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000534655.jpg",\n        "coco_id": 534655,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "surfboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 544054,\n          "object_id": 1481342,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_534655_544054_1481342_528966_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 528966,\n          "object_id": 1481342,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a surfboard",\n            "bounds": [\n              [\n                0.12132812291383743,\n                0.36138173937797546,\n                0.22426562011241913,\n                0.7257845401763916\n              ],\n              [\n                0.0,\n                0.4235363006591797,\n                0.1346093714237213,\n                0.5157142877578735\n              ],\n              [\n                0.0,\n                0.36138173937797546,\n                0.22426562011241913,\n                0.7257845401763916\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a surfboard",\n            "bounds": [\n              [\n                0.016499999910593033,\n                0.321358323097229,\n                0.0794687494635582,\n                0.6584309339523315\n              ],\n              [\n                0.0,\n                0.4235363006591797,\n                0.1346093714237213,\n                0.5157142877578735\n              ],\n              [\n                0.0,\n                0.321358323097229,\n                0.1346093714237213,\n                0.6584309339523315\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000537617.jpg",\n        "coco_id": 537617,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 199064,\n          "object_id": 153476,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_537617_199064_153476_200272_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 200272,\n          "object_id": 153476,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.38468751311302185,\n                0.23086650669574738,\n                0.47935938835144043,\n                0.640772819519043\n              ],\n              [\n                0.3063593804836273,\n                0.2744262218475342,\n                0.5792343616485596,\n                0.7732787132263184\n              ],\n              [\n                0.3063593804836273,\n                0.23086650669574738,\n                0.5792343616485596,\n                0.7732787132263184\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.34078124165534973,\n                0.1606089025735855,\n                0.5554843544960022,\n                0.7327634692192078\n              ],\n              [\n                0.3063593804836273,\n                0.2744262218475342,\n                0.5792343616485596,\n                0.7732787132263184\n              ],\n              [\n                0.3063593804836273,\n                0.1606089025735855,\n                0.5792343616485596,\n                0.7732787132263184\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000053939.jpg",\n        "coco_id": 53939,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1213314,\n          "object_id": 1946569,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_53939_1213314_1946569_2022499_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2022499,\n          "object_id": 1946569,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.10645312815904617,\n                0.25172916054725647,\n                0.22123438119888306,\n                0.6110625267028809\n              ],\n              [\n                0.35935938358306885,\n                0.7336875200271606,\n                0.476624995470047,\n                0.9312708377838135\n              ],\n              [\n                0.10645312815904617,\n                0.25172916054725647,\n                0.476624995470047,\n                0.9312708377838135\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.37517186999320984,\n                0.4414791762828827,\n                0.5834375023841858,\n                0.942395806312561\n              ],\n              [\n                0.35935938358306885,\n                0.7336875200271606,\n                0.476624995470047,\n                0.9312708377838135\n              ],\n              [\n                0.35935938358306885,\n                0.4414791762828827,\n                0.5834375023841858,\n                0.942395806312561\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000053939.jpg",\n        "coco_id": 53939,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1268968,\n          "object_id": 1415443,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_53939_1268968_1415443_2022499_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2022499,\n          "object_id": 1415443,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.10645312815904617,\n                0.25172916054725647,\n                0.22123438119888306,\n                0.6110625267028809\n              ],\n              [\n                0.23045311868190765,\n                0.3217083215713501,\n                0.3200937509536743,\n                0.4310624897480011\n              ],\n              [\n                0.10645312815904617,\n                0.25172916054725647,\n                0.3200937509536743,\n                0.6110625267028809\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.19740624725818634,\n                0.2776041626930237,\n                0.3386250138282776,\n                0.6082291603088379\n              ],\n              [\n                0.23045311868190765,\n                0.3217083215713501,\n                0.3200937509536743,\n                0.4310624897480011\n              ],\n              [\n                0.19740624725818634,\n                0.2776041626930237,\n                0.3386250138282776,\n                0.6082291603088379\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000054130.jpg",\n        "coco_id": 54130,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skis",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1309022,\n          "object_id": 1454846,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_54130_1309022_1454846_1208053_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1208053,\n          "object_id": 1454846,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding skis",\n            "bounds": [\n              [\n                0.2337031215429306,\n                0.379093736410141,\n                0.3887656331062317,\n                0.6442499756813049\n              ],\n              [\n                0.19437499344348907,\n                0.6311249732971191,\n                0.431765615940094,\n                0.6455312371253967\n              ],\n              [\n                0.19437499344348907,\n                0.379093736410141,\n                0.431765615940094,\n                0.6455312371253967\n              ]\n            ]\n          },\n          {\n            "text": "a person holding skis",\n            "bounds": [\n              [\n                0.688406229019165,\n                0.3871406316757202,\n                0.822390615940094,\n                0.672921895980835\n              ],\n              [\n                0.19437499344348907,\n                0.6311249732971191,\n                0.431765615940094,\n                0.6455312371253967\n              ],\n              [\n                0.19437499344348907,\n                0.3871406316757202,\n                0.822390615940094,\n                0.672921895980835\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000542925.jpg",\n        "coco_id": 542925,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1292662,\n          "object_id": 577990,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_542925_1292662_577990_1289976_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1289976,\n          "object_id": 577990,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.02292187511920929,\n                0.4906572699546814,\n                0.1159375011920929,\n                0.7511502504348755\n              ],\n              [\n                0.9004843831062317,\n                0.6480985879898071,\n                0.9961562752723694,\n                0.6805399060249329\n              ],\n              [\n                0.02292187511920929,\n                0.4906572699546814,\n                0.9961562752723694,\n                0.7511502504348755\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.8451250195503235,\n                0.5078638792037964,\n                0.9333750009536743,\n                0.7730281949043274\n              ],\n              [\n                0.9004843831062317,\n                0.6480985879898071,\n                0.9961562752723694,\n                0.6805399060249329\n              ],\n              [\n                0.8451250195503235,\n                0.5078638792037964,\n                0.9961562752723694,\n                0.7730281949043274\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000543585.jpg",\n        "coco_id": 543585,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 432825,\n          "object_id": 629736,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_543585_432825_629736_461402_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 461402,\n          "object_id": 629736,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.03641566261649132,\n                0.4187999963760376,\n                0.42090362310409546,\n                0.9998599886894226\n              ],\n              [\n                0.4162951707839966,\n                0.5804200172424316,\n                0.8559939861297607,\n                0.667739987373352\n              ],\n              [\n                0.03641566261649132,\n                0.4187999963760376,\n                0.8559939861297607,\n                0.9998599886894226\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.17560240626335144,\n                0.5469800233840942,\n                0.7327409386634827,\n                0.9864400029182434\n              ],\n              [\n                0.4162951707839966,\n                0.5804200172424316,\n                0.8559939861297607,\n                0.667739987373352\n              ],\n              [\n                0.17560240626335144,\n                0.5469800233840942,\n                0.8559939861297607,\n                0.9864400029182434\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000546586.jpg",\n        "coco_id": 546586,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1239772,\n          "object_id": 1862744,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_546586_1239772_1862744_2156244_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2156244,\n          "object_id": 1862744,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.051552288234233856,\n                0.3777124285697937,\n                0.11616013199090958,\n                0.47540849447250366\n              ],\n              [\n                0.018807189539074898,\n                0.43900325894355774,\n                0.1568300724029541,\n                0.5343464016914368\n              ],\n              [\n                0.018807189539074898,\n                0.3777124285697937,\n                0.1568300724029541,\n                0.5343464016914368\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.13696078956127167,\n                0.33944445848464966,\n                0.1884150356054306,\n                0.4880392253398895\n              ],\n              [\n                0.018807189539074898,\n                0.43900325894355774,\n                0.1568300724029541,\n                0.5343464016914368\n              ],\n              [\n                0.018807189539074898,\n                0.33944445848464966,\n                0.1884150356054306,\n                0.5343464016914368\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000550028.jpg",\n        "coco_id": 550028,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1711406,\n          "object_id": 631238,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_550028_1711406_631238_542449_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 542449,\n          "object_id": 631238,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.6000781059265137,\n                0.21345703303813934,\n                0.9222656488418579,\n                0.8565820455551147\n              ],\n              [\n                0.9116250276565552,\n                0.16828125715255737,\n                1.0,\n                0.29249998927116394\n              ],\n              [\n                0.6000781059265137,\n                0.16828125715255737,\n                1.0,\n                0.8565820455551147\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.00465625012293458,\n                0.39164063334465027,\n                0.22048437595367432,\n                0.8531835675239563\n              ],\n              [\n                0.9116250276565552,\n                0.16828125715255737,\n                1.0,\n                0.29249998927116394\n              ],\n              [\n                0.00465625012293458,\n                0.16828125715255737,\n                1.0,\n                0.8531835675239563\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000550028.jpg",\n        "coco_id": 550028,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 540266,\n          "object_id": 635529,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_550028_540266_635529_542449_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 542449,\n          "object_id": 635529,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.00465625012293458,\n                0.39164063334465027,\n                0.22048437595367432,\n                0.8531835675239563\n              ],\n              [\n                0.45759373903274536,\n                0.5794336199760437,\n                0.5259843468666077,\n                0.6845117211341858\n              ],\n              [\n                0.00465625012293458,\n                0.39164063334465027,\n                0.5259843468666077,\n                0.8531835675239563\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.0775468721985817,\n                0.44583985209465027,\n                0.5168750286102295,\n                0.857421875\n              ],\n              [\n                0.45759373903274536,\n                0.5794336199760437,\n                0.5259843468666077,\n                0.6845117211341858\n              ],\n              [\n                0.0775468721985817,\n                0.44583985209465027,\n                0.5259843468666077,\n                0.857421875\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000055385.jpg",\n        "coco_id": 55385,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 187714,\n          "object_id": 1420798,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_55385_187714_1420798_237039_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 237039,\n          "object_id": 1420798,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.3807656168937683,\n                0.01505639124661684,\n                0.6900312304496765,\n                0.8128947615623474\n              ],\n              [\n                0.4799531102180481,\n                0.19097743928432465,\n                0.6662343740463257,\n                0.4093233048915863\n              ],\n              [\n                0.3807656168937683,\n                0.01505639124661684,\n                0.6900312304496765,\n                0.8128947615623474\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.6103749871253967,\n                0.11729323118925095,\n                0.9339843988418579,\n                1.0\n              ],\n              [\n                0.4799531102180481,\n                0.19097743928432465,\n                0.6662343740463257,\n                0.4093233048915863\n              ],\n              [\n                0.4799531102180481,\n                0.11729323118925095,\n                0.9339843988418579,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000554886.jpg",\n        "coco_id": 554886,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 422587,\n          "object_id": 634865,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_554886_422587_634865_1209499_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1209499,\n          "object_id": 634865,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.0052656251937150955,\n                0.2876400053501129,\n                0.5846250057220459,\n                0.9280800223350525\n              ],\n              [\n                0.4748593866825104,\n                0.5048800110816956,\n                0.616406261920929,\n                0.6391599774360657\n              ],\n              [\n                0.0052656251937150955,\n                0.2876400053501129,\n                0.616406261920929,\n                0.9280800223350525\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.5947343707084656,\n                0.016899999231100082,\n                0.9396093487739563,\n                0.9583399891853333\n              ],\n              [\n                0.4748593866825104,\n                0.5048800110816956,\n                0.616406261920929,\n                0.6391599774360657\n              ],\n              [\n                0.4748593866825104,\n                0.016899999231100082,\n                0.9396093487739563,\n                0.9583399891853333\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000556342.jpg",\n        "coco_id": 556342,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "person",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 537851,\n          "object_id": 2165998,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_556342_537851_2165998_451874_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 451874,\n          "object_id": 2165998,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.4526093602180481,\n                0.46595123410224915,\n                0.5174843668937683,\n                0.7564877867698669\n              ],\n              [\n                0.8920624852180481,\n                0.32639023661613464,\n                0.9581875205039978,\n                0.6003658771514893\n              ],\n              [\n                0.4526093602180481,\n                0.32639023661613464,\n                0.9581875205039978,\n                0.7564877867698669\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.8809375166893005,\n                0.4027073085308075,\n                0.9907343983650208,\n                0.841853678226471\n              ],\n              [\n                0.8920624852180481,\n                0.32639023661613464,\n                0.9581875205039978,\n                0.6003658771514893\n              ],\n              [\n                0.8809375166893005,\n                0.32639023661613464,\n                0.9907343983650208,\n                0.841853678226471\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000556599.jpg",\n        "coco_id": 556599,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 509270,\n          "object_id": 1421619,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_556599_509270_1421619_437116_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 437116,\n          "object_id": 1421619,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.19718749821186066,\n                0.32374998927116394,\n                0.34507811069488525,\n                0.8383610844612122\n              ],\n              [\n                0.6720625162124634,\n                0.13819444179534912,\n                0.7452499866485596,\n                0.3573055565357208\n              ],\n              [\n                0.19718749821186066,\n                0.13819444179534912,\n                0.7452499866485596,\n                0.8383610844612122\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.6282343864440918,\n                0.06447222083806992,\n                0.7483124732971191,\n                0.6689721941947937\n              ],\n              [\n                0.6720625162124634,\n                0.13819444179534912,\n                0.7452499866485596,\n                0.3573055565357208\n              ],\n              [\n                0.6282343864440918,\n                0.06447222083806992,\n                0.7483124732971191,\n                0.6689721941947937\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000556830.jpg",\n        "coco_id": 556830,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 497809,\n          "object_id": 629343,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_556830_497809_629343_472188_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 472188,\n          "object_id": 629343,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.10442187637090683,\n                0.3791787326335907,\n                0.34375,\n                0.957656979560852\n              ],\n              [\n                0.48003125190734863,\n                0.04876811429858208,\n                0.6071875095367432,\n                0.26142510771751404\n              ],\n              [\n                0.10442187637090683,\n                0.04876811429858208,\n                0.6071875095367432,\n                0.957656979560852\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.5083437561988831,\n                0.24140097200870514,\n                0.9257187247276306,\n                0.8591787219047546\n              ],\n              [\n                0.48003125190734863,\n                0.04876811429858208,\n                0.6071875095367432,\n                0.26142510771751404\n              ],\n              [\n                0.48003125190734863,\n                0.04876811429858208,\n                0.9257187247276306,\n                0.8591787219047546\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000557746.jpg",\n        "coco_id": 557746,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "pizza",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1221816,\n          "object_id": 1072550,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_557746_1221816_1072550_1240788_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1240788,\n          "object_id": 1072550,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.4523593783378601,\n                0.11568749696016312,\n                0.8761249780654907,\n                0.6852083206176758\n              ],\n              [\n                0.5780937671661377,\n                0.6468124985694885,\n                0.901687502861023,\n                0.8580625057220459\n              ],\n              [\n                0.4523593783378601,\n                0.11568749696016312,\n                0.901687502861023,\n                0.8580625057220459\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.10134375095367432,\n                0.0664374977350235,\n                0.4712812602519989,\n                0.7443749904632568\n              ],\n              [\n                0.5780937671661377,\n                0.6468124985694885,\n                0.901687502861023,\n                0.8580625057220459\n              ],\n              [\n                0.10134375095367432,\n                0.0664374977350235,\n                0.901687502861023,\n                0.8580625057220459\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000557746.jpg",\n        "coco_id": 557746,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1240788,\n          "object_id": 323707,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_557746_1240788_323707_1221816_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1221816,\n          "object_id": 323707,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.4523593783378601,\n                0.11568749696016312,\n                0.8761249780654907,\n                0.6852083206176758\n              ],\n              [\n                0.2668437361717224,\n                0.2600833475589752,\n                0.31296876072883606,\n                0.3605208396911621\n              ],\n              [\n                0.2668437361717224,\n                0.11568749696016312,\n                0.8761249780654907,\n                0.6852083206176758\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.10134375095367432,\n                0.0664374977350235,\n                0.4712812602519989,\n                0.7443749904632568\n              ],\n              [\n                0.2668437361717224,\n                0.2600833475589752,\n                0.31296876072883606,\n                0.3605208396911621\n              ],\n              [\n                0.10134375095367432,\n                0.0664374977350235,\n                0.4712812602519989,\n                0.7443749904632568\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000557944.jpg",\n        "coco_id": 557944,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1233236,\n          "object_id": 629161,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_557944_1233236_629161_1258561_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1258561,\n          "object_id": 629161,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.16349999606609344,\n                0.500041663646698,\n                0.418156236410141,\n                0.7656041383743286\n              ],\n              [\n                0.6267499923706055,\n                0.589062511920929,\n                0.7284843921661377,\n                0.620312511920929\n              ],\n              [\n                0.16349999606609344,\n                0.500041663646698,\n                0.7284843921661377,\n                0.7656041383743286\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.6825781464576721,\n                0.46291667222976685,\n                0.8393281102180481,\n                0.7662916779518127\n              ],\n              [\n                0.6267499923706055,\n                0.589062511920929,\n                0.7284843921661377,\n                0.620312511920929\n              ],\n              [\n                0.6267499923706055,\n                0.46291667222976685,\n                0.8393281102180481,\n                0.7662916779518127\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000560384.jpg",\n        "coco_id": 560384,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 446634,\n          "object_id": 658697,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_560384_446634_658697_485127_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 485127,\n          "object_id": 658697,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.17735937237739563,\n                0.0044999998062849045,\n                0.28209376335144043,\n                0.19593749940395355\n              ],\n              [\n                0.5502656102180481,\n                0.4855625033378601,\n                0.5912187695503235,\n                0.590583324432373\n              ],\n              [\n                0.17735937237739563,\n                0.0044999998062849045,\n                0.5912187695503235,\n                0.590583324432373\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.40770313143730164,\n                0.43422916531562805,\n                0.5796093940734863,\n                0.7875624895095825\n              ],\n              [\n                0.5502656102180481,\n                0.4855625033378601,\n                0.5912187695503235,\n                0.590583324432373\n              ],\n              [\n                0.40770313143730164,\n                0.43422916531562805,\n                0.5912187695503235,\n                0.7875624895095825\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000561028.jpg",\n        "coco_id": 561028,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 424834,\n          "object_id": 656299,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_561028_424834_656299_509621_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 509621,\n          "object_id": 656299,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.4074375033378601,\n                0.1391666680574417,\n                0.6159843802452087,\n                0.8328750133514404\n              ],\n              [\n                0.40434375405311584,\n                0.5350833535194397,\n                0.4814531207084656,\n                0.6713958382606506\n              ],\n              [\n                0.40434375405311584,\n                0.1391666680574417,\n                0.6159843802452087,\n                0.8328750133514404\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.5425000190734863,\n                0.07472916692495346,\n                0.7720156311988831,\n                0.534375011920929\n              ],\n              [\n                0.40434375405311584,\n                0.5350833535194397,\n                0.4814531207084656,\n                0.6713958382606506\n              ],\n              [\n                0.40434375405311584,\n                0.07472916692495346,\n                0.7720156311988831,\n                0.6713958382606506\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000561713.jpg",\n        "coco_id": 561713,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1278536,\n          "object_id": 631323,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_561713_1278536_631323_1220292_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1220292,\n          "object_id": 631323,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.46639999747276306,\n                0.370788037776947,\n                0.732699990272522,\n                0.8494293689727783\n              ],\n              [\n                0.4903799891471863,\n                0.34798914194107056,\n                0.6844599843025208,\n                0.38165760040283203\n              ],\n              [\n                0.46639999747276306,\n                0.34798914194107056,\n                0.732699990272522,\n                0.8494293689727783\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.1782200038433075,\n                0.10788043588399887,\n                0.5000200271606445,\n                0.8074456453323364\n              ],\n              [\n                0.4903799891471863,\n                0.34798914194107056,\n                0.6844599843025208,\n                0.38165760040283203\n              ],\n              [\n                0.1782200038433075,\n                0.10788043588399887,\n                0.6844599843025208,\n                0.8074456453323364\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000564349.jpg",\n        "coco_id": 564349,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 444940,\n          "object_id": 1183087,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_564349_444940_1183087_477407_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 477407,\n          "object_id": 1183087,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.3109859228134155,\n                0.5126093626022339,\n                0.7843661904335022,\n                0.8791875243186951\n              ],\n              [\n                0.37258216738700867,\n                0.6309999823570251,\n                0.6897417902946472,\n                0.9251875281333923\n              ],\n              [\n                0.3109859228134155,\n                0.5126093626022339,\n                0.7843661904335022,\n                0.9251875281333923\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.44251173734664917,\n                0.165953129529953,\n                0.8360328674316406,\n                0.7752812504768372\n              ],\n              [\n                0.37258216738700867,\n                0.6309999823570251,\n                0.6897417902946472,\n                0.9251875281333923\n              ],\n              [\n                0.37258216738700867,\n                0.165953129529953,\n                0.8360328674316406,\n                0.9251875281333923\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000564349.jpg",\n        "coco_id": 564349,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "person",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 477407,\n          "object_id": 444940,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_564349_477407_444940_444940_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 444940,\n          "object_id": 444940,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a person",\n            "bounds": [\n              [\n                0.44251173734664917,\n                0.165953129529953,\n                0.8360328674316406,\n                0.7752812504768372\n              ],\n              [\n                0.3109859228134155,\n                0.5126093626022339,\n                0.7843661904335022,\n                0.8791875243186951\n              ],\n              [\n                0.3109859228134155,\n                0.165953129529953,\n                0.8360328674316406,\n                0.8791875243186951\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a person",\n            "bounds": [\n              [\n                0.3109859228134155,\n                0.5126093626022339,\n                0.7843661904335022,\n                0.8791875243186951\n              ],\n              [\n                0.3109859228134155,\n                0.5126093626022339,\n                0.7843661904335022,\n                0.8791875243186951\n              ],\n              [\n                0.3109859228134155,\n                0.5126093626022339,\n                0.7843661904335022,\n                0.8791875243186951\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000056456.jpg",\n        "coco_id": 56456,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1231918,\n          "object_id": 601967,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_56456_1231918_601967_1236069_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1236069,\n          "object_id": 601967,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.2511250078678131,\n                0.6179069876670837,\n                0.49118751287460327,\n                0.9989302158355713\n              ],\n              [\n                0.7961562275886536,\n                0.4169767498970032,\n                0.878781259059906,\n                0.5582790970802307\n              ],\n              [\n                0.2511250078678131,\n                0.4169767498970032,\n                0.878781259059906,\n                0.9989302158355713\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.5713124871253967,\n                0.18902325630187988,\n                0.904953122138977,\n                0.7006046772003174\n              ],\n              [\n                0.7961562275886536,\n                0.4169767498970032,\n                0.878781259059906,\n                0.5582790970802307\n              ],\n              [\n                0.5713124871253967,\n                0.18902325630187988,\n                0.904953122138977,\n                0.7006046772003174\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000566547.jpg",\n        "coco_id": 566547,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 441385,\n          "object_id": 1836362,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_566547_441385_1836362_549982_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 549982,\n          "object_id": 1836362,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.312687486410141,\n                0.20156541466712952,\n                0.47574999928474426,\n                0.9085046648979187\n              ],\n              [\n                0.29789063334465027,\n                0.31362149119377136,\n                0.41218748688697815,\n                0.6511214971542358\n              ],\n              [\n                0.29789063334465027,\n                0.20156541466712952,\n                0.47574999928474426,\n                0.9085046648979187\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.0,\n                0.15343457460403442,\n                0.10312499850988388,\n                0.9312149286270142\n              ],\n              [\n                0.29789063334465027,\n                0.31362149119377136,\n                0.41218748688697815,\n                0.6511214971542358\n              ],\n              [\n                0.0,\n                0.15343457460403442,\n                0.41218748688697815,\n                0.9312149286270142\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000566547.jpg",\n        "coco_id": 566547,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 441385,\n          "object_id": 1836362,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_566547_441385_1836362_549982_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 549982,\n          "object_id": 1836362,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.0,\n                0.15343457460403442,\n                0.10312499850988388,\n                0.9312149286270142\n              ],\n              [\n                0.29789063334465027,\n                0.31362149119377136,\n                0.41218748688697815,\n                0.6511214971542358\n              ],\n              [\n                0.0,\n                0.15343457460403442,\n                0.41218748688697815,\n                0.9312149286270142\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.312687486410141,\n                0.20156541466712952,\n                0.47574999928474426,\n                0.9085046648979187\n              ],\n              [\n                0.29789063334465027,\n                0.31362149119377136,\n                0.41218748688697815,\n                0.6511214971542358\n              ],\n              [\n                0.29789063334465027,\n                0.20156541466712952,\n                0.47574999928474426,\n                0.9085046648979187\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000567036.jpg",\n        "coco_id": 567036,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "person",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1277263,\n          "object_id": 1261040,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_567036_1277263_1261040_1245669_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1245669,\n          "object_id": 1261040,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.029906250536441803,\n                0.20496471226215363,\n                0.27220311760902405,\n                0.7049646973609924\n              ],\n              [\n                0.19539062678813934,\n                0.26491764187812805,\n                0.3167968690395355,\n                0.7273176312446594\n              ],\n              [\n                0.029906250536441803,\n                0.20496471226215363,\n                0.3167968690395355,\n                0.7273176312446594\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.2536875009536743,\n                0.3640470504760742,\n                0.5685625076293945,\n                0.8247293829917908\n              ],\n              [\n                0.19539062678813934,\n                0.26491764187812805,\n                0.3167968690395355,\n                0.7273176312446594\n              ],\n              [\n                0.19539062678813934,\n                0.26491764187812805,\n                0.5685625076293945,\n                0.8247293829917908\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000570768.jpg",\n        "coco_id": 570768,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1243462,\n          "object_id": 657145,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_570768_1243462_657145_447957_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 447957,\n          "object_id": 657145,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.05037499964237213,\n                0.35107991099357605,\n                0.21610938012599945,\n                0.7695464491844177\n              ],\n              [\n                0.5828593969345093,\n                0.29222461581230164,\n                0.7541093826293945,\n                0.40622031688690186\n              ],\n              [\n                0.05037499964237213,\n                0.29222461581230164,\n                0.7541093826293945,\n                0.7695464491844177\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.5137187242507935,\n                0.2606695592403412,\n                0.8762500286102295,\n                0.9303455948829651\n              ],\n              [\n                0.5828593969345093,\n                0.29222461581230164,\n                0.7541093826293945,\n                0.40622031688690186\n              ],\n              [\n                0.5137187242507935,\n                0.2606695592403412,\n                0.8762500286102295,\n                0.9303455948829651\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000570768.jpg",\n        "coco_id": 570768,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 447957,\n          "object_id": 375387,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_570768_447957_375387_1243462_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1243462,\n          "object_id": 375387,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.5137187242507935,\n                0.2606695592403412,\n                0.8762500286102295,\n                0.9303455948829651\n              ],\n              [\n                0.03285937383770943,\n                0.4625917971134186,\n                0.17745313048362732,\n                0.7519870400428772\n              ],\n              [\n                0.03285937383770943,\n                0.2606695592403412,\n                0.8762500286102295,\n                0.9303455948829651\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.05037499964237213,\n                0.35107991099357605,\n                0.21610938012599945,\n                0.7695464491844177\n              ],\n              [\n                0.03285937383770943,\n                0.4625917971134186,\n                0.17745313048362732,\n                0.7519870400428772\n              ],\n              [\n                0.03285937383770943,\n                0.35107991099357605,\n                0.21610938012599945,\n                0.7695464491844177\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000571799.jpg",\n        "coco_id": 571799,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "knife",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 431213,\n          "object_id": 698570,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_571799_431213_698570_428214_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 428214,\n          "object_id": 698570,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.0,\n                0.0,\n                0.47865626215934753,\n                0.882770836353302\n              ],\n              [\n                0.32490625977516174,\n                0.7172291874885559,\n                0.5558906197547913,\n                0.8251458406448364\n              ],\n              [\n                0.0,\n                0.0,\n                0.5558906197547913,\n                0.882770836353302\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.43623438477516174,\n                0.14114584028720856,\n                0.7818750143051147,\n                0.7787500023841858\n              ],\n              [\n                0.32490625977516174,\n                0.7172291874885559,\n                0.5558906197547913,\n                0.8251458406448364\n              ],\n              [\n                0.32490625977516174,\n                0.14114584028720856,\n                0.7818750143051147,\n                0.8251458406448364\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000578345.jpg",\n        "coco_id": 578345,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1732916,\n          "object_id": 365228,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_578345_1732916_365228_554661_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 554661,\n          "object_id": 365228,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.5812031030654907,\n                0.4464375078678131,\n                0.6834999918937683,\n                0.6895833611488342\n              ],\n              [\n                0.3438437581062317,\n                0.5400624871253967,\n                0.542062520980835,\n                0.8476250171661377\n              ],\n              [\n                0.3438437581062317,\n                0.4464375078678131,\n                0.6834999918937683,\n                0.8476250171661377\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.3726406395435333,\n                0.37327083945274353,\n                0.5237656235694885,\n                0.741937518119812\n              ],\n              [\n                0.3438437581062317,\n                0.5400624871253967,\n                0.542062520980835,\n                0.8476250171661377\n              ],\n              [\n                0.3438437581062317,\n                0.37327083945274353,\n                0.542062520980835,\n                0.8476250171661377\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000578345.jpg",\n        "coco_id": 578345,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1732916,\n          "object_id": 365228,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_578345_1732916_365228_554661_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 554661,\n          "object_id": 365228,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.5812031030654907,\n                0.4464375078678131,\n                0.6834999918937683,\n                0.6895833611488342\n              ],\n              [\n                0.3438437581062317,\n                0.5400624871253967,\n                0.542062520980835,\n                0.8476250171661377\n              ],\n              [\n                0.3438437581062317,\n                0.4464375078678131,\n                0.6834999918937683,\n                0.8476250171661377\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.3726406395435333,\n                0.37327083945274353,\n                0.5237656235694885,\n                0.741937518119812\n              ],\n              [\n                0.3438437581062317,\n                0.5400624871253967,\n                0.542062520980835,\n                0.8476250171661377\n              ],\n              [\n                0.3438437581062317,\n                0.37327083945274353,\n                0.542062520980835,\n                0.8476250171661377\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000579729.jpg",\n        "coco_id": 579729,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 201075,\n          "object_id": 127989,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_579729_201075_127989_196525_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 196525,\n          "object_id": 127989,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.4985312521457672,\n                0.1612880527973175,\n                0.6567343473434448,\n                0.7228103280067444\n              ],\n              [\n                0.6930624842643738,\n                0.47306790947914124,\n                0.934640645980835,\n                0.7769320607185364\n              ],\n              [\n                0.4985312521457672,\n                0.1612880527973175,\n                0.934640645980835,\n                0.7769320607185364\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.7181562781333923,\n                0.34156909584999084,\n                0.9190624952316284,\n                0.6898828744888306\n              ],\n              [\n                0.6930624842643738,\n                0.47306790947914124,\n                0.934640645980835,\n                0.7769320607185364\n              ],\n              [\n                0.6930624842643738,\n                0.34156909584999084,\n                0.934640645980835,\n                0.7769320607185364\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000581657.jpg",\n        "coco_id": 581657,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 556799,\n          "object_id": 322715,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_581657_556799_322715_1758997_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1758997,\n          "object_id": 322715,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.7910208106040955,\n                0.0036718749906867743,\n                1.0,\n                0.3769843876361847\n              ],\n              [\n                0.5585416555404663,\n                0.5365312695503235,\n                0.7080416679382324,\n                0.626046895980835\n              ],\n              [\n                0.5585416555404663,\n                0.0036718749906867743,\n                1.0,\n                0.626046895980835\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.5425208210945129,\n                0.5907031297683716,\n                1.0,\n                0.9481250047683716\n              ],\n              [\n                0.5585416555404663,\n                0.5365312695503235,\n                0.7080416679382324,\n                0.626046895980835\n              ],\n              [\n                0.5425208210945129,\n                0.5365312695503235,\n                1.0,\n                0.9481250047683716\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000062872.jpg",\n        "coco_id": 62872,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 531891,\n          "object_id": 605469,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_62872_531891_605469_1707150_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1707150,\n          "object_id": 605469,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.7196962833404541,\n                0.39457812905311584,\n                0.9063551425933838,\n                0.7468125224113464\n              ],\n              [\n                0.36235982179641724,\n                0.27256250381469727,\n                0.4637850522994995,\n                0.3465000092983246\n              ],\n              [\n                0.36235982179641724,\n                0.27256250381469727,\n                0.9063551425933838,\n                0.7468125224113464\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.09158878773450851,\n                0.3304218649864197,\n                0.5613083839416504,\n                0.7840625047683716\n              ],\n              [\n                0.36235982179641724,\n                0.27256250381469727,\n                0.4637850522994995,\n                0.3465000092983246\n              ],\n              [\n                0.09158878773450851,\n                0.27256250381469727,\n                0.5613083839416504,\n                0.7840625047683716\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000006780.jpg",\n        "coco_id": 6780,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 466407,\n          "object_id": 1448219,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_6780_466407_1448219_492335_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 492335,\n          "object_id": 1448219,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.3998124897480011,\n                0.6316713094711304,\n                0.6865468621253967,\n                0.8523119688034058\n              ],\n              [\n                0.4970156252384186,\n                0.6776044368743896,\n                0.5195156335830688,\n                0.7403063774108887\n              ],\n              [\n                0.3998124897480011,\n                0.6316713094711304,\n                0.6865468621253967,\n                0.8523119688034058\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.29231250286102295,\n                0.28231197595596313,\n                0.4070156216621399,\n                0.7598885893821716\n              ],\n              [\n                0.4970156252384186,\n                0.6776044368743896,\n                0.5195156335830688,\n                0.7403063774108887\n              ],\n              [\n                0.29231250286102295,\n                0.28231197595596313,\n                0.5195156335830688,\n                0.7598885893821716\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000078614.jpg",\n        "coco_id": 78614,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tie",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 467356,\n          "object_id": 294341,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_78614_467356_294341_456734_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 456734,\n          "object_id": 294341,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tie",\n            "bounds": [\n              [\n                0.49987998604774475,\n                0.33482035994529724,\n                0.8646399974822998,\n                0.988772451877594\n              ],\n              [\n                0.20027999579906464,\n                0.4330538809299469,\n                0.4679200053215027,\n                0.8852993845939636\n              ],\n              [\n                0.20027999579906464,\n                0.33482035994529724,\n                0.8646399974822998,\n                0.988772451877594\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tie",\n            "bounds": [\n              [\n                0.06710000336170197,\n                0.15643712878227234,\n                0.5191799998283386,\n                1.0\n              ],\n              [\n                0.20027999579906464,\n                0.4330538809299469,\n                0.4679200053215027,\n                0.8852993845939636\n              ],\n              [\n                0.06710000336170197,\n                0.15643712878227234,\n                0.5191799998283386,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000079852.jpg",\n        "coco_id": 79852,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skateboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 490867,\n          "object_id": 639354,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_79852_490867_639354_473267_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 473267,\n          "object_id": 639354,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.35096874833106995,\n                0.38216784596443176,\n                0.8360000252723694,\n                0.8945221304893494\n              ],\n              [\n                0.3604687452316284,\n                0.8685081601142883,\n                0.6587812304496765,\n                0.9573193192481995\n              ],\n              [\n                0.35096874833106995,\n                0.38216784596443176,\n                0.8360000252723694,\n                0.9573193192481995\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.17323437333106995,\n                0.006736596580594778,\n                0.38109374046325684,\n                0.7258508205413818\n              ],\n              [\n                0.3604687452316284,\n                0.8685081601142883,\n                0.6587812304496765,\n                0.9573193192481995\n              ],\n              [\n                0.17323437333106995,\n                0.006736596580594778,\n                0.6587812304496765,\n                0.9573193192481995\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000080553.jpg",\n        "coco_id": 80553,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 452775,\n          "object_id": 1841790,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_80553_452775_1841790_493787_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 493787,\n          "object_id": 1841790,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.300656259059906,\n                0.40772300958633423,\n                0.42210936546325684,\n                0.8224647641181946\n              ],\n              [\n                0.349609375,\n                0.6357980966567993,\n                0.4331406354904175,\n                0.8246009349822998\n              ],\n              [\n                0.300656259059906,\n                0.40772300958633423,\n                0.4331406354904175,\n                0.8246009349822998\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a suitcase",\n            "bounds": [\n              [\n                0.562125027179718,\n                0.3942722976207733,\n                0.6538281440734863,\n                0.7799295783042908\n              ],\n              [\n                0.349609375,\n                0.6357980966567993,\n                0.4331406354904175,\n                0.8246009349822998\n              ],\n              [\n                0.349609375,\n                0.3942722976207733,\n                0.6538281440734863,\n                0.8246009349822998\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000008571.jpg",\n        "coco_id": 8571,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 431470,\n          "object_id": 658128,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_8571_431470_658128_516015_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 516015,\n          "object_id": 658128,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.4410156309604645,\n                0.2755269408226013,\n                0.5824687480926514,\n                0.8932318687438965\n              ],\n              [\n                0.4524218738079071,\n                0.5541452169418335,\n                0.6204531192779541,\n                0.6535128951072693\n              ],\n              [\n                0.4410156309604645,\n                0.2755269408226013,\n                0.6204531192779541,\n                0.8932318687438965\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.2214062511920929,\n                0.2780327796936035,\n                0.3440625071525574,\n                0.8722013831138611\n              ],\n              [\n                0.4524218738079071,\n                0.5541452169418335,\n                0.6204531192779541,\n                0.6535128951072693\n              ],\n              [\n                0.2214062511920929,\n                0.2780327796936035,\n                0.6204531192779541,\n                0.8722013831138611\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000008571.jpg",\n        "coco_id": 8571,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 511294,\n          "object_id": 1481777,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_8571_511294_1481777_516015_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 516015,\n          "object_id": 1481777,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.3138124942779541,\n                0.3055269420146942,\n                0.4660624861717224,\n                0.8525761365890503\n              ],\n              [\n                0.39771875739097595,\n                0.5365339517593384,\n                0.4351406395435333,\n                0.7431849837303162\n              ],\n              [\n                0.3138124942779541,\n                0.3055269420146942,\n                0.4660624861717224,\n                0.8525761365890503\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.2214062511920929,\n                0.2780327796936035,\n                0.3440625071525574,\n                0.8722013831138611\n              ],\n              [\n                0.39771875739097595,\n                0.5365339517593384,\n                0.4351406395435333,\n                0.7431849837303162\n              ],\n              [\n                0.2214062511920929,\n                0.2780327796936035,\n                0.4351406395435333,\n                0.8722013831138611\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000087027.jpg",\n        "coco_id": 87027,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 435154,\n          "object_id": 1844707,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_87027_435154_1844707_452636_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 452636,\n          "object_id": 1844707,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.6495937705039978,\n                0.05774080380797386,\n                0.9703750014305115,\n                0.8869526982307434\n              ],\n              [\n                0.7332500219345093,\n                0.09241681545972824,\n                0.8578593730926514,\n                0.20208406448364258\n              ],\n              [\n                0.6495937705039978,\n                0.05774080380797386,\n                0.9703750014305115,\n                0.8869526982307434\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.008015625178813934,\n                0.30336251854896545,\n                0.16039063036441803,\n                0.867408037185669\n              ],\n              [\n                0.7332500219345093,\n                0.09241681545972824,\n                0.8578593730926514,\n                0.20208406448364258\n              ],\n              [\n                0.008015625178813934,\n                0.09241681545972824,\n                0.8578593730926514,\n                0.867408037185669\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000089248.jpg",\n        "coco_id": 89248,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "mouse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 193033,\n          "object_id": 1107009,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_89248_193033_1107009_212839_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 212839,\n          "object_id": 1107009,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a mouse",\n            "bounds": [\n              [\n                0.5907187461853027,\n                0.015737704932689667,\n                1.0,\n                0.997751772403717\n              ],\n              [\n                0.5900156497955322,\n                0.6994613409042358,\n                0.6669999957084656,\n                0.7675409913063049\n              ],\n              [\n                0.5900156497955322,\n                0.015737704932689667,\n                1.0,\n                0.997751772403717\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a mouse",\n            "bounds": [\n              [\n                0.001500000013038516,\n                0.2801639437675476,\n                0.4128749966621399,\n                0.9864403009414673\n              ],\n              [\n                0.5900156497955322,\n                0.6994613409042358,\n                0.6669999957084656,\n                0.7675409913063049\n              ],\n              [\n                0.001500000013038516,\n                0.2801639437675476,\n                0.6669999957084656,\n                0.9864403009414673\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000089390.jpg",\n        "coco_id": 89390,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "car",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1271940,\n          "object_id": 1779148,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_89390_1271940_1779148_1269735_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1269735,\n          "object_id": 1779148,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a car",\n            "bounds": [\n              [\n                0.2624531388282776,\n                0.34562501311302185,\n                0.4034687578678131,\n                0.5270416736602783\n              ],\n              [\n                0.2947031259536743,\n                0.3764791786670685,\n                1.0,\n                0.8399999737739563\n              ],\n              [\n                0.2624531388282776,\n                0.34562501311302185,\n                1.0,\n                0.8399999737739563\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a car",\n            "bounds": [\n              [\n                0.6900625228881836,\n                0.38064584136009216,\n                0.8231406211853027,\n                0.5454583168029785\n              ],\n              [\n                0.2947031259536743,\n                0.3764791786670685,\n                1.0,\n                0.8399999737739563\n              ],\n              [\n                0.2947031259536743,\n                0.3764791786670685,\n                1.0,\n                0.8399999737739563\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000089390.jpg",\n        "coco_id": 89390,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1271940,\n          "object_id": 323275,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_89390_1271940_323275_1269735_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1269735,\n          "object_id": 323275,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.2624531388282776,\n                0.34562501311302185,\n                0.4034687578678131,\n                0.5270416736602783\n              ],\n              [\n                0.7591249942779541,\n                0.4176458418369293,\n                0.7941718697547913,\n                0.4648958444595337\n              ],\n              [\n                0.2624531388282776,\n                0.34562501311302185,\n                0.7941718697547913,\n                0.5270416736602783\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.6900625228881836,\n                0.38064584136009216,\n                0.8231406211853027,\n                0.5454583168029785\n              ],\n              [\n                0.7591249942779541,\n                0.4176458418369293,\n                0.7941718697547913,\n                0.4648958444595337\n              ],\n              [\n                0.6900625228881836,\n                0.38064584136009216,\n                0.8231406211853027,\n                0.5454583168029785\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000089390.jpg",\n        "coco_id": 89390,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1271940,\n          "object_id": 323275,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_89390_1271940_323275_1269735_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1269735,\n          "object_id": 323275,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.6900625228881836,\n                0.38064584136009216,\n                0.8231406211853027,\n                0.5454583168029785\n              ],\n              [\n                0.7591249942779541,\n                0.4176458418369293,\n                0.7941718697547913,\n                0.4648958444595337\n              ],\n              [\n                0.6900625228881836,\n                0.38064584136009216,\n                0.8231406211853027,\n                0.5454583168029785\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.2624531388282776,\n                0.34562501311302185,\n                0.4034687578678131,\n                0.5270416736602783\n              ],\n              [\n                0.7591249942779541,\n                0.4176458418369293,\n                0.7941718697547913,\n                0.4648958444595337\n              ],\n              [\n                0.2624531388282776,\n                0.34562501311302185,\n                0.7941718697547913,\n                0.5270416736602783\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000089788.jpg",\n        "coco_id": 89788,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 477606,\n          "object_id": 1878500,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_89788_477606_1878500_430394_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 430394,\n          "object_id": 1878500,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.004999999888241291,\n                0.34239062666893005,\n                0.5240208506584167,\n                0.9978593587875366\n              ],\n              [\n                0.5078333616256714,\n                0.6235312223434448,\n                0.6514999866485596,\n                0.7392500042915344\n              ],\n              [\n                0.004999999888241291,\n                0.34239062666893005,\n                0.6514999866485596,\n                0.9978593587875366\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.051229167729616165,\n                0.148375004529953,\n                0.9243958592414856,\n                0.8075781464576721\n              ],\n              [\n                0.5078333616256714,\n                0.6235312223434448,\n                0.6514999866485596,\n                0.7392500042915344\n              ],\n              [\n                0.051229167729616165,\n                0.148375004529953,\n                0.9243958592414856,\n                0.8075781464576721\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000092098.jpg",\n        "coco_id": 92098,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 541550,\n          "object_id": 1180718,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_92098_541550_1180718_426896_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 426896,\n          "object_id": 1180718,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.425968736410141,\n                0.304749459028244,\n                0.5695624947547913,\n                0.8023093938827515\n              ],\n              [\n                0.5246250033378601,\n                0.5901525020599365,\n                0.6168749928474426,\n                0.739389955997467\n              ],\n              [\n                0.425968736410141,\n                0.304749459028244,\n                0.6168749928474426,\n                0.8023093938827515\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.5366093516349792,\n                0.26714596152305603,\n                0.6584062576293945,\n                0.8396949768066406\n              ],\n              [\n                0.5246250033378601,\n                0.5901525020599365,\n                0.6168749928474426,\n                0.739389955997467\n              ],\n              [\n                0.5246250033378601,\n                0.26714596152305603,\n                0.6584062576293945,\n                0.8396949768066406\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000009669.jpg",\n        "coco_id": 9669,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1709645,\n          "object_id": 321902,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_9669_1709645_321902_1700706_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1700706,\n          "object_id": 321902,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.12070093303918839,\n                0.17489062249660492,\n                0.9689486026763916,\n                0.7870000004768372\n              ],\n              [\n                0.48649531602859497,\n                0.5219374895095825,\n                0.6251401901245117,\n                0.5786093473434448\n              ],\n              [\n                0.12070093303918839,\n                0.17489062249660492,\n                0.9689486026763916,\n                0.7870000004768372\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.17647196352481842,\n                0.7288437485694885,\n                0.6961682438850403,\n                1.0\n              ],\n              [\n                0.48649531602859497,\n                0.5219374895095825,\n                0.6251401901245117,\n                0.5786093473434448\n              ],\n              [\n                0.17647196352481842,\n                0.5219374895095825,\n                0.6961682438850403,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000097632.jpg",\n        "coco_id": 97632,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 491061,\n          "object_id": 657959,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_97632_491061_657959_508713_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 508713,\n          "object_id": 657959,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.020984375849366188,\n                0.1659250557422638,\n                0.1034531220793724,\n                0.6288290619850159\n              ],\n              [\n                0.3473437428474426,\n                0.4473067820072174,\n                0.39125001430511475,\n                0.590585470199585\n              ],\n              [\n                0.020984375849366188,\n                0.1659250557422638,\n                0.39125001430511475,\n                0.6288290619850159\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.3447187542915344,\n                0.3669086694717407,\n                0.5561249852180481,\n                0.9257845282554626\n              ],\n              [\n                0.3473437428474426,\n                0.4473067820072174,\n                0.39125001430511475,\n                0.590585470199585\n              ],\n              [\n                0.3447187542915344,\n                0.3669086694717407,\n                0.5561249852180481,\n                0.9257845282554626\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000099293.jpg",\n        "coco_id": 99293,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "donut",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 495749,\n          "object_id": 1082294,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_99293_495749_1082294_487648_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 487648,\n          "object_id": 1082294,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a donut",\n            "bounds": [\n              [\n                0.11756250262260437,\n                0.24944444000720978,\n                0.42598438262939453,\n                0.9842777848243713\n              ],\n              [\n                0.5966406464576721,\n                0.7012222409248352,\n                0.6513749957084656,\n                0.7916944622993469\n              ],\n              [\n                0.11756250262260437,\n                0.24944444000720978,\n                0.6513749957084656,\n                0.9842777848243713\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a donut",\n            "bounds": [\n              [\n                0.5788593888282776,\n                0.2162500023841858,\n                0.9991562366485596,\n                0.984333336353302\n              ],\n              [\n                0.5966406464576721,\n                0.7012222409248352,\n                0.6513749957084656,\n                0.7916944622993469\n              ],\n              [\n                0.5788593888282776,\n                0.2162500023841858,\n                0.9991562366485596,\n                0.984333336353302\n              ]\n            ]\n          }\n        ]\n      }\n    ],\n    "calibration": [\n      {\n        "filename": "COCO_train2014_000000134648.jpg",\n        "coco_id": 134648,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 439751,\n          "object_id": 1416162,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_134648_439751_1416162_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 439751,\n          "object_id": 1416162,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.33550000190734863,\n                0.4596691131591797,\n                0.5694375038146973,\n                0.6646690964698792\n              ],\n              [\n                0.5306666493415833,\n                0.5534191131591797,\n                0.6581458449363708,\n                0.7872058749198914\n              ],\n              [\n                0.33550000190734863,\n                0.4596691131591797,\n                0.6581458449363708,\n                0.7872058749198914\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.33550000190734863,\n                0.4596691131591797,\n                0.5694375038146973,\n                0.6646690964698792\n              ],\n              [\n                0.5306666493415833,\n                0.5534191131591797,\n                0.6581458449363708,\n                0.7872058749198914\n              ],\n              [\n                0.33550000190734863,\n                0.4596691131591797,\n                0.6581458449363708,\n                0.7872058749198914\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000014873.jpg",\n        "coco_id": 14873,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1284477,\n          "object_id": 1172516,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_14873_1284477_1172516_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1284477,\n          "object_id": 1172516,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.6698125004768372,\n                0.4712916612625122,\n                0.7501875162124634,\n                0.6712083220481873\n              ],\n              [\n                0.6967968940734863,\n                0.5112916827201843,\n                0.7297343611717224,\n                0.5739374756813049\n              ],\n              [\n                0.6698125004768372,\n                0.4712916612625122,\n                0.7501875162124634,\n                0.6712083220481873\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.6698125004768372,\n                0.4712916612625122,\n                0.7501875162124634,\n                0.6712083220481873\n              ],\n              [\n                0.6967968940734863,\n                0.5112916827201843,\n                0.7297343611717224,\n                0.5739374756813049\n              ],\n              [\n                0.6698125004768372,\n                0.4712916612625122,\n                0.7501875162124634,\n                0.6712083220481873\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000014873.jpg",\n        "coco_id": 14873,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2020813,\n          "object_id": 1163596,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_14873_2020813_1163596_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 2020813,\n          "object_id": 1163596,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.2779531180858612,\n                0.49002084136009216,\n                0.3393281102180481,\n                0.7013124823570251\n              ],\n              [\n                0.3012968897819519,\n                0.5220624804496765,\n                0.3409687578678131,\n                0.5870833396911621\n              ],\n              [\n                0.2779531180858612,\n                0.49002084136009216,\n                0.3409687578678131,\n                0.7013124823570251\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.2779531180858612,\n                0.49002084136009216,\n                0.3393281102180481,\n                0.7013124823570251\n              ],\n              [\n                0.3012968897819519,\n                0.5220624804496765,\n                0.3409687578678131,\n                0.5870833396911621\n              ],\n              [\n                0.2779531180858612,\n                0.49002084136009216,\n                0.3409687578678131,\n                0.7013124823570251\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000203809.jpg",\n        "coco_id": 203809,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 485667,\n          "object_id": 1838190,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_203809_485667_1838190_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 485667,\n          "object_id": 1838190,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.5514758825302124,\n                0.3398599922657013,\n                0.7981927990913391,\n                0.9114000201225281\n              ],\n              [\n                0.649216890335083,\n                0.44082000851631165,\n                0.8434939980506897,\n                0.7061200141906738\n              ],\n              [\n                0.5514758825302124,\n                0.3398599922657013,\n                0.8434939980506897,\n                0.9114000201225281\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.5514758825302124,\n                0.3398599922657013,\n                0.7981927990913391,\n                0.9114000201225281\n              ],\n              [\n                0.649216890335083,\n                0.44082000851631165,\n                0.8434939980506897,\n                0.7061200141906738\n              ],\n              [\n                0.5514758825302124,\n                0.3398599922657013,\n                0.8434939980506897,\n                0.9114000201225281\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000221202.jpg",\n        "coco_id": 221202,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 205330,\n          "object_id": 59568,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_221202_205330_59568_sit_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 205330,\n          "object_id": 59568,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.3952343761920929,\n                0.42478349804878235,\n                0.4674062430858612,\n                0.631134033203125\n              ],\n              [\n                0.4090625047683716,\n                0.5397937893867493,\n                0.4937968850135803,\n                0.7247216701507568\n              ],\n              [\n                0.3952343761920929,\n                0.42478349804878235,\n                0.4937968850135803,\n                0.7247216701507568\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.3952343761920929,\n                0.42478349804878235,\n                0.4674062430858612,\n                0.631134033203125\n              ],\n              [\n                0.4090625047683716,\n                0.5397937893867493,\n                0.4937968850135803,\n                0.7247216701507568\n              ],\n              [\n                0.3952343761920929,\n                0.42478349804878235,\n                0.4937968850135803,\n                0.7247216701507568\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000224508.jpg",\n        "coco_id": 224508,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 196169,\n          "object_id": 58177,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_224508_196169_58177_ride_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 196169,\n          "object_id": 58177,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.6844687461853027,\n                0.3150467276573181,\n                0.764968752861023,\n                0.5955139994621277\n              ],\n              [\n                0.5902343988418579,\n                0.3894859850406647,\n                0.8990781307220459,\n                0.746425211429596\n              ],\n              [\n                0.5902343988418579,\n                0.3150467276573181,\n                0.8990781307220459,\n                0.746425211429596\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.6844687461853027,\n                0.3150467276573181,\n                0.764968752861023,\n                0.5955139994621277\n              ],\n              [\n                0.5902343988418579,\n                0.3894859850406647,\n                0.8990781307220459,\n                0.746425211429596\n              ],\n              [\n                0.5902343988418579,\n                0.3150467276573181,\n                0.8990781307220459,\n                0.746425211429596\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000229193.jpg",\n        "coco_id": 229193,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "pizza",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 472104,\n          "object_id": 1074854,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_229193_472104_1074854_hold_eat",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 472104,\n          "object_id": 1074854,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a pizza",\n            "bounds": [\n              [\n                0.10727272927761078,\n                0.0,\n                0.9185780882835388,\n                0.9887655973434448\n              ],\n              [\n                0.13489510118961334,\n                0.7625156044960022,\n                0.5945920944213867,\n                0.9760156273841858\n              ],\n              [\n                0.10727272927761078,\n                0.0,\n                0.9185780882835388,\n                0.9887655973434448\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.10727272927761078,\n                0.0,\n                0.9185780882835388,\n                0.9887655973434448\n              ],\n              [\n                0.13489510118961334,\n                0.7625156044960022,\n                0.5945920944213867,\n                0.9760156273841858\n              ],\n              [\n                0.10727272927761078,\n                0.0,\n                0.9185780882835388,\n                0.9887655973434448\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000251696.jpg",\n        "coco_id": 251696,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 423468,\n          "object_id": 326265,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_251696_423468_326265_carry_talk_on_phone",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 423468,\n          "object_id": 326265,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.04214062541723251,\n                0.2383541613817215,\n                0.37415623664855957,\n                0.9799166917800903\n              ],\n              [\n                0.15198437869548798,\n                0.45399999618530273,\n                0.1847812533378601,\n                0.5068125128746033\n              ],\n              [\n                0.04214062541723251,\n                0.2383541613817215,\n                0.37415623664855957,\n                0.9799166917800903\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.04214062541723251,\n                0.2383541613817215,\n                0.37415623664855957,\n                0.9799166917800903\n              ],\n              [\n                0.15198437869548798,\n                0.45399999618530273,\n                0.1847812533378601,\n                0.5068125128746033\n              ],\n              [\n                0.04214062541723251,\n                0.2383541613817215,\n                0.37415623664855957,\n                0.9799166917800903\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000255983.jpg",\n        "coco_id": 255983,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1281068,\n          "object_id": 1177113,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_255983_1281068_1177113_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1281068,\n          "object_id": 1177113,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.8483906388282776,\n                0.27676814794540405,\n                1.0,\n                0.9996019005775452\n              ],\n              [\n                0.8628437519073486,\n                0.4332318603992462,\n                0.9169843792915344,\n                0.7278454303741455\n              ],\n              [\n                0.8483906388282776,\n                0.27676814794540405,\n                1.0,\n                0.9996019005775452\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.8483906388282776,\n                0.27676814794540405,\n                1.0,\n                0.9996019005775452\n              ],\n              [\n                0.8628437519073486,\n                0.4332318603992462,\n                0.9169843792915344,\n                0.7278454303741455\n              ],\n              [\n                0.8483906388282776,\n                0.27676814794540405,\n                1.0,\n                0.9996019005775452\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000029456.jpg",\n        "coco_id": 29456,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1225449,\n          "object_id": 1415684,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_29456_1225449_1415684_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1225449,\n          "object_id": 1415684,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.4297812581062317,\n                0.38875293731689453,\n                0.6595781445503235,\n                1.0\n              ],\n              [\n                0.46284374594688416,\n                0.4990588128566742,\n                0.5969843864440918,\n                0.6926588416099548\n              ],\n              [\n                0.4297812581062317,\n                0.38875293731689453,\n                0.6595781445503235,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.4297812581062317,\n                0.38875293731689453,\n                0.6595781445503235,\n                1.0\n              ],\n              [\n                0.46284374594688416,\n                0.4990588128566742,\n                0.5969843864440918,\n                0.6926588416099548\n              ],\n              [\n                0.4297812581062317,\n                0.38875293731689453,\n                0.6595781445503235,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000324823.jpg",\n        "coco_id": 324823,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 435827,\n          "object_id": 321404,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_324823_435827_321404_talk_on_phone_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 435827,\n          "object_id": 321404,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.1147887334227562,\n                0.2921406328678131,\n                0.6076995134353638,\n                0.6966249942779541\n              ],\n              [\n                0.26647886633872986,\n                0.3555000126361847,\n                0.297488272190094,\n                0.40146875381469727\n              ],\n              [\n                0.1147887334227562,\n                0.2921406328678131,\n                0.6076995134353638,\n                0.6966249942779541\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.1147887334227562,\n                0.2921406328678131,\n                0.6076995134353638,\n                0.6966249942779541\n              ],\n              [\n                0.26647886633872986,\n                0.3555000126361847,\n                0.297488272190094,\n                0.40146875381469727\n              ],\n              [\n                0.1147887334227562,\n                0.2921406328678131,\n                0.6076995134353638,\n                0.6966249942779541\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000327110.jpg",\n        "coco_id": 327110,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1205336,\n          "object_id": 329566,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_327110_1205336_329566_carry_talk_on_phone",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1205336,\n          "object_id": 329566,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.4677799940109253,\n                0.26442351937294006,\n                0.7217400074005127,\n                0.9956706166267395\n              ],\n              [\n                0.5027599930763245,\n                0.5267529487609863,\n                0.535260021686554,\n                0.5522588491439819\n              ],\n              [\n                0.4677799940109253,\n                0.26442351937294006,\n                0.7217400074005127,\n                0.9956706166267395\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.4677799940109253,\n                0.26442351937294006,\n                0.7217400074005127,\n                0.9956706166267395\n              ],\n              [\n                0.5027599930763245,\n                0.5267529487609863,\n                0.535260021686554,\n                0.5522588491439819\n              ],\n              [\n                0.4677799940109253,\n                0.26442351937294006,\n                0.7217400074005127,\n                0.9956706166267395\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000341429.jpg",\n        "coco_id": 341429,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "pizza",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 190150,\n          "object_id": 1569733,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_341429_190150_1569733_eat_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 190150,\n          "object_id": 1569733,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a pizza",\n            "bounds": [\n              [\n                0.003247058717533946,\n                0.10107812285423279,\n                0.9618117809295654,\n                0.9569844007492065\n              ],\n              [\n                0.3499999940395355,\n                0.7987968921661377,\n                0.6093176603317261,\n                0.8770624995231628\n              ],\n              [\n                0.003247058717533946,\n                0.10107812285423279,\n                0.9618117809295654,\n                0.9569844007492065\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.003247058717533946,\n                0.10107812285423279,\n                0.9618117809295654,\n                0.9569844007492065\n              ],\n              [\n                0.3499999940395355,\n                0.7987968921661377,\n                0.6093176603317261,\n                0.8770624995231628\n              ],\n              [\n                0.003247058717533946,\n                0.10107812285423279,\n                0.9618117809295654,\n                0.9569844007492065\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000343215.jpg",\n        "coco_id": 343215,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 530048,\n          "object_id": 1172788,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_343215_530048_1172788_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 530048,\n          "object_id": 1172788,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.46485936641693115,\n                0.5948134064674377,\n                0.6511874794960022,\n                0.9229104518890381\n              ],\n              [\n                0.5416093468666077,\n                0.7089738845825195,\n                0.6004531383514404,\n                0.7918283343315125\n              ],\n              [\n                0.46485936641693115,\n                0.5948134064674377,\n                0.6511874794960022,\n                0.9229104518890381\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.46485936641693115,\n                0.5948134064674377,\n                0.6511874794960022,\n                0.9229104518890381\n              ],\n              [\n                0.5416093468666077,\n                0.7089738845825195,\n                0.6004531383514404,\n                0.7918283343315125\n              ],\n              [\n                0.46485936641693115,\n                0.5948134064674377,\n                0.6511874794960022,\n                0.9229104518890381\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000405361.jpg",\n        "coco_id": 405361,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 211141,\n          "object_id": 1181400,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_405361_211141_1181400_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 211141,\n          "object_id": 1181400,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.4372187554836273,\n                0.2216981202363968,\n                0.5765312314033508,\n                0.834669828414917\n              ],\n              [\n                0.5374374985694885,\n                0.350566029548645,\n                0.5791249871253967,\n                0.5325943231582642\n              ],\n              [\n                0.4372187554836273,\n                0.2216981202363968,\n                0.5791249871253967,\n                0.834669828414917\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.4372187554836273,\n                0.2216981202363968,\n                0.5765312314033508,\n                0.834669828414917\n              ],\n              [\n                0.5374374985694885,\n                0.350566029548645,\n                0.5791249871253967,\n                0.5325943231582642\n              ],\n              [\n                0.4372187554836273,\n                0.2216981202363968,\n                0.5791249871253967,\n                0.834669828414917\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000416705.jpg",\n        "coco_id": 416705,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "pizza",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1229237,\n          "object_id": 1072881,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_416705_1229237_1072881_eat_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1229237,\n          "object_id": 1072881,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.4634583294391632,\n                0.001234375056810677,\n                1.0,\n                0.2026406228542328\n              ],\n              [\n                0.4674166738986969,\n                0.17753125727176666,\n                1.0,\n                0.3325781226158142\n              ],\n              [\n                0.4634583294391632,\n                0.001234375056810677,\n                1.0,\n                0.3325781226158142\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a pizza",\n            "bounds": [\n              [\n                0.4634583294391632,\n                0.001234375056810677,\n                1.0,\n                0.2026406228542328\n              ],\n              [\n                0.4674166738986969,\n                0.17753125727176666,\n                1.0,\n                0.3325781226158142\n              ],\n              [\n                0.4634583294391632,\n                0.001234375056810677,\n                1.0,\n                0.3325781226158142\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000421086.jpg",\n        "coco_id": 421086,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "pizza",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 488434,\n          "object_id": 1076790,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_421086_488434_1076790_hold_eat",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 488434,\n          "object_id": 1076790,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a pizza",\n            "bounds": [\n              [\n                0.3402968645095825,\n                0.003958333283662796,\n                1.0,\n                0.5818333625793457\n              ],\n              [\n                0.44493749737739563,\n                0.4584166705608368,\n                0.6876406073570251,\n                0.6943749785423279\n              ],\n              [\n                0.3402968645095825,\n                0.003958333283662796,\n                1.0,\n                0.6943749785423279\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.3402968645095825,\n                0.003958333283662796,\n                1.0,\n                0.5818333625793457\n              ],\n              [\n                0.44493749737739563,\n                0.4584166705608368,\n                0.6876406073570251,\n                0.6943749785423279\n              ],\n              [\n                0.3402968645095825,\n                0.003958333283662796,\n                1.0,\n                0.6943749785423279\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000444278.jpg",\n        "coco_id": 444278,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 493142,\n          "object_id": 652977,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_444278_493142_652977_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 493142,\n          "object_id": 652977,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a surfboard",\n            "bounds": [\n              [\n                0.6663593649864197,\n                0.5896875262260437,\n                0.7678750157356262,\n                0.742562472820282\n              ],\n              [\n                0.7080625295639038,\n                0.6651666760444641,\n                0.8239218592643738,\n                0.746708333492279\n              ],\n              [\n                0.6663593649864197,\n                0.5896875262260437,\n                0.8239218592643738,\n                0.746708333492279\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.6663593649864197,\n                0.5896875262260437,\n                0.7678750157356262,\n                0.742562472820282\n              ],\n              [\n                0.7080625295639038,\n                0.6651666760444641,\n                0.8239218592643738,\n                0.746708333492279\n              ],\n              [\n                0.6663593649864197,\n                0.5896875262260437,\n                0.8239218592643738,\n                0.746708333492279\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000457033.jpg",\n        "coco_id": 457033,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 191710,\n          "object_id": 56343,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_457033_191710_56343_hold_ride",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 191710,\n          "object_id": 56343,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.36921873688697815,\n                0.2296803593635559,\n                0.6380312442779541,\n                0.8730136752128601\n              ],\n              [\n                0.23451562225818634,\n                0.11746575683355331,\n                0.9778906106948853,\n                0.878242015838623\n              ],\n              [\n                0.23451562225818634,\n                0.11746575683355331,\n                0.9778906106948853,\n                0.878242015838623\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.36921873688697815,\n                0.2296803593635559,\n                0.6380312442779541,\n                0.8730136752128601\n              ],\n              [\n                0.23451562225818634,\n                0.11746575683355331,\n                0.9778906106948853,\n                0.878242015838623\n              ],\n              [\n                0.23451562225818634,\n                0.11746575683355331,\n                0.9778906106948853,\n                0.878242015838623\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000457033.jpg",\n        "coco_id": 457033,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 191710,\n          "object_id": 56343,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_457033_191710_56343_hold_sit",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 191710,\n          "object_id": 56343,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.36921873688697815,\n                0.2296803593635559,\n                0.6380312442779541,\n                0.8730136752128601\n              ],\n              [\n                0.23451562225818634,\n                0.11746575683355331,\n                0.9778906106948853,\n                0.878242015838623\n              ],\n              [\n                0.23451562225818634,\n                0.11746575683355331,\n                0.9778906106948853,\n                0.878242015838623\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.36921873688697815,\n                0.2296803593635559,\n                0.6380312442779541,\n                0.8730136752128601\n              ],\n              [\n                0.23451562225818634,\n                0.11746575683355331,\n                0.9778906106948853,\n                0.878242015838623\n              ],\n              [\n                0.23451562225818634,\n                0.11746575683355331,\n                0.9778906106948853,\n                0.878242015838623\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000484646.jpg",\n        "coco_id": 484646,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "surfboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 427595,\n          "object_id": 648089,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_484646_427595_648089_carry_hold",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 427595,\n          "object_id": 648089,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a surfboard",\n            "bounds": [\n              [\n                0.22247999906539917,\n                0.49663999676704407,\n                0.30000001192092896,\n                0.7078666687011719\n              ],\n              [\n                0.2253199964761734,\n                0.5670400261878967,\n                0.2484399974346161,\n                0.6393866539001465\n              ],\n              [\n                0.22247999906539917,\n                0.49663999676704407,\n                0.30000001192092896,\n                0.7078666687011719\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.22247999906539917,\n                0.49663999676704407,\n                0.30000001192092896,\n                0.7078666687011719\n              ],\n              [\n                0.2253199964761734,\n                0.5670400261878967,\n                0.2484399974346161,\n                0.6393866539001465\n              ],\n              [\n                0.22247999906539917,\n                0.49663999676704407,\n                0.30000001192092896,\n                0.7078666687011719\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000495476.jpg",\n        "coco_id": 495476,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1234948,\n          "object_id": 1823893,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_495476_1234948_1823893_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 1234948,\n          "object_id": 1823893,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.16593749821186066,\n                0.5517656207084656,\n                0.2729843854904175,\n                0.8132343888282776\n              ],\n              [\n                0.2146874964237213,\n                0.5888437628746033,\n                0.23921875655651093,\n                0.6354687213897705\n              ],\n              [\n                0.16593749821186066,\n                0.5517656207084656,\n                0.2729843854904175,\n                0.8132343888282776\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.16593749821186066,\n                0.5517656207084656,\n                0.2729843854904175,\n                0.8132343888282776\n              ],\n              [\n                0.2146874964237213,\n                0.5888437628746033,\n                0.23921875655651093,\n                0.6354687213897705\n              ],\n              [\n                0.16593749821186066,\n                0.5517656207084656,\n                0.2729843854904175,\n                0.8132343888282776\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000528712.jpg",\n        "coco_id": 528712,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 225265,\n          "object_id": 1436105,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_528712_225265_1436105_hold_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 225265,\n          "object_id": 1436105,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.06573999673128128,\n                0.15279999375343323,\n                0.30338001251220703,\n                0.905626654624939\n              ],\n              [\n                0.2537800073623657,\n                0.5835999846458435,\n                0.3059599995613098,\n                0.724399983882904\n              ],\n              [\n                0.06573999673128128,\n                0.15279999375343323,\n                0.3059599995613098,\n                0.905626654624939\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.06573999673128128,\n                0.15279999375343323,\n                0.30338001251220703,\n                0.905626654624939\n              ],\n              [\n                0.2537800073623657,\n                0.5835999846458435,\n                0.3059599995613098,\n                0.724399983882904\n              ],\n              [\n                0.06573999673128128,\n                0.15279999375343323,\n                0.3059599995613098,\n                0.905626654624939\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000536607.jpg",\n        "coco_id": 536607,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 189050,\n          "object_id": 1636463,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "verb_536607_189050_1636463_talk_on_phone_carry",\n        "task": "wrong_predicate",\n        "source_negative": {\n          "subject_id": 189050,\n          "object_id": 1636463,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.11935937404632568,\n                0.3682708442211151,\n                0.5209687352180481,\n                0.9897916913032532\n              ],\n              [\n                0.3244374990463257,\n                0.46293750405311584,\n                0.3345312476158142,\n                0.49658334255218506\n              ],\n              [\n                0.11935937404632568,\n                0.3682708442211151,\n                0.5209687352180481,\n                0.9897916913032532\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.11935937404632568,\n                0.3682708442211151,\n                0.5209687352180481,\n                0.9897916913032532\n              ],\n              [\n                0.3244374990463257,\n                0.46293750405311584,\n                0.3345312476158142,\n                0.49658334255218506\n              ],\n              [\n                0.11935937404632568,\n                0.3682708442211151,\n                0.5209687352180481,\n                0.9897916913032532\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000100611.jpg",\n        "coco_id": 100611,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 532255,\n          "object_id": 383115,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_100611_532255_383115_486517_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 486517,\n          "object_id": 383115,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.22372916340827942,\n                0.006796875037252903,\n                0.3567708432674408,\n                0.25850000977516174\n              ],\n              [\n                0.34835416078567505,\n                0.2038281261920929,\n                0.42641666531562805,\n                0.35923436284065247\n              ],\n              [\n                0.22372916340827942,\n                0.006796875037252903,\n                0.42641666531562805,\n                0.35923436284065247\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.39887499809265137,\n                0.06029687449336052,\n                0.637541651725769,\n                0.31853124499320984\n              ],\n              [\n                0.34835416078567505,\n                0.2038281261920929,\n                0.42641666531562805,\n                0.35923436284065247\n              ],\n              [\n                0.34835416078567505,\n                0.06029687449336052,\n                0.637541651725769,\n                0.35923436284065247\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000119979.jpg",\n        "coco_id": 119979,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 469685,\n          "object_id": 604599,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_119979_469685_604599_505410_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 505410,\n          "object_id": 604599,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.35532811284065247,\n                0.289508193731308,\n                0.5832343697547913,\n                0.9861358404159546\n              ],\n              [\n                0.357234388589859,\n                0.21967212855815887,\n                0.4173593819141388,\n                0.34210774302482605\n              ],\n              [\n                0.35532811284065247,\n                0.21967212855815887,\n                0.5832343697547913,\n                0.9861358404159546\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.46875,\n                0.22740046679973602,\n                0.7562500238418579,\n                1.0\n              ],\n              [\n                0.357234388589859,\n                0.21967212855815887,\n                0.4173593819141388,\n                0.34210774302482605\n              ],\n              [\n                0.357234388589859,\n                0.21967212855815887,\n                0.7562500238418579,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000124841.jpg",\n        "coco_id": 124841,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "sports ball",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1272295,\n          "object_id": 301539,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_124841_1272295_301539_1301829_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1301829,\n          "object_id": 301539,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.580984354019165,\n                0.3228289484977722,\n                0.842968761920929,\n                0.8802850842475891\n              ],\n              [\n                0.5660781264305115,\n                0.4467543959617615,\n                0.6258593797683716,\n                0.504188597202301\n              ],\n              [\n                0.5660781264305115,\n                0.3228289484977722,\n                0.842968761920929,\n                0.8802850842475891\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a sports ball",\n            "bounds": [\n              [\n                0.5176562666893005,\n                0.17605262994766235,\n                0.670828104019165,\n                0.7142324447631836\n              ],\n              [\n                0.5660781264305115,\n                0.4467543959617615,\n                0.6258593797683716,\n                0.504188597202301\n              ],\n              [\n                0.5176562666893005,\n                0.17605262994766235,\n                0.670828104019165,\n                0.7142324447631836\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000124841.jpg",\n        "coco_id": 124841,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "sports ball",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1272295,\n          "object_id": 305292,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_124841_1272295_305292_1301829_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1301829,\n          "object_id": 305292,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a sports ball",\n            "bounds": [\n              [\n                0.5176562666893005,\n                0.17605262994766235,\n                0.670828104019165,\n                0.7142324447631836\n              ],\n              [\n                0.7539530992507935,\n                0.528092086315155,\n                0.8090156316757202,\n                0.5932456254959106\n              ],\n              [\n                0.5176562666893005,\n                0.17605262994766235,\n                0.8090156316757202,\n                0.7142324447631836\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a sports ball",\n            "bounds": [\n              [\n                0.580984354019165,\n                0.3228289484977722,\n                0.842968761920929,\n                0.8802850842475891\n              ],\n              [\n                0.7539530992507935,\n                0.528092086315155,\n                0.8090156316757202,\n                0.5932456254959106\n              ],\n              [\n                0.580984354019165,\n                0.3228289484977722,\n                0.842968761920929,\n                0.8802850842475891\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000133229.jpg",\n        "coco_id": 133229,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "carrot",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 480732,\n          "object_id": 1066298,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_133229_480732_1066298_455492_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 455492,\n          "object_id": 1066298,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a carrot",\n            "bounds": [\n              [\n                0.09796874970197678,\n                0.14752084016799927,\n                0.5912187695503235,\n                0.8389583230018616\n              ],\n              [\n                0.49217188358306885,\n                0.8372916579246521,\n                0.6134374737739563,\n                0.9398124814033508\n              ],\n              [\n                0.09796874970197678,\n                0.14752084016799927,\n                0.6134374737739563,\n                0.9398124814033508\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a carrot",\n            "bounds": [\n              [\n                0.5157343745231628,\n                0.24268749356269836,\n                0.930343747138977,\n                0.9191041588783264\n              ],\n              [\n                0.49217188358306885,\n                0.8372916579246521,\n                0.6134374737739563,\n                0.9398124814033508\n              ],\n              [\n                0.49217188358306885,\n                0.24268749356269836,\n                0.930343747138977,\n                0.9398124814033508\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000134639.jpg",\n        "coco_id": 134639,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2164785,\n          "object_id": 659498,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_134639_2164785_659498_1708419_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1708419,\n          "object_id": 659498,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.022703124210238457,\n                0.2313849776983261,\n                0.12095312774181366,\n                0.3619014024734497\n              ],\n              [\n                0.18125000596046448,\n                0.5546009540557861,\n                0.2090156227350235,\n                0.6738497614860535\n              ],\n              [\n                0.022703124210238457,\n                0.2313849776983261,\n                0.2090156227350235,\n                0.6738497614860535\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.1941562443971634,\n                0.5335211157798767,\n                0.30553126335144043,\n                0.9088732600212097\n              ],\n              [\n                0.18125000596046448,\n                0.5546009540557861,\n                0.2090156227350235,\n                0.6738497614860535\n              ],\n              [\n                0.18125000596046448,\n                0.5335211157798767,\n                0.30553126335144043,\n                0.9088732600212097\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000142637.jpg",\n        "coco_id": 142637,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1737811,\n          "object_id": 601797,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_142637_1737811_601797_1271736_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1271736,\n          "object_id": 601797,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.21581250429153442,\n                0.17531615495681763,\n                0.7001405954360962,\n                0.9876580834388733\n              ],\n              [\n                0.10309375077486038,\n                0.4154098331928253,\n                0.25770312547683716,\n                0.6097424030303955\n              ],\n              [\n                0.10309375077486038,\n                0.17531615495681763,\n                0.7001405954360962,\n                0.9876580834388733\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.10795312374830246,\n                0.1415690928697586,\n                0.39881250262260437,\n                1.0\n              ],\n              [\n                0.10309375077486038,\n                0.4154098331928253,\n                0.25770312547683716,\n                0.6097424030303955\n              ],\n              [\n                0.10309375077486038,\n                0.1415690928697586,\n                0.39881250262260437,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000014873.jpg",\n        "coco_id": 14873,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1260282,\n          "object_id": 1178117,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_14873_1260282_1178117_1213217_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1213217,\n          "object_id": 1178117,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.4574531316757202,\n                0.4442499876022339,\n                0.5458906292915344,\n                0.706166684627533\n              ],\n              [\n                0.06585937738418579,\n                0.5600833296775818,\n                0.11170312762260437,\n                0.6302083134651184\n              ],\n              [\n                0.06585937738418579,\n                0.4442499876022339,\n                0.5458906292915344,\n                0.706166684627533\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.06467187404632568,\n                0.5221250057220459,\n                0.16756249964237213,\n                0.7467708587646484\n              ],\n              [\n                0.06585937738418579,\n                0.5600833296775818,\n                0.11170312762260437,\n                0.6302083134651184\n              ],\n              [\n                0.06467187404632568,\n                0.5221250057220459,\n                0.16756249964237213,\n                0.7467708587646484\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000014873.jpg",\n        "coco_id": 14873,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1260282,\n          "object_id": 1178117,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_14873_1260282_1178117_1284477_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1284477,\n          "object_id": 1178117,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.6698125004768372,\n                0.4712916612625122,\n                0.7501875162124634,\n                0.6712083220481873\n              ],\n              [\n                0.06585937738418579,\n                0.5600833296775818,\n                0.11170312762260437,\n                0.6302083134651184\n              ],\n              [\n                0.06585937738418579,\n                0.4712916612625122,\n                0.7501875162124634,\n                0.6712083220481873\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.06467187404632568,\n                0.5221250057220459,\n                0.16756249964237213,\n                0.7467708587646484\n              ],\n              [\n                0.06585937738418579,\n                0.5600833296775818,\n                0.11170312762260437,\n                0.6302083134651184\n              ],\n              [\n                0.06467187404632568,\n                0.5221250057220459,\n                0.16756249964237213,\n                0.7467708587646484\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000014873.jpg",\n        "coco_id": 14873,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1284477,\n          "object_id": 1172516,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_14873_1284477_1172516_1213217_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1213217,\n          "object_id": 1172516,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.6698125004768372,\n                0.4712916612625122,\n                0.7501875162124634,\n                0.6712083220481873\n              ],\n              [\n                0.6967968940734863,\n                0.5112916827201843,\n                0.7297343611717224,\n                0.5739374756813049\n              ],\n              [\n                0.6698125004768372,\n                0.4712916612625122,\n                0.7501875162124634,\n                0.6712083220481873\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.4574531316757202,\n                0.4442499876022339,\n                0.5458906292915344,\n                0.706166684627533\n              ],\n              [\n                0.6967968940734863,\n                0.5112916827201843,\n                0.7297343611717224,\n                0.5739374756813049\n              ],\n              [\n                0.4574531316757202,\n                0.4442499876022339,\n                0.7297343611717224,\n                0.706166684627533\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000014873.jpg",\n        "coco_id": 14873,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2020813,\n          "object_id": 1163596,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_14873_2020813_1163596_1283044_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1283044,\n          "object_id": 1163596,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.5844218730926514,\n                0.47479167580604553,\n                0.6362344026565552,\n                0.6873958110809326\n              ],\n              [\n                0.3012968897819519,\n                0.5220624804496765,\n                0.3409687578678131,\n                0.5870833396911621\n              ],\n              [\n                0.3012968897819519,\n                0.47479167580604553,\n                0.6362344026565552,\n                0.6873958110809326\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.2779531180858612,\n                0.49002084136009216,\n                0.3393281102180481,\n                0.7013124823570251\n              ],\n              [\n                0.3012968897819519,\n                0.5220624804496765,\n                0.3409687578678131,\n                0.5870833396911621\n              ],\n              [\n                0.2779531180858612,\n                0.49002084136009216,\n                0.3409687578678131,\n                0.7013124823570251\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000174135.jpg",\n        "coco_id": 174135,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "scissors",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2207659,\n          "object_id": 1994819,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_174135_2207659_1994819_431462_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 431462,\n          "object_id": 1994819,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding scissors",\n            "bounds": [\n              [\n                0.0,\n                0.002468750113621354,\n                0.995526909828186,\n                0.7185156345367432\n              ],\n              [\n                0.2992739975452423,\n                0.13378125429153442,\n                0.6625527143478394,\n                0.25779688358306885\n              ],\n              [\n                0.0,\n                0.002468750113621354,\n                0.995526909828186,\n                0.7185156345367432\n              ]\n            ]\n          },\n          {\n            "text": "a person holding scissors",\n            "bounds": [\n              [\n                0.0,\n                0.3280937373638153,\n                0.9969789385795593,\n                0.9887655973434448\n              ],\n              [\n                0.2992739975452423,\n                0.13378125429153442,\n                0.6625527143478394,\n                0.25779688358306885\n              ],\n              [\n                0.0,\n                0.13378125429153442,\n                0.9969789385795593,\n                0.9887655973434448\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000174738.jpg",\n        "coco_id": 174738,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1677791,\n          "object_id": 2212638,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_174738_1677791_2212638_197195_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 197195,\n          "object_id": 2212638,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.9517343640327454,\n                0.26508334279060364,\n                1.0,\n                0.5016458630561829\n              ],\n              [\n                0.24815624952316284,\n                0.4007291793823242,\n                1.0,\n                1.0\n              ],\n              [\n                0.24815624952316284,\n                0.26508334279060364,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.600265622138977,\n                0.0015208333497866988,\n                0.7202500104904175,\n                0.17929166555404663\n              ],\n              [\n                0.24815624952316284,\n                0.4007291793823242,\n                1.0,\n                1.0\n              ],\n              [\n                0.24815624952316284,\n                0.0015208333497866988,\n                1.0,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000017520.jpg",\n        "coco_id": 17520,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 197392,\n          "object_id": 1178665,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_17520_197392_1178665_196780_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 196780,\n          "object_id": 1178665,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.18262499570846558,\n                0.4507981240749359,\n                0.3176249861717224,\n                0.826478898525238\n              ],\n              [\n                0.08476562798023224,\n                0.5748826265335083,\n                0.11284375190734863,\n                0.6608685255050659\n              ],\n              [\n                0.08476562798023224,\n                0.4507981240749359,\n                0.3176249861717224,\n                0.826478898525238\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.06949999928474426,\n                0.4758685529232025,\n                0.18976563215255737,\n                0.827863872051239\n              ],\n              [\n                0.08476562798023224,\n                0.5748826265335083,\n                0.11284375190734863,\n                0.6608685255050659\n              ],\n              [\n                0.06949999928474426,\n                0.4758685529232025,\n                0.18976563215255737,\n                0.827863872051239\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000188693.jpg",\n        "coco_id": 188693,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "book",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 198590,\n          "object_id": 1648464,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_188693_198590_1648464_202781_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 202781,\n          "object_id": 1648464,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a book",\n            "bounds": [\n              [\n                0.8223541378974915,\n                0.28626561164855957,\n                1.0,\n                0.7244687676429749\n              ],\n              [\n                0.40177083015441895,\n                0.4096718728542328,\n                0.4949583411216736,\n                0.4689687490463257\n              ],\n              [\n                0.40177083015441895,\n                0.28626561164855957,\n                1.0,\n                0.7244687676429749\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a book",\n            "bounds": [\n              [\n                0.3929791748523712,\n                0.3266249895095825,\n                0.5725625157356262,\n                0.7218906283378601\n              ],\n              [\n                0.40177083015441895,\n                0.4096718728542328,\n                0.4949583411216736,\n                0.4689687490463257\n              ],\n              [\n                0.3929791748523712,\n                0.3266249895095825,\n                0.5725625157356262,\n                0.7218906283378601\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000193025.jpg",\n        "coco_id": 193025,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 198934,\n          "object_id": 322699,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_193025_198934_322699_200903_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 200903,\n          "object_id": 322699,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.4334813058376312,\n                0.19550000131130219,\n                0.7527102828025818,\n                0.5595468878746033\n              ],\n              [\n                0.6727336645126343,\n                0.7152343988418579,\n                0.7344158887863159,\n                0.758859395980835\n              ],\n              [\n                0.4334813058376312,\n                0.19550000131130219,\n                0.7527102828025818,\n                0.758859395980835\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.547383189201355,\n                0.6260625123977661,\n                0.8660747408866882,\n                1.0\n              ],\n              [\n                0.6727336645126343,\n                0.7152343988418579,\n                0.7344158887863159,\n                0.758859395980835\n              ],\n              [\n                0.547383189201355,\n                0.6260625123977661,\n                0.8660747408866882,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000193025.jpg",\n        "coco_id": 193025,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 200903,\n          "object_id": 127335,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_193025_200903_127335_198934_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 198934,\n          "object_id": 127335,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.547383189201355,\n                0.6260625123977661,\n                0.8660747408866882,\n                1.0\n              ],\n              [\n                0.354976624250412,\n                0.37795311212539673,\n                0.8971728682518005,\n                0.6587499976158142\n              ],\n              [\n                0.354976624250412,\n                0.37795311212539673,\n                0.8971728682518005,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.4334813058376312,\n                0.19550000131130219,\n                0.7527102828025818,\n                0.5595468878746033\n              ],\n              [\n                0.354976624250412,\n                0.37795311212539673,\n                0.8971728682518005,\n                0.6587499976158142\n              ],\n              [\n                0.354976624250412,\n                0.19550000131130219,\n                0.8971728682518005,\n                0.6587499976158142\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000193025.jpg",\n        "coco_id": 193025,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bicycle",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 200903,\n          "object_id": 127335,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_193025_200903_127335_198934_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 198934,\n          "object_id": 127335,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.4334813058376312,\n                0.19550000131130219,\n                0.7527102828025818,\n                0.5595468878746033\n              ],\n              [\n                0.354976624250412,\n                0.37795311212539673,\n                0.8971728682518005,\n                0.6587499976158142\n              ],\n              [\n                0.354976624250412,\n                0.19550000131130219,\n                0.8971728682518005,\n                0.6587499976158142\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bicycle",\n            "bounds": [\n              [\n                0.547383189201355,\n                0.6260625123977661,\n                0.8660747408866882,\n                1.0\n              ],\n              [\n                0.354976624250412,\n                0.37795311212539673,\n                0.8971728682518005,\n                0.6587499976158142\n              ],\n              [\n                0.354976624250412,\n                0.37795311212539673,\n                0.8971728682518005,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000021079.jpg",\n        "coco_id": 21079,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "spoon",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1284744,\n          "object_id": 2185246,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_21079_1284744_2185246_1308401_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1308401,\n          "object_id": 2185246,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a spoon",\n            "bounds": [\n              [\n                0.6736299991607666,\n                0.45842188596725464,\n                1.0,\n                1.0\n              ],\n              [\n                0.10121779888868332,\n                0.892828106880188,\n                0.22901639342308044,\n                0.9057187438011169\n              ],\n              [\n                0.10121779888868332,\n                0.45842188596725464,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a spoon",\n            "bounds": [\n              [\n                0.10199063271284103,\n                0.6854062676429749,\n                0.6694613695144653,\n                1.0\n              ],\n              [\n                0.10121779888868332,\n                0.892828106880188,\n                0.22901639342308044,\n                0.9057187438011169\n              ],\n              [\n                0.10121779888868332,\n                0.6854062676429749,\n                0.6694613695144653,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000221202.jpg",\n        "coco_id": 221202,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 199544,\n          "object_id": 54099,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_221202_199544_54099_264332_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 264332,\n          "object_id": 54099,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.6715624928474426,\n                0.4152783453464508,\n                0.7846562266349792,\n                0.6898350715637207\n              ],\n              [\n                0.668093740940094,\n                0.5013195872306824,\n                0.8538749814033508,\n                0.7810721397399902\n              ],\n              [\n                0.668093740940094,\n                0.4152783453464508,\n                0.8538749814033508,\n                0.7810721397399902\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.07082812488079071,\n                0.5407422780990601,\n                0.12387499958276749,\n                0.7157319784164429\n              ],\n              [\n                0.668093740940094,\n                0.5013195872306824,\n                0.8538749814033508,\n                0.7810721397399902\n              ],\n              [\n                0.07082812488079071,\n                0.5013195872306824,\n                0.8538749814033508,\n                0.7810721397399902\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000221202.jpg",\n        "coco_id": 221202,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 199544,\n          "object_id": 54099,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_221202_199544_54099_264332_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 264332,\n          "object_id": 54099,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.6715624928474426,\n                0.4152783453464508,\n                0.7846562266349792,\n                0.6898350715637207\n              ],\n              [\n                0.668093740940094,\n                0.5013195872306824,\n                0.8538749814033508,\n                0.7810721397399902\n              ],\n              [\n                0.668093740940094,\n                0.4152783453464508,\n                0.8538749814033508,\n                0.7810721397399902\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.07082812488079071,\n                0.5407422780990601,\n                0.12387499958276749,\n                0.7157319784164429\n              ],\n              [\n                0.668093740940094,\n                0.5013195872306824,\n                0.8538749814033508,\n                0.7810721397399902\n              ],\n              [\n                0.07082812488079071,\n                0.5013195872306824,\n                0.8538749814033508,\n                0.7810721397399902\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000221202.jpg",\n        "coco_id": 221202,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 205330,\n          "object_id": 59568,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_221202_205330_59568_264332_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 264332,\n          "object_id": 59568,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.3952343761920929,\n                0.42478349804878235,\n                0.4674062430858612,\n                0.631134033203125\n              ],\n              [\n                0.4090625047683716,\n                0.5397937893867493,\n                0.4937968850135803,\n                0.7247216701507568\n              ],\n              [\n                0.3952343761920929,\n                0.42478349804878235,\n                0.4937968850135803,\n                0.7247216701507568\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.07082812488079071,\n                0.5407422780990601,\n                0.12387499958276749,\n                0.7157319784164429\n              ],\n              [\n                0.4090625047683716,\n                0.5397937893867493,\n                0.4937968850135803,\n                0.7247216701507568\n              ],\n              [\n                0.07082812488079071,\n                0.5397937893867493,\n                0.4937968850135803,\n                0.7247216701507568\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000221202.jpg",\n        "coco_id": 221202,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 205330,\n          "object_id": 59568,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_221202_205330_59568_264332_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 264332,\n          "object_id": 59568,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.07082812488079071,\n                0.5407422780990601,\n                0.12387499958276749,\n                0.7157319784164429\n              ],\n              [\n                0.4090625047683716,\n                0.5397937893867493,\n                0.4937968850135803,\n                0.7247216701507568\n              ],\n              [\n                0.07082812488079071,\n                0.5397937893867493,\n                0.4937968850135803,\n                0.7247216701507568\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.3952343761920929,\n                0.42478349804878235,\n                0.4674062430858612,\n                0.631134033203125\n              ],\n              [\n                0.4090625047683716,\n                0.5397937893867493,\n                0.4937968850135803,\n                0.7247216701507568\n              ],\n              [\n                0.3952343761920929,\n                0.42478349804878235,\n                0.4937968850135803,\n                0.7247216701507568\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000221202.jpg",\n        "coco_id": 221202,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 256066,\n          "object_id": 57539,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_221202_256066_57539_264332_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 264332,\n          "object_id": 57539,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.5357656478881836,\n                0.42538145184516907,\n                0.6115312576293945,\n                0.6336288452148438\n              ],\n              [\n                0.538937509059906,\n                0.5029484629631042,\n                0.6559374928474426,\n                0.7407216429710388\n              ],\n              [\n                0.5357656478881836,\n                0.42538145184516907,\n                0.6559374928474426,\n                0.7407216429710388\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.07082812488079071,\n                0.5407422780990601,\n                0.12387499958276749,\n                0.7157319784164429\n              ],\n              [\n                0.538937509059906,\n                0.5029484629631042,\n                0.6559374928474426,\n                0.7407216429710388\n              ],\n              [\n                0.07082812488079071,\n                0.5029484629631042,\n                0.6559374928474426,\n                0.7407216429710388\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000221202.jpg",\n        "coco_id": 221202,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 256066,\n          "object_id": 57539,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_221202_256066_57539_264332_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 264332,\n          "object_id": 57539,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.5357656478881836,\n                0.42538145184516907,\n                0.6115312576293945,\n                0.6336288452148438\n              ],\n              [\n                0.538937509059906,\n                0.5029484629631042,\n                0.6559374928474426,\n                0.7407216429710388\n              ],\n              [\n                0.5357656478881836,\n                0.42538145184516907,\n                0.6559374928474426,\n                0.7407216429710388\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.07082812488079071,\n                0.5407422780990601,\n                0.12387499958276749,\n                0.7157319784164429\n              ],\n              [\n                0.538937509059906,\n                0.5029484629631042,\n                0.6559374928474426,\n                0.7407216429710388\n              ],\n              [\n                0.07082812488079071,\n                0.5029484629631042,\n                0.6559374928474426,\n                0.7407216429710388\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000238982.jpg",\n        "coco_id": 238982,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1215230,\n          "object_id": 150567,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_238982_1215230_150567_2004986_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2004986,\n          "object_id": 150567,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.02056249976158142,\n                0.29236534237861633,\n                0.12950000166893005,\n                0.6312412023544312\n              ],\n              [\n                0.5107187628746033,\n                0.3509133458137512,\n                0.8451406359672546,\n                0.6964637041091919\n              ],\n              [\n                0.02056249976158142,\n                0.29236534237861633,\n                0.8451406359672546,\n                0.6964637041091919\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a motorcycle",\n            "bounds": [\n              [\n                0.618359386920929,\n                0.29192036390304565,\n                0.7251406311988831,\n                0.6327400207519531\n              ],\n              [\n                0.5107187628746033,\n                0.3509133458137512,\n                0.8451406359672546,\n                0.6964637041091919\n              ],\n              [\n                0.5107187628746033,\n                0.29192036390304565,\n                0.8451406359672546,\n                0.6964637041091919\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000238982.jpg",\n        "coco_id": 238982,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1242023,\n          "object_id": 54767,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_238982_1242023_54767_2004986_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2004986,\n          "object_id": 54767,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.3013593852519989,\n                0.12149883061647415,\n                0.44679686427116394,\n                0.530491828918457\n              ],\n              [\n                0.1764843761920929,\n                0.18859484791755676,\n                0.5567031502723694,\n                0.7821311354637146\n              ],\n              [\n                0.1764843761920929,\n                0.12149883061647415,\n                0.5567031502723694,\n                0.7821311354637146\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.02056249976158142,\n                0.29236534237861633,\n                0.12950000166893005,\n                0.6312412023544312\n              ],\n              [\n                0.1764843761920929,\n                0.18859484791755676,\n                0.5567031502723694,\n                0.7821311354637146\n              ],\n              [\n                0.02056249976158142,\n                0.18859484791755676,\n                0.5567031502723694,\n                0.7821311354637146\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000246735.jpg",\n        "coco_id": 246735,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 489525,\n          "object_id": 630576,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_246735_489525_630576_524746_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 524746,\n          "object_id": 630576,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.2876458466053009,\n                0.4696562588214874,\n                0.437458336353302,\n                0.6292187571525574\n              ],\n              [\n                0.38214582204818726,\n                0.5647343993186951,\n                0.4635624885559082,\n                0.6253281235694885\n              ],\n              [\n                0.2876458466053009,\n                0.4696562588214874,\n                0.4635624885559082,\n                0.6292187571525574\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.0,\n                0.5446249842643738,\n                0.1576666682958603,\n                0.7138281464576721\n              ],\n              [\n                0.38214582204818726,\n                0.5647343993186951,\n                0.4635624885559082,\n                0.6253281235694885\n              ],\n              [\n                0.0,\n                0.5446249842643738,\n                0.4635624885559082,\n                0.7138281464576721\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000251696.jpg",\n        "coco_id": 251696,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 423468,\n          "object_id": 326265,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_251696_423468_326265_436671_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 436671,\n          "object_id": 326265,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.5022500157356262,\n                0.2176041603088379,\n                0.7348281145095825,\n                0.9861458539962769\n              ],\n              [\n                0.15198437869548798,\n                0.45399999618530273,\n                0.1847812533378601,\n                0.5068125128746033\n              ],\n              [\n                0.15198437869548798,\n                0.2176041603088379,\n                0.7348281145095825,\n                0.9861458539962769\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.04214062541723251,\n                0.2383541613817215,\n                0.37415623664855957,\n                0.9799166917800903\n              ],\n              [\n                0.15198437869548798,\n                0.45399999618530273,\n                0.1847812533378601,\n                0.5068125128746033\n              ],\n              [\n                0.04214062541723251,\n                0.2383541613817215,\n                0.37415623664855957,\n                0.9799166917800903\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000251696.jpg",\n        "coco_id": 251696,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 436671,\n          "object_id": 323229,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_251696_436671_323229_423468_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 423468,\n          "object_id": 323229,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.04214062541723251,\n                0.2383541613817215,\n                0.37415623664855957,\n                0.9799166917800903\n              ],\n              [\n                0.6117187738418579,\n                0.2996875047683716,\n                0.6603749990463257,\n                0.3738541603088379\n              ],\n              [\n                0.04214062541723251,\n                0.2383541613817215,\n                0.6603749990463257,\n                0.9799166917800903\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.5022500157356262,\n                0.2176041603088379,\n                0.7348281145095825,\n                0.9861458539962769\n              ],\n              [\n                0.6117187738418579,\n                0.2996875047683716,\n                0.6603749990463257,\n                0.3738541603088379\n              ],\n              [\n                0.5022500157356262,\n                0.2176041603088379,\n                0.7348281145095825,\n                0.9861458539962769\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000255983.jpg",\n        "coco_id": 255983,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1228675,\n          "object_id": 576828,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_255983_1228675_576828_1262173_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1262173,\n          "object_id": 576828,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.7333124876022339,\n                0.26238876581192017,\n                0.8625312447547913,\n                0.86147540807724\n              ],\n              [\n                0.015250000171363354,\n                0.7200702428817749,\n                0.26343750953674316,\n                0.9455737471580505\n              ],\n              [\n                0.015250000171363354,\n                0.26238876581192017,\n                0.8625312447547913,\n                0.9455737471580505\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.30418750643730164,\n                0.4151756465435028,\n                0.6873437762260437,\n                0.9511709809303284\n              ],\n              [\n                0.015250000171363354,\n                0.7200702428817749,\n                0.26343750953674316,\n                0.9455737471580505\n              ],\n              [\n                0.015250000171363354,\n                0.4151756465435028,\n                0.6873437762260437,\n                0.9511709809303284\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000255983.jpg",\n        "coco_id": 255983,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1262173,\n          "object_id": 1175095,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_255983_1262173_1175095_1322934_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1322934,\n          "object_id": 1175095,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.18590624630451202,\n                0.3685480058193207,\n                0.3688281178474426,\n                0.9438173174858093\n              ],\n              [\n                0.7327656149864197,\n                0.3387353718280792,\n                0.7882500290870667,\n                0.5478219985961914\n              ],\n              [\n                0.18590624630451202,\n                0.3387353718280792,\n                0.7882500290870667,\n                0.9438173174858093\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.7333124876022339,\n                0.26238876581192017,\n                0.8625312447547913,\n                0.86147540807724\n              ],\n              [\n                0.7327656149864197,\n                0.3387353718280792,\n                0.7882500290870667,\n                0.5478219985961914\n              ],\n              [\n                0.7327656149864197,\n                0.26238876581192017,\n                0.8625312447547913,\n                0.86147540807724\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000255983.jpg",\n        "coco_id": 255983,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1281068,\n          "object_id": 1177113,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_255983_1281068_1177113_1228675_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1228675,\n          "object_id": 1177113,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.30418750643730164,\n                0.4151756465435028,\n                0.6873437762260437,\n                0.9511709809303284\n              ],\n              [\n                0.8628437519073486,\n                0.4332318603992462,\n                0.9169843792915344,\n                0.7278454303741455\n              ],\n              [\n                0.30418750643730164,\n                0.4151756465435028,\n                0.9169843792915344,\n                0.9511709809303284\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.8483906388282776,\n                0.27676814794540405,\n                1.0,\n                0.9996019005775452\n              ],\n              [\n                0.8628437519073486,\n                0.4332318603992462,\n                0.9169843792915344,\n                0.7278454303741455\n              ],\n              [\n                0.8483906388282776,\n                0.27676814794540405,\n                1.0,\n                0.9996019005775452\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000255983.jpg",\n        "coco_id": 255983,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1322934,\n          "object_id": 576828,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_255983_1322934_576828_1262173_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1262173,\n          "object_id": 576828,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.7333124876022339,\n                0.26238876581192017,\n                0.8625312447547913,\n                0.86147540807724\n              ],\n              [\n                0.015250000171363354,\n                0.7200702428817749,\n                0.26343750953674316,\n                0.9455737471580505\n              ],\n              [\n                0.015250000171363354,\n                0.26238876581192017,\n                0.8625312447547913,\n                0.9455737471580505\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.18590624630451202,\n                0.3685480058193207,\n                0.3688281178474426,\n                0.9438173174858093\n              ],\n              [\n                0.015250000171363354,\n                0.7200702428817749,\n                0.26343750953674316,\n                0.9455737471580505\n              ],\n              [\n                0.015250000171363354,\n                0.3685480058193207,\n                0.3688281178474426,\n                0.9455737471580505\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000269986.jpg",\n        "coco_id": 269986,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1743301,\n          "object_id": 1173101,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_269986_1743301_1173101_459165_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 459165,\n          "object_id": 1173101,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.6258281469345093,\n                0.0641966462135315,\n                0.8944844007492065,\n                0.9876498579978943\n              ],\n              [\n                0.7885937690734863,\n                0.4145084023475647,\n                0.9353281259536743,\n                0.7529016733169556\n              ],\n              [\n                0.6258281469345093,\n                0.0641966462135315,\n                0.9353281259536743,\n                0.9876498579978943\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.1947343796491623,\n                0.02695443667471409,\n                0.531499981880188,\n                0.9887530207633972\n              ],\n              [\n                0.7885937690734863,\n                0.4145084023475647,\n                0.9353281259536743,\n                0.7529016733169556\n              ],\n              [\n                0.1947343796491623,\n                0.02695443667471409,\n                0.9353281259536743,\n                0.9887530207633972\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000276711.jpg",\n        "coco_id": 276711,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "remote",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 481560,\n          "object_id": 1109130,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_276711_481560_1109130_532901_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 532901,\n          "object_id": 1109130,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.7432656288146973,\n                0.25841665267944336,\n                1.0,\n                0.9887708425521851\n              ],\n              [\n                0.5501406192779541,\n                0.8535416722297668,\n                0.6552656292915344,\n                0.9260625243186951\n              ],\n              [\n                0.5501406192779541,\n                0.25841665267944336,\n                1.0,\n                0.9887708425521851\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.18556250631809235,\n                0.11654166877269745,\n                0.6386250257492065,\n                1.0\n              ],\n              [\n                0.5501406192779541,\n                0.8535416722297668,\n                0.6552656292915344,\n                0.9260625243186951\n              ],\n              [\n                0.18556250631809235,\n                0.11654166877269745,\n                0.6552656292915344,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000277284.jpg",\n        "coco_id": 277284,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1742952,\n          "object_id": 601799,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_277284_1742952_601799_212141_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 212141,\n          "object_id": 601799,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.8525312542915344,\n                0.0632159635424614,\n                1.0,\n                0.9819483757019043\n              ],\n              [\n                0.8680312633514404,\n                0.6066901683807373,\n                0.9913281202316284,\n                0.7110563516616821\n              ],\n              [\n                0.8525312542915344,\n                0.0632159635424614,\n                1.0,\n                0.9819483757019043\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.4442499876022339,\n                0.3887558579444885,\n                0.560921847820282,\n                0.8359624147415161\n              ],\n              [\n                0.8680312633514404,\n                0.6066901683807373,\n                0.9913281202316284,\n                0.7110563516616821\n              ],\n              [\n                0.4442499876022339,\n                0.3887558579444885,\n                0.9913281202316284,\n                0.8359624147415161\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000279632.jpg",\n        "coco_id": 279632,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "snowboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1734292,\n          "object_id": 618227,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_279632_1734292_618227_489830_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 489830,\n          "object_id": 618227,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.28910937905311584,\n                0.45352083444595337,\n                0.4693906307220459,\n                0.8299375176429749\n              ],\n              [\n                0.5511249899864197,\n                0.28314584493637085,\n                0.6370781064033508,\n                0.7932500243186951\n              ],\n              [\n                0.28910937905311584,\n                0.28314584493637085,\n                0.6370781064033508,\n                0.8299375176429749\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.5506718754768372,\n                0.13512499630451202,\n                0.7331093549728394,\n                0.7770208120346069\n              ],\n              [\n                0.5511249899864197,\n                0.28314584493637085,\n                0.6370781064033508,\n                0.7932500243186951\n              ],\n              [\n                0.5506718754768372,\n                0.13512499630451202,\n                0.7331093549728394,\n                0.7932500243186951\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000279632.jpg",\n        "coco_id": 279632,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "snowboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 521304,\n          "object_id": 620639,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_279632_521304_620639_489830_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 489830,\n          "object_id": 620639,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.28910937905311584,\n                0.45352083444595337,\n                0.4693906307220459,\n                0.8299375176429749\n              ],\n              [\n                0.010140624828636646,\n                0.21835416555404663,\n                0.19503125548362732,\n                0.9160000085830688\n              ],\n              [\n                0.010140624828636646,\n                0.21835416555404663,\n                0.4693906307220459,\n                0.9160000085830688\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a snowboard",\n            "bounds": [\n              [\n                0.0708906278014183,\n                0.12472916394472122,\n                0.27482813596725464,\n                0.844083309173584\n              ],\n              [\n                0.010140624828636646,\n                0.21835416555404663,\n                0.19503125548362732,\n                0.9160000085830688\n              ],\n              [\n                0.010140624828636646,\n                0.12472916394472122,\n                0.27482813596725464,\n                0.9160000085830688\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000313541.jpg",\n        "coco_id": 313541,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 448656,\n          "object_id": 658451,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_313541_448656_658451_490374_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 490374,\n          "object_id": 658451,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.2515312433242798,\n                0.5229039788246155,\n                0.5088750123977661,\n                0.8776346445083618\n              ],\n              [\n                0.17442187666893005,\n                0.6303513050079346,\n                0.2700156271457672,\n                0.7041451930999756\n              ],\n              [\n                0.17442187666893005,\n                0.5229039788246155,\n                0.5088750123977661,\n                0.8776346445083618\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.28337499499320984,\n                0.2943793833255768,\n                0.3793281316757202,\n                0.5573067665100098\n              ],\n              [\n                0.17442187666893005,\n                0.6303513050079346,\n                0.2700156271457672,\n                0.7041451930999756\n              ],\n              [\n                0.17442187666893005,\n                0.2943793833255768,\n                0.3793281316757202,\n                0.7041451930999756\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000327110.jpg",\n        "coco_id": 327110,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1205336,\n          "object_id": 329566,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_327110_1205336_329566_1216673_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1216673,\n          "object_id": 329566,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.4677799940109253,\n                0.26442351937294006,\n                0.7217400074005127,\n                0.9956706166267395\n              ],\n              [\n                0.5027599930763245,\n                0.5267529487609863,\n                0.535260021686554,\n                0.5522588491439819\n              ],\n              [\n                0.4677799940109253,\n                0.26442351937294006,\n                0.7217400074005127,\n                0.9956706166267395\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a cell phone",\n            "bounds": [\n              [\n                0.7102400064468384,\n                0.12049411982297897,\n                0.9973999857902527,\n                1.0\n              ],\n              [\n                0.5027599930763245,\n                0.5267529487609863,\n                0.535260021686554,\n                0.5522588491439819\n              ],\n              [\n                0.5027599930763245,\n                0.12049411982297897,\n                0.9973999857902527,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000327110.jpg",\n        "coco_id": 327110,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1216673,\n          "object_id": 2137442,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_327110_1216673_2137442_1205336_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1205336,\n          "object_id": 2137442,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.4677799940109253,\n                0.26442351937294006,\n                0.7217400074005127,\n                0.9956706166267395\n              ],\n              [\n                0.8334800004959106,\n                0.25025883316993713,\n                0.8890399932861328,\n                0.34054118394851685\n              ],\n              [\n                0.4677799940109253,\n                0.25025883316993713,\n                0.8890399932861328,\n                0.9956706166267395\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.7102400064468384,\n                0.12049411982297897,\n                0.9973999857902527,\n                1.0\n              ],\n              [\n                0.8334800004959106,\n                0.25025883316993713,\n                0.8890399932861328,\n                0.34054118394851685\n              ],\n              [\n                0.7102400064468384,\n                0.12049411982297897,\n                0.9973999857902527,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000327110.jpg",\n        "coco_id": 327110,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1265321,\n          "object_id": 1180452,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_327110_1265321_1180452_1216673_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1216673,\n          "object_id": 1180452,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.7102400064468384,\n                0.12049411982297897,\n                0.9973999857902527,\n                1.0\n              ],\n              [\n                0.2911800146102905,\n                0.5290352702140808,\n                0.38741999864578247,\n                0.9183294177055359\n              ],\n              [\n                0.2911800146102905,\n                0.12049411982297897,\n                0.9973999857902527,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.10159999877214432,\n                0.17663529515266418,\n                0.38995999097824097,\n                1.0\n              ],\n              [\n                0.2911800146102905,\n                0.5290352702140808,\n                0.38741999864578247,\n                0.9183294177055359\n              ],\n              [\n                0.10159999877214432,\n                0.17663529515266418,\n                0.38995999097824097,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000330952.jpg",\n        "coco_id": 330952,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "umbrella",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1206812,\n          "object_id": 286841,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_330952_1206812_286841_1212367_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1212367,\n          "object_id": 286841,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.10132812708616257,\n                0.29973891377449036,\n                0.24851562082767487,\n                0.9322715401649475\n              ],\n              [\n                0.11954687535762787,\n                0.22023499011993408,\n                0.35401561856269836,\n                0.43522194027900696\n              ],\n              [\n                0.10132812708616257,\n                0.22023499011993408,\n                0.35401561856269836,\n                0.9322715401649475\n              ]\n            ]\n          },\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.2407187521457672,\n                0.3483028709888458,\n                0.36982813477516174,\n                0.8786423206329346\n              ],\n              [\n                0.11954687535762787,\n                0.22023499011993408,\n                0.35401561856269836,\n                0.43522194027900696\n              ],\n              [\n                0.11954687535762787,\n                0.22023499011993408,\n                0.36982813477516174,\n                0.8786423206329346\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000356374.jpg",\n        "coco_id": 356374,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 474578,\n          "object_id": 322677,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_356374_474578_322677_535115_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 535115,\n          "object_id": 322677,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.4915468692779541,\n                0.014645833522081375,\n                0.947640597820282,\n                0.9876041412353516\n              ],\n              [\n                0.620968759059906,\n                0.1456666737794876,\n                0.6942968964576721,\n                0.3761875033378601\n              ],\n              [\n                0.4915468692779541,\n                0.014645833522081375,\n                0.947640597820282,\n                0.9876041412353516\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.001687500043772161,\n                0.003958333283662796,\n                0.7178281545639038,\n                0.9881458282470703\n              ],\n              [\n                0.620968759059906,\n                0.1456666737794876,\n                0.6942968964576721,\n                0.3761875033378601\n              ],\n              [\n                0.001687500043772161,\n                0.003958333283662796,\n                0.7178281545639038,\n                0.9881458282470703\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000365164.jpg",\n        "coco_id": 365164,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "skateboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1266219,\n          "object_id": 1861832,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_365164_1266219_1861832_1238219_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1238219,\n          "object_id": 1861832,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.11934426426887512,\n                0.033265624195337296,\n                0.4357376992702484,\n                0.4375312626361847\n              ],\n              [\n                0.14426229894161224,\n                0.2426406294107437,\n                0.21266978979110718,\n                0.4177187383174896\n              ],\n              [\n                0.11934426426887512,\n                0.033265624195337296,\n                0.4357376992702484,\n                0.4375312626361847\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.01578454300761223,\n                0.07060937583446503,\n                0.31421545147895813,\n                0.46437498927116394\n              ],\n              [\n                0.14426229894161224,\n                0.2426406294107437,\n                0.21266978979110718,\n                0.4177187383174896\n              ],\n              [\n                0.01578454300761223,\n                0.07060937583446503,\n                0.31421545147895813,\n                0.46437498927116394\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000366948.jpg",\n        "coco_id": 366948,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1292493,\n          "object_id": 671493,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_366948_1292493_671493_1271216_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1271216,\n          "object_id": 671493,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.3689843714237213,\n                0.03920374810695648,\n                0.6521250009536743,\n                0.9850351214408875\n              ],\n              [\n                0.5027031302452087,\n                0.5780327916145325,\n                0.5685625076293945,\n                0.7418501377105713\n              ],\n              [\n                0.3689843714237213,\n                0.03920374810695648,\n                0.6521250009536743,\n                0.9850351214408875\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.4934062361717224,\n                0.10180328041315079,\n                0.9056875109672546,\n                1.0\n              ],\n              [\n                0.5027031302452087,\n                0.5780327916145325,\n                0.5685625076293945,\n                0.7418501377105713\n              ],\n              [\n                0.4934062361717224,\n                0.10180328041315079,\n                0.9056875109672546,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000370234.jpg",\n        "coco_id": 370234,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "wine glass",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 467300,\n          "object_id": 1496820,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_370234_467300_1496820_447773_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 447773,\n          "object_id": 1496820,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a wine glass",\n            "bounds": [\n              [\n                0.0,\n                0.4711875021457672,\n                0.4234218895435333,\n                1.0\n              ],\n              [\n                0.31156250834465027,\n                0.4711041748523712,\n                0.40312498807907104,\n                0.6346874833106995\n              ],\n              [\n                0.0,\n                0.4711041748523712,\n                0.4234218895435333,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a wine glass",\n            "bounds": [\n              [\n                0.028468750417232513,\n                0.35972917079925537,\n                0.38887500762939453,\n                0.937333345413208\n              ],\n              [\n                0.31156250834465027,\n                0.4711041748523712,\n                0.40312498807907104,\n                0.6346874833106995\n              ],\n              [\n                0.028468750417232513,\n                0.35972917079925537,\n                0.40312498807907104,\n                0.937333345413208\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000379944.jpg",\n        "coco_id": 379944,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 506930,\n          "object_id": 627837,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_379944_506930_627837_1203297_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1203297,\n          "object_id": 627837,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.32975998520851135,\n                0.2487460821866989,\n                0.6967800259590149,\n                0.9498745799064636\n              ],\n              [\n                0.14083999395370483,\n                0.5512539148330688,\n                0.41312000155448914,\n                0.6960501670837402\n              ],\n              [\n                0.14083999395370483,\n                0.2487460821866989,\n                0.6967800259590149,\n                0.9498745799064636\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.755079984664917,\n                0.0381191223859787,\n                0.9727799892425537,\n                0.8088088035583496\n              ],\n              [\n                0.14083999395370483,\n                0.5512539148330688,\n                0.41312000155448914,\n                0.6960501670837402\n              ],\n              [\n                0.14083999395370483,\n                0.0381191223859787,\n                0.9727799892425537,\n                0.8088088035583496\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000038380.jpg",\n        "coco_id": 38380,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1292729,\n          "object_id": 1471826,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_38380_1292729_1471826_1326860_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1326860,\n          "object_id": 1471826,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.9061562418937683,\n                0.26119565963745117,\n                0.9978593587875366,\n                0.7916304469108582\n              ],\n              [\n                0.9348124861717224,\n                0.5822826027870178,\n                0.9822187423706055,\n                0.6795651912689209\n              ],\n              [\n                0.9061562418937683,\n                0.26119565963745117,\n                0.9978593587875366,\n                0.7916304469108582\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.29201561212539673,\n                0.2516847848892212,\n                0.4793750047683716,\n                0.5797826051712036\n              ],\n              [\n                0.9348124861717224,\n                0.5822826027870178,\n                0.9822187423706055,\n                0.6795651912689209\n              ],\n              [\n                0.29201561212539673,\n                0.2516847848892212,\n                0.9822187423706055,\n                0.6795651912689209\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000038380.jpg",\n        "coco_id": 38380,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2162200,\n          "object_id": 628381,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_38380_2162200_628381_1326860_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1326860,\n          "object_id": 628381,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.29201561212539673,\n                0.2516847848892212,\n                0.4793750047683716,\n                0.5797826051712036\n              ],\n              [\n                0.15821875631809235,\n                0.4795108735561371,\n                0.2587187588214874,\n                0.5867391228675842\n              ],\n              [\n                0.15821875631809235,\n                0.2516847848892212,\n                0.4793750047683716,\n                0.5867391228675842\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.05078125,\n                0.13247282803058624,\n                0.19565625488758087,\n                0.6026086807250977\n              ],\n              [\n                0.15821875631809235,\n                0.4795108735561371,\n                0.2587187588214874,\n                0.5867391228675842\n              ],\n              [\n                0.05078125,\n                0.13247282803058624,\n                0.2587187588214874,\n                0.6026086807250977\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000393534.jpg",\n        "coco_id": 393534,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1205310,\n          "object_id": 56466,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_393534_1205310_56466_2163307_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2163307,\n          "object_id": 56466,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.8950625061988831,\n                0.8507746458053589,\n                0.9997968673706055,\n                0.999178409576416\n              ],\n              [\n                0.42890626192092896,\n                0.3732629120349884,\n                0.5979843735694885,\n                0.7327464818954468\n              ],\n              [\n                0.42890626192092896,\n                0.3732629120349884,\n                0.9997968673706055,\n                0.999178409576416\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.5058437585830688,\n                0.3260093927383423,\n                0.5865781307220459,\n                0.5685915350914001\n              ],\n              [\n                0.42890626192092896,\n                0.3732629120349884,\n                0.5979843735694885,\n                0.7327464818954468\n              ],\n              [\n                0.42890626192092896,\n                0.3260093927383423,\n                0.5979843735694885,\n                0.7327464818954468\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000393534.jpg",\n        "coco_id": 393534,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1205310,\n          "object_id": 56466,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_393534_1205310_56466_2163307_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2163307,\n          "object_id": 56466,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.5058437585830688,\n                0.3260093927383423,\n                0.5865781307220459,\n                0.5685915350914001\n              ],\n              [\n                0.42890626192092896,\n                0.3732629120349884,\n                0.5979843735694885,\n                0.7327464818954468\n              ],\n              [\n                0.42890626192092896,\n                0.3260093927383423,\n                0.5979843735694885,\n                0.7327464818954468\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.8950625061988831,\n                0.8507746458053589,\n                0.9997968673706055,\n                0.999178409576416\n              ],\n              [\n                0.42890626192092896,\n                0.3732629120349884,\n                0.5979843735694885,\n                0.7327464818954468\n              ],\n              [\n                0.42890626192092896,\n                0.3732629120349884,\n                0.9997968673706055,\n                0.999178409576416\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000393534.jpg",\n        "coco_id": 393534,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1211447,\n          "object_id": 56679,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_393534_1211447_56679_2163307_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2163307,\n          "object_id": 56679,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.8950625061988831,\n                0.8507746458053589,\n                0.9997968673706055,\n                0.999178409576416\n              ],\n              [\n                0.7843906283378601,\n                0.4923474192619324,\n                0.8930156230926514,\n                0.7991549372673035\n              ],\n              [\n                0.7843906283378601,\n                0.4923474192619324,\n                0.9997968673706055,\n                0.999178409576416\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.8032343983650208,\n                0.3505633771419525,\n                0.9154219031333923,\n                0.6134741902351379\n              ],\n              [\n                0.7843906283378601,\n                0.4923474192619324,\n                0.8930156230926514,\n                0.7991549372673035\n              ],\n              [\n                0.7843906283378601,\n                0.3505633771419525,\n                0.9154219031333923,\n                0.7991549372673035\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000393534.jpg",\n        "coco_id": 393534,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1211447,\n          "object_id": 56679,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_393534_1211447_56679_2163307_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2163307,\n          "object_id": 56679,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.8032343983650208,\n                0.3505633771419525,\n                0.9154219031333923,\n                0.6134741902351379\n              ],\n              [\n                0.7843906283378601,\n                0.4923474192619324,\n                0.8930156230926514,\n                0.7991549372673035\n              ],\n              [\n                0.7843906283378601,\n                0.3505633771419525,\n                0.9154219031333923,\n                0.7991549372673035\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.8950625061988831,\n                0.8507746458053589,\n                0.9997968673706055,\n                0.999178409576416\n              ],\n              [\n                0.7843906283378601,\n                0.4923474192619324,\n                0.8930156230926514,\n                0.7991549372673035\n              ],\n              [\n                0.7843906283378601,\n                0.4923474192619324,\n                0.9997968673706055,\n                0.999178409576416\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000394583.jpg",\n        "coco_id": 394583,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1702101,\n          "object_id": 1172019,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_394583_1702101_1172019_1690648_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1690648,\n          "object_id": 1172019,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.34721875190734863,\n                0.17397646605968475,\n                0.5502499938011169,\n                0.9671058654785156\n              ],\n              [\n                0.40517187118530273,\n                0.28195294737815857,\n                0.5130624771118164,\n                0.37978824973106384\n              ],\n              [\n                0.34721875190734863,\n                0.17397646605968475,\n                0.5502499938011169,\n                0.9671058654785156\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.7672656178474426,\n                0.16778823733329773,\n                0.9123437404632568,\n                0.7691529393196106\n              ],\n              [\n                0.40517187118530273,\n                0.28195294737815857,\n                0.5130624771118164,\n                0.37978824973106384\n              ],\n              [\n                0.40517187118530273,\n                0.16778823733329773,\n                0.9123437404632568,\n                0.7691529393196106\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000405361.jpg",\n        "coco_id": 405361,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 211141,\n          "object_id": 1181400,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_405361_211141_1181400_222294_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 222294,\n          "object_id": 1181400,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.5614843964576721,\n                0.2008962333202362,\n                0.7902500033378601,\n                0.9901415109634399\n              ],\n              [\n                0.5374374985694885,\n                0.350566029548645,\n                0.5791249871253967,\n                0.5325943231582642\n              ],\n              [\n                0.5374374985694885,\n                0.2008962333202362,\n                0.7902500033378601,\n                0.9901415109634399\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.4372187554836273,\n                0.2216981202363968,\n                0.5765312314033508,\n                0.834669828414917\n              ],\n              [\n                0.5374374985694885,\n                0.350566029548645,\n                0.5791249871253967,\n                0.5325943231582642\n              ],\n              [\n                0.4372187554836273,\n                0.2216981202363968,\n                0.5791249871253967,\n                0.834669828414917\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000405361.jpg",\n        "coco_id": 405361,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 222294,\n          "object_id": 1446525,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_405361_222294_1446525_211141_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 211141,\n          "object_id": 1446525,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.4372187554836273,\n                0.2216981202363968,\n                0.5765312314033508,\n                0.834669828414917\n              ],\n              [\n                0.7242187261581421,\n                0.666863203048706,\n                0.7807812690734863,\n                0.8317924737930298\n              ],\n              [\n                0.4372187554836273,\n                0.2216981202363968,\n                0.7807812690734863,\n                0.834669828414917\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.5614843964576721,\n                0.2008962333202362,\n                0.7902500033378601,\n                0.9901415109634399\n              ],\n              [\n                0.7242187261581421,\n                0.666863203048706,\n                0.7807812690734863,\n                0.8317924737930298\n              ],\n              [\n                0.5614843964576721,\n                0.2008962333202362,\n                0.7902500033378601,\n                0.9901415109634399\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000405361.jpg",\n        "coco_id": 405361,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 236444,\n          "object_id": 1177247,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_405361_236444_1177247_211141_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 211141,\n          "object_id": 1177247,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.4372187554836273,\n                0.2216981202363968,\n                0.5765312314033508,\n                0.834669828414917\n              ],\n              [\n                0.8485937714576721,\n                0.37941038608551025,\n                0.9781249761581421,\n                0.6378301978111267\n              ],\n              [\n                0.4372187554836273,\n                0.2216981202363968,\n                0.9781249761581421,\n                0.834669828414917\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.8296093940734863,\n                0.21959905326366425,\n                1.0,\n                0.9876179099082947\n              ],\n              [\n                0.8485937714576721,\n                0.37941038608551025,\n                0.9781249761581421,\n                0.6378301978111267\n              ],\n              [\n                0.8296093940734863,\n                0.21959905326366425,\n                1.0,\n                0.9876179099082947\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000405361.jpg",\n        "coco_id": 405361,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 236444,\n          "object_id": 1177247,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_405361_236444_1177247_222294_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 222294,\n          "object_id": 1177247,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.8296093940734863,\n                0.21959905326366425,\n                1.0,\n                0.9876179099082947\n              ],\n              [\n                0.8485937714576721,\n                0.37941038608551025,\n                0.9781249761581421,\n                0.6378301978111267\n              ],\n              [\n                0.8296093940734863,\n                0.21959905326366425,\n                1.0,\n                0.9876179099082947\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.5614843964576721,\n                0.2008962333202362,\n                0.7902500033378601,\n                0.9901415109634399\n              ],\n              [\n                0.8485937714576721,\n                0.37941038608551025,\n                0.9781249761581421,\n                0.6378301978111267\n              ],\n              [\n                0.5614843964576721,\n                0.2008962333202362,\n                0.9781249761581421,\n                0.9901415109634399\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000413248.jpg",\n        "coco_id": 413248,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 225823,\n          "object_id": 149488,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_413248_225823_149488_227738_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 227738,\n          "object_id": 149488,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.7048437595367432,\n                0.3372074365615845,\n                0.7957187294960022,\n                0.8286170363426208\n              ],\n              [\n                0.4508124887943268,\n                0.4455053210258484,\n                0.825390636920929,\n                0.9242553114891052\n              ],\n              [\n                0.4508124887943268,\n                0.3372074365615845,\n                0.825390636920929,\n                0.9242553114891052\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.6099374890327454,\n                0.35337767004966736,\n                0.7234843969345093,\n                0.8432712554931641\n              ],\n              [\n                0.4508124887943268,\n                0.4455053210258484,\n                0.825390636920929,\n                0.9242553114891052\n              ],\n              [\n                0.4508124887943268,\n                0.35337767004966736,\n                0.825390636920929,\n                0.9242553114891052\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000433336.jpg",\n        "coco_id": 433336,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "snowboard",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 507160,\n          "object_id": 620754,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_433336_507160_620754_424469_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 424469,\n          "object_id": 620754,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a snowboard",\n            "bounds": [\n              [\n                0.5216321349143982,\n                0.20860937237739563,\n                0.8085232973098755,\n                0.8310937285423279\n              ],\n              [\n                0.13041450083255768,\n                0.3752500116825104,\n                0.5411139726638794,\n                0.8337031006813049\n              ],\n              [\n                0.13041450083255768,\n                0.20860937237739563,\n                0.8085232973098755,\n                0.8337031006813049\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a snowboard",\n            "bounds": [\n              [\n                0.2701813578605652,\n                0.2602187395095825,\n                0.5600000023841858,\n                0.8392968773841858\n              ],\n              [\n                0.13041450083255768,\n                0.3752500116825104,\n                0.5411139726638794,\n                0.8337031006813049\n              ],\n              [\n                0.13041450083255768,\n                0.2602187395095825,\n                0.5600000023841858,\n                0.8392968773841858\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000444278.jpg",\n        "coco_id": 444278,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 493142,\n          "object_id": 652977,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_444278_493142_652977_490445_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 490445,\n          "object_id": 652977,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.6663593649864197,\n                0.5896875262260437,\n                0.7678750157356262,\n                0.742562472820282\n              ],\n              [\n                0.7080625295639038,\n                0.6651666760444641,\n                0.8239218592643738,\n                0.746708333492279\n              ],\n              [\n                0.6663593649864197,\n                0.5896875262260437,\n                0.8239218592643738,\n                0.746708333492279\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.25475001335144043,\n                0.38337498903274536,\n                0.3547656238079071,\n                0.546541690826416\n              ],\n              [\n                0.7080625295639038,\n                0.6651666760444641,\n                0.8239218592643738,\n                0.746708333492279\n              ],\n              [\n                0.25475001335144043,\n                0.38337498903274536,\n                0.8239218592643738,\n                0.746708333492279\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000450175.jpg",\n        "coco_id": 450175,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1220223,\n          "object_id": 628089,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_450175_1220223_628089_1747556_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1747556,\n          "object_id": 628089,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.3747812509536743,\n                0.46127083897590637,\n                0.5318437218666077,\n                0.7684583067893982\n              ],\n              [\n                0.3774999976158142,\n                0.406208336353302,\n                0.42265623807907104,\n                0.5533124804496765\n              ],\n              [\n                0.3747812509536743,\n                0.406208336353302,\n                0.5318437218666077,\n                0.7684583067893982\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.12962499260902405,\n                0.4938333332538605,\n                0.23518750071525574,\n                0.7827083468437195\n              ],\n              [\n                0.3774999976158142,\n                0.406208336353302,\n                0.42265623807907104,\n                0.5533124804496765\n              ],\n              [\n                0.12962499260902405,\n                0.406208336353302,\n                0.42265623807907104,\n                0.7827083468437195\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000450175.jpg",\n        "coco_id": 450175,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1681420,\n          "object_id": 633465,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_450175_1681420_633465_1747556_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1747556,\n          "object_id": 633465,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.12962499260902405,\n                0.4938333332538605,\n                0.23518750071525574,\n                0.7827083468437195\n              ],\n              [\n                0.3327656388282776,\n                0.6605208516120911,\n                0.3641406297683716,\n                0.7043125033378601\n              ],\n              [\n                0.12962499260902405,\n                0.4938333332538605,\n                0.3641406297683716,\n                0.7827083468437195\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.22678124904632568,\n                0.5334166884422302,\n                0.365359365940094,\n                0.7781041860580444\n              ],\n              [\n                0.3327656388282776,\n                0.6605208516120911,\n                0.3641406297683716,\n                0.7043125033378601\n              ],\n              [\n                0.22678124904632568,\n                0.5334166884422302,\n                0.365359365940094,\n                0.7781041860580444\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000045089.jpg",\n        "coco_id": 45089,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 204714,\n          "object_id": 57887,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_45089_204714_57887_1761374_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1761374,\n          "object_id": 57887,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.3438437581062317,\n                0.15718969702720642,\n                0.47292187809944153,\n                0.5866276621818542\n              ],\n              [\n                0.2955625057220459,\n                0.23772834241390228,\n                0.4531562626361847,\n                0.846908688545227\n              ],\n              [\n                0.2955625057220459,\n                0.15718969702720642,\n                0.47292187809944153,\n                0.846908688545227\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.9342812299728394,\n                0.38932085037231445,\n                1.0,\n                0.8759719133377075\n              ],\n              [\n                0.2955625057220459,\n                0.23772834241390228,\n                0.4531562626361847,\n                0.846908688545227\n              ],\n              [\n                0.2955625057220459,\n                0.23772834241390228,\n                1.0,\n                0.8759719133377075\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000045089.jpg",\n        "coco_id": 45089,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "horse",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2151556,\n          "object_id": 60387,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_45089_2151556_60387_1761374_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1761374,\n          "object_id": 60387,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.9342812299728394,\n                0.38932085037231445,\n                1.0,\n                0.8759719133377075\n              ],\n              [\n                0.6112343668937683,\n                0.2752693295478821,\n                0.7475312352180481,\n                0.8817095756530762\n              ],\n              [\n                0.6112343668937683,\n                0.2752693295478821,\n                1.0,\n                0.8817095756530762\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a horse",\n            "bounds": [\n              [\n                0.5807187557220459,\n                0.1293676793575287,\n                0.7640312314033508,\n                0.6175878047943115\n              ],\n              [\n                0.6112343668937683,\n                0.2752693295478821,\n                0.7475312352180481,\n                0.8817095756530762\n              ],\n              [\n                0.5807187557220459,\n                0.1293676793575287,\n                0.7640312314033508,\n                0.8817095756530762\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000452457.jpg",\n        "coco_id": 452457,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 446822,\n          "object_id": 634012,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_452457_446822_634012_453742_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 453742,\n          "object_id": 634012,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.07975000143051147,\n                0.16929411888122559,\n                0.2848281264305115,\n                0.691811740398407\n              ],\n              [\n                0.0996718779206276,\n                0.3271999955177307,\n                0.16837500035762787,\n                0.39814117550849915\n              ],\n              [\n                0.07975000143051147,\n                0.16929411888122559,\n                0.2848281264305115,\n                0.691811740398407\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.48060938715934753,\n                0.23703528940677643,\n                0.7590625286102295,\n                0.671999990940094\n              ],\n              [\n                0.0996718779206276,\n                0.3271999955177307,\n                0.16837500035762787,\n                0.39814117550849915\n              ],\n              [\n                0.0996718779206276,\n                0.23703528940677643,\n                0.7590625286102295,\n                0.671999990940094\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000453979.jpg",\n        "coco_id": 453979,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "laptop",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 445754,\n          "object_id": 1098181,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_453979_445754_1098181_427756_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 427756,\n          "object_id": 1098181,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a laptop",\n            "bounds": [\n              [\n                0.023900000378489494,\n                0.7011246681213379,\n                0.8749799728393555,\n                0.9033740758895874\n              ],\n              [\n                0.14390000700950623,\n                0.5868459939956665,\n                0.30570000410079956,\n                0.793838620185852\n              ],\n              [\n                0.023900000378489494,\n                0.5868459939956665,\n                0.8749799728393555,\n                0.9033740758895874\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a laptop",\n            "bounds": [\n              [\n                0.6702799797058105,\n                0.10102689266204834,\n                0.9380199909210205,\n                0.9091442823410034\n              ],\n              [\n                0.14390000700950623,\n                0.5868459939956665,\n                0.30570000410079956,\n                0.793838620185852\n              ],\n              [\n                0.14390000700950623,\n                0.10102689266204834,\n                0.9380199909210205,\n                0.9091442823410034\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000004843.jpg",\n        "coco_id": 4843,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "umbrella",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 448671,\n          "object_id": 284275,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_4843_448671_284275_541528_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 541528,\n          "object_id": 284275,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.32360416650772095,\n                0.3820156157016754,\n                0.5363333225250244,\n                0.8764062523841858\n              ],\n              [\n                0.10720833390951157,\n                0.14714062213897705,\n                0.7454166412353516,\n                0.6999531388282776\n              ],\n              [\n                0.10720833390951157,\n                0.14714062213897705,\n                0.7454166412353516,\n                0.8764062523841858\n              ]\n            ]\n          },\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.06591666489839554,\n                0.269656240940094,\n                0.35356250405311584,\n                0.7775312662124634\n              ],\n              [\n                0.10720833390951157,\n                0.14714062213897705,\n                0.7454166412353516,\n                0.6999531388282776\n              ],\n              [\n                0.06591666489839554,\n                0.14714062213897705,\n                0.7454166412353516,\n                0.7775312662124634\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000484646.jpg",\n        "coco_id": 484646,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 497591,\n          "object_id": 647868,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_484646_497591_647868_427595_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 427595,\n          "object_id": 647868,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.6426399946212769,\n                0.469733327627182,\n                0.7158799767494202,\n                0.6961066722869873\n              ],\n              [\n                0.692080020904541,\n                0.5313066840171814,\n                0.7178800106048584,\n                0.6678666472434998\n              ],\n              [\n                0.6426399946212769,\n                0.469733327627182,\n                0.7178800106048584,\n                0.6961066722869873\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.22247999906539917,\n                0.49663999676704407,\n                0.30000001192092896,\n                0.7078666687011719\n              ],\n              [\n                0.692080020904541,\n                0.5313066840171814,\n                0.7178800106048584,\n                0.6678666472434998\n              ],\n              [\n                0.22247999906539917,\n                0.49663999676704407,\n                0.7178800106048584,\n                0.7078666687011719\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000495476.jpg",\n        "coco_id": 495476,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "elephant",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1223724,\n          "object_id": 581518,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_495476_1223724_581518_528923_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 528923,\n          "object_id": 581518,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.2637656331062317,\n                0.4851406216621399,\n                0.39774999022483826,\n                0.817984402179718\n              ],\n              [\n                0.3040468692779541,\n                0.3074375092983246,\n                0.6531562209129333,\n                0.8862656354904175\n              ],\n              [\n                0.2637656331062317,\n                0.3074375092983246,\n                0.6531562209129333,\n                0.8862656354904175\n              ]\n            ]\n          },\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.32718750834465027,\n                0.13585937023162842,\n                0.43028125166893005,\n                0.39253124594688416\n              ],\n              [\n                0.3040468692779541,\n                0.3074375092983246,\n                0.6531562209129333,\n                0.8862656354904175\n              ],\n              [\n                0.3040468692779541,\n                0.13585937023162842,\n                0.6531562209129333,\n                0.8862656354904175\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000495476.jpg",\n        "coco_id": 495476,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1234948,\n          "object_id": 1823893,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_495476_1234948_1823893_528923_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 528923,\n          "object_id": 1823893,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.2637656331062317,\n                0.4851406216621399,\n                0.39774999022483826,\n                0.817984402179718\n              ],\n              [\n                0.2146874964237213,\n                0.5888437628746033,\n                0.23921875655651093,\n                0.6354687213897705\n              ],\n              [\n                0.2146874964237213,\n                0.4851406216621399,\n                0.39774999022483826,\n                0.817984402179718\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.16593749821186066,\n                0.5517656207084656,\n                0.2729843854904175,\n                0.8132343888282776\n              ],\n              [\n                0.2146874964237213,\n                0.5888437628746033,\n                0.23921875655651093,\n                0.6354687213897705\n              ],\n              [\n                0.16593749821186066,\n                0.5517656207084656,\n                0.2729843854904175,\n                0.8132343888282776\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000495476.jpg",\n        "coco_id": 495476,\n        "caption_split": "train",\n        "predicate": "riding",\n        "object_label": "elephant",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 427934,\n          "object_id": 581518,\n          "action": "ride",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_495476_427934_581518_528923_ride",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 528923,\n          "object_id": 581518,\n          "action": "ride",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.2637656331062317,\n                0.4851406216621399,\n                0.39774999022483826,\n                0.817984402179718\n              ],\n              [\n                0.3040468692779541,\n                0.3074375092983246,\n                0.6531562209129333,\n                0.8862656354904175\n              ],\n              [\n                0.2637656331062317,\n                0.3074375092983246,\n                0.6531562209129333,\n                0.8862656354904175\n              ]\n            ]\n          },\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.41337499022483826,\n                0.14678125083446503,\n                0.5726875066757202,\n                0.41110938787460327\n              ],\n              [\n                0.3040468692779541,\n                0.3074375092983246,\n                0.6531562209129333,\n                0.8862656354904175\n              ],\n              [\n                0.3040468692779541,\n                0.14678125083446503,\n                0.6531562209129333,\n                0.8862656354904175\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000495476.jpg",\n        "coco_id": 495476,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "elephant",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 427934,\n          "object_id": 581518,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_495476_427934_581518_528923_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 528923,\n          "object_id": 581518,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on an elephant",\n            "bounds": [\n              [\n                0.41337499022483826,\n                0.14678125083446503,\n                0.5726875066757202,\n                0.41110938787460327\n              ],\n              [\n                0.3040468692779541,\n                0.3074375092983246,\n                0.6531562209129333,\n                0.8862656354904175\n              ],\n              [\n                0.3040468692779541,\n                0.14678125083446503,\n                0.6531562209129333,\n                0.8862656354904175\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on an elephant",\n            "bounds": [\n              [\n                0.2637656331062317,\n                0.4851406216621399,\n                0.39774999022483826,\n                0.817984402179718\n              ],\n              [\n                0.3040468692779541,\n                0.3074375092983246,\n                0.6531562209129333,\n                0.8862656354904175\n              ],\n              [\n                0.2637656331062317,\n                0.3074375092983246,\n                0.6531562209129333,\n                0.8862656354904175\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000501814.jpg",\n        "coco_id": 501814,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "book",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 518409,\n          "object_id": 1137358,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_501814_518409_1137358_550666_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 550666,\n          "object_id": 1137358,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a book",\n            "bounds": [\n              [\n                0.5713281035423279,\n                0.4943764805793762,\n                0.9811406135559082,\n                0.9853647351264954\n              ],\n              [\n                0.3347499966621399,\n                0.5998352766036987,\n                0.5657656192779541,\n                0.8849647045135498\n              ],\n              [\n                0.3347499966621399,\n                0.4943764805793762,\n                0.9811406135559082,\n                0.9853647351264954\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a book",\n            "bounds": [\n              [\n                0.0,\n                0.29159998893737793,\n                0.42710936069488525,\n                1.0\n              ],\n              [\n                0.3347499966621399,\n                0.5998352766036987,\n                0.5657656192779541,\n                0.8849647045135498\n              ],\n              [\n                0.0,\n                0.29159998893737793,\n                0.5657656192779541,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000503689.jpg",\n        "coco_id": 503689,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 513679,\n          "object_id": 629709,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_503689_513679_629709_520656_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 520656,\n          "object_id": 629709,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.2588437497615814,\n                0.23283372819423676,\n                0.5464218854904175,\n                0.9886885285377502\n              ],\n              [\n                0.5402968525886536,\n                0.42007026076316833,\n                0.70703125,\n                0.48946136236190796\n              ],\n              [\n                0.2588437497615814,\n                0.23283372819423676,\n                0.70703125,\n                0.9886885285377502\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.4701874852180481,\n                0.1973770558834076,\n                0.6439062356948853,\n                0.8779156804084778\n              ],\n              [\n                0.5402968525886536,\n                0.42007026076316833,\n                0.70703125,\n                0.48946136236190796\n              ],\n              [\n                0.4701874852180481,\n                0.1973770558834076,\n                0.70703125,\n                0.8779156804084778\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000504122.jpg",\n        "coco_id": 504122,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "donut",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 2165999,\n          "object_id": 1572288,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_504122_2165999_1572288_435263_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 435263,\n          "object_id": 1572288,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a donut",\n            "bounds": [\n              [\n                0.0017343750223517418,\n                0.0,\n                0.3821406364440918,\n                0.7610280513763428\n              ],\n              [\n                0.14862500131130219,\n                0.22098131477832794,\n                0.22037500143051147,\n                0.30485981702804565\n              ],\n              [\n                0.0017343750223517418,\n                0.0,\n                0.3821406364440918,\n                0.7610280513763428\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a donut",\n            "bounds": [\n              [\n                0.7003124952316284,\n                0.04943925142288208,\n                0.9993749856948853,\n                0.6382009387016296\n              ],\n              [\n                0.14862500131130219,\n                0.22098131477832794,\n                0.22037500143051147,\n                0.30485981702804565\n              ],\n              [\n                0.14862500131130219,\n                0.04943925142288208,\n                0.9993749856948853,\n                0.6382009387016296\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000504122.jpg",\n        "coco_id": 504122,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "donut",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2165999,\n          "object_id": 1572288,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_504122_2165999_1572288_435263_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 435263,\n          "object_id": 1572288,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a donut",\n            "bounds": [\n              [\n                0.7003124952316284,\n                0.04943925142288208,\n                0.9993749856948853,\n                0.6382009387016296\n              ],\n              [\n                0.14862500131130219,\n                0.22098131477832794,\n                0.22037500143051147,\n                0.30485981702804565\n              ],\n              [\n                0.14862500131130219,\n                0.04943925142288208,\n                0.9993749856948853,\n                0.6382009387016296\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a donut",\n            "bounds": [\n              [\n                0.0017343750223517418,\n                0.0,\n                0.3821406364440918,\n                0.7610280513763428\n              ],\n              [\n                0.14862500131130219,\n                0.22098131477832794,\n                0.22037500143051147,\n                0.30485981702804565\n              ],\n              [\n                0.0017343750223517418,\n                0.0,\n                0.3821406364440918,\n                0.7610280513763428\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000506417.jpg",\n        "coco_id": 506417,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1203873,\n          "object_id": 631078,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_506417_1203873_631078_1734826_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1734826,\n          "object_id": 631078,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.23076562583446503,\n                0.0966273620724678,\n                0.562749981880188,\n                0.9460613131523132\n              ],\n              [\n                0.03620312362909317,\n                0.0328773595392704,\n                0.23990625143051147,\n                0.2012735903263092\n              ],\n              [\n                0.03620312362909317,\n                0.0328773595392704,\n                0.562749981880188,\n                0.9460613131523132\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.6450937390327454,\n                0.31683963537216187,\n                0.9999687671661377,\n                0.9971933960914612\n              ],\n              [\n                0.03620312362909317,\n                0.0328773595392704,\n                0.23990625143051147,\n                0.2012735903263092\n              ],\n              [\n                0.03620312362909317,\n                0.0328773595392704,\n                0.9999687671661377,\n                0.9971933960914612\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000515937.jpg",\n        "coco_id": 515937,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "backpack",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1752163,\n          "object_id": 1169878,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_515937_1752163_1169878_514202_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 514202,\n          "object_id": 1169878,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.31368544697761536,\n                0.4139062464237213,\n                0.5420892238616943,\n                0.7745000123977661\n              ],\n              [\n                0.48551642894744873,\n                0.47560936212539673,\n                0.6236385107040405,\n                0.5890781283378601\n              ],\n              [\n                0.31368544697761536,\n                0.4139062464237213,\n                0.6236385107040405,\n                0.7745000123977661\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a backpack",\n            "bounds": [\n              [\n                0.5248591303825378,\n                0.44446873664855957,\n                0.7510094046592712,\n                0.7403437495231628\n              ],\n              [\n                0.48551642894744873,\n                0.47560936212539673,\n                0.6236385107040405,\n                0.5890781283378601\n              ],\n              [\n                0.48551642894744873,\n                0.44446873664855957,\n                0.7510094046592712,\n                0.7403437495231628\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000520569.jpg",\n        "coco_id": 520569,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 443921,\n          "object_id": 1184242,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_520569_443921_1184242_472543_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 472543,\n          "object_id": 1184242,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.43017613887786865,\n                0.6825156211853027,\n                0.5568689107894897,\n                0.9474531412124634\n              ],\n              [\n                0.3072211444377899,\n                0.8272500038146973,\n                0.4653424620628357,\n                0.9555781483650208\n              ],\n              [\n                0.3072211444377899,\n                0.6825156211853027,\n                0.5568689107894897,\n                0.9555781483650208\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.02594911865890026,\n                0.6249531507492065,\n                0.16757339239120483,\n                0.9006718993186951\n              ],\n              [\n                0.3072211444377899,\n                0.8272500038146973,\n                0.4653424620628357,\n                0.9555781483650208\n              ],\n              [\n                0.02594911865890026,\n                0.6249531507492065,\n                0.4653424620628357,\n                0.9555781483650208\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000520569.jpg",\n        "coco_id": 520569,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 443921,\n          "object_id": 1184242,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_520569_443921_1184242_472543_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 472543,\n          "object_id": 1184242,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.02594911865890026,\n                0.6249531507492065,\n                0.16757339239120483,\n                0.9006718993186951\n              ],\n              [\n                0.3072211444377899,\n                0.8272500038146973,\n                0.4653424620628357,\n                0.9555781483650208\n              ],\n              [\n                0.02594911865890026,\n                0.6249531507492065,\n                0.4653424620628357,\n                0.9555781483650208\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.43017613887786865,\n                0.6825156211853027,\n                0.5568689107894897,\n                0.9474531412124634\n              ],\n              [\n                0.3072211444377899,\n                0.8272500038146973,\n                0.4653424620628357,\n                0.9555781483650208\n              ],\n              [\n                0.3072211444377899,\n                0.6825156211853027,\n                0.5568689107894897,\n                0.9555781483650208\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000520569.jpg",\n        "coco_id": 520569,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 452258,\n          "object_id": 1185208,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_520569_452258_1185208_472543_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 472543,\n          "object_id": 1185208,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.6723679304122925,\n                0.751296877861023,\n                0.8547554016113281,\n                0.9872812628746033\n              ],\n              [\n                0.5794911980628967,\n                0.8752031326293945,\n                0.711154580116272,\n                0.9782031178474426\n              ],\n              [\n                0.5794911980628967,\n                0.751296877861023,\n                0.8547554016113281,\n                0.9872812628746033\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a suitcase",\n            "bounds": [\n              [\n                0.02594911865890026,\n                0.6249531507492065,\n                0.16757339239120483,\n                0.9006718993186951\n              ],\n              [\n                0.5794911980628967,\n                0.8752031326293945,\n                0.711154580116272,\n                0.9782031178474426\n              ],\n              [\n                0.02594911865890026,\n                0.6249531507492065,\n                0.711154580116272,\n                0.9782031178474426\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000520569.jpg",\n        "coco_id": 520569,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 452258,\n          "object_id": 1185208,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_520569_452258_1185208_472543_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 472543,\n          "object_id": 1185208,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.6723679304122925,\n                0.751296877861023,\n                0.8547554016113281,\n                0.9872812628746033\n              ],\n              [\n                0.5794911980628967,\n                0.8752031326293945,\n                0.711154580116272,\n                0.9782031178474426\n              ],\n              [\n                0.5794911980628967,\n                0.751296877861023,\n                0.8547554016113281,\n                0.9872812628746033\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.02594911865890026,\n                0.6249531507492065,\n                0.16757339239120483,\n                0.9006718993186951\n              ],\n              [\n                0.5794911980628967,\n                0.8752031326293945,\n                0.711154580116272,\n                0.9782031178474426\n              ],\n              [\n                0.02594911865890026,\n                0.6249531507492065,\n                0.711154580116272,\n                0.9782031178474426\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000522338.jpg",\n        "coco_id": 522338,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "donut",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 501365,\n          "object_id": 1078831,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_522338_501365_1078831_484710_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 484710,\n          "object_id": 1078831,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a donut",\n            "bounds": [\n              [\n                0.005062500014901161,\n                0.20897915959358215,\n                0.5797812342643738,\n                0.9887708425521851\n              ],\n              [\n                0.3775312602519989,\n                0.6966249942779541,\n                0.5427031517028809,\n                0.8741666674613953\n              ],\n              [\n                0.005062500014901161,\n                0.20897915959358215,\n                0.5797812342643738,\n                0.9887708425521851\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a donut",\n            "bounds": [\n              [\n                0.5561875104904175,\n                0.1146041676402092,\n                1.0,\n                0.9865208268165588\n              ],\n              [\n                0.3775312602519989,\n                0.6966249942779541,\n                0.5427031517028809,\n                0.8741666674613953\n              ],\n              [\n                0.3775312602519989,\n                0.1146041676402092,\n                1.0,\n                0.9865208268165588\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000052440.jpg",\n        "coco_id": 52440,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "person",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 493917,\n          "object_id": 1706584,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_52440_493917_1706584_1706584_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1706584,\n          "object_id": 1706584,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.3104887902736664,\n                0.04426562413573265,\n                0.5887576341629028,\n                0.5184218883514404\n              ],\n              [\n                0.011710794642567635,\n                0.06517187505960464,\n                0.3427087664604187,\n                0.48539063334465027\n              ],\n              [\n                0.011710794642567635,\n                0.04426562413573265,\n                0.5887576341629028,\n                0.5184218883514404\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.011710794642567635,\n                0.06517187505960464,\n                0.3427087664604187,\n                0.48539063334465027\n              ],\n              [\n                0.011710794642567635,\n                0.06517187505960464,\n                0.3427087664604187,\n                0.48539063334465027\n              ],\n              [\n                0.011710794642567635,\n                0.06517187505960464,\n                0.3427087664604187,\n                0.48539063334465027\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000528712.jpg",\n        "coco_id": 528712,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 225265,\n          "object_id": 1436105,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_528712_225265_1436105_216869_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 216869,\n          "object_id": 1436105,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.3244599997997284,\n                0.16581332683563232,\n                0.4592199921607971,\n                0.7466933131217957\n              ],\n              [\n                0.2537800073623657,\n                0.5835999846458435,\n                0.3059599995613098,\n                0.724399983882904\n              ],\n              [\n                0.2537800073623657,\n                0.16581332683563232,\n                0.4592199921607971,\n                0.7466933131217957\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.06573999673128128,\n                0.15279999375343323,\n                0.30338001251220703,\n                0.905626654624939\n              ],\n              [\n                0.2537800073623657,\n                0.5835999846458435,\n                0.3059599995613098,\n                0.724399983882904\n              ],\n              [\n                0.06573999673128128,\n                0.15279999375343323,\n                0.3059599995613098,\n                0.905626654624939\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000528712.jpg",\n        "coco_id": 528712,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 258505,\n          "object_id": 1437125,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_528712_258505_1437125_216869_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 216869,\n          "object_id": 1437125,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.00675999978557229,\n                0.1801866739988327,\n                0.11655999720096588,\n                0.7364799976348877\n              ],\n              [\n                0.09109999984502792,\n                0.4949600100517273,\n                0.13458000123500824,\n                0.5993066430091858\n              ],\n              [\n                0.00675999978557229,\n                0.1801866739988327,\n                0.13458000123500824,\n                0.7364799976348877\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.3244599997997284,\n                0.16581332683563232,\n                0.4592199921607971,\n                0.7466933131217957\n              ],\n              [\n                0.09109999984502792,\n                0.4949600100517273,\n                0.13458000123500824,\n                0.5993066430091858\n              ],\n              [\n                0.09109999984502792,\n                0.16581332683563232,\n                0.4592199921607971,\n                0.7466933131217957\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000528712.jpg",\n        "coco_id": 528712,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "handbag",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 258505,\n          "object_id": 1437125,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_528712_258505_1437125_225265_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 225265,\n          "object_id": 1437125,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.00675999978557229,\n                0.1801866739988327,\n                0.11655999720096588,\n                0.7364799976348877\n              ],\n              [\n                0.09109999984502792,\n                0.4949600100517273,\n                0.13458000123500824,\n                0.5993066430091858\n              ],\n              [\n                0.00675999978557229,\n                0.1801866739988327,\n                0.13458000123500824,\n                0.7364799976348877\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a handbag",\n            "bounds": [\n              [\n                0.06573999673128128,\n                0.15279999375343323,\n                0.30338001251220703,\n                0.905626654624939\n              ],\n              [\n                0.09109999984502792,\n                0.4949600100517273,\n                0.13458000123500824,\n                0.5993066430091858\n              ],\n              [\n                0.06573999673128128,\n                0.15279999375343323,\n                0.30338001251220703,\n                0.905626654624939\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000562826.jpg",\n        "coco_id": 562826,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "wine glass",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1214890,\n          "object_id": 661935,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_562826_1214890_661935_453037_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 453037,\n          "object_id": 661935,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a wine glass",\n            "bounds": [\n              [\n                0.2140468806028366,\n                0.029208334162831306,\n                0.8730312585830688,\n                0.979770839214325\n              ],\n              [\n                0.45506250858306885,\n                0.27122917771339417,\n                0.5814531445503235,\n                0.6083124876022339\n              ],\n              [\n                0.2140468806028366,\n                0.029208334162831306,\n                0.8730312585830688,\n                0.979770839214325\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a wine glass",\n            "bounds": [\n              [\n                0.6219062209129333,\n                0.03595833480358124,\n                0.9977499842643738,\n                0.9213541746139526\n              ],\n              [\n                0.45506250858306885,\n                0.27122917771339417,\n                0.5814531445503235,\n                0.6083124876022339\n              ],\n              [\n                0.45506250858306885,\n                0.03595833480358124,\n                0.9977499842643738,\n                0.9213541746139526\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000569301.jpg",\n        "coco_id": 569301,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 533744,\n          "object_id": 628074,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_569301_533744_628074_533964_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 533964,\n          "object_id": 628074,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.5546093583106995,\n                0.00526932068169117,\n                0.8026562333106995,\n                0.8693910837173462\n              ],\n              [\n                0.40031251311302185,\n                0.5618032813072205,\n                0.6581875085830688,\n                0.9123653173446655\n              ],\n              [\n                0.40031251311302185,\n                0.00526932068169117,\n                0.8026562333106995,\n                0.9123653173446655\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.3734999895095825,\n                0.09651053696870804,\n                0.646109402179718,\n                0.988149881362915\n              ],\n              [\n                0.40031251311302185,\n                0.5618032813072205,\n                0.6581875085830688,\n                0.9123653173446655\n              ],\n              [\n                0.3734999895095825,\n                0.09651053696870804,\n                0.6581875085830688,\n                0.988149881362915\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000573287.jpg",\n        "coco_id": 573287,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "remote",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2156158,\n          "object_id": 1113575,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_573287_2156158_1113575_452470_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 452470,\n          "object_id": 1113575,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.12476562708616257,\n                0.4163958430290222,\n                0.513296902179718,\n                0.9996458292007446\n              ],\n              [\n                0.16810937225818634,\n                0.6571041941642761,\n                0.18610937893390656,\n                0.6828749775886536\n              ],\n              [\n                0.12476562708616257,\n                0.4163958430290222,\n                0.513296902179718,\n                0.9996458292007446\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.0,\n                0.421979159116745,\n                0.33899998664855957,\n                0.9987708330154419\n              ],\n              [\n                0.16810937225818634,\n                0.6571041941642761,\n                0.18610937893390656,\n                0.6828749775886536\n              ],\n              [\n                0.0,\n                0.421979159116745,\n                0.33899998664855957,\n                0.9987708330154419\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000573287.jpg",\n        "coco_id": 573287,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cup",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2160137,\n          "object_id": 1504317,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_573287_2160137_1504317_452470_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 452470,\n          "object_id": 1504317,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.12476562708616257,\n                0.4163958430290222,\n                0.513296902179718,\n                0.9996458292007446\n              ],\n              [\n                0.8651562333106995,\n                0.53927081823349,\n                0.9031718969345093,\n                0.596791684627533\n              ],\n              [\n                0.12476562708616257,\n                0.4163958430290222,\n                0.9031718969345093,\n                0.9996458292007446\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cup",\n            "bounds": [\n              [\n                0.6874218583106995,\n                0.40314581990242004,\n                1.0,\n                0.901395857334137\n              ],\n              [\n                0.8651562333106995,\n                0.53927081823349,\n                0.9031718969345093,\n                0.596791684627533\n              ],\n              [\n                0.6874218583106995,\n                0.40314581990242004,\n                1.0,\n                0.901395857334137\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000573287.jpg",\n        "coco_id": 573287,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "remote",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 509215,\n          "object_id": 1108093,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_573287_509215_1108093_452470_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 452470,\n          "object_id": 1108093,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.3800624907016754,\n                0.35360416769981384,\n                0.6638437509536743,\n                0.9053958058357239\n              ],\n              [\n                0.5419843792915344,\n                0.5574374794960022,\n                0.5683281421661377,\n                0.5872083306312561\n              ],\n              [\n                0.3800624907016754,\n                0.35360416769981384,\n                0.6638437509536743,\n                0.9053958058357239\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a remote",\n            "bounds": [\n              [\n                0.12476562708616257,\n                0.4163958430290222,\n                0.513296902179718,\n                0.9996458292007446\n              ],\n              [\n                0.5419843792915344,\n                0.5574374794960022,\n                0.5683281421661377,\n                0.5872083306312561\n              ],\n              [\n                0.12476562708616257,\n                0.4163958430290222,\n                0.5683281421661377,\n                0.9996458292007446\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000057614.jpg",\n        "coco_id": 57614,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "knife",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 469225,\n          "object_id": 698922,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_57614_469225_698922_464976_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 464976,\n          "object_id": 698922,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.43479686975479126,\n                0.29625293612480164,\n                0.7061718702316284,\n                0.9546838402748108\n              ],\n              [\n                0.3523593842983246,\n                0.37081965804100037,\n                0.4408593773841858,\n                0.4123653471469879\n              ],\n              [\n                0.3523593842983246,\n                0.29625293612480164,\n                0.7061718702316284,\n                0.9546838402748108\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.5980625152587891,\n                0.35023418068885803,\n                0.8114374876022339,\n                0.9673536419868469\n              ],\n              [\n                0.3523593842983246,\n                0.37081965804100037,\n                0.4408593773841858,\n                0.4123653471469879\n              ],\n              [\n                0.3523593842983246,\n                0.35023418068885803,\n                0.8114374876022339,\n                0.9673536419868469\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000579414.jpg",\n        "coco_id": 579414,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1293483,\n          "object_id": 630775,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_579414_1293483_630775_1727161_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1727161,\n          "object_id": 630775,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.5255625247955322,\n                0.31168749928474426,\n                0.7757499814033508,\n                0.729770839214325\n              ],\n              [\n                0.4535781145095825,\n                0.3407500088214874,\n                0.5341406464576721,\n                0.39535418152809143\n              ],\n              [\n                0.4535781145095825,\n                0.31168749928474426,\n                0.7757499814033508,\n                0.729770839214325\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.25559374690055847,\n                0.4894166588783264,\n                0.4652499854564667,\n                0.7447291612625122\n              ],\n              [\n                0.4535781145095825,\n                0.3407500088214874,\n                0.5341406464576721,\n                0.39535418152809143\n              ],\n              [\n                0.25559374690055847,\n                0.3407500088214874,\n                0.5341406464576721,\n                0.7447291612625122\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000007049.jpg",\n        "coco_id": 7049,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "umbrella",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 512841,\n          "object_id": 281261,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_7049_512841_281261_517921_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 517921,\n          "object_id": 281261,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.7409836053848267,\n                0.14606249332427979,\n                1.0,\n                0.9865156412124634\n              ],\n              [\n                0.0,\n                0.13189062476158142,\n                0.8319438099861145,\n                0.3229062557220459\n              ],\n              [\n                0.0,\n                0.13189062476158142,\n                1.0,\n                0.9865156412124634\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying an umbrella",\n            "bounds": [\n              [\n                0.0338875874876976,\n                0.21457812190055847,\n                0.5129976868629456,\n                1.0\n              ],\n              [\n                0.0,\n                0.13189062476158142,\n                0.8319438099861145,\n                0.3229062557220459\n              ],\n              [\n                0.0,\n                0.13189062476158142,\n                0.8319438099861145,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000007049.jpg",\n        "coco_id": 7049,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "umbrella",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 512841,\n          "object_id": 281261,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_7049_512841_281261_517921_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 517921,\n          "object_id": 281261,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.7409836053848267,\n                0.14606249332427979,\n                1.0,\n                0.9865156412124634\n              ],\n              [\n                0.0,\n                0.13189062476158142,\n                0.8319438099861145,\n                0.3229062557220459\n              ],\n              [\n                0.0,\n                0.13189062476158142,\n                1.0,\n                0.9865156412124634\n              ]\n            ]\n          },\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.0338875874876976,\n                0.21457812190055847,\n                0.5129976868629456,\n                1.0\n              ],\n              [\n                0.0,\n                0.13189062476158142,\n                0.8319438099861145,\n                0.3229062557220459\n              ],\n              [\n                0.0,\n                0.13189062476158142,\n                0.8319438099861145,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000072002.jpg",\n        "coco_id": 72002,\n        "caption_split": "train",\n        "predicate": "carrying",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1211286,\n          "object_id": 602028,\n          "action": "carry",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_72002_1211286_602028_1265918_carry",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1265918,\n          "object_id": 602028,\n          "action": "carry",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person carrying a frisbee",\n            "bounds": [\n              [\n                0.38243749737739563,\n                0.315607488155365,\n                0.4765625,\n                0.8925233483314514\n              ],\n              [\n                0.43056249618530273,\n                0.42761683464050293,\n                0.45631250739097595,\n                0.5065420269966125\n              ],\n              [\n                0.38243749737739563,\n                0.315607488155365,\n                0.4765625,\n                0.8925233483314514\n              ]\n            ]\n          },\n          {\n            "text": "a person carrying a frisbee",\n            "bounds": [\n              [\n                0.7981250286102295,\n                0.29018691182136536,\n                0.8920156359672546,\n                0.9032710194587708\n              ],\n              [\n                0.43056249618530273,\n                0.42761683464050293,\n                0.45631250739097595,\n                0.5065420269966125\n              ],\n              [\n                0.43056249618530273,\n                0.29018691182136536,\n                0.8920156359672546,\n                0.9032710194587708\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000072002.jpg",\n        "coco_id": 72002,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1211286,\n          "object_id": 602028,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_72002_1211286_602028_1690562_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1690562,\n          "object_id": 602028,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.38243749737739563,\n                0.315607488155365,\n                0.4765625,\n                0.8925233483314514\n              ],\n              [\n                0.43056249618530273,\n                0.42761683464050293,\n                0.45631250739097595,\n                0.5065420269966125\n              ],\n              [\n                0.38243749737739563,\n                0.315607488155365,\n                0.4765625,\n                0.8925233483314514\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.7406562566757202,\n                0.3350934684276581,\n                0.7939531207084656,\n                0.7689252495765686\n              ],\n              [\n                0.43056249618530273,\n                0.42761683464050293,\n                0.45631250739097595,\n                0.5065420269966125\n              ],\n              [\n                0.43056249618530273,\n                0.3350934684276581,\n                0.7939531207084656,\n                0.7689252495765686\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000072002.jpg",\n        "coco_id": 72002,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1265918,\n          "object_id": 604578,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_72002_1265918_604578_1690562_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1690562,\n          "object_id": 604578,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.7981250286102295,\n                0.29018691182136536,\n                0.8920156359672546,\n                0.9032710194587708\n              ],\n              [\n                0.8280937671661377,\n                0.39920559525489807,\n                0.852218747138977,\n                0.47864484786987305\n              ],\n              [\n                0.7981250286102295,\n                0.29018691182136536,\n                0.8920156359672546,\n                0.9032710194587708\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.7406562566757202,\n                0.3350934684276581,\n                0.7939531207084656,\n                0.7689252495765686\n              ],\n              [\n                0.8280937671661377,\n                0.39920559525489807,\n                0.852218747138977,\n                0.47864484786987305\n              ],\n              [\n                0.7406562566757202,\n                0.3350934684276581,\n                0.852218747138977,\n                0.7689252495765686\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000073639.jpg",\n        "coco_id": 73639,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1218113,\n          "object_id": 378678,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_73639_1218113_378678_1295465_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1295465,\n          "object_id": 378678,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.33125001192092896,\n                0.4217798709869385,\n                0.574999988079071,\n                0.7800936698913574\n              ],\n              [\n                0.349421888589859,\n                0.7092739939689636,\n                0.5345625281333923,\n                0.8529508113861084\n              ],\n              [\n                0.33125001192092896,\n                0.4217798709869385,\n                0.574999988079071,\n                0.8529508113861084\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.8935781121253967,\n                0.18426229059696198,\n                1.0,\n                0.9640515446662903\n              ],\n              [\n                0.349421888589859,\n                0.7092739939689636,\n                0.5345625281333923,\n                0.8529508113861084\n              ],\n              [\n                0.349421888589859,\n                0.18426229059696198,\n                1.0,\n                0.9640515446662903\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000073639.jpg",\n        "coco_id": 73639,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 2153088,\n          "object_id": 376551,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_73639_2153088_376551_1295465_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 1295465,\n          "object_id": 376551,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.8935781121253967,\n                0.18426229059696198,\n                1.0,\n                0.9640515446662903\n              ],\n              [\n                0.5877187252044678,\n                0.730327844619751,\n                0.7076718807220459,\n                0.8853864073753357\n              ],\n              [\n                0.5877187252044678,\n                0.18426229059696198,\n                1.0,\n                0.9640515446662903\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.5676562786102295,\n                0.43334895372390747,\n                0.8168749809265137,\n                0.7475410103797913\n              ],\n              [\n                0.5877187252044678,\n                0.730327844619751,\n                0.7076718807220459,\n                0.8853864073753357\n              ],\n              [\n                0.5676562786102295,\n                0.43334895372390747,\n                0.8168749809265137,\n                0.8853864073753357\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000084550.jpg",\n        "coco_id": 84550,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 187574,\n          "object_id": 327913,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_84550_187574_327913_198359_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 198359,\n          "object_id": 327913,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.5064600110054016,\n                0.3526933193206787,\n                0.6370999813079834,\n                0.9741866588592529\n              ],\n              [\n                0.5676400065422058,\n                0.40405333042144775,\n                0.5976799726486206,\n                0.4309333264827728\n              ],\n              [\n                0.5064600110054016,\n                0.3526933193206787,\n                0.6370999813079834,\n                0.9741866588592529\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.08258000016212463,\n                0.3322133421897888,\n                0.3202199935913086,\n                0.986133337020874\n              ],\n              [\n                0.5676400065422058,\n                0.40405333042144775,\n                0.5976799726486206,\n                0.4309333264827728\n              ],\n              [\n                0.08258000016212463,\n                0.3322133421897888,\n                0.5976799726486206,\n                0.986133337020874\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000084550.jpg",\n        "coco_id": 84550,\n        "caption_split": "train",\n        "predicate": "talking_on",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 187574,\n          "object_id": 327913,\n          "action": "talk_on_phone",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_84550_187574_327913_198359_talk_on_phone",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 198359,\n          "object_id": 327913,\n          "action": "talk_on_phone",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.08258000016212463,\n                0.3322133421897888,\n                0.3202199935913086,\n                0.986133337020874\n              ],\n              [\n                0.5676400065422058,\n                0.40405333042144775,\n                0.5976799726486206,\n                0.4309333264827728\n              ],\n              [\n                0.08258000016212463,\n                0.3322133421897888,\n                0.5976799726486206,\n                0.986133337020874\n              ]\n            ]\n          },\n          {\n            "text": "a person talking on a cell phone",\n            "bounds": [\n              [\n                0.5064600110054016,\n                0.3526933193206787,\n                0.6370999813079834,\n                0.9741866588592529\n              ],\n              [\n                0.5676400065422058,\n                0.40405333042144775,\n                0.5976799726486206,\n                0.4309333264827728\n              ],\n              [\n                0.5064600110054016,\n                0.3526933193206787,\n                0.6370999813079834,\n                0.9741866588592529\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000089181.jpg",\n        "coco_id": 89181,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 505347,\n          "object_id": 603484,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_89181_505347_603484_514884_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 514884,\n          "object_id": 603484,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.3330526351928711,\n                0.09437499940395355,\n                0.9272631406784058,\n                0.9887655973434448\n              ],\n              [\n                0.30294737219810486,\n                0.09485937654972076,\n                0.49876314401626587,\n                0.23371875286102295\n              ],\n              [\n                0.30294737219810486,\n                0.09437499940395355,\n                0.9272631406784058,\n                0.9887655973434448\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.049368422478437424,\n                0.3251250088214874,\n                0.5610789656639099,\n                1.0\n              ],\n              [\n                0.30294737219810486,\n                0.09485937654972076,\n                0.49876314401626587,\n                0.23371875286102295\n              ],\n              [\n                0.049368422478437424,\n                0.09485937654972076,\n                0.5610789656639099,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000090251.jpg",\n        "coco_id": 90251,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "cell phone",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 508208,\n          "object_id": 323082,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_90251_508208_323082_2162039_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2162039,\n          "object_id": 323082,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.0,\n                0.13765625655651093,\n                0.8136534094810486,\n                0.9870156049728394\n              ],\n              [\n                0.7778220176696777,\n                0.735953152179718,\n                0.8862997889518738,\n                0.8054062724113464\n              ],\n              [\n                0.0,\n                0.13765625655651093,\n                0.8862997889518738,\n                0.9870156049728394\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a cell phone",\n            "bounds": [\n              [\n                0.35444965958595276,\n                0.3063125014305115,\n                0.9992271661758423,\n                1.0\n              ],\n              [\n                0.7778220176696777,\n                0.735953152179718,\n                0.8862997889518738,\n                0.8054062724113464\n              ],\n              [\n                0.35444965958595276,\n                0.3063125014305115,\n                0.9992271661758423,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000090251.jpg",\n        "coco_id": 90251,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "chair",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 508208,\n          "object_id": 373403,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_90251_508208_373403_2162039_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 2162039,\n          "object_id": 373403,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.35444965958595276,\n                0.3063125014305115,\n                0.9992271661758423,\n                1.0\n              ],\n              [\n                0.19918033480644226,\n                0.3997812569141388,\n                0.4591100811958313,\n                0.5484218597412109\n              ],\n              [\n                0.19918033480644226,\n                0.3063125014305115,\n                0.9992271661758423,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a chair",\n            "bounds": [\n              [\n                0.0,\n                0.13765625655651093,\n                0.8136534094810486,\n                0.9870156049728394\n              ],\n              [\n                0.19918033480644226,\n                0.3997812569141388,\n                0.4591100811958313,\n                0.5484218597412109\n              ],\n              [\n                0.0,\n                0.13765625655651093,\n                0.8136534094810486,\n                0.9870156049728394\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000090862.jpg",\n        "coco_id": 90862,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "pizza",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 1760659,\n          "object_id": 1571440,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_90862_1760659_1571440_430129_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 430129,\n          "object_id": 1571440,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.28120312094688416,\n                0.11922916769981384,\n                0.5640156269073486,\n                0.6117291450500488\n              ],\n              [\n                0.2525937557220459,\n                0.4502291679382324,\n                0.36403125524520874,\n                0.4933125078678131\n              ],\n              [\n                0.2525937557220459,\n                0.11922916769981384,\n                0.5640156269073486,\n                0.6117291450500488\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.5376718640327454,\n                0.23670832812786102,\n                0.7503437399864197,\n                0.7361875176429749\n              ],\n              [\n                0.2525937557220459,\n                0.4502291679382324,\n                0.36403125524520874,\n                0.4933125078678131\n              ],\n              [\n                0.2525937557220459,\n                0.23670832812786102,\n                0.7503437399864197,\n                0.7361875176429749\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000090862.jpg",\n        "coco_id": 90862,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "knife",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 1760659,\n          "object_id": 1891779,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_90862_1760659_1891779_430129_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 430129,\n          "object_id": 1891779,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.5376718640327454,\n                0.23670832812786102,\n                0.7503437399864197,\n                0.7361875176429749\n              ],\n              [\n                0.2985000014305115,\n                0.4581458270549774,\n                0.3859218657016754,\n                0.47187501192092896\n              ],\n              [\n                0.2985000014305115,\n                0.23670832812786102,\n                0.7503437399864197,\n                0.7361875176429749\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a knife",\n            "bounds": [\n              [\n                0.28120312094688416,\n                0.11922916769981384,\n                0.5640156269073486,\n                0.6117291450500488\n              ],\n              [\n                0.2985000014305115,\n                0.4581458270549774,\n                0.3859218657016754,\n                0.47187501192092896\n              ],\n              [\n                0.28120312094688416,\n                0.11922916769981384,\n                0.5640156269073486,\n                0.6117291450500488\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000090862.jpg",\n        "coco_id": 90862,\n        "caption_split": "train",\n        "predicate": "eating",\n        "object_label": "pizza",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 566921,\n          "object_id": 1570959,\n          "action": "eat",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_90862_566921_1570959_430129_eat",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 430129,\n          "object_id": 1570959,\n          "action": "eat",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.5376718640327454,\n                0.23670832812786102,\n                0.7503437399864197,\n                0.7361875176429749\n              ],\n              [\n                0.2554374933242798,\n                0.7519583106040955,\n                0.33460938930511475,\n                0.8421041369438171\n              ],\n              [\n                0.2554374933242798,\n                0.23670832812786102,\n                0.7503437399864197,\n                0.8421041369438171\n              ]\n            ]\n          },\n          {\n            "text": "a person eating a pizza",\n            "bounds": [\n              [\n                0.0,\n                0.6217291951179504,\n                0.36662501096725464,\n                1.0\n              ],\n              [\n                0.2554374933242798,\n                0.7519583106040955,\n                0.33460938930511475,\n                0.8421041369438171\n              ],\n              [\n                0.0,\n                0.6217291951179504,\n                0.36662501096725464,\n                1.0\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000090862.jpg",\n        "coco_id": 90862,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "fork",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 566921,\n          "object_id": 690527,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_90862_566921_690527_430129_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 430129,\n          "object_id": 690527,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.0,\n                0.6217291951179504,\n                0.36662501096725464,\n                1.0\n              ],\n              [\n                0.1302812546491623,\n                0.6328333616256714,\n                0.24592188000679016,\n                0.7699375152587891\n              ],\n              [\n                0.0,\n                0.6217291951179504,\n                0.36662501096725464,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a fork",\n            "bounds": [\n              [\n                0.5376718640327454,\n                0.23670832812786102,\n                0.7503437399864197,\n                0.7361875176429749\n              ],\n              [\n                0.1302812546491623,\n                0.6328333616256714,\n                0.24592188000679016,\n                0.7699375152587891\n              ],\n              [\n                0.1302812546491623,\n                0.23670832812786102,\n                0.7503437399864197,\n                0.7699375152587891\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000091926.jpg",\n        "coco_id": 91926,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 184556,\n          "object_id": 1810396,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_91926_184556_1810396_200764_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 200764,\n          "object_id": 1810396,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.43034374713897705,\n                0.28765013813972473,\n                0.6535781025886536,\n                0.9887728691101074\n              ],\n              [\n                0.3829062581062317,\n                0.5175979137420654,\n                0.4731093645095825,\n                0.968563973903656\n              ],\n              [\n                0.3829062581062317,\n                0.28765013813972473,\n                0.6535781025886536,\n                0.9887728691101074\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.6629843711853027,\n                0.11010444164276123,\n                0.8068749904632568,\n                0.9730287194252014\n              ],\n              [\n                0.3829062581062317,\n                0.5175979137420654,\n                0.4731093645095825,\n                0.968563973903656\n              ],\n              [\n                0.3829062581062317,\n                0.11010444164276123,\n                0.8068749904632568,\n                0.9730287194252014\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000091926.jpg",\n        "coco_id": 91926,\n        "caption_split": "train",\n        "predicate": "sitting_on",\n        "object_label": "bench",\n        "positive_index": 1,\n        "source_positive": {\n          "subject_id": 192948,\n          "object_id": 1810396,\n          "action": "sit",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_91926_192948_1810396_200764_sit",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 200764,\n          "object_id": 1810396,\n          "action": "sit",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.6629843711853027,\n                0.11010444164276123,\n                0.8068749904632568,\n                0.9730287194252014\n              ],\n              [\n                0.3829062581062317,\n                0.5175979137420654,\n                0.4731093645095825,\n                0.968563973903656\n              ],\n              [\n                0.3829062581062317,\n                0.11010444164276123,\n                0.8068749904632568,\n                0.9730287194252014\n              ]\n            ]\n          },\n          {\n            "text": "a person sitting on a bench",\n            "bounds": [\n              [\n                0.14357812702655792,\n                0.25112271308898926,\n                0.4642656147480011,\n                0.9865535497665405\n              ],\n              [\n                0.3829062581062317,\n                0.5175979137420654,\n                0.4731093645095825,\n                0.968563973903656\n              ],\n              [\n                0.14357812702655792,\n                0.25112271308898926,\n                0.4731093645095825,\n                0.9865535497665405\n              ]\n            ]\n          }\n        ]\n      },\n      {\n        "filename": "COCO_train2014_000000094179.jpg",\n        "coco_id": 94179,\n        "caption_split": "train",\n        "predicate": "holding",\n        "object_label": "laptop",\n        "positive_index": 0,\n        "source_positive": {\n          "subject_id": 505704,\n          "object_id": 1101727,\n          "action": "hold",\n          "label": 1\n        },\n        "label_source": "vcoco_explicit_role_positive_and_actor_zero",\n        "case_id": "actor_94179_505704_1101727_508374_hold",\n        "task": "wrong_subject_instance",\n        "source_negative": {\n          "subject_id": 508374,\n          "object_id": 1101727,\n          "action": "hold",\n          "label": 0\n        },\n        "candidates": [\n          {\n            "text": "a person holding a laptop",\n            "bounds": [\n              [\n                0.4174707233905792,\n                0.09214062243700027,\n                0.995526909828186,\n                0.7313593626022339\n              ],\n              [\n                0.7089929580688477,\n                0.34700000286102295,\n                0.9993442893028259,\n                0.46159374713897705\n              ],\n              [\n                0.4174707233905792,\n                0.09214062243700027,\n                0.9993442893028259,\n                0.7313593626022339\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a laptop",\n            "bounds": [\n              [\n                0.0,\n                0.07796874642372131,\n                0.9666510820388794,\n                0.9835781455039978\n              ],\n              [\n                0.7089929580688477,\n                0.34700000286102295,\n                0.9993442893028259,\n                0.46159374713897705\n              ],\n              [\n                0.0,\n                0.07796874642372131,\n                0.9993442893028259,\n                0.9835781455039978\n              ]\n            ]\n          }\n        ]\n      }\n    ],\n    "probe": [\n      {\n        "case_id": "target_0c3f9530294c",\n        "filename": "COCO_train2014_000000402041.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "riding",\n        "object_label": "elephant",\n        "candidates": [\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.37229686975479126,\n                0.11458221077919006,\n                0.4605312645435333,\n                0.38474392890930176\n              ],\n              [\n                0.2417343705892563,\n                0.1992991864681244,\n                0.607796847820282,\n                0.8188948631286621\n              ],\n              [\n                0.2417343705892563,\n                0.11458221077919006,\n                0.607796847820282,\n                0.8188948631286621\n              ]\n            ]\n          },\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.37229686975479126,\n                0.11458221077919006,\n                0.4605312645435333,\n                0.38474392890930176\n              ],\n              [\n                0.03096874989569187,\n                0.18946091830730438,\n                0.3441718816757202,\n                0.8205660581588745\n              ],\n              [\n                0.03096874989569187,\n                0.11458221077919006,\n                0.4605312645435333,\n                0.8205660581588745\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 402041,\n        "caption_split": "train"\n      },\n      {\n        "case_id": "target_0e4cbf68736f",\n        "filename": "COCO_train2014_000000329469.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.44103124737739563,\n                0.12934426963329315,\n                0.8724687695503235,\n                0.7125058770179749\n              ],\n              [\n                0.379031240940094,\n                0.4518032670021057,\n                0.9988593459129333,\n                0.9889227151870728\n              ],\n              [\n                0.379031240940094,\n                0.12934426963329315,\n                0.9988593459129333,\n                0.9889227151870728\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.44103124737739563,\n                0.12934426963329315,\n                0.8724687695503235,\n                0.7125058770179749\n              ],\n              [\n                0.00040625000838190317,\n                0.1662997603416443,\n                0.33406248688697815,\n                0.9707728624343872\n              ],\n              [\n                0.00040625000838190317,\n                0.12934426963329315,\n                0.8724687695503235,\n                0.9707728624343872\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 329469,\n        "caption_split": "train"\n      },\n      {\n        "case_id": "target_1c641d5eb7b4",\n        "filename": "COCO_train2014_000000329469.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.054734375327825546,\n                0.002974238945171237,\n                0.3283750116825104,\n                0.6980562210083008\n              ],\n              [\n                0.379031240940094,\n                0.4518032670021057,\n                0.9988593459129333,\n                0.9889227151870728\n              ],\n              [\n                0.054734375327825546,\n                0.002974238945171237,\n                0.9988593459129333,\n                0.9889227151870728\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.054734375327825546,\n                0.002974238945171237,\n                0.3283750116825104,\n                0.6980562210083008\n              ],\n              [\n                0.00040625000838190317,\n                0.1662997603416443,\n                0.33406248688697815,\n                0.9707728624343872\n              ],\n              [\n                0.00040625000838190317,\n                0.002974238945171237,\n                0.33406248688697815,\n                0.9707728624343872\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 329469,\n        "caption_split": "train"\n      },\n      {\n        "case_id": "target_4847bc6c9dbd",\n        "filename": "COCO_train2014_000000529613.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "holding",\n        "object_label": "pizza",\n        "candidates": [\n          {\n            "text": "a person holding a pizza",\n            "bounds": [\n              [\n                0.7104374766349792,\n                0.05479079484939575,\n                0.9990781545639038,\n                0.9891631603240967\n              ],\n              [\n                0.5698281526565552,\n                0.7525104880332947,\n                0.7062343955039978,\n                0.9071129560470581\n              ],\n              [\n                0.5698281526565552,\n                0.05479079484939575,\n                0.9990781545639038,\n                0.9891631603240967\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a pizza",\n            "bounds": [\n              [\n                0.7104374766349792,\n                0.05479079484939575,\n                0.9990781545639038,\n                0.9891631603240967\n              ],\n              [\n                0.7311093807220459,\n                0.5250208973884583,\n                0.8207187652587891,\n                0.6130962371826172\n              ],\n              [\n                0.7104374766349792,\n                0.05479079484939575,\n                0.9990781545639038,\n                0.9891631603240967\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 529613,\n        "caption_split": "train"\n      },\n      {\n        "case_id": "target_a3e9de3d1a05",\n        "filename": "COCO_train2014_000000306800.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.12490624934434891,\n                0.07544626295566559,\n                0.2906718850135803,\n                0.4726047217845917\n              ],\n              [\n                0.12729687988758087,\n                0.2585063874721527,\n                0.3114687502384186,\n                0.5208925604820251\n              ],\n              [\n                0.12490624934434891,\n                0.07544626295566559,\n                0.3114687502384186,\n                0.5208925604820251\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.12490624934434891,\n                0.07544626295566559,\n                0.2906718850135803,\n                0.4726047217845917\n              ],\n              [\n                0.36570313572883606,\n                0.24967212975025177,\n                0.504281222820282,\n                0.5496357083320618\n              ],\n              [\n                0.12490624934434891,\n                0.07544626295566559,\n                0.504281222820282,\n                0.5496357083320618\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 306800,\n        "caption_split": "train"\n      },\n      {\n        "case_id": "target_aad7419af8d7",\n        "filename": "COCO_train2014_000000090148.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "riding",\n        "object_label": "horse",\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.44264063239097595,\n                0.4294103682041168,\n                0.5676562786102295,\n                0.828773558139801\n              ],\n              [\n                0.20103125274181366,\n                0.4789622724056244,\n                0.45512500405311584,\n                0.9908726215362549\n              ],\n              [\n                0.20103125274181366,\n                0.4294103682041168,\n                0.5676562786102295,\n                0.9908726215362549\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.44264063239097595,\n                0.4294103682041168,\n                0.5676562786102295,\n                0.828773558139801\n              ],\n              [\n                0.37553125619888306,\n                0.5447641611099243,\n                0.6722031235694885,\n                0.9727122783660889\n              ],\n              [\n                0.37553125619888306,\n                0.4294103682041168,\n                0.6722031235694885,\n                0.9727122783660889\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 90148,\n        "caption_split": "train"\n      },\n      {\n        "case_id": "target_be7cb012482d",\n        "filename": "COCO_train2014_000000029525.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.5501093864440918,\n                0.5265630483627319,\n                0.5977500081062317,\n                0.6634003520011902\n              ],\n              [\n                0.7240312695503235,\n                0.5825777053833008,\n                0.779046893119812,\n                0.6706947088241577\n              ],\n              [\n                0.5501093864440918,\n                0.5265630483627319,\n                0.779046893119812,\n                0.6706947088241577\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.5501093864440918,\n                0.5265630483627319,\n                0.5977500081062317,\n                0.6634003520011902\n              ],\n              [\n                0.534250020980835,\n                0.5866178870201111,\n                0.6139531135559082,\n                0.6900548338890076\n              ],\n              [\n                0.534250020980835,\n                0.5265630483627319,\n                0.6139531135559082,\n                0.6900548338890076\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 29525,\n        "caption_split": "train"\n      },\n      {\n        "case_id": "target_db610fa7def8",\n        "filename": "COCO_train2014_000000448839.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.6131406426429749,\n                0.27385416626930237,\n                0.7441562414169312,\n                0.7698749899864197\n              ],\n              [\n                0.3524687588214874,\n                0.4518125057220459,\n                0.48356249928474426,\n                0.7711041569709778\n              ],\n              [\n                0.3524687588214874,\n                0.27385416626930237,\n                0.7441562414169312,\n                0.7711041569709778\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.6131406426429749,\n                0.27385416626930237,\n                0.7441562414169312,\n                0.7698749899864197\n              ],\n              [\n                0.6295156478881836,\n                0.5354999899864197,\n                0.6914843916893005,\n                0.9014583230018616\n              ],\n              [\n                0.6131406426429749,\n                0.27385416626930237,\n                0.7441562414169312,\n                0.9014583230018616\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 448839,\n        "caption_split": "train"\n      },\n      {\n        "case_id": "target_ddf0eb72e2b2",\n        "filename": "COCO_train2014_000000136302.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "holding",\n        "object_label": "surfboard",\n        "candidates": [\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.20231249928474426,\n                0.5519999861717224,\n                0.25632813572883606,\n                0.7668125033378601\n              ],\n              [\n                0.16318750381469727,\n                0.5120416879653931,\n                0.29820311069488525,\n                0.5669375061988831\n              ],\n              [\n                0.16318750381469727,\n                0.5120416879653931,\n                0.29820311069488525,\n                0.7668125033378601\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a surfboard",\n            "bounds": [\n              [\n                0.20231249928474426,\n                0.5519999861717224,\n                0.25632813572883606,\n                0.7668125033378601\n              ],\n              [\n                0.16573438048362732,\n                0.6129791736602783,\n                0.25764063000679016,\n                0.6891250014305115\n              ],\n              [\n                0.16573438048362732,\n                0.5519999861717224,\n                0.25764063000679016,\n                0.7668125033378601\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 136302,\n        "caption_split": "train"\n      },\n      {\n        "case_id": "target_e0ba7ff124ef",\n        "filename": "COCO_train2014_000000334758.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "riding",\n        "object_label": "horse",\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.24637646973133087,\n                0.5350156426429749,\n                0.4445176422595978,\n                0.7931249737739563\n              ],\n              [\n                0.047152940183877945,\n                0.6251093745231628,\n                0.2621176540851593,\n                0.9177968502044678\n              ],\n              [\n                0.047152940183877945,\n                0.5350156426429749,\n                0.4445176422595978,\n                0.9177968502044678\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.24637646973133087,\n                0.5350156426429749,\n                0.4445176422595978,\n                0.7931249737739563\n              ],\n              [\n                0.2870352864265442,\n                0.5880937576293945,\n                0.41312941908836365,\n                0.9177031517028809\n              ],\n              [\n                0.24637646973133087,\n                0.5350156426429749,\n                0.4445176422595978,\n                0.9177031517028809\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 334758,\n        "caption_split": "train"\n      },\n      {\n        "case_id": "target_f4c47f24a715",\n        "filename": "COCO_train2014_000000432019.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "holding",\n        "object_label": "umbrella",\n        "candidates": [\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.6438906192779541,\n                0.8699444532394409,\n                0.7256718873977661,\n                0.9988611340522766\n              ],\n              [\n                0.5967968702316284,\n                0.7785555720329285,\n                0.8040156364440918,\n                0.9226111173629761\n              ],\n              [\n                0.5967968702316284,\n                0.7785555720329285,\n                0.8040156364440918,\n                0.9988611340522766\n              ]\n            ]\n          },\n          {\n            "text": "a person holding an umbrella",\n            "bounds": [\n              [\n                0.6438906192779541,\n                0.8699444532394409,\n                0.7256718873977661,\n                0.9988611340522766\n              ],\n              [\n                0.312624990940094,\n                0.7567777633666992,\n                0.5142655968666077,\n                0.9590277671813965\n              ],\n              [\n                0.312624990940094,\n                0.7567777633666992,\n                0.7256718873977661,\n                0.9988611340522766\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 432019,\n        "caption_split": "train"\n      },\n      {\n        "case_id": "target_fb525b4c45ce",\n        "filename": "COCO_train2014_000000402041.jpg",\n        "task": "wrong_object_instance",\n        "predicate": "riding",\n        "object_label": "elephant",\n        "candidates": [\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.6753281354904175,\n                0.2014285773038864,\n                0.772390604019165,\n                0.47886791825294495\n              ],\n              [\n                0.5250625014305115,\n                0.2986792325973511,\n                0.9226093888282776,\n                0.8185714483261108\n              ],\n              [\n                0.5250625014305115,\n                0.2014285773038864,\n                0.9226093888282776,\n                0.8185714483261108\n              ]\n            ]\n          },\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.6753281354904175,\n                0.2014285773038864,\n                0.772390604019165,\n                0.47886791825294495\n              ],\n              [\n                0.03096874989569187,\n                0.18946091830730438,\n                0.3441718816757202,\n                0.8205660581588745\n              ],\n              [\n                0.03096874989569187,\n                0.18946091830730438,\n                0.772390604019165,\n                0.8205660581588745\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "label_source": "human_explicit_supported_vs_contradicted",\n        "coco_id": 402041,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000116512.jpg",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_116512_217722_1765055_riding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.6652287840843201,\n                0.4274509847164154,\n                0.7954902052879333,\n                0.7481045722961426\n              ],\n              [\n                0.6561437845230103,\n                0.5800653696060181,\n                0.7348529696464539,\n                0.7608823776245117\n              ],\n              [\n                0.6561437845230103,\n                0.4274509847164154,\n                0.7954902052879333,\n                0.7608823776245117\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.6652287840843201,\n                0.4274509847164154,\n                0.7954902052879333,\n                0.7481045722961426\n              ],\n              [\n                0.6561437845230103,\n                0.5800653696060181,\n                0.7348529696464539,\n                0.7608823776245117\n              ],\n              [\n                0.6561437845230103,\n                0.4274509847164154,\n                0.7954902052879333,\n                0.7608823776245117\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 116512,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000117371.jpg",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_117371_259686_129675_holding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.26661795377731323,\n                0.04307812452316284,\n                0.7634864449501038,\n                0.8140624761581421\n              ],\n              [\n                0.4051356911659241,\n                0.3806406259536743,\n                0.764321506023407,\n                0.982796847820282\n              ],\n              [\n                0.26661795377731323,\n                0.04307812452316284,\n                0.764321506023407,\n                0.982796847820282\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.26661795377731323,\n                0.04307812452316284,\n                0.7634864449501038,\n                0.8140624761581421\n              ],\n              [\n                0.4051356911659241,\n                0.3806406259536743,\n                0.764321506023407,\n                0.982796847820282\n              ],\n              [\n                0.26661795377731323,\n                0.04307812452316284,\n                0.764321506023407,\n                0.982796847820282\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 117371,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000167743.jpg",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_167743_196203_127444_holding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.1712656319141388,\n                0.3935999870300293,\n                0.5667499899864197,\n                1.0\n              ],\n              [\n                0.5612343549728394,\n                0.9231294393539429,\n                0.6469218730926514,\n                0.9897882342338562\n              ],\n              [\n                0.1712656319141388,\n                0.3935999870300293,\n                0.6469218730926514,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.1712656319141388,\n                0.3935999870300293,\n                0.5667499899864197,\n                1.0\n              ],\n              [\n                0.5612343549728394,\n                0.9231294393539429,\n                0.6469218730926514,\n                0.9897882342338562\n              ],\n              [\n                0.1712656319141388,\n                0.3935999870300293,\n                0.6469218730926514,\n                1.0\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 167743,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000231524.jpg",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_231524_234083_130532_riding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.7275624871253967,\n                0.23311474919319153,\n                0.9436718821525574,\n                1.0\n              ],\n              [\n                0.7545156478881836,\n                0.595714271068573,\n                1.0,\n                0.9862763285636902\n              ],\n              [\n                0.7275624871253967,\n                0.23311474919319153,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.7275624871253967,\n                0.23311474919319153,\n                0.9436718821525574,\n                1.0\n              ],\n              [\n                0.7545156478881836,\n                0.595714271068573,\n                1.0,\n                0.9862763285636902\n              ],\n              [\n                0.7275624871253967,\n                0.23311474919319153,\n                1.0,\n                1.0\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 231524,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000255495.jpg",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_255495_1716967_128488_riding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.1861562430858612,\n                0.0,\n                0.7047500014305115,\n                0.9887615442276001\n              ],\n              [\n                0.5949843525886536,\n                0.5667652487754822,\n                0.9813593626022339,\n                0.9895194172859192\n              ],\n              [\n                0.1861562430858612,\n                0.0,\n                0.9813593626022339,\n                0.9895194172859192\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.1861562430858612,\n                0.0,\n                0.7047500014305115,\n                0.9887615442276001\n              ],\n              [\n                0.5949843525886536,\n                0.5667652487754822,\n                0.9813593626022339,\n                0.9895194172859192\n              ],\n              [\n                0.1861562430858612,\n                0.0,\n                0.9813593626022339,\n                0.9895194172859192\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 255495,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000268313.jpg",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_268313_189084_128301_riding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.31773436069488525,\n                0.16183333098888397,\n                0.5048437714576721,\n                0.7833333611488342\n              ],\n              [\n                0.37257811427116394,\n                0.526520848274231,\n                0.4612968862056732,\n                0.8512499928474426\n              ],\n              [\n                0.31773436069488525,\n                0.16183333098888397,\n                0.5048437714576721,\n                0.8512499928474426\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.31773436069488525,\n                0.16183333098888397,\n                0.5048437714576721,\n                0.7833333611488342\n              ],\n              [\n                0.37257811427116394,\n                0.526520848274231,\n                0.4612968862056732,\n                0.8512499928474426\n              ],\n              [\n                0.31773436069488525,\n                0.16183333098888397,\n                0.5048437714576721,\n                0.8512499928474426\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 268313,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000340406.jpg",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_340406_1265058_241798_holding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.8238281011581421,\n                0.4722500145435333,\n                0.9077187776565552,\n                0.8055833578109741\n              ],\n              [\n                0.8103437423706055,\n                0.648562490940094,\n                0.9169374704360962,\n                0.8288541436195374\n              ],\n              [\n                0.8103437423706055,\n                0.4722500145435333,\n                0.9169374704360962,\n                0.8288541436195374\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.8238281011581421,\n                0.4722500145435333,\n                0.9077187776565552,\n                0.8055833578109741\n              ],\n              [\n                0.8103437423706055,\n                0.648562490940094,\n                0.9169374704360962,\n                0.8288541436195374\n              ],\n              [\n                0.8103437423706055,\n                0.4722500145435333,\n                0.9169374704360962,\n                0.8288541436195374\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 340406,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000411238.jpg",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_411238_254641_1334919_holding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.7264062762260437,\n                0.2202291637659073,\n                1.0,\n                0.9865208268165588\n              ],\n              [\n                0.7104843854904175,\n                0.8171666860580444,\n                0.8185312747955322,\n                0.984458327293396\n              ],\n              [\n                0.7104843854904175,\n                0.2202291637659073,\n                1.0,\n                0.9865208268165588\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.7264062762260437,\n                0.2202291637659073,\n                1.0,\n                0.9865208268165588\n              ],\n              [\n                0.7104843854904175,\n                0.8171666860580444,\n                0.8185312747955322,\n                0.984458327293396\n              ],\n              [\n                0.7104843854904175,\n                0.2202291637659073,\n                1.0,\n                0.9865208268165588\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 411238,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000459543.jpg",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_459543_185107_152411_riding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.45814061164855957,\n                0.3211241364479065,\n                0.7355937361717224,\n                0.5616159439086914\n              ],\n              [\n                0.5517968535423279,\n                0.5464637279510498,\n                0.8114687204360962,\n                0.7542856931686401\n              ],\n              [\n                0.45814061164855957,\n                0.3211241364479065,\n                0.8114687204360962,\n                0.7542856931686401\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.45814061164855957,\n                0.3211241364479065,\n                0.7355937361717224,\n                0.5616159439086914\n              ],\n              [\n                0.5517968535423279,\n                0.5464637279510498,\n                0.8114687204360962,\n                0.7542856931686401\n              ],\n              [\n                0.45814061164855957,\n                0.3211241364479065,\n                0.8114687204360962,\n                0.7542856931686401\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 459543,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000482810.jpg",\n        "predicate": "holding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_482810_197961_126737_holding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.3148750066757202,\n                0.14569412171840668,\n                0.5939218997955322,\n                0.9322117567062378\n              ],\n              [\n                0.024281250312924385,\n                0.5677411556243896,\n                0.711187481880188,\n                0.9784941077232361\n              ],\n              [\n                0.024281250312924385,\n                0.14569412171840668,\n                0.711187481880188,\n                0.9784941077232361\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.3148750066757202,\n                0.14569412171840668,\n                0.5939218997955322,\n                0.9322117567062378\n              ],\n              [\n                0.024281250312924385,\n                0.5677411556243896,\n                0.711187481880188,\n                0.9784941077232361\n              ],\n              [\n                0.024281250312924385,\n                0.14569412171840668,\n                0.711187481880188,\n                0.9784941077232361\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 482810,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000562296.jpg",\n        "predicate": "holding",\n        "object_label": "motorcycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_562296_195750_148370_holding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person holding a motorcycle",\n            "bounds": [\n              [\n                0.3037812411785126,\n                0.0015690376749262214,\n                0.8358281254768372,\n                0.9813389182090759\n              ],\n              [\n                0.001687500043772161,\n                0.3502301275730133,\n                1.0,\n                0.9898744821548462\n              ],\n              [\n                0.001687500043772161,\n                0.0015690376749262214,\n                1.0,\n                0.9898744821548462\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.3037812411785126,\n                0.0015690376749262214,\n                0.8358281254768372,\n                0.9813389182090759\n              ],\n              [\n                0.001687500043772161,\n                0.3502301275730133,\n                1.0,\n                0.9898744821548462\n              ],\n              [\n                0.001687500043772161,\n                0.0015690376749262214,\n                1.0,\n                0.9898744821548462\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 562296,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000098596.jpg",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_verb_98596_202874_125334_riding",\n        "task": "wrong_predicate",\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.21579687297344208,\n                0.015435815788805485,\n                0.6891718506813049,\n                0.9950396418571472\n              ],\n              [\n                0.0,\n                0.6626624464988708,\n                0.8251405954360962,\n                1.0\n              ],\n              [\n                0.0,\n                0.015435815788805485,\n                0.8251405954360962,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a bicycle",\n            "bounds": [\n              [\n                0.21579687297344208,\n                0.015435815788805485,\n                0.6891718506813049,\n                0.9950396418571472\n              ],\n              [\n                0.0,\n                0.6626624464988708,\n                0.8251405954360962,\n                1.0\n              ],\n              [\n                0.0,\n                0.015435815788805485,\n                0.8251405954360962,\n                1.0\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 98596,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000101094.jpg",\n        "predicate": "riding",\n        "object_label": "elephant",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_101094_1717869_583064_233930_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.40681248903274536,\n                0.15318749845027924,\n                0.4965781271457672,\n                0.29897916316986084\n              ],\n              [\n                0.19001562893390656,\n                0.3318333327770233,\n                0.6894687414169312,\n                0.8296041488647461\n              ],\n              [\n                0.19001562893390656,\n                0.15318749845027924,\n                0.6894687414169312,\n                0.8296041488647461\n              ]\n            ]\n          },\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.8354687690734863,\n                0.5525416731834412,\n                0.8940312266349792,\n                0.7758333086967468\n              ],\n              [\n                0.19001562893390656,\n                0.3318333327770233,\n                0.6894687414169312,\n                0.8296041488647461\n              ],\n              [\n                0.19001562893390656,\n                0.3318333327770233,\n                0.8940312266349792,\n                0.8296041488647461\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 101094,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000105714.jpg",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_105714_482561_602761_483544_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.6565937399864197,\n                0.22222483158111572,\n                1.0,\n                0.9841217994689941\n              ],\n              [\n                0.3793906271457672,\n                0.3893676698207855,\n                0.44584375619888306,\n                0.5232552886009216\n              ],\n              [\n                0.3793906271457672,\n                0.22222483158111572,\n                1.0,\n                0.9841217994689941\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.4212968647480011,\n                0.22177985310554504,\n                0.908578097820282,\n                0.9813348650932312\n              ],\n              [\n                0.3793906271457672,\n                0.3893676698207855,\n                0.44584375619888306,\n                0.5232552886009216\n              ],\n              [\n                0.3793906271457672,\n                0.22177985310554504,\n                0.908578097820282,\n                0.9813348650932312\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 105714,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000126001.jpg",\n        "predicate": "riding",\n        "object_label": "elephant",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_126001_563454_582401_559045_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.5190312266349792,\n                0.43960416316986084,\n                0.7751093506813049,\n                0.8023958206176758\n              ],\n              [\n                0.23310936987400055,\n                0.7961666584014893,\n                0.8969531059265137,\n                0.9898750185966492\n              ],\n              [\n                0.23310936987400055,\n                0.43960416316986084,\n                0.8969531059265137,\n                0.9898750185966492\n              ]\n            ]\n          },\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.32167187333106995,\n                0.5434583425521851,\n                0.4909687638282776,\n                0.9881458282470703\n              ],\n              [\n                0.23310936987400055,\n                0.7961666584014893,\n                0.8969531059265137,\n                0.9898750185966492\n              ],\n              [\n                0.23310936987400055,\n                0.5434583425521851,\n                0.8969531059265137,\n                0.9898750185966492\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 126001,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000147674.jpg",\n        "predicate": "holding",\n        "object_label": "handbag",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_147674_524611_1172736_2007716_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.0,\n                0.730772852897644,\n                0.1152968779206276,\n                1.0\n              ],\n              [\n                0.14671875536441803,\n                0.8357142806053162,\n                0.20200000703334808,\n                0.9436768293380737\n              ],\n              [\n                0.0,\n                0.730772852897644,\n                0.20200000703334808,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a handbag",\n            "bounds": [\n              [\n                0.13314062356948853,\n                0.7485246062278748,\n                0.2345000058412552,\n                0.9775409698486328\n              ],\n              [\n                0.14671875536441803,\n                0.8357142806053162,\n                0.20200000703334808,\n                0.9436768293380737\n              ],\n              [\n                0.13314062356948853,\n                0.7485246062278748,\n                0.2345000058412552,\n                0.9775409698486328\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 147674,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000159176.jpg",\n        "predicate": "holding",\n        "object_label": "skateboard",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_159176_1301495_1474767_1231990_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.5081250071525574,\n                0.42853185534477234,\n                0.5763280987739563,\n                0.7341551184654236\n              ],\n              [\n                0.2615000009536743,\n                0.5813850164413452,\n                0.2896406352519989,\n                0.7102493047714233\n              ],\n              [\n                0.2615000009536743,\n                0.42853185534477234,\n                0.5763280987739563,\n                0.7341551184654236\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a skateboard",\n            "bounds": [\n              [\n                0.2423125058412552,\n                0.49980610609054565,\n                0.28987500071525574,\n                0.7234072089195251\n              ],\n              [\n                0.2615000009536743,\n                0.5813850164413452,\n                0.2896406352519989,\n                0.7102493047714233\n              ],\n              [\n                0.2423125058412552,\n                0.49980610609054565,\n                0.28987500071525574,\n                0.7234072089195251\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 159176,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000165334.jpg",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_165334_225612_1335895_217739_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.6270312666893005,\n                0.3283958435058594,\n                0.8047812581062317,\n                0.6598958373069763\n              ],\n              [\n                0.48310938477516174,\n                0.5619375109672546,\n                0.5456093549728394,\n                0.8479791879653931\n              ],\n              [\n                0.48310938477516174,\n                0.3283958435058594,\n                0.8047812581062317,\n                0.8479791879653931\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.44987499713897705,\n                0.3238750100135803,\n                0.5939218997955322,\n                0.7230833172798157\n              ],\n              [\n                0.48310938477516174,\n                0.5619375109672546,\n                0.5456093549728394,\n                0.8479791879653931\n              ],\n              [\n                0.44987499713897705,\n                0.3238750100135803,\n                0.5939218997955322,\n                0.8479791879653931\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 165334,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000170636.jpg",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_170636_219821_150628_236396_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.8752031326293945,\n                0.31016626954078674,\n                0.9940780997276306,\n                0.6931828856468201\n              ],\n              [\n                0.5203906297683716,\n                0.3354869484901428,\n                0.7796562314033508,\n                0.8285035490989685\n              ],\n              [\n                0.5203906297683716,\n                0.31016626954078674,\n                0.9940780997276306,\n                0.8285035490989685\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.5973437428474426,\n                0.266128271818161,\n                0.7292500138282776,\n                0.739406168460846\n              ],\n              [\n                0.5203906297683716,\n                0.3354869484901428,\n                0.7796562314033508,\n                0.8285035490989685\n              ],\n              [\n                0.5203906297683716,\n                0.266128271818161,\n                0.7796562314033508,\n                0.8285035490989685\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 170636,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000180778.jpg",\n        "predicate": "holding",\n        "object_label": "laptop",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_180778_1739914_1101257_454965_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a laptop",\n            "bounds": [\n              [\n                0.25120311975479126,\n                0.17098326981067657,\n                0.49904686212539673,\n                0.6585773825645447\n              ],\n              [\n                0.06456249952316284,\n                0.4424686133861542,\n                0.585281252861023,\n                0.8800627589225769\n              ],\n              [\n                0.06456249952316284,\n                0.17098326981067657,\n                0.585281252861023,\n                0.8800627589225769\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a laptop",\n            "bounds": [\n              [\n                0.0,\n                0.22715480625629425,\n                0.32087498903274536,\n                1.0\n              ],\n              [\n                0.06456249952316284,\n                0.4424686133861542,\n                0.585281252861023,\n                0.8800627589225769\n              ],\n              [\n                0.0,\n                0.22715480625629425,\n                0.585281252861023,\n                1.0\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 180778,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000203671.jpg",\n        "predicate": "riding",\n        "object_label": "horse",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_203671_2161494_1403380_192903_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.8595468997955322,\n                0.1662295013666153,\n                1.0,\n                0.9870023131370544\n              ],\n              [\n                0.874093770980835,\n                0.5901405215263367,\n                0.9988124966621399,\n                0.997751772403717\n              ],\n              [\n                0.8595468997955322,\n                0.1662295013666153,\n                1.0,\n                0.997751772403717\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.4392968714237213,\n                0.5415691137313843,\n                0.725656270980835,\n                0.9887588024139404\n              ],\n              [\n                0.874093770980835,\n                0.5901405215263367,\n                0.9988124966621399,\n                0.997751772403717\n              ],\n              [\n                0.4392968714237213,\n                0.5415691137313843,\n                0.9988124966621399,\n                0.997751772403717\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 203671,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000209842.jpg",\n        "predicate": "riding",\n        "object_label": "horse",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_209842_188740_54707_205928_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.518750011920929,\n                0.4000000059604645,\n                0.6521875262260437,\n                0.8943793773651123\n              ],\n              [\n                0.2000156193971634,\n                0.44131147861480713,\n                0.45003125071525574,\n                0.9895082116127014\n              ],\n              [\n                0.2000156193971634,\n                0.4000000059604645,\n                0.6521875262260437,\n                0.9895082116127014\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.19757813215255737,\n                0.1193208396434784,\n                0.4896562397480011,\n                0.8832552433013916\n              ],\n              [\n                0.2000156193971634,\n                0.44131147861480713,\n                0.45003125071525574,\n                0.9895082116127014\n              ],\n              [\n                0.19757813215255737,\n                0.1193208396434784,\n                0.4896562397480011,\n                0.9895082116127014\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 209842,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000231315.jpg",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_231315_1252218_2204126_1275844_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.435734361410141,\n                0.0441787950694561,\n                0.7153280973434448,\n                0.889896035194397\n              ],\n              [\n                0.22248438000679016,\n                0.1283160150051117,\n                0.9484531283378601,\n                0.8904989361763\n              ],\n              [\n                0.22248438000679016,\n                0.0441787950694561,\n                0.9484531283378601,\n                0.8904989361763\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.01015624962747097,\n                0.0934719368815422,\n                0.3622343838214874,\n                0.9853638410568237\n              ],\n              [\n                0.22248438000679016,\n                0.1283160150051117,\n                0.9484531283378601,\n                0.8904989361763\n              ],\n              [\n                0.01015624962747097,\n                0.0934719368815422,\n                0.9484531283378601,\n                0.9853638410568237\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 231315,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000233901.jpg",\n        "predicate": "holding",\n        "object_label": "baseball glove",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_233901_458865_635536_435537_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.20884375274181366,\n                0.37626761198043823,\n                0.3518750071525574,\n                0.8165258169174194\n              ],\n              [\n                0.5512031316757202,\n                0.46718311309814453,\n                0.6017968654632568,\n                0.5823708772659302\n              ],\n              [\n                0.20884375274181366,\n                0.37626761198043823,\n                0.6017968654632568,\n                0.8165258169174194\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball glove",\n            "bounds": [\n              [\n                0.4794843792915344,\n                0.3456572890281677,\n                0.8562343716621399,\n                0.8762206435203552\n              ],\n              [\n                0.5512031316757202,\n                0.46718311309814453,\n                0.6017968654632568,\n                0.5823708772659302\n              ],\n              [\n                0.4794843792915344,\n                0.3456572890281677,\n                0.8562343716621399,\n                0.8762206435203552\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 233901,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000268804.jpg",\n        "predicate": "riding",\n        "object_label": "horse",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_268804_1713522_54927_1739648_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.5056250095367432,\n                0.3685416579246521,\n                0.5882031321525574,\n                0.5415833592414856\n              ],\n              [\n                0.5102656483650208,\n                0.4934374988079071,\n                0.656906247138977,\n                0.7776874899864197\n              ],\n              [\n                0.5056250095367432,\n                0.3685416579246521,\n                0.656906247138977,\n                0.7776874899864197\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.6987344026565552,\n                0.5288958549499512,\n                0.7687187790870667,\n                0.6422291398048401\n              ],\n              [\n                0.5102656483650208,\n                0.4934374988079071,\n                0.656906247138977,\n                0.7776874899864197\n              ],\n              [\n                0.5102656483650208,\n                0.4934374988079071,\n                0.7687187790870667,\n                0.7776874899864197\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 268804,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000274267.jpg",\n        "predicate": "holding",\n        "object_label": "spoon",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_274267_1278180_706817_1682103_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a spoon",\n            "bounds": [\n              [\n                0.6503437757492065,\n                0.4178541600704193,\n                1.0,\n                0.7131458520889282\n              ],\n              [\n                0.22503125667572021,\n                0.5867083072662354,\n                0.2802656292915344,\n                0.6143749952316284\n              ],\n              [\n                0.22503125667572021,\n                0.4178541600704193,\n                1.0,\n                0.7131458520889282\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a spoon",\n            "bounds": [\n              [\n                0.10643749684095383,\n                0.3324166536331177,\n                0.3179374933242798,\n                0.633145809173584\n              ],\n              [\n                0.22503125667572021,\n                0.5867083072662354,\n                0.2802656292915344,\n                0.6143749952316284\n              ],\n              [\n                0.10643749684095383,\n                0.3324166536331177,\n                0.3179374933242798,\n                0.633145809173584\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 274267,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000027902.jpg",\n        "predicate": "holding",\n        "object_label": "sandwich",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_27902_2152824_2218436_450287_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a sandwich",\n            "bounds": [\n              [\n                0.28196653723716736,\n                0.2348593771457672,\n                0.9247907996177673,\n                0.9676406383514404\n              ],\n              [\n                0.5727823972702026,\n                0.5203593969345093,\n                0.6809204816818237,\n                0.5690000057220459\n              ],\n              [\n                0.28196653723716736,\n                0.2348593771457672,\n                0.9247907996177673,\n                0.9676406383514404\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a sandwich",\n            "bounds": [\n              [\n                0.8183891177177429,\n                0.10973437130451202,\n                1.0,\n                0.8647968769073486\n              ],\n              [\n                0.5727823972702026,\n                0.5203593969345093,\n                0.6809204816818237,\n                0.5690000057220459\n              ],\n              [\n                0.5727823972702026,\n                0.10973437130451202,\n                1.0,\n                0.8647968769073486\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 27902,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000284543.jpg",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_284543_187471_127037_2160149_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.32910001277923584,\n                0.12243243306875229,\n                0.49305999279022217,\n                0.3870870769023895\n              ],\n              [\n                0.24291999638080597,\n                0.4700600504875183,\n                0.42847999930381775,\n                0.8366066217422485\n              ],\n              [\n                0.24291999638080597,\n                0.12243243306875229,\n                0.49305999279022217,\n                0.8366066217422485\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.29934000968933105,\n                0.4795795679092407,\n                0.42824000120162964,\n                0.7569969892501831\n              ],\n              [\n                0.24291999638080597,\n                0.4700600504875183,\n                0.42847999930381775,\n                0.8366066217422485\n              ],\n              [\n                0.24291999638080597,\n                0.4700600504875183,\n                0.42847999930381775,\n                0.8366066217422485\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 284543,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000304445.jpg",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_304445_221015_124788_206773_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.6746562719345093,\n                0.3462603986263275,\n                0.7539218664169312,\n                0.7309695482254028\n              ],\n              [\n                0.2081875056028366,\n                0.532188355922699,\n                0.4018593728542328,\n                0.9270359873771667\n              ],\n              [\n                0.2081875056028366,\n                0.3462603986263275,\n                0.7539218664169312,\n                0.9270359873771667\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.2630000114440918,\n                0.23828254640102386,\n                0.408265620470047,\n                0.880166232585907\n              ],\n              [\n                0.2081875056028366,\n                0.532188355922699,\n                0.4018593728542328,\n                0.9270359873771667\n              ],\n              [\n                0.2081875056028366,\n                0.23828254640102386,\n                0.408265620470047,\n                0.9270359873771667\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 304445,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000304866.jpg",\n        "predicate": "holding",\n        "object_label": "backpack",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_304866_1249970_1169166_1254857_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.8590937256813049,\n                0.7146135568618774,\n                1.0,\n                0.9887588024139404\n              ],\n              [\n                0.726187527179718,\n                0.7536534070968628,\n                0.7729062438011169,\n                0.8705855011940002\n              ],\n              [\n                0.726187527179718,\n                0.7146135568618774,\n                1.0,\n                0.9887588024139404\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a backpack",\n            "bounds": [\n              [\n                0.7341406345367432,\n                0.6760655641555786,\n                0.8307656049728394,\n                1.0\n              ],\n              [\n                0.726187527179718,\n                0.7536534070968628,\n                0.7729062438011169,\n                0.8705855011940002\n              ],\n              [\n                0.726187527179718,\n                0.6760655641555786,\n                0.8307656049728394,\n                1.0\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 304866,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000317306.jpg",\n        "predicate": "holding",\n        "object_label": "baseball bat",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_317306_462594_630774_437359_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.10923437774181366,\n                0.04030245915055275,\n                0.3489062488079071,\n                0.5273156762123108\n              ],\n              [\n                0.33107811212539673,\n                0.27202269434928894,\n                0.3860468864440918,\n                0.43829867243766785\n              ],\n              [\n                0.10923437774181366,\n                0.04030245915055275,\n                0.3860468864440918,\n                0.5273156762123108\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a baseball bat",\n            "bounds": [\n              [\n                0.6426718831062317,\n                0.24718336760997772,\n                0.8191249966621399,\n                0.8337051272392273\n              ],\n              [\n                0.33107811212539673,\n                0.27202269434928894,\n                0.3860468864440918,\n                0.43829867243766785\n              ],\n              [\n                0.33107811212539673,\n                0.24718336760997772,\n                0.8191249966621399,\n                0.8337051272392273\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 317306,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000356015.jpg",\n        "predicate": "holding",\n        "object_label": "horse",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_356015_1257852_274048_1328311_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.24833999574184418,\n                0.40259701013565063,\n                0.3918600082397461,\n                0.6481492519378662\n              ],\n              [\n                0.5688999891281128,\n                0.4605970084667206,\n                0.6986799836158752,\n                0.8277015089988708\n              ],\n              [\n                0.24833999574184418,\n                0.40259701013565063,\n                0.6986799836158752,\n                0.8277015089988708\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a horse",\n            "bounds": [\n              [\n                0.5841799974441528,\n                0.37077611684799194,\n                0.6880599856376648,\n                0.6831343173980713\n              ],\n              [\n                0.5688999891281128,\n                0.4605970084667206,\n                0.6986799836158752,\n                0.8277015089988708\n              ],\n              [\n                0.5688999891281128,\n                0.37077611684799194,\n                0.6986799836158752,\n                0.8277015089988708\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 356015,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000384626.jpg",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_384626_2166601_151557_219924_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.1306000053882599,\n                0.3088372051715851,\n                0.5370200276374817,\n                0.8507493734359741\n              ],\n              [\n                0.0,\n                0.4883979260921478,\n                0.5101600289344788,\n                0.8718087673187256\n              ],\n              [\n                0.0,\n                0.3088372051715851,\n                0.5370200276374817,\n                0.8718087673187256\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.0,\n                0.002093023154884577,\n                0.09672000259160995,\n                0.4489147365093231\n              ],\n              [\n                0.0,\n                0.4883979260921478,\n                0.5101600289344788,\n                0.8718087673187256\n              ],\n              [\n                0.0,\n                0.002093023154884577,\n                0.5101600289344788,\n                0.8718087673187256\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 384626,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000407779.jpg",\n        "predicate": "holding",\n        "object_label": "teddy bear",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_407779_184993_1163135_213130_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a teddy bear",\n            "bounds": [\n              [\n                0.10090625286102295,\n                0.3345282971858978,\n                0.20973438024520874,\n                0.8309905529022217\n              ],\n              [\n                0.7837968468666077,\n                0.5545990467071533,\n                0.8712812662124634,\n                0.7482547163963318\n              ],\n              [\n                0.10090625286102295,\n                0.3345282971858978,\n                0.8712812662124634,\n                0.8309905529022217\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a teddy bear",\n            "bounds": [\n              [\n                0.703249990940094,\n                0.3645518720149994,\n                0.8288437724113464,\n                0.8353065848350525\n              ],\n              [\n                0.7837968468666077,\n                0.5545990467071533,\n                0.8712812662124634,\n                0.7482547163963318\n              ],\n              [\n                0.703249990940094,\n                0.3645518720149994,\n                0.8712812662124634,\n                0.8353065848350525\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 407779,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000421840.jpg",\n        "predicate": "holding",\n        "object_label": "person",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_421840_494406_506246_2206965_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.0,\n                0.36423438787460327,\n                0.41433027386665344,\n                0.7944999933242798\n              ],\n              [\n                0.3844220042228699,\n                0.3788124918937683,\n                0.9715780019760132,\n                0.8631093502044678\n              ],\n              [\n                0.0,\n                0.36423438787460327,\n                0.9715780019760132,\n                0.8631093502044678\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.43541285395622253,\n                0.08764062821865082,\n                0.9631926417350769,\n                0.6943749785423279\n              ],\n              [\n                0.3844220042228699,\n                0.3788124918937683,\n                0.9715780019760132,\n                0.8631093502044678\n              ],\n              [\n                0.3844220042228699,\n                0.08764062821865082,\n                0.9715780019760132,\n                0.8631093502044678\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 421840,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000445495.jpg",\n        "predicate": "holding",\n        "object_label": "book",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_445495_219382_1648592_229552_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a book",\n            "bounds": [\n              [\n                0.0042500002309679985,\n                0.24942928552627563,\n                0.47968751192092896,\n                0.9348387122154236\n              ],\n              [\n                0.34056249260902405,\n                0.3237965404987335,\n                0.4603906273841858,\n                0.6244168877601624\n              ],\n              [\n                0.0042500002309679985,\n                0.24942928552627563,\n                0.47968751192092896,\n                0.9348387122154236\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a book",\n            "bounds": [\n              [\n                0.1743437498807907,\n                0.23230768740177155,\n                0.8647500276565552,\n                0.8135980367660522\n              ],\n              [\n                0.34056249260902405,\n                0.3237965404987335,\n                0.4603906273841858,\n                0.6244168877601624\n              ],\n              [\n                0.1743437498807907,\n                0.23230768740177155,\n                0.8647500276565552,\n                0.8135980367660522\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 445495,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000453968.jpg",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_453968_520651_1482094_539754_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.8533124923706055,\n                0.23717132210731506,\n                0.9932812452316284,\n                0.8383864760398865\n              ],\n              [\n                0.6481562256813049,\n                0.3039242923259735,\n                0.7539530992507935,\n                0.5649800896644592\n              ],\n              [\n                0.6481562256813049,\n                0.23717132210731506,\n                0.9932812452316284,\n                0.8383864760398865\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.5517030954360962,\n                0.1640438288450241,\n                0.7649843692779541,\n                0.9573107361793518\n              ],\n              [\n                0.6481562256813049,\n                0.3039242923259735,\n                0.7539530992507935,\n                0.5649800896644592\n              ],\n              [\n                0.5517030954360962,\n                0.1640438288450241,\n                0.7649843692779541,\n                0.9573107361793518\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 453968,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000456245.jpg",\n        "predicate": "riding",\n        "object_label": "boat",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_456245_201734_177098_213452_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a boat",\n            "bounds": [\n              [\n                0.47514063119888306,\n                0.5089976787567139,\n                0.6184531450271606,\n                0.7161072492599487\n              ],\n              [\n                0.3027656376361847,\n                0.6382051110267639,\n                0.8420312404632568,\n                0.7730302810668945\n              ],\n              [\n                0.3027656376361847,\n                0.5089976787567139,\n                0.8420312404632568,\n                0.7730302810668945\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a boat",\n            "bounds": [\n              [\n                0.5741093754768372,\n                0.4980885684490204,\n                0.6578906178474426,\n                0.7168298363685608\n              ],\n              [\n                0.3027656376361847,\n                0.6382051110267639,\n                0.8420312404632568,\n                0.7730302810668945\n              ],\n              [\n                0.3027656376361847,\n                0.4980885684490204,\n                0.8420312404632568,\n                0.7730302810668945\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 456245,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000457011.jpg",\n        "predicate": "riding",\n        "object_label": "horse",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_457011_230651_58352_234138_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.3334687352180481,\n                0.11304166913032532,\n                0.47556251287460327,\n                0.5081250071525574\n              ],\n              [\n                0.15517187118530273,\n                0.2737083435058594,\n                0.6514062285423279,\n                0.7844791412353516\n              ],\n              [\n                0.15517187118530273,\n                0.11304166913032532,\n                0.6514062285423279,\n                0.7844791412353516\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a horse",\n            "bounds": [\n              [\n                0.7066875100135803,\n                0.31860417127609253,\n                0.8377968668937683,\n                0.780916690826416\n              ],\n              [\n                0.15517187118530273,\n                0.2737083435058594,\n                0.6514062285423279,\n                0.7844791412353516\n              ],\n              [\n                0.15517187118530273,\n                0.2737083435058594,\n                0.8377968668937683,\n                0.7844791412353516\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 457011,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000464888.jpg",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_464888_200122_125247_256669_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.5639328360557556,\n                0.27248436212539673,\n                0.6367786526679993,\n                0.42353126406669617\n              ],\n              [\n                0.02442687749862671,\n                0.6890937685966492,\n                0.2849999964237213,\n                0.8178437352180481\n              ],\n              [\n                0.02442687749862671,\n                0.27248436212539673,\n                0.6367786526679993,\n                0.8178437352180481\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.10278656333684921,\n                0.5953750014305115,\n                0.2369762808084488,\n                0.7872499823570251\n              ],\n              [\n                0.02442687749862671,\n                0.6890937685966492,\n                0.2849999964237213,\n                0.8178437352180481\n              ],\n              [\n                0.02442687749862671,\n                0.5953750014305115,\n                0.2849999964237213,\n                0.8178437352180481\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 464888,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000464888.jpg",\n        "predicate": "holding",\n        "object_label": "suitcase",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_464888_223869_1186932_190039_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.5904347896575928,\n                0.5920156240463257,\n                0.6591106653213501,\n                0.7785624861717224\n              ],\n              [\n                0.45915019512176514,\n                0.6760937571525574,\n                0.5139525532722473,\n                0.7692343592643738\n              ],\n              [\n                0.45915019512176514,\n                0.5920156240463257,\n                0.6591106653213501,\n                0.7785624861717224\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a suitcase",\n            "bounds": [\n              [\n                0.4890909194946289,\n                0.5966249704360962,\n                0.5853161811828613,\n                0.7798905968666077\n              ],\n              [\n                0.45915019512176514,\n                0.6760937571525574,\n                0.5139525532722473,\n                0.7692343592643738\n              ],\n              [\n                0.45915019512176514,\n                0.5966249704360962,\n                0.5853161811828613,\n                0.7798905968666077\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 464888,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000474190.jpg",\n        "predicate": "holding",\n        "object_label": "person",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_474190_512370_482591_482591_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.43229687213897705,\n                0.36110416054725647,\n                0.555343747138977,\n                0.6881874799728394\n              ],\n              [\n                0.466859370470047,\n                0.4606666564941406,\n                0.5258437395095825,\n                0.6921250224113464\n              ],\n              [\n                0.43229687213897705,\n                0.36110416054725647,\n                0.555343747138977,\n                0.6921250224113464\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a person",\n            "bounds": [\n              [\n                0.466859370470047,\n                0.4606666564941406,\n                0.5258437395095825,\n                0.6921250224113464\n              ],\n              [\n                0.466859370470047,\n                0.4606666564941406,\n                0.5258437395095825,\n                0.6921250224113464\n              ],\n              [\n                0.466859370470047,\n                0.4606666564941406,\n                0.5258437395095825,\n                0.6921250224113464\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 474190,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000481036.jpg",\n        "predicate": "riding",\n        "object_label": "train",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_481036_1237019_268194_1275881_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a train",\n            "bounds": [\n              [\n                0.44089582562446594,\n                0.3227187395095825,\n                0.6310416460037231,\n                0.5555468797683716\n              ],\n              [\n                0.0,\n                0.0650937482714653,\n                0.7427291870117188,\n                0.9890312552452087\n              ],\n              [\n                0.0,\n                0.0650937482714653,\n                0.7427291870117188,\n                0.9890312552452087\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a train",\n            "bounds": [\n              [\n                0.8748124837875366,\n                0.14228124916553497,\n                0.9995833039283752,\n                0.45720311999320984\n              ],\n              [\n                0.0,\n                0.0650937482714653,\n                0.7427291870117188,\n                0.9890312552452087\n              ],\n              [\n                0.0,\n                0.0650937482714653,\n                0.9995833039283752,\n                0.9890312552452087\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 481036,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000502756.jpg",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_502756_442481_604412_474343_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.0,\n                0.4480421543121338,\n                0.23180000483989716,\n                1.0\n              ],\n              [\n                0.4429999887943268,\n                0.5345180630683899,\n                0.5017600059509277,\n                0.624518096446991\n              ],\n              [\n                0.0,\n                0.4480421543121338,\n                0.5017600059509277,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.4032000005245209,\n                0.3403012156486511,\n                0.5485799908638,\n                0.8865662813186646\n              ],\n              [\n                0.4429999887943268,\n                0.5345180630683899,\n                0.5017600059509277,\n                0.624518096446991\n              ],\n              [\n                0.4032000005245209,\n                0.3403012156486511,\n                0.5485799908638,\n                0.8865662813186646\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 502756,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000517485.jpg",\n        "predicate": "riding",\n        "object_label": "bicycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_517485_2151314_126923_220994_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.23281249403953552,\n                0.24875587224960327,\n                0.41340625286102295,\n                0.8048121929168701\n              ],\n              [\n                0.4263593852519989,\n                0.4787324070930481,\n                0.974359393119812,\n                1.0\n              ],\n              [\n                0.23281249403953552,\n                0.24875587224960327,\n                0.974359393119812,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a bicycle",\n            "bounds": [\n              [\n                0.6464999914169312,\n                0.18669013679027557,\n                1.0,\n                0.9361032843589783\n              ],\n              [\n                0.4263593852519989,\n                0.4787324070930481,\n                0.974359393119812,\n                1.0\n              ],\n              [\n                0.4263593852519989,\n                0.18669013679027557,\n                1.0,\n                1.0\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 517485,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000519053.jpg",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_519053_1255229_151303_1315996_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.4609687626361847,\n                0.19868852198123932,\n                0.5960624814033508,\n                0.5819672346115112\n              ],\n              [\n                0.38967186212539673,\n                0.2789227068424225,\n                0.6225937604904175,\n                0.7133255004882812\n              ],\n              [\n                0.38967186212539673,\n                0.19868852198123932,\n                0.6225937604904175,\n                0.7133255004882812\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.04284375160932541,\n                0.26208430528640747,\n                0.12017187476158142,\n                0.633583128452301\n              ],\n              [\n                0.38967186212539673,\n                0.2789227068424225,\n                0.6225937604904175,\n                0.7133255004882812\n              ],\n              [\n                0.04284375160932541,\n                0.26208430528640747,\n                0.6225937604904175,\n                0.7133255004882812\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 519053,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000542450.jpg",\n        "predicate": "riding",\n        "object_label": "motorcycle",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_542450_222035_149273_238770_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.3241249918937683,\n                0.4592037498950958,\n                0.4429062604904175,\n                0.7897658348083496\n              ],\n              [\n                0.3343437612056732,\n                0.557564377784729,\n                0.4557812511920929,\n                0.8519437909126282\n              ],\n              [\n                0.3241249918937683,\n                0.4592037498950958,\n                0.4557812511920929,\n                0.8519437909126282\n              ]\n            ]\n          },\n          {\n            "text": "a person riding a motorcycle",\n            "bounds": [\n              [\n                0.6363750100135803,\n                0.49852457642555237,\n                0.7122499942779541,\n                0.9100233912467957\n              ],\n              [\n                0.3343437612056732,\n                0.557564377784729,\n                0.4557812511920929,\n                0.8519437909126282\n              ],\n              [\n                0.3343437612056732,\n                0.49852457642555237,\n                0.7122499942779541,\n                0.9100233912467957\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 542450,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000566245.jpg",\n        "predicate": "riding",\n        "object_label": "elephant",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_566245_490789_580783_1231374_riding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.8583199977874756,\n                0.5258399844169617,\n                1.0,\n                0.9929199814796448\n              ],\n              [\n                0.041039999574422836,\n                0.5013800263404846,\n                0.8638666868209839,\n                1.0\n              ],\n              [\n                0.041039999574422836,\n                0.5013800263404846,\n                1.0,\n                1.0\n              ]\n            ]\n          },\n          {\n            "text": "a person riding an elephant",\n            "bounds": [\n              [\n                0.07077333331108093,\n                0.12849999964237213,\n                0.4320266544818878,\n                0.6620200276374817\n              ],\n              [\n                0.041039999574422836,\n                0.5013800263404846,\n                0.8638666868209839,\n                1.0\n              ],\n              [\n                0.041039999574422836,\n                0.12849999964237213,\n                0.8638666868209839,\n                1.0\n              ]\n            ]\n          }\n        ],\n        "positive_index": 1,\n        "coco_id": 566245,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000068459.jpg",\n        "predicate": "holding",\n        "object_label": "tennis racket",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_68459_441167_657426_457917_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.19895312190055847,\n                0.15716665983200073,\n                0.3883593678474426,\n                0.7863749861717224\n              ],\n              [\n                0.23720312118530273,\n                0.11820833384990692,\n                0.2933281362056732,\n                0.25458332896232605\n              ],\n              [\n                0.19895312190055847,\n                0.11820833384990692,\n                0.3883593678474426,\n                0.7863749861717224\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a tennis racket",\n            "bounds": [\n              [\n                0.03700000047683716,\n                0.1300416737794876,\n                0.22870312631130219,\n                0.6031458377838135\n              ],\n              [\n                0.23720312118530273,\n                0.11820833384990692,\n                0.2933281362056732,\n                0.25458332896232605\n              ],\n              [\n                0.03700000047683716,\n                0.11820833384990692,\n                0.2933281362056732,\n                0.6031458377838135\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 68459,\n        "caption_split": "train"\n      },\n      {\n        "filename": "COCO_train2014_000000073903.jpg",\n        "predicate": "holding",\n        "object_label": "frisbee",\n        "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n        "case_id": "vcoco_actor_73903_1188939_603900_1716867_holding",\n        "task": "wrong_subject_instance",\n        "candidates": [\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.6294353008270264,\n                0.21123437583446503,\n                0.8358588218688965,\n                0.9011250138282776\n              ],\n              [\n                0.7537882328033447,\n                0.11801562458276749,\n                0.872494101524353,\n                0.16107812523841858\n              ],\n              [\n                0.6294353008270264,\n                0.11801562458276749,\n                0.872494101524353,\n                0.9011250138282776\n              ]\n            ]\n          },\n          {\n            "text": "a person holding a frisbee",\n            "bounds": [\n              [\n                0.12207058817148209,\n                0.5058281421661377,\n                0.4531058967113495,\n                0.9892656207084656\n              ],\n              [\n                0.7537882328033447,\n                0.11801562458276749,\n                0.872494101524353,\n                0.16107812523841858\n              ],\n              [\n                0.12207058817148209,\n                0.11801562458276749,\n                0.872494101524353,\n                0.9892656207084656\n              ]\n            ]\n          }\n        ],\n        "positive_index": 0,\n        "coco_id": 73903,\n        "caption_split": "train"\n      }\n    ]\n  },\n  "counts": {\n    "fit": {\n      "pairs": 503,\n      "images": 310,\n      "tasks": {\n        "wrong_predicate": 80,\n        "wrong_subject_instance": 423\n      },\n      "predicates": {\n        "carrying": 68,\n        "holding": 283,\n        "eating": 22,\n        "riding": 35,\n        "sitting_on": 75,\n        "talking_on": 20\n      }\n    },\n    "calibration": {\n      "pairs": 140,\n      "images": 79,\n      "tasks": {\n        "wrong_predicate": 24,\n        "wrong_subject_instance": 116\n      },\n      "predicates": {\n        "holding": 68,\n        "carrying": 29,\n        "sitting_on": 20,\n        "riding": 9,\n        "talking_on": 7,\n        "eating": 7\n      }\n    },\n    "probe": {\n      "pairs": 62,\n      "images": 59,\n      "tasks": {\n        "wrong_object_instance": 12,\n        "wrong_predicate": 12,\n        "wrong_subject_instance": 38\n      },\n      "predicates": {\n        "riding": 34,\n        "holding": 28\n      }\n    }\n  },\n  "sources_sha256": {\n    "actions": "2267afe7adc0a6c3e8ac90fb35f6d3a5bd5cd3c7edebf0622d0cbb7a67fddf19",\n    "instances": "2b26ef46c5a06201d863e3f888fc9cac28f60b16a39ff9a66364800f94458680",\n    "dataset": "2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d",\n    "groups": "6a95392e450a4b810f2937ea792b77abcc6dd4c467f22f140f2867f3b5630249",\n    "probe": "fcc69d897c5751a4480a4cb413fb278582619a4599b4a50452dd8c091acf30bb"\n  },\n  "fit_compatibility": [\n    [\n      "apple",\n      "eat"\n    ],\n    [\n      "apple",\n      "hold"\n    ],\n    [\n      "backpack",\n      "carry"\n    ],\n    [\n      "backpack",\n      "hold"\n    ],\n    [\n      "banana",\n      "carry"\n    ],\n    [\n      "banana",\n      "eat"\n    ],\n    [\n      "banana",\n      "hold"\n    ],\n    [\n      "baseball bat",\n      "hold"\n    ],\n    [\n      "baseball glove",\n      "hold"\n    ],\n    [\n      "bed",\n      "sit"\n    ],\n    [\n      "bench",\n      "sit"\n    ],\n    [\n      "bicycle",\n      "hold"\n    ],\n    [\n      "bicycle",\n      "ride"\n    ],\n    [\n      "bicycle",\n      "sit"\n    ],\n    [\n      "boat",\n      "hold"\n    ],\n    [\n      "boat",\n      "ride"\n    ],\n    [\n      "book",\n      "hold"\n    ],\n    [\n      "bottle",\n      "hold"\n    ],\n    [\n      "bowl",\n      "hold"\n    ],\n    [\n      "broccoli",\n      "eat"\n    ],\n    [\n      "broccoli",\n      "hold"\n    ],\n    [\n      "bus",\n      "ride"\n    ],\n    [\n      "cake",\n      "eat"\n    ],\n    [\n      "cake",\n      "hold"\n    ],\n    [\n      "car",\n      "ride"\n    ],\n    [\n      "carrot",\n      "hold"\n    ],\n    [\n      "cat",\n      "hold"\n    ],\n    [\n      "cell phone",\n      "carry"\n    ],\n    [\n      "cell phone",\n      "hold"\n    ],\n    [\n      "cell phone",\n      "talk_on_phone"\n    ],\n    [\n      "chair",\n      "hold"\n    ],\n    [\n      "chair",\n      "sit"\n    ],\n    [\n      "couch",\n      "sit"\n    ],\n    [\n      "cup",\n      "hold"\n    ],\n    [\n      "dining table",\n      "sit"\n    ],\n    [\n      "dog",\n      "hold"\n    ],\n    [\n      "donut",\n      "eat"\n    ],\n    [\n      "donut",\n      "hold"\n    ],\n    [\n      "elephant",\n      "hold"\n    ],\n    [\n      "elephant",\n      "ride"\n    ],\n    [\n      "elephant",\n      "sit"\n    ],\n    [\n      "fork",\n      "hold"\n    ],\n    [\n      "frisbee",\n      "hold"\n    ],\n    [\n      "handbag",\n      "carry"\n    ],\n    [\n      "handbag",\n      "hold"\n    ],\n    [\n      "horse",\n      "hold"\n    ],\n    [\n      "horse",\n      "ride"\n    ],\n    [\n      "horse",\n      "sit"\n    ],\n    [\n      "hot dog",\n      "eat"\n    ],\n    [\n      "hot dog",\n      "hold"\n    ],\n    [\n      "kite",\n      "carry"\n    ],\n    [\n      "kite",\n      "hold"\n    ],\n    [\n      "knife",\n      "hold"\n    ],\n    [\n      "laptop",\n      "hold"\n    ],\n    [\n      "motorcycle",\n      "hold"\n    ],\n    [\n      "motorcycle",\n      "ride"\n    ],\n    [\n      "motorcycle",\n      "sit"\n    ],\n    [\n      "mouse",\n      "hold"\n    ],\n    [\n      "orange",\n      "carry"\n    ],\n    [\n      "person",\n      "carry"\n    ],\n    [\n      "person",\n      "hold"\n    ],\n    [\n      "pizza",\n      "eat"\n    ],\n    [\n      "pizza",\n      "hold"\n    ],\n    [\n      "refrigerator",\n      "hold"\n    ],\n    [\n      "remote",\n      "hold"\n    ],\n    [\n      "sandwich",\n      "eat"\n    ],\n    [\n      "sandwich",\n      "hold"\n    ],\n    [\n      "scissors",\n      "hold"\n    ],\n    [\n      "sheep",\n      "hold"\n    ],\n    [\n      "skateboard",\n      "carry"\n    ],\n    [\n      "skateboard",\n      "hold"\n    ],\n    [\n      "skis",\n      "carry"\n    ],\n    [\n      "skis",\n      "hold"\n    ],\n    [\n      "snowboard",\n      "carry"\n    ],\n    [\n      "snowboard",\n      "hold"\n    ],\n    [\n      "spoon",\n      "hold"\n    ],\n    [\n      "sports ball",\n      "carry"\n    ],\n    [\n      "sports ball",\n      "hold"\n    ],\n    [\n      "suitcase",\n      "carry"\n    ],\n    [\n      "suitcase",\n      "hold"\n    ],\n    [\n      "suitcase",\n      "sit"\n    ],\n    [\n      "surfboard",\n      "carry"\n    ],\n    [\n      "surfboard",\n      "hold"\n    ],\n    [\n      "teddy bear",\n      "hold"\n    ],\n    [\n      "tennis racket",\n      "carry"\n    ],\n    [\n      "tennis racket",\n      "hold"\n    ],\n    [\n      "tie",\n      "hold"\n    ],\n    [\n      "toilet",\n      "sit"\n    ],\n    [\n      "toothbrush",\n      "hold"\n    ],\n    [\n      "train",\n      "ride"\n    ],\n    [\n      "truck",\n      "hold"\n    ],\n    [\n      "truck",\n      "ride"\n    ],\n    [\n      "umbrella",\n      "carry"\n    ],\n    [\n      "umbrella",\n      "hold"\n    ],\n    [\n      "wine glass",\n      "carry"\n    ],\n    [\n      "wine glass",\n      "hold"\n    ]\n  ],\n  "split_sha256": {\n    "fit": "06513ede3d41919422aaf17238fd379e4c96f066137aab9310340c7c0d4804ba",\n    "calibration": "247a7bd6c03a2a3b44b6139ab8cc97c3ba895278072aac266fa25a3368ec468a",\n    "probe": "d6f95949cccc476ee4bfbcff37cfdccc756db83d5a4957ecffaf45d251ca16b3"\n  },\n  "schedule": [\n    0,\n    80,\n    1,\n    81,\n    2,\n    82,\n    3,\n    83,\n    4,\n    84,\n    5,\n    85,\n    6,\n    86,\n    7,\n    87,\n    8,\n    88,\n    9,\n    89,\n    10,\n    90,\n    11,\n    91,\n    12,\n    92,\n    13,\n    93,\n    14,\n    94,\n    15,\n    95,\n    16,\n    96,\n    17,\n    97,\n    18,\n    98,\n    19,\n    99,\n    20,\n    100,\n    21,\n    101,\n    22,\n    102,\n    23,\n    103,\n    24,\n    104,\n    25,\n    105,\n    26,\n    106,\n    27,\n    107,\n    28,\n    108,\n    29,\n    109,\n    30,\n    110,\n    31,\n    111,\n    32,\n    112,\n    33,\n    113,\n    34,\n    114,\n    35,\n    115,\n    36,\n    116,\n    37,\n    117,\n    38,\n    118,\n    39,\n    119,\n    40,\n    120,\n    41,\n    121,\n    42,\n    122,\n    43,\n    123,\n    44,\n    124,\n    45,\n    125,\n    46,\n    126,\n    47,\n    127,\n    48,\n    128,\n    49,\n    129,\n    50,\n    130,\n    51,\n    131,\n    52,\n    132,\n    53,\n    133,\n    54,\n    134,\n    55,\n    135,\n    56,\n    136,\n    57,\n    137,\n    58,\n    138,\n    59,\n    139,\n    60,\n    140,\n    61,\n    141,\n    62,\n    142,\n    63,\n    143,\n    64,\n    144,\n    65,\n    145,\n    66,\n    146,\n    67,\n    147,\n    68,\n    148,\n    69,\n    149,\n    70,\n    150,\n    71,\n    151,\n    72,\n    152,\n    73,\n    153,\n    74,\n    154,\n    75,\n    155,\n    76,\n    156,\n    77,\n    157,\n    78,\n    158,\n    79,\n    159,\n    0,\n    160,\n    1,\n    161,\n    2,\n    162,\n    3,\n    163,\n    4,\n    164,\n    5,\n    165,\n    6,\n    166,\n    7,\n    167,\n    8,\n    168,\n    9,\n    169,\n    10,\n    170,\n    11,\n    171,\n    12,\n    172,\n    13,\n    173,\n    14,\n    174,\n    15,\n    175,\n    16,\n    176,\n    17,\n    177,\n    18,\n    178,\n    19,\n    179,\n    20,\n    180,\n    21,\n    181,\n    22,\n    182,\n    23,\n    183,\n    24,\n    184,\n    25,\n    185,\n    26,\n    186,\n    27,\n    187,\n    28,\n    188,\n    29,\n    189,\n    30,\n    190,\n    31,\n    191,\n    32,\n    192,\n    33,\n    193,\n    34,\n    194,\n    35,\n    195,\n    36,\n    196,\n    37,\n    197,\n    38,\n    198,\n    39,\n    199,\n    40,\n    200,\n    41,\n    201,\n    42,\n    202,\n    43,\n    203,\n    44,\n    204,\n    45,\n    205,\n    46,\n    206,\n    47,\n    207,\n    48,\n    208,\n    49,\n    209,\n    50,\n    210,\n    51,\n    211,\n    52,\n    212,\n    53,\n    213,\n    54,\n    214,\n    55,\n    215,\n    56,\n    216,\n    57,\n    217,\n    58,\n    218,\n    59,\n    219,\n    60,\n    220,\n    61,\n    221,\n    62,\n    222,\n    63,\n    223,\n    64,\n    224,\n    65,\n    225,\n    66,\n    226,\n    67,\n    227,\n    68,\n    228,\n    69,\n    229,\n    70,\n    230,\n    71,\n    231,\n    72,\n    232,\n    73,\n    233,\n    74,\n    234,\n    75,\n    235,\n    76,\n    236,\n    77,\n    237,\n    78,\n    238,\n    79,\n    239,\n    0,\n    240,\n    1,\n    241,\n    2,\n    242,\n    3,\n    243,\n    4,\n    244,\n    5,\n    245,\n    6,\n    246,\n    7,\n    247,\n    8,\n    248,\n    9,\n    249,\n    10,\n    250,\n    11,\n    251,\n    12,\n    252,\n    13,\n    253,\n    14,\n    254,\n    15,\n    255,\n    16,\n    256,\n    17,\n    257,\n    18,\n    258,\n    19,\n    259,\n    20,\n    260,\n    21,\n    261,\n    22,\n    262,\n    23,\n    263,\n    24,\n    264,\n    25,\n    265,\n    26,\n    266,\n    27,\n    267,\n    28,\n    268,\n    29,\n    269,\n    30,\n    270,\n    31,\n    271,\n    32,\n    272,\n    33,\n    273,\n    34,\n    274,\n    35,\n    275,\n    36,\n    276,\n    37,\n    277,\n    38,\n    278,\n    39,\n    279,\n    40,\n    280,\n    41,\n    281,\n    42,\n    282,\n    43,\n    283,\n    44,\n    284,\n    45,\n    285,\n    46,\n    286,\n    47,\n    287,\n    48,\n    288,\n    49,\n    289,\n    50,\n    290,\n    51,\n    291,\n    52,\n    292,\n    53,\n    293,\n    54,\n    294,\n    55,\n    295,\n    56,\n    296,\n    57,\n    297,\n    58,\n    298,\n    59,\n    299,\n    60,\n    300,\n    61,\n    301,\n    62,\n    302,\n    63,\n    303,\n    64,\n    304,\n    65,\n    305,\n    66,\n    306,\n    67,\n    307,\n    68,\n    308,\n    69,\n    309,\n    70,\n    310,\n    71,\n    311,\n    72,\n    312,\n    73,\n    313,\n    74,\n    314,\n    75,\n    315,\n    76,\n    316,\n    77,\n    317,\n    78,\n    318,\n    79,\n    319,\n    0,\n    320,\n    1,\n    321,\n    2,\n    322,\n    3,\n    323,\n    4,\n    324,\n    5,\n    325,\n    6,\n    326,\n    7,\n    327,\n    8,\n    328,\n    9,\n    329,\n    10,\n    330,\n    11,\n    331,\n    12,\n    332,\n    13,\n    333,\n    14,\n    334,\n    15,\n    335,\n    16,\n    336,\n    17,\n    337,\n    18,\n    338,\n    19,\n    339,\n    20,\n    340,\n    21,\n    341,\n    22,\n    342,\n    23,\n    343,\n    24,\n    344,\n    25,\n    345,\n    26,\n    346,\n    27,\n    347,\n    28,\n    348,\n    29,\n    349,\n    30,\n    350,\n    31,\n    351,\n    32,\n    352,\n    33,\n    353,\n    34,\n    354,\n    35,\n    355,\n    36,\n    356,\n    37,\n    357,\n    38,\n    358,\n    39,\n    359,\n    40,\n    360,\n    41,\n    361,\n    42,\n    362,\n    43,\n    363,\n    44,\n    364,\n    45,\n    365,\n    46,\n    366,\n    47,\n    367,\n    48,\n    368,\n    49,\n    369,\n    50,\n    370,\n    51,\n    371,\n    52,\n    372,\n    53,\n    373,\n    54,\n    374,\n    55,\n    375,\n    56,\n    376,\n    57,\n    377,\n    58,\n    378,\n    59,\n    379,\n    60,\n    380,\n    61,\n    381,\n    62,\n    382,\n    63,\n    383,\n    64,\n    384,\n    65,\n    385,\n    66,\n    386,\n    67,\n    387,\n    68,\n    388,\n    69,\n    389,\n    70,\n    390,\n    71,\n    391,\n    72,\n    392,\n    73,\n    393,\n    74,\n    394,\n    75,\n    395,\n    76,\n    396,\n    77,\n    397,\n    78,\n    398,\n    79,\n    399,\n    0,\n    400,\n    1,\n    401,\n    2,\n    402,\n    3,\n    403,\n    4,\n    404,\n    5,\n    405,\n    6,\n    406,\n    7,\n    407,\n    8,\n    408,\n    9,\n    409,\n    10,\n    410,\n    11,\n    411,\n    12,\n    412,\n    13,\n    413,\n    14,\n    414,\n    15,\n    415,\n    16,\n    416,\n    17,\n    417,\n    18,\n    418,\n    19,\n    419,\n    20,\n    420,\n    21,\n    421,\n    22,\n    422,\n    23,\n    423,\n    24,\n    424,\n    25,\n    425,\n    26,\n    426,\n    27,\n    427,\n    28,\n    428,\n    29,\n    429,\n    30,\n    430,\n    31,\n    431,\n    32,\n    432,\n    33,\n    433,\n    34,\n    434,\n    35,\n    435,\n    36,\n    436,\n    37,\n    437,\n    38,\n    438,\n    39,\n    439,\n    40,\n    440,\n    41,\n    441,\n    42,\n    442,\n    43,\n    443,\n    44,\n    444,\n    45,\n    445,\n    46,\n    446,\n    47,\n    447,\n    48,\n    448,\n    49,\n    449,\n    50,\n    450,\n    51,\n    451,\n    52,\n    452,\n    53,\n    453,\n    54,\n    454,\n    55,\n    455,\n    56,\n    456,\n    57,\n    457,\n    58,\n    458,\n    59,\n    459,\n    60,\n    460,\n    61,\n    461,\n    62,\n    462,\n    63,\n    463,\n    64,\n    464,\n    65,\n    465,\n    66,\n    466,\n    67,\n    467,\n    68,\n    468,\n    69,\n    469,\n    70,\n    470,\n    71,\n    471,\n    72,\n    472,\n    73,\n    473,\n    74,\n    474,\n    75,\n    475,\n    76,\n    476,\n    77,\n    477,\n    78,\n    478,\n    79,\n    479,\n    0,\n    480,\n    1,\n    481,\n    2,\n    482,\n    3,\n    483,\n    4,\n    484,\n    5,\n    485,\n    6,\n    486,\n    7,\n    487,\n    8,\n    488,\n    9,\n    489,\n    10,\n    490,\n    11,\n    491,\n    12,\n    492,\n    13,\n    493,\n    14,\n    494,\n    15,\n    495,\n    16,\n    496,\n    17,\n    497,\n    18,\n    498,\n    19,\n    499,\n    20,\n    500,\n    21,\n    501,\n    22,\n    502,\n    23,\n    80,\n    24,\n    81,\n    25,\n    82,\n    26,\n    83,\n    27,\n    84,\n    28,\n    85,\n    29,\n    86,\n    30,\n    87,\n    31,\n    88,\n    32,\n    89,\n    33,\n    90,\n    34,\n    91,\n    35,\n    92,\n    36,\n    93,\n    37,\n    94,\n    38,\n    95,\n    39,\n    96,\n    40,\n    97,\n    41,\n    98,\n    42,\n    99,\n    43,\n    100,\n    44,\n    101,\n    45,\n    102,\n    46,\n    103,\n    47,\n    104,\n    48,\n    105,\n    49,\n    106,\n    50,\n    107,\n    51,\n    108,\n    52,\n    109,\n    53,\n    110,\n    54,\n    111,\n    55,\n    112,\n    56,\n    113,\n    57,\n    114,\n    58,\n    115,\n    59,\n    116,\n    60,\n    117,\n    61,\n    118,\n    62,\n    119,\n    63,\n    120,\n    64,\n    121,\n    65,\n    122,\n    66,\n    123,\n    67,\n    124,\n    68,\n    125,\n    69,\n    126,\n    70,\n    127,\n    71,\n    128,\n    72,\n    129,\n    73,\n    130,\n    74,\n    131,\n    75,\n    132,\n    76,\n    133,\n    77,\n    134,\n    78,\n    135,\n    79,\n    136,\n    0,\n    137,\n    1,\n    138,\n    2,\n    139,\n    3,\n    140,\n    4,\n    141,\n    5,\n    142,\n    6,\n    143,\n    7,\n    144,\n    8,\n    145,\n    9,\n    146,\n    10,\n    147,\n    11,\n    148,\n    12,\n    149,\n    13,\n    150,\n    14,\n    151,\n    15,\n    152,\n    16,\n    153,\n    17,\n    154,\n    18,\n    155,\n    19,\n    156,\n    20,\n    157,\n    21,\n    158,\n    22,\n    159,\n    23,\n    160,\n    24,\n    161,\n    25,\n    162,\n    26,\n    163,\n    27,\n    164,\n    28,\n    165,\n    29,\n    166,\n    30,\n    167,\n    31,\n    168,\n    32,\n    169,\n    33,\n    170,\n    34,\n    171,\n    35,\n    172,\n    36,\n    173,\n    37,\n    174,\n    38,\n    175,\n    39,\n    176,\n    40,\n    177,\n    41,\n    178,\n    42,\n    179,\n    43,\n    180,\n    44,\n    181,\n    45,\n    182,\n    46,\n    183,\n    47,\n    184,\n    48,\n    185,\n    49,\n    186,\n    50,\n    187,\n    51,\n    188,\n    52,\n    189,\n    53,\n    190,\n    54,\n    191,\n    55,\n    192,\n    56,\n    193,\n    57,\n    194,\n    58,\n    195,\n    59,\n    196,\n    60,\n    197,\n    61,\n    198,\n    62,\n    199,\n    63,\n    200,\n    64,\n    201,\n    65,\n    202,\n    66,\n    203,\n    67,\n    204,\n    68,\n    205,\n    69,\n    206,\n    70,\n    207,\n    71,\n    208,\n    72,\n    209,\n    73,\n    210,\n    74,\n    211,\n    75,\n    212,\n    76,\n    213,\n    77,\n    214,\n    78,\n    215,\n    79,\n    216,\n    0,\n    217,\n    1,\n    218,\n    2,\n    219,\n    3,\n    220,\n    4,\n    221,\n    5,\n    222,\n    6,\n    223,\n    7,\n    224,\n    8,\n    225,\n    9,\n    226,\n    10,\n    227,\n    11,\n    228,\n    12,\n    229,\n    13,\n    230,\n    14,\n    231,\n    15,\n    232,\n    16,\n    233,\n    17,\n    234,\n    18,\n    235,\n    19,\n    236,\n    20,\n    237,\n    21,\n    238,\n    22,\n    239,\n    23,\n    240,\n    24,\n    241,\n    25,\n    242,\n    26,\n    243,\n    27,\n    244,\n    28,\n    245,\n    29,\n    246,\n    30,\n    247,\n    31,\n    248,\n    32,\n    249,\n    33,\n    250,\n    34,\n    251,\n    35,\n    252,\n    36,\n    253,\n    37,\n    254,\n    38,\n    255,\n    39,\n    256,\n    40,\n    257,\n    41,\n    258,\n    42,\n    259,\n    43,\n    260,\n    44,\n    261,\n    45,\n    262,\n    46,\n    263,\n    47,\n    264,\n    48,\n    265,\n    49,\n    266,\n    50,\n    267,\n    51,\n    268,\n    52,\n    269,\n    53,\n    270,\n    54,\n    271,\n    55,\n    272,\n    56,\n    273,\n    57,\n    274,\n    58,\n    275,\n    59,\n    276,\n    60,\n    277,\n    61,\n    278,\n    62,\n    279,\n    63,\n    280,\n    64,\n    281,\n    65,\n    282,\n    66,\n    283,\n    67,\n    284,\n    68,\n    285,\n    69,\n    286,\n    70,\n    287,\n    71,\n    288,\n    72,\n    289,\n    73,\n    290,\n    74,\n    291,\n    75,\n    292,\n    76,\n    293,\n    77,\n    294,\n    78,\n    295,\n    79,\n    296,\n    0,\n    297,\n    1,\n    298,\n    2,\n    299,\n    3,\n    300,\n    4,\n    301,\n    5,\n    302,\n    6,\n    303,\n    7,\n    304,\n    8,\n    305,\n    9,\n    306,\n    10,\n    307,\n    11,\n    308,\n    12,\n    309,\n    13,\n    310,\n    14,\n    311,\n    15,\n    312,\n    16,\n    313,\n    17,\n    314,\n    18,\n    315,\n    19,\n    316,\n    20,\n    317,\n    21,\n    318,\n    22,\n    319,\n    23,\n    320,\n    24,\n    321,\n    25,\n    322,\n    26,\n    323,\n    27,\n    324,\n    28,\n    325,\n    29,\n    326,\n    30,\n    327,\n    31,\n    328,\n    32,\n    329,\n    33,\n    330,\n    34,\n    331,\n    35,\n    332,\n    36,\n    333,\n    37,\n    334,\n    38,\n    335,\n    39,\n    336,\n    40,\n    337,\n    41,\n    338,\n    42,\n    339,\n    43,\n    340,\n    44,\n    341,\n    45,\n    342,\n    46,\n    343,\n    47,\n    344,\n    48,\n    345,\n    49,\n    346,\n    50,\n    347,\n    51,\n    348,\n    52,\n    349,\n    53,\n    350,\n    54,\n    351,\n    55,\n    352,\n    56,\n    353,\n    57,\n    354,\n    58,\n    355,\n    59,\n    356,\n    60,\n    357,\n    61,\n    358,\n    62,\n    359,\n    63,\n    360,\n    64,\n    361,\n    65,\n    362,\n    66,\n    363,\n    67,\n    364,\n    68,\n    365,\n    69,\n    366,\n    70,\n    367,\n    71,\n    368,\n    72,\n    369,\n    73,\n    370,\n    74,\n    371,\n    75,\n    372,\n    76,\n    373,\n    77,\n    374,\n    78,\n    375,\n    79,\n    376,\n    0,\n    377,\n    1,\n    378,\n    2,\n    379,\n    3,\n    380,\n    4,\n    381,\n    5,\n    382,\n    6,\n    383,\n    7,\n    384,\n    8,\n    385,\n    9,\n    386,\n    10,\n    387,\n    11,\n    388,\n    12,\n    389,\n    13,\n    390,\n    14,\n    391,\n    15,\n    392,\n    16,\n    393,\n    17,\n    394,\n    18,\n    395,\n    19,\n    396,\n    20,\n    397,\n    21,\n    398,\n    22,\n    399,\n    23,\n    400,\n    24,\n    401,\n    25,\n    402,\n    26,\n    403,\n    27,\n    404,\n    28,\n    405,\n    29,\n    406,\n    30,\n    407,\n    31,\n    408,\n    32,\n    409,\n    33,\n    410,\n    34,\n    411,\n    35,\n    412,\n    36,\n    413,\n    37,\n    414,\n    38,\n    415,\n    39,\n    416,\n    40,\n    417,\n    41,\n    418,\n    42,\n    419,\n    43,\n    420,\n    44,\n    421,\n    45,\n    422,\n    46,\n    423,\n    47,\n    424,\n    48,\n    425,\n    49,\n    426,\n    50,\n    427,\n    51,\n    428,\n    52,\n    429,\n    53,\n    430,\n    54,\n    431,\n    55,\n    432,\n    56,\n    433,\n    57,\n    434,\n    58,\n    435,\n    59,\n    436,\n    60,\n    437,\n    61,\n    438,\n    62,\n    439,\n    63,\n    440,\n    64,\n    441,\n    65,\n    442,\n    66,\n    443,\n    67,\n    444,\n    68,\n    445,\n    69,\n    446,\n    70,\n    447,\n    71,\n    448,\n    72,\n    449,\n    73,\n    450,\n    74,\n    451,\n    75,\n    452,\n    76,\n    453,\n    77,\n    454,\n    78,\n    455,\n    79,\n    456,\n    0,\n    457,\n    1,\n    458,\n    2,\n    459,\n    3,\n    460,\n    4,\n    461,\n    5,\n    462,\n    6,\n    463,\n    7,\n    464,\n    8,\n    465,\n    9,\n    466,\n    10,\n    467,\n    11,\n    468,\n    12,\n    469,\n    13,\n    470,\n    14,\n    471,\n    15,\n    472,\n    16,\n    473,\n    17,\n    474,\n    18,\n    475,\n    19,\n    476,\n    20,\n    477,\n    21,\n    478,\n    22,\n    479,\n    23,\n    480,\n    24,\n    481,\n    25,\n    482,\n    26,\n    483,\n    27,\n    484,\n    28,\n    485,\n    29,\n    486,\n    30,\n    487,\n    31,\n    488,\n    32,\n    489,\n    33,\n    490,\n    34,\n    491,\n    35,\n    492,\n    36,\n    493,\n    37,\n    494,\n    38,\n    495,\n    39,\n    496,\n    40,\n    497,\n    41,\n    498,\n    42,\n    499,\n    43,\n    500,\n    44,\n    501,\n    45,\n    502,\n    46,\n    80,\n    47,\n    81,\n    48,\n    82,\n    49,\n    83,\n    50,\n    84,\n    51,\n    85,\n    52,\n    86,\n    53,\n    87,\n    54,\n    88,\n    55,\n    89,\n    56,\n    90,\n    57,\n    91,\n    58,\n    92,\n    59,\n    93,\n    60,\n    94,\n    61,\n    95,\n    62,\n    96,\n    63,\n    97,\n    64,\n    98,\n    65,\n    99,\n    66,\n    100,\n    67,\n    101,\n    68,\n    102,\n    69,\n    103,\n    70,\n    104,\n    71,\n    105,\n    72,\n    106,\n    73,\n    107,\n    74,\n    108,\n    75,\n    109,\n    76,\n    110,\n    77,\n    111,\n    78,\n    112,\n    79,\n    113,\n    0,\n    114,\n    1,\n    115,\n    2,\n    116,\n    3,\n    117,\n    4,\n    118,\n    5,\n    119,\n    6,\n    120,\n    7,\n    121,\n    8,\n    122,\n    9,\n    123,\n    10,\n    124,\n    11,\n    125,\n    12,\n    126,\n    13,\n    127,\n    14,\n    128,\n    15,\n    129,\n    16,\n    130,\n    17,\n    131,\n    18,\n    132,\n    19,\n    133,\n    20,\n    134,\n    21,\n    135,\n    22,\n    136,\n    23,\n    137,\n    24,\n    138,\n    25,\n    139,\n    26,\n    140,\n    27,\n    141,\n    28,\n    142,\n    29,\n    143,\n    30,\n    144,\n    31,\n    145,\n    32,\n    146,\n    33,\n    147,\n    34,\n    148,\n    35,\n    149,\n    36,\n    150,\n    37,\n    151,\n    38,\n    152,\n    39,\n    153,\n    40,\n    154,\n    41,\n    155,\n    42,\n    156,\n    43,\n    157,\n    44,\n    158,\n    45,\n    159,\n    46,\n    160,\n    47,\n    161,\n    48,\n    162,\n    49,\n    163,\n    50,\n    164,\n    51,\n    165,\n    52,\n    166,\n    53,\n    167,\n    54,\n    168,\n    55,\n    169,\n    56,\n    170,\n    57,\n    171,\n    58,\n    172,\n    59,\n    173,\n    60,\n    174,\n    61,\n    175,\n    62,\n    176,\n    63,\n    177,\n    64,\n    178,\n    65,\n    179,\n    66,\n    180,\n    67,\n    181,\n    68,\n    182,\n    69,\n    183,\n    70,\n    184,\n    71,\n    185,\n    72,\n    186,\n    73,\n    187,\n    74,\n    188,\n    75,\n    189,\n    76,\n    190,\n    77,\n    191,\n    78,\n    192,\n    79,\n    193,\n    0,\n    194,\n    1,\n    195,\n    2,\n    196,\n    3,\n    197,\n    4,\n    198,\n    5,\n    199,\n    6,\n    200,\n    7,\n    201,\n    8,\n    202,\n    9,\n    203,\n    10,\n    204,\n    11,\n    205,\n    12,\n    206,\n    13,\n    207,\n    14,\n    208,\n    15,\n    209,\n    16,\n    210,\n    17,\n    211,\n    18,\n    212,\n    19,\n    213,\n    20,\n    214,\n    21,\n    215,\n    22,\n    216,\n    23,\n    217,\n    24,\n    218,\n    25,\n    219,\n    26,\n    220,\n    27,\n    221,\n    28,\n    222,\n    29,\n    223,\n    30,\n    224,\n    31,\n    225,\n    32,\n    226,\n    33,\n    227,\n    34,\n    228,\n    35,\n    229,\n    36,\n    230,\n    37,\n    231,\n    38,\n    232,\n    39,\n    233,\n    40,\n    234,\n    41,\n    235,\n    42,\n    236,\n    43,\n    237,\n    44,\n    238,\n    45,\n    239,\n    46,\n    240,\n    47,\n    241,\n    48,\n    242,\n    49,\n    243,\n    50,\n    244,\n    51,\n    245,\n    52,\n    246,\n    53,\n    247,\n    54,\n    248,\n    55,\n    249,\n    56,\n    250,\n    57,\n    251,\n    58,\n    252,\n    59,\n    253,\n    60,\n    254,\n    61,\n    255,\n    62,\n    256,\n    63,\n    257,\n    64,\n    258,\n    65,\n    259,\n    66,\n    260,\n    67,\n    261,\n    68,\n    262,\n    69,\n    263,\n    70,\n    264,\n    71,\n    265,\n    72,\n    266,\n    73,\n    267,\n    74,\n    268,\n    75,\n    269,\n    76,\n    270,\n    77,\n    271,\n    78,\n    272,\n    79,\n    273,\n    0,\n    274,\n    1,\n    275,\n    2,\n    276,\n    3,\n    277,\n    4,\n    278,\n    5,\n    279,\n    6,\n    280,\n    7,\n    281,\n    8,\n    282,\n    9,\n    283,\n    10,\n    284,\n    11,\n    285,\n    12,\n    286,\n    13,\n    287,\n    14,\n    288,\n    15,\n    289,\n    16,\n    290,\n    17,\n    291,\n    18,\n    292,\n    19,\n    293,\n    20,\n    294,\n    21,\n    295,\n    22,\n    296,\n    23,\n    297,\n    24,\n    298,\n    25,\n    299,\n    26,\n    300,\n    27,\n    301,\n    28,\n    302,\n    29,\n    303,\n    30,\n    304,\n    31,\n    305,\n    32,\n    306,\n    33,\n    307,\n    34,\n    308,\n    35,\n    309,\n    36,\n    310,\n    37,\n    311,\n    38,\n    312,\n    39,\n    313,\n    40,\n    314,\n    41,\n    315,\n    42,\n    316,\n    43,\n    317,\n    44,\n    318,\n    45,\n    319,\n    46,\n    320,\n    47,\n    321,\n    48,\n    322,\n    49,\n    323,\n    50,\n    324,\n    51,\n    325,\n    52,\n    326,\n    53,\n    327,\n    54,\n    328,\n    55,\n    329,\n    56,\n    330,\n    57,\n    331,\n    58,\n    332,\n    59,\n    333,\n    60,\n    334,\n    61,\n    335,\n    62,\n    336,\n    63,\n    337,\n    64,\n    338,\n    65,\n    339,\n    66,\n    340,\n    67,\n    341,\n    68,\n    342,\n    69,\n    343,\n    70,\n    344,\n    71,\n    345,\n    72,\n    346,\n    73,\n    347,\n    74,\n    348,\n    75,\n    349,\n    76,\n    350,\n    77,\n    351,\n    78,\n    352,\n    79,\n    353,\n    0,\n    354,\n    1,\n    355,\n    2,\n    356,\n    3,\n    357,\n    4,\n    358,\n    5,\n    359,\n    6,\n    360,\n    7,\n    361,\n    8,\n    362,\n    9,\n    363,\n    10,\n    364,\n    11,\n    365,\n    12,\n    366,\n    13,\n    367,\n    14,\n    368,\n    15,\n    369,\n    16,\n    370,\n    17,\n    371,\n    18,\n    372,\n    19,\n    373,\n    20,\n    374,\n    21,\n    375,\n    22,\n    376,\n    23,\n    377,\n    24,\n    378,\n    25,\n    379,\n    26,\n    380,\n    27,\n    381,\n    28,\n    382,\n    29,\n    383,\n    30,\n    384,\n    31,\n    385,\n    32,\n    386,\n    33,\n    387,\n    34,\n    388,\n    35,\n    389,\n    36,\n    390,\n    37,\n    391,\n    38,\n    392,\n    39,\n    393,\n    40,\n    394,\n    41,\n    395,\n    42,\n    396,\n    43,\n    397,\n    44,\n    398,\n    45,\n    399,\n    46,\n    400,\n    47,\n    401,\n    48,\n    402,\n    49,\n    403,\n    50,\n    404,\n    51,\n    405,\n    52,\n    406,\n    53,\n    407,\n    54,\n    408,\n    55,\n    409,\n    56,\n    410,\n    57,\n    411,\n    58,\n    412,\n    59,\n    413,\n    60,\n    414,\n    61,\n    415,\n    62,\n    416,\n    63,\n    417,\n    64,\n    418,\n    65,\n    419,\n    66,\n    420,\n    67,\n    421,\n    68,\n    422,\n    69,\n    423,\n    70,\n    424,\n    71,\n    425,\n    72,\n    426,\n    73,\n    427,\n    74,\n    428,\n    75,\n    429,\n    76,\n    430,\n    77,\n    431,\n    78,\n    432,\n    79,\n    433,\n    0,\n    434,\n    1,\n    435,\n    2,\n    436,\n    3,\n    437,\n    4,\n    438,\n    5,\n    439,\n    6,\n    440,\n    7,\n    441,\n    8,\n    442,\n    9,\n    443,\n    10,\n    444,\n    11,\n    445,\n    12,\n    446,\n    13,\n    447,\n    14,\n    448,\n    15,\n    449,\n    16,\n    450,\n    17,\n    451,\n    18,\n    452,\n    19,\n    453,\n    20,\n    454,\n    21,\n    455,\n    22,\n    456,\n    23,\n    457,\n    24,\n    458,\n    25,\n    459,\n    26,\n    460,\n    27,\n    461,\n    28,\n    462,\n    29,\n    463,\n    30,\n    464,\n    31,\n    465,\n    32,\n    466,\n    33,\n    467,\n    34,\n    468,\n    35,\n    469,\n    36,\n    470,\n    37,\n    471,\n    38,\n    472,\n    39,\n    473,\n    40,\n    474,\n    41,\n    475,\n    42,\n    476,\n    43,\n    477,\n    44,\n    478,\n    45,\n    479,\n    46,\n    480,\n    47,\n    481,\n    48,\n    482,\n    49,\n    483,\n    50,\n    484,\n    51,\n    485,\n    52,\n    486,\n    53,\n    487,\n    54,\n    488,\n    55,\n    489,\n    56,\n    490,\n    57,\n    491,\n    58,\n    492,\n    59,\n    493,\n    60,\n    494,\n    61,\n    495,\n    62,\n    496,\n    63,\n    497,\n    64,\n    498,\n    65,\n    499,\n    66,\n    500,\n    67,\n    501,\n    68,\n    502,\n    69,\n    80,\n    70,\n    81,\n    71,\n    82,\n    72,\n    83,\n    73,\n    84,\n    74,\n    85,\n    75,\n    86,\n    76,\n    87,\n    77,\n    88,\n    78,\n    89,\n    79,\n    90,\n    0,\n    91,\n    1,\n    92,\n    2,\n    93,\n    3,\n    94,\n    4,\n    95,\n    5,\n    96,\n    6,\n    97,\n    7,\n    98,\n    8,\n    99,\n    9,\n    100,\n    10,\n    101,\n    11,\n    102,\n    12,\n    103,\n    13,\n    104,\n    14,\n    105,\n    15,\n    106,\n    16,\n    107,\n    17,\n    108,\n    18,\n    109,\n    19,\n    110,\n    20,\n    111,\n    21,\n    112,\n    22,\n    113,\n    23,\n    114,\n    24,\n    115,\n    25,\n    116,\n    26,\n    117,\n    27,\n    118,\n    28,\n    119,\n    29,\n    120,\n    30,\n    121,\n    31,\n    122,\n    32,\n    123,\n    33,\n    124,\n    34,\n    125,\n    35,\n    126,\n    36,\n    127,\n    37,\n    128,\n    38,\n    129,\n    39,\n    130,\n    40,\n    131,\n    41,\n    132,\n    42,\n    133,\n    43,\n    134,\n    44,\n    135,\n    45,\n    136,\n    46,\n    137,\n    47,\n    138,\n    48,\n    139,\n    49,\n    140,\n    50,\n    141,\n    51,\n    142,\n    52,\n    143,\n    53,\n    144,\n    54,\n    145,\n    55,\n    146,\n    56,\n    147,\n    57,\n    148,\n    58,\n    149,\n    59,\n    150,\n    60,\n    151,\n    61,\n    152,\n    62,\n    153,\n    63,\n    154,\n    64,\n    155,\n    65,\n    156,\n    66,\n    157,\n    67,\n    158,\n    68,\n    159,\n    69,\n    160,\n    70,\n    161,\n    71,\n    162,\n    72,\n    163,\n    73,\n    164,\n    74,\n    165,\n    75,\n    166,\n    76,\n    167,\n    77,\n    168,\n    78,\n    169,\n    79,\n    170,\n    0,\n    171,\n    1,\n    172,\n    2,\n    173,\n    3,\n    174,\n    4,\n    175,\n    5,\n    176,\n    6,\n    177,\n    7,\n    178,\n    8,\n    179,\n    9,\n    180,\n    10,\n    181,\n    11,\n    182,\n    12,\n    183,\n    13,\n    184,\n    14,\n    185,\n    15,\n    186,\n    16,\n    187,\n    17,\n    188,\n    18,\n    189,\n    19,\n    190,\n    20,\n    191,\n    21,\n    192,\n    22,\n    193,\n    23,\n    194,\n    24,\n    195,\n    25,\n    196,\n    26,\n    197,\n    27,\n    198,\n    28,\n    199,\n    29,\n    200,\n    30,\n    201,\n    31,\n    202,\n    32,\n    203,\n    33,\n    204,\n    34,\n    205,\n    35,\n    206,\n    36,\n    207,\n    37,\n    208,\n    38,\n    209,\n    39,\n    210,\n    40,\n    211,\n    41,\n    212,\n    42,\n    213,\n    43,\n    214,\n    44,\n    215,\n    45,\n    216,\n    46,\n    217,\n    47,\n    218,\n    48,\n    219,\n    49,\n    220,\n    50,\n    221,\n    51,\n    222,\n    52,\n    223,\n    53,\n    224,\n    54,\n    225,\n    55,\n    226,\n    56,\n    227,\n    57,\n    228,\n    58,\n    229,\n    59,\n    230,\n    60,\n    231,\n    61,\n    232,\n    62,\n    233,\n    63,\n    234,\n    64,\n    235,\n    65,\n    236,\n    66,\n    237,\n    67,\n    238,\n    68,\n    239,\n    69,\n    240,\n    70,\n    241,\n    71,\n    242,\n    72,\n    243,\n    73,\n    244,\n    74,\n    245,\n    75,\n    246,\n    76,\n    247,\n    77,\n    248,\n    78,\n    249,\n    79,\n    250,\n    0,\n    251,\n    1,\n    252,\n    2,\n    253,\n    3,\n    254,\n    4,\n    255,\n    5,\n    256,\n    6,\n    257,\n    7,\n    258,\n    8,\n    259,\n    9,\n    260,\n    10,\n    261,\n    11,\n    262,\n    12,\n    263,\n    13,\n    264,\n    14,\n    265,\n    15,\n    266,\n    16,\n    267,\n    17,\n    268,\n    18,\n    269,\n    19,\n    270,\n    20,\n    271,\n    21,\n    272,\n    22,\n    273,\n    23,\n    274,\n    24,\n    275,\n    25,\n    276,\n    26,\n    277,\n    27,\n    278,\n    28,\n    279,\n    29,\n    280,\n    30,\n    281,\n    31,\n    282,\n    32,\n    283,\n    33,\n    284,\n    34,\n    285,\n    35,\n    286,\n    36,\n    287,\n    37,\n    288,\n    38,\n    289,\n    39,\n    290,\n    40,\n    291,\n    41,\n    292,\n    42,\n    293,\n    43,\n    294,\n    44,\n    295,\n    45,\n    296,\n    46,\n    297,\n    47,\n    298,\n    48,\n    299,\n    49,\n    300,\n    50,\n    301,\n    51,\n    302,\n    52,\n    303,\n    53,\n    304,\n    54,\n    305,\n    55,\n    306,\n    56,\n    307,\n    57,\n    308,\n    58,\n    309,\n    59,\n    310,\n    60,\n    311,\n    61,\n    312,\n    62,\n    313,\n    63,\n    314,\n    64,\n    315,\n    65,\n    316,\n    66,\n    317,\n    67,\n    318,\n    68,\n    319,\n    69,\n    320,\n    70,\n    321,\n    71,\n    322,\n    72,\n    323,\n    73,\n    324,\n    74,\n    325,\n    75,\n    326,\n    76,\n    327,\n    77,\n    328,\n    78,\n    329,\n    79,\n    330,\n    0,\n    331,\n    1,\n    332,\n    2,\n    333,\n    3,\n    334,\n    4,\n    335,\n    5,\n    336,\n    6,\n    337,\n    7,\n    338,\n    8,\n    339,\n    9,\n    340,\n    10,\n    341,\n    11,\n    342,\n    12,\n    343,\n    13,\n    344,\n    14,\n    345,\n    15,\n    346,\n    16,\n    347,\n    17,\n    348,\n    18,\n    349,\n    19,\n    350,\n    20,\n    351,\n    21,\n    352,\n    22,\n    353,\n    23,\n    354,\n    24,\n    355,\n    25,\n    356,\n    26,\n    357,\n    27,\n    358,\n    28,\n    359,\n    29,\n    360,\n    30,\n    361,\n    31,\n    362,\n    32,\n    363,\n    33,\n    364,\n    34,\n    365,\n    35,\n    366,\n    36,\n    367,\n    37,\n    368,\n    38,\n    369,\n    39,\n    370,\n    40,\n    371,\n    41,\n    372,\n    42,\n    373,\n    43,\n    374,\n    44,\n    375,\n    45,\n    376,\n    46,\n    377,\n    47,\n    378,\n    48,\n    379,\n    49,\n    380,\n    50,\n    381,\n    51,\n    382,\n    52,\n    383,\n    53,\n    384,\n    54,\n    385,\n    55,\n    386,\n    56,\n    387,\n    57,\n    388,\n    58,\n    389,\n    59,\n    390,\n    60,\n    391,\n    61,\n    392,\n    62,\n    393,\n    63,\n    394,\n    64,\n    395,\n    65,\n    396,\n    66,\n    397,\n    67,\n    398,\n    68,\n    399,\n    69,\n    400,\n    70,\n    401,\n    71,\n    402,\n    72,\n    403,\n    73,\n    404,\n    74,\n    405,\n    75,\n    406,\n    76,\n    407,\n    77,\n    408,\n    78,\n    409,\n    79,\n    410,\n    0,\n    411,\n    1,\n    412,\n    2,\n    413,\n    3,\n    414,\n    4,\n    415,\n    5,\n    416,\n    6,\n    417,\n    7,\n    418,\n    8,\n    419,\n    9,\n    420,\n    10,\n    421,\n    11,\n    422,\n    12,\n    423,\n    13,\n    424,\n    14,\n    425,\n    15,\n    426,\n    16,\n    427,\n    17,\n    428,\n    18,\n    429,\n    19,\n    430,\n    20,\n    431,\n    21,\n    432,\n    22,\n    433,\n    23,\n    434,\n    24,\n    435,\n    25,\n    436,\n    26,\n    437,\n    27,\n    438,\n    28,\n    439,\n    29,\n    440,\n    30,\n    441,\n    31,\n    442,\n    32,\n    443,\n    33,\n    444,\n    34,\n    445,\n    35,\n    446,\n    36,\n    447,\n    37,\n    448,\n    38,\n    449,\n    39,\n    450,\n    40,\n    451,\n    41,\n    452,\n    42,\n    453,\n    43,\n    454,\n    44,\n    455,\n    45,\n    456,\n    46,\n    457,\n    47,\n    458,\n    48,\n    459,\n    49,\n    460,\n    50,\n    461,\n    51,\n    462,\n    52,\n    463,\n    53,\n    464,\n    54,\n    465,\n    55,\n    466,\n    56,\n    467,\n    57,\n    468,\n    58,\n    469,\n    59,\n    470,\n    60,\n    471,\n    61,\n    472,\n    62,\n    473,\n    63,\n    474,\n    64,\n    475,\n    65,\n    476,\n    66,\n    477,\n    67,\n    478,\n    68,\n    479,\n    69,\n    480,\n    70,\n    481,\n    71,\n    482,\n    72,\n    483,\n    73,\n    484,\n    74,\n    485,\n    75,\n    486,\n    76,\n    487,\n    77,\n    488,\n    78,\n    489,\n    79,\n    490,\n    0,\n    491,\n    1,\n    492,\n    2,\n    493,\n    3,\n    494,\n    4,\n    495,\n    5,\n    496,\n    6,\n    497,\n    7,\n    498,\n    8,\n    499,\n    9,\n    500,\n    10,\n    501,\n    11,\n    502,\n    12,\n    80,\n    13,\n    81,\n    14,\n    82,\n    15,\n    83,\n    16,\n    84,\n    17,\n    85,\n    18,\n    86,\n    19,\n    87,\n    20,\n    88,\n    21,\n    89,\n    22,\n    90,\n    23,\n    91,\n    24,\n    92,\n    25,\n    93,\n    26,\n    94,\n    27,\n    95,\n    28,\n    96,\n    29,\n    97,\n    30,\n    98,\n    31,\n    99,\n    32,\n    100,\n    33,\n    101,\n    34,\n    102,\n    35,\n    103,\n    36,\n    104,\n    37,\n    105,\n    38,\n    106,\n    39,\n    107,\n    40,\n    108,\n    41,\n    109,\n    42,\n    110,\n    43,\n    111,\n    44,\n    112,\n    45,\n    113,\n    46,\n    114,\n    47,\n    115,\n    48,\n    116,\n    49,\n    117,\n    50,\n    118,\n    51,\n    119,\n    52,\n    120,\n    53,\n    121,\n    54,\n    122,\n    55,\n    123,\n    56,\n    124,\n    57,\n    125,\n    58,\n    126,\n    59,\n    127,\n    60,\n    128,\n    61,\n    129,\n    62,\n    130,\n    63,\n    131,\n    64,\n    132,\n    65,\n    133,\n    66,\n    134,\n    67,\n    135,\n    68,\n    136,\n    69,\n    137,\n    70,\n    138,\n    71,\n    139,\n    72,\n    140,\n    73,\n    141,\n    74,\n    142,\n    75,\n    143,\n    76,\n    144,\n    77,\n    145,\n    78,\n    146,\n    79,\n    147,\n    0,\n    148,\n    1,\n    149,\n    2,\n    150,\n    3,\n    151,\n    4,\n    152,\n    5,\n    153,\n    6,\n    154,\n    7,\n    155,\n    8,\n    156,\n    9,\n    157,\n    10,\n    158,\n    11,\n    159,\n    12,\n    160,\n    13,\n    161,\n    14,\n    162,\n    15,\n    163,\n    16,\n    164,\n    17,\n    165,\n    18,\n    166,\n    19,\n    167,\n    20,\n    168,\n    21,\n    169,\n    22,\n    170,\n    23,\n    171,\n    24,\n    172,\n    25,\n    173,\n    26,\n    174,\n    27,\n    175,\n    28,\n    176,\n    29,\n    177,\n    30,\n    178,\n    31,\n    179,\n    32,\n    180,\n    33,\n    181,\n    34,\n    182,\n    35,\n    183,\n    36,\n    184,\n    37,\n    185,\n    38,\n    186,\n    39,\n    187,\n    40,\n    188,\n    41,\n    189,\n    42,\n    190,\n    43,\n    191,\n    44,\n    192,\n    45,\n    193,\n    46,\n    194,\n    47,\n    195,\n    48,\n    196,\n    49,\n    197,\n    50,\n    198,\n    51,\n    199,\n    52,\n    200,\n    53,\n    201,\n    54,\n    202,\n    55,\n    203,\n    56,\n    204,\n    57,\n    205,\n    58,\n    206,\n    59,\n    207,\n    60,\n    208,\n    61,\n    209,\n    62,\n    210,\n    63,\n    211,\n    64,\n    212,\n    65,\n    213,\n    66,\n    214,\n    67,\n    215,\n    68,\n    216,\n    69,\n    217,\n    70,\n    218,\n    71,\n    219,\n    72,\n    220,\n    73,\n    221,\n    74,\n    222,\n    75,\n    223,\n    76,\n    224,\n    77,\n    225,\n    78,\n    226,\n    79,\n    227,\n    0,\n    228,\n    1,\n    229,\n    2,\n    230,\n    3,\n    231,\n    4,\n    232,\n    5,\n    233,\n    6,\n    234,\n    7,\n    235,\n    8,\n    236,\n    9,\n    237,\n    10,\n    238,\n    11,\n    239,\n    12,\n    240,\n    13,\n    241,\n    14,\n    242,\n    15,\n    243,\n    16,\n    244,\n    17,\n    245,\n    18,\n    246,\n    19,\n    247,\n    20,\n    248,\n    21,\n    249,\n    22,\n    250,\n    23,\n    251,\n    24,\n    252,\n    25,\n    253,\n    26,\n    254,\n    27,\n    255,\n    28,\n    256,\n    29,\n    257,\n    30,\n    258,\n    31,\n    259,\n    32,\n    260,\n    33,\n    261,\n    34,\n    262,\n    35,\n    263,\n    36,\n    264,\n    37,\n    265,\n    38,\n    266,\n    39,\n    267,\n    40,\n    268,\n    41,\n    269,\n    42,\n    270,\n    43,\n    271,\n    44,\n    272,\n    45,\n    273,\n    46,\n    274,\n    47,\n    275,\n    48,\n    276,\n    49,\n    277,\n    50,\n    278,\n    51,\n    279,\n    52,\n    280,\n    53,\n    281,\n    54,\n    282,\n    55,\n    283,\n    56,\n    284,\n    57,\n    285,\n    58,\n    286,\n    59,\n    287,\n    60,\n    288,\n    61,\n    289,\n    62,\n    290,\n    63,\n    291,\n    64,\n    292,\n    65,\n    293,\n    66,\n    294,\n    67,\n    295,\n    68,\n    296,\n    69,\n    297,\n    70,\n    298,\n    71,\n    299,\n    72,\n    300,\n    73,\n    301,\n    74,\n    302,\n    75,\n    303,\n    76,\n    304,\n    77,\n    305,\n    78,\n    306,\n    79,\n    307,\n    0,\n    308,\n    1,\n    309,\n    2,\n    310,\n    3,\n    311,\n    4,\n    312,\n    5,\n    313,\n    6,\n    314,\n    7,\n    315,\n    8,\n    316,\n    9,\n    317,\n    10,\n    318,\n    11,\n    319,\n    12,\n    320,\n    13,\n    321,\n    14,\n    322,\n    15,\n    323,\n    16,\n    324,\n    17,\n    325,\n    18,\n    326,\n    19,\n    327,\n    20,\n    328,\n    21,\n    329,\n    22,\n    330,\n    23,\n    331,\n    24,\n    332,\n    25,\n    333,\n    26,\n    334,\n    27,\n    335,\n    28,\n    336,\n    29,\n    337,\n    30,\n    338,\n    31,\n    339,\n    32,\n    340,\n    33,\n    341,\n    34,\n    342,\n    35,\n    343,\n    36,\n    344,\n    37,\n    345,\n    38,\n    346,\n    39,\n    347,\n    40,\n    348,\n    41,\n    349,\n    42,\n    350,\n    43,\n    351,\n    44,\n    352,\n    45,\n    353,\n    46,\n    354,\n    47,\n    355,\n    48,\n    356,\n    49,\n    357,\n    50,\n    358,\n    51,\n    359,\n    52,\n    360,\n    53,\n    361,\n    54,\n    362,\n    55,\n    363,\n    56,\n    364,\n    57,\n    365,\n    58,\n    366,\n    59,\n    367,\n    60,\n    368,\n    61,\n    369,\n    62,\n    370,\n    63,\n    371,\n    64,\n    372,\n    65,\n    373,\n    66,\n    374,\n    67,\n    375,\n    68,\n    376,\n    69,\n    377,\n    70,\n    378,\n    71,\n    379,\n    72,\n    380,\n    73,\n    381,\n    74,\n    382,\n    75,\n    383,\n    76,\n    384,\n    77,\n    385,\n    78,\n    386,\n    79,\n    387,\n    0,\n    388,\n    1,\n    389,\n    2,\n    390,\n    3,\n    391,\n    4,\n    392,\n    5,\n    393,\n    6,\n    394,\n    7,\n    395,\n    8,\n    396,\n    9,\n    397,\n    10,\n    398,\n    11,\n    399,\n    12,\n    400,\n    13,\n    401,\n    14,\n    402,\n    15,\n    403,\n    16,\n    404,\n    17,\n    405,\n    18,\n    406,\n    19,\n    407,\n    20,\n    408,\n    21,\n    409,\n    22,\n    410,\n    23,\n    411,\n    24,\n    412,\n    25,\n    413,\n    26,\n    414,\n    27,\n    415,\n    28,\n    416,\n    29,\n    417,\n    30,\n    418,\n    31,\n    419,\n    32,\n    420,\n    33,\n    421,\n    34,\n    422,\n    35,\n    423,\n    36,\n    424,\n    37,\n    425,\n    38,\n    426,\n    39,\n    427,\n    40,\n    428,\n    41,\n    429,\n    42,\n    430,\n    43,\n    431,\n    44,\n    432,\n    45,\n    433,\n    46,\n    434,\n    47,\n    435,\n    48,\n    436,\n    49,\n    437,\n    50,\n    438,\n    51,\n    439,\n    52,\n    440,\n    53,\n    441,\n    54,\n    442,\n    55,\n    443,\n    56,\n    444,\n    57,\n    445,\n    58,\n    446,\n    59,\n    447,\n    60,\n    448,\n    61,\n    449,\n    62,\n    450,\n    63,\n    451,\n    64,\n    452,\n    65,\n    453,\n    66,\n    454,\n    67,\n    455,\n    68,\n    456,\n    69,\n    457,\n    70,\n    458,\n    71,\n    459,\n    72,\n    460,\n    73,\n    461,\n    74,\n    462,\n    75,\n    463,\n    76,\n    464,\n    77,\n    465,\n    78,\n    466,\n    79,\n    467,\n    0,\n    468,\n    1,\n    469,\n    2,\n    470,\n    3,\n    471,\n    4,\n    472,\n    5,\n    473,\n    6,\n    474,\n    7,\n    475,\n    8,\n    476,\n    9,\n    477,\n    10,\n    478,\n    11,\n    479,\n    12,\n    480,\n    13,\n    481,\n    14,\n    482,\n    15,\n    483,\n    16,\n    484,\n    17,\n    485,\n    18,\n    486,\n    19,\n    487,\n    20,\n    488,\n    21,\n    489,\n    22,\n    490,\n    23,\n    491,\n    24,\n    492,\n    25,\n    493,\n    26,\n    494,\n    27,\n    495,\n    28,\n    496,\n    29,\n    497,\n    30,\n    498,\n    31,\n    499,\n    32,\n    500,\n    33,\n    501,\n    34,\n    502,\n    35,\n    80,\n    36,\n    81,\n    37,\n    82,\n    38,\n    83,\n    39,\n    84,\n    40,\n    85,\n    41,\n    86,\n    42,\n    87,\n    43,\n    88,\n    44,\n    89,\n    45,\n    90,\n    46,\n    91,\n    47,\n    92,\n    48,\n    93,\n    49,\n    94,\n    50,\n    95,\n    51,\n    96,\n    52,\n    97,\n    53,\n    98,\n    54,\n    99,\n    55,\n    100,\n    56,\n    101,\n    57,\n    102,\n    58,\n    103,\n    59,\n    104,\n    60,\n    105,\n    61,\n    106,\n    62,\n    107,\n    63,\n    108,\n    64,\n    109,\n    65,\n    110,\n    66,\n    111,\n    67,\n    112,\n    68,\n    113,\n    69,\n    114,\n    70,\n    115,\n    71,\n    116,\n    72,\n    117,\n    73,\n    118,\n    74,\n    119,\n    75,\n    120,\n    76,\n    121,\n    77,\n    122,\n    78,\n    123,\n    79,\n    124,\n    0,\n    125,\n    1,\n    126,\n    2,\n    127,\n    3,\n    128,\n    4,\n    129,\n    5,\n    130,\n    6,\n    131,\n    7,\n    132,\n    8,\n    133,\n    9,\n    134,\n    10,\n    135,\n    11,\n    136,\n    12,\n    137,\n    13,\n    138,\n    14,\n    139,\n    15,\n    140,\n    16,\n    141,\n    17,\n    142,\n    18,\n    143,\n    19,\n    144,\n    20,\n    145,\n    21,\n    146,\n    22,\n    147,\n    23,\n    148,\n    24,\n    149,\n    25,\n    150,\n    26,\n    151,\n    27,\n    152,\n    28,\n    153,\n    29,\n    154,\n    30,\n    155,\n    31,\n    156,\n    32,\n    157,\n    33,\n    158,\n    34,\n    159,\n    35,\n    160,\n    36,\n    161,\n    37,\n    162,\n    38,\n    163,\n    39,\n    164,\n    40,\n    165,\n    41,\n    166,\n    42,\n    167,\n    43,\n    168,\n    44,\n    169,\n    45,\n    170,\n    46,\n    171,\n    47,\n    172,\n    48,\n    173,\n    49,\n    174,\n    50,\n    175,\n    51,\n    176,\n    52,\n    177,\n    53,\n    178,\n    54,\n    179,\n    55,\n    180,\n    56,\n    181,\n    57,\n    182,\n    58,\n    183,\n    59,\n    184,\n    60,\n    185,\n    61,\n    186,\n    62,\n    187,\n    63,\n    188,\n    64,\n    189,\n    65,\n    190,\n    66,\n    191,\n    67,\n    192,\n    68,\n    193,\n    69,\n    194,\n    70,\n    195,\n    71,\n    196,\n    72,\n    197,\n    73,\n    198,\n    74,\n    199,\n    75,\n    200,\n    76,\n    201,\n    77,\n    202,\n    78,\n    203,\n    79,\n    204,\n    0,\n    205,\n    1,\n    206,\n    2,\n    207,\n    3,\n    208,\n    4,\n    209,\n    5,\n    210,\n    6,\n    211,\n    7,\n    212,\n    8,\n    213,\n    9,\n    214,\n    10,\n    215,\n    11,\n    216,\n    12,\n    217,\n    13,\n    218,\n    14,\n    219,\n    15,\n    220,\n    16,\n    221,\n    17,\n    222,\n    18,\n    223,\n    19,\n    224,\n    20,\n    225,\n    21,\n    226,\n    22,\n    227,\n    23,\n    228,\n    24,\n    229,\n    25,\n    230,\n    26,\n    231,\n    27,\n    232,\n    28,\n    233,\n    29,\n    234,\n    30,\n    235,\n    31,\n    236,\n    32,\n    237,\n    33,\n    238,\n    34,\n    239,\n    35,\n    240,\n    36,\n    241,\n    37,\n    242,\n    38,\n    243,\n    39,\n    244,\n    40,\n    245,\n    41,\n    246,\n    42,\n    247,\n    43,\n    248,\n    44,\n    249,\n    45,\n    250,\n    46,\n    251,\n    47,\n    252,\n    48,\n    253,\n    49,\n    254,\n    50,\n    255,\n    51,\n    256,\n    52,\n    257,\n    53,\n    258,\n    54,\n    259,\n    55,\n    260,\n    56,\n    261,\n    57,\n    262,\n    58,\n    263,\n    59,\n    264,\n    60,\n    265,\n    61,\n    266,\n    62,\n    267,\n    63,\n    268,\n    64,\n    269,\n    65,\n    270,\n    66,\n    271,\n    67,\n    272,\n    68,\n    273,\n    69,\n    274,\n    70,\n    275,\n    71,\n    276,\n    72,\n    277,\n    73,\n    278,\n    74,\n    279,\n    75,\n    280,\n    76,\n    281,\n    77,\n    282,\n    78,\n    283,\n    79,\n    284,\n    0,\n    285,\n    1,\n    286,\n    2,\n    287,\n    3,\n    288,\n    4,\n    289,\n    5,\n    290,\n    6,\n    291,\n    7,\n    292,\n    8,\n    293,\n    9,\n    294,\n    10,\n    295,\n    11,\n    296,\n    12,\n    297,\n    13,\n    298,\n    14,\n    299,\n    15,\n    300,\n    16,\n    301,\n    17,\n    302,\n    18,\n    303,\n    19,\n    304,\n    20,\n    305,\n    21,\n    306,\n    22,\n    307,\n    23,\n    308,\n    24,\n    309,\n    25,\n    310,\n    26,\n    311,\n    27,\n    312,\n    28,\n    313,\n    29,\n    314,\n    30,\n    315,\n    31,\n    316,\n    32,\n    317,\n    33,\n    318,\n    34,\n    319,\n    35,\n    320,\n    36,\n    321,\n    37,\n    322,\n    38,\n    323,\n    39,\n    324,\n    40,\n    325,\n    41,\n    326,\n    42,\n    327,\n    43,\n    328,\n    44,\n    329,\n    45,\n    330,\n    46,\n    331,\n    47,\n    332,\n    48,\n    333,\n    49,\n    334,\n    50,\n    335,\n    51,\n    336,\n    52,\n    337,\n    53,\n    338,\n    54,\n    339,\n    55,\n    340,\n    56,\n    341,\n    57,\n    342,\n    58,\n    343,\n    59,\n    344,\n    60,\n    345,\n    61,\n    346,\n    62,\n    347,\n    63,\n    348,\n    64,\n    349,\n    65,\n    350,\n    66,\n    351,\n    67,\n    352,\n    68,\n    353,\n    69,\n    354,\n    70,\n    355,\n    71,\n    356,\n    72,\n    357,\n    73,\n    358,\n    74,\n    359,\n    75,\n    360,\n    76,\n    361,\n    77,\n    362,\n    78,\n    363,\n    79,\n    364,\n    0,\n    365,\n    1,\n    366,\n    2,\n    367,\n    3,\n    368,\n    4,\n    369,\n    5,\n    370,\n    6,\n    371,\n    7,\n    372,\n    8,\n    373,\n    9,\n    374,\n    10,\n    375,\n    11,\n    376,\n    12,\n    377,\n    13,\n    378,\n    14,\n    379,\n    15,\n    380,\n    16,\n    381,\n    17,\n    382,\n    18,\n    383,\n    19,\n    384,\n    20,\n    385,\n    21,\n    386,\n    22,\n    387,\n    23,\n    388,\n    24,\n    389,\n    25,\n    390,\n    26,\n    391,\n    27,\n    392,\n    28,\n    393,\n    29,\n    394,\n    30,\n    395,\n    31,\n    396,\n    32,\n    397,\n    33,\n    398,\n    34,\n    399,\n    35,\n    400,\n    36,\n    401,\n    37,\n    402,\n    38,\n    403,\n    39,\n    404,\n    40,\n    405,\n    41,\n    406,\n    42,\n    407,\n    43,\n    408,\n    44,\n    409,\n    45,\n    410,\n    46,\n    411,\n    47,\n    412,\n    48,\n    413,\n    49,\n    414,\n    50,\n    415,\n    51,\n    416,\n    52,\n    417,\n    53,\n    418,\n    54,\n    419,\n    55,\n    420,\n    56,\n    421,\n    57,\n    422,\n    58,\n    423,\n    59,\n    424,\n    60,\n    425,\n    61,\n    426,\n    62,\n    427,\n    63,\n    428,\n    64,\n    429,\n    65,\n    430,\n    66,\n    431,\n    67,\n    432,\n    68,\n    433,\n    69,\n    434,\n    70,\n    435,\n    71,\n    436,\n    72,\n    437,\n    73,\n    438,\n    74,\n    439,\n    75,\n    440,\n    76,\n    441,\n    77,\n    442,\n    78,\n    443,\n    79,\n    444,\n    0,\n    445,\n    1,\n    446,\n    2,\n    447,\n    3,\n    448,\n    4,\n    449,\n    5,\n    450,\n    6,\n    451,\n    7,\n    452,\n    8,\n    453,\n    9,\n    454,\n    10,\n    455,\n    11,\n    456,\n    12,\n    457,\n    13,\n    458,\n    14,\n    459,\n    15,\n    460,\n    16,\n    461,\n    17,\n    462,\n    18,\n    463,\n    19,\n    464,\n    20,\n    465,\n    21,\n    466,\n    22,\n    467,\n    23,\n    468,\n    24,\n    469,\n    25,\n    470,\n    26,\n    471,\n    27,\n    472,\n    28,\n    473,\n    29,\n    474,\n    30,\n    475,\n    31,\n    476,\n    32,\n    477,\n    33,\n    478,\n    34,\n    479,\n    35,\n    480,\n    36,\n    481,\n    37,\n    482,\n    38,\n    483,\n    39,\n    484,\n    40,\n    485,\n    41,\n    486,\n    42,\n    487,\n    43,\n    488,\n    44,\n    489,\n    45,\n    490,\n    46,\n    491,\n    47,\n    492,\n    48,\n    493,\n    49,\n    494,\n    50,\n    495,\n    51,\n    496,\n    52,\n    497,\n    53,\n    498,\n    54,\n    499,\n    55,\n    500,\n    56,\n    501,\n    57,\n    502,\n    58,\n    80,\n    59,\n    81,\n    60,\n    82,\n    61,\n    83,\n    62,\n    84,\n    63,\n    85,\n    64,\n    86,\n    65,\n    87,\n    66,\n    88,\n    67,\n    89,\n    68,\n    90,\n    69,\n    91,\n    70,\n    92,\n    71,\n    93,\n    72,\n    94,\n    73,\n    95,\n    74,\n    96,\n    75,\n    97,\n    76,\n    98,\n    77,\n    99,\n    78,\n    100,\n    79,\n    101,\n    0,\n    102,\n    1,\n    103,\n    2,\n    104,\n    3,\n    105,\n    4,\n    106,\n    5,\n    107,\n    6,\n    108,\n    7,\n    109,\n    8,\n    110,\n    9,\n    111,\n    10,\n    112,\n    11,\n    113,\n    12,\n    114,\n    13,\n    115,\n    14,\n    116,\n    15,\n    117,\n    16,\n    118,\n    17,\n    119,\n    18,\n    120,\n    19,\n    121,\n    20,\n    122,\n    21,\n    123,\n    22,\n    124,\n    23,\n    125,\n    24,\n    126,\n    25,\n    127,\n    26,\n    128,\n    27,\n    129,\n    28,\n    130,\n    29,\n    131,\n    30,\n    132,\n    31,\n    133,\n    32,\n    134,\n    33,\n    135,\n    34,\n    136,\n    35,\n    137,\n    36,\n    138,\n    37,\n    139,\n    38,\n    140,\n    39,\n    141,\n    40,\n    142,\n    41,\n    143,\n    42,\n    144,\n    43,\n    145,\n    44,\n    146,\n    45,\n    147,\n    46,\n    148,\n    47,\n    149,\n    48,\n    150,\n    49,\n    151,\n    50,\n    152,\n    51,\n    153,\n    52,\n    154,\n    53,\n    155,\n    54,\n    156,\n    55,\n    157,\n    56,\n    158,\n    57,\n    159,\n    58,\n    160,\n    59,\n    161,\n    60,\n    162,\n    61,\n    163,\n    62,\n    164,\n    63,\n    165,\n    64,\n    166,\n    65,\n    167,\n    66,\n    168,\n    67,\n    169,\n    68,\n    170,\n    69,\n    171,\n    70,\n    172,\n    71,\n    173,\n    72,\n    174,\n    73,\n    175,\n    74,\n    176,\n    75,\n    177,\n    76,\n    178,\n    77,\n    179,\n    78,\n    180,\n    79,\n    181,\n    0,\n    182,\n    1,\n    183,\n    2,\n    184,\n    3,\n    185,\n    4,\n    186,\n    5,\n    187,\n    6,\n    188,\n    7,\n    189,\n    8,\n    190,\n    9,\n    191,\n    10,\n    192,\n    11,\n    193,\n    12,\n    194,\n    13,\n    195,\n    14,\n    196,\n    15,\n    197,\n    16,\n    198,\n    17,\n    199,\n    18,\n    200,\n    19,\n    201,\n    20,\n    202,\n    21,\n    203,\n    22,\n    204,\n    23,\n    205,\n    24,\n    206,\n    25,\n    207,\n    26,\n    208,\n    27,\n    209,\n    28,\n    210,\n    29,\n    211,\n    30,\n    212,\n    31,\n    213,\n    32,\n    214,\n    33,\n    215,\n    34,\n    216,\n    35,\n    217,\n    36,\n    218,\n    37,\n    219,\n    38,\n    220,\n    39,\n    221,\n    40,\n    222,\n    41,\n    223,\n    42,\n    224,\n    43,\n    225,\n    44,\n    226,\n    45,\n    227,\n    46,\n    228,\n    47,\n    229,\n    48,\n    230,\n    49,\n    231,\n    50,\n    232,\n    51,\n    233,\n    52,\n    234,\n    53,\n    235,\n    54,\n    236,\n    55,\n    237,\n    56,\n    238,\n    57,\n    239,\n    58,\n    240,\n    59,\n    241,\n    60,\n    242,\n    61,\n    243,\n    62,\n    244,\n    63,\n    245,\n    64,\n    246,\n    65,\n    247,\n    66,\n    248,\n    67,\n    249,\n    68,\n    250,\n    69,\n    251,\n    70,\n    252,\n    71,\n    253,\n    72,\n    254,\n    73,\n    255,\n    74,\n    256,\n    75,\n    257,\n    76,\n    258,\n    77,\n    259,\n    78,\n    260,\n    79,\n    261,\n    0,\n    262,\n    1,\n    263,\n    2,\n    264,\n    3,\n    265,\n    4,\n    266,\n    5,\n    267,\n    6,\n    268,\n    7,\n    269,\n    8,\n    270,\n    9,\n    271,\n    10,\n    272,\n    11,\n    273,\n    12,\n    274,\n    13,\n    275,\n    14,\n    276,\n    15,\n    277,\n    16,\n    278,\n    17,\n    279,\n    18,\n    280,\n    19,\n    281,\n    20,\n    282,\n    21,\n    283,\n    22,\n    284,\n    23,\n    285,\n    24,\n    286,\n    25,\n    287,\n    26,\n    288,\n    27,\n    289,\n    28,\n    290,\n    29,\n    291,\n    30,\n    292,\n    31,\n    293,\n    32,\n    294,\n    33,\n    295,\n    34,\n    296,\n    35,\n    297,\n    36,\n    298,\n    37,\n    299,\n    38,\n    300,\n    39,\n    301,\n    40,\n    302,\n    41,\n    303,\n    42,\n    304,\n    43,\n    305,\n    44,\n    306,\n    45,\n    307,\n    46,\n    308,\n    47,\n    309,\n    48,\n    310,\n    49,\n    311,\n    50,\n    312,\n    51,\n    313,\n    52,\n    314,\n    53,\n    315,\n    54,\n    316,\n    55,\n    317,\n    56,\n    318,\n    57,\n    319,\n    58,\n    320,\n    59,\n    321,\n    60,\n    322,\n    61,\n    323,\n    62,\n    324,\n    63,\n    325,\n    64,\n    326,\n    65,\n    327,\n    66,\n    328,\n    67,\n    329,\n    68,\n    330,\n    69,\n    331,\n    70,\n    332,\n    71,\n    333,\n    72,\n    334,\n    73,\n    335,\n    74,\n    336,\n    75,\n    337,\n    76,\n    338,\n    77,\n    339,\n    78,\n    340,\n    79,\n    341,\n    0,\n    342,\n    1,\n    343,\n    2,\n    344,\n    3,\n    345,\n    4,\n    346,\n    5,\n    347,\n    6,\n    348,\n    7,\n    349,\n    8,\n    350,\n    9,\n    351,\n    10,\n    352,\n    11,\n    353,\n    12,\n    354,\n    13,\n    355,\n    14,\n    356,\n    15,\n    357,\n    16,\n    358,\n    17,\n    359,\n    18,\n    360,\n    19,\n    361,\n    20,\n    362,\n    21,\n    363,\n    22,\n    364,\n    23,\n    365,\n    24,\n    366,\n    25,\n    367,\n    26,\n    368,\n    27,\n    369,\n    28,\n    370,\n    29,\n    371,\n    30,\n    372,\n    31,\n    373,\n    32,\n    374,\n    33,\n    375,\n    34,\n    376,\n    35,\n    377,\n    36,\n    378,\n    37,\n    379,\n    38,\n    380,\n    39,\n    381,\n    40,\n    382,\n    41,\n    383,\n    42,\n    384,\n    43,\n    385,\n    44,\n    386,\n    45,\n    387,\n    46,\n    388,\n    47,\n    389,\n    48,\n    390,\n    49,\n    391,\n    50,\n    392,\n    51,\n    393,\n    52,\n    394,\n    53,\n    395,\n    54,\n    396,\n    55,\n    397,\n    56,\n    398,\n    57,\n    399,\n    58,\n    400,\n    59,\n    401,\n    60,\n    402,\n    61,\n    403,\n    62,\n    404,\n    63,\n    405,\n    64,\n    406,\n    65,\n    407,\n    66,\n    408,\n    67,\n    409,\n    68,\n    410,\n    69,\n    411,\n    70,\n    412,\n    71,\n    413,\n    72,\n    414,\n    73,\n    415,\n    74,\n    416,\n    75,\n    417,\n    76,\n    418,\n    77,\n    419,\n    78,\n    420,\n    79,\n    421,\n    0,\n    422,\n    1,\n    423,\n    2,\n    424,\n    3,\n    425,\n    4,\n    426,\n    5,\n    427,\n    6,\n    428,\n    7,\n    429,\n    8,\n    430,\n    9,\n    431,\n    10,\n    432,\n    11,\n    433,\n    12,\n    434,\n    13,\n    435,\n    14,\n    436,\n    15,\n    437,\n    16,\n    438,\n    17,\n    439,\n    18,\n    440,\n    19,\n    441,\n    20,\n    442,\n    21,\n    443,\n    22,\n    444,\n    23,\n    445,\n    24,\n    446,\n    25,\n    447,\n    26,\n    448,\n    27,\n    449,\n    28,\n    450,\n    29,\n    451,\n    30,\n    452,\n    31,\n    453,\n    32,\n    454,\n    33,\n    455,\n    34,\n    456,\n    35,\n    457,\n    36,\n    458,\n    37,\n    459,\n    38,\n    460,\n    39,\n    461,\n    40,\n    462,\n    41,\n    463,\n    42,\n    464,\n    43,\n    465,\n    44,\n    466,\n    45,\n    467,\n    46,\n    468,\n    47,\n    469,\n    48,\n    470,\n    49,\n    471,\n    50,\n    472,\n    51,\n    473,\n    52,\n    474,\n    53,\n    475,\n    54,\n    476,\n    55,\n    477,\n    56,\n    478,\n    57,\n    479,\n    58,\n    480,\n    59,\n    481,\n    60,\n    482,\n    61,\n    483,\n    62,\n    484,\n    63,\n    485,\n    64,\n    486,\n    65,\n    487,\n    66,\n    488,\n    67,\n    489,\n    68,\n    490,\n    69,\n    491,\n    70,\n    492,\n    71,\n    493,\n    72,\n    494,\n    73,\n    495,\n    74,\n    496,\n    75,\n    497,\n    76,\n    498,\n    77,\n    499,\n    78,\n    500,\n    79,\n    501,\n    0,\n    502,\n    1,\n    80,\n    2,\n    81,\n    3,\n    82,\n    4,\n    83,\n    5,\n    84,\n    6,\n    85,\n    7,\n    86,\n    8,\n    87,\n    9,\n    88,\n    10,\n    89,\n    11,\n    90,\n    12,\n    91,\n    13,\n    92,\n    14,\n    93,\n    15,\n    94,\n    16,\n    95,\n    17,\n    96,\n    18,\n    97,\n    19,\n    98,\n    20,\n    99,\n    21,\n    100,\n    22,\n    101,\n    23,\n    102,\n    24,\n    103,\n    25,\n    104,\n    26,\n    105,\n    27,\n    106,\n    28,\n    107,\n    29,\n    108,\n    30,\n    109,\n    31,\n    110,\n    32,\n    111,\n    33,\n    112,\n    34,\n    113,\n    35,\n    114,\n    36,\n    115,\n    37,\n    116,\n    38,\n    117,\n    39,\n    118,\n    40,\n    119,\n    41,\n    120,\n    42,\n    121,\n    43,\n    122,\n    44,\n    123,\n    45,\n    124,\n    46,\n    125,\n    47,\n    126,\n    48,\n    127,\n    49,\n    128,\n    50,\n    129,\n    51,\n    130,\n    52,\n    131,\n    53,\n    132,\n    54,\n    133,\n    55,\n    134,\n    56,\n    135,\n    57,\n    136,\n    58,\n    137,\n    59,\n    138,\n    60,\n    139,\n    61,\n    140,\n    62,\n    141,\n    63,\n    142,\n    64,\n    143,\n    65,\n    144,\n    66,\n    145,\n    67,\n    146,\n    68,\n    147,\n    69,\n    148,\n    70,\n    149,\n    71,\n    150,\n    72,\n    151,\n    73,\n    152,\n    74,\n    153,\n    75,\n    154,\n    76,\n    155,\n    77,\n    156,\n    78,\n    157,\n    79,\n    158,\n    0,\n    159,\n    1,\n    160,\n    2,\n    161,\n    3,\n    162,\n    4,\n    163,\n    5,\n    164,\n    6,\n    165,\n    7,\n    166,\n    8,\n    167,\n    9,\n    168,\n    10,\n    169,\n    11,\n    170,\n    12,\n    171,\n    13,\n    172,\n    14,\n    173,\n    15,\n    174,\n    16,\n    175,\n    17,\n    176,\n    18,\n    177,\n    19,\n    178,\n    20,\n    179,\n    21,\n    180,\n    22,\n    181,\n    23,\n    182,\n    24,\n    183,\n    25,\n    184,\n    26,\n    185,\n    27,\n    186,\n    28,\n    187,\n    29,\n    188,\n    30,\n    189,\n    31,\n    190,\n    32,\n    191,\n    33,\n    192,\n    34,\n    193,\n    35,\n    194,\n    36,\n    195,\n    37,\n    196,\n    38,\n    197,\n    39,\n    198,\n    40,\n    199,\n    41,\n    200,\n    42,\n    201,\n    43,\n    202,\n    44,\n    203,\n    45,\n    204,\n    46,\n    205,\n    47,\n    206,\n    48,\n    207,\n    49,\n    208,\n    50,\n    209,\n    51,\n    210,\n    52,\n    211,\n    53,\n    212,\n    54,\n    213,\n    55,\n    214,\n    56,\n    215,\n    57,\n    216,\n    58,\n    217,\n    59,\n    218,\n    60,\n    219,\n    61,\n    220,\n    62,\n    221,\n    63,\n    222,\n    64,\n    223,\n    65,\n    224,\n    66,\n    225,\n    67,\n    226,\n    68,\n    227,\n    69,\n    228,\n    70,\n    229,\n    71,\n    230,\n    72,\n    231,\n    73,\n    232,\n    74,\n    233,\n    75,\n    234,\n    76,\n    235,\n    77,\n    236,\n    78,\n    237,\n    79,\n    238,\n    0,\n    239,\n    1,\n    240,\n    2,\n    241,\n    3,\n    242,\n    4,\n    243,\n    5,\n    244,\n    6,\n    245,\n    7,\n    246,\n    8,\n    247,\n    9,\n    248,\n    10,\n    249,\n    11,\n    250,\n    12,\n    251,\n    13,\n    252,\n    14,\n    253,\n    15,\n    254,\n    16,\n    255,\n    17,\n    256,\n    18,\n    257,\n    19,\n    258,\n    20,\n    259,\n    21,\n    260,\n    22,\n    261,\n    23,\n    262,\n    24,\n    263,\n    25,\n    264,\n    26,\n    265,\n    27,\n    266,\n    28,\n    267,\n    29,\n    268,\n    30,\n    269,\n    31,\n    270,\n    32,\n    271,\n    33,\n    272,\n    34,\n    273,\n    35,\n    274,\n    36,\n    275,\n    37,\n    276,\n    38,\n    277,\n    39,\n    278,\n    40,\n    279,\n    41,\n    280,\n    42,\n    281,\n    43,\n    282,\n    44,\n    283,\n    45,\n    284,\n    46,\n    285,\n    47,\n    286,\n    48,\n    287,\n    49,\n    288,\n    50,\n    289,\n    51,\n    290,\n    52,\n    291,\n    53,\n    292,\n    54,\n    293,\n    55,\n    294,\n    56,\n    295,\n    57,\n    296,\n    58,\n    297,\n    59,\n    298,\n    60,\n    299,\n    61,\n    300,\n    62,\n    301,\n    63,\n    302,\n    64,\n    303,\n    65,\n    304,\n    66,\n    305,\n    67,\n    306,\n    68,\n    307,\n    69,\n    308,\n    70,\n    309,\n    71,\n    310,\n    72,\n    311,\n    73,\n    312,\n    74,\n    313,\n    75,\n    314,\n    76,\n    315,\n    77,\n    316,\n    78,\n    317,\n    79,\n    318,\n    0,\n    319,\n    1,\n    320,\n    2,\n    321,\n    3,\n    322,\n    4,\n    323,\n    5,\n    324,\n    6,\n    325,\n    7,\n    326,\n    8,\n    327,\n    9,\n    328,\n    10,\n    329,\n    11,\n    330,\n    12,\n    331,\n    13,\n    332,\n    14,\n    333,\n    15,\n    334,\n    16,\n    335,\n    17,\n    336,\n    18,\n    337,\n    19,\n    338,\n    20,\n    339,\n    21,\n    340,\n    22,\n    341,\n    23,\n    342,\n    24,\n    343,\n    25,\n    344,\n    26,\n    345,\n    27,\n    346,\n    28,\n    347,\n    29,\n    348,\n    30,\n    349,\n    31,\n    350,\n    32,\n    351,\n    33,\n    352,\n    34,\n    353,\n    35,\n    354,\n    36,\n    355,\n    37,\n    356,\n    38,\n    357,\n    39,\n    358,\n    40,\n    359,\n    41,\n    360,\n    42,\n    361,\n    43,\n    362,\n    44,\n    363,\n    45,\n    364,\n    46,\n    365,\n    47,\n    366,\n    48,\n    367,\n    49,\n    368,\n    50,\n    369,\n    51,\n    370,\n    52,\n    371,\n    53,\n    372,\n    54,\n    373,\n    55,\n    374,\n    56,\n    375,\n    57,\n    376,\n    58,\n    377,\n    59,\n    378,\n    60,\n    379,\n    61,\n    380,\n    62,\n    381,\n    63,\n    382,\n    64,\n    383,\n    65,\n    384,\n    66,\n    385,\n    67,\n    386,\n    68,\n    387,\n    69,\n    388,\n    70,\n    389,\n    71,\n    390,\n    72,\n    391,\n    73,\n    392,\n    74,\n    393,\n    75,\n    394,\n    76,\n    395,\n    77,\n    396,\n    78,\n    397,\n    79,\n    398,\n    0,\n    399,\n    1,\n    400,\n    2,\n    401,\n    3,\n    402,\n    4,\n    403,\n    5,\n    404,\n    6,\n    405,\n    7,\n    406,\n    8,\n    407,\n    9,\n    408,\n    10,\n    409,\n    11,\n    410,\n    12,\n    411,\n    13,\n    412,\n    14,\n    413,\n    15,\n    414,\n    16,\n    415,\n    17,\n    416,\n    18,\n    417,\n    19,\n    418,\n    20,\n    419,\n    21,\n    420,\n    22,\n    421,\n    23,\n    422,\n    24,\n    423,\n    25,\n    424,\n    26,\n    425,\n    27,\n    426,\n    28,\n    427,\n    29,\n    428,\n    30,\n    429,\n    31,\n    430,\n    32,\n    431,\n    33,\n    432,\n    34,\n    433,\n    35,\n    434,\n    36,\n    435,\n    37,\n    436,\n    38,\n    437,\n    39,\n    438,\n    40,\n    439,\n    41,\n    440,\n    42,\n    441,\n    43,\n    442,\n    44,\n    443,\n    45,\n    444,\n    46,\n    445,\n    47,\n    446,\n    48,\n    447,\n    49,\n    448,\n    50,\n    449,\n    51,\n    450,\n    52,\n    451,\n    53,\n    452,\n    54,\n    453,\n    55,\n    454,\n    56,\n    455,\n    57,\n    456,\n    58,\n    457,\n    59,\n    458,\n    60,\n    459,\n    61,\n    460,\n    62,\n    461,\n    63,\n    462,\n    64,\n    463,\n    65,\n    464,\n    66,\n    465,\n    67,\n    466,\n    68,\n    467,\n    69,\n    468,\n    70,\n    469,\n    71,\n    470,\n    72,\n    471,\n    73,\n    472,\n    74,\n    473,\n    75,\n    474,\n    76,\n    475,\n    77,\n    476,\n    78,\n    477,\n    79,\n    478,\n    0,\n    479,\n    1,\n    480,\n    2,\n    481,\n    3,\n    482,\n    4,\n    483,\n    5,\n    484,\n    6,\n    485,\n    7,\n    486,\n    8,\n    487,\n    9,\n    488,\n    10,\n    489,\n    11,\n    490,\n    12,\n    491,\n    13,\n    492,\n    14,\n    493,\n    15,\n    494,\n    16,\n    495,\n    17,\n    496,\n    18,\n    497,\n    19,\n    498,\n    20,\n    499,\n    21,\n    500,\n    22,\n    501,\n    23,\n    502,\n    24,\n    80,\n    25,\n    81,\n    26,\n    82,\n    27,\n    83,\n    28,\n    84,\n    29,\n    85,\n    30,\n    86,\n    31,\n    87,\n    32,\n    88,\n    33,\n    89,\n    34,\n    90,\n    35,\n    91,\n    36,\n    92,\n    37,\n    93,\n    38,\n    94,\n    39,\n    95,\n    40,\n    96,\n    41,\n    97,\n    42,\n    98,\n    43,\n    99,\n    44,\n    100,\n    45,\n    101,\n    46,\n    102,\n    47,\n    103,\n    48,\n    104,\n    49,\n    105,\n    50,\n    106,\n    51,\n    107,\n    52,\n    108,\n    53,\n    109,\n    54,\n    110,\n    55,\n    111,\n    56,\n    112,\n    57,\n    113,\n    58,\n    114,\n    59,\n    115,\n    60,\n    116,\n    61,\n    117,\n    62,\n    118,\n    63,\n    119,\n    64,\n    120,\n    65,\n    121,\n    66,\n    122,\n    67,\n    123,\n    68,\n    124,\n    69,\n    125,\n    70,\n    126,\n    71,\n    127,\n    72,\n    128,\n    73,\n    129,\n    74,\n    130,\n    75,\n    131,\n    76,\n    132,\n    77,\n    133,\n    78,\n    134,\n    79,\n    135,\n    0,\n    136,\n    1,\n    137,\n    2,\n    138,\n    3,\n    139,\n    4,\n    140,\n    5,\n    141,\n    6,\n    142,\n    7,\n    143,\n    8,\n    144,\n    9,\n    145,\n    10,\n    146,\n    11,\n    147,\n    12,\n    148,\n    13,\n    149,\n    14,\n    150,\n    15,\n    151,\n    16,\n    152,\n    17,\n    153,\n    18,\n    154,\n    19,\n    155,\n    20,\n    156,\n    21,\n    157,\n    22,\n    158,\n    23,\n    159,\n    24,\n    160,\n    25,\n    161,\n    26,\n    162,\n    27,\n    163,\n    28,\n    164,\n    29,\n    165,\n    30,\n    166,\n    31,\n    167,\n    32,\n    168,\n    33,\n    169,\n    34,\n    170,\n    35,\n    171,\n    36,\n    172,\n    37,\n    173,\n    38,\n    174,\n    39,\n    175,\n    40,\n    176,\n    41,\n    177,\n    42,\n    178,\n    43,\n    179,\n    44,\n    180,\n    45,\n    181,\n    46,\n    182,\n    47,\n    183,\n    48,\n    184,\n    49,\n    185,\n    50,\n    186,\n    51,\n    187,\n    52,\n    188,\n    53,\n    189,\n    54,\n    190,\n    55,\n    191,\n    56,\n    192,\n    57,\n    193,\n    58,\n    194,\n    59,\n    195,\n    60,\n    196,\n    61,\n    197,\n    62,\n    198,\n    63,\n    199,\n    64,\n    200,\n    65,\n    201,\n    66,\n    202,\n    67,\n    203,\n    68,\n    204,\n    69,\n    205,\n    70,\n    206,\n    71,\n    207,\n    72,\n    208,\n    73,\n    209,\n    74,\n    210,\n    75,\n    211,\n    76,\n    212,\n    77,\n    213,\n    78,\n    214,\n    79,\n    215,\n    0,\n    216,\n    1,\n    217,\n    2,\n    218,\n    3,\n    219,\n    4,\n    220,\n    5,\n    221,\n    6,\n    222,\n    7,\n    223,\n    8,\n    224,\n    9,\n    225,\n    10,\n    226,\n    11,\n    227,\n    12,\n    228,\n    13,\n    229,\n    14,\n    230,\n    15,\n    231,\n    16,\n    232,\n    17,\n    233,\n    18,\n    234,\n    19,\n    235,\n    20,\n    236,\n    21,\n    237,\n    22,\n    238,\n    23,\n    239,\n    24,\n    240,\n    25,\n    241,\n    26,\n    242,\n    27,\n    243,\n    28,\n    244,\n    29,\n    245,\n    30,\n    246,\n    31,\n    247,\n    32,\n    248,\n    33,\n    249,\n    34,\n    250,\n    35,\n    251,\n    36,\n    252,\n    37,\n    253,\n    38,\n    254,\n    39,\n    255,\n    40,\n    256,\n    41,\n    257,\n    42,\n    258,\n    43,\n    259,\n    44,\n    260,\n    45,\n    261,\n    46,\n    262,\n    47,\n    263,\n    48,\n    264,\n    49,\n    265,\n    50,\n    266,\n    51,\n    267,\n    52,\n    268,\n    53,\n    269,\n    54,\n    270,\n    55,\n    271,\n    56,\n    272,\n    57,\n    273,\n    58,\n    274,\n    59,\n    275,\n    60,\n    276,\n    61,\n    277,\n    62,\n    278,\n    63,\n    279,\n    64,\n    280,\n    65,\n    281,\n    66,\n    282,\n    67,\n    283,\n    68,\n    284,\n    69,\n    285,\n    70,\n    286,\n    71,\n    287,\n    72,\n    288,\n    73,\n    289,\n    74,\n    290,\n    75,\n    291,\n    76,\n    292,\n    77,\n    293,\n    78,\n    294,\n    79,\n    295,\n    0,\n    296,\n    1,\n    297,\n    2,\n    298,\n    3,\n    299,\n    4,\n    300,\n    5,\n    301,\n    6,\n    302,\n    7,\n    303,\n    8,\n    304,\n    9,\n    305,\n    10,\n    306,\n    11,\n    307,\n    12,\n    308,\n    13,\n    309,\n    14,\n    310,\n    15,\n    311,\n    16,\n    312,\n    17,\n    313,\n    18,\n    314,\n    19,\n    315,\n    20,\n    316,\n    21,\n    317,\n    22,\n    318,\n    23,\n    319,\n    24,\n    320,\n    25,\n    321,\n    26,\n    322,\n    27,\n    323,\n    28,\n    324,\n    29,\n    325,\n    30,\n    326,\n    31,\n    327,\n    32,\n    328,\n    33,\n    329,\n    34,\n    330,\n    35,\n    331,\n    36,\n    332,\n    37,\n    333,\n    38,\n    334,\n    39,\n    335,\n    40,\n    336,\n    41,\n    337,\n    42,\n    338,\n    43,\n    339,\n    44,\n    340,\n    45,\n    341,\n    46,\n    342,\n    47,\n    343,\n    48,\n    344,\n    49,\n    345,\n    50,\n    346,\n    51,\n    347,\n    52,\n    348,\n    53,\n    349,\n    54,\n    350,\n    55,\n    351,\n    56,\n    352,\n    57,\n    353,\n    58,\n    354,\n    59,\n    355,\n    60,\n    356,\n    61,\n    357,\n    62,\n    358,\n    63,\n    359,\n    64,\n    360,\n    65,\n    361,\n    66,\n    362,\n    67,\n    363,\n    68,\n    364,\n    69,\n    365,\n    70,\n    366,\n    71,\n    367,\n    72,\n    368,\n    73,\n    369,\n    74,\n    370,\n    75,\n    371,\n    76,\n    372,\n    77,\n    373,\n    78,\n    374,\n    79,\n    375,\n    0,\n    376,\n    1,\n    377,\n    2,\n    378,\n    3,\n    379,\n    4,\n    380,\n    5,\n    381,\n    6,\n    382,\n    7,\n    383,\n    8,\n    384,\n    9,\n    385,\n    10,\n    386,\n    11,\n    387,\n    12,\n    388,\n    13,\n    389,\n    14,\n    390,\n    15,\n    391,\n    16,\n    392,\n    17,\n    393,\n    18,\n    394,\n    19,\n    395,\n    20,\n    396,\n    21,\n    397,\n    22,\n    398,\n    23,\n    399,\n    24,\n    400,\n    25,\n    401,\n    26,\n    402,\n    27,\n    403,\n    28,\n    404,\n    29,\n    405,\n    30,\n    406,\n    31,\n    407,\n    32,\n    408,\n    33,\n    409,\n    34,\n    410,\n    35,\n    411,\n    36,\n    412,\n    37,\n    413,\n    38,\n    414,\n    39,\n    415,\n    40,\n    416,\n    41,\n    417,\n    42,\n    418,\n    43,\n    419,\n    44,\n    420,\n    45,\n    421,\n    46,\n    422,\n    47,\n    423,\n    48,\n    424,\n    49,\n    425,\n    50,\n    426,\n    51,\n    427,\n    52,\n    428,\n    53,\n    429,\n    54,\n    430,\n    55,\n    431,\n    56,\n    432,\n    57,\n    433,\n    58,\n    434,\n    59,\n    435,\n    60,\n    436,\n    61,\n    437,\n    62,\n    438,\n    63,\n    439,\n    64,\n    440,\n    65,\n    441,\n    66,\n    442,\n    67,\n    443,\n    68,\n    444,\n    69,\n    445,\n    70,\n    446,\n    71,\n    447,\n    72,\n    448,\n    73,\n    449,\n    74,\n    450,\n    75,\n    451,\n    76,\n    452,\n    77,\n    453,\n    78,\n    454,\n    79,\n    455,\n    0,\n    456,\n    1,\n    457,\n    2,\n    458,\n    3,\n    459,\n    4,\n    460,\n    5,\n    461,\n    6,\n    462,\n    7,\n    463,\n    8,\n    464,\n    9,\n    465,\n    10,\n    466,\n    11,\n    467,\n    12,\n    468,\n    13,\n    469,\n    14,\n    470,\n    15,\n    471,\n    16,\n    472,\n    17,\n    473,\n    18,\n    474,\n    19,\n    475,\n    20,\n    476,\n    21,\n    477,\n    22,\n    478,\n    23,\n    479,\n    24,\n    480,\n    25,\n    481,\n    26,\n    482,\n    27,\n    483,\n    28,\n    484,\n    29,\n    485,\n    30,\n    486,\n    31,\n    487,\n    32,\n    488,\n    33,\n    489,\n    34,\n    490,\n    35,\n    491,\n    36,\n    492,\n    37,\n    493,\n    38,\n    494,\n    39,\n    495,\n    40,\n    496,\n    41,\n    497,\n    42,\n    498,\n    43,\n    499,\n    44,\n    500,\n    45,\n    501,\n    46,\n    502,\n    47,\n    80,\n    48,\n    81,\n    49,\n    82,\n    50,\n    83,\n    51,\n    84,\n    52,\n    85,\n    53,\n    86,\n    54,\n    87,\n    55,\n    88,\n    56,\n    89,\n    57,\n    90,\n    58,\n    91,\n    59,\n    92,\n    60,\n    93,\n    61,\n    94,\n    62,\n    95,\n    63,\n    96,\n    64,\n    97,\n    65,\n    98,\n    66,\n    99,\n    67,\n    100,\n    68,\n    101,\n    69,\n    102,\n    70,\n    103,\n    71,\n    104,\n    72,\n    105,\n    73,\n    106,\n    74,\n    107,\n    75,\n    108,\n    76,\n    109,\n    77,\n    110,\n    78,\n    111,\n    79,\n    112,\n    0,\n    113,\n    1,\n    114,\n    2,\n    115,\n    3,\n    116,\n    4,\n    117,\n    5,\n    118,\n    6,\n    119,\n    7,\n    120,\n    8,\n    121,\n    9,\n    122,\n    10,\n    123,\n    11,\n    124,\n    12,\n    125,\n    13,\n    126,\n    14,\n    127,\n    15,\n    128,\n    16,\n    129,\n    17,\n    130,\n    18,\n    131,\n    19,\n    132,\n    20,\n    133,\n    21,\n    134,\n    22,\n    135,\n    23,\n    136,\n    24,\n    137,\n    25,\n    138,\n    26,\n    139,\n    27,\n    140,\n    28,\n    141,\n    29,\n    142,\n    30,\n    143,\n    31,\n    144,\n    32,\n    145,\n    33,\n    146,\n    34,\n    147,\n    35,\n    148,\n    36,\n    149,\n    37,\n    150,\n    38,\n    151,\n    39,\n    152,\n    40,\n    153,\n    41,\n    154,\n    42,\n    155,\n    43,\n    156,\n    44,\n    157,\n    45,\n    158,\n    46,\n    159,\n    47,\n    160,\n    48,\n    161,\n    49,\n    162,\n    50,\n    163,\n    51,\n    164,\n    52,\n    165,\n    53,\n    166,\n    54,\n    167,\n    55,\n    168,\n    56,\n    169,\n    57,\n    170,\n    58,\n    171,\n    59,\n    172,\n    60,\n    173,\n    61,\n    174,\n    62,\n    175,\n    63,\n    176,\n    64,\n    177,\n    65,\n    178,\n    66,\n    179,\n    67,\n    180,\n    68,\n    181,\n    69,\n    182,\n    70,\n    183,\n    71,\n    184,\n    72,\n    185,\n    73,\n    186,\n    74,\n    187,\n    75,\n    188,\n    76,\n    189,\n    77,\n    190,\n    78,\n    191,\n    79,\n    192,\n    0,\n    193,\n    1,\n    194,\n    2,\n    195,\n    3,\n    196,\n    4,\n    197,\n    5,\n    198,\n    6,\n    199,\n    7,\n    200,\n    8,\n    201,\n    9,\n    202,\n    10,\n    203,\n    11,\n    204,\n    12,\n    205,\n    13,\n    206,\n    14,\n    207,\n    15,\n    208,\n    16,\n    209,\n    17,\n    210,\n    18,\n    211,\n    19,\n    212,\n    20,\n    213,\n    21,\n    214,\n    22,\n    215,\n    23,\n    216,\n    24,\n    217,\n    25,\n    218,\n    26,\n    219,\n    27,\n    220,\n    28,\n    221,\n    29,\n    222,\n    30,\n    223,\n    31,\n    224,\n    32,\n    225,\n    33,\n    226,\n    34,\n    227,\n    35,\n    228,\n    36,\n    229,\n    37,\n    230,\n    38,\n    231,\n    39,\n    232,\n    40,\n    233,\n    41,\n    234,\n    42,\n    235,\n    43,\n    236,\n    44,\n    237,\n    45,\n    238,\n    46,\n    239,\n    47,\n    240,\n    48,\n    241,\n    49,\n    242,\n    50,\n    243,\n    51,\n    244,\n    52,\n    245,\n    53,\n    246,\n    54,\n    247,\n    55,\n    248,\n    56,\n    249,\n    57,\n    250,\n    58,\n    251,\n    59,\n    252,\n    60,\n    253,\n    61,\n    254,\n    62,\n    255,\n    63,\n    256,\n    64,\n    257,\n    65,\n    258,\n    66,\n    259,\n    67,\n    260,\n    68,\n    261,\n    69,\n    262,\n    70,\n    263,\n    71,\n    264,\n    72,\n    265,\n    73,\n    266,\n    74,\n    267,\n    75,\n    268,\n    76,\n    269,\n    77,\n    270,\n    78,\n    271,\n    79,\n    272,\n    0,\n    273,\n    1,\n    274,\n    2,\n    275,\n    3,\n    276,\n    4,\n    277,\n    5,\n    278,\n    6,\n    279,\n    7,\n    280,\n    8,\n    281,\n    9,\n    282,\n    10,\n    283,\n    11,\n    284,\n    12,\n    285,\n    13,\n    286,\n    14,\n    287,\n    15,\n    288,\n    16,\n    289,\n    17,\n    290,\n    18,\n    291,\n    19,\n    292,\n    20,\n    293,\n    21,\n    294,\n    22,\n    295,\n    23,\n    296,\n    24,\n    297,\n    25,\n    298,\n    26,\n    299,\n    27,\n    300,\n    28,\n    301,\n    29,\n    302,\n    30,\n    303,\n    31,\n    304,\n    32,\n    305,\n    33,\n    306,\n    34,\n    307,\n    35,\n    308,\n    36,\n    309,\n    37,\n    310,\n    38,\n    311,\n    39,\n    312,\n    40,\n    313,\n    41,\n    314,\n    42,\n    315,\n    43,\n    316,\n    44,\n    317,\n    45,\n    318,\n    46,\n    319,\n    47,\n    320,\n    48,\n    321,\n    49,\n    322,\n    50,\n    323,\n    51,\n    324,\n    52,\n    325,\n    53,\n    326,\n    54,\n    327,\n    55,\n    328,\n    56,\n    329,\n    57,\n    330,\n    58,\n    331,\n    59,\n    332,\n    60,\n    333,\n    61,\n    334,\n    62,\n    335,\n    63,\n    336,\n    64,\n    337,\n    65,\n    338,\n    66,\n    339,\n    67,\n    340,\n    68,\n    341,\n    69,\n    342,\n    70,\n    343,\n    71,\n    344,\n    72,\n    345,\n    73,\n    346,\n    74,\n    347,\n    75,\n    348,\n    76,\n    349,\n    77,\n    350,\n    78,\n    351,\n    79,\n    352,\n    0,\n    353,\n    1,\n    354,\n    2,\n    355,\n    3,\n    356,\n    4,\n    357,\n    5,\n    358,\n    6,\n    359,\n    7,\n    360,\n    8,\n    361,\n    9,\n    362,\n    10,\n    363,\n    11,\n    364,\n    12,\n    365,\n    13,\n    366,\n    14,\n    367,\n    15,\n    368,\n    16,\n    369,\n    17,\n    370,\n    18,\n    371,\n    19,\n    372,\n    20,\n    373,\n    21,\n    374,\n    22,\n    375,\n    23,\n    376,\n    24,\n    377,\n    25,\n    378,\n    26,\n    379,\n    27,\n    380,\n    28,\n    381,\n    29,\n    382,\n    30,\n    383,\n    31,\n    384,\n    32,\n    385,\n    33,\n    386,\n    34,\n    387,\n    35,\n    388,\n    36,\n    389,\n    37,\n    390,\n    38,\n    391,\n    39,\n    392,\n    40,\n    393,\n    41,\n    394,\n    42,\n    395,\n    43,\n    396,\n    44,\n    397,\n    45,\n    398,\n    46,\n    399,\n    47,\n    400,\n    48,\n    401,\n    49,\n    402,\n    50,\n    403,\n    51,\n    404,\n    52,\n    405,\n    53,\n    406,\n    54,\n    407,\n    55,\n    408,\n    56,\n    409,\n    57,\n    410,\n    58,\n    411,\n    59,\n    412,\n    60,\n    413,\n    61,\n    414,\n    62,\n    415,\n    63,\n    416,\n    64,\n    417,\n    65,\n    418,\n    66,\n    419,\n    67,\n    420,\n    68,\n    421,\n    69,\n    422,\n    70,\n    423,\n    71,\n    424,\n    72,\n    425,\n    73,\n    426,\n    74,\n    427,\n    75,\n    428,\n    76,\n    429,\n    77,\n    430,\n    78,\n    431,\n    79,\n    432,\n    0,\n    433,\n    1,\n    434,\n    2,\n    435,\n    3,\n    436,\n    4,\n    437,\n    5,\n    438,\n    6,\n    439,\n    7,\n    440,\n    8,\n    441,\n    9,\n    442,\n    10,\n    443,\n    11,\n    444,\n    12,\n    445,\n    13,\n    446,\n    14,\n    447,\n    15,\n    448,\n    16,\n    449,\n    17,\n    450,\n    18,\n    451,\n    19,\n    452,\n    20,\n    453,\n    21,\n    454,\n    22,\n    455,\n    23,\n    456,\n    24,\n    457,\n    25,\n    458,\n    26,\n    459,\n    27,\n    460,\n    28,\n    461,\n    29,\n    462,\n    30,\n    463,\n    31,\n    464,\n    32,\n    465,\n    33,\n    466,\n    34,\n    467,\n    35,\n    468,\n    36,\n    469,\n    37,\n    470,\n    38,\n    471,\n    39,\n    472,\n    40,\n    473,\n    41,\n    474,\n    42,\n    475,\n    43,\n    476,\n    44,\n    477,\n    45,\n    478,\n    46,\n    479,\n    47,\n    480,\n    48,\n    481,\n    49,\n    482,\n    50,\n    483,\n    51,\n    484,\n    52,\n    485,\n    53,\n    486,\n    54,\n    487,\n    55,\n    488,\n    56,\n    489,\n    57,\n    490,\n    58,\n    491,\n    59,\n    492,\n    60,\n    493,\n    61,\n    494,\n    62,\n    495,\n    63,\n    496,\n    64,\n    497,\n    65,\n    498,\n    66,\n    499,\n    67,\n    500,\n    68,\n    501,\n    69,\n    502,\n    70,\n    80,\n    71,\n    81,\n    72,\n    82,\n    73,\n    83,\n    74,\n    84,\n    75,\n    85,\n    76,\n    86,\n    77,\n    87,\n    78,\n    88,\n    79,\n    89,\n    0,\n    90,\n    1,\n    91,\n    2,\n    92,\n    3,\n    93,\n    4,\n    94,\n    5,\n    95,\n    6,\n    96,\n    7,\n    97,\n    8,\n    98,\n    9,\n    99,\n    10,\n    100,\n    11,\n    101,\n    12,\n    102,\n    13,\n    103,\n    14,\n    104,\n    15,\n    105,\n    16,\n    106,\n    17,\n    107,\n    18,\n    108,\n    19,\n    109,\n    20,\n    110,\n    21,\n    111,\n    22,\n    112,\n    23,\n    113,\n    24,\n    114,\n    25,\n    115,\n    26,\n    116,\n    27,\n    117,\n    28,\n    118,\n    29,\n    119,\n    30,\n    120,\n    31,\n    121,\n    32,\n    122,\n    33,\n    123,\n    34,\n    124,\n    35,\n    125,\n    36,\n    126,\n    37,\n    127,\n    38,\n    128,\n    39,\n    129,\n    40,\n    130,\n    41,\n    131,\n    42,\n    132,\n    43,\n    133,\n    44,\n    134,\n    45,\n    135,\n    46,\n    136,\n    47,\n    137,\n    48,\n    138,\n    49,\n    139,\n    50,\n    140,\n    51,\n    141,\n    52,\n    142,\n    53,\n    143,\n    54,\n    144,\n    55,\n    145,\n    56,\n    146,\n    57,\n    147,\n    58,\n    148,\n    59,\n    149,\n    60,\n    150,\n    61,\n    151,\n    62,\n    152,\n    63,\n    153,\n    64,\n    154,\n    65,\n    155,\n    66,\n    156,\n    67,\n    157,\n    68,\n    158,\n    69,\n    159,\n    70,\n    160,\n    71,\n    161,\n    72,\n    162,\n    73,\n    163,\n    74,\n    164,\n    75,\n    165,\n    76,\n    166,\n    77,\n    167,\n    78,\n    168,\n    79,\n    169,\n    0,\n    170,\n    1,\n    171,\n    2,\n    172,\n    3,\n    173,\n    4,\n    174,\n    5,\n    175,\n    6,\n    176,\n    7,\n    177,\n    8,\n    178,\n    9,\n    179,\n    10,\n    180,\n    11,\n    181,\n    12,\n    182,\n    13,\n    183,\n    14,\n    184,\n    15,\n    185,\n    16,\n    186,\n    17,\n    187,\n    18,\n    188,\n    19,\n    189,\n    20,\n    190,\n    21,\n    191,\n    22,\n    192,\n    23,\n    193,\n    24,\n    194,\n    25,\n    195,\n    26,\n    196,\n    27,\n    197,\n    28,\n    198,\n    29,\n    199,\n    30,\n    200,\n    31,\n    201,\n    32,\n    202,\n    33,\n    203,\n    34,\n    204,\n    35,\n    205,\n    36,\n    206,\n    37,\n    207,\n    38,\n    208,\n    39,\n    209,\n    40,\n    210,\n    41,\n    211,\n    42,\n    212,\n    43,\n    213,\n    44,\n    214,\n    45,\n    215,\n    46,\n    216,\n    47,\n    217,\n    48,\n    218,\n    49,\n    219,\n    50,\n    220,\n    51,\n    221,\n    52,\n    222,\n    53,\n    223,\n    54,\n    224,\n    55,\n    225,\n    56,\n    226,\n    57,\n    227,\n    58,\n    228,\n    59,\n    229,\n    60,\n    230,\n    61,\n    231,\n    62,\n    232,\n    63,\n    233,\n    64,\n    234,\n    65,\n    235,\n    66,\n    236,\n    67,\n    237,\n    68,\n    238,\n    69,\n    239,\n    70,\n    240,\n    71,\n    241,\n    72,\n    242,\n    73,\n    243,\n    74,\n    244,\n    75,\n    245,\n    76,\n    246,\n    77,\n    247,\n    78,\n    248,\n    79,\n    249,\n    0,\n    250,\n    1,\n    251,\n    2,\n    252,\n    3,\n    253,\n    4,\n    254,\n    5,\n    255,\n    6,\n    256,\n    7,\n    257,\n    8,\n    258,\n    9,\n    259,\n    10,\n    260,\n    11,\n    261,\n    12,\n    262,\n    13,\n    263,\n    14,\n    264,\n    15,\n    265,\n    16,\n    266,\n    17,\n    267,\n    18,\n    268,\n    19,\n    269,\n    20,\n    270,\n    21,\n    271,\n    22,\n    272,\n    23,\n    273,\n    24,\n    274,\n    25,\n    275,\n    26,\n    276,\n    27,\n    277,\n    28,\n    278,\n    29,\n    279,\n    30,\n    280,\n    31,\n    281,\n    32,\n    282,\n    33,\n    283,\n    34,\n    284,\n    35,\n    285,\n    36,\n    286,\n    37,\n    287,\n    38,\n    288,\n    39,\n    289,\n    40,\n    290,\n    41,\n    291,\n    42,\n    292,\n    43,\n    293,\n    44,\n    294,\n    45,\n    295,\n    46,\n    296,\n    47,\n    297,\n    48,\n    298,\n    49,\n    299,\n    50,\n    300,\n    51,\n    301,\n    52,\n    302,\n    53,\n    303,\n    54,\n    304,\n    55,\n    305,\n    56,\n    306,\n    57,\n    307,\n    58,\n    308,\n    59,\n    309,\n    60,\n    310,\n    61,\n    311,\n    62,\n    312,\n    63,\n    313,\n    64,\n    314,\n    65,\n    315,\n    66,\n    316,\n    67,\n    317,\n    68,\n    318,\n    69,\n    319,\n    70,\n    320,\n    71,\n    321,\n    72,\n    322,\n    73,\n    323,\n    74,\n    324,\n    75,\n    325,\n    76,\n    326,\n    77,\n    327,\n    78,\n    328,\n    79,\n    329,\n    0,\n    330,\n    1,\n    331,\n    2,\n    332,\n    3,\n    333,\n    4,\n    334,\n    5,\n    335,\n    6,\n    336,\n    7,\n    337,\n    8,\n    338,\n    9,\n    339,\n    10,\n    340,\n    11,\n    341,\n    12,\n    342,\n    13,\n    343,\n    14,\n    344,\n    15,\n    345,\n    16,\n    346,\n    17,\n    347,\n    18,\n    348,\n    19,\n    349,\n    20,\n    350,\n    21,\n    351,\n    22,\n    352,\n    23,\n    353,\n    24,\n    354,\n    25,\n    355,\n    26,\n    356,\n    27,\n    357,\n    28,\n    358,\n    29,\n    359,\n    30,\n    360,\n    31,\n    361,\n    32,\n    362,\n    33,\n    363,\n    34,\n    364,\n    35,\n    365,\n    36,\n    366,\n    37,\n    367,\n    38,\n    368,\n    39,\n    369,\n    40,\n    370,\n    41,\n    371,\n    42,\n    372,\n    43,\n    373,\n    44,\n    374,\n    45,\n    375,\n    46,\n    376,\n    47,\n    377,\n    48,\n    378,\n    49,\n    379,\n    50,\n    380,\n    51,\n    381,\n    52,\n    382,\n    53,\n    383,\n    54,\n    384,\n    55,\n    385,\n    56,\n    386,\n    57,\n    387,\n    58,\n    388,\n    59,\n    389,\n    60,\n    390,\n    61,\n    391,\n    62,\n    392,\n    63,\n    393,\n    64,\n    394,\n    65,\n    395,\n    66,\n    396,\n    67,\n    397,\n    68,\n    398,\n    69,\n    399,\n    70,\n    400,\n    71,\n    401,\n    72,\n    402,\n    73,\n    403,\n    74,\n    404,\n    75,\n    405,\n    76,\n    406,\n    77,\n    407,\n    78,\n    408,\n    79,\n    409,\n    0,\n    410,\n    1,\n    411,\n    2,\n    412,\n    3,\n    413,\n    4,\n    414,\n    5,\n    415,\n    6,\n    416,\n    7,\n    417,\n    8,\n    418,\n    9,\n    419,\n    10,\n    420,\n    11,\n    421,\n    12,\n    422,\n    13,\n    423,\n    14,\n    424,\n    15,\n    425,\n    16,\n    426,\n    17,\n    427,\n    18,\n    428,\n    19,\n    429,\n    20,\n    430,\n    21,\n    431,\n    22,\n    432,\n    23,\n    433,\n    24,\n    434,\n    25,\n    435,\n    26,\n    436,\n    27,\n    437,\n    28,\n    438,\n    29,\n    439,\n    30,\n    440,\n    31,\n    441,\n    32,\n    442,\n    33,\n    443,\n    34,\n    444,\n    35,\n    445,\n    36,\n    446,\n    37,\n    447,\n    38,\n    448,\n    39,\n    449,\n    40,\n    450,\n    41,\n    451,\n    42,\n    452,\n    43,\n    453,\n    44,\n    454,\n    45,\n    455,\n    46,\n    456,\n    47,\n    457,\n    48,\n    458,\n    49,\n    459,\n    50,\n    460,\n    51,\n    461,\n    52,\n    462,\n    53,\n    463,\n    54,\n    464,\n    55,\n    465,\n    56,\n    466,\n    57,\n    467,\n    58,\n    468,\n    59,\n    469,\n    60,\n    470,\n    61,\n    471,\n    62,\n    472,\n    63,\n    473,\n    64,\n    474,\n    65,\n    475,\n    66,\n    476,\n    67,\n    477,\n    68,\n    478,\n    69,\n    479,\n    70,\n    480,\n    71,\n    481,\n    72,\n    482,\n    73,\n    483,\n    74,\n    484,\n    75,\n    485,\n    76,\n    486,\n    77,\n    487,\n    78,\n    488,\n    79,\n    489,\n    0,\n    490,\n    1,\n    491,\n    2,\n    492,\n    3,\n    493,\n    4,\n    494,\n    5,\n    495,\n    6,\n    496,\n    7,\n    497,\n    8,\n    498,\n    9,\n    499,\n    10,\n    500,\n    11,\n    501,\n    12,\n    502,\n    13,\n    80,\n    14,\n    81,\n    15,\n    82,\n    16,\n    83,\n    17,\n    84,\n    18,\n    85,\n    19,\n    86,\n    20,\n    87,\n    21,\n    88,\n    22,\n    89,\n    23,\n    90,\n    24,\n    91,\n    25,\n    92,\n    26,\n    93,\n    27,\n    94,\n    28,\n    95,\n    29,\n    96,\n    30,\n    97,\n    31,\n    98,\n    32,\n    99,\n    33,\n    100,\n    34,\n    101,\n    35,\n    102,\n    36,\n    103,\n    37,\n    104,\n    38,\n    105,\n    39,\n    106,\n    40,\n    107,\n    41,\n    108,\n    42,\n    109,\n    43,\n    110,\n    44,\n    111,\n    45,\n    112,\n    46,\n    113,\n    47,\n    114,\n    48,\n    115,\n    49,\n    116,\n    50,\n    117,\n    51,\n    118,\n    52,\n    119,\n    53,\n    120,\n    54,\n    121,\n    55,\n    122,\n    56,\n    123,\n    57,\n    124,\n    58,\n    125,\n    59,\n    126,\n    60,\n    127,\n    61,\n    128,\n    62,\n    129,\n    63,\n    130,\n    64,\n    131,\n    65,\n    132,\n    66,\n    133,\n    67,\n    134,\n    68,\n    135,\n    69,\n    136,\n    70,\n    137,\n    71,\n    138,\n    72,\n    139,\n    73,\n    140,\n    74,\n    141,\n    75,\n    142,\n    76,\n    143,\n    77,\n    144,\n    78,\n    145,\n    79,\n    146,\n    0,\n    147,\n    1,\n    148,\n    2,\n    149,\n    3,\n    150,\n    4,\n    151,\n    5,\n    152,\n    6,\n    153,\n    7,\n    154,\n    8,\n    155,\n    9,\n    156,\n    10,\n    157,\n    11,\n    158,\n    12,\n    159,\n    13,\n    160,\n    14,\n    161,\n    15,\n    162,\n    16,\n    163,\n    17,\n    164,\n    18,\n    165,\n    19,\n    166,\n    20,\n    167,\n    21,\n    168,\n    22,\n    169,\n    23,\n    170,\n    24,\n    171,\n    25,\n    172,\n    26,\n    173,\n    27,\n    174,\n    28,\n    175,\n    29,\n    176,\n    30,\n    177,\n    31,\n    178,\n    32,\n    179,\n    33,\n    180,\n    34,\n    181,\n    35,\n    182,\n    36,\n    183,\n    37,\n    184,\n    38,\n    185,\n    39,\n    186,\n    40,\n    187,\n    41,\n    188,\n    42,\n    189,\n    43,\n    190,\n    44,\n    191,\n    45,\n    192,\n    46,\n    193,\n    47,\n    194,\n    48,\n    195,\n    49,\n    196,\n    50,\n    197,\n    51,\n    198,\n    52,\n    199,\n    53,\n    200,\n    54,\n    201,\n    55,\n    202,\n    56,\n    203,\n    57,\n    204,\n    58,\n    205,\n    59,\n    206,\n    60,\n    207,\n    61,\n    208,\n    62,\n    209,\n    63,\n    210,\n    64,\n    211,\n    65,\n    212,\n    66,\n    213,\n    67,\n    214,\n    68,\n    215,\n    69,\n    216,\n    70,\n    217,\n    71,\n    218,\n    72,\n    219,\n    73,\n    220,\n    74,\n    221,\n    75,\n    222,\n    76,\n    223,\n    77,\n    224,\n    78,\n    225,\n    79,\n    226,\n    0,\n    227,\n    1,\n    228,\n    2,\n    229,\n    3,\n    230,\n    4,\n    231,\n    5,\n    232,\n    6,\n    233,\n    7,\n    234,\n    8,\n    235,\n    9,\n    236,\n    10,\n    237,\n    11,\n    238,\n    12,\n    239,\n    13,\n    240,\n    14,\n    241,\n    15,\n    242,\n    16,\n    243,\n    17,\n    244,\n    18,\n    245,\n    19,\n    246,\n    20,\n    247,\n    21,\n    248,\n    22,\n    249,\n    23,\n    250,\n    24,\n    251,\n    25,\n    252,\n    26,\n    253,\n    27,\n    254,\n    28,\n    255,\n    29,\n    256,\n    30,\n    257,\n    31,\n    258,\n    32,\n    259,\n    33,\n    260,\n    34,\n    261,\n    35,\n    262,\n    36,\n    263,\n    37,\n    264,\n    38,\n    265,\n    39,\n    266,\n    40,\n    267,\n    41,\n    268,\n    42,\n    269,\n    43,\n    270,\n    44,\n    271,\n    45,\n    272,\n    46,\n    273,\n    47,\n    274,\n    48,\n    275,\n    49,\n    276,\n    50,\n    277,\n    51,\n    278,\n    52,\n    279,\n    53,\n    280,\n    54,\n    281,\n    55,\n    282,\n    56,\n    283,\n    57,\n    284,\n    58,\n    285,\n    59,\n    286,\n    60,\n    287,\n    61,\n    288,\n    62,\n    289,\n    63,\n    290,\n    64,\n    291,\n    65,\n    292,\n    66,\n    293,\n    67,\n    294,\n    68,\n    295,\n    69,\n    296,\n    70,\n    297,\n    71,\n    298,\n    72,\n    299,\n    73,\n    300,\n    74,\n    301,\n    75,\n    302,\n    76,\n    303,\n    77,\n    304,\n    78,\n    305,\n    79,\n    306,\n    0,\n    307,\n    1,\n    308,\n    2,\n    309,\n    3,\n    310,\n    4,\n    311,\n    5,\n    312,\n    6,\n    313,\n    7,\n    314,\n    8,\n    315,\n    9,\n    316,\n    10,\n    317,\n    11,\n    318,\n    12,\n    319,\n    13,\n    320,\n    14,\n    321,\n    15,\n    322,\n    16,\n    323,\n    17,\n    324,\n    18,\n    325,\n    19,\n    326,\n    20,\n    327,\n    21,\n    328,\n    22,\n    329,\n    23,\n    330,\n    24,\n    331,\n    25,\n    332,\n    26,\n    333,\n    27,\n    334,\n    28,\n    335,\n    29,\n    336,\n    30,\n    337,\n    31,\n    338,\n    32,\n    339,\n    33,\n    340,\n    34,\n    341,\n    35,\n    342,\n    36,\n    343,\n    37,\n    344,\n    38,\n    345,\n    39,\n    346,\n    40,\n    347,\n    41,\n    348,\n    42,\n    349,\n    43,\n    350,\n    44,\n    351,\n    45,\n    352,\n    46,\n    353,\n    47,\n    354,\n    48,\n    355,\n    49,\n    356,\n    50,\n    357,\n    51,\n    358,\n    52,\n    359,\n    53,\n    360,\n    54,\n    361,\n    55,\n    362,\n    56,\n    363,\n    57,\n    364,\n    58,\n    365,\n    59,\n    366,\n    60,\n    367,\n    61,\n    368,\n    62,\n    369,\n    63,\n    370,\n    64,\n    371,\n    65,\n    372,\n    66,\n    373,\n    67,\n    374,\n    68,\n    375,\n    69,\n    376,\n    70,\n    377,\n    71,\n    378,\n    72,\n    379,\n    73,\n    380,\n    74,\n    381,\n    75,\n    382,\n    76,\n    383,\n    77,\n    384,\n    78,\n    385,\n    79,\n    386,\n    0,\n    387,\n    1,\n    388,\n    2,\n    389,\n    3,\n    390,\n    4,\n    391,\n    5,\n    392,\n    6,\n    393,\n    7,\n    394,\n    8,\n    395,\n    9,\n    396,\n    10,\n    397,\n    11,\n    398,\n    12,\n    399,\n    13,\n    400,\n    14,\n    401,\n    15,\n    402,\n    16,\n    403,\n    17,\n    404,\n    18,\n    405,\n    19,\n    406,\n    20,\n    407,\n    21,\n    408,\n    22,\n    409,\n    23,\n    410,\n    24,\n    411,\n    25,\n    412,\n    26,\n    413,\n    27,\n    414,\n    28,\n    415,\n    29,\n    416,\n    30,\n    417,\n    31,\n    418,\n    32,\n    419,\n    33,\n    420,\n    34,\n    421,\n    35,\n    422,\n    36,\n    423,\n    37,\n    424,\n    38,\n    425,\n    39,\n    426\n  ],\n  "test_used": false,\n  "policy": "Fit uses explicit actor/action zero only. Missing role is UNKNOWN. Predicate pairs balance both directions within object class/action pair. No probe/calibration loss.",\n  "caption_overlap": "All auxiliary images are caption train/restval, historically seen by the backbone; relation partitions are disjoint, not unseen caption tests.",\n  "object_instance_scope": "Wrong-object probe is transfer only; no unconfirmed object negatives are synthesized."\n}'
VERIFIED_MANIFEST = REPO/'verified_manifest.json'
VERIFIED_MANIFEST.write_text(VERIFIED_LABELS,encoding='utf-8')
VERIFIED_MANIFEST_SHA = '7242a9a632ecf715d47cb2a9838a4ecacfad06b9eeca9849f6aa7ebc9e4beffa'
print('Verified F source and locked labels installed',flush=True)


## 2. Preflight
Kiểm tra dữ liệu, loss và gradient bằng dữ liệu giả, sau đó đồng bộ hai GPU và metric. Notebook dừng nếu kiểm tra lỗi.


In [ ]:
for name in ('test_pilot_data.py','test_verified_data.py','test_verified_model.py'):
    run([sys.executable,'-u',REPO/name],cwd=REPO)
run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
     '--max_restarts=0',REPO/'test_verified_ddp.py'],cwd=REPO)
run([sys.executable,'-u',REPO/'pilot_metric_smoke.py',CHAIR],cwd=REPO)
print('PREFLIGHT PASS',flush=True)


## 3. Chuẩn bị dữ liệu
Stage cùng 11.000 ảnh caption như A/E và các ảnh trong bộ nhãn xác nhận.
Embedding CLIP đóng băng; riêng tập fit được đưa vào loss. Hai tập còn lại chỉ dùng chẩn đoán.


In [ ]:
CONFIG = Path('/tmp/semantic_evidence_pilot_config_' + ARM + '.json')
expected = {
    'checkpoint':'5f8eead3f4e960189345b1a9bf492763fe52ec71f70115b73244ffe9d03d2e75',
    'legacy_prompt':'d7910e9bf87e3aad86d806c77ec336b34793e3bef68a3f662fdcb2e9456ca38d',
    'upt_prompt':'24e7861a4aa0e7e731d67689dce195688096232bcbef1b4264034cf520d6957d',
    'dataset_json':'2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d',
    'detections':'498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c',
}
cfg = dict(arm=ARM,checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),upt_prompt=str(UPT_PROMPT),
           visual_cache=str(VISUAL_CACHE),dataset_json=str(COCO_JSON),detections=str(DETECTIONS),
           output=str(OUTPUT),stage='/tmp/semantic_evidence_stage_'+ARM+'.h5',chair=str(CHAIR),
           global_batch_size=8,eval_batch_size=16,warmup_updates=200,base_lr=5e-6,branch_lr=1e-4,
           scst_weight=.05,expected_hashes=expected,seed=42,project_commit=PROJECT_COMMIT,
           embedded_source_sha256=SOURCE_HASHES,verified_manifest=str(VERIFIED_MANIFEST),
           verified_manifest_sha256=VERIFIED_MANIFEST_SHA,aux_stage='/tmp/verified_same_image_aux_F.h5')
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
run([sys.executable,'-u',REPO/'run_verified.py','prepare','--config',CONFIG],cwd=REPO)
PROTOCOL = OUTPUT/'protocol.json'
(OUTPUT/'source').mkdir()
for name,source in SOURCES.items(): (OUTPUT/'source'/name).write_text(source,encoding='utf-8')
shutil.copy2(SETUP_LOG,OUTPUT/'setup.log')
locked = json.loads(PROTOCOL.read_text())
print('LOCKED COHORT:',locked['cohort_sha256'],locked['counts_by_mode'],flush=True)

print('LOCKED VERIFIED COUNTS:',locked['verified_counts'],flush=True)


## 4. Huấn luyện F trên hai GPU
1.250 cập nhật; batch toàn cục 8, mỗi GPU 4 ảnh caption + 4 cặp xác nhận.
200 cập nhật đầu: CE + 0,1 loss căn chỉnh. Sau đó thêm 0,05 SCST với reward CIDEr-D + 0,5 thưởng đối tượng.
Loss căn chỉnh: max(0; 0,2 − điểm đúng + điểm sai), dùng cặp đã xác nhận trên cùng ảnh.
Lịch cố định 5.000 lượt sai quan hệ và 5.000 lượt sai người; không dùng loss giả nhãn E cũ.
Giữ checkpoint cuối; không chọn bằng calibration/probe/test. Đây là nhiều giám sát và tính toán hơn A/E.


In [ ]:
TRAIN_LOG = OUTPUT/'run.log'
try:
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
         '--max_restarts=0',REPO/'run_verified.py','worker','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
except Exception:
    run([sys.executable,'-u',REPO/'run_verified.py','archive','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
    raise


## 5. Đánh giá và lưu
Caption greedy val1000, cùng A/E; thêm tắt nhánh, tráo text, tráo vùng.
Matcher trước/sau học trên calibration và probe cố định; có toàn ảnh và xoá đặc trưng làm kiểm tra đối chứng.
Calibration/probe thuộc caption train/restval và đã từng dùng trong nghiên cứu trước, không phải test hoàn toàn mới.
Sai vật thể chỉ kiểm tra chuyển giao, không có cặp âm vật thể được tự suy diễn để học.
Gửi ZIP review; giữ riêng checkpoint.pth trong Saved Output. ZIP không chứa checkpoint/HDF5.


In [ ]:
try:
    run([sys.executable,'-u',REPO/'run_verified.py','evaluate','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
finally:
    run([sys.executable,'-u',REPO/'run_verified.py','archive','--config',PROTOCOL],cwd=REPO,log=TRAIN_LOG)
result = json.loads((OUTPUT/'run_summary.json').read_text())
print(json.dumps(result,indent=2,ensure_ascii=False),flush=True)
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)
print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.pth',flush=True)
